# Kati_tryout — the complete documented pipeline

This notebook contains the Python source for an independent copy of Keats using the **frozen suggested 50 characteristics from Characteristics_tryout**. Data lives in `data/`, scripts in `scripts/`, configurations in `configs/`, and results in `outputs/`. Keats remains separate.

The notebook is a readable source edition and an execution hub. Every source file has its own explained code cell. Source cells use `%%writefile`: running them writes the organised file and **does not execute its model fit**. Calculation-library cells are collapsed initially where the notebook viewer supports it. Use the notebook outline and search to find a model, iteration, or function.

**Build status:** this edition is built and verified before the expensive experiment run. Existing Keats forecasts, coefficients, covariance caches and backtest results are not imported as Kati results. The dashboard reports which results are available.

1. Run **Locate the pipeline**.
2. Read or edit the embedded source sections as needed. Run source cells if you want to materialize that edition.
3. Inspect the dependency plan and run verification.
4. Set the model controls deliberately when ready to start fresh fits.
5. Refresh and explore the charts and tables.

The source notebook requires the accompanying local `data/` folder. It does not contain gigabytes of stock data inside its JSON.


## Locate the pipeline

**Purpose:** select this experiment's folder and establish safe local paths. **Input:** the notebook's working directory. **Output:** `KATI_ROOT`, local folders, and the Python version. This cell does not read credentials, download data, or fit models.


In [ ]:
from pathlib import Path
import os
import sys
import subprocess
import json

candidates = [Path.cwd(), Path.cwd() / "Kati_tryout"]
candidates += [p for p in Path.cwd().parents if p.name == "Kati_tryout"]
KATI_ROOT = next((p.resolve() for p in candidates
                  if p.name == "Kati_tryout" and (p / "data").is_dir()), None)
if KATI_ROOT is None:
    raise FileNotFoundError("Open this notebook in Kati_tryout or its parent repository.")
if not (3, 11) <= sys.version_info[:2] < (3, 14):
    raise RuntimeError("Choose a Python 3.11–3.13 kernel for the project's dependency range.")
for name in ("scripts", "configs", "data", "outputs", "docs", "tests"):
    (KATI_ROOT / name).mkdir(exist_ok=True)
os.chdir(KATI_ROOT)
sys.path.insert(0, str(KATI_ROOT / "scripts"))
print("Pipeline:", KATI_ROOT)
print("Python:", sys.executable)

# Create the embedded source folders before any %%writefile cell runs.
for relative_directory in ['.', 'configs', 'configs/eu_large', 'configs/us_large', 'configs/us_mid', 'scripts', 'scripts/analysis', 'scripts/analysis/ml_comparison', 'scripts/base', 'scripts/iteration_1', 'scripts/iteration_2', 'scripts/iteration_3', 'scripts/iteration_4', 'scripts/iteration_5', 'scripts/jkmp', 'scripts/jkmp/data', 'scripts/jkmp/evaluation', 'scripts/jkmp/models', 'scripts/jkmp/portfolio', 'scripts/legacy_iterations/iteration_1', 'scripts/legacy_iterations/iteration_2', 'scripts/legacy_iterations/iteration_3', 'scripts/shared', 'tests']:
    (KATI_ROOT / relative_directory).mkdir(parents=True, exist_ok=True)


## Research scope and timing

| Run | Markets and methods | Comparison |
|---|---|---|
| Base | Static ML* in US high-cap only; Portfolio ML in US high-cap, US mid-cap and EU high-cap | Four retained base ML results and the requested Market references |
| Iteration 1 | Annual refitting already included in Base | Retained as design; P(better) N/A |
| Iteration 2 | Gamma 5 | Previous accepted version and Base/Static ML*/Market |
| Iteration 3 | Exit 120 and costs x2 | Previous accepted version; rejected changes are not inherited |
| Iteration 4 | Market plus pretest-calibrated 1% TE tilt | Inherits only accepted preceding changes |
| Iteration 5 | Recent-history weighting | Previous accepted version; all results retained |
| Final | Only accepted changes | P(better) >= 80% on the unchanged 2020–2025 test period |

Base return-forecast RFF settings are selected separately on historical members of each market, using `return_model.validation_scope: universe`. Coefficients still fit on screened regional training data. Portfolio ML has its own separate RFF portfolio-weight model and market-specific utility tuning. See `docs/MARKET_SPECIFIC_RFF_TUNING.md`. This source update has not fitted models or refreshed backtest results.

Base ridge calibration uses **validation selection from 101 penalties**. Return-model settings are selected on each market's historical validation members; Portfolio ML selects its own coefficients and settings using preceding portfolio utility. RFF and Static ML* k/u/g procedures and iteration settings are retained.

The current 50 are selected using **US high-cap observations from 2005–2014 only**, within the verified published JKP whitelist. Selection uses association statistics without ML training. No later outcome or other-market observation enters the shortlist. The existing model-refit and portfolio-weighting implementation is preserved for a later explicit run. See outputs/us_high_cap_selection/charts/selection_decision_path_jkp_only.pdf and recommended_50_jkp_only.csv for current results.

The 50 cover 13 published JKP groups, with no provisional labels. Candidates must appear in both published JKP Cluster Labels and Factor Details and in WRDS Global Factor Data: 153 candidates, 152 eligible, 50 selected. The 13 A, 13 T and 24 S reasons retain anchors, group representation and the association/redundancy rule. Covariance themes use published directions. The selected 50 include dolvol_126d as a ranked signal; costs separately use its original unranked dollar-volume value. There are exactly 50 ML inputs.


## Embedded source edition

The following sections contain the actual source, including all calculation libraries and verification tests. Each cell materializes only the named local file. The source is also available as normal files under `scripts/`; editing a script does not automatically update this notebook. Rebuild this edition with `python scripts/build_notebook.py` after reviewing script changes. Running an older notebook's source cells intentionally restores its embedded edition.


## Source — Workflow entry points and preparation


### `scripts/_bootstrap.py`

Resolve only the isolated package; never silently import the original jkmp.

**Role:** Pipeline implementation or configuration. This cell writes the file without running its main function.

**Input:** the embedded source below. **Output:** `scripts/_bootstrap.py`. Expand the code cell to read or edit it.


In [ ]:
%%writefile scripts/_bootstrap.py
"""Resolve only the isolated package; never silently import the original jkmp."""
from pathlib import Path
import sys
for _stream in (sys.stdout,sys.stderr):
    if hasattr(_stream,"reconfigure"):_stream.reconfigure(encoding="utf-8")

ROOT = Path(__file__).resolve().parents[1]
sys.path.insert(0, str(ROOT / "scripts"))
sys.path.insert(0, str(ROOT / "scripts"))

from jkmp.imputation_audit import enable_reporting, reporting_enabled
if not reporting_enabled():
    enable_reporting(Path(sys.argv[0]).name)


def configure(script_file):
    """Safe defaults for running a copied script directly in VS Code."""
    if "--config" not in sys.argv and Path(script_file).name not in {
        "00_download.py", "01_prepare.py", "02_covariance.py", "download_once.py",
        "15_holdout.py", "compare_breadth.py", "compare_halflife.py",
    }:
        folder=Path(script_file).parent.name
        group=folder if folder in ("base","iteration_1","iteration_2","iteration_3") else "base"
        if Path(script_file).name in ("04_benchmarks.py","active_engine.py"):group="iteration_1"
        cfg = ROOT / "configs" / "us_large" / f"{group}.yaml"
        sys.argv += ["--config", str(cfg)]


### `scripts/association_statistics.py`

Transparent predictive diagnostics with pairwise missing-data handling.

**Role:** Pipeline implementation or configuration. This cell writes the file without running its main function.

**Input:** the embedded source below. **Output:** `scripts/association_statistics.py`. Expand the code cell to read or edit it.


In [ ]:
%%writefile scripts/association_statistics.py
"""Transparent predictive diagnostics with pairwise missing-data handling."""
from __future__ import annotations

import numpy as np
import pandas as pd
from scipy.stats import norm


def numeric_array(values):
    a=np.asarray(values)
    if a.dtype.kind=='O':
        return pd.to_numeric(pd.Series(a.ravel()),errors='raise').to_numpy(dtype=float,na_value=np.nan).reshape(a.shape)
    return np.asarray(a,float)


def paired_columns(x,y,min_pairs=20):
    """Pearson and covariance for each column with its own valid target pairs."""
    x=numeric_array(x);y=numeric_array(y)
    if y.ndim==1:y=np.broadcast_to(y[:,None],x.shape)
    if x.shape!=y.shape:raise ValueError('Target matrix and features do not align')
    m=np.isfinite(x)&np.isfinite(y)
    n=m.sum(axis=0)
    a=np.where(m,x,0.);b=np.where(m,y,0.)
    sx=a.sum(axis=0);sy=b.sum(axis=0)
    den=np.maximum(n,1)
    cross=(a*b).sum(axis=0)-sx*sy/den
    xx=(a*a).sum(axis=0)-sx*sx/den
    yy=(b*b).sum(axis=0)-sy*sy/den
    corr=np.divide(cross,np.sqrt(np.maximum(xx,0)*np.maximum(yy,0)),out=np.full(x.shape[1],np.nan),where=(xx>0)&(yy>0)&(n>=min_pairs))
    cov=np.divide(cross,n-1,out=np.full(x.shape[1],np.nan),where=n>=max(2,min_pairs))
    return n,corr,cov


def monthly_return_stats(frame,features,min_pairs=20):
    """Exact pairwise Spearman ranks and raw/rank-return covariance, no imputation."""
    x=frame[features].apply(pd.to_numeric,errors='coerce').replace([np.inf,-np.inf],np.nan)
    y=pd.to_numeric(frame.observed_ret_ld1,errors='coerce').replace([np.inf,-np.inf],np.nan)
    n,pearson,cov=paired_columns(x.to_numpy(dtype=float,na_value=np.nan),y.to_numpy(dtype=float,na_value=np.nan),min_pairs)
    joint=x.notna().to_numpy()&y.notna().to_numpy()[:,None]
    a=x.where(joint).rank(method='average',pct=True)
    b=pd.DataFrame(np.broadcast_to(y.to_numpy()[:,None],x.shape),index=x.index,columns=x.columns).where(joint).rank(method='average',pct=True)
    _,ic,rankcov=paired_columns(a.to_numpy(),b.to_numpy(),min_pairs)
    _,_,rank_ycov=paired_columns(a.to_numpy(),y.to_numpy(),min_pairs)
    top=a.ge(.8).to_numpy()&joint;bottom=a.le(.2).to_numpy()&joint
    yt=np.broadcast_to(y.to_numpy()[:,None],x.shape)
    nt=top.sum(axis=0);nb=bottom.sum(axis=0)
    high=np.divide(np.where(top,yt,0.).sum(axis=0),nt,out=np.full(len(features),np.nan),where=nt>0)
    low=np.divide(np.where(bottom,yt,0.).sum(axis=0),nb,out=np.full(len(features),np.nan),where=nb>0)
    spread=high-low;spread[n<min_pairs]=np.nan
    return pd.DataFrame(dict(characteristic=features,n_stocks=n,pearson=pearson,rank_ic=ic,
        raw_characteristic_return_covariance=cov,rank_rank_covariance=rankcov,
        ranked_characteristic_return_covariance=rank_ycov,gross_quintile_spread=spread,
        observed_feature_count=x.notna().sum().to_numpy(),population_size=len(frame)))


def hac_mean(values):
    x=np.asarray(values,float);valid=np.isfinite(x);n=int(valid.sum())
    if n<3:return dict(n=n,mean=np.nan,se=np.nan,t=np.nan,p=np.nan,ci_low=np.nan,ci_high=np.nan,lags=0)
    mu=x[valid].mean();z=np.where(valid,x-mu,0.)
    lags=min(len(x)-1,int(np.floor(4*(n/100)**(2/9))))
    # Missing ICs retain their calendar positions: zero score contributions do
    # not impute outcomes, and two separated months never become adjacent.
    variance=float(z@z)
    for k in range(1,lags+1):variance+=2*(1-k/(lags+1))*float(z[k:]@z[:-k])
    se=np.sqrt(max(variance,0))/n
    t=mu/se if se>0 else (0. if mu==0 else np.sign(mu)*np.inf)
    return dict(n=n,mean=mu,se=se,t=t,p=float(2*norm.sf(abs(t))),
        ci_low=mu-1.95996398454*se,ci_high=mu+1.95996398454*se,lags=lags)


def bh_adjust(values):
    p=np.asarray(values,float);result=np.full(len(p),np.nan)
    ix=np.flatnonzero(np.isfinite(p));order=ix[np.argsort(p[ix])]
    if not len(order):return result
    v=p[order]*len(order)/np.arange(1,len(order)+1)
    result[order]=np.minimum(1.,np.minimum.accumulate(v[::-1])[::-1]);return result


def pairwise_matrices(x,min_pairs=24):
    """Pairwise complete covariance/correlation; no zero filling of observed data."""
    a=numeric_array(x);m=np.isfinite(a).astype(float);v=np.where(np.isfinite(a),a,0.)
    n=m.T@m;s=v.T@m;q=(v*v).T@m
    denom=np.maximum(n,1.)
    cross=v.T@v-s*s.T/denom
    variance=q-s*s/denom
    corr=np.divide(cross,np.sqrt(np.maximum(variance,0.)*np.maximum(variance.T,0.)),
        out=np.full(n.shape,np.nan),where=(variance>0)&(variance.T>0)&(n>=min_pairs))
    cov=np.divide(cross,n-1,out=np.full(n.shape,np.nan),where=n>=min_pairs)
    return corr,cov,n


def panel_metrics(panel,features,universe,min_pairs=20,prior_state=None,return_state=False):
    rows=[];prior,prior_date=(None,None) if prior_state is None else prior_state
    for date,g in panel.groupby('eom',sort=True):
        result=monthly_return_stats(g,features,min_pairs)
        result['universe']=universe;result['formation_month']=date;result['return_month']=date+pd.offsets.MonthEnd(1)
        ranks=g.set_index('id')[features].replace([np.inf,-np.inf],np.nan).rank(pct=True)
        result['rank_persistence']=np.nan;result['mean_abs_rank_change']=np.nan
        if prior is not None and date==prior_date+pd.offsets.MonthEnd(1):
            ids=ranks.index.intersection(prior.index)
            _,persist,_=paired_columns(ranks.loc[ids].to_numpy(),prior.loc[ids].to_numpy(),min_pairs)
            result['rank_persistence']=persist
            changes=np.abs(ranks.loc[ids].to_numpy()-prior.loc[ids].to_numpy())
            count=np.isfinite(changes).sum(axis=0)
            result['mean_abs_rank_change']=np.divide(np.nansum(changes,axis=0),count,
                out=np.full(len(features),np.nan),where=count>0)
        prior,prior_date=ranks,date;rows.append(result)
    out=pd.concat(rows,ignore_index=True)
    return (out,(prior,prior_date)) if return_state else out


def summarize(monthly):
    rows=[]
    for (universe,char),g in monthly.groupby(['universe','characteristic']):
        ic=hac_mean(g.rank_ic)
        rows.append(dict(universe=universe,characteristic=char,months=len(g),valid_ic_months=ic['n'],
            coverage=g.observed_feature_count.sum()/g.population_size.sum(),
            mean_rank_ic=ic['mean'],ic_hac_t=ic['t'],ic_hac_se=ic['se'],ic_hac_p=ic['p'],
            ic_ci_low=ic['ci_low'],ic_ci_high=ic['ci_high'],mean_pearson=g.pearson.mean(),
            mean_raw_return_covariance=g.raw_characteristic_return_covariance.mean(),
            mean_rank_return_covariance=g.ranked_characteristic_return_covariance.mean(),
            mean_gross_spread_annual=12*g.gross_quintile_spread.mean(),
            rank_persistence=g.rank_persistence.mean(),rank_change_proxy=g.mean_abs_rank_change.mean(),
            sample_start=g.return_month.min(),sample_end=g.return_month.max()))
    out=pd.DataFrame(rows)
    out['ic_hac_p_bh']=out.groupby('universe').ic_hac_p.transform(lambda v:bh_adjust(v.to_numpy()))
    return out


### `scripts/build_notebook.py`

Build the complete documented source notebook without fitting an ML model. Each Python file is embedded as an editable %%writefile cell. Running those cells materializes the organised files; it does not execute their entry points. The final cells inspect the plan, verify inputs, and optionally run models.

**Role:** Pipeline implementation or configuration. This cell writes the file without running its main function.

**Input:** the embedded source below. **Output:** `scripts/build_notebook.py`. Expand the code cell to read or edit it.


In [ ]:
%%writefile scripts/build_notebook.py
"""Build the complete documented source notebook without fitting an ML model.

Each Python file is embedded as an editable %%writefile cell. Running those
cells materializes the organised files; it does not execute their entry points.
The final cells inspect the plan, verify inputs, and optionally run models.
"""
from __future__ import annotations

import ast
import hashlib
import json
from pathlib import Path

ROOT = Path(__file__).resolve().parents[1]


def markdown(text):
    return {"cell_type": "markdown", "metadata": {}, "source": text.strip() + "\n"}


def code(text, *, collapsed=False):
    metadata = {"tags": ["embedded-source"]} if collapsed else {}
    if collapsed:
        metadata.update(jupyter={"source_hidden": True}, collapsed=True)
    return {"cell_type": "code", "execution_count": None, "metadata": metadata,
            "outputs": [], "source": text.rstrip() + "\n"}


def caption(path, source):
    relative = path.relative_to(ROOT).as_posix()
    if path.suffix == ".py":
        try:
            doc = ast.get_docstring(ast.parse(source)) or ""
        except SyntaxError:
            doc = ""
        purpose = " ".join(doc.split())[:550] or "Supporting implementation for the Kati workflow."
    else:
        purpose = "Declarative settings for the local Kati workflow; paths resolve inside Kati_tryout."
    if "tests/" in relative:
        role = "Verification source. Tests run only when explicitly invoked; this cell writes the test file."
    elif "/jkmp/" in relative:
        role = "Calculation library. Other stages import its functions; this cell writes the module."
    elif "/analysis/" in relative or path.stem in {"dashboard", "comparison_analysis", "organise_results"}:
        role = "Reporting source. Reads available local results when called; this cell writes the module."
    else:
        role = "Pipeline implementation or configuration. This cell writes the file without running its main function."
    return f"### `{relative}`\n\n{purpose}\n\n**Role:** {role}\n\n**Input:** the embedded source below. **Output:** `{relative}`. Expand the code cell to read or edit it."


def source_groups():
    scripts = sorted((ROOT / "scripts").rglob("*.py"))
    tests = sorted((ROOT / "tests").rglob("*.py"))
    configs = sorted((ROOT / "configs").rglob("*.yaml")) + [ROOT/'configs/us_high_cap_iterations.json']
    settings = ROOT / "settings.json"
    grouped = [
        ("Workflow entry points and preparation", [p for p in scripts if p.parent == ROOT / "scripts"]),
        ("Shared data and estimation stages", [p for p in scripts if p.relative_to(ROOT / "scripts").parts[0] == "shared"]),
        ("Base: US high-cap Static ML* and regional Portfolio ML", [p for p in scripts if p.relative_to(ROOT / "scripts").parts[0] == "base"]),
        ("Iteration 1: annual refitting already in Base", [p for p in scripts if p.relative_to(ROOT / "scripts").parts[0] == "iteration_1"]),
        ("Iteration 2: gamma 5", [p for p in scripts if p.relative_to(ROOT / "scripts").parts[0] == "iteration_2"]),
        ("Iteration 3: exit 120 and doubled costs", [p for p in scripts if p.relative_to(ROOT / "scripts").parts[0] == "iteration_3"]),
        ("Iteration 4: Market plus 1% TE tilt", [p for p in scripts if p.relative_to(ROOT / "scripts").parts[0] == "iteration_4"]),
        ("Iteration 5: recent-history weighting", [p for p in scripts if p.relative_to(ROOT / "scripts").parts[0] == "iteration_5"]),
        ("Legacy iteration source (reference only)", [p for p in scripts if p.relative_to(ROOT / "scripts").parts[0] == "legacy_iterations"]),
        ("Reporting and statistical comparisons", [p for p in scripts if p.relative_to(ROOT / "scripts").parts[0] == "analysis"]),
        ("Calculation libraries", [p for p in scripts if p.relative_to(ROOT / "scripts").parts[0] == "jkmp"]),
        ("Verification tests", tests),
        ("Experiment configurations", configs + ([settings] if settings.exists() else [])),
    ]
    included = {p for _, files in grouped for p in files}
    remaining = [p for p in scripts if p not in included]
    if remaining:
        grouped.append(("Additional supporting modules", remaining))
    return grouped


def build():
    cells = [markdown("""
# Kati_tryout — the complete documented pipeline

This notebook contains the Python source for an independent copy of Keats using the **frozen suggested 50 characteristics from Characteristics_tryout**. Data lives in `data/`, scripts in `scripts/`, configurations in `configs/`, and results in `outputs/`. Keats remains separate.

The notebook is a readable source edition and an execution hub. Every source file has its own explained code cell. Source cells use `%%writefile`: running them writes the organised file and **does not execute its model fit**. Calculation-library cells are collapsed initially where the notebook viewer supports it. Use the notebook outline and search to find a model, iteration, or function.

**Build status:** this edition is built and verified before the expensive experiment run. Existing Keats forecasts, coefficients, covariance caches and backtest results are not imported as Kati results. The dashboard reports which results are available.

1. Run **Locate the pipeline**.
2. Read or edit the embedded source sections as needed. Run source cells if you want to materialize that edition.
3. Inspect the dependency plan and run verification.
4. Set the model controls deliberately when ready to start fresh fits.
5. Refresh and explore the charts and tables.

The source notebook requires the accompanying local `data/` folder. It does not contain gigabytes of stock data inside its JSON.
"""), markdown("""
## Locate the pipeline

**Purpose:** select this experiment's folder and establish safe local paths. **Input:** the notebook's working directory. **Output:** `KATI_ROOT`, local folders, and the Python version. This cell does not read credentials, download data, or fit models.
"""), code('''from pathlib import Path
import os
import sys
import subprocess
import json

candidates = [Path.cwd(), Path.cwd() / "Kati_tryout"]
candidates += [p for p in Path.cwd().parents if p.name == "Kati_tryout"]
KATI_ROOT = next((p.resolve() for p in candidates
                  if p.name == "Kati_tryout" and (p / "data").is_dir()), None)
if KATI_ROOT is None:
    raise FileNotFoundError("Open this notebook in Kati_tryout or its parent repository.")
if not (3, 11) <= sys.version_info[:2] < (3, 14):
    raise RuntimeError("Choose a Python 3.11–3.13 kernel for the project's dependency range.")
for name in ("scripts", "configs", "data", "outputs", "docs", "tests"):
    (KATI_ROOT / name).mkdir(exist_ok=True)
os.chdir(KATI_ROOT)
sys.path.insert(0, str(KATI_ROOT / "scripts"))
print("Pipeline:", KATI_ROOT)
print("Python:", sys.executable)
'''), markdown("""
## Research scope and timing

| Run | Markets and methods | Comparison |
|---|---|---|
| Base | Static ML* in US high-cap only; Portfolio ML in US high-cap, US mid-cap and EU high-cap | Four retained base ML results and the requested Market references |
| Iteration 1 | Annual refitting already included in Base | Retained as design; P(better) N/A |
| Iteration 2 | Gamma 5 | Previous accepted version and Base/Static ML*/Market |
| Iteration 3 | Exit 120 and costs x2 | Previous accepted version; rejected changes are not inherited |
| Iteration 4 | Market plus pretest-calibrated 1% TE tilt | Inherits only accepted preceding changes |
| Iteration 5 | Recent-history weighting | Previous accepted version; all results retained |
| Final | Only accepted changes | P(better) >= 80% on the unchanged 2020–2025 test period |

Base return-forecast RFF settings are selected separately on historical members of each market, using `return_model.validation_scope: universe`. Coefficients still fit on screened regional training data. Portfolio ML has its own separate RFF portfolio-weight model and market-specific utility tuning. See `docs/MARKET_SPECIFIC_RFF_TUNING.md`. This source update has not fitted models or refreshed backtest results.

Base ridge calibration uses **validation selection from 101 penalties**. Return-model settings are selected on each market's historical validation members; Portfolio ML selects its own coefficients and settings using preceding portfolio utility. RFF and Static ML* k/u/g procedures and iteration settings are retained.

The current 50 are selected using **US high-cap observations from 2005–2014 only**, within the verified published JKP whitelist. Selection uses association statistics without ML training. No later outcome or other-market observation enters the shortlist. The existing model-refit and portfolio-weighting implementation is preserved for a later explicit run. See outputs/us_high_cap_selection/charts/selection_decision_path_jkp_only.pdf and recommended_50_jkp_only.csv for current results.

The 50 cover 13 published JKP groups, with no provisional labels. Candidates must appear in both published JKP Cluster Labels and Factor Details and in WRDS Global Factor Data: 153 candidates, 152 eligible, 50 selected. The 13 A, 13 T and 24 S reasons retain anchors, group representation and the association/redundancy rule. Covariance themes use published directions. The selected 50 include dolvol_126d as a ranked signal; costs separately use its original unranked dollar-volume value. There are exactly 50 ML inputs.
"""), markdown("""
## Embedded source edition

The following sections contain the actual source, including all calculation libraries and verification tests. Each cell materializes only the named local file. The source is also available as normal files under `scripts/`; editing a script does not automatically update this notebook. Rebuild this edition with `python scripts/build_notebook.py` after reviewing script changes. Running an older notebook's source cells intentionally restores its embedded edition.
""")]
    hashes = {}
    count = 0
    for group, paths in source_groups():
        if not paths:
            continue
        cells.append(markdown(f"## Source — {group}"))
        for path in paths:
            relative = path.relative_to(ROOT).as_posix()
            content = path.read_text(encoding="utf-8")
            hashes[relative] = hashlib.sha256((content.rstrip()+"\n").encode("utf-8")).hexdigest()
            cells.append(markdown(caption(path, content)))
            cells.append(code(f"%%writefile {relative}\n{content}", collapsed=True))
            count += 1
    cells.extend([
        markdown("""
## US high-cap correlation and covariance selection — no ML training

**Purpose:** reproduce the current 50 from 153 verified published JKP candidates using only US high-cap observations in 2005–2014. **Input:** the self-contained US high-cap snapshot and published reference metadata. **Output:** rankings, A/T/S reasons, 50×50 feature matrices, return-augmented matrices, per-stock associations and interactive charts. This optional cell performs no ML fit and does not install a changed list over fitted models.
"""), code('''RECOMPUTE_ASSOCIATIONS = False
if RECOMPUTE_ASSOCIATIONS:
    subprocess.run([sys.executable, "-X", "utf8", str(KATI_ROOT / "scripts/select_characteristics.py")],
                   cwd=KATI_ROOT, check=True)
else:
    print("Existing association report: outputs/us_high_cap_selection/index.html")
'''),
        markdown("""
## Inspect the workflow before fitting

**Purpose:** list the real job dependencies. **Input:** organised scripts/configurations. **Output:** printed execution plan. `--list-jobs` does not launch model fits. The first run needs fresh forecasts, covariance and portfolio coefficients for the new features.
"""), code('''command = [sys.executable, "-X", "utf8", str(KATI_ROOT / "scripts/run_all.py"),
           "--group", "all", "--scope", "full", "--universe", "all", "--list-jobs"]
subprocess.run(command, cwd=KATI_ROOT, check=True)
'''), markdown("""
## Verify the local build

**Purpose:** check the frozen inputs, local paths, source/notebook structure and implementation safeguards. **Input:** this local build. **Output:** the verification report and its printed status. This check does not establish portfolio performance; model results require the fresh fitting stages below.
"""), code('''subprocess.run([sys.executable, "-X", "utf8", str(KATI_ROOT / "scripts/verify.py")],
               cwd=KATI_ROOT, check=True)
'''), markdown("""
## Choose whether to run models

**Purpose:** make the long computation explicit. **Input:** the controls below. **Output:** an execution command, followed by model results only if `RUN_MODELS=True`.

Leave both flags false to browse and verify the build. Set `RUN_MODELS=True` for the full current experiment scope. `RUN_FIGURE1` controls the additional frontier fits: false adds `--skip-figure1`, while true includes the frontier stage. The full CLI command is in the user guide. The model run may take many hours and reuses only compatible completed Kati outputs on a later invocation.
"""), code('''RUN_MODELS = False
RUN_FIGURE1 = False
RUN_GROUP = "all"
MARKETS = "all"

runner = KATI_ROOT / "scripts/run_all.py"
model_command = [sys.executable, "-u", "-X", "utf8", str(runner),
                 "--group", RUN_GROUP, "--scope", "full", "--universe", MARKETS]
if not RUN_FIGURE1:
    model_command.append("--skip-figure1")
print("Command:", subprocess.list2cmdline(model_command))
if RUN_MODELS:
    subprocess.run(model_command, cwd=KATI_ROOT, check=True)
else:
    print("Model fits have not been started by this cell.")
'''), markdown("""
## Refresh the results explorer

**Purpose:** create a local dashboard from results that actually exist. **Input:** Kati result files. **Output:** charts, tables and report navigation. Before fitting, the explorer shows selection evidence and marks missing model results as unavailable. It does not substitute Keats performance. Use market, run and measure controls in the explorer where offered.
"""), code('''subprocess.run([sys.executable, "-X", "utf8", str(KATI_ROOT / "scripts/dashboard.py")],
               cwd=KATI_ROOT, check=True)
from IPython.display import HTML, display
dashboard = next((p for p in [KATI_ROOT / "outputs/dashboard.html", KATI_ROOT / "outputs/index.html"]
                  if p.exists()), None)
if dashboard is not None:
    relative = dashboard.relative_to(KATI_ROOT).as_posix()
    display(HTML(f'<p><a href="{relative}" target="_blank">Open the full results explorer</a></p>'
                 f'<iframe src="{relative}" style="width:100%;height:850px;border:1px solid #d5dee5"></iframe>'))
else:
    print("No dashboard has been generated yet; inspect the dashboard command's output.")
'''), markdown("""
## Explain the frozen 50 and their groups

**Purpose:** display the shortlist explanation, separately from fitted investment performance. **Input:** selection charts copied into Kati outputs. **Output:** presentation overview and detailed evidence chart, when available. The three selection codes mean **A** published group anchor, **T** minimum theme representation, and **S** balanced score with a redundancy penalty. They are reasons for inclusion, not ML coefficients or importance weights.
"""), code('''from IPython.display import Image, display
for filename in ("selection_groups_and_reasons.png", "selected50_rank_correlation.png", "selected50_rank_covariance.png"):
    image_path = KATI_ROOT / "outputs/us_high_cap_selection/charts" / filename
    if image_path.exists():
        display(Image(filename=str(image_path)))
    else:
        print(filename, "is not present; the frozen names and metadata are under data/characteristics/.")
'''), markdown("""
## Where to inspect completed results

| Question | Local result location |
|---|---|
| Which characteristic data and labels are used? | `data/characteristics/` and `data/SNAPSHOT_MANIFEST.json` |
| Which base model/market performs better? | `outputs/base/overall analysis/` |
| What changed in a specific iteration? | `outputs/iteration_N/us_large/overall analysis/` |
| How do the four Portfolio ML versions compare? | `outputs/base/overall analysis/us_high_cap_progression/` |
| How do all models and iterations compare? | `outputs/base/overall analysis/all_models_and_iterations/` |
| Where are Figure 1 charts? | `outputs/base/overall analysis/figure1/` and iteration extensions |
| What ran, failed, or was blocked? | `outputs/organisation_audit/run_logs/` for the all-group workflow |

CSV/Parquet tables and charts sit beside their corresponding report. Read the availability/status files before citing a result. A blocked downstream job means its prerequisite did not finish; it is not a zero-return observation. Read `docs/ASSUMPTIONS_AND_CHANGES.md` for timing, imputation, selection and interpretation limits.
""")])
    source_directories = sorted({str(Path(name).parent).replace("\\", "/") for name in hashes})
    cells[2]["source"] += "\n# Create the embedded source folders before any %%writefile cell runs.\n"
    cells[2]["source"] += f"for relative_directory in {source_directories!r}:\n    (KATI_ROOT / relative_directory).mkdir(parents=True, exist_ok=True)\n"
    notebook = {"cells": cells,
                "metadata": {"kernelspec": {"display_name": "Python 3", "language": "python", "name": "python3"},
                             "language_info": {"name": "python", "version": "3.13"},
                             "kati_source_manifest": hashes,
                             "kati_source_hash_convention": "UTF-8 text, LF newlines, trailing whitespace/newlines canonicalized to one final newline",
                             "kati_source_file_count": count,
                             "kati_model_results_embedded": False},
                "nbformat": 4, "nbformat_minor": 5}
    for i, cell in enumerate(cells):
        cell["id"] = f"kati-{i:04d}"
    target = ROOT / "Kati_tryout.ipynb"
    target.write_text(json.dumps(notebook, ensure_ascii=False, indent=1) + "\n", encoding="utf-8")
    print(f"Built {target}: {count} embedded files, {len(cells)} explained cells; no model fits run.")
    return target


if __name__ == "__main__":
    build()


### `scripts/comparison_analysis.py`

Saved-output cross-market and US high-cap iteration comparisons. No investment model is fitted here. Market references use the existing market-target and portfolio-accounting routines on each base stock basket.

**Role:** Reporting source. Reads available local results when called; this cell writes the module.

**Input:** the embedded source below. **Output:** `scripts/comparison_analysis.py`. Expand the code cell to read or edit it.


In [ ]:
%%writefile scripts/comparison_analysis.py
"""Saved-output cross-market and US high-cap iteration comparisons.

No investment model is fitted here. Market references use the existing
market-target and portfolio-accounting routines on each base stock basket.
"""
from __future__ import annotations

import argparse
import html
import json
from pathlib import Path
import sys

import numpy as np
import pandas as pd

from workflow import ROOT, UNIVERSES, config_path
from jkmp.config import load_config

sys.path.insert(0, str(ROOT / "scripts"))
from analysis.ml_comparison.data import sha256
from analysis.ml_comparison.metrics import statistics, annual, paths_and_rolling, metric_dictionary
from analysis.ml_comparison.inference import compare_pairs, bootstrap_rank_stability, newey_west_mean

BASE_OUT = ROOT / "outputs/base/overall analysis"
NAMES = {"us_large": "US high-cap", "us_mid": "US mid-cap", "eu_large": "EU high-cap"}
BASE_PF = "US high-cap Portfolio ML base"
STATIC = "US high-cap Static ML"
STATIC_STAR = "US high-cap Static ML*"
MARKET = "US high-cap Market"
ITERATIONS = {i: f"US high-cap Portfolio ML iteration {i}" for i in range(1,6)}
STYLE = """<style>body{max-width:1180px;margin:40px auto;padding:0 24px;font:16px/1.55 system-ui;color:#23313c}h1{font-size:30px}h2{margin-top:32px}table{border-collapse:collapse;font-size:13px;width:100%}th,td{padding:9px;border:1px solid #d5dee5;text-align:right}th:first-child,td:first-child{text-align:left}th{background:#e9eff4}.scroll{overflow:auto}img{max-width:100%;margin:18px 0}a{color:#125981}.note{background:#eef3f7;padding:15px;border-radius:6px}</style>"""


def market_dir(universe):
    return BASE_OUT / "market_references" / universe


def build_market_reference(universe):
    """Compute only Market, never unused equal-weight/min-variance strategies."""
    from jkmp.portfolio.benchmarks import market_targets
    from jkmp.portfolio.accounting import add_start_weights, portfolio_returns
    cfg = load_config(config_path(universe))
    panel_path = cfg.paths.processed / "panel.parquet"
    wealth_path = cfg.paths.processed / "wealth.parquet"
    panel, wealth = pd.read_parquet(panel_path), pd.read_parquet(wealth_path)
    dates = pd.date_range(cfg.dates.test_start, cfg.dates.test_end, freq="ME") - pd.offsets.MonthEnd(1)
    universe_rows = panel.loc[panel.in_universe & panel.eom.isin(dates)]
    targets = market_targets(universe_rows)
    weights = add_start_weights(targets, panel, wealth, dates, cfg.portfolio.charge_exit_tc)
    pf = portfolio_returns(weights, panel, wealth).assign(type="Market")
    folder = market_dir(universe)
    folder.mkdir(parents=True, exist_ok=True)
    weights.assign(type="Market").to_parquet(folder / "weights.parquet", index=False)
    pf.to_parquet(folder / "pf.parquet", index=False)
    provenance = dict(universe=universe, method="value-weighted existing base stock basket",
        panel_sha256=sha256(panel_path), wealth_sha256=sha256(wealth_path),
        pf_sha256=sha256(folder / "pf.parquet"), cost_convention="saved modelled quadratic costs included",
        initialisation="same existing add_start_weights convention, first test formation month",
        external_index=False, models_fitted=False)
    (folder / "provenance.json").write_text(json.dumps(provenance, indent=2), encoding="utf-8")
    return 0


def base_sources():
    records = []
    for universe in UNIVERSES:
        root = ROOT / "outputs/base" / universe
        for method, rel, kind in (("Portfolio ML", "portfolio_ml/rff/pf.parquet", "Portfolio-ML"),
                                  ("Static ML*", "static_ml_star/pf.parquet", "Static-ML*"),
                                  ("Market", None, "Market")):
            if kind=="Static-ML*" and universe!="us_large":continue
            label = f"{NAMES[universe]} {method}" + (" base" if method == "Portfolio ML" else "")
            records.append(dict(strategy=label, universe=universe, group="base",
                path=market_dir(universe) / "pf.parquet" if rel is None else root / rel,
                expected_type=kind, te_target=np.nan))
    return records


def iteration_source(number, target=None):
    return dict(strategy=ITERATIONS[number], universe="us_large", group=f"iteration_{number}",
        path=ROOT / f"outputs/iteration_{number}/us_large/pf.parquet",
        expected_type="Portfolio-ML", te_target=.01 if number in (4,5) else np.nan)


def iteration_comparison_sources(number):
    """Current iteration plus all earlier versions; never include a later one."""
    if number not in range(1,6):raise ValueError("Iteration must be 1 through 5")
    return iteration_baseline_sources()+[iteration_source(i) for i in range(1,number+1)]


def iteration_baseline_sources():
    """Current US high-cap base references, including Static ML*."""
    return [r for r in base_sources() if r['universe']=='us_large']


def load_sources(records):
    """Require the entire declared test calendar; never fill, trim or deduplicate."""
    frames, inventory = [], []
    fields = ["r", "tc", "turnover", "inv", "shorting"]
    for record in records:
        path = record["path"]
        pf = pd.read_parquet(path).copy()
        if set(pf.type) != {record["expected_type"]}:
            raise ValueError(f"Unexpected strategy type: {path}")
        pf["eom_ret"], pf["eom"] = pd.to_datetime(pf.eom_ret), pd.to_datetime(pf.eom)
        cfg = load_config(config_path(record["universe"]))
        expected = pd.date_range(cfg.dates.test_start, cfg.dates.test_end, freq="ME")
        pf = pf.sort_values("eom_ret")
        if not pd.DatetimeIndex(pf.eom_ret).equals(expected):
            raise ValueError(f"Incomplete or duplicate test months: {path}")
        if not (pf.eom + pd.offsets.MonthEnd(1)).eq(pf.eom_ret).all():
            raise ValueError(f"Formation/return mismatch: {path}")
        if not np.isfinite(pf[fields].to_numpy(float)).all():
            raise ValueError(f"Missing or nonfinite performance: {path}")
        frames.append(pf.assign(strategy=record["strategy"], universe=record["universe"], group=record["group"]))
        inventory.append({**{k: v for k, v in record.items() if k != "expected_type"},
            "path": str(path.relative_to(ROOT)), "sha256": sha256(path), "months": len(pf),
            "first_return": str(pf.eom_ret.min().date()), "last_return": str(pf.eom_ret.max().date()),
            "excluded_months": 0})
    result = pd.concat(frames, ignore_index=True)
    if result.duplicated(["strategy", "eom_ret"]).any():
        raise ValueError("Repeated comparison source")
    return result, pd.DataFrame(inventory)


def relative_metrics(monthly, benchmark):
    """Paper IR uses gross-active SD; net-active SD IR is separately reported."""
    b = benchmark.set_index("eom_ret")
    rows = []
    for label, d in monthly.groupby("strategy", sort=False):
        d = d.set_index("eom_ret").sort_index()
        ref = b.reindex(d.index)
        if ref[["r", "tc"]].isna().any().any():
            raise ValueError("Benchmark dates differ")
        gross = (d.r - ref.r).to_numpy()
        net = (d.r - d.tc - ref.r + ref.tc).to_numpy()
        te_g, te_n = np.std(gross, ddof=1) * np.sqrt(12), np.std(net, ddof=1) * np.sqrt(12)
        test = newey_west_mean(net)
        rows.append(dict(strategy=label, benchmark=benchmark.strategy.iloc[0], months=len(d),
            active_gross_return=12 * gross.mean(), active_net_return=12 * net.mean(),
            tracking_error_gross=te_g, tracking_error_net=te_n,
            information_ratio_paper=12 * net.mean() / te_g if te_g > 1e-12 else np.nan,
            information_ratio_net_sd=12 * net.mean() / te_n if te_n > 1e-12 else np.nan,
            beta_net=np.cov((d.r-d.tc), (ref.r-ref.tc), ddof=1)[0, 1] / (ref.r-ref.tc).var(ddof=1),
            months_beating_benchmark=int((net > 0).sum()), active_net_hac_p=test["p"],
            active_net_hac_ci_low=12 * (test["mean"] - 1.959964 * test["se"]),
            active_net_hac_ci_high=12 * (test["mean"] + 1.959964 * test["se"]),
            inference_note="Unadjusted descriptive HAC test; paired utility and Sharpe files apply Holm"))
    return pd.DataFrame(rows)


def charts(monthly, stats, out):
    import matplotlib
    matplotlib.use("Agg")
    import matplotlib.pyplot as plt
    folder = out / "charts"
    folder.mkdir(exist_ok=True)
    colors = dict(zip(monthly.strategy.unique(), plt.get_cmap("tab10").colors))
    fig, axes = plt.subplots(2, 1, figsize=(12, 8), sharex=True)
    for label, d in monthly.groupby("strategy", sort=False):
        d = d.sort_values("eom_ret")
        axes[0].plot(d.eom_ret, 100 * (d.r-d.tc).cumsum(), label=label, color=colors[label])
        axes[1].plot(d.eom_ret, 100 * d.tc.cumsum(), color=colors[label])
    axes[0].set(ylabel="Percentage points", title="Cumulative net excess return   arithmetic")
    axes[1].set(ylabel="Percentage points", title="Cumulative modelled trading costs")
    for ax,values in ((axes[0],monthly.r-monthly.tc),(axes[1],monthly.tc)):
        if abs(values).max()>.5:
            ax.set_yscale("symlog",linthresh=10)
            ax.set_title(ax.get_title()+"   symmetric log scale")
    axes[0].legend(fontsize=9, loc="best")
    for ax in axes: ax.grid(alpha=.2)
    fig.tight_layout(); fig.savefig(folder / "net_returns_and_costs.png", dpi=160); plt.close(fig)
    fig, axes = plt.subplots(1, 2, figsize=(12, 5.5))
    x = np.arange(len(stats))
    axes[0].barh(x, 100*stats.R_TC, color=[colors[s] for s in stats.strategy])
    axes[0].set_yticks(x, stats.strategy, fontsize=9); axes[0].set_xlabel("Annual net excess return (%)")
    axes[1].barh(x, stats.SR_net, color=[colors[s] for s in stats.strategy])
    axes[1].set_yticks(x, [""]*len(x)); axes[1].set_xlabel("Paper net Sharpe")
    if stats.R_TC.abs().max()>1:
        axes[0].set_xscale("symlog",linthresh=10);axes[0].set_xlabel("Annual net excess return (%; symmetric log scale)")
    if stats.SR_net.abs().max()>5:
        axes[1].set_xscale("symlog",linthresh=1);axes[1].set_xlabel("Paper net Sharpe (symmetric log scale)")
    for ax in axes: ax.grid(axis="x", alpha=.2); ax.axvline(0, color="#555", lw=.7)
    fig.tight_layout(); fig.savefig(folder / "return_and_sharpe.png", dpi=160); plt.close(fig)
    rolling = paths_and_rolling(monthly)
    fig, axes = plt.subplots(2, 1, figsize=(12, 8), sharex=True)
    for label, d in rolling.groupby("strategy", sort=False):
        axes[0].plot(d.eom_ret, d.rolling_net_sharpe_paper, label=label, color=colors[label])
        axes[1].plot(d.eom_ret, 100*d.drawdown_excess, color=colors[label])
    axes[0].set_title("Rolling 24-month paper net Sharpe"); axes[0].legend(fontsize=8)
    axes[1].set(title="Additive net excess-return drawdown", ylabel="Percentage points")
    for ax in axes: ax.grid(alpha=.2)
    fig.tight_layout(); fig.savefig(folder / "rolling_and_drawdown.png", dpi=160); plt.close(fig)


def overview_table(stats):
    show = stats[["strategy", "n", "R", "TC", "R_TC", "Vol", "SR_net", "Utility", "Lev", "Turnover"]].copy()
    for c in ("R", "TC", "R_TC", "Vol", "Utility"): show[c] *= 100
    return show.rename(columns={"R":"Gross %", "TC":"Cost %", "R_TC":"Net %", "Vol":"Risk %", "SR_net":"Net Sharpe", "Utility":"Utility %", "Lev":"Leverage"})


def economic_findings(stats, monthly, benchmark, inference):
    ranked = stats.sort_values("Utility", ascending=False)
    best = ranked.iloc[0]
    findings = [f"On the saved 2020-2025 sample, {best.strategy} has the highest mean-variance utility ({best.Utility:.2%} per year at evaluation gamma 10). This is an in-sample comparison of historical backtests, not a forecast of future leadership."]
    for row in stats.itertuples():
        findings.append(f"{row.strategy}: gross excess return {row.R:.2%}, modelled cost {row.TC:.2%}, net excess return {row.R_TC:.2%} per year; paper net Sharpe {row.SR_net:.3f}, gross leverage {row.Lev:.2f} and monthly turnover {row.Turnover:.2%}.")
    rel = relative_metrics(monthly, benchmark)
    for r in rel.loc[rel.strategy.ne(benchmark.strategy.iloc[0])].itertuples():
        findings.append(f"Against {r.benchmark}, {r.strategy} has annual net return difference {r.active_net_return:+.2%} and paper information ratio {r.information_ratio_paper:.3f}. This difference includes market exposure and universe effects.")
    central = inference.loc[inference.bootstrap_block_months.eq(6)]
    significant = int((central.utility_hac_p_holm < .05).sum())
    findings.append(f"Of {len(central)} pairwise utility comparisons, {significant} have Holm-adjusted HAC p below 0.05. Read paired confidence intervals before describing a difference as robust. Six years of monthly results provide limited evidence; the tests do not undo prior research selection.")
    if all(label in set(monthly.strategy) for label in (ITERATIONS[2], ITERATIONS[3])):
        a=monthly.loc[monthly.strategy.eq(ITERATIONS[2])].sort_values("eom_ret")
        b=monthly.loc[monthly.strategy.eq(ITERATIONS[3])].sort_values("eom_ret")
        if np.array_equal(a[["r","tc"]].to_numpy(),b[["r","tc"]].to_numpy()):
            findings.append("Iterations 2 and 3 have exactly identical saved monthly returns and costs over 2020-2025. The half-life search selected no decay in every test year; iteration 3 produces no incremental performance in this test period.")
    if (monthly.r-monthly.tc).le(-1).any():
        findings.append("At least one strategy has a monthly net excess return at or below -100%. Additive paths remain arithmetic model diagnostics, not feasible investor wealth.")
    return findings


def write_report(monthly, inventory, out, title, benchmark, reps=2000, context=""):
    out.mkdir(parents=True, exist_ok=True)
    monthly = monthly.assign(dataset=title)
    stats = pd.DataFrame([dict(strategy=s, **statistics(d)) for s, d in monthly.groupby("strategy", sort=False)])
    monthly.to_csv(out / "monthly_returns.csv", index=False)
    inventory.to_csv(out / "source_inventory.csv", index=False)
    stats.to_csv(out / "performance_summary.csv", index=False)
    annual(monthly).to_csv(out / "annual_performance.csv", index=False)
    paths_and_rolling(monthly).to_csv(out / "rolling_and_cumulative.csv", index=False)
    periods = []
    for start, end in ((2020, 2022), (2023, 2025)):
        for s, d in monthly.loc[monthly.eom_ret.dt.year.between(start, end)].groupby("strategy", sort=False):
            periods.append(dict(period=f"{start}-{end}", strategy=s, **statistics(d)))
    pd.DataFrame(periods).to_csv(out / "subperiod_performance.csv", index=False)
    for field, filename in (("r", "gross_return_correlation.csv"), ("net", "net_return_correlation.csv")):
        monthly.assign(net=monthly.r-monthly.tc).pivot(index="eom_ret", columns="strategy", values=field).corr().to_csv(out / filename)
    inference = pd.concat([compare_pairs(monthly, bootstrap_reps=reps, block_length=b) for b in (3, 6, 12)], ignore_index=True)
    inference.to_csv(out / "paired_inference.csv", index=False)
    bootstrap_rank_stability(monthly, bootstrap_reps=reps).to_csv(out / "bootstrap_rank_stability.csv", index=False)
    relative_metrics(monthly, benchmark).to_csv(out / "versus_market.csv", index=False)
    sensitivities = []
    for multiple in (0, .5, 1, 2):
        for s, d in monthly.groupby("strategy", sort=False):
            sensitivities.append(dict(strategy=s, cost_multiple=multiple, **statistics(d.assign(tc=d.tc*multiple))))
    pd.DataFrame(sensitivities).to_csv(out / "fixed_weight_cost_sensitivity.csv", index=False)
    metric_dictionary().to_csv(out / "metric_dictionary.csv", index=False)
    # Independent direct formulas, separate from the reused metric implementation.
    checks = []
    for row in stats.itertuples():
        d = monthly.loc[monthly.strategy.eq(row.strategy)]
        r = d.r.to_numpy(); net = (d.r-d.tc).to_numpy(); variance = sum((r-sum(r)/len(r))**2)/(len(r)-1)
        passed = all(np.isclose(a, b, rtol=1e-11, atol=1e-12) for a,b in (
            (row.R_TC, 12*sum(net)/len(net)), (row.Utility, 12*(sum(net)/len(net)-5*variance)),
            (row.Vol, np.sqrt(12*variance)), (row.TC, 12*sum(d.tc)/len(d))))
        checks.append(dict(strategy=row.strategy, status="PASS" if passed else "FAIL", months=len(d), excluded_months=0))
    pd.DataFrame(checks).to_csv(out / "independent_verification.csv", index=False)
    if not all(c["status"] == "PASS" for c in checks): raise AssertionError("Comparison formula check failed")
    charts(monthly, stats, out)
    notes = economic_findings(stats, monthly, benchmark, inference)
    methods = """All main sources must contain exactly the same 72 return months from January 2020 through December 2025. No missing return is filled and no month is silently dropped. Returns are USD excess returns; subtract modelled trading costs before net comparisons. Reported annual returns are arithmetic monthly means times 12. Utility uses common evaluation gamma 10 and gross-return sample variance. Paper net Sharpe divides mean net returns by gross-return volatility; the CSV also supplies net-volatility Sharpe. The benchmark is the existing value-weighted stock basket, not a downloaded commercial index. Its own modelled trading costs are included. Cumulative returns add monthly net excess returns, matching the paper replication code. Drawdown is the drop from the additive path's running peak including initial zero; multiply by 100 for percentage points, not percent loss of capital. This drawdown is a reporting extension of the additive chart convention; CAGR and geometric Calmar are omitted. Models update annually on expanding past history; this is not a separate unseen holdout. Paired HAC tests, Holm adjustment within each report and utility/Sharpe family, and paired circular block bootstraps (2000 default replications; 3, 6 and 12 months) describe uncertainty conditional on these saved portfolios. Cross-report testing and earlier research choices are not corrected. Cost scenarios scale saved costs at fixed weights, without reoptimisation. Original missing-return assumptions and initialization conventions are retained. Cross-region rankings reflect different stock universes, opportunity sets and risk exposures; they do not isolate forecasting skill or currency-neutral alpha."""
    md = f"# {title}\n\n{context}\n\n## Economic interpretation\n\n" + "\n\n".join("- "+n for n in notes)
    md += "\n\n## Methods assumptions and exclusions\n\n"+methods+"\n\nSee index.html for readable tables and charts. CSV values are decimal fractions unless the metric dictionary says otherwise.\n"
    (out / "README.md").write_text(md, encoding="utf-8")
    (out / "METHODS_ASSUMPTIONS_AND_EXCLUSIONS.md").write_text(methods+"\n\n"+context, encoding="utf-8")
    links = "".join(f'<li><a href="{f.name}">{html.escape(f.stem.replace("_", " "))}</a></li>' for f in sorted(out.glob("*.csv")))
    body = f"<h1>{html.escape(title)}</h1><p>{html.escape(context)}</p><p class='note'>2020–2025 test return months. Percent columns are annualised; Sharpe, leverage and turnover are ratios. All costs are model estimates.</p>"
    body += '<h2>Performance</h2><div class="scroll">'+overview_table(stats).to_html(index=False, float_format=lambda x:f"{x:,.4g}")+"</div>"
    body += "<h2>Economic interpretation</h2><ul>"+"".join("<li>"+html.escape(n)+"</li>" for n in notes)+"</ul>"
    for f in sorted((out / "charts").glob("*.png")): body += f'<img src="charts/{f.name}" alt="{html.escape(f.stem.replace("_", " "))}">'
    body += "<h2>Methods and limits</h2><p>"+html.escape(methods)+"</p><h2>Data and detailed tests</h2><ul>"+links+"</ul>"
    (out / "index.html").write_text('<!doctype html><meta charset="utf-8"><title>'+html.escape(title)+"</title>"+STYLE+body, encoding="utf-8")
    return stats


def run_reports(only="all", reps=2000):
    if only in ('final',*(f'iteration_{i}' for i in range(1,6))):
        from iteration_sequence import report_saved
        return report_saved()
    if only=='all':
        from iteration_sequence import report_saved
        report_saved()
        run_reports('base',reps)
        return run_reports('master',reps)
    records = base_sources() if only in ("base","master","all") else iteration_baseline_sources()
    needed = UNIVERSES if only in ("base", "master", "all") else ("us_large",)
    records = [r for r in records if r["universe"] in needed]
    for universe in needed:
        # Rebuild cheaply from source accounting so cached Market cannot go stale.
        build_market_reference(universe)
    data, inventory = load_sources(records)
    benchmark = data.loc[data.strategy.eq(MARKET)]
    if only in ("base", "all"):
        wanted = [r["strategy"] for r in records if r["expected_type"]!="Market" or r["strategy"]==MARKET]
        keep = data.strategy.isin(wanted)
        write_report(data.loc[keep], inventory.loc[inventory.strategy.isin(wanted)], BASE_OUT,
            "Base overall comparison", benchmark, reps,
            "Four base ML results: US high-cap Static ML* and Portfolio ML in US high-cap, US mid-cap and EU high-cap, plus US high-cap Market. Static ML is excluded for US mid-cap and EU high-cap. Same-region Market comparisons are supplied separately.")
        all_stats = pd.DataFrame([dict(strategy=s, **statistics(d)) for s,d in data.groupby("strategy", sort=False)])
        all_stats.to_csv(BASE_OUT / "all_base_models_and_regional_markets.csv", index=False)
        regional = pd.concat([relative_metrics(data.loc[data.universe.eq(u)], data.loc[data.universe.eq(u)&data.type.eq("Market")]).assign(universe=u) for u in UNIVERSES])
        regional.to_csv(BASE_OUT / "same_region_market_comparison.csv", index=False)
        data.to_csv(BASE_OUT / "all_base_monthly_returns.csv", index=False)
    if only in ("master","all"):
        chosen=[r for r in records if r["expected_type"]!="Market" or r["strategy"]==MARKET]
        combined, inv=load_sources(chosen+[iteration_source(i) for i in range(1,6)])
        write_report(combined,inv,BASE_OUT/"all_models_and_iterations",
            "All retained base ML models and US high-cap Portfolio ML iterations",benchmark,reps,
            "Unchanged three-market Base and five US high-cap sequential experiments. Rejected candidates remain visible but are not inherited. See outputs/iteration_summary for P(better), accepted ancestry and the final strategy.")
    add_navigation(include_iterations=only in ("all","iteration_1","iteration_2","iteration_3"))
    return 0


def add_navigation(include_iterations=True):
    entries = [("Base overall comparison", "outputs/base/overall analysis/index.html"),
               ("Figure 1 implementable frontiers", "outputs/base/overall analysis/figure1/index.html"),
               ("All base ML models and iterations", "outputs/base/overall analysis/all_models_and_iterations/index.html"),
               ("Final US high-cap Portfolio ML progression", "outputs/base/overall analysis/us_high_cap_progression/index.html")]
    entries += [(f"US high-cap iteration {i}", f"outputs/iteration_{i}/us_large/index.html") for i in range(1,6)]
    entries.append(('Five experiment decisions and final strategy','outputs/iteration_summary/index.html'))
    for title, rel in entries:
        if not include_iterations and rel.startswith("outputs/iteration_"):continue
        page = ROOT / rel
        if not page.exists(): continue
        import os
        links = " · ".join(f'<a href="{os.path.relpath(ROOT/r,page.parent).replace(chr(92),"/")}">{html.escape(t)}</a>' for t,r in entries if (ROOT/r).exists())
        text = page.read_text(encoding="utf-8")
        if "<!-- comparison-navigation -->" in text: text = text.split("<!-- comparison-navigation -->")[0]
        extra="".join(f'<li><a href="{p.name}">{html.escape(p.stem.replace("_"," "))}</a></li>' for p in sorted(page.parent.glob("*.csv")) if f'href="{p.name}"' not in text)
        if extra:text+="<h2>Additional comparisons</h2><ul>"+extra+"</ul>"
        page.write_text(text+"<!-- comparison-navigation --><h2>Related comparisons</h2><p>"+links+"</p>", encoding="utf-8")
    for page in [ROOT/"outputs/index.html", ROOT/"outputs/base/index.html"] + ([ROOT/f"outputs/iteration_{i}/index.html" for i in (1,2,3)] if include_iterations else []):
        if not page.exists(): continue
        import os
        href = os.path.relpath(BASE_OUT/"index.html", page.parent).replace("\\", "/")
        text = page.read_text(encoding="utf-8")
        if "<!-- overall-analysis-link -->" in text: text = text.split("<!-- overall-analysis-link -->")[0]
        page.write_text(text+f'<!-- overall-analysis-link --><h2>Overall comparisons</h2><p><a href="{href}">Base overall analysis and US high-cap iteration comparisons</a></p>', encoding="utf-8")


def main(argv=None):
    parser = argparse.ArgumentParser(description=__doc__)
    parser.add_argument("--only-group", choices=("all", "base", *(f'iteration_{i}' for i in range(1,6)), "final", "master"), default="all")
    parser.add_argument("--bootstrap-reps", type=int, default=2000)
    args = parser.parse_args(argv)
    if args.bootstrap_reps < 1: parser.error("bootstrap-reps must be positive")
    return run_reports(args.only_group, args.bootstrap_reps)


if __name__ == "__main__":
    raise SystemExit(main())


### `scripts/dashboard.py`

Build a local interactive viewer of actual Kati results; never fit a model. Selectors switch market, experiment, metric and visible strategies. Missing results remain explicitly unavailable. Arithmetic cumulative returns are shown because unconstrained strategies can have monthly net returns below -100%.

**Role:** Reporting source. Reads available local results when called; this cell writes the module.

**Input:** the embedded source below. **Output:** `scripts/dashboard.py`. Expand the code cell to read or edit it.


In [ ]:
%%writefile scripts/dashboard.py
"""Build a local interactive viewer of actual Kati results; never fit a model.

Selectors switch market, experiment, metric and visible strategies. Missing
results remain explicitly unavailable. Arithmetic cumulative returns are shown
because unconstrained strategies can have monthly net returns below -100%.
"""
from __future__ import annotations

import html
import json
from pathlib import Path

import numpy as np
import pandas as pd

ROOT = Path(__file__).resolve().parents[1]
MARKETS = {"us_large": "US high-cap", "us_mid": "US mid-cap", "eu_large": "EU high-cap"}


def collect_series(root=ROOT):
    """Read only local completed portfolio files; keep reference identities clear."""
    root = Path(root)
    records = []
    for market in MARKETS:
        base = root / "outputs/base" / market
        folders=([("static_ml_star", "Static ML*")] if market=="us_large" else [])+[("portfolio_ml/rff", "Portfolio ML base")]
        for folder, label in folders:
            records.append((market, "base", label, base / folder / "pf.parquet"))
        records.append((market, "base", "Market (base basket)", root / "outputs/base/overall analysis/market_references" / market / "pf.parquet"))
    for number in range(1,6):
        base = root / f"outputs/iteration_{number}/us_large"
        records.append(('us_large',f'iteration_{number}',f'Portfolio ML iteration {number}',base/'pf.parquet'))
    series = []
    for market, group, label, path in records:
        if not path.exists():
            continue
        frame = pd.read_parquet(path)
        if "type" in frame and label.startswith("Market"):
            frame = frame[frame.type.eq("Market")]
        elif "type" in frame:
            expected = "Static-ML*" if label == "Static ML*" else ("Portfolio-ML" if group == "base" else "Portfolio-ML (active)")
            if group == "base" and set(frame.type) != {expected}:
                raise ValueError(f"Unexpected strategy identity in {path}")
        if frame.empty:
            continue
        required = {"eom_ret", "r", "tc", "turnover"}
        if not required.issubset(frame):
            raise ValueError(f"Incomplete portfolio result: {path}")
        frame = frame.sort_values("eom_ret")
        if frame.eom_ret.duplicated().any():
            raise ValueError(f"Duplicate result month: {path}")
        values = frame[["r", "tc", "turnover"]].to_numpy(float)
        if not np.isfinite(values).all():
            raise ValueError(f"Nonfinite result: {path}")
        net = values[:, 0] - values[:, 1]
        sd = float(np.std(net, ddof=1)) if len(net) > 1 else 0.
        points = [dict(date=pd.Timestamp(row.eom_ret).strftime("%Y-%m-%d"), gross=float(row.r), cost=float(row.tc), net=float(row.r-row.tc), turnover=float(row.turnover)) for row in frame.itertuples()]
        series.append(dict(market=market, group=group, label=label, source=str(path.relative_to(root)).replace("\\", "/"), points=points,
            summary=dict(months=len(frame), first=points[0]["date"], last=points[-1]["date"], annual_net=float(12*net.mean()), annual_cost=float(12*values[:, 1].mean()), net_vol=float(np.sqrt(12)*sd), net_sharpe=float(np.sqrt(12)*net.mean()/sd) if sd else None,
                         nonpositive_wealth_months=int((1+net <= 0).sum()))))
    return series


def build_dashboard(root=ROOT):
    root = Path(root)
    out = root / "outputs"
    out.mkdir(parents=True, exist_ok=True)
    series = collect_series(root)
    feature_path = root / "data/characteristics/selected_feature_metadata.csv"
    features = pd.read_csv(feature_path).replace({np.nan: None}).to_dict("records") if feature_path.exists() else []
    payload = json.dumps(dict(series=series, features=features), allow_nan=False, default=str).replace("<", "\\u003c")
    template = r'''<!doctype html><html lang="en"><meta charset="utf-8"><title>Kati_tryout results</title>
<style>body{max-width:1250px;margin:30px auto;padding:0 24px;font:15px/1.55 system-ui;color:#18364a;background:#f9fbfc}h1{font-size:34px}h2{margin-top:32px}select,input,button{font:inherit}select{padding:7px}label{margin-right:18px;display:inline-block}a{color:#126084}.card{padding:20px;background:white;border:1px solid #dce5eb;border-radius:12px;margin:18px 0}.note{background:#edf3f8;padding:15px;border-radius:8px}.scroll{overflow:auto}table{border-collapse:collapse;width:100%;font-size:13px}th,td{padding:8px;border-bottom:1px solid #dce5eb;text-align:right}th:first-child,td:first-child{text-align:left}th{background:#edf3f8}img{max-width:100%}svg{width:100%;height:auto}.muted{color:#597080}.warn{color:#9b4a18}#legend label{font-size:13px;margin:5px 15px 5px 0}nav a{margin-right:18px}summary{cursor:pointer;font-weight:650}</style>
<h1>Kati_tryout</h1><p>The frozen research 50, using the Keats model mathematics in an independent pipeline.</p>
<nav><a href="us_high_cap_selection/index.html">US high-cap selection and matrices</a><a href="#models">Model results</a><a href="#features">Why these 50?</a><a href="../docs/USER_GUIDE.md">User guide</a><a href="../docs/ASSUMPTIONS_AND_CHANGES.md">Assumptions and changes</a></nav>
<div class="note"><b>Current selection: US high-cap only, 2005–2014.</b> These 50 were ranked using observed correlations, coverage, stability and persistence, with group quotas and redundancy penalties. No ML model was fitted for this selection. Old characteristic archives and their fitted results have been removed. Performance for the new 50 requires a later explicit refit.</div>
<section id="models"><h2>Model comparisons</h2><div class="card"><label>Market <select id="market"><option value="us_large">US high-cap</option><option value="us_mid">US mid-cap</option><option value="eu_large">EU high-cap</option></select></label><label>Comparison <select id="group"><option value="base">Base</option><option value="iteration_1">Iteration 1 + base</option><option value="iteration_2">Iteration 2 + earlier versions</option><option value="iteration_3">All iterations + base</option></select></label><label>Chart <select id="metric"><option value="net">Cumulative arithmetic net excess return</option><option value="gross">Cumulative arithmetic gross excess return</option><option value="cost">Cumulative modelled cost</option><option value="turnover">Monthly turnover</option></select></label><div id="legend"></div><p id="status" class="muted"></p><div id="plot"></div><div class="scroll"><table id="metrics"></table></div><p class="muted">Charts compare common available months. Cumulative return lines add monthly excess returns; they do not represent compounded investor wealth. The table uses the same common months. Modelled costs and unconstrained weights follow Keats.</p></div>
<details class="card"><summary>Native reports, CSV tables and frontier charts</summary><ul id="reports"></ul><p>After running all models, the native overall-analysis reports include statistical comparisons, uncertainty, annual/subperiod tables and static charts. Figure 1 requires its separate fresh grid fits.</p></details></section>
<section id="features"><h2>The selected 50 and their groups</h2><div class="note">50 published JKP group labels; no provisional groups. The candidate whitelist requires both JKP reference files and WRDS source availability. Selection uses US high-cap observations from 2005–2014. A = published anchor, T = group quota, S = association score after redundancy penalty. <a href="us_high_cap_selection/index.html">Open the decision chart, correlations, covariances and matrix downloads</a>.</div>
<div class="card"><label>Group <select id="theme"><option value="">All groups</option></select></label><label>Find <input id="search" placeholder="Characteristic or description"></label><p id="feature_count"></p><div class="scroll"><table id="featuretable"></table></div></div>
<details class="card" open><summary>Selection overview</summary>OVERVIEW</details><details class="card"><summary>Evidence for each of the 50</summary>DETAIL</details></section>
<script>const DATA=PAYLOAD;
const $=x=>document.getElementById(x), esc=x=>String(x??'').replace(/[&<>"']/g,c=>({'&':'&amp;','<':'&lt;','>':'&gt;','"':'&quot;',"'":'&#39;'}[c]));
const colors=['#16739b','#b35a27','#4b885e','#845aa0','#ca9b2d','#c45065','#35636a','#535e96'];
const pct=x=>Number.isFinite(x)?(100*x).toFixed(2)+'%':'—', num=x=>Number.isFinite(x)?x.toFixed(3):'—'; let visible=new Set(DATA.series.map(s=>s.label));
function candidates(){let level=Number($('group').value.split('_')[1]||0),market=$('market').value;return DATA.series.filter(s=>s.market===market&&(s.group==='base'||Number(s.group.split('_')[1])<=level));}
function draw(){let all=candidates();$('legend').innerHTML=all.map((s,i)=>`<label style="color:${colors[i%colors.length]}"><input type="checkbox" data-label="${esc(s.label)}" ${visible.has(s.label)?'checked':''}> ${esc(s.label)}</label>`).join('');$('legend').querySelectorAll('input').forEach(el=>el.onchange=()=>{el.checked?visible.add(el.dataset.label):visible.delete(el.dataset.label);draw()});
let ss=all.filter(s=>visible.has(s.label));if(!ss.length){$('status').textContent=all.length?'Select at least one strategy.':'No completed Kati model results yet. Run the models using the notebook controls or scripts/run_all.py.';$('plot').innerHTML='';$('metrics').innerHTML='';return;}
let dates=ss[0].points.map(p=>p.date).filter(d=>ss.every(s=>s.points.some(p=>p.date===d)));if(!dates.length){$('status').textContent='These strategies have no common result months.';$('plot').innerHTML='';$('metrics').innerHTML='';return;}
let key=$('metric').value; let rows=ss.map(s=>({...s,p:dates.map(d=>s.points.find(p=>p.date===d))}));let curves=rows.map(s=>{let c=0;return s.p.map(p=>key==='turnover'?p[key]:(c+=p[key]));});let vals=curves.flat(),lo=Math.min(0,...vals),hi=Math.max(0,...vals);if(hi===lo)hi=lo+1;let pad=(hi-lo)*.08;lo-=pad;hi+=pad;
const W=1080,H=390,L=78,R=22,T=25,B=50,x=i=>L+(W-L-R)*i/Math.max(1,dates.length-1),y=v=>T+(H-T-B)*(hi-v)/(hi-lo);
let svg=`<svg viewBox="0 0 ${W} ${H}" role="img" aria-label="Selected strategy comparison">`;
for(let i=0;i<5;i++){let v=lo+(hi-lo)*i/4;svg+=`<line x1="${L}" x2="${W-R}" y1="${y(v)}" y2="${y(v)}" stroke="#dde5eb"/><text x="${L-9}" y="${y(v)+4}" text-anchor="end" font-size="12">${key==='turnover'?v.toFixed(1):pct(v)}</text>`;}
curves.forEach((v,j)=>{let index=all.findIndex(s=>s.label===rows[j].label);svg+=`<polyline fill="none" stroke="${colors[index%colors.length]}" stroke-width="2.5" points="${v.map((z,i)=>`${x(i)},${y(z)}`).join(' ')}"/>`;});
[0,Math.floor((dates.length-1)/2),dates.length-1].forEach(i=>svg+=`<text x="${x(i)}" y="${H-15}" text-anchor="middle" font-size="12">${dates[i]}</text>`);$('plot').innerHTML=svg+'</svg>';
$('status').textContent=`${dates.length} common monthly observations: ${dates[0]} to ${dates.at(-1)}. Source files shown below.`;
$('metrics').innerHTML='<tr><th>Strategy</th><th>Annual net return</th><th>Annual cost</th><th>Annual net volatility</th><th>Net Sharpe</th><th>Months net ≤ −100%</th><th>Data</th></tr>'+rows.map(s=>{let n=s.p.length,mean=s.p.reduce((a,p)=>a+p.net,0)/n,sd=n>1?Math.sqrt(s.p.reduce((a,p)=>a+(p.net-mean)**2,0)/(n-1)):0,cost=s.p.reduce((a,p)=>a+p.cost,0)/n;return `<tr><td>${esc(s.label)}</td><td>${pct(12*mean)}</td><td>${pct(12*cost)}</td><td>${pct(Math.sqrt(12)*sd)}</td><td>${sd?num(Math.sqrt(12)*mean/sd):'—'}</td><td>${s.p.filter(p=>p.net<=-1).length}</td><td><a href="${esc(s.source.replace(/^outputs\//,''))}">Parquet</a></td></tr>`}).join('');}
['market','group','metric'].forEach(id=>$(id).onchange=()=>{if($('market').value!=='us_large')$('group').value='base';draw()});
let themes=[...new Set(DATA.features.map(f=>f.theme))].sort();$('theme').innerHTML+=themes.map(t=>`<option>${esc(t)}</option>`).join('');function features(){let query=$('search').value.toLowerCase(),theme=$('theme').value,ff=DATA.features.filter(f=>(!theme||f.theme===theme)&&`${f.characteristic} ${f.description}`.toLowerCase().includes(query));$('feature_count').textContent=`${ff.length} of ${DATA.features.length} characteristics`;$('featuretable').innerHTML='<tr><th>Characteristic</th><th>Description</th><th>Group</th><th>Group evidence</th><th>US high-cap 2005–2014 rank IC</th><th>US high-cap coverage</th><th>Selection reason</th></tr>'+ff.map(f=>`<tr><td>${esc(f.characteristic)}</td><td style="text-align:left">${esc(f.description)}</td><td>${esc(f.theme)}</td><td>${f.published_jkp_153===true?'Published':'Provisional proxy'}</td><td>${num(Number(f.training_mean_rank_ic))}</td><td>${pct(Number(f.score_coverage))}</td><td>${esc(f.reason_code||f.selection_reason)}</td></tr>`).join('');}['theme','search'].forEach(id=>$(id).oninput=features);
const reports=REPORTS;$('reports').innerHTML=reports.map(r=>`<li><a href="${esc(r.path)}">${esc(r.title)}</a></li>`).join('')||'<li>No fitted-model reports yet.</li>';features();draw();
</script></html>'''
    links = []
    template=template.replace('<option value="iteration_3">All iterations + base</option>','<option value="iteration_3">Iteration 3 + earlier versions</option><option value="iteration_4">Iteration 4 + earlier versions</option><option value="iteration_5">All five experiments + base</option>')
    for rel, title in (("base/overall analysis/index.html", "Base: all markets"), ("iteration_summary/index.html", "Five experiments: P(better), verdicts and final strategy"), ("base/overall analysis/figure1/index.html", "Figure 1 frontiers")):
        if (out / rel).exists():
            links.append(dict(path=rel, title=title))
    for i in range(1,6):
        rel = f"iteration_{i}/us_large/index.html"
        if (out / rel).exists():
            links.append(dict(path=rel, title=f"Iteration {i} comparison"))
    for token, filename in (("OVERVIEW", "selection_groups_and_reasons.png"), ("DETAIL", "selected50_return_association.png")):
        rel = "us_high_cap_selection/charts/" + filename
        content = f'<img src="{rel}" alt="Frozen characteristic selection evidence">' if (out / rel).exists() else '<p>Selection chart not bundled; the table above retains the frozen evidence.</p>'
        template = template.replace(token, content)
    template = template.replace("PAYLOAD", payload).replace("REPORTS", json.dumps(links))
    (out / "dashboard.html").write_text(template, encoding="utf-8")
    (out / "dashboard_status.json").write_text(json.dumps(dict(completed_series=len(series), feature_count=len(features), models_fitted_by_dashboard=False, reported_results="Kati local outputs only"), indent=2)+"\n", encoding="utf-8")
    print(f"Dashboard: {out/'dashboard.html'} ({len(series)} completed model/reference series)")
    return 0


if __name__ == "__main__":
    raise SystemExit(build_dashboard())


### `scripts/figure1.py`

Figure 1 adapted to the retained Keats models, with independently fitted grid points. Base: risk aversion x fund size, US high-cap Static ML* and regional Portfolio ML. No retired ML families are run. Iteration extensions use active TE settings on a separate axis; a TE sweep is never labelled as the paper's absolute-risk frontier.

**Role:** Pipeline implementation or configuration. This cell writes the file without running its main function.

**Input:** the embedded source below. **Output:** `scripts/figure1.py`. Expand the code cell to read or edit it.


In [ ]:
%%writefile scripts/figure1.py
"""Figure 1 adapted to the retained Keats models, with independently fitted grid points.

Base: risk aversion x fund size, US high-cap Static ML* and regional Portfolio ML. No retired ML
families are run. Iteration extensions use active TE settings on a separate
axis; a TE sweep is never labelled as the paper's absolute-risk frontier.
"""
from __future__ import annotations

import argparse
from dataclasses import fields, is_dataclass, replace
import hashlib
import html
import json
import logging
import os
from pathlib import Path
import shutil
import time
from collections.abc import Mapping

import numpy as np
import pandas as pd
from joblib import Parallel, delayed, parallel_config
from scipy.linalg import cho_factor, cho_solve

from workflow import ROOT, UNIVERSES, config_path
from jkmp.config import load_config
from jkmp.covariance import load_covariance
from jkmp.data.prepare import risk_free_monthly
from jkmp.portfolio.grids import grid_inputs, scaled_wealth, point_key, _signal_kwargs
from jkmp.portfolio.portfolio_ml import (build_yearly_moments, save_yearly_moments,
    load_yearly_moments, search_yearly, run_portfolio_ml, signal_set)
from jkmp.portfolio.static_ml_star import hyper_grid, run_static_ml_star
from jkmp.evaluation.metrics import summary_stats
from comparison_analysis import BASE_OUT, NAMES, STYLE, relative_metrics, market_dir, build_market_reference

OUT = BASE_OUT / "figure1"
ENGINE_FILES = ("scripts/jkmp/portfolio/portfolio_ml.py", "scripts/jkmp/portfolio/static_ml.py",
    "scripts/jkmp/portfolio/trading_speed.py", "scripts/jkmp/portfolio/accounting.py",
    "scripts/jkmp/portfolio/recursion.py", "scripts/jkmp/models/random_features.py")
log = logging.getLogger(__name__)


def frontier_methods(universe):
    return ([("Static-ML*","pf_static.parquet")] if universe=="us_large" else [])+[("Portfolio-ML","pf_portfolio.parquet")]


def portfolio_compatible(previous,current):
    """Reuse unchanged Portfolio ML fits only after checking every model/data input."""
    def inputs(record):
        config=dict(record.get("config",{}))
        for key in ("reporting","static_ml","regimes"):config.pop(key,None)
        sources={p:h for p,h in record.get("sources",{}).items() if not p.replace(chr(92),"/").endswith(("/static_ml.py","/static_ml_star.py"))}
        return config,sources
    return bool(previous) and inputs(previous)==inputs(current)


def moments_compatible(previous,current):
    """Ridge and return forecasts do not enter Portfolio ML sufficient statistics."""
    def inputs(record):
        config=dict(record.get('config',{}))
        for key in ('reporting','regimes','static_ml','ridge_grid','return_model'):config.pop(key,None)
        sources={p:h for p,h in record.get('sources',{}).items()
                 if not p.replace(chr(92),'/').endswith(('/predictions.parquet','/static_ml.py','/static_ml_star.py'))}
        return config,sources
    return bool(previous) and inputs(previous)==inputs(current)


def digest(path):
    h = hashlib.sha256()
    with Path(path).open("rb") as stream:
        for block in iter(lambda: stream.read(4*1024*1024), b""): h.update(block)
    return h.hexdigest()


def primitive(value):
    if is_dataclass(value):return {f.name:primitive(getattr(value,f.name)) for f in fields(value)}
    if isinstance(value,Mapping):return {str(k):primitive(v) for k,v in value.items()}
    if isinstance(value,(list,tuple)):return [primitive(x) for x in value]
    if isinstance(value,(str,int,float,bool)) or value is None:return value
    return str(value)


def write_json(path, value):
    path.parent.mkdir(parents=True, exist_ok=True)
    temp=path.with_suffix(".writing.json")
    temp.write_text(json.dumps(value,indent=2,default=str)+"\n",encoding="utf-8")
    temp.replace(path)


def provenance(universe,variant="base"):
    cfg=load_config(config_path(universe,variant))
    sources=[cfg.paths.processed/name for name in ("panel.parquet","predictions.parquet","wealth.parquet",
        "covariance/cov_factor.parquet","covariance/cov_loadings.parquet","covariance/provenance.json")]
    engine=list(ENGINE_FILES)
    if variant=="base":engine[1]="scripts/jkmp/portfolio/static_ml_star.py"
    sources += [cfg.paths.raw/"ff_factors_monthly.parquet"]+[ROOT/p for p in engine]
    sources += [p for region in ("US","EU") for p in (ROOT/f"data/inputs/lag_history_{region}.parquet",) if p.exists()]
    if variant!="base":sources.append(ROOT/"scripts/jkmp/portfolio/active.py")
    record=dict(config=primitive(cfg),sources={str(p):digest(p) for p in sources},
        estimator=("Annually tuned US high-cap Static ML* and regional RFF Portfolio ML" if variant=="base" else "Keats fixed Static ML and expanding utility-trained RFF Portfolio ML"),format_version=2)
    record["signature"]=hashlib.sha256(json.dumps(record,sort_keys=True).encode()).hexdigest()
    return record


def load_inputs(cfg):
    p=cfg.paths.processed
    # grid_inputs uses only stocks ever held. Push that identical filter into
    # the parquet reader rather than merging a million-row regional panel.
    members=pd.read_parquet(p/'panel.parquet',columns=['id'],filters=[('in_universe','=',True)])
    ids=sorted(members.id.unique().tolist())
    columns=list(dict.fromkeys(['id','eom','me','in_universe','lambda','g','tr_ld1','ret_ld1',*cfg.features.names]))
    panel=pd.read_parquet(p/'panel.parquet',columns=columns,filters=[('id','in',ids)])
    preds=pd.read_parquet(p/'predictions.parquet',columns=['id','eom','pred_ld1'],filters=[('id','in',ids)])
    panel=panel.merge(preds,on=["id","eom"],how="left",validate="one_to_one")
    wealth=pd.read_parquet(p/"wealth.parquet")
    cov=load_covariance(p/"covariance")
    rf=risk_free_monthly(pd.read_parquet(cfg.paths.raw/"ff_factors_monthly.parquet"))
    return grid_inputs(cfg,panel,wealth,cov,rf)


def validate_pf(pf, cfg, method):
    expected=pd.date_range(cfg.dates.test_start,cfg.dates.test_end,freq="ME")
    d=pf.sort_values("eom_ret")
    if set(d.type)!={method} or not pd.DatetimeIndex(d.eom_ret).equals(expected):
        raise ValueError(f"Wrong strategy or incomplete calendar: {method}")
    if not np.isfinite(d[["r","tc","turnover","inv","shorting"]].to_numpy(float)).all():
        raise ValueError("Nonfinite frontier performance")
    return d


def valid_point(folder, signature, cfg):
    marker=folder/"completion.json"
    if not marker.exists():return False
    record=json.loads(marker.read_text())
    if record.get("signature")!=signature:return False
    for method,filename in frontier_methods(cfg.paths.outputs.name):
        path=folder/filename
        if not path.exists() or digest(path)!=record.get("hashes",{}).get(filename):return False
        validate_pf(pd.read_parquet(path),cfg,method)
    return True


def point_signature(source_signature,gamma,level):
    return hashlib.sha256(f"{source_signature}|{gamma:.17g}|{level:.17g}".encode()).hexdigest()


def fit_point(universe,gamma,level,source_signature,force=False,previous_signature=None,previous_moments_signature=None):
    """New fit at each (gamma,wealth); source headline portfolios are read-only."""
    cfg=load_config(config_path(universe))
    folder=OUT/universe/"points"/point_key(gamma,level)
    signature=point_signature(source_signature,gamma,level)
    if not force and valid_point(folder,signature,cfg):return dict(point=folder.name,status="reused")
    old=json.loads((folder/"completion.json").read_text()) if (folder/"completion.json").exists() else {}
    portfolio_reuse=(not force and previous_signature is not None and
        old.get("signature")==point_signature(previous_signature,gamma,level) and
        (folder/"pf_portfolio.parquet").exists() and
        digest(folder/"pf_portfolio.parquet")==old.get("hashes",{}).get("pf_portfolio.parquet"))
    folder.mkdir(parents=True,exist_ok=True)
    started=time.monotonic()
    write_json(folder/"status.json",dict(status="running",gamma=gamma,wealth=level,pid=os.getpid()))
    try:
        native=gamma==cfg.investor.gamma and level==cfg.investor.wealth
        if native:
            # Exact anchor point is the already-fitted retained strategy, not a
            # rescaled approximation. It also provides a preservation check.
            copies=[
                            ("portfolio_ml/rff/pf.parquet","pf_portfolio.parquet"),
                            ("portfolio_ml/rff/chosen.csv","chosen_portfolio.csv")]
            if universe=="us_large":copies += [("static_ml_star/pf.parquet","pf_static.parquet"),("static_ml_star/weights.parquet","weights_static.parquet"),("static_ml_star/chosen.csv","chosen_static.csv")]
            for src,dst in copies:
                shutil.copy2(cfg.paths.outputs/src,folder/dst)
            origin="existing exact baseline gamma and wealth"
        else:
            if universe=="us_large" or not portfolio_reuse:
                inp=load_inputs(cfg)
                wealth=scaled_wealth(inp.wealth,level,cfg.investor.wealth)
            if universe=="us_large":
                grid=hyper_grid(cfg.static_ml.k_grid,cfg.static_ml.u_grid,cfg.static_ml.g_grid)
                static=run_static_ml_star(inp.panel,wealth,inp.cov,inp.val_dates,inp.eval_dates,gamma,grid,cfg.portfolio.charge_exit_tc)
                static["pf"].to_parquet(folder/"pf_static.parquet",index=False)
                static["weights"].to_parquet(folder/"weights_static.parquet",index=False)
                static["chosen"].to_csv(folder/"chosen_static.csv",index=False)
                static["validation"].to_csv(folder/"validation_static.csv",index=False)
            if portfolio_reuse:
                validate_pf(pd.read_parquet(folder/"pf_portfolio.parquet"),cfg,"Portfolio-ML")
            else:
                old_moments=point_signature(previous_moments_signature,gamma,level) if previous_moments_signature else None
                fit_portfolio_point(inp,cfg,wealth,gamma,signature,force,folder,old_moments)
            origin="Static ML* retuned where enabled; Portfolio ML reused only after matching source hashes and model config" if portfolio_reuse else "fresh coefficients and past-utility hyperparameter selection at this gamma and wealth"
        for method,filename in frontier_methods(universe):
            validate_pf(pd.read_parquet(folder/filename),cfg,method)
        record=dict(status="complete",signature=signature,gamma=gamma,wealth=level,origin=origin,
            seconds=round(time.monotonic()-started,2),
            hashes={n:digest(folder/n) for _,n in frontier_methods(universe)})
        write_json(folder/"completion.json",record);write_json(folder/"status.json",record)
        print(f"COMPLETE {universe} gamma={gamma:g} wealth=${level/1e9:g}B ({record['seconds']:.1f}s)",flush=True)
        return dict(point=folder.name,status="complete")
    except Exception as error:
        write_json(folder/"status.json",dict(status="failed",gamma=gamma,wealth=level,error=str(error)))
        raise


def fit_portfolio_point(inp,cfg,wealth,gamma,signature,force,folder,previous_signature=None):
    universe=cfg.paths.outputs.name
    pm=cfg.portfolio_ml;results={}
    _,tags,label=signal_set(pm.use_rff,pm.rff_variance_grid)
    # Compact root avoids Windows MAX_PATH in this deeply nested repo.
    # The full signature is retained in each cache and checked below.
    cache=ROOT/"data/figure1"/universe/signature[:16]
    def read_cache(path):
        import zipfile
        if force or not path.exists():return None
        try:return load_yearly_moments(path)
        except (zipfile.BadZipFile,EOFError,ValueError,OSError) as error:
            log.warning('Incomplete/corrupt derived cache %s; rebuilding from observed inputs (%s)',path,error)
            return None
    for tag,g in zip(tags,pm.rff_variance_grid):
        path=cache/f"{tag}.npz"
        ym=read_cache(path)
        if ym is None and not force and previous_signature:
            old=ROOT/'data/figure1'/universe/previous_signature[:16]/f'{tag}.npz'
            if old.exists():
                cached=read_cache(old)
                if cached is not None and cached.meta.get('figure1_signature')==previous_signature:
                    ym=cached
                    # In-memory provenance only; keep the original cache intact.
                    ym.meta['figure1_signature']=signature
        if ym is None or ym.meta.get("figure1_signature")!=signature:
            ym=build_yearly_moments(inp.panel,wealth,inp.cov,inp.rf,inp.pfml_dates,cfg.features.names,
                signal_dates=inp.eval_dates,gamma=gamma,mu=cfg.investor.mu,lb=cfg.universe.lookback_months,
                iterations=cfg.investor.m_iterations,scale_by_vol=pm.scale_by_vol,
                add_constant=pm.add_constant,**_signal_kwargs(cfg,g))
            ym.meta["figure1_signature"]=signature
            path.parent.mkdir(parents=True,exist_ok=True)
            temporary=path.with_name(path.stem+'.writing.npz')
            save_yearly_moments(ym,temporary);temporary.replace(path)
        results[tag]=search_yearly(ym,tag,pm.p_grid,cfg.ridge_grid.values,inp.hp_years,pm.half_life_years)
    result=run_portfolio_ml(results,inp.panel,wealth,inp.cov,inp.rf,inp.eval_dates,inp.eval_dates,
        gamma=gamma,mu=cfg.investor.mu,iterations=cfg.investor.m_iterations,
        charge_exit_tc=cfg.portfolio.charge_exit_tc,label=label)
    for name in ("pf","weights","aim_coefs"):
        result[name].to_parquet(folder/f"{name}_portfolio.parquet",index=False)
    result["chosen"].to_csv(folder/"chosen_portfolio.csv",index=False)
    result["validation"].to_csv(folder/"validation_portfolio.csv",index=False)


def frictionless_reference(inp,cfg):
    """Paper Eq 42, evaluated on forward returns; a diagnostic, not another ML family."""
    targets=np.r_[np.arange(-.5,.50001,.05),.6,.75,1.,2.]/12
    per_target={float(k):[] for k in targets};constraints=[]
    for date in inp.eval_dates:
        d=inp.panel.loc[inp.panel.eom.eq(date)&inp.panel.in_universe].sort_values("id")
        ids=d.id.to_numpy();mu=d.pred_ld1.to_numpy();one=np.ones(len(ids))
        sigma=inp.cov[date].full(ids)
        # Center before solving: the textbook a*c-b*b form suffers severe
        # cancellation when stock forecasts are almost identical.
        if np.ptp(mu)<=1e-10*max(1.,np.max(np.abs(mu))):
            raise ValueError(f"Degenerate frictionless reference: nearly constant forecasts at {date.date()}")
        z=mu-mu.mean()
        solved=cho_solve(cho_factor(sigma),np.column_stack([z,one]))
        x,y=solved[:,0],solved[:,1];c=float(one@y)
        w0=y/c;v=x-y*float(one@x)/c
        mu0=float(mu@w0);delta=float(z@v)
        if delta<=0:raise ValueError(f"Degenerate frictionless reference at {date.date()}")
        for k in targets:
            w=w0+(k-mu0)*v/delta
            if abs(w.sum()-1)>1e-8 or abs(w@mu-k)>1e-8:
                raise ValueError(f"Degenerate frictionless reference: unstable constraints at {date.date()}")
            per_target[float(k)].append(dict(eom=date,eom_ret=date+pd.offsets.MonthEnd(1),r=float(w@d.ret_ld1),
                tc=0.,turnover=np.nan,inv=float(abs(w).sum()),shorting=float(-w.clip(max=0).sum())))
            constraints.append(dict(eom=date,target_monthly=k,weight_sum=float(w.sum()),predicted_return=float(w@mu)))
    monthly=pd.concat([pd.DataFrame(v).assign(target_monthly=k) for k,v in per_target.items()],ignore_index=True)
    summary=pd.DataFrame([dict(target_monthly=k,**summary_stats(pd.DataFrame(v),cfg.investor.gamma).to_dict()) for k,v in per_target.items()])
    return monthly,summary,pd.DataFrame(constraints)


def summarize_market(universe,gammas,levels,signature):
    cfg=load_config(config_path(universe));folder=OUT/universe
    rows=[];monthly=[];checks=[]
    for level in levels:
        for gamma in gammas:
            point=folder/"points"/point_key(gamma,level)
            if not valid_point(point,point_signature(signature,gamma,level),cfg):raise ValueError(f"Incomplete frontier point: {point}")
            for method,name in frontier_methods(universe):
                d=validate_pf(pd.read_parquet(point/name),cfg,method)
                row=dict(universe=universe,type=method,gamma=gamma,wealth=level,**summary_stats(d,gamma).to_dict())
                rows.append(row);monthly.append(d.assign(gamma=gamma,wealth=level))
                # Independent scalar arithmetic check against the metric library.
                r=d.r.to_numpy();tc=d.tc.to_numpy();mean=sum(r)/len(r)
                variance=sum((x-mean)**2 for x in r)/(len(r)-1)
                passed=np.isclose(row["R_TC"],12*(sum(r-tc)/len(r)),rtol=1e-11,atol=1e-12) and np.isclose(row["Vol"],np.sqrt(12*variance))
                if level==0:passed=passed and bool(d.tc.eq(0).all())
                checks.append(dict(type=method,gamma=gamma,wealth=level,status="PASS" if passed else "FAIL"))
    summary=pd.DataFrame(rows);summary.to_csv(folder/"frontier_summary.csv",index=False)
    pd.concat(monthly,ignore_index=True).to_csv(folder/"monthly_returns.csv",index=False)
    pd.DataFrame(checks).to_csv(folder/"independent_verification.csv",index=False)
    if any(c["status"]!="PASS" for c in checks):raise AssertionError("Frontier arithmetic verification failed")
    anchor=summary.loc[summary.gamma.eq(cfg.investor.gamma)&summary.wealth.eq(cfg.investor.wealth)]
    for row in anchor.itertuples():
        rel="static_ml_star/pf.parquet" if row.type=="Static-ML*" else "portfolio_ml/rff/pf.parquet"
        native=pd.read_parquet(cfg.paths.outputs/rel)
        d=pd.concat(monthly).loc[lambda x:x.type.eq(row.type)&x.gamma.eq(row.gamma)&x.wealth.eq(row.wealth)]
        np.testing.assert_allclose(d[["r","tc"]],native.sort_values("eom_ret")[["r","tc"]],rtol=0,atol=0)
    ref_path=folder/"frictionless_reference.csv";ref_status=folder/"frictionless_reference_status.json"
    cached=json.loads(ref_status.read_text()) if ref_status.exists() else {}
    if cached.get('source_signature')!=signature or cached.get('method_version')!=3:
        inp=load_inputs(cfg)
        try:
            m,s,c=frictionless_reference(inp,cfg)
            status=dict(status='available',method_version=3,source_signature=signature,excluded_months=0,
                turnover='not calculated for the diagnostic reference; reported as missing')
        except ValueError as error:
            if not str(error).startswith('Degenerate frictionless reference'):raise
            # Do not draw a shortened-sample or numerically infeasible curve.
            m=pd.DataFrame(columns=['eom','eom_ret','target_monthly','r','tc'])
            s=pd.DataFrame(columns=['target_monthly','n','R','Vol','R_TC'])
            c=pd.DataFrame(columns=['eom','target_monthly','weight_sum','predicted_return'])
            status=dict(status='unavailable',reason=str(error),method_version=3,source_signature=signature,
                excluded_months=0,entire_reference_omitted=True,retained_model_frontiers_affected=False)
        m.to_csv(folder/"frictionless_monthly.csv",index=False);s.to_csv(ref_path,index=False)
        c.to_csv(folder/"frictionless_constraint_checks.csv",index=False);write_json(ref_status,status)
    constraints=pd.read_csv(folder/"frictionless_constraint_checks.csv")
    if not constraints.empty:
        np.testing.assert_allclose(constraints.weight_sum,1,atol=1e-8)
        np.testing.assert_allclose(constraints.predicted_return,constraints.target_monthly,atol=1e-8)
    return summary,pd.read_csv(ref_path)


def verify_grid_accounting(universe,summary):
    """Independently reconcile every fitted point to its saved stock weights."""
    cfg=load_config(config_path(universe));folder=OUT/universe
    panel=pd.read_parquet(cfg.paths.processed/'panel.parquet',columns=['id','eom','ret_ld1','lambda','in_universe'])
    ids=panel.loc[panel.in_universe,'id'].unique()
    dates=pd.date_range(cfg.dates.test_start,cfg.dates.test_end,freq='ME')-pd.offsets.MonthEnd(1)
    panel=panel.loc[panel.id.isin(ids)&panel.eom.isin(dates),['id','eom','ret_ld1','lambda']]
    wealth=pd.read_parquet(cfg.paths.processed/'wealth.parquet').set_index('eom').wealth
    rows=[];names=['r','tc','turnover','inv','shorting']
    for row in summary.itertuples():
        point=folder/'points'/point_key(row.gamma,row.wealth)
        method='static' if row.type=='Static-ML*' else 'portfolio'
        weights_path=point/f'weights_{method}.parquet'
        if not weights_path.exists():
            rel='static_ml_star/weights.parquet' if method=='static' else 'portfolio_ml/rff/weights.parquet'
            weights_path=cfg.paths.outputs/rel
        weights=pd.read_parquet(weights_path)
        d=weights.merge(panel,on=['id','eom'],how='left',validate='one_to_one')
        held=~d['exit']
        if d['lambda'].isna().any() or d.loc[held,'ret_ld1'].isna().any():raise ValueError('Incomplete frontier weight accounting sources')
        d['r_i']=d.w*d.ret_ld1.where(held,0.)
        d['tc_i']=.5*d.eom.map(wealth)*(row.wealth/cfg.investor.wealth)*d['lambda']*(d.w-d.w_start)**2
        d['turnover_i']=(d.w-d.w_start).abs();d['inv_i']=d.w.abs();d['short_i']=-d.w.clip(upper=0)
        calc=d.groupby('eom').agg(r=('r_i','sum'),tc=('tc_i','sum'),turnover=('turnover_i','sum'),inv=('inv_i','sum'),shorting=('short_i','sum'))
        pf=pd.read_parquet(point/f'pf_{method}.parquet').sort_values('eom')
        observed=calc.reindex(pf.eom)[names].to_numpy()
        expected=pf[names].to_numpy()
        passed=np.allclose(observed,expected,rtol=1e-10,atol=1e-12)
        rows.append(dict(type=row.type,gamma=row.gamma,wealth=row.wealth,months=len(pf),
            status='PASS' if passed else 'FAIL',max_return_error=float(abs(observed[:,0]-expected[:,0]).max()),
            max_cost_error=float(abs(observed[:,1]-expected[:,1]).max()),weights_path=str(weights_path.relative_to(ROOT))))
    result=pd.DataFrame(rows);result.to_csv(folder/'accounting_verification.csv',index=False)
    if result.status.ne('PASS').any():raise AssertionError('Frontier weight accounting failed')
    return result


def save_figure(fig,path):
    import matplotlib.pyplot as plt
    path.parent.mkdir(parents=True,exist_ok=True)
    for suffix in ("png","pdf"):
        target=path.with_suffix('.'+suffix)
        try:fig.savefig(target,dpi=180,bbox_inches='tight')
        except PermissionError:
            target=path.with_name(path.name+'_current').with_suffix('.'+suffix)
            fig.savefig(target,dpi=180,bbox_inches='tight')
            log.warning('Existing chart is open/locked; current export saved to %s',target)
    plt.close(fig)


def plot_market(universe,summary,reference):
    import matplotlib
    matplotlib.use("Agg")
    import matplotlib.pyplot as plt
    cfg=load_config(config_path(universe));folder=OUT/universe
    fig,axes=plt.subplots(1,2,figsize=(14,5.3))
    markers={1:"o",5:"^",10:"s",20:"+",100:"x"}
    s=summary.loc[summary.wealth.eq(cfg.investor.wealth)]
    for method,color in (("Portfolio-ML","#126c9a"),("Static-ML*","#d67b29")):
        d=s.loc[s.type.eq(method)].sort_values("Vol")
        if d.empty:continue
        axes[0].plot(100*d.Vol,100*d.R_TC,"-",color=color,label=method)
        for r in d.itertuples():
            axes[0].scatter([100*r.Vol],[100*r.R_TC],marker=markers.get(r.gamma,"o"),s=35,color=color)
        base=d.loc[d.gamma.eq(cfg.investor.gamma)]
        if not base.empty:
            r=base.iloc[0];x=np.linspace(0,max(s.Vol)*1.1,150)
            axes[0].plot(100*x,100*(r.Utility+.5*cfg.investor.gamma*x*x),ls="--",color=color,alpha=.5,lw=.8)
    if not (market_dir(universe)/"pf.parquet").exists():build_market_reference(universe)
    market=summary_stats(pd.read_parquet(market_dir(universe)/"pf.parquet"),cfg.investor.gamma)
    axes[0].scatter([100*market.Vol],[100*market.R_TC],marker="D",s=35,color="#222",label="Market reference",zorder=5)
    levels=sorted(summary.wealth.unique());cmap=plt.get_cmap("viridis")
    for i,level in enumerate(levels):
        d=summary.loc[summary.type.eq("Portfolio-ML")&summary.wealth.eq(level)].sort_values("Vol")
        color=cmap(i/max(len(levels)-1,1))
        axes[1].plot(100*d.Vol,100*d.R_TC,"-",color=color,label="Zero-cost limit" if level==0 else f"${level/1e9:g}B")
        for r in d.itertuples():axes[1].scatter([100*r.Vol],[100*r.R_TC],marker=markers.get(r.gamma,"o"),s=25,color=color)
    for ax in axes:
        if not reference.empty:ax.plot(100*reference.Vol,100*reference.R_TC,ls=":",lw=1,color="#777",label="Frictionless risky-assets reference")
        ax.axhline(0,color="#777",lw=.6);ax.grid(alpha=.2)
        ax.set(xlabel="Realised gross volatility (% per year)",ylabel="Net excess return (% per year)")
        ax.legend(fontsize=7)
        # Reference targets include very high-risk portfolios; keep the view on
        # actual retained-model points, while exporting every reference point.
    for ax,points,extra_r,extra_v in ((axes[0],s,[market.R_TC],[market.Vol]),
                                    (axes[1],summary.loc[summary.type.eq("Portfolio-ML")],[],[])):
        y=np.r_[points.R_TC,extra_r];x=np.r_[points.Vol,extra_v]
        pad=max(.01,float(np.ptp(y))*.15)
        ax.set(xlim=(0,100*max(x)*1.15),ylim=(100*(min(0,y.min())-pad),100*(max(0,y.max())+pad)))
        if not reference.empty and reference.Vol.min()>max(x)*1.15:
            for line in ax.lines:
                if line.get_label()=="Frictionless risky-assets reference":line.set_label("Frictionless ref. (outside displayed risk range)")
        ax.legend(fontsize=7)
    axes[1].legend(fontsize=7,loc="upper left",bbox_to_anchor=(1.02,1))
    axes[0].set_title(f"A  Retained models at ${cfg.investor.wealth/1e9:g}B\nSymbols identify risk aversion")
    axes[1].set_title("B  Portfolio ML by investor size\nEach point is independently fitted")
    fig.suptitle(f"{NAMES[universe]}   Estimated implementable frontier   2020–2025")
    fig.text(.5,.015,"Risk aversion gamma: 1 circle; 5 triangle; 10 square; 20 plus; 100 cross",ha="center",fontsize=9)
    fig.tight_layout(rect=(0,.045,1,1));save_figure(fig,folder/"charts/figure1_implementable_frontier")


def write_market_page(universe,summary):
    cfg=load_config(config_path(universe));folder=OUT/universe
    text=("Figure 1 includes US high-cap Static ML* and regional Portfolio ML. Every risk-aversion/fund-size point refits the portfolio rule; "
        "Portfolio ML reselects its RFF scale, model size and ridge using cumulative past utility. US high-cap Static ML* reselects "
        "k/u/g each December using cumulative preceding utility and existing annual return forecasts. Existing baseline model outputs are reused only at their exact original gamma and wealth. "
        "Zero wealth denotes the exact zero-impact mathematical limit, not a real fund. All wealth figures are anchored at end-2019, "
        "unlike the paper's end-2020 anchor. Realised 2020–2025 gross risk and net excess returns are evaluated on identical 72 months. "
        "Lines join fitted grid points and need not be efficient or monotone out of sample. Dashed curves show gamma-10 indifference curves. "
        "Crossing fund-size curves are possible because these are finite-sample fitted strategies, not the known true optimum. "
        "The dotted risky-assets reference applies the paper's constrained mean-variance formula to saved forecasts and contemporaneous covariance; "
        "it is a zero-cost diagnostic, not an additional headline ML method. Fixed Static ML, Markowitz ML and Factor ML are omitted. US mid-cap and EU high-cap have no Static ML strategy. "
        "No separate unseen holdout is claimed. Original missing-return and initialization conventions remain in force. "
        "Returns and costs are arithmetic model diagnostics, not compounded investor wealth. Unconstrained allocations can imply extreme costs and infeasible losses, in unconstrained rules. "
        "Where the frictionless reference is outside the displayed risk range, the legend says so and its full values remain in the CSV. "
        "No optimality claim follows from choosing the best-looking point on this test plot.")
    reference_status=json.loads((folder/'frictionless_reference_status.json').read_text())
    if reference_status['status']=='unavailable':
        text+=' The optional frictionless reference is unavailable for this market: '+reference_status['reason']+'. No months were dropped or returns fabricated; the retained model frontiers remain complete.'
    (folder/"METHODS_AND_EXCLUSIONS.md").write_text("# Figure 1 methods\n\n"+text+"\n",encoding="utf-8")
    shown=summary[["type","gamma","wealth","R_TC","Vol","SR_net","Utility","TC"]].copy()
    shown["wealth"]/=1e9
    for col in ("R_TC","Vol","Utility","TC"):shown[col]*=100
    shown=shown.rename(columns={"wealth":"Wealth $B","R_TC":"Net return %","Vol":"Risk %","Utility":"Utility %","TC":"Cost %"})
    links=''.join(f'<li><a href="{p.name}">{p.stem.replace("_"," ")}</a></li>' for p in sorted(folder.glob("*.csv")))
    page=f'<h1>{NAMES[universe]} Figure 1</h1><p>{html.escape(text)}</p><img src="charts/figure1_implementable_frontier.png" alt="Two-panel implementable frontier"><p><a href="charts/figure1_implementable_frontier.pdf">Download PDF chart</a></p><div class="scroll">'+shown.to_html(index=False,float_format=lambda x:f"{x:,.4g}")+f'</div><ul>{links}</ul><p><a href="../index.html">All Figure 1 results</a></p>'
    (folder/"index.html").write_text('<!doctype html><meta charset="utf-8">'+STYLE+page,encoding="utf-8")


def main(argv=None):
    parser=argparse.ArgumentParser(description=__doc__)
    parser.add_argument("--selected-universe",choices=("all",*UNIVERSES),default="all")
    parser.add_argument("--jobs",type=int,default=3)
    parser.add_argument("--gammas",type=float,nargs="+")
    parser.add_argument("--wealth-billions",type=float,nargs="+")
    parser.add_argument("--force",action="store_true")
    parser.add_argument("--plots-only",action="store_true")
    parser.add_argument("--include-iterations",action="store_true")
    args=parser.parse_args(argv)
    if args.jobs<1:parser.error("jobs must be positive")
    universes=UNIVERSES if args.selected_universe=="all" else (args.selected_universe,)
    OUT.mkdir(parents=True,exist_ok=True)
    logging.basicConfig(level=logging.INFO,format="%(asctime)s %(message)s")
    for universe in universes:
        cfg=load_config(config_path(universe));gammas=tuple(args.gammas or cfg.investor.gamma_grid)
        levels=tuple(w*1e9 for w in args.wealth_billions) if args.wealth_billions is not None else (0.,*cfg.investor.wealth_grid)
        if any(not np.isfinite(g) or g<=0 for g in gammas) or any(not np.isfinite(w) or w<0 for w in levels):parser.error("Require gamma>0 and wealth>=0")
        if len(set(gammas))!=len(gammas) or len(set(levels))!=len(levels):parser.error("Duplicate grid points")
        if cfg.investor.gamma not in gammas or cfg.investor.wealth not in levels:
            parser.error("Figure 1 must include the configured baseline gamma and wealth anchor")
        record=provenance(universe);folder=OUT/universe;folder.mkdir(parents=True,exist_ok=True)
        old_path=folder/"source_provenance.json"
        previous=json.loads(old_path.read_text()) if old_path.exists() else {}
        moment_signature=previous.get('signature') if moments_compatible(previous,record) else None
        if moment_signature==record['signature']:moment_signature=None
        moment_record=folder/'moment_reuse_provenance.json'
        if moment_record.exists() and moment_signature is None:
            saved=json.loads(moment_record.read_text())
            if moments_compatible(saved,record):moment_signature=saved['signature']
        elif moment_signature and moment_signature!=record['signature']:
            write_json(moment_record,previous)
        reuse_signature=previous.get("signature") if portfolio_compatible(previous,record) else None
        # Keep the previous manifest until point migration finishes so interrupted runs resume.
        migration=folder/"portfolio_reuse_provenance.json"
        if migration.exists() and reuse_signature is None:
            previous=json.loads(migration.read_text())
            reuse_signature=previous.get("signature") if portfolio_compatible(previous,record) else None
        if reuse_signature and reuse_signature!=record["signature"]:write_json(migration,previous)
        write_json(folder/"source_provenance.json",record)
        points=[(g,w) for w in levels for g in gammas]
        print(f"FIGURE 1 {universe}: {len(points)} points, {args.jobs} workers",flush=True)
        if not args.plots_only:
            with parallel_config(backend="loky",inner_max_num_threads=1):
                Parallel(n_jobs=args.jobs,verbose=10)(delayed(fit_point)(universe,g,w,record["signature"],args.force,reuse_signature,moment_signature) for g,w in points)
        summary,reference=summarize_market(universe,gammas,levels,record["signature"])
        verify_grid_accounting(universe,summary)
        plot_market(universe,summary,reference);write_market_page(universe,summary)
        # Data/model sources must be unchanged after fitting new output-only grids.
        after=provenance(universe)
        if after["signature"]!=record["signature"]:raise AssertionError("Source inputs changed during frontier run")
        write_json(folder/"completion.json",dict(status="complete",grid_points=len(points),methods=[m for m,_ in frontier_methods(universe)],
            monthly_rows=len(pd.date_range(cfg.dates.test_start,cfg.dates.test_end,freq="ME"))*len(points)*len(frontier_methods(universe)),signature=record["signature"],source_inputs_unchanged=True))
    if args.include_iterations and "us_large" in universes:
        from iteration_sequence import report_saved
        report_saved()
    plot_combined()
    index_page()
    return 0


def index_page():
    rows=[]
    for universe in UNIVERSES:
        p=OUT/universe/"completion.json"
        if p.exists():rows.append(f'<li><a href="{universe}/index.html">{NAMES[universe]} Figure 1</a></li>')
    text='<h1>Figure 1 Implementable frontiers</h1><p>Independently fitted retained models across risk aversion and investor size.</p><ul>'+''.join(rows)+'</ul>'
    if (OUT/"charts/figure1_combined.png").exists():
        pdf='figure1_combined_current.pdf' if (OUT/'charts/figure1_combined_current.pdf').exists() and (OUT/'charts/figure1_combined_current.pdf').stat().st_mtime_ns>(OUT/'charts/figure1_combined.pdf').stat().st_mtime_ns else 'figure1_combined.pdf'
        text+=f'<h2>Combined base Figure 1: all retained frontiers</h2><img src="charts/figure1_combined.png" alt="Combined method and investor-size frontiers for all three markets"><p><a href="charts/{pdf}">Combined PDF</a> | <a href="combined_frontier_summary.csv">All underlying points</a></p>'
    recommendations=("# Recommended next figures for your iterations\n\n"
        "| Priority | Paper figure | Why it helps | Required work |\n|---|---|---|---|\n"
        "| 1 | Positions by liquidity | Diagnose the exit-120/costs-x2 experiment against its previous accepted setup | Aggregate saved long/short weights into contemporaneous liquidity deciles; no model refit |\n"
        "| 2 | Stock-weight paths | Show iteration 4's benchmark-relative tilts and iteration 5's weighting changes | Plot actual and active weights for consistently available stocks; no model refit |\n"
        "| 3 | Complete Figure 2 gross return net return and utility panels | Makes each iteration's gain or loss relative to its predecessor visible after costs and risk | Saved monthly results suffice; clarify arithmetic accumulation and utility convention |\n"
        "| Later | Figure 5 economic feature importance | Explains which economic themes matter under the fitted strategy, rather than ranking raw correlations | Permute themes, keep fitted coefficients fixed, rebuild portfolio decisions and measure utility loss; repeated counterfactual runs |\n\n"
        "The current five-step sequence has no 250-stock breadth experiment. Borrowing/lending-fee figures require data not currently present. The experiment decision table, returns, drawdowns, relative index and costs/turnover plots are available in outputs/iteration_summary and each iteration folder.\n")
    (OUT/"RECOMMENDED_NEXT_FIGURES.md").write_text(recommendations,encoding="utf-8")
    text+='<p><a href="RECOMMENDED_NEXT_FIGURES.md">Recommended next figures for the iterations</a></p>'
    text+='<p><a href="../../../iteration_summary/index.html">Five US high-cap experiments and final-strategy charts</a></p>'
    (OUT/"index.html").write_text('<!doctype html><meta charset="utf-8">'+STYLE+text,encoding="utf-8")
    from comparison_analysis import add_navigation
    add_navigation(include_iterations=False)


def plot_combined():
    """Combined paper-style A/B figure using every verified retained base curve."""
    import matplotlib
    matplotlib.use("Agg")
    import matplotlib.pyplot as plt
    from matplotlib.lines import Line2D
    frames=[]
    for universe in UNIVERSES:
        folder=OUT/universe;path=folder/"frontier_summary.csv";marker=folder/"completion.json"
        if not path.exists() or not marker.exists():return False
        complete=json.loads(marker.read_text())
        if complete.get("signature")!=provenance(universe)["signature"]:return False
        d=pd.read_csv(path)
        if set(d.type)!={m for m,_ in frontier_methods(universe)}:
            raise ValueError(f"Unexpected methods in combined frontier: {universe}")
        if d.duplicated(["type","gamma","wealth"]).any():raise ValueError("Duplicate combined frontier points")
        frames.append(d.assign(universe=universe,market=NAMES[universe]))
    all_points=pd.concat(frames,ignore_index=True)
    if not np.isfinite(all_points[["Vol","R_TC"]]).all().all():raise ValueError("Nonfinite combined frontier")
    all_points.to_csv(OUT/"combined_frontier_summary.csv",index=False)
    fig,axes=plt.subplots(2,1,figsize=(12,12))
    colors={"us_large":"#146c9a","us_mid":"#ae4362","eu_large":"#159472"}
    anchors=[]
    for universe in UNIVERSES:
        cfg=load_config(config_path(universe));d=all_points.loc[all_points.universe.eq(universe)]
        for method,_ in frontier_methods(universe):
            color="#d87920" if method=="Static-ML*" else colors[universe]
            label=f"{NAMES[universe]} {method}"
            for level,style in ((cfg.investor.wealth,"-"),(0.,":")):
                curve=d.loc[d.type.eq(method)&d.wealth.eq(level)].sort_values("Vol")
                if curve.empty:continue
                axes[0].plot(100*curve.Vol,100*curve.R_TC,style,color=color,lw=1.8 if level else 1.1,
                    marker="o" if level else None,ms=3,label=label if level else "_nolegend_")
                if level:
                    anchor=curve.loc[curve.gamma.eq(cfg.investor.gamma)]
                    axes[0].scatter(100*anchor.Vol,100*anchor.R_TC,color=color,marker="s",s=58,zorder=5)
                    anchors.append(curve)
        pf=pd.read_parquet(market_dir(universe)/"pf.parquet")
        st=summary_stats(pf,cfg.investor.gamma)
        axes[0].scatter(100*st.Vol,100*st.R_TC,marker="D",color=colors[universe],s=40,label=f"{NAMES[universe]} Market")
    axes[0].plot([],[],":",color="#666",label="Dotted: zero-impact refit")
    levels=sorted(all_points.wealth.unique());cmap=plt.get_cmap("viridis")
    styles={"us_large":"-","us_mid":"--","eu_large":":"}
    for universe in UNIVERSES:
        for method,_ in frontier_methods(universe):
            for i,level in enumerate(levels):
                d=all_points.loc[all_points.universe.eq(universe)&all_points.type.eq(method)&all_points.wealth.eq(level)].sort_values("Vol")
                if d.empty:continue
                axes[1].plot(100*d.Vol,100*d.R_TC,ls="-." if method=="Static-ML*" else styles[universe],
                    color=cmap(i/max(1,len(levels)-1)),lw=1.5,marker="o",ms=2.5,label="_nolegend_")
    handles=[Line2D([0],[0],color=cmap(i/max(1,len(levels)-1)),label="Zero-impact limit" if level==0 else f"${level/1e9:g}B") for i,level in enumerate(levels)]
    handles += [Line2D([0],[0],color="#333",ls=styles[u],label=NAMES[u]+" Portfolio ML") for u in UNIVERSES]
    handles += [Line2D([0],[0],color="#333",ls="-.",label="US high-cap Static ML*")]
    axes[1].legend(handles=handles,fontsize=8,ncol=3,loc="best")
    axes[0].legend(fontsize=8,ncol=2,loc="best")
    axes[0].set_title("A   Implementable frontiers by portfolio method and market",loc="left",fontsize=13)
    axes[1].set_title("B   All retained frontiers by investor size, method and market",loc="left",fontsize=13)
    for ax in axes:
        ax.set(xlabel="Realised gross volatility (% per year)",ylabel="Excess return net of trading costs (% per year)")
        ax.axhline(0,color="#777",lw=.6);ax.grid(alpha=.18)
    fig.suptitle("Combined base Figure 1 | US high-cap, US mid-cap and EU high-cap",fontsize=15)
    fig.text(.5,.016,"2020–2025; A: solid lines at each market's configured fund size, squares at configured risk aversion.\n"
        "B: every fitted investor-size curve. Lines join separate risk-aversion fits; test curves need not be monotone or efficient.",ha="center",fontsize=9)
    fig.tight_layout(rect=(0,.055,1,.97))
    save_figure(fig,OUT/"charts/figure1_combined")
    (OUT/"COMBINED_FIGURE_METHODS.md").write_text(
        "# Combined base Figure 1\n\nPanel A compares US high-cap Static ML* and Portfolio ML in all three markets at their configured fund size. "
        "Dotted curves use separately fitted zero-impact rules. Diamonds are same-market value-weighted references. "
        "Panel B shows every retained model's fund-size curve, including US high-cap Static ML*; color identifies size and line style identifies method/market. "
        "Each point varies risk aversion, with model parameters chosen from preceding observations. "
        "Static ML* is retuned separately at every risk/fund-size point. US mid-cap and EU high-cap have no Static ML model. "
        "Returns are annual arithmetic excess returns after modelled costs; risk is annualised gross-return SD. "
        "All curves use 72 months in 2020–2025; fund size is anchored at end-2019. "
        "Cross-market curves represent different stock universes. These are realised fitted strategies, not a known optimal efficient envelope. "
        "No iteration fit or result is part of this base figure.\n",encoding="utf-8")
    return True




if __name__=="__main__":raise SystemExit(main())


### `scripts/iteration_results_panels.py`

Create presentation panels from saved iteration 2/4 returns; no model fitting.

**Role:** Pipeline implementation or configuration. This cell writes the file without running its main function.

**Input:** the embedded source below. **Output:** `scripts/iteration_results_panels.py`. Expand the code cell to read or edit it.


In [ ]:
%%writefile scripts/iteration_results_panels.py
"""Create presentation panels from saved iteration 2/4 returns; no model fitting."""
from pathlib import Path
import json

import matplotlib
matplotlib.use('Agg')
import matplotlib.dates as mdates
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

from workflow import ROOT
from analysis.ml_comparison.metrics import additive_path

BASE = ROOT / 'outputs/base/us_large/portfolio_ml/rff'
PROTOCOL = ROOT / 'configs/us_high_cap_iterations.json'


def build(number):
    from iteration_sequence import paired_probability
    folder = ROOT / f'outputs/iteration_{number}/us_large'
    frames = {
        'Base Portfolio ML': pd.read_parquet(BASE / 'pf.parquet'),
        f'Iteration {number}': pd.read_parquet(folder / 'pf.parquet'),
        'Market': pd.read_parquet(ROOT / 'outputs/base/overall analysis/market_references/us_large/pf.parquet'),
    }
    frames = {k: v.sort_values('eom_ret').reset_index(drop=True) for k, v in frames.items()}
    candidate, market = frames[f'Iteration {number}'], frames['Market']
    assert all(v.eom_ret.equals(candidate.eom_ret) and len(v) == 72 for v in frames.values())
    protocol = json.loads(PROTOCOL.read_text())
    probability = paired_probability(candidate, market, protocol['bootstrap_reps'],
                                     protocol['bootstrap_block_months'], protocol['bootstrap_seed'])['probability']
    stats = pd.read_csv(folder / 'performance_summary.csv').set_index('strategy')
    net = candidate.r.to_numpy() - candidate.tc.to_numpy()
    market_net = market.r.to_numpy() - market.tc.to_numpy()
    active = net - market_net
    annual_active = 12 * active.mean()
    te = active.std(ddof=1) * np.sqrt(12)
    ir = annual_active / te
    decision = json.loads((folder / 'decision.json').read_text())
    title = 'Gamma 5' if number == 2 else 'Market + 1% TE-target tilt'
    fig = plt.figure(figsize=(13.5, 7.2), dpi=180, facecolor='white')
    navy = '#1c304d'
    fig.text(.035, .95, '3 | RESULTS', color='#00868b', fontsize=11, weight='bold')
    fig.text(.035, .89, f'Iteration {number} vs the market, 2020 to 2025',
             color=navy, fontsize=23, weight='bold')
    inherited = ', '.join(map(str, decision['included_after']))
    fig.text(.035, .845, f'US high-cap | {title} | Included iterations: {inherited}', fontsize=11, color='#535e6b')
    axes = [fig.add_axes([.07 + j * .31, .445, .275, .32]) for j in range(3)]
    paths = {}
    for label, frame in frames.items():
        values = frame.r.to_numpy() - frame.tc.to_numpy()
        path, drawdown = additive_path(values)
        paths[label] = path
        color = {'Base Portfolio ML': '#9cbed3', f'Iteration {number}': '#dd842b', 'Market': '#172333'}[label]
        for ax, series in zip(axes[:2], [100 * path, 100 * drawdown]):
            ax.plot(frame.eom_ret, series, color=color, lw=1.6 if label != 'Base Portfolio ML' else 1.1, label=label)
    relative = 100 * (paths[f'Iteration {number}'] - paths['Market'])
    axes[2].plot(candidate.eom_ret, relative, color='#179d53', lw=1.6)
    axes[2].fill_between(candidate.eom_ret, relative, 0, color='#179d53', alpha=.14)
    titles = ['Cumulative net return (additive)', 'Additive drawdown from peak', 'Cumulative return minus Market']
    for ax, name in zip(axes, titles):
        ax.set_title(name, fontsize=10, color=navy)
        ax.set_ylabel('Percentage points', fontsize=9)
        ax.xaxis.set_major_locator(mdates.YearLocator())
        ax.xaxis.set_major_formatter(mdates.DateFormatter('%Y'))
        ax.tick_params(labelsize=8)
        ax.grid(alpha=.15)
        ax.axhline(0, color='#777777', lw=.6)
    axes[0].legend(fontsize=7, frameon=False, loc='upper left')
    table_rows = [
        ['Net excess return / year', f"{stats.loc['Candidate', 'R_TC']:.2%}", f"{stats.loc['Market', 'R_TC']:.2%}", 'Active return / year', f'{annual_active:.3%}'],
        ['Net Sharpe', f"{stats.loc['Candidate', 'SR_net']:.3f}", f"{stats.loc['Market', 'SR_net']:.3f}", 'Information ratio', f'{ir:.3f}'],
        ['Worst additive DD (pp)', f"{100*stats.loc['Candidate', 'max_drawdown_excess']:.2f}", f"{100*stats.loc['Market', 'max_drawdown_excess']:.2f}", 'P(better Sharpe vs market)', f'{probability:.1%}'],
    ]
    ax = fig.add_axes([.035, .205, .93, .175]); ax.axis('off')
    table = ax.table(cellText=table_rows, colLabels=['Measure', f'Iteration {number}', 'Market', 'Relative to market', 'Value'],
                     colWidths=[.25, .15, .15, .30, .15], cellLoc='left', colLoc='left', bbox=[0, 0, 1, 1])
    table.auto_set_font_size(False); table.set_fontsize(10)
    for (r, c), cell in table.get_celld().items():
        cell.set_linewidth(.6); cell.set_edgecolor('#68717d'); cell.PAD=.025
        cell.set_facecolor(navy if r == 0 else ('#eef2f7' if r % 2 else 'white'))
        cell.set_text_props(color='white' if r == 0 else '#253140', weight='bold' if r == 0 else 'normal')
    conclusion = ('Gamma 5 improves on Base, but substantially underperforms the market.' if number == 2 else
                  'The tilt closely tracks the market; evidence of a higher Sharpe versus market is below 80%.')
    fig.text(.035, .15, conclusion, fontsize=11, color=navy, weight='bold')
    fig.text(.035, .105, 'Net returns are annualised arithmetic excess returns after costs. Curves and drawdowns add monthly returns (percentage points).', fontsize=9, color='#555f6b')
    fig.text(.035, .075, 'P is paired block-bootstrap support, not a conventional p-value. It compares against Market here, not the prior iteration.', fontsize=9, color='#555f6b')
    final_note = 'Test-period selection results; no independent unseen confirmation. Additive drawdown is not a percentage loss of capital.'
    fig.text(.035, .045, final_note, fontsize=9, color='#555f6b')
    output = folder / 'charts'; output.mkdir(exist_ok=True)
    fig.savefig(output / 'market_comparison_panel.png', dpi=180, facecolor='white')
    plt.close(fig)
    pd.DataFrame(table_rows, columns=['Measure', f'Iteration {number}', 'Market', 'Relative to market', 'Value']).to_csv(output / 'market_comparison_panel.csv', index=False)
    print(output / 'market_comparison_panel.png')


def build_summary():
    """Refresh the compact screenshot from the current five-iteration CSV."""
    folder = ROOT / 'outputs/iteration_summary'
    data = pd.read_csv(folder / 'statistical_tests_all_iterations.csv')
    names = {1:'Refit yearly', 2:'Gamma 5', 3:'Exit 120, costs x2',
             4:'Market + 1% TE tilt', 5:'Recent weighting'}
    rows = []
    for r in data.itertuples():
        verdict = 'Retained as design' if r.iteration == 1 else ('Kept' if r.included else 'Not included')
        rows.append([f'{r.iteration}  {names[r.iteration]}', f'{r.Sharpe:.3f}',
                     'N/A' if pd.isna(r.P_better) else f'{r.P_better:.1%}',
                     f'{100*r.worst_drawdown:.1f}', verdict])
    fig = plt.figure(figsize=(12,5.1), dpi=180, facecolor='white')
    fig.text(.025,.925,'Statistical tests across all five iterations',fontsize=19,weight='bold',color='#242750')
    fig.text(.025,.865,'Kati_tryout | US high-cap | Test period: 2020-2025 | Additive returns',fontsize=11,color='#555b68')
    ax=fig.add_axes([.025,.27,.95,.52]); ax.axis('off')
    t=ax.table(cellText=rows,colLabels=['Iteration','Net Sharpe','P(better)','Additive DD (pp)','Verdict'],
               colWidths=[.30,.12,.13,.18,.27],cellLoc='left',colLoc='left',bbox=[0,0,1,1])
    t.auto_set_font_size(False); t.set_fontsize(11)
    for (r,c),cell in t.get_celld().items():
        cell.set_edgecolor('#626777');cell.set_linewidth(.65);cell.PAD=.025
        if r==0:
            cell.set_facecolor('#242750');cell.set_text_props(color='white',weight='bold')
        else:
            cell.set_facecolor('#f0f3f7' if r%2 else 'white');cell.get_text().set_color('#252a35')
            if c==4:
                cell.set_text_props(weight='bold',color='#242750' if r==1 else ('#008b85' if rows[r-1][4]=='Kept' else '#bd351e'))
    kept = data.loc[data.included, 'iteration'].astype(str).tolist()
    fig.text(.025,.195,'Accepted sequence: '+', '.join(kept)+'. Annual refitting is already in Base.',fontsize=11,weight='bold',color='#242750')
    fig.text(.025,.137,'P(better) compares net Sharpe with the previous accepted version using a paired block bootstrap.',fontsize=10,color='#454b56')
    fig.text(.025,.091,'DD: drop from the additive return peak, in percentage points (pp). Drawdown is not an acceptance gate.',fontsize=10,color='#454b56')
    fig.text(.025,.045,'Test-period selection results; these are not independent unseen-data confirmation.',fontsize=9,color='#666b75')
    fig.savefig(folder/'statistical_tests_screenshot.png', dpi=180, facecolor='white');plt.close(fig)


if __name__ == '__main__':
    for iteration in (2, 4):
        build(iteration)
    build_summary()


### `scripts/iteration_sequence.py`

Five sequential US high-cap experiments, with test-only P(better) and explicit inheritance. Existing base models, input dates and characteristics are read-only. The retained strategy changes only when a candidate's paired bootstrap Sharpe comparison reaches the declared threshold. Rejected candidates remain fully reported.

**Role:** Pipeline implementation or configuration. This cell writes the file without running its main function.

**Input:** the embedded source below. **Output:** `scripts/iteration_sequence.py`. Expand the code cell to read or edit it.


In [ ]:
%%writefile scripts/iteration_sequence.py
"""Five sequential US high-cap experiments, with test-only P(better) and explicit inheritance.

Existing base models, input dates and characteristics are read-only. The retained
strategy changes only when a candidate's paired bootstrap Sharpe comparison reaches
the declared threshold. Rejected candidates remain fully reported.
"""
from __future__ import annotations

import argparse
import dataclasses
import hashlib
import html
import json
import logging
from pathlib import Path
import shutil

import numpy as np
import pandas as pd
from scipy.stats import norm

from workflow import ROOT, config_path
from jkmp.config import load_config
from jkmp.covariance import load_covariance
from jkmp.data.prepare import risk_free_monthly
from jkmp.portfolio.grids import grid_inputs, _signal_kwargs
from jkmp.portfolio.portfolio_ml import build_yearly_moments, search_yearly, run_portfolio_ml
from jkmp.portfolio.active import calibrate
from jkmp.evaluation.metrics import active_weights, ex_ante_exposures
from analysis.ml_comparison.inference import _bootstrap_differences
from analysis.ml_comparison.metrics import additive_path, statistics

log=logging.getLogger(__name__)
PROTOCOL=ROOT/'configs/us_high_cap_iterations.json'
OUT=ROOT/'outputs/iteration_summary'
BASE=ROOT/'outputs/base/us_large/portfolio_ml/rff'
MONTH=pd.offsets.MonthEnd(1)
STYLE='<style>body{font:16px/1.5 system-ui;max-width:1250px;margin:35px auto;color:#203044;padding:0 20px}table{border-collapse:collapse;width:100%;font-size:13px}td,th{border:1px solid #ccd5de;padding:9px;text-align:right}td:first-child,th:first-child{text-align:left}th{background:#203451;color:white}img{max-width:100%}.note{background:#edf3f8;padding:15px}a{color:#126084}</style>'


def write_json(path,value):
    path=Path(path)
    if not path.resolve().is_relative_to(ROOT.resolve()):raise ValueError('Output outside Kati')
    path.parent.mkdir(parents=True,exist_ok=True)
    temporary=path.with_suffix('.writing.json')
    temporary.write_text(json.dumps(value,indent=2,default=str,allow_nan=False)+'\n',encoding='utf-8')
    temporary.replace(path)


def digest(path):
    with Path(path).open('rb') as stream:return hashlib.file_digest(stream,'sha256').hexdigest()


def default_state(cfg):
    return dict(gamma=cfg.investor.gamma,exit_rank=cfg.universe.exit_rank,cost_multiple=1.0,
                benchmark_relative=False,te_target=None,half_lives=[None])


def candidate_state(previous,overrides):
    return {**previous,**overrides}


def inheritance(previous,number,accepted):
    return [*previous,number] if accepted and number not in previous else list(previous)


def paired_probability(candidate,reference,reps=10000,block=6,seed=20261102):
    """Paired bootstrap support, not a posterior or a conventional p-value."""
    a=candidate.sort_values('eom_ret').set_index('eom_ret')
    b=reference.sort_values('eom_ret').set_index('eom_ret')
    if not a.index.equals(b.index) or len(a)<2:raise ValueError('Paired calendars differ')
    if np.array_equal(a[['r','tc']],b[['r','tc']]):
        return dict(probability=None,ci_low=0.,ci_high=0.,identical=True,valid_reps=reps)
    _,delta=_bootstrap_differences(a.r.to_numpy(),a.tc.to_numpy(),b.r.to_numpy(),b.tc.to_numpy(),
                                  10.,reps,block,np.random.default_rng(seed))
    delta=delta[np.isfinite(delta)]
    if len(delta)!=reps:raise ValueError('Undefined bootstrap Sharpe replicates')
    # Half weight to numerical ties; equality cannot manufacture evidence.
    tied=np.isclose(delta,0.,rtol=0.,atol=1e-12)
    p=float(np.mean((delta>1e-12).astype(float)+.5*tied))
    lo,hi=np.quantile(delta,[.025,.975])
    return dict(probability=p,ci_low=float(lo),ci_high=float(hi),identical=False,valid_reps=len(delta))


def buffered_membership(screened,exit_rank,target_n=80,entry_rank=70):
    """Keats source timing: December formation selection, held December--November.

    Rankings use contemporaneous market cap and original screened row order.
    No forward target is consulted. Missing calendar rows terminate a membership.
    """
    dec=screened[screened.eom.dt.month.eq(12)].copy()
    dec['mcap_rank']=dec.groupby('eom').me.rank(ascending=False,method='first')
    previous=set();decisions=[]
    for eom,g in dec.groupby('eom',sort=True):
        ranks=g.set_index('id').mcap_rank
        retained={i for i in previous if i in ranks.index and ranks[i]<=exit_rank}
        entrants=ranks[ranks.le(entry_rank)].drop(list(retained),errors='ignore')
        chosen=retained|set(entrants.index)
        fill=ranks.drop(list(chosen),errors='ignore').sort_values(kind='stable')
        chosen |= set(fill.index[:max(0,target_n-len(chosen))])
        decisions.append(g[g.id.isin(chosen)][['id','mcap_rank']].assign(form_year=eom.year))
        previous=chosen
    membership=pd.concat(decisions,ignore_index=True)
    rows=screened[['id','eom']].copy()
    rows['form_year']=np.where(rows.eom.dt.month.eq(12),rows.eom.dt.year,rows.eom.dt.year-1)
    rows=rows.merge(membership,on=['id','form_year'],validate='many_to_one').sort_values(['id','form_year','eom'])
    prior=rows.groupby(['id','form_year']).eom.shift()
    gap=prior.notna()&(prior+MONTH!=rows.eom)
    return rows[~gap.groupby([rows.id,rows.form_year]).cummax()]


class ExperimentEngine:
    def __init__(self):
        self.cfg=load_config(config_path('us_large'))
        p=self.cfg.paths.processed
        # Portfolio fitting uses only stocks ever held under a declared rule.
        # Keep that exact union at read time, not million-row copies per trial.
        members=pd.read_parquet(p/'panel.parquet',columns=['id'],filters=[('in_universe','=',True)])
        ids=set(members.id.unique())
        source=pd.read_parquet(ROOT/'data/source/screened_US.parquet',columns=['id','eom','me'])
        declared=json.loads(PROTOCOL.read_text())['iterations']
        self._memberships={}
        for rank in {s.get('override',{}).get('exit_rank') for s in declared}-{None}:
            membership=buffered_membership(source,rank,self.cfg.universe.target_n,self.cfg.universe.entry_rank)
            self._memberships[rank]=membership
            ids.update(membership.id.unique())
        ids=sorted(ids)
        columns=list(dict.fromkeys(['id','eom','me','in_universe','lambda','g','tr_ld1','ret_ld1',*self.cfg.features.names]))
        self.panel=pd.read_parquet(p/'panel.parquet',columns=columns,filters=[('id','in',ids)])
        preds=pd.read_parquet(p/'predictions.parquet',columns=['id','eom','pred_ld1'],filters=[('id','in',ids)])
        self.panel=self.panel.merge(preds,on=['id','eom'],how='left',validate='one_to_one')
        self.wealth=pd.read_parquet(p/'wealth.parquet')
        self.cov=load_covariance(p/'covariance')
        self.rf=risk_free_monthly(pd.read_parquet(self.cfg.paths.raw/'ff_factors_monthly.parquet'))
        self._panels={self.cfg.universe.exit_rank:self.panel}
        self._expanded_cov=None
        paths=[p/'panel.parquet',p/'wealth.parquet',p/'covariance/cov_loadings.parquet',p/'covariance/cov_factor.parquet',BASE/'pf.parquet',PROTOCOL,Path(__file__)]
        self.signature=hashlib.sha256(json.dumps({str(x):digest(x) for x in paths},sort_keys=True).encode()).hexdigest()

    def panel_for(self,state,folder):
        rank=state['exit_rank']
        if rank not in self._panels:
            from jkmp.data.lag_history import exclude_unavailable
            from workflow import zero_missing_held_returns
            if rank not in self._memberships:raise ValueError('Exit-rank change is not in the declared protocol')
            membership=self._memberships[rank]
            keys=pd.MultiIndex.from_frame(membership[['id','eom']])
            result=self.panel.copy()
            result['in_universe']=pd.MultiIndex.from_frame(result[['id','eom']]).isin(keys)
            result,excluded=exclude_unavailable(result,'us_large',self.cfg.universe.lookback_months)
            result=zero_missing_held_returns(result,self.rf)
            if not result.loc[result.in_universe,['g','lambda','ret_ld1','me']].notna().all().all():
                raise ValueError('New buffered members lack required observed current inputs')
            folder.mkdir(parents=True,exist_ok=True)
            membership.to_csv(folder/'membership.csv',index=False)
            excluded.to_csv(folder/'history_exclusions.csv',index=False)
            self._panels[rank]=result
        return self._panels[rank].assign(**{'lambda':self._panels[rank]['lambda']*state['cost_multiple']})

    def covariance_for(self,panel):
        missing=any(not set(g.id).issubset(set(self.cov[d].ids)) for d,g in
                    panel[panel.in_universe&panel.eom.isin(self.cov)].groupby('eom'))
        if not missing:return self.cov
        if self._expanded_cov is None:
            path=ROOT/'data/processed/us_large/universe_250/covariance'
            if not path.exists():raise FileNotFoundError('Extended buffered members require the existing wider covariance cache')
            extended=load_covariance(path)
            for d,c in self.cov.items():
                common=np.intersect1d(c.ids,extended[d].ids)
                np.testing.assert_allclose(c.full(common),extended[d].full(common),rtol=1e-10,atol=1e-12)
            self._expanded_cov=extended
        return self._expanded_cov

    def fit(self,state,folder,*,calibration=False,gamma_te=None):
        cfg=self.cfg
        panel=self.panel_for(state,folder)
        cov=self.covariance_for(panel)
        pm=dataclasses.replace(cfg.portfolio_ml,benchmark_relative=state['benchmark_relative'],
             add_constant=not state['benchmark_relative'],half_life_years=tuple(state['half_lives']))
        cfg=dataclasses.replace(cfg,portfolio_ml=pm)
        inp=grid_inputs(cfg,panel,self.wealth,cov,self.rf)
        gamma=float(gamma_te if state['benchmark_relative'] else state['gamma'])
        if calibration:
            end=pd.Timestamp(cfg.dates.validation_end)
            dates=inp.pfml_dates[inp.pfml_dates+MONTH<=end]
            years=[y for y in inp.hp_years if y<=end.year]
            aim=dates[(dates+MONTH).year>years[0]]
        else:
            dates=inp.pfml_dates;years=inp.hp_years
            aim=dates[(dates+MONTH).year>years[0]]
        evaluation=aim if calibration else inp.eval_dates
        results={}
        for g in pm.rff_variance_grid:
            tag=f'rff_g{g:.4f}'
            log.info('Fit %s: gamma=%g %s; %d months',folder.name,gamma,tag,len(dates))
            ym=build_yearly_moments(inp.panel,self.wealth,cov,self.rf,dates,cfg.features.names,
                signal_dates=aim,gamma=gamma,mu=cfg.investor.mu,lb=cfg.universe.lookback_months,
                iterations=cfg.investor.m_iterations,scale_by_vol=pm.scale_by_vol,
                add_constant=pm.add_constant,**_signal_kwargs(cfg,g))
            results[tag]=search_yearly(ym,tag,pm.p_grid,cfg.ridge_grid.values,years,pm.half_life_years)
            del ym
        res=run_portfolio_ml(results,inp.panel,self.wealth,cov,self.rf,aim,evaluation,
            gamma=gamma,mu=cfg.investor.mu,iterations=cfg.investor.m_iterations,
            charge_exit_tc=cfg.portfolio.charge_exit_tc,benchmark_relative=state['benchmark_relative'])
        if calibration:
            ex=ex_ante_exposures(active_weights(res['weights'],inp.panel),cov)
            return dict(gamma_te=gamma,te_ex_ante=float(ex.te.mean()*np.sqrt(12)),
                        months=len(evaluation),last_return=str((evaluation[-1]+MONTH).date()))
        for name,frame in res.items():
            if name=='chosen':frame.to_csv(folder/'chosen.csv',index=False)
            else:frame.to_parquet(folder/f'{name}.parquet',index=False)
        return res

    def run_candidate(self,state,folder):
        folder.mkdir(parents=True,exist_ok=True)
        key=hashlib.sha256(json.dumps(dict(state=state,source=self.signature),sort_keys=True).encode()).hexdigest()
        marker=folder/'fit_manifest.json'
        files=['pf.parquet','weights.parquet','chosen.csv']
        if marker.exists():
            old=json.loads(marker.read_text())
            if old.get('signature')==key and all((folder/f).exists() and digest(folder/f)==old.get('hashes',{}).get(f) for f in files):
                log.info('Reuse verified candidate %s',folder)
                return pd.read_parquet(folder/'pf.parquet')
        gamma_te=None
        if state['benchmark_relative']:
            points=[]
            # Use a bracket, not an OOS-selected TE risk penalty.
            for g in (100.,250.,50.,500.,25.,1000.,10.,2500.):
                point=self.fit(state,folder,calibration=True,gamma_te=g)
                points.append(point)
                table=pd.DataFrame(points)
                table.to_csv(folder/'calibration.csv',index=False)
                if len(points)>1 and table.te_ex_ante.min()<=state['te_target']<=table.te_ex_ante.max():break
            chosen=calibrate(pd.DataFrame(points),[state['te_target']])
            chosen.to_csv(folder/'calibrated.csv',index=False)
            gamma_te=float(chosen.gamma_te.iloc[0])
        self.fit(state,folder,gamma_te=gamma_te)
        write_json(marker,dict(signature=key,state=state,gamma_te=gamma_te,
                              hashes={f:digest(folder/f) for f in files}))
        return pd.read_parquet(folder/'pf.parquet')


def verify_accounting(folder,panel,wealth):
    """Reconstruct each stock's return/cost directly, independently of the model optimiser."""
    w=pd.read_parquet(folder/'weights.parquet')
    pf=pd.read_parquet(folder/'pf.parquet').sort_values('eom_ret').set_index('eom_ret')
    d=w.merge(panel[['id','eom','ret_ld1','lambda']],on=['id','eom'],how='left',validate='many_to_one').merge(wealth[['eom','wealth']],on='eom',validate='many_to_one')
    if d['lambda'].isna().any() or d.loc[~d.exit,'ret_ld1'].isna().any():raise ValueError('Incomplete accounting inputs')
    dw=d.w-d.w_start
    d=d.assign(r=d.w*d.ret_ld1.where(~d.exit,0),tc=.5*d.wealth*d['lambda']*dw**2,
               inv=d.w.abs(),shorting=d.w.clip(upper=0).abs(),turnover=dw.abs())
    recomputed=d.groupby('eom')[['r','tc','inv','shorting','turnover']].sum()
    recomputed.index+=MONTH
    np.testing.assert_allclose(recomputed.loc[pf.index],pf[recomputed.columns],rtol=1e-10,atol=1e-12)
    if not pf.index.equals(pd.date_range('2020-01-31','2025-12-31',freq='ME')):raise ValueError('Expected all 72 existing test months')
    write_json(folder/'accounting_verification.json',dict(status='PASS',months=len(pf),metrics=list(recomputed.columns)))


def plot_comparison(frames,out,title):
    import matplotlib
    matplotlib.use('Agg')
    import matplotlib.pyplot as plt
    out.mkdir(parents=True,exist_ok=True)
    fig,axes=plt.subplots(1,3,figsize=(15,4.1))
    market=frames['Market'].sort_values('eom_ret')
    mb,_=additive_path(market.r-market.tc)
    for label,pf in frames.items():
        d=pf.sort_values('eom_ret');net=d.r-d.tc;path,dd=additive_path(net)
        if not d.eom_ret.reset_index(drop=True).equals(market.eom_ret.reset_index(drop=True)):
            raise ValueError('Additive market comparisons require identical monthly calendars')
        axes[0].plot(d.eom_ret,100*path,label=label)
        axes[1].plot(d.eom_ret,100*dd,label=label)
        if label!='Market':axes[2].plot(d.eom_ret,100*(path-mb),label=label)
    for ax in axes:ax.grid(alpha=.2);ax.tick_params(axis='x',rotation=25);ax.set_ylabel('Percentage points')
    axes[0].set_title('Cumulative net excess return (additive)')
    axes[1].set_title('Additive drawdown from previous peak')
    axes[2].set_title('Cumulative net return minus Market');axes[2].axhline(0,color='grey',lw=.8)
    axes[0].legend(fontsize=7);fig.suptitle(title);fig.tight_layout()
    for suffix in ('png','pdf'):fig.savefig(out/f'net_returns_and_costs.{suffix}',dpi=160)
    plt.close(fig)
    fig,axes=plt.subplots(1,2,figsize=(12,3.5))
    for label,pf in frames.items():
        d=pf.sort_values('eom_ret');axes[0].plot(d.eom_ret,100*d.tc.cumsum(),label=label)
        axes[1].plot(d.eom_ret,d.turnover,label=label)
    axes[0].set_title('Cumulative modelled costs (%)');axes[1].set_title('Monthly turnover')
    for ax in axes:ax.grid(alpha=.2)
    axes[0].legend(fontsize=7);fig.tight_layout()
    for suffix in ('png','pdf'):fig.savefig(out/f'costs_and_turnover.{suffix}',dpi=160)
    plt.close(fig)


def write_iteration_report(number,decision,candidate,reference,base,static,market):
    folder=ROOT/f'outputs/iteration_{number}/us_large'
    frames={'Candidate':candidate,'Previous accepted':reference,'Base Portfolio ML':base,'Static ML*':static,'Market':market}
    table=pd.DataFrame([dict(strategy=name,**statistics(pf)) for name,pf in frames.items()])
    table.to_csv(folder/'performance_summary.csv',index=False)
    pd.concat([p.assign(strategy=n) for n,p in frames.items()],ignore_index=True).to_csv(folder/'monthly_returns.csv',index=False)
    pd.DataFrame([dict(strategy=n,year=y,**statistics(g)) for n,p in frames.items() for y,g in p.groupby(p.eom_ret.dt.year)]).to_csv(folder/'annual_performance.csv',index=False)
    plot_comparison(frames,folder/'charts',f'Iteration {number}: {decision["name"]}')
    inherited=decision['inherited_iterations']
    narrative=f'Iteration 1 (annual refitting) is already part of iteration {number} and Base.'
    narrative+=f' Other inherited accepted changes: {[i for i in inherited if i!=1] or "none"}.'
    narrative+=f' Compared against: {decision["reference"]}. Verdict: {decision["verdict"]}.'
    narrative+=f' Included in subsequent accepted setup: {decision["accepted"]}.'
    if decision['candidate_state']['benchmark_relative']:
        narrative+=' The tilt uses calibrated gamma_TE; an inherited gamma-5 decision is not an extra multiplier of active risk.'
    text=f'# Iteration {number}: {decision["name"]}\n\n{narrative}\n\nP(better net Sharpe): {decision["p_better"] if decision["p_better"] is not None else "N/A (already in Base or identical)"}. Acceptance threshold: 80%.\n\n'+METHODS
    (folder/'SUMMARY.md').write_text(text,encoding='utf-8')
    (folder/'README.md').write_text(text,encoding='utf-8')
    body=f'<h1>Iteration {number}: {html.escape(decision["name"])}</h1><p class="note">{html.escape(narrative)}</p><p>P(better): {decision["p_better"] if decision["p_better"] is not None else "N/A"}</p>'
    body+=table.to_html(index=False,float_format=lambda v:f'{v:.5g}')+'<img src="charts/net_returns_and_costs.png"><img src="charts/costs_and_turnover.png">'
    body+='<p><a href="SUMMARY.md">Summary and methods</a> | <a href="decision.json">Decision/inheritance</a> | <a href="performance_summary.csv">Performance CSV</a> | <a href="monthly_returns.csv">Monthly returns</a> | <a href="../../iteration_summary/index.html">All five experiments</a></p>'
    (folder/'index.html').write_text('<!doctype html><meta charset="utf-8">'+STYLE+body,encoding='utf-8')


METHODS='''All comparisons use the existing 72 test return months, January 2020--December 2025. Kati OOS labels overlap this test history; no separate unseen confirmation is claimed. Annual expanding coefficient refits and monthly portfolio updates are preserved. P(better) is the fraction of 10,000 paired circular-block bootstrap samples (six-month blocks, seed 20261102) with a positive net-Sharpe difference, with half weight to numerical ties. It is not a frequentist p-value or a Bayesian probability of future outperformance. Net Sharpe uses gross-return volatility, following the paper convention. Acceptance is P(better) >= 80% against the last accepted version, not against Market. Identical strategies are not accepted as an improvement. Iteration 1 is retained as an existing design, without a statistical claim. Drawdown is descriptive and adds no rejection gate. Decisions use already inspected test outcomes; subsequent reported performance on those same months is selected-sample evidence, not independent validation. The experiment log retains rejected trials. Nested hyperparameter searches and earlier research choices are not fully corrected by these five comparisons. Cumulative returns add monthly net excess returns, matching the paper replication code (cumsum(r-tc)). Additive drawdown is the cumulative sum minus its running maximum including initial zero; multiply by 100 for percentage points. Relative performance is the cumulative difference against Market. Additive drawdown is a reporting extension of the paper chart convention, not a verified drawdown definition in the paper or a total-wealth loss. TE 1% is an ex-ante pretest calibration target, not a hard realised cap. Base markets, stock screens, selected 50, dates, costs and risk equations are retained except for each explicitly tested override. Exit 120 changes buffering within the same US large-cap candidate population and may change the stock count; costs x2 are applied both when fitting and when trading. The original wider covariance cache may supply extra buffered stocks, only after equality checks against common base covariance entries. No 250-stock breadth experiment is part of this five-step sequence.'''


def report_saved(upto=5):
    protocol=json.loads(PROTOCOL.read_text())
    decisions=[]
    base=pd.read_parquet(BASE/'pf.parquet')
    market=pd.read_parquet(ROOT/'outputs/base/overall analysis/market_references/us_large/pf.parquet')
    for spec in protocol['iterations'][:upto]:
        folder=ROOT/f'outputs/iteration_{spec["number"]}/us_large'
        if (folder/'decision.json').exists():decisions.append(json.loads((folder/'decision.json').read_text()))
    if not decisions:return 0
    OUT.mkdir(parents=True,exist_ok=True)
    rows=[]
    static=pd.read_parquet(ROOT/'outputs/base/us_large/static_ml_star/pf.parquet')
    for d in decisions:
        pf=pd.read_parquet(ROOT/f'outputs/iteration_{d["number"]}/us_large/pf.parquet')
        previous_number=next((i for i in reversed(d['inherited_iterations']) if i!=1),1)
        previous_folder=BASE if previous_number==1 else ROOT/f'outputs/iteration_{previous_number}/us_large'
        reference=pd.read_parquet(previous_folder/'pf.parquet')
        write_iteration_report(d['number'],d,pf,reference,base,static,market)
        s=statistics(pf)
        rows.append(dict(iteration=d['number'],change=d['name'],Sharpe=s['SR_net'],P_better=d['p_better'],
            worst_drawdown=s['max_drawdown_excess'],verdict=d['verdict'],included=d['accepted'],
            reference=d['reference'],inherited_iterations=','.join(map(str,d['inherited_iterations'])),
            final_included_iterations=','.join(map(str,d['included_after'])),annual_net_return=s['R_TC'],
            path_convention='additive',drawdown_unit='decimal return difference (x100 = percentage points)'))
    table=pd.DataFrame(rows);table.to_csv(OUT/'statistical_tests_all_iterations.csv',index=False)
    import matplotlib
    matplotlib.use('Agg')
    import matplotlib.pyplot as plt
    chart_table=table[['iteration','change','Sharpe','P_better','worst_drawdown','verdict']].copy()
    chart_table['Sharpe']=chart_table.Sharpe.map(lambda x:f'{x:.3f}')
    chart_table['P_better']=chart_table.P_better.map(lambda x:'N/A' if pd.isna(x) else f'{x:.1%}')
    chart_table['worst_drawdown']=chart_table.worst_drawdown.map(lambda x:f'{100*x:.1f}')
    chart_table['verdict']=chart_table.verdict.map(lambda x:'Already in Base; retained as design' if 'Annual refitting' in x else 'Kept' if x=='Kept' else 'Not included')
    fig,ax=plt.subplots(figsize=(15,3.8));ax.axis('off')
    chart=ax.table(cellText=chart_table.values,colLabels=['Step','Change','Net Sharpe','P(better)','Additive DD (pp)','Verdict'],
                   colWidths=[.04,.24,.09,.09,.12,.35],loc='center',cellLoc='left')
    chart.auto_set_font_size(False);chart.set_fontsize(10);chart.scale(1,2)
    for (row,col),cell in chart.get_celld().items():
        cell.set_edgecolor('#ccd4dd')
        if row==0:cell.set_facecolor('#203451');cell.set_text_props(color='white',weight='bold')
        else:cell.set_facecolor('#f0f4f8' if row%2 else 'white')
    fig.suptitle('Statistical tests across all five iterations | US high-cap, 2020–2025',fontsize=14)
    fig.tight_layout()
    for suffix in ('png','pdf'):fig.savefig(OUT/f'statistical_tests_all_iterations.{suffix}',dpi=180,bbox_inches='tight')
    plt.close(fig)
    final_number=decisions[-1]['last_accepted_after']
    final_path=BASE if final_number==1 else ROOT/f'outputs/iteration_{final_number}/us_large'
    final=pd.read_parquet(final_path/'pf.parquet')
    final_folder=OUT/'final';final_folder.mkdir(exist_ok=True)
    for file in ('pf.parquet','weights.parquet'):shutil.copy2(final_path/file,final_folder/file)
    final_cmp=paired_probability(final,market,protocol['bootstrap_reps'],protocol['bootstrap_block_months'],protocol['bootstrap_seed'])
    net=final.r.to_numpy()-final.tc.to_numpy();mn=market.r.to_numpy()-market.tc.to_numpy();active=net-mn
    te=float(active.std(ddof=1)*np.sqrt(12))
    final_stats=pd.DataFrame([dict(strategy=n,**statistics(p)) for n,p in [('Final Portfolio ML',final),('Market',market)]])
    final_stats.to_csv(final_folder/'performance_summary.csv',index=False)
    write_json(final_folder/'summary.json',dict(last_accepted_iteration=final_number,included_iterations=decisions[-1]['included_after'],
        p_beats_market=final_cmp['probability'],active_return=float(12*active.mean()),tracking_error=te,
        information_ratio=float(12*active.mean()/te) if te>0 else None,independent_unseen_confirmation=False))
    plot_comparison({'Final Portfolio ML':final,'Market':market},final_folder/'charts','Final accepted strategy vs Market, 2020--2025')
    (final_folder/'SUMMARY.md').write_text('# Final accepted US high-cap Portfolio ML\n\nIncluded iterations: '+str(decisions[-1]['included_after'])+'\n\n'+METHODS,encoding='utf-8')
    write_json(OUT/'run_status.json',dict(completed_iterations=len(decisions),total_iterations=5,
        sequence_complete=len(decisions)==5,final_is_provisional=len(decisions)<5))
    display=table[['iteration','change','Sharpe','P_better','worst_drawdown','verdict']].copy()
    display['P_better']=display.P_better.map(lambda x:'N/A' if pd.isna(x) else f'{x:.1%}')
    display['worst_drawdown']=display.worst_drawdown.map(lambda x:f'{100*x:.1f} pp')
    links=''.join(f'<li><a href="../iteration_{d["number"]}/us_large/index.html">Iteration {d["number"]}: {html.escape(d["name"])}</a> — {html.escape(d["verdict"])}</li>' for d in decisions)
    body='<h1>Statistical tests across all five US high-cap iterations</h1>'+display.to_html(index=False,float_format=lambda x:f'{x:.3f}')
    body+='<ul>'+links+'</ul><h2>Final accepted strategy vs Market</h2>'+final_stats.to_html(index=False,float_format=lambda x:f'{x:.5g}')
    body+='<img src="final/charts/net_returns_and_costs.png"><p><a href="statistical_tests_all_iterations.csv">Experiment table CSV</a> | <a href="final/summary.json">Final active metrics and P versus Market</a> | <a href="METHODS.md">Definitions and limitations</a></p>'
    body+='<p class="note">'+html.escape(METHODS)+'</p>'
    (OUT/'index.html').write_text('<!doctype html><meta charset="utf-8">'+STYLE+body,encoding='utf-8')
    (OUT/'METHODS.md').write_text(METHODS,encoding='utf-8')
    # Replace the former four-version landing page with the current five-step report.
    alias=ROOT/'outputs/base/overall analysis/us_high_cap_progression';alias.mkdir(parents=True,exist_ok=True)
    (alias/'index.html').write_text('<!doctype html><meta charset="utf-8"><meta http-equiv="refresh" content="0; url=../../../iteration_summary/index.html"><a href="../../../iteration_summary/index.html">Current five-experiment decisions and final strategy</a>',encoding='utf-8')
    progression=[dict(strategy='Base Portfolio ML',**statistics(base))]
    for d in decisions:
        progression.append(dict(strategy=f'Iteration {d["number"]}: {d["name"]}',**statistics(pd.read_parquet(ROOT/f'outputs/iteration_{d["number"]}/us_large/pf.parquet'))))
    pd.DataFrame(progression).to_csv(alias/'performance_summary.csv',index=False)
    from iteration_results_panels import build,build_summary
    for number in (2,4):
        if any(d['number']==number for d in decisions):build(number)
    build_summary()
    print(display.to_string(index=False),flush=True)
    return 0


def main(argv=None):
    parser=argparse.ArgumentParser(description=__doc__)
    parser.add_argument('--upto',type=int,choices=range(1,6),default=5)
    parser.add_argument('--reports-only',action='store_true')
    args=parser.parse_args(argv)
    if args.reports_only:return report_saved()
    logging.basicConfig(level=logging.INFO,format='%(asctime)s %(message)s')
    protocol=json.loads(PROTOCOL.read_text());engine=ExperimentEngine()
    base=pd.read_parquet(BASE/'pf.parquet')
    static=pd.read_parquet(ROOT/'outputs/base/us_large/static_ml_star/pf.parquet')
    market=pd.read_parquet(ROOT/'outputs/base/overall analysis/market_references/us_large/pf.parquet')
    accepted_state=default_state(engine.cfg);accepted=[1];previous=base;previous_number=1
    base_hashes={str(p):digest(p) for p in [BASE/'pf.parquet',ROOT/'outputs/base/us_large/static_ml_star/pf.parquet',ROOT/'outputs/base/us_mid/portfolio_ml/rff/pf.parquet',ROOT/'outputs/base/eu_large/portfolio_ml/rff/pf.parquet']}
    for spec in protocol['iterations'][:args.upto]:
        n=spec['number'];folder=ROOT/f'outputs/iteration_{n}/us_large';folder.mkdir(parents=True,exist_ok=True)
        before=list(accepted);reference=previous;reference_label='Base (annual refitting)' if previous_number==1 else f'Iteration {previous_number}'
        if n==1:
            state=dict(accepted_state)
            for name in ('pf.parquet','weights.parquet','chosen.csv'):shutil.copy2(BASE/name,folder/name)
            candidate=base;test=dict(probability=None,ci_low=0.,ci_high=0.,identical=True,valid_reps=0)
            keep=True;verdict='Annual refitting already included in Base; retained as design'
        else:
            state=candidate_state(accepted_state,spec['override'])
            candidate=engine.run_candidate(state,folder)
            test=paired_probability(candidate,reference,protocol['bootstrap_reps'],protocol['bootstrap_block_months'],protocol['bootstrap_seed'])
            keep=test['probability'] is not None and test['probability']>=protocol['acceptance_threshold'] and not test['identical']
            verdict='Kept' if keep else ('Not included: identical to previous accepted version' if test['identical'] else 'Not included: P(better) below 80%')
        verify_accounting(folder,engine.panel_for(state,folder),engine.wealth)
        if keep:
            accepted_state=state;accepted=inheritance(accepted,n,True);previous=candidate;previous_number=n
        decision=dict(number=n,name=spec['name'],reference=reference_label,inherited_iterations=before,
            p_better=test['probability'],bootstrap=test,threshold=protocol['acceptance_threshold'],
            accepted=bool(keep),enabled_in_next_accepted_setup=bool(keep),included_after=accepted,
            last_accepted_after=previous_number,verdict=verdict,candidate_state=state,accepted_state_after=accepted_state,
            evaluation_start=str(engine.cfg.dates.test_start),evaluation_end=str(engine.cfg.dates.test_end),
            p_is_conventional_p_value=False,independent_unseen_confirmation=False)
        write_json(folder/'decision.json',decision)
        write_iteration_report(n,decision,candidate,reference,base,static,market)
        report_saved(upto=n)
    if any(digest(Path(p))!=h for p,h in base_hashes.items()):raise AssertionError('Base portfolios changed during iteration work')
    write_json(OUT/'verification.json',dict(status='PASS',base_portfolios_unchanged=True,iterations=args.upto,
        acceptance_threshold=protocol['acceptance_threshold'],test_months=72,source_signature=engine.signature))
    return 0


if __name__=='__main__':raise SystemExit(main())


### `scripts/organise_results.py`

Index saved base/iteration outputs without fitting or completing experiments.

**Role:** Reporting source. Reads available local results when called; this cell writes the module.

**Input:** the embedded source below. **Output:** `scripts/organise_results.py`. Expand the code cell to read or edit it.


In [ ]:
%%writefile scripts/organise_results.py
"""Index saved base/iteration outputs without fitting or completing experiments."""
from pathlib import Path
import hashlib
import html
import json
import numpy as np
import pandas as pd

ROOT=Path(__file__).resolve().parents[1]
GROUPS=('base',*(f'iteration_{i}' for i in range(1,6)))
MARKETS=('us_large','us_mid','eu_large')
DESCRIPTIONS={
    'base':'US high-cap Static ML* and default Portfolio ML in all three markets; overall analysis adds Market references.',
    'iteration_1':'Annual refitting, already in Base; retained as design, P(better) N/A.',
    'iteration_2':'Gamma 5 against the previous accepted version.',
    'iteration_3':'Exit rank 120 and costs x2, inheriting accepted changes.',
    'iteration_4':'Market plus pretest-calibrated 1% TE-target ML tilt.',
    'iteration_5':'Recent-history weighting with 3/5/10-year half-lives and equal weighting.'}


def markets_for_group(group):
    """Only publish current scope; never recreate removed historical folders."""
    return MARKETS if group=='base' else ('us_large',)


def sha256(path):
    h=hashlib.sha256()
    with path.open('rb') as f:
        for block in iter(lambda:f.read(1024*1024),b''):h.update(block)
    return h.hexdigest()


def active_label(path,root):
    calibration=root/'active/calibrated.csv'
    if calibration.exists():
        for row in pd.read_csv(calibration).itertuples():
            if f'gte{row.gamma_te:g}'==path.parent.name:return f'Portfolio-ML, TE {row.te_target:.0%}'
    return f'Portfolio-ML, {path.parent.name}'


def discover(group,market):
    root=ROOT/'outputs'/group/market
    files=[]
    if group=='base':
        files=([(root/'static_ml_star/pf.parquet','experiment',None)] if market=='us_large' else [])+[(root/'portfolio_ml/rff/pf.parquet','experiment',None)]
    else:
        files=[(root/'pf.parquet','experiment',f'Portfolio ML iteration {group.split("_")[1]}')]
        base=ROOT/'outputs/base/us_large'
        files += [(base/'portfolio_ml/rff/pf.parquet','reference','Base Portfolio ML'),
                  (base/'static_ml_star/pf.parquet','reference','Static ML*'),
                  (ROOT/'outputs/base/overall analysis/market_references/us_large/pf.parquet','reference','Market')]
    frames=[];inventory=[]
    for path,role,label in files:
        if not path.exists():continue
        data=pd.read_parquet(path)
        for name,d in data.groupby('type',sort=False):
            if group=='base':
                from jkmp.strategies import base_strategies
                if name not in base_strategies(market):raise ValueError('Other method in base')
            d=d.copy();d['strategy']=label or name;d['role']=role;d['group']=group;d['market']=market
            frames.append(d)
            inventory.append(dict(group=group,market=market,strategy=label or name,role=role,
                file=str(path.relative_to(ROOT/'outputs')),sha256=sha256(path),months=len(d)))
    columns=['type','eom','eom_ret','r','tc','turnover','inv','shorting','strategy','role','group','market']
    return (pd.concat(frames,ignore_index=True) if frames else pd.DataFrame(columns=columns)),inventory


def stats(data,market_root):
    rows=[];benchmark=None
    path=market_root/'benchmarks/pf_test.parquet'
    if path.exists():benchmark=pd.read_parquet(path).query("type == 'Market'").set_index('eom_ret')
    for (strategy,role),d in data.groupby(['strategy','role'],sort=False):
        r=d.r.to_numpy(float);tc=d.tc.to_numpy(float);sd=np.std(r,ddof=1)
        row=dict(strategy=strategy,role=role,months=len(d),first_return=str(pd.Timestamp(d.eom_ret.min()).date()),
            last_return=str(pd.Timestamp(d.eom_ret.max()).date()),annual_gross_return=12*r.mean(),annual_cost=12*tc.mean(),
            annual_net_return=12*(r-tc).mean(),annual_gross_volatility=np.sqrt(12)*sd,
            net_sharpe_paper=np.sqrt(12)*(r-tc).mean()/sd if sd>0 else np.nan,
            mean_leverage=d.inv.mean(),mean_monthly_turnover=d.turnover.mean())
        if benchmark is not None:
            b=benchmark.reindex(d.eom_ret)
            if b[['r','tc']].notna().all().all():
                active=r-b.r.to_numpy();active_net=(r-tc)-(b.r-b.tc).to_numpy();te=active.std(ddof=1)*np.sqrt(12)
                row.update(annual_active_net=12*active_net.mean(),tracking_error_realised=te,
                    information_ratio_net=12*active_net.mean()/te if te>0 else np.nan,
                    benchmark_cost_convention='net strategy minus net same-universe Market')
        rows.append(row)
    return pd.DataFrame(rows)


def plot(data,destination,title):
    if data.empty:return
    import matplotlib
    matplotlib.use('Agg')
    import matplotlib.pyplot as plt
    destination.parent.mkdir(parents=True,exist_ok=True)
    fig,axes=plt.subplots(1,2,figsize=(12,4.5))
    for label,d in data.groupby('strategy'):
        d=d.sort_values('eom_ret')
        axes[0].plot(d.eom_ret,100*d.r.cumsum(),label=label)
        axes[1].plot(d.eom_ret,100*d.tc.cumsum(),label=label)
    axes[0].set(title='Cumulative gross excess return (arithmetic)',ylabel='Percentage points')
    axes[1].set(title='Cumulative modelled trading cost',ylabel='Percentage points')
    if data.tc.max()>1:axes[1].set_yscale('symlog',linthresh=1);axes[1].set_title('Cumulative cost (symlog)')
    for ax in axes:ax.grid(alpha=.2);ax.legend(fontsize=7)
    fig.suptitle(title);fig.tight_layout()
    for suffix in ('png','pdf'):fig.savefig(destination.with_suffix('.'+suffix),dpi=170)
    plt.close(fig)


def main(argv=None):
    import argparse
    parser=argparse.ArgumentParser(description=__doc__)
    parser.add_argument('--only-group',choices=GROUPS,default=None)
    args=parser.parse_args(argv)
    availability=[];inventory=[]
    style='<style>body{max-width:1300px;margin:36px auto;padding:0 20px;font:15px/1.5 system-ui;color:#21323e}table{border-collapse:collapse;font-size:13px}td,th{border:1px solid #ced8df;padding:8px}th{background:#e8eff4}.scroll{overflow-x:auto}img{width:100%}</style>'
    for group in ([args.only_group] if args.only_group else GROUPS):
        group_root=ROOT/'outputs'/group;group_root.mkdir(exist_ok=True)
        for market in markets_for_group(group):
            root=group_root/market;reports=root/'reports';reports.mkdir(parents=True,exist_ok=True)
            data,files=discover(group,market);inventory.extend(files)
            experiment=data.loc[data.role.eq('experiment')];reference=data.loc[data.role.ne('experiment')]
            if group=='base':status='available' if experiment.strategy.nunique()==(2 if market=='us_large' else 1) else 'not run' if experiment.empty else 'partial'
            elif not experiment.empty:status='saved experiment results available'
            elif not reference.empty:status='preparatory/reference outputs only; experiment unfinished'
            elif any(p.is_file() for p in root.rglob('*') if 'reports' not in p.relative_to(root).parts and p.name not in ('status.json','README.md','index.html')):status='input audits/references only; experiment unfinished'
            else:status='not run; no saved experiment performance'
            record=dict(group=group,market=market,status=status,experiment_series=int(experiment.strategy.nunique()),
                reference_series=int(reference.strategy.nunique()),new_experiments_run=False,
                scheduled_in_current_scope=group=='base' or market=='us_large')
            if group!='base' and (root/'decision.json').exists():
                decision=json.loads((root/'decision.json').read_text())
                record.update(verdict=decision['verdict'],p_better=decision['p_better'],
                    included=decision['accepted'],inherited_iterations=decision['inherited_iterations'])
            availability.append(record)
            (root/'status.json').write_text(json.dumps(record,indent=2),encoding='utf-8')
            table=stats(data,root)
            if not data.empty:
                data.to_csv(reports/'available_monthly_returns.csv',index=False)
                table.to_csv(reports/'available_performance.csv',index=False)
            plot(experiment,reports/'charts/experiment_returns_and_costs',f'{group} | {market} | saved experiment results')
            plot(reference,reports/'charts/reference_returns_and_costs',f'{group} | {market} | reference/preparatory results')
            intro=f'<h1>{group}: {market}</h1><p>{html.escape(DESCRIPTIONS[group])}</p><p><strong>Status: {html.escape(status)}</strong></p>'
            text='Only freshly fitted Kati outputs are indexed here. This reporting step does not fit models. Test returns are 2020-2025; characteristic selection used US high-cap 2005-2014 only. Iterations apply only to US high-cap Portfolio ML. Reference/preparatory statistics are not a completed iteration result.'
            body=intro+'<p>'+text+'</p>'
            if group!='base' and (root/'decision.json').exists():
                decision=json.loads((root/'decision.json').read_text())
                probability=decision['p_better']
                ptext='N/A (already in Base)' if probability is None else f'{probability:.1%}'
                body+=f'<p><b>{html.escape(decision["verdict"])}</b>; P(better) = {ptext}. Inherited iterations: {decision["inherited_iterations"]}. Included after this decision: {decision["included_after"]}.</p><p><a href="SUMMARY.md">Inheritance and decision summary</a> | <a href="../../iteration_summary/index.html">All five experiments and final comparison</a></p>'
                body+='<img src="charts/net_returns_and_costs.png" alt="Candidate, previous accepted version, Base, Static ML star and Market: returns, drawdowns and relative performance"><p><a href="performance_summary.csv">Detailed performance CSV</a> | <a href="monthly_returns.csv">Monthly series</a> | <a href="decision.json">Decision and exact inherited settings</a></p>'
            if not table.empty:body+='<h2>Available saved performance</h2><p>Returns and costs are annual decimal fractions; leverage and turnover are ratios.</p><div class="scroll">'+table.to_html(index=False,float_format=lambda x:f'{x:.5g}')+'</div>'
            if (root/'active/summary.csv').exists():body+='<p><a href="active/summary.csv">Original iteration summary (unchanged)</a></p>'
            for role in ('experiment','reference'):
                image=f'reports/charts/{role}_returns_and_costs.png'
                if (root/image).exists():body+=f'<img src="{image}" alt="{role} results">'
            (root/'index.html').write_text('<!doctype html><meta charset="utf-8">'+style+body,encoding='utf-8')
            (root/'README.md').write_text(f'# {group}: {market}\n\n{DESCRIPTIONS[group]}\n\n**Status: {status}.**\n\n{text}\n\nOpen index.html. CSV, Parquet and chart files remain in their respective run folders. Newly generated descriptive summaries/charts are under reports/.\n',encoding='utf-8')
            if group!='base' and (root/'SUMMARY.md').exists():
                summary=(root/'SUMMARY.md').read_text(encoding='utf-8')
                (root/'README.md').write_text(summary,encoding='utf-8')
                (group_root/'SUMMARY.md').write_text(summary+'\n\nDetailed results: us_large/index.html\n',encoding='utf-8')
        rows=pd.DataFrame(availability).query('group == @group')
        rows.to_csv(group_root/'availability.csv',index=False)
        links=''.join(f'<li><a href="{m}/index.html">{m}</a></li>' for m in markets_for_group(group))
        # Base keeps its richer validated two-method comparison as index.html.
        if group!='base' or not (group_root/'index.html').exists():
            (group_root/'index.html').write_text('<!doctype html><meta charset="utf-8">'+style+f'<h1>{group}</h1><p>{DESCRIPTIONS[group]}</p>'+rows.to_html(index=False)+f'<ul>{links}</ul><p>Only completed Kati results are shown; experiments without new-feature outputs remain unfinished.</p>',encoding='utf-8')
    out=ROOT/'outputs'
    if args.only_group:
        previous=out/'availability_by_group_and_market.csv'
        if previous.exists():availability += pd.read_csv(previous).query('group != @args.only_group').to_dict('records')
        previous_inventory=out/'organisation_audit/result_source_inventory.csv'
        if previous_inventory.exists():inventory += pd.read_csv(previous_inventory).query('group != @args.only_group').to_dict('records')
    pd.DataFrame(availability).to_csv(out/'availability_by_group_and_market.csv',index=False)
    (out/'organisation_audit').mkdir(exist_ok=True)
    pd.DataFrame(inventory).to_csv(out/'organisation_audit/result_source_inventory.csv',index=False)
    links='<li><a href="us_high_cap_selection/index.html">US high-cap 2005–2014 characteristic selection, correlation and covariance</a></li><li><a href="dashboard.html">Interactive results explorer</a></li>'+''.join(f'<li><a href="{g}/index.html">{g}</a>: {DESCRIPTIONS[g]}</li>' for g in GROUPS)
    (out/'index.html').write_text('<!doctype html><meta charset="utf-8">'+style+'<h1>Kati results by run group</h1><ul>'+links+'</ul>'+pd.DataFrame(availability).to_html(index=False)+'<p>Base contains US high-cap Static ML* and regional Portfolio ML. Iteration scope follows Keats; results require fresh fits with the new 50. Open dashboard.html for interactive comparisons.</p>',encoding='utf-8')
    print(pd.DataFrame(availability).to_string(index=False))
    from comparison_analysis import add_navigation
    add_navigation(include_iterations=args.only_group!="base")
    return 0


if __name__=='__main__':raise SystemExit(main())


### `scripts/prepare_inputs.py`

Create an independent local input snapshot for the frozen research 50. This is a one-time, offline adapter, not a characteristic selector or ML fit. The first call reads the two sibling projects; subsequent calls verify and reuse the local snapshot. Pipeline calculations need only Kati_tryout/data afterward.

**Role:** Pipeline implementation or configuration. This cell writes the file without running its main function.

**Input:** the embedded source below. **Output:** `scripts/prepare_inputs.py`. Expand the code cell to read or edit it.


In [ ]:
%%writefile scripts/prepare_inputs.py
"""Create an independent local input snapshot for the frozen research 50.

This is a one-time, offline adapter, not a characteristic selector or ML fit.
The first call reads the two sibling projects; subsequent calls verify and reuse
the local snapshot. Pipeline calculations need only Kati_tryout/data afterward.
"""
from __future__ import annotations

import argparse
import hashlib
import json
from pathlib import Path
import shutil

import numpy as np
import pandas as pd
import pyarrow.parquet as pq

ROOT = Path(__file__).resolve().parents[1]
MARKETS = {"US": ("us_large", "us_mid"), "EU": ("eu_large",)}
KEYS = ["id", "eom"]
BASE_COLUMNS = ["id", "eom", "sic", "size_grp", "me", "ret_exc_lead1m", "hist_months", "dolvol_126d"]
MEMBER_COLUMNS = ["id", "eom", "form_year", "mcap_rank", "status"]
REFERENCES = ("ff_factors_monthly.parquet", "ff_factors_daily.parquet", "factor_details.parquet", "cluster_labels.parquet")


def digest(path):
    h = hashlib.sha256()
    with Path(path).open("rb") as stream:
        for block in iter(lambda: stream.read(8 * 1024 * 1024), b""):
            h.update(block)
    return h.hexdigest()


def normalize(frame):
    frame = frame.copy()
    frame["id"] = pd.to_numeric(frame.id, errors="raise").astype("int64")
    frame["eom"] = pd.to_datetime(frame.eom)
    if frame[KEYS].isna().any().any() or frame.duplicated(KEYS).any():
        raise ValueError("Missing or duplicate stock-month keys")
    if not frame.eom.dt.is_month_end.all():
        raise ValueError("Expected month-end formation dates")
    return frame


def same_keys(left, right):
    a = pd.MultiIndex.from_frame(left[KEYS])
    b = pd.MultiIndex.from_frame(right[KEYS])
    return len(a) == len(b) and a.is_unique and b.is_unique and a.difference(b).empty and b.difference(a).empty


def _write_json(path, value):
    path.parent.mkdir(parents=True, exist_ok=True)
    path.write_text(json.dumps(value, indent=2, default=str) + "\n", encoding="utf-8")


def compare_source_vintages(original, pipeline=ROOT):
    """Compare shared raw features, without replacing either source vintage.

    This is an offline bootstrap audit. It is not needed by subsequent model
    runs, which use the independent local snapshot only.
    """
    original = Path(original)
    pipeline = Path(pipeline)
    data = pipeline / "data"
    names = json.loads((data / "characteristics/recommended_50.json").read_text())
    records = []
    for region in MARKETS:
        source_path = original / f"screened_{region}.parquet"
        shared = [name for name in names if name in pq.read_schema(source_path).names]
        new = normalize(pd.read_parquet(data / f"characteristics/raw50_{region}.parquet", columns=[*KEYS, *shared]))
        old = normalize(pd.read_parquet(source_path, columns=[*KEYS, *shared]))
        old = old.loc[old.eom.between(new.eom.min(), new.eom.max())].copy()
        if not same_keys(old, new):
            raise AssertionError("Source-vintage comparison requires exact matching stock-month keys")
        joined = old.merge(new, on=KEYS, validate="one_to_one", suffixes=("_old", "_new"))
        for name in shared:
            a = joined[name + "_old"].to_numpy(dtype=float, na_value=np.nan)
            b = joined[name + "_new"].to_numpy(dtype=float, na_value=np.nan)
            observed_a, observed_b = np.isfinite(a), np.isfinite(b)
            joint = observed_a & observed_b
            difference = np.abs(a[joint] - b[joint])
            records.append({"region": region, "characteristic": name, "stock_months": len(joined),
                "joint_nonmissing": int(joint.sum()), "both_missing": int((~observed_a & ~observed_b).sum()),
                "old_missing_new_observed": int((~observed_a & observed_b).sum()),
                "old_observed_new_missing": int((observed_a & ~observed_b).sum()),
                "different_at_absolute_tolerance_1e_10": int((difference > 1e-10).sum()),
                "maximum_absolute_difference": float(difference.max()) if len(difference) else None,
                "mean_absolute_difference": float(difference.mean()) if len(difference) else None,
                "comparison": "original Keats raw screened feature vs frozen Characteristics_tryout raw value; absolute tolerance1e-10, no rescaling"})
    report = pd.DataFrame(records)
    path = data / "characteristics/source_vintage_comparison.csv"
    report.to_csv(path, index=False)
    summary = {"status": "PASS", "comparison_completed": True, "feature_region_pairs": len(report),
        "overlap_features": report.characteristic.unique().tolist(),
        "pairs_with_observed_value_revisions": int(report.different_at_absolute_tolerance_1e_10.gt(0).sum()),
        "pairs_with_missingness_revisions": int((report.old_missing_new_observed + report.old_observed_new_missing).gt(0).sum()),
        "observed_values_different": int(report.different_at_absolute_tolerance_1e_10.sum()),
        "files_or_models_modified": "Only this comparison CSV and audit JSON; neither source values nor feature selection/model caches changed",
        "interpretation": "Any revisions mean the experiment combines the frozen new feature set with its supplied source vintage, rather than isolating feature names alone."}
    _write_json(pipeline / "outputs/validation/source_vintage_comparison.json", summary)
    return path, summary


def bootstrap(repo_root=None, pipeline_root=None):
    """Materialize local inputs, or verify every previously snapshotted file.

    Existing snapshots are immutable: a changed/damaged file raises instead of
    silently replacing inputs underneath saved results. No network is used.
    """
    pipeline = Path(pipeline_root).resolve() if pipeline_root else ROOT
    repo = Path(repo_root).resolve() if repo_root else pipeline.parent
    data = pipeline / "data"
    manifest_path = data / "SNAPSHOT_MANIFEST.json"
    if manifest_path.exists():
        manifest = json.loads(manifest_path.read_text(encoding="utf-8"))
        for item in manifest["files"]:
            target = (pipeline / item["path"]).resolve()
            if not target.is_relative_to(data.resolve()):
                raise ValueError(f"Snapshot manifest path escapes data: {target}")
            if not target.is_file() or digest(target) != item["sha256"]:
                raise ValueError(f"Snapshot changed or incomplete: {target}. Existing results were not modified.")
        if json.loads((data / "characteristics/recommended_50.json").read_text()) != manifest["features"]:
            raise ValueError("Frozen feature list differs from snapshot")
        print(f"Verified independent local snapshot: {len(manifest['files'])} files; 50 characteristics", flush=True)
        return manifest

    if (pipeline/'settings.json').exists() and json.loads((pipeline/'settings.json').read_text()).get('selection_scope')=='us_large':
        raise FileNotFoundError('Restore the current US high-cap input snapshot; do not bootstrap the archived three-market feature selection. See scripts/update_us_selection.py.')

    keats = repo / "keats_pipeline"
    tryout = repo / "Characteristics_tryout/data"
    settings = json.loads((keats / "settings.json").read_text(encoding="utf-8"))
    original = (keats / settings["source"]).resolve()
    reference = (keats / settings["reference_data"]).resolve()
    selected_path = tryout / "recommended_50.json"
    names = json.loads(selected_path.read_text(encoding="utf-8"))
    if len(names) != 50 or len(set(names)) != 50 or set(names) & {"source_ret_exc_lead1m", "ret_exc", "ret_exc_lead1m", "id", "eom"}:
        raise ValueError("Expected exactly 50 unique predictor characteristics")
    selected_metadata_path = repo / "Characteristics_tryout/outputs/selection_explanation/source_tables/recommended_50.csv"
    if not selected_metadata_path.exists():
        selected_metadata_path = repo / "Characteristics_tryout/outputs/recommended_50.csv"
    selected = pd.read_csv(selected_metadata_path)
    if selected.characteristic.tolist() != names or not selected.training_direction.isin([-1, 1]).all():
        raise ValueError("Frozen selection evidence does not match the 50 names/directions")

    files = []
    sources = []
    audits = []

    def source_record(path):
        item = {"path": str(path.resolve()), "bytes": path.stat().st_size, "sha256": digest(path)}
        sources.append(item)
        return item

    def register(path, role, source=None):
        item = {"path": str(path.relative_to(pipeline)).replace("\\", "/"), "bytes": path.stat().st_size,
                "sha256": digest(path), "role": role}
        if source is not None:
            item["copied_from"] = str(source.resolve())
        files.append(item)

    def copy(path, destination, role):
        source_record(path)
        destination.parent.mkdir(parents=True, exist_ok=True)
        if destination.exists() and digest(destination) != digest(path):
            raise ValueError(f"Refusing to overwrite a different input: {destination}")
        if not destination.exists():
            shutil.copy2(path, destination)
        if digest(path) != digest(destination):
            raise AssertionError(f"Copy hash mismatch: {path}")
        register(destination, role, path)

    def parquet(frame, destination, role):
        destination.parent.mkdir(parents=True, exist_ok=True)
        temporary = destination.with_suffix(destination.suffix + ".building")
        frame.to_parquet(temporary, index=False)
        if destination.exists():
            if digest(temporary) != digest(destination):
                raise ValueError(f"Refusing to overwrite a different generated input: {destination}")
            temporary.unlink()
        else:
            temporary.replace(destination)
        check = pd.read_parquet(destination, columns=KEYS)
        if not same_keys(frame, check):
            raise AssertionError(f"Written keys changed: {destination}")
        register(destination, role)

    copy(selected_path, data / "characteristics/recommended_50.json", "Frozen, ordered predictor list")
    copy(selected_metadata_path, data / "characteristics/selected_feature_metadata.csv", "Frozen group, sign and selection evidence; proxy labels explicitly retained")
    directions = selected[["characteristic", "theme", "theme_assignment_method", "published_jkp_153", "direction", "training_direction"]].copy()
    directions["model_theme"] = directions.theme.str.strip().str.lower().str.replace(" ", "_", regex=False).str.replace("-", "_", regex=False)
    directions["model_theme_direction"] = np.where(directions.published_jkp_153, directions.direction, directions.training_direction)
    directions["model_direction_source"] = np.where(directions.published_jkp_153, "published JKP direction", "2005-2014 learned direction for provisional proxy group")
    if not directions.model_theme_direction.isin([-1, 1]).all():
        raise ValueError("Every theme exposure requires an explicit valid direction")
    directions_path = data / "characteristics/theme_directions.csv"
    directions.to_csv(directions_path, index=False)
    register(directions_path, "Theme covariance direction policy: published28 signs, training22 proxy signs; raw ML ranks remain unflipped")
    for filename in REFERENCES:
        copy(reference / filename, data / "reference" / filename, "Unchanged reference input")
    # Infrastructure copies deliberately exclude old normalized/raw feature lag
    # files and the old aggregate manifest; new lag files are built below.
    for path in sorted((keats / "data/inputs").iterdir()):
        if path.is_file() and path.suffix in {".parquet", ".json", ".csv"} and not (
                path.name.startswith(("lag_history", "unavailable_lag_history")) or path.name == "manifest.json"):
            copy(path, data / "inputs" / path.name, "Unchanged returns, daily covariance, regime or infrastructure audit input")

    for region, universes in MARKETS.items():
        print(f"Snapshot {region}: joining frozen raw50 to original infrastructure metadata", flush=True)
        metadata_path = tryout / f"metadata_{region}.parquet"
        source_record(metadata_path)
        meta = normalize(pd.read_parquet(metadata_path))
        parts = []
        for path in sorted((tryout / "panels").glob(f"{region}_*.parquet")):
            source_record(path)
            frame = normalize(pd.read_parquet(path, columns=[*KEYS, *names, "source_row_found"]))
            frame[names] = frame[names].apply(pd.to_numeric, errors="raise").astype(float)
            if np.isinf(frame[names].to_numpy()).any():
                raise ValueError(f"Infinite source characteristic: {path}")
            parts.append(frame)
        if not parts:
            raise ValueError(f"No characteristic partitions for {region}")
        raw = normalize(pd.concat(parts, ignore_index=True)).sort_values(["eom", "id"], ignore_index=True)
        if not same_keys(meta, raw):
            raise AssertionError(f"Downloaded {region} keys differ from the frozen broad training/held-stock keys")
        # Source flags refer to missing entire observations; feature-level NaNs
        # are retained independently and become 0.5 only after monthly ranking.
        parquet(raw, data / f"characteristics/raw50_{region}.parquet", "Raw selected characteristics; original NaNs and source-row flags preserved")

        source_path = original / f"screened_{region}.parquet"
        source_record(source_path)
        broad = normalize(pd.read_parquet(source_path, columns=BASE_COLUMNS))
        key_index = pd.MultiIndex.from_frame(raw[KEYS])
        in_scope = pd.MultiIndex.from_frame(broad[KEYS]).isin(key_index)
        excluded_beyond_horizon = broad.loc[~in_scope, KEYS].copy()
        broad = broad.loc[in_scope].copy()
        if not same_keys(broad, raw):
            raise AssertionError(f"Original metadata does not exactly cover frozen {region} input keys")
        if len(excluded_beyond_horizon) and not excluded_beyond_horizon.eom.gt(raw.eom.max()).all():
            raise AssertionError("A pre-horizon original source key was unexpectedly omitted")
        broad = broad.merge(raw, on=KEYS, how="left", validate="one_to_one").sort_values(["eom", "id"], ignore_index=True)
        parquet(broad, data / f"source/screened_{region}.parquet", "Original screened population and infrastructure joined to raw selected50; no source forward-return audit field admitted")
        ranks = raw[names].groupby(raw.eom).rank(method="average", pct=True).fillna(0.5)
        rank_frame = pd.concat([raw[KEYS], ranks.add_suffix("_rank")], axis=1)
        a = ranks.to_numpy()
        if not np.isfinite(a).all() or (a < 0).any() or (a > 1).any():
            raise AssertionError("Ranked characteristics outside [0,1]")
        coverage = raw[names].notna().mean().rename("observed_fraction").rename_axis("characteristic").reset_index()
        coverage["region"] = region
        coverage_path = data / f"characteristics/coverage_{region}.csv"
        coverage.to_csv(coverage_path, index=False)
        register(coverage_path, "Raw feature availability; includes warmup and test dates without inspecting return outcomes")

        membership_audit = {}
        for universe in universes:
            member_path = original / f"panel_{universe}.parquet"
            source_record(member_path)
            members_all = normalize(pd.read_parquet(member_path, columns=MEMBER_COLUMNS))
            members = members_all.loc[members_all.eom.between(raw.eom.min(), raw.eom.max())].copy()
            prepared = members.merge(rank_frame, on=KEYS, how="left", validate="one_to_one", indicator=True)
            if not prepared._merge.eq("both").all() or not same_keys(members, prepared):
                raise AssertionError(f"Missing characteristic data for original {universe} membership")
            prepared = prepared.drop(columns="_merge")
            parquet(prepared, data / f"source/panel_{universe}.parquet", "Original declared-period membership metadata with newly computed selected50 ranks")
            membership_audit[universe] = {"members_original_in_declared_period": len(members), "members_snapshotted": len(prepared),
                "exact_membership_keys_preserved": True, "original_member_rows_beyond_input_horizon": len(members_all) - len(members)}

        lag_path = tryout / f"lag_history/{region}.parquet"
        source_record(lag_path)
        lag = normalize(pd.read_parquet(lag_path, columns=[*KEYS, *names, "source_row_found"]))
        old_lag_path = keats / f"data/inputs/lag_history_{region}.parquet"
        source_record(old_lag_path)
        old_lag = normalize(pd.read_parquet(old_lag_path, columns=[*KEYS, "g", "growth_imputed_neutral"]))
        old_unavailable_path = keats / f"data/inputs/unavailable_lag_history_{region}.csv"
        source_record(old_unavailable_path)
        old_unavailable = normalize(pd.read_csv(old_unavailable_path, parse_dates=["eom"]))
        requested = pd.concat([old_lag[KEYS], old_unavailable[KEYS]], ignore_index=True).drop_duplicates(KEYS)
        if not same_keys(lag, requested):
            raise AssertionError(f"Supplemental {region} requested lag keys differ from Keats")
        actual = lag.loc[lag.source_row_found].copy()
        usable = actual.merge(old_lag, on=KEYS, how="inner", validate="one_to_one")
        # A usable feature source must have an actual saved infrastructure growth
        # observation. Absence remains a key exclusion, never a synthetic row.
        unavailable_keys = pd.MultiIndex.from_frame(requested[KEYS]).difference(pd.MultiIndex.from_frame(usable[KEYS]))
        unavailable = unavailable_keys.to_frame(index=False).sort_values(KEYS, ignore_index=True)
        normalized_lag = usable[[*KEYS, "g", "growth_imputed_neutral"]].copy()
        for name in names:
            normalized_lag[name] = 0.5
        grouped_reference = {date: group for date, group in raw.groupby("eom", sort=False)}
        for date, group in usable.groupby("eom", sort=False):
            if date not in grouped_reference:
                raise ValueError(f"No actual regional rank reference for supplemental month {date}")
            reference_group = grouped_reference[date]
            for name in names:
                baseline = np.sort(reference_group[name].dropna().to_numpy(dtype=float))
                values = pd.to_numeric(group[name], errors="raise").to_numpy(dtype=float, na_value=np.nan)
                valid = np.isfinite(values)
                normalized = np.full(len(group), 0.5)
                if len(baseline):
                    lo = np.searchsorted(baseline, values[valid], side="left")
                    hi = np.searchsorted(baseline, values[valid], side="right")
                    normalized[valid] = (lo + hi + 2.0) / (2.0 * (len(baseline) + 1))
                normalized_lag.loc[group.index, name] = normalized
        normalized_lag["ret_ld1"] = np.nan
        normalized_lag["lambda"] = np.nan
        normalized_lag["lag_history_only"] = True
        if not np.isfinite(normalized_lag[names].to_numpy()).all():
            raise AssertionError("Non-finite supplemental feature rank")
        parquet(normalized_lag, data / f"inputs/lag_history_{region}.parquet", "Selected50 normalized pre-entry history; saved growth infrastructure, no training targets")
        parquet(lag, data / f"characteristics/lag_raw50_{region}.parquet", "Raw selected50 supplemental history, including explicit unavailable source rows")
        unavailable_path = data / f"inputs/unavailable_lag_history_{region}.csv"
        unavailable.to_csv(unavailable_path, index=False)
        register(unavailable_path, "Actual missing supplemental keys; apply explicit membership-exclusion policy")
        lag_info_path = data / f"inputs/lag_history_{region}.json"
        _write_json(lag_info_path, {"source": "../characteristics/lag_raw50_" + region + ".parquet",
            "feature_list": "../characteristics/recommended_50.json", "requested_keys": len(requested), "usable_keys": len(usable),
            "unavailable_keys": len(unavailable), "rank_reference": "New raw selected50 regional month plus inserted actual observation; average ties",
            "missing_feature_or_reference_rank": 0.5, "growth": "Unchanged original normalized Keats lag g by exact key",
            "purpose": "Input history only: ret_ld1 and lambda deliberately missing, never additional training targets"})
        register(lag_info_path, "Supplemental history provenance")
        audits.append({"region": region, "stock_months": len(raw), "formation_start": str(raw.eom.min().date()),
            "formation_end": str(raw.eom.max().date()), "exact_broad_keys_preserved": True, "absent_source_rows": int((~raw.source_row_found).sum()),
            "minimum_full_history_feature_coverage": float(coverage.observed_fraction.min()), "memberships": membership_audit,
            "original_broad_rows_beyond_declared_return_horizon": len(excluded_beyond_horizon),
            "lag_requested": len(requested), "lag_usable": len(usable), "lag_unavailable": len(unavailable),
            "zero_filled_raw_features": False, "source_forward_return_audit_used_as_predictor": False})
        print(f"{region}: {len(raw):,} broad rows; {len(usable)} usable lag rows; {len(unavailable)} unavailable keys", flush=True)

    vintage_path, vintage_summary = compare_source_vintages(original, pipeline)
    register(vintage_path, "Shared-feature source-vintage audit; values are never substituted")
    manifest = {"status": "PASS", "pipeline": "Kati_tryout", "features": names, "feature_count": len(names),
        "network_used": False, "models_refitted": False, "runtime_requires_sibling_projects": False,
        "source_vintage_comparison": vintage_summary,
        "theme_direction_policy": {"published_direction_count": int(directions.published_jkp_153.sum()),
            "training_direction_proxy_count": int((~directions.published_jkp_153).sum()),
            "published_vs_training_sign_disagreements": directions.loc[directions.published_jkp_153 & directions.direction.ne(directions.training_direction), "characteristic"].tolist(),
            "applies_to": "theme covariance exposures only; the 50 raw ML ranks are unflipped"},
        "source_hashes": sources, "files": files, "regional_audits": audits,
        "assumptions": ["Original Keats memberships, returns, liquidity, daily covariance observations and regimes are preserved.",
            "Formation December2025 is excluded because its next return is January2026, outside the declared 1995–2025 return horizon.",
            "Raw feature NaNs remain NaN; monthly regional average percentile ranks impute missing observations to 0.5.",
            "The selected50 were chosen with pre2020 evidence; published and proxy labels/signs remain explicitly distinguished.",
            "Theme exposures use published JKP directions for28 published members and 2005–2014 learned directions for22 proxies; raw ML feature ranks are not sign-flipped.",
            "No original feature rank is reused; dolvol_126d remains an infrastructure cost input, not an extra ML predictor.",
            "Old outputs, fitted coefficients, covariance estimates and model caches are not copied."]}
    _write_json(manifest_path, manifest)
    _write_json(pipeline / "outputs/validation/data_snapshot_audit.json", {k: v for k, v in manifest.items() if k not in {"files", "source_hashes"}})
    print(f"Independent input snapshot complete: {manifest_path}", flush=True)
    return manifest


def main(argv=None):
    parser = argparse.ArgumentParser(description=__doc__)
    parser.add_argument("--repo-root", type=Path, default=None, help="Existing repository to read during first snapshot only")
    args = parser.parse_args(argv)
    bootstrap(repo_root=args.repo_root)
    return 0


if __name__ == "__main__":
    raise SystemExit(main())


### `scripts/remove_old_characteristics.py`

Remove the obsolete feature archives explicitly requested by the user. Keeps only hashes as evidence that the current ML implementation was preserved. Candidate data used to reproduce the current selection are retained.

**Role:** Pipeline implementation or configuration. This cell writes the file without running its main function.

**Input:** the embedded source below. **Output:** `scripts/remove_old_characteristics.py`. Expand the code cell to read or edit it.


In [ ]:
%%writefile scripts/remove_old_characteristics.py
"""Remove the obsolete feature archives explicitly requested by the user.

Keeps only hashes as evidence that the current ML implementation was preserved.
Candidate data used to reproduce the current selection are retained.
"""
from pathlib import Path
import hashlib
import json
import shutil

ROOT = Path(__file__).resolve().parents[1]


def main():
    from jkmp.config import _read_yaml
    evidence = ROOT / 'data/characteristics/implementation_baseline.json'
    if not evidence.exists():
        files = list((ROOT/'scripts/jkmp/models').glob('*.py')) + list((ROOT/'scripts/jkmp/portfolio').glob('*.py'))
        configs = {}
        for path in (ROOT/'configs').rglob('*.yaml'):
            value = _read_yaml(path, seen=())
            value.pop('features', None)
            configs[path.relative_to(ROOT).as_posix()] = value
        record = dict(source_hashes={p.relative_to(ROOT).as_posix():hashlib.sha256(p.read_bytes()).hexdigest() for p in files},
                      non_feature_configs=configs, old_feature_archives_removed=True)
        evidence.write_text(json.dumps(record, indent=2, default=str)+'\n', encoding='utf-8')
    targets = [ROOT/'data/archives/20261102_121716_previous_50',
               ROOT/'outputs/archive/20261102_121716_previous_50',
               ROOT.parent/'Characteristics_tryout/data/selection_archive/20261102_122021_previous_cross_market']
    removed = []
    for target in targets:
        resolved = target.resolve()
        if not resolved.is_relative_to(ROOT.parent.resolve()) or resolved == ROOT.parent.resolve():
            raise ValueError(f'Unsafe archive target: {resolved}')
        if resolved.exists():
            shutil.rmtree('\\\\?\\' + str(resolved))
            removed.append(str(resolved.relative_to(ROOT.parent)))
    for path, key in [(ROOT/'settings.json','old_feature_archive'), (ROOT/'data/SNAPSHOT_MANIFEST.json','previous_feature_archive')]:
        value = json.loads(path.read_text())
        value.pop(key, None)
        value['old_feature_archives_removed'] = True
        path.write_text(json.dumps(value, indent=2)+'\n', encoding='utf-8')
    report = dict(status='PASS', old_feature_archives_removed=True, removed=removed,
                  active_selection='US high-cap 2005–2014 association 50', models_fitted=False,
                  recovery='Removed from local disk; no repository backup of these archives was created.')
    (ROOT/'outputs/validation/archive_removal.json').write_text(json.dumps(report, indent=2)+'\n', encoding='utf-8')
    print(json.dumps(report, indent=2))


if __name__ == '__main__':
    main()


### `scripts/run.py`

Offline Kati stages: three base markets and US high-cap Portfolio ML iterations.

**Role:** Pipeline implementation or configuration. This cell writes the file without running its main function.

**Input:** the embedded source below. **Output:** `scripts/run.py`. Expand the code cell to read or edit it.


In [ ]:
%%writefile scripts/run.py
"""Offline Kati stages: three base markets and US high-cap Portfolio ML iterations."""
from __future__ import annotations
import argparse,importlib.util,sys
for _stream in (sys.stdout,sys.stderr):
    if hasattr(_stream,"reconfigure"):_stream.reconfigure(encoding="utf-8")
from workflow import ROOT,UNIVERSES,VARIANTS,audit,config_path,covariance,originals_unchanged,prepare,validate_prepared,write_json
from script_groups import GROUPS,VARIANTS as GROUP_VARIANTS,normalise_group,script_path
SCRIPTS={x:x for x in ("03","04","05","06","07","10","11","12","13","14","monitor")}

def calibrated_gamma(universe, variant, target=0.02):
    import pandas as pd
    from jkmp.config import load_config
    owner = "iteration_4"
    cfg = load_config(config_path(universe, owner))
    path = cfg.paths.outputs / "calibrated.csv"
    if not path.exists():
        raise FileNotFoundError(f"Calibrate this universe first; missing {path}. Never reuse the original gamma_TE.")
    points = pd.read_csv(path)
    selected = points[(points.te_target - target).abs().lt(1e-12)]
    if len(selected) != 1:
        raise ValueError(f"No unique calibration for TE target {target}")
    return float(selected.gamma_te.iloc[0])



def run_stage(stage,universe,variant="base",extra=(),target=.02):
    if stage=="select-characteristics":
        from us_high_cap_selection import main
        return main(list(extra))
    if stage=="dashboard":
        from dashboard import build_dashboard
        return build_dashboard()
    if stage=="verify":
        from verify import main
        return main(list(extra))
    if stage=='verify-iterations':
        from verify_iterations import main
        return main()
    group=normalise_group(variant);variant=GROUP_VARIANTS[group]
    if stage in ('experiment','12'):
        if group=='base' or universe!='us_large':raise ValueError('Choose US high-cap iteration_1 through iteration_5')
        from iteration_sequence import main
        return main(['--upto',group.split('_')[1],*extra])
    if group!='base' and stage in (*SCRIPTS,'prepare','01','02'):
        raise ValueError('Iteration inputs are read-only Base data. Run the experiment stage or scripts/iteration_N/run.py.')
    if group!="base" and universe!="us_large" and stage not in ("audit","00","validate","compare","organise","overall","figure1"):
        raise ValueError("Iterations are scoped to US high-cap Portfolio ML only; no other-market output is created.")
    from jkmp.imputation_audit import enable_reporting
    enable_reporting(f"{group} {stage} {universe} te={target:g}")
    if stage=="figure1":
        from figure1 import main
        return main(["--selected-universe",universe,*extra])
    if stage=="overall":
        from comparison_analysis import main
        return main(list(extra))
    if stage=="market":
        from comparison_analysis import build_market_reference
        return build_market_reference(universe)
    if stage in ("audit","00"):return audit()
    if stage=="validate":return validate_prepared()
    if stage in ("prepare","01"):return prepare(universe,variant)
    if stage=="02":return covariance(universe,variant)
    if stage=="15":
        report=dict(universe=universe,available=False,reason="Kati test is 2020-2025; no separate unseen holdout exists.")
        write_json(ROOT/"outputs"/group/universe/"availability/holdout.json",report);print(report["reason"]);return
    if stage=="compare":
        if group!="base":raise ValueError("Use organise for existing iteration results")
        sys.path.insert(0,str(ROOT/"scripts"))
        from analysis.ml_comparison.cli import main
        return main(list(extra))
    if stage=="organise":
        from organise_results import main
        return main(list(extra))
    from jkmp.config import load_config
    cfg_path=config_path(universe,variant);cfg=load_config(cfg_path)
    if stage=="05" and group=="base" and universe!="us_large":
        raise ValueError("Static-ML* is enabled only for US high-cap base")
    required=[cfg.paths.processed/"panel.parquet"]
    if stage!="03":required.append(cfg.paths.processed/"covariance/cov_factor.parquet")
    missing=[str(p) for p in required if not p.exists()]
    if missing:raise FileNotFoundError("Run upstream stages first: "+", ".join(missing))
    if stage in ("04","05","12"):
        from workflow import ensure_prediction_coverage
        ensure_prediction_coverage(cfg)
    script=script_path(stage,group);args=list(extra)
    if stage in ("03","04","05","06","07","10","11","12"):
        args += ["--config",str(cfg_path)]
    if stage=="12" and args and args[0] in ("run","capacity") and not any("investor.gamma_te=" in a for a in args):
        args += ["--override",f"investor.gamma_te={calibrated_gamma(universe,variant,target):g}"]
    sys.argv=[str(script),*args];sys.path.insert(0,str(ROOT/"scripts"))
    if stage=="12":
        import runpy
        return runpy.run_path(str(script),run_name="__main__")
    spec=importlib.util.spec_from_file_location("keats_stage_"+stage,script);module=importlib.util.module_from_spec(spec);spec.loader.exec_module(module)
    if stage=="13":module.CONFIGS={"80-stock":str(config_path(universe,"active_pfml")),"250-stock":str(config_path(universe,"universe_250"))}
    if stage=="14":module.REF=str(config_path(universe,"universe_250"));module.VAR=str(config_path(universe,"halflife"))
    if stage=="monitor":module.SELECTED_CONFIG=config_path(universe,"universe_250")
    module.main()


def main():
    parser=argparse.ArgumentParser(description=__doc__,allow_abbrev=False)
    parser.add_argument("stage",choices=["audit","validate","prepare","00","01","02","15","compare","organise","market","overall","figure1","dashboard","verify","verify-iterations","select-characteristics","experiment",*SCRIPTS])
    parser.add_argument("--universe",choices=[*UNIVERSES,"all"],default="us_large")
    parser.add_argument("--group",choices=GROUPS,default="base")
    parser.add_argument("--variant",choices=VARIANTS,default=None,help="Legacy alias for a group")
    parser.add_argument("--te-target",type=float,default=.02)
    args,extra=parser.parse_known_args();group=normalise_group(args.variant or args.group)
    if args.stage not in SCRIPTS and args.stage not in ("compare","overall","figure1","organise","experiment") and extra:parser.error(f"Unexpected arguments: {extra}")
    universes=UNIVERSES if args.universe=="all" else [args.universe]
    if group!="base" and args.universe=="all":universes=["us_large"]
    if args.stage in ("audit","validate","00","compare","organise","overall","dashboard","verify","select-characteristics"):universes=["us_large"]
    if args.stage=="figure1":universes=[args.universe]
    failed=[]
    for universe in universes:
        try:
            result=run_stage(args.stage,universe,group,extra,args.te_target)
            if isinstance(result,int) and not isinstance(result,bool) and result!=0:failed.append(universe)
        except (FileNotFoundError,ValueError) as error:print(f"STOPPED {universe}/{group}: {error}",file=sys.stderr);failed.append(universe)
    originals_unchanged();return 2 if failed else 0

if __name__=="__main__":raise SystemExit(main())


### `scripts/run_all.py`

Run the Kati workflow in dependency order with persistent logs and checkpoints. Examples: python Kati_tryout/scripts/run_all.py --scope core python Kati_tryout/scripts/run_all.py --group iteration_3 --scope full python Kati_tryout/scripts/run_all.py --group iteration_3 --scope full --universe us_large Completed outputs are retained. Failed jobs block their dependants; independent universes/stages continue. Rerunning resumes incomplete work. Nothing is committed, pushed or downloaded by this runner.

**Role:** Pipeline implementation or configuration. This cell writes the file without running its main function.

**Input:** the embedded source below. **Output:** `scripts/run_all.py`. Expand the code cell to read or edit it.


In [ ]:
%%writefile scripts/run_all.py
"""Run the Kati workflow in dependency order with persistent logs and checkpoints.

Examples:
    python Kati_tryout/scripts/run_all.py --scope core
    python Kati_tryout/scripts/run_all.py --group iteration_3 --scope full
    python Kati_tryout/scripts/run_all.py --group iteration_3 --scope full --universe us_large

Completed outputs are retained. Failed jobs block their dependants; independent
universes/stages continue. Rerunning resumes incomplete work. Nothing is committed,
pushed or downloaded by this runner.
"""
from __future__ import annotations

import argparse
from dataclasses import dataclass
import datetime as dt
import json
import os
from pathlib import Path
import subprocess
import sys
import time

from workflow import ROOT, UNIVERSES, config_path, originals_unchanged
from jkmp.config import load_config


@dataclass
class Job:
    name: str
    stage: str
    universe: str
    variant: str = "base"
    extra: tuple[str, ...] = ()
    target: float = 0.02
    dependencies: tuple[str, ...] = ()


def plan(universes,scope,group="base",skip_figure1=False):
    from script_groups import normalise_group
    level=5 if group=="all" else (0 if group=='base' else int(normalise_group(group).split('_')[1]))
    jobs=[Job("audit","audit","all"),Job("prepare","prepare","all",dependencies=("audit",)),Job("validate","validate","all",dependencies=("prepare",))]
    tables=[]
    for u in universes:
        def add(label,stage,variant="base",extra=(),target=.02,deps=()):
            name=f"{u}.{label}";jobs.append(Job(name,stage,u,variant,tuple(extra),target,tuple(deps)));return name
        cov=add("base.covariance","02",deps=("validate",));ret=add("base.returns","03",deps=("validate",))
        static=add("base.static","05",deps=(cov,ret)) if u=="us_large" else None
        inputs=add("base.inputs","06",deps=(cov,));pf=add("base.portfolio","07",deps=(inputs,))
        table=add("base.tables","11",deps=tuple(x for x in (static,pf) if x));tables.append(table)
        if scope=="full":add("base.regimes","10",deps=(table,));add("base.availability","15",deps=(table,))
        add("base.market_reference","market",deps=(cov,))
    jobs.append(Job("base.comparison","compare","all",dependencies=tuple(tables)))
    if set(universes)==set(UNIVERSES):
        jobs.append(Job("base.overall_analysis","overall","all",extra=("--only-group","base"),dependencies=tuple(tables)+tuple(f"{u}.base.market_reference" for u in universes)))
    if level and 'us_large' in universes:
        previous='us_large.base.tables'
        for number in range(1,level+1):
            name=f'us_large.iteration_{number}.experiment'
            jobs.append(Job(name,'experiment','us_large',f'iteration_{number}',dependencies=(previous,'us_large.base.market_reference')))
            previous=name
        if level==5 and set(universes)==set(UNIVERSES):
            jobs.append(Job('us_large.iteration_verification','verify-iterations','us_large',dependencies=(previous,)))
            jobs.append(Job('overall.master_comparison','overall','all',extra=('--only-group','master'),dependencies=('us_large.iteration_verification','base.overall_analysis')))
    if scope=="full" and not skip_figure1:
        chosen="all" if set(universes)==set(UNIVERSES) else universes[0]
        extras=("--jobs","2")
        deps=tuple(tables)
        jobs.append(Job("figure1.frontiers","figure1",chosen,extra=extras,dependencies=deps))
    jobs.append(Job("organisation","organise","all",extra=("--only-group","base") if group=="base" else ()))
    jobs.append(Job("dashboard","dashboard","all"))
    return jobs


def output_paths(job):
    if job.stage in ("audit","validate","15","compare","organise","overall","figure1","experiment","verify-iterations"):return []
    cfg=load_config(config_path(job.universe,job.variant));p,o=cfg.paths.processed,cfg.paths.outputs
    if job.stage=="prepare":return [p/"preparation.json",p/"panel.parquet"]
    if job.stage=="02":return [p/"covariance"/n for n in ("cov_factor.parquet","cov_loadings.parquet","cov_diagnostics.csv","provenance.json")]
    if job.stage=="03":
        from jkmp.models.return_model import tuning_marker
        return [p/n for n in ("predictions.parquet","return_model/models.npz","return_model/chosen.csv","return_model/diagnostics.csv")]+[tuning_marker(cfg)]
    if job.stage=="04":return [o/"benchmarks"/n for n in ("pf_test.parquet","weights_test.parquet","table2_test.csv")]
    if job.stage=="market":return [ROOT/"outputs/base/overall analysis/market_references"/job.universe/n for n in ("pf.parquet","weights.parquet","provenance.json")]
    if job.stage=="05":
        from jkmp.portfolio.static_ml_star import tuning_grid_marker
        return [o/"static_ml_star"/n for n in ("pf.parquet","weights.parquet","validation.parquet","chosen.csv")]+[tuning_grid_marker(cfg)]
    if job.stage=="06":return [p/"pfml"/f"rff_g{g:.4f}"/n for g in cfg.portfolio_ml.rff_variance_grid for n in ("meta.json","2025-11-30.npz")]
    if job.stage=="07":
        from jkmp.strategies import portfolio_tuning_marker
        return [o/"portfolio_ml/rff"/n for n in ("pf.parquet","weights.parquet","aim_coefs.parquet","chosen.csv")]+[portfolio_tuning_marker(cfg)]
    if job.stage in ("10","11"):
        from jkmp.strategies import base_sources_marker
        folder="regimes" if job.stage=="10" else "tables"
        names=("regime_table.csv","model_scope.json") if job.stage=="10" else ("table2.csv","table3.csv","table4.csv","subperiods.csv","model_scope.json")
        return [o/folder/n for n in names]+[base_sources_marker(o,folder)]
    if job.stage=="13":return [o/"breadth/capacity_aum.csv"]
    if job.stage=="14":return [o/"halflife/summary.csv"]
    if job.stage=="monitor":return [o/"monitor/monitor.csv"]
    if job.stage=="12":
        if job.extra[0]=="calibrate":return [o/"active/calibrated.csv"]
        if job.extra[0]=="report":return [o/"active/summary.csv"]
        from run import calibrated_gamma
        try:g=calibrated_gamma(job.universe,job.variant,job.target)
        except (FileNotFoundError,ValueError):return []
        if job.extra[0]=="run":return [o/"active"/f"gte{g:g}"/n for n in ("pf.parquet","weights.parquet","chosen.csv")]
        return [o/"active/capacity"/f"gte{g:g}_w{w/1e9:g}B/pf.parquet" for w in cfg.investor.wealth_grid]
    return []


def cached_outputs_ready(paths):
    """A historical marker cannot validate files overwritten by a later run.

    This matters when restoring an earlier parameter choice: its old marker
    may still exist even though the shared output files describe another fit.
    Versioned completion markers are written after their output files.
    """
    if not paths or not all(p.is_file() and p.stat().st_size for p in paths):
        return False
    marker=paths[-1]
    if marker.name.startswith(("tuning_","sources_")):
        return marker.stat().st_mtime_ns >= max(p.stat().st_mtime_ns for p in paths[:-1])
    return True


def main():
    parser = argparse.ArgumentParser(description=__doc__)
    parser.add_argument("--scope", choices=("core", "full"), default="core")
    parser.add_argument("--universe", choices=(*UNIVERSES, "all"), default="all")
    parser.add_argument("--group",choices=("base",*(f'iteration_{i}' for i in range(1,6)),"all"),default="base")
    parser.add_argument("--list-jobs",action="store_true",help="Show dependencies without running experiments")
    parser.add_argument("--skip-figure1",action="store_true",help="Run full diagnostics without fitting the separate frontier grid")
    args = parser.parse_args()
    if args.group!="base" and args.universe not in ("all","us_large"):
        parser.error("Iterations apply only to US high-cap Portfolio ML. Use --universe all for all base markets plus US high-cap iterations.")
    universes = UNIVERSES if args.universe == "all" else [args.universe]
    jobs = plan(universes, args.scope, args.group,args.skip_figure1)
    if args.list_jobs:
        for job in jobs:print(job.name,job.stage,job.variant,"depends on",job.dependencies)
        return 0
    logs = ROOT / "outputs" / (args.group if args.group!="all" else "organisation_audit") / "run_logs"
    logs.mkdir(parents=True, exist_ok=True)
    state_path = logs / f"{args.group}_{args.scope}_{args.universe}_state.json"
    state = json.loads(state_path.read_text()) if state_path.exists() else {"jobs": {}}
    names={job.name for job in jobs}
    historical={k:v for k,v in state["jobs"].items() if k not in names}
    if historical:state.setdefault("historical_out_of_scope_jobs",{}).update(historical)
    state["jobs"]={k:v for k,v in state["jobs"].items() if k in names}
    state["iteration_scope"]="us_large Portfolio ML only; base retains selected markets"
    statuses = {}
    def save():
        state["updated_utc"] = dt.datetime.now(dt.timezone.utc).isoformat()
        temporary = state_path.with_suffix(".writing.json")
        temporary.write_text(json.dumps(state, indent=2))
        temporary.replace(state_path)
    environment = dict(os.environ, PYTHONUNBUFFERED="1", PYTHONIOENCODING="utf-8", PYTHONUTF8="1",
                       OPENBLAS_NUM_THREADS="4", OMP_NUM_THREADS="4", MKL_NUM_THREADS="4")
    for index, job in enumerate(jobs, 1):
        unmet = [name for name in job.dependencies if statuses.get(name) not in ("complete", "reused")]
        if unmet:
            statuses[job.name] = "blocked"
            state["jobs"][job.name] = {"status": "blocked", "dependencies": unmet}
            save()
            print(f"[{index}/{len(jobs)}] BLOCKED {job.name}: {', '.join(unmet)}", flush=True)
            continue
        markers = output_paths(job) if job.universe != "all" else []
        if cached_outputs_ready(markers):
            statuses[job.name] = "reused"
            state["jobs"][job.name] = {"status": "reused", "outputs": [str(p) for p in markers]}
            save()
            print(f"[{index}/{len(jobs)}] REUSE {job.name}", flush=True)
            continue
        stamp = dt.datetime.now().strftime("%Y%m%d_%H%M%S")
        log = logs / f"{job.name}_{stamp}.log"
        command = [sys.executable, "-u", str(ROOT / "scripts/run.py"), job.stage, "--universe", job.universe,
                   "--variant", job.variant, "--te-target", str(job.target), *job.extra]
        if job.stage=="compare":command += ["--selected-universe",args.universe]
        started = time.monotonic()
        state["jobs"][job.name] = {"status": "running", "command": command, "log": str(log)}
        save()
        print(f"[{index}/{len(jobs)}] RUN {job.name} -> {log.name}", flush=True)
        with log.open("w", encoding="utf-8") as stream:
            process = subprocess.run(command, stdout=stream, stderr=subprocess.STDOUT, env=environment)
        status = "complete" if process.returncode == 0 else "failed"
        statuses[job.name] = status
        state["jobs"][job.name].update(status=status, exit_code=process.returncode,
                                          seconds=round(time.monotonic() - started, 2))
        save()
        print(f"[{index}/{len(jobs)}] {status.upper()} {job.name} ({time.monotonic()-started:.1f}s)", flush=True)
        if process.returncode:
            print("\n".join(log.read_text(encoding="utf-8", errors="replace").splitlines()[-12:]), flush=True)
    originals_unchanged()
    failed = [name for name, value in statuses.items() if value in ("failed", "blocked")]
    print(f"Run finished: {len(jobs)-len(failed)}/{len(jobs)} jobs complete/reused; state: {state_path}", flush=True)
    return 2 if failed else 0


if __name__ == "__main__":
    raise SystemExit(main())


### `scripts/script_groups.py`

Authoritative mapping from run groups to script entry points.

**Role:** Pipeline implementation or configuration. This cell writes the file without running its main function.

**Input:** the embedded source below. **Output:** `scripts/script_groups.py`. Expand the code cell to read or edit it.


In [ ]:
%%writefile scripts/script_groups.py
"""Authoritative mapping from run groups to script entry points."""
from pathlib import Path
ROOT=Path(__file__).resolve().parents[1]
GROUPS=("base",*(f"iteration_{i}" for i in range(1,6)))
VARIANTS={group:group for group in GROUPS}
ALIASES={**{x:x for x in GROUPS},**{v:k for k,v in VARIANTS.items()}}


def normalise_group(value):
    if value not in ALIASES:raise ValueError(f"Unknown group: {value}")
    return ALIASES[value]


def baseline_reference_dir(cfg):
    """Route reference portfolios without copying base results into iteration reports."""
    output=cfg.paths.outputs
    group=output.parent.name
    return ROOT/'outputs/base'/output.name


def script_path(stage,group):
    group=normalise_group(group)
    common={"03":"shared/03_return_model.py","04":"shared/04_benchmarks.py"}
    base={"05":"base/05_static_ml.py","06":"shared/06_pfml_inputs.py","07":"base/07_portfolio_ml.py","10":"base/10_regimes.py","11":"base/11_tables.py"}
    if stage=="04":
        if group=="base":raise ValueError("Market is an iteration reference; base reports contain only the two ML strategies")
        return ROOT/"scripts"/common[stage]
    if stage in common:return ROOT/"scripts"/common[stage]
    if group=="base" and stage in base:return ROOT/"scripts"/base[stage]
    if stage=="06" and group=="iteration_2":return ROOT/"scripts/shared/06_pfml_inputs.py"
    if stage in ("12","experiment") and group!="base":return ROOT/"scripts"/group/"run.py"
    if stage=="13" and group=="iteration_2":return ROOT/"scripts/iteration_2/compare_breadth.py"
    if stage=="14" and group=="iteration_3":return ROOT/"scripts/iteration_3/compare_halflife.py"
    if stage=="monitor" and group=="iteration_2":return ROOT/"scripts/iteration_2/monitor.py"
    raise ValueError(f"Stage {stage} is not part of {group}")


### `scripts/select_characteristics.py`

Run the self-contained US high-cap association analysis without fitting ML.

**Role:** Pipeline implementation or configuration. This cell writes the file without running its main function.

**Input:** the embedded source below. **Output:** `scripts/select_characteristics.py`. Expand the code cell to read or edit it.


In [ ]:
%%writefile scripts/select_characteristics.py
"""Run the self-contained US high-cap association analysis without fitting ML."""
from us_high_cap_selection import main

if __name__=='__main__':raise SystemExit(main())


### `scripts/selection_rules.py`

Frozen transparent group/quota/redundancy rules; no ML fit.

**Role:** Pipeline implementation or configuration. This cell writes the file without running its main function.

**Input:** the embedded source below. **Output:** `scripts/selection_rules.py`. Expand the code cell to read or edit it.


In [ ]:
%%writefile scripts/selection_rules.py
"""Frozen transparent group/quota/redundancy rules; no ML fit."""
import numpy as np
import pandas as pd

def assign_themes(registry,corr):
    result=registry.copy();groups=result.loc[result.published_theme.notna()].groupby('published_theme').characteristic.apply(list).to_dict()
    themes=[];methods=[];strengths=[]
    for row in result.itertuples():
        if pd.notna(row.published_theme):themes.append(row.published_theme);methods.append('published JKP cluster');strengths.append(np.nan)
        else:
            proxy={theme:float(corr.loc[row.characteristic,chars].abs().mean()) for theme,chars in groups.items()}
            theme=max(proxy,key=lambda k:proxy[k] if np.isfinite(proxy[k]) else -np.inf)
            themes.append(theme);methods.append('empirical training correlation proxy; not published semantic cluster');strengths.append(proxy[theme])
    result['theme']=themes;result['theme_assignment_method']=methods;result['theme_assignment_strength']=strengths
    return result

def diverse_pick(scores,corr,count,min_theme=2,max_theme=None,score_col='screen_score',redundancy_penalty=.15,published_anchor=False):
    """Greedy transparent coverage/diversity selection, not a global-optimality claim."""
    pool=scores.loc[scores.eligible].copy().set_index('characteristic',drop=False)
    if len(pool)<count:raise ValueError(f'Only {len(pool)} coverage-qualified candidates for requested {count}')
    chosen=[];reasons={};theme_counts={}
    def add(char,reason):
        chosen.append(char);theme=pool.loc[char,'theme'];theme_counts[theme]=theme_counts.get(theme,0)+1;reasons[char]=reason
    def best(chars):
        values={}
        for char in chars:
            redundant=float(corr.loc[char,chosen].abs().max()) if chosen else 0.
            if not np.isfinite(redundant):redundant=0.
            # Small documented preference for published interpretable cluster membership.
            values[char]=float(pool.loc[char,score_col])+.02*bool(pool.loc[char,'published_jkp_153'])-redundancy_penalty*redundant
        return max(values,key=lambda c:(values[c],c))
    for theme in sorted(pool.theme.unique()):
        chars=pool.index[pool.theme.eq(theme)].tolist()
        if published_anchor:
            anchors=[c for c in chars if pool.loc[c,'published_jkp_153']]
            if not anchors:raise ValueError(f'No eligible published representative for theme {theme}')
            if len(chosen)>=count:raise ValueError('Theme anchors make requested subset infeasible')
            add(best(anchors),'published JKP representative for economic theme')
        for _ in range(max(0,min(min_theme,len(chars))-theme_counts.get(theme,0))):
            available=[c for c in chars if c not in chosen]
            if not available or len(chosen)>=count:break
            add(best(available),'minimum economic-theme representation')
    while len(chosen)<count:
        available=[c for c in pool.index if c not in chosen and (max_theme is None or theme_counts.get(pool.loc[c,'theme'],0)<max_theme)]
        if not available:raise ValueError('Theme cap makes requested subset infeasible; no silent relaxation')
        add(best(available),'balanced predictive score with redundancy penalty')
    selected=pool.loc[chosen].copy();selected['selection_order']=np.arange(1,len(selected)+1);selected['selection_reason']=[reasons[c] for c in chosen]
    return selected.reset_index(drop=True)


### `scripts/update_us_selection.py`

Install the US high-cap association shortlist and archive old-feature fits. This is an offline input migration after selection/acquisition. It never calls the return forecaster, covariance estimator or either portfolio ML fit.

**Role:** Pipeline implementation or configuration. This cell writes the file without running its main function.

**Input:** the embedded source below. **Output:** `scripts/update_us_selection.py`. Expand the code cell to read or edit it.


In [ ]:
%%writefile scripts/update_us_selection.py
"""Install the US high-cap association shortlist and archive old-feature fits.

This is an offline input migration after selection/acquisition. It never calls
the return forecaster, covariance estimator or either portfolio ML fit.
"""
from __future__ import annotations
import ast
import datetime as dt
import hashlib
import json
from pathlib import Path
import shutil

import numpy as np
import pandas as pd
import yaml

ROOT=Path(__file__).resolve().parents[1]
PROJECT=ROOT.parent
CHAR=PROJECT/'Characteristics_tryout'


def digest(path):
    h=hashlib.sha256()
    with Path(path).open('rb') as stream:
        for block in iter(lambda:stream.read(4*1024*1024),b''):h.update(block)
    return h.hexdigest()


def json_file(path,value):
    path.parent.mkdir(parents=True,exist_ok=True)
    path.write_text(json.dumps(value,indent=2,default=str)+'\n',encoding='utf-8')


def checked_move(source,destination):
    source=source.resolve();destination=destination.resolve()
    if not source.is_relative_to(ROOT) or not destination.is_relative_to(ROOT):raise ValueError('Archive target outside Kati')
    if not source.exists():return
    if destination.exists():raise ValueError(f'Archive destination already exists: {destination}')
    destination.parent.mkdir(parents=True,exist_ok=True)
    shutil.move('\\\\?\\'+str(source),'\\\\?\\'+str(destination))


def refresh_shared_manifest():
    """Retain existing hashes and register the shared inputs used by new fits."""
    path=ROOT/'data/SNAPSHOT_MANIFEST.json'
    manifest=json.loads(path.read_text())
    for item in manifest['files']:
        if digest(ROOT/item['path'])!=item['sha256']:raise AssertionError('Existing frozen input changed')
    present={item['path'] for item in manifest['files']}
    for folder in ('inputs','reference','characteristics'):
        for p in (ROOT/'data'/folder).rglob('*'):
            if not p.is_file():continue
            relative=str(p.relative_to(ROOT)).replace('\\','/')
            if relative not in present:
                manifest['files'].append(dict(path=relative,sha256=digest(p),bytes=p.stat().st_size,role='Shared source input or self-contained US association sample'))
    json_file(path,manifest)
    return len(manifest['files'])


def portable_analysis():
    """Keep all analysis source and measurement data local to Kati."""
    origin=CHAR/'characteristics'
    source=(origin/'us_high_cap_selection.py').read_text(encoding='utf-8')
    source=source.replace('from .paths import DATA, OUTPUT',
        "ROOT=Path(__file__).resolve().parents[1]\nDATA=ROOT/'data/characteristics/us_high_cap_inputs'\nOUTPUT=ROOT/'outputs'")
    source=source.replace('from .selection import assign_themes, diverse_pick','from selection_rules import assign_themes, diverse_pick')
    source=source.replace('from .statistics import panel_metrics, summarize, pairwise_matrices, paired_columns',
        'from association_statistics import panel_metrics, summarize, pairwise_matrices, paired_columns')
    (ROOT/'scripts/us_high_cap_selection.py').write_text(source,encoding='utf-8')
    shutil.copy2(origin/'statistics.py',ROOT/'scripts/association_statistics.py')
    original=(origin/'selection.py').read_text(encoding='utf-8');tree=ast.parse(original)
    funcs=[ast.get_source_segment(original,node) for node in tree.body if isinstance(node,ast.FunctionDef) and node.name in ('assign_themes','diverse_pick')]
    (ROOT/'scripts/selection_rules.py').write_text('"""Frozen transparent group/quota/redundancy rules; no ML fit."""\nimport numpy as np\nimport pandas as pd\n\n'+'\n\n'.join(funcs)+'\n',encoding='utf-8')
    shutil.copy2(CHAR/'data/us_high_cap_selection/us_high_cap_sample.parquet',ROOT/'data/characteristics/us_high_cap_inputs/us_high_cap_sample.parquet')
    shutil.copy2(CHAR/'data/us_high_cap_selection/candidate_registry.csv',ROOT/'data/characteristics/us_high_cap_inputs/candidate_registry.csv')


def update():
    result=CHAR/'outputs/us_high_cap_selection'
    selected=pd.read_csv(result/'recommended_50.csv')
    names=selected.characteristic.tolist()
    if len(names)!=len(set(names)) or len(names)!=50:raise ValueError('Expected exactly 50 selected characteristics')
    decision=json.loads((result/'selection_manifest.json').read_text())
    if decision['other_markets_used'] or decision['models_fitted'] or decision['selection_return_end']!='2014-12-31':raise ValueError('Selection must be association-only, US high-cap, 2005–2014')
    current=ROOT/'data/characteristics/recommended_50.json'
    if (ROOT/'data/.us_high_cap_selection_build').exists() and not (ROOT/'data/SNAPSHOT_MANIFEST.json').exists():
        return resume_staged_install()
    if current.exists() and json.loads(current.read_text())==names:
        print('Current US high-cap 50 inputs already installed.');return 0
    missing=[x for x in names if x not in json.loads((CHAR/'data/download_200.json').read_text())]
    supplemental=CHAR/'data/us_high_cap_selection/supplemental'
    verified_supplemental=CHAR/'data/us_high_cap_selection/jkp_verified_supplemental'
    def extra_features(region,lag=False):
        remaining=set(missing); merged=None
        import pyarrow.parquet as pq
        for folder in (supplemental,verified_supplemental):
            paths=[folder/f'lag_{region}.parquet'] if lag else sorted(folder.glob(f'{region}_*.parquet'))
            paths=[p for p in paths if p.exists()]
            if not paths:continue
            cols=[x for x in missing if x in remaining and x in pq.read_schema(paths[0]).names]
            if not cols:continue
            values=pd.concat([pd.read_parquet(p,columns=['id','eom',*cols]) for p in paths],ignore_index=True)
            merged=values if merged is None else merged.merge(values,on=['id','eom'],how='outer',validate='one_to_one')
            remaining-=set(cols)
        if remaining:raise FileNotFoundError(f'Missing selected JKP columns: {sorted(remaining)}')
        return merged
    if missing and not (supplemental/'manifest.json').exists():raise FileNotFoundError('Acquire the missing new-characteristic columns before updating Kati')
    stage=ROOT/'data/.us_high_cap_selection_build'
    if stage.exists():raise ValueError('A prior staging folder exists; inspect it before another migration')
    stage.mkdir()
    files=[];audits=[]
    def register(path,role):
        files.append(dict(path='data/'+str(path.relative_to(stage)).replace('\\','/'),sha256=digest(path),bytes=path.stat().st_size,role=role))
    def save(frame,path,role):
        path.parent.mkdir(parents=True,exist_ok=True);frame.to_parquet(path,index=False,compression='zstd');register(path,role)
    chars=stage/'characteristics';chars.mkdir()
    baseline=ROOT/'data/characteristics/implementation_baseline.json'
    if baseline.exists():shutil.copy2(baseline,chars/baseline.name)
    shutil.copy2(result/'recommended_50.json',chars/'recommended_50.json');register(chars/'recommended_50.json','US high-cap-only 2005–2014 association shortlist')
    shutil.copy2(result/'recommended_50.csv',chars/'selected_feature_metadata.csv');register(chars/'selected_feature_metadata.csv','Selection reasons, group status and association diagnostics')
    directions=selected[['characteristic','theme','theme_assignment_method','published_jkp_153','direction','training_direction']].copy()
    directions['model_theme']=directions.theme.str.lower().str.replace(' ','_',regex=False).str.replace('-','_',regex=False)
    directions['model_theme_direction']=np.where(directions.published_jkp_153,directions.direction,directions.training_direction)
    directions['model_direction_source']=np.where(directions.published_jkp_153,'published JKP direction','2005–2014 US high-cap empirical sign for provisional group')
    if not directions.model_theme_direction.isin([-1,1]).all():raise ValueError('Invalid theme direction')
    directions.to_csv(chars/'theme_directions.csv',index=False);register(chars/'theme_directions.csv','Published signs or US high-cap empirical signs; raw ML ranks unflipped')
    for region,markets in [('US',('us_large','us_mid')),('EU',('eu_large',))]:
        print('Build new input snapshot:',region,flush=True)
        old_source=ROOT/f'data/source/screened_{region}.parquet'
        basecols=['id','eom','sic','size_grp','me','ret_exc_lead1m','hist_months','dolvol_126d']
        base=pd.read_parquet(old_source,columns=basecols)
        cached=[x for x in names if x not in missing]
        raw=pd.concat([pd.read_parquet(p,columns=['id','eom',*cached]) for p in sorted((CHAR/'data/panels').glob(f'{region}_*.parquet'))],ignore_index=True)
        if missing:
            extra=extra_features(region)
            raw=raw.merge(extra,on=['id','eom'],how='left',validate='one_to_one',indicator=True)
            if not raw._merge.eq('both').all():raise ValueError('Missing supplemental keys')
            raw=raw.drop(columns='_merge')
        raw=raw[['id','eom',*names]]
        if raw.duplicated(['id','eom']).any():raise ValueError('Duplicate raw characteristic keys')
        raw[names]=raw[names].apply(pd.to_numeric,errors='raise').astype(float).replace([np.inf,-np.inf],np.nan)
        raw=base[['id','eom']].merge(raw,on=['id','eom'],validate='one_to_one',how='left',indicator=True)
        if not raw._merge.eq('both').all():raise ValueError('Original broad regional keys not covered')
        raw=raw.drop(columns='_merge')
        if 'dolvol_126d' in names:
            np.testing.assert_allclose(base.dolvol_126d,raw.dolvol_126d,equal_nan=True,rtol=1e-12,atol=1e-10)
        save(raw,chars/f'raw50_{region}.parquet','Actual full-history selected50 observations; only US high-cap evidence selected names')
        frame=base.drop(columns=[x for x in names if x in base]).merge(raw,on=['id','eom'],validate='one_to_one')
        save(frame,stage/f'source/screened_{region}.parquet','Original infrastructure with newly selected raw inputs')
        ranks=frame.groupby('eom')[names].rank(method='average',pct=True).fillna(.5)
        rankframe=pd.concat([frame[['id','eom']],ranks.add_suffix('_rank')],axis=1)
        for market in markets:
            member=pd.read_parquet(ROOT/f'data/source/panel_{market}.parquet',columns=['id','eom','form_year','mcap_rank','status'])
            rebuilt=member.merge(rankframe,on=['id','eom'],how='left',validate='one_to_one',indicator=True)
            if not rebuilt._merge.eq('both').all():raise ValueError('Original member keys absent')
            save(rebuilt.drop(columns='_merge'),stage/f'source/panel_{market}.parquet','Unchanged membership and independent new-feature ranks')
        old_lag=pd.read_parquet(ROOT/f'data/inputs/lag_history_{region}.parquet',columns=['id','eom','g','growth_imputed_neutral'])
        lag=pd.read_parquet(CHAR/f'data/lag_history/{region}.parquet',columns=['id','eom',*cached])
        if missing:
            extra=extra_features(region,lag=True)
            lag=lag.merge(extra,on=['id','eom'],how='left',validate='one_to_one')
        lag=old_lag.merge(lag,on=['id','eom'],how='left',validate='one_to_one',indicator=True)
        if not lag._merge.eq('both').all():raise ValueError('Actual lag keys absent from characteristic source')
        lag=lag.drop(columns='_merge')
        save(lag[['id','eom',*names]],chars/f'lag_raw50_{region}.parquet','Actual selected-feature lag observations')
        normalized=lag[['id','eom','g','growth_imputed_neutral']].copy()
        for name in names:normalized[name]=.5
        group_indices=frame.groupby('eom').groups
        for date,g in lag.groupby('eom'):
            reference=frame.loc[group_indices[date],names]
            for name in names:
                ref=np.sort(reference[name].dropna().to_numpy(float));values=g[name].to_numpy(float)
                valid=np.isfinite(values);a=np.full(len(g),.5)
                if len(ref):a[valid]=(np.searchsorted(ref,values[valid],side='left')+np.searchsorted(ref,values[valid],side='right')+2)/(2*(len(ref)+1))
                normalized.loc[g.index,name]=a
        normalized['ret_ld1']=np.nan;normalized['lambda']=np.nan;normalized['lag_history_only']=True
        save(normalized,stage/f'inputs/lag_history_{region}.parquet','New signal ranks, unchanged existing lag growth infrastructure')
        audits.append(dict(region=region,stock_months=len(frame),exact_broad_keys_preserved=True,absent_source_rows=0,
            formation_start=str(frame.eom.min().date()),formation_end=str(frame.eom.max().date())))
        del frame,raw,ranks,rankframe,base,lag,normalized
    stamp=dt.datetime.now().strftime('%Y%m%d_%H%M%S')+'_previous_50'
    archive_data=ROOT/'data/archives'/stamp;archive_outputs=ROOT/'outputs/archive'/stamp
    archive_data.mkdir(parents=True);archive_outputs.mkdir(parents=True)
    # The original feature set, coefficients, fitted caches and results move
    # together. They are never relabelled as fits for the new selected 50.
    shutil.copytree(ROOT/'configs',archive_data/'configs')
    shutil.copy2(ROOT/'settings.json',archive_data/'settings.json')
    if (ROOT/'Kati_tryout.ipynb').exists():shutil.copy2(ROOT/'Kati_tryout.ipynb',archive_data/'Kati_tryout.ipynb')
    shutil.copytree(ROOT/'scripts',archive_data/'scripts',ignore=shutil.ignore_patterns('__pycache__'))
    fitted=[p for p in (ROOT/'data/processed').rglob('*') if p.is_file() and p.name in ('models.npz','aim_coefs.parquet','chosen.csv')]
    fitted_hashes=[dict(path=str(p.relative_to(ROOT)),sha256=digest(p)) for p in fitted]
    output_hashes=[dict(path=str(p.relative_to(ROOT/'outputs')),sha256=digest(p)) for p in (ROOT/'outputs').rglob('*')
        if p.is_file() and 'archive' not in p.relative_to(ROOT/'outputs').parts and p.name in ('pf.parquet','weights.parquet','aim_coefs.parquet','chosen.csv')]
    if fitted_hashes or output_hashes:raise ValueError('Existing fits require explicit review before replacing inputs')
    moved=[]
    for name in ('characteristics','source','processed','raw','figure1'):
        p=ROOT/'data'/name
        if p.exists():checked_move(p,archive_data/name);moved.append('data/'+name)
    if (ROOT/'data/SNAPSHOT_MANIFEST.json').exists():checked_move(ROOT/'data/SNAPSHOT_MANIFEST.json',archive_data/'SNAPSHOT_MANIFEST.json')
    for p in list((ROOT/'outputs').iterdir()):
        if p.name=='archive':continue
        checked_move(p,archive_outputs/p.name)
    # Return/current risk inputs stay shared; only feature-dependent lag ranks change.
    for region in ('US','EU'):
        for suffix in ('.parquet','.json'):
            p=ROOT/f'data/inputs/lag_history_{region}{suffix}'
            if p.exists():checked_move(p,archive_data/f'inputs/{p.name}')
    for name in ('characteristics','source'):
        checked_move(stage/name,ROOT/'data'/name)
    for p in (stage/'inputs').iterdir():checked_move(p,ROOT/'data/inputs'/p.name)
    for region in ('US','EU'):
        json_file(ROOT/f'data/inputs/lag_history_{region}.json',dict(selection_scope='US high-cap 2005–2014',models_fitted=False,
            rank_policy='Monthly regional percentile ranks; actual inserted lag observation; neutral0.5 for missing feature',
            growth_policy='Prior actual lag infrastructure preserved by exact key'))
    themes={t:g.characteristic.tolist() for t,g in directions.groupby('model_theme',sort=False)}
    for p in (ROOT/'configs').rglob('*.yaml'):
        cfg=yaml.safe_load(p.read_text());cfg.setdefault('features',{})['themes']=themes
        p.write_text(yaml.safe_dump(cfg,sort_keys=False),encoding='utf-8')
    settings=json.loads((ROOT/'settings.json').read_text())
    settings.update(features='US high-cap-only association-selected 50',selection_scope='us_large',selection_return_start='2005-01-31',
        selection_return_end='2014-12-31',selection_models_fitted=False,old_feature_archive=stamp)
    json_file(ROOT/'settings.json',settings)
    (ROOT/'data/characteristics/us_high_cap_inputs').mkdir()
    portable_analysis()
    shutil.copytree(result,ROOT/'outputs/us_high_cap_selection')
    for item in files:
        path=ROOT/item['path']
        if digest(path)!=item['sha256']:raise AssertionError('Staged input changed during installation')
    for item in fitted_hashes:
        relative=Path(item['path']).relative_to('data')
        if digest(archive_data/relative)!=item['sha256']:raise AssertionError('Archived fit coefficient hash changed')
    for item in output_hashes:
        if digest(archive_outputs/item['path'])!=item['sha256']:raise AssertionError('Archived portfolio or coefficient output changed')
    json_file(ROOT/'data/SNAPSHOT_MANIFEST.json',dict(status='PASS',pipeline='Kati_tryout',features=names,feature_count=50,
        selection_scope='us_large',selection_return_start='2005-01-31',selection_return_end='2014-12-31',models_refitted=False,
        runtime_requires_sibling_projects=False,files=files,regional_audits=audits,previous_feature_archive=stamp,
        theme_direction_policy=dict(published_direction_count=int(directions.published_jkp_153.sum()),training_direction_proxy_count=int((~directions.published_jkp_153).sum()))))
    migration=dict(status='PASS',new_selection=decision,archive_data=str(archive_data.relative_to(ROOT)),archive_results=str(archive_outputs.relative_to(ROOT)),
        preserved_fit_hashes=fitted_hashes,preserved_portfolio_output_hashes=output_hashes,models_fitted=False,old_fits_preserved=True,old_feature_data_archived=moved)
    json_file(ROOT/'outputs/validation/us_high_cap_selection_migration.json',migration)
    (archive_outputs/'README.md').write_text('# Previous characteristic-set results\n\nThese fits use the earlier three-market-selected 50. They were archived intact when switching to the US high-cap-only association shortlist. They are not results for the new 50.\n\nInputs, original configurations and code/notebook are under `'+str(archive_data.relative_to(ROOT))+'`.\n',encoding='utf-8')
    # The user explicitly requested no obsolete characteristic archives.
    for target in (archive_data,archive_outputs):
        resolved=target.resolve()
        if not resolved.is_relative_to(ROOT.resolve()):raise ValueError('Unsafe cleanup target')
        shutil.rmtree('\\\\?\\'+str(resolved))
    manifest_path=ROOT/'data/SNAPSHOT_MANIFEST.json'
    manifest=json.loads(manifest_path.read_text());manifest.pop('previous_feature_archive',None)
    manifest.update(jkp_only=True,old_feature_archives_removed=True)
    json_file(manifest_path,manifest)
    settings.pop('old_feature_archive',None);settings.update(jkp_only=True,features='Published JKP-only US high-cap association-selected 50')
    json_file(ROOT/'settings.json',settings)
    json_file(ROOT/'outputs/validation/us_high_cap_selection_migration.json',dict(status='PASS',jkp_only=True,models_fitted=False,old_feature_archives_removed=True,new_selection=decision))
    for target in (stage/'inputs',stage):
        if target.exists():target.rmdir()
    print('Installed verified JKP-only 50; superseded local input snapshot removed.',flush=True)
    return 0


def resume_staged_install():
    """Finish a fully built snapshot after an interrupted Windows directory move."""
    stage=ROOT/'data/.us_high_cap_selection_build'
    previous=sorted((ROOT/'data/archives').glob('*_previous_50'))
    prior=previous[-1] if previous else None
    result=CHAR/'outputs/us_high_cap_selection'
    selected=pd.read_csv(result/'recommended_50.csv');names=selected.characteristic.tolist()
    current_chars=stage/'characteristics' if (stage/'characteristics').exists() else ROOT/'data/characteristics'
    current_source=stage/'source' if (stage/'source').exists() else ROOT/'data/source'
    if json.loads((current_chars/'recommended_50.json').read_text())!=names:
        raise ValueError('Staged selection differs from current verified selection')
    audits=[]
    for region in ('US','EU'):
        new=pd.read_parquet(current_source/f'screened_{region}.parquet',columns=['id','eom','dolvol_126d'])
        if prior:
            old=pd.read_parquet(prior/f'source/screened_{region}.parquet',columns=['id','eom','dolvol_126d'])
            pd.testing.assert_frame_equal(old,new,check_dtype=False)
        else:
            expected=pd.read_parquet(CHAR/f'data/metadata_{region}.parquet',columns=['id','eom'])
            actual=pd.MultiIndex.from_frame(new[['id','eom']]);keys=pd.MultiIndex.from_frame(expected)
            if len(actual)!=len(keys) or not actual.difference(keys).empty:raise ValueError('Source keys differ from acquisition metadata')
        audits.append(dict(region=region,stock_months=len(new),exact_broad_keys_preserved=True,absent_source_rows=0,
                           formation_start=str(new.eom.min().date()),formation_end=str(new.eom.max().date())))
    for name in ('characteristics','source'):
        if (stage/name).exists():checked_move(stage/name,ROOT/'data'/name)
    for path in (stage/'inputs').glob('*.parquet'):
        destination=ROOT/'data/inputs'/path.name
        path.replace(destination)
    directions=pd.read_csv(ROOT/'data/characteristics/theme_directions.csv')
    themes={t:g.characteristic.tolist() for t,g in directions.groupby('model_theme',sort=False)}
    for path in (ROOT/'configs').rglob('*.yaml'):
        config=yaml.safe_load(path.read_text());config.setdefault('features',{})['themes']=themes
        path.write_text(yaml.safe_dump(config,sort_keys=False),encoding='utf-8')
    settings=json.loads((ROOT/'settings.json').read_text())
    settings.pop('old_feature_archive',None)
    settings.update(features='Published JKP-only US high-cap association-selected 50',jkp_only=True,
                    selection_scope='us_large',selection_return_start='2005-01-31',selection_return_end='2014-12-31',selection_models_fitted=False,old_feature_archives_removed=True)
    json_file(ROOT/'settings.json',settings)
    (ROOT/'data/characteristics/us_high_cap_inputs').mkdir(exist_ok=True)
    portable_analysis()
    # Remove only the superseded outputs named by this migration.
    for target in (() if prior is None else (prior,ROOT/'outputs/archive'/prior.name)):
        resolved=target.resolve()
        if not resolved.is_relative_to(ROOT.resolve()):raise ValueError('Cleanup outside Kati')
        if resolved.exists():shutil.rmtree('\\\\?\\'+str(resolved))
    for path in result.rglob('*'):
        if not path.is_file() or path.name=='selection_decision_path.pdf':continue
        destination=ROOT/'outputs/us_high_cap_selection'/path.relative_to(result)
        destination.parent.mkdir(parents=True,exist_ok=True)
        try:shutil.copy2(path,destination)
        except PermissionError:
            alternate=destination.with_name(destination.stem+'_jkp_only'+destination.suffix)
            shutil.copy2(path,alternate)
            print('Open legacy report retained; current result saved as:',alternate.name,flush=True)
    files=[]
    for folder in ('characteristics','source','inputs','reference'):
        for path in (ROOT/'data'/folder).rglob('*'):
            if path.is_file():files.append(dict(path=path.relative_to(ROOT).as_posix(),sha256=digest(path),bytes=path.stat().st_size,role='Current verified JKP-only input snapshot'))
    json_file(ROOT/'data/SNAPSHOT_MANIFEST.json',dict(status='PASS',pipeline='Kati_tryout',features=names,feature_count=50,
        selection_scope='us_large',selection_return_start='2005-01-31',selection_return_end='2014-12-31',models_refitted=False,
        runtime_requires_sibling_projects=False,files=files,regional_audits=audits,jkp_only=True,old_feature_archives_removed=True,
        theme_direction_policy=dict(published_direction_count=50,training_direction_proxy_count=0)))
    json_file(ROOT/'outputs/validation/us_high_cap_selection_migration.json',dict(status='PASS',jkp_only=True,models_fitted=False,
        exact_broad_keys_and_cost_liquidity_preserved=True,old_feature_archives_removed=True))
    for folder in (stage/'inputs',stage):
        if folder.exists():folder.rmdir()
    print('Installed JKP-only snapshot; exact keys and raw cost liquidity independently verified.',flush=True)
    return 0


if __name__=='__main__':raise SystemExit(update())


### `scripts/us_high_cap_selection.py`

US high-cap-only association selection; no forecast or portfolio model fit. Candidates must appear in both published JKP Cluster Labels and Factor Details and in the WRDS source schema. The measurement window is US high-cap 2005–2014.

**Role:** Pipeline implementation or configuration. This cell writes the file without running its main function.

**Input:** the embedded source below. **Output:** `scripts/us_high_cap_selection.py`. Expand the code cell to read or edit it.


In [ ]:
%%writefile scripts/us_high_cap_selection.py
"""US high-cap-only association selection; no forecast or portfolio model fit.

Candidates must appear in both published JKP Cluster Labels and Factor Details
and in the WRDS source schema. The measurement window is US high-cap 2005–2014.
"""
from __future__ import annotations

import argparse
import hashlib
import html
import json
from pathlib import Path

import numpy as np
import pandas as pd

ROOT=Path(__file__).resolve().parents[1]
DATA=ROOT/'data/characteristics/us_high_cap_inputs'
OUTPUT=ROOT/'outputs'
from selection_rules import assign_themes, diverse_pick
from association_statistics import panel_metrics, summarize, pairwise_matrices, paired_columns

REASONS = {"published JKP representative for economic theme": "A",
           "minimum economic-theme representation": "T",
           "balanced predictive score with redundancy penalty": "S"}


def sha(path):
    h = hashlib.sha256()
    with Path(path).open('rb') as stream:
        for block in iter(lambda: stream.read(4*1024*1024), b''): h.update(block)
    return h.hexdigest()


def verified_jkp_registry(registry, data=DATA):
    """Whitelist published JKP names; never infer membership from column position."""
    reference = Path(data)/'reference'
    if not reference.exists(): reference = Path(data).parents[1]/'reference'
    cluster_path=reference/'cluster_labels.parquet'
    details_path=reference/'factor_details.parquet'
    clusters=pd.read_parquet(cluster_path).rename(columns={'cluster':'published_theme'})
    details=pd.read_parquet(details_path).dropna(subset=['abr_jkp'])
    if clusters.characteristic.duplicated().any() or details.abr_jkp.duplicated().any():
        raise ValueError('Published JKP metadata has duplicate identifiers')
    allowed=set(clusters.characteristic)&set(details.abr_jkp)
    verified=registry.loc[registry.characteristic.isin(allowed)].copy()
    # Independently replace cached labels/signs with published reference values.
    verified=verified.drop(columns=['published_theme','direction','name_new'],errors='ignore')
    verified=verified.merge(clusters,on='characteristic',validate='one_to_one')
    verified=verified.merge(details[['abr_jkp','direction','name_new']],left_on='characteristic',right_on='abr_jkp',validate='one_to_one').drop(columns='abr_jkp')
    verified['published_jkp_153']=True
    verified['jkp_membership_verified']=True
    if not verified.direction.isin([-1,1]).all():raise ValueError('Missing published JKP direction')
    return verified.reindex(sorted(verified.columns),axis=1),[cluster_path,details_path]


def load_sample(mode='training', data=DATA):
    """Filter membership and return dates before computing any statistics."""
    data = Path(data)
    local = data/'us_high_cap_sample.parquet'
    if local.exists():
        if mode != 'training':raise ValueError('The self-contained sample contains 2005–2014 observations only')
        frame = pd.read_parquet(local)
        registry = pd.read_csv(data/'candidate_registry.csv')
        names = registry.characteristic.tolist()
        if not frame.member_us_large.all() or not frame.return_month.between('2005-01-31','2014-12-31').all():
            raise ValueError('Standalone sample has out-of-scope rows')
        return frame,names,registry,[local,data/'candidate_registry.csv']
    meta_path = data/'metadata_US.parquet'
    meta = pd.read_parquet(meta_path)
    end = '2014-12-31' if mode == 'training' else '2019-12-31'
    meta = meta.loc[meta.member_us_large & meta.return_month.between('2005-01-31', end)].copy()
    if meta.duplicated(['id','eom']).any(): raise ValueError('Duplicate US high-cap membership keys')
    registry = pd.read_csv(data/'candidate_registry.csv')
    if mode == 'training':
        names = registry.characteristic.tolist()
        paths = sorted((data/'pilot').glob('US_*.parquet'))
    else:
        names = json.loads((data/'download_200.json').read_text())
        paths = [p for p in sorted((data/'panels').glob('US_*.parquet')) if int(p.stem.split('_')[1]) <= 2019]
    parts = []
    sources = [meta_path, data/'candidate_registry.csv']
    for path in paths:
        frame = pd.read_parquet(path)
        frame = frame.merge(meta[['id','eom']], on=['id','eom'], how='inner', validate='one_to_one')
        parts.append(frame)
        sources.append(path)
    raw = pd.concat(parts, ignore_index=True)
    price = data/'pilot/prc_US.parquet'
    if 'prc' in names and price.exists():
        supplement = pd.read_parquet(price).rename(columns={'prc':'price_supplement'})
        raw = raw.merge(supplement, on=['id','eom'], how='left', validate='one_to_one')
        raw['prc'] = raw.get('prc', pd.Series(np.nan, index=raw.index)).combine_first(raw.price_supplement)
        raw = raw.drop(columns='price_supplement')
        sources.append(price)
    if raw.duplicated(['id','eom']).any() or not set(names).issubset(raw):
        raise ValueError('Incomplete or duplicate US characteristic observations')
    frame = meta.merge(raw[['id','eom',*names]], on=['id','eom'], how='left', validate='one_to_one', indicator=True)
    if not frame._merge.eq('both').all(): raise ValueError('US high-cap keys absent from raw source')
    frame = frame.drop(columns='_merge').sort_values(['eom','id']).reset_index(drop=True)
    frame[names] = frame[names].apply(pd.to_numeric, errors='raise').astype(float).replace([np.inf,-np.inf],np.nan)
    # Missing observed targets stay missing; do not load model-imputed returns.
    assert frame.member_us_large.all() and frame.return_month.max() <= pd.Timestamp(end)
    return frame, names, registry.loc[registry.characteristic.isin(names)].copy(), sources


def association_scores(monthly, mode='training'):
    if set(monthly.universe) != {'us_large'}:raise ValueError('Selection statistics must contain only US high-cap stocks')
    if not monthly.return_month.between('2005-01-31','2019-12-31').all():raise ValueError('Out-of-scope return dates')
    train = summarize(monthly.loc[monthly.return_month.dt.year.le(2014)]).set_index('characteristic')
    fit = train.copy()
    direction = np.where(train.mean_rank_ic.ge(0), 1, -1)
    fit['training_direction'] = direction
    fit['training_mean_rank_ic'] = train.mean_rank_ic
    fit['minimum_training_coverage'] = train.coverage
    fit['score_period_mean_rank_ic'] = train.mean_rank_ic
    fit['score_coverage'] = train.coverage
    fit['minimum_validation_coverage'] = np.nan
    fit['validation_oriented_ic'] = np.nan
    scoring = monthly.loc[monthly.return_month.dt.year.le(2014)].copy()
    if mode == 'validation':
        val = summarize(monthly.loc[monthly.return_month.dt.year.ge(2015)]).set_index('characteristic').reindex(train.index)
        fit['score_period_mean_rank_ic'] = val.mean_rank_ic
        fit['score_coverage'] = val.coverage
        fit['minimum_validation_coverage'] = val.coverage
        fit['validation_oriented_ic'] = val.mean_rank_ic*direction
        fit['eligible'] = (train.coverage.ge(.6)&val.coverage.ge(.6)&train.valid_ic_months.ge(60)&val.valid_ic_months.ge(30))
        scoring = monthly.loc[monthly.return_month.dt.year.ge(2015)].copy()
    else:
        fit['eligible'] = train.coverage.ge(.6)&train.valid_ic_months.ge(60)
    fit['oriented_score_ic'] = fit.score_period_mean_rank_ic*direction
    yearly = scoring.assign(year=scoring.return_month.dt.year).groupby(['characteristic','year']).rank_ic.mean()
    fit['directional_year_stability'] = [float((yearly.loc[c].dropna()*d>0).mean()) for c,d in zip(fit.index,direction)]
    fit['mean_persistence'] = scoring.groupby('characteristic').rank_persistence.mean().reindex(fit.index)
    # All components are observed associations; no regression, ML or portfolio fit.
    fit['final_score'] = (.55*fit.oriented_score_ic.rank(pct=True)+.20*fit.score_coverage.rank(pct=True)
        +.15*fit.directional_year_stability.rank(pct=True)+.10*fit.mean_persistence.fillna(0).rank(pct=True))
    return fit.reset_index()


def export_matrices(frame, names, folder):
    folder.mkdir(parents=True, exist_ok=True)
    raw = frame[names].to_numpy(float)
    ranked = frame.groupby('eom')[names].rank(method='average',pct=True)
    for scale, values in [('raw',raw),('rank',ranked.to_numpy(float))]:
        corr,cov,n = pairwise_matrices(values, min_pairs=120)
        for label, array in [('correlation',corr),('covariance',cov),('pair_counts',n)]:
            pd.DataFrame(array,index=names,columns=names).to_csv(folder/f'{scale}_{label}.csv')
    # A labelled return column makes feature-to-return associations explicit.
    target = frame.observed_ret_ld1.to_numpy(float)
    target_rank = frame.groupby('eom').observed_ret_ld1.rank(method='average',pct=True).to_numpy(float)
    for scale, values, y in [('raw',raw,target),('rank',ranked.to_numpy(float),target_rank)]:
        corr,cov,n = pairwise_matrices(np.column_stack([values,y]),min_pairs=120)
        labels = names+['NEXT_MONTH_EXCESS_RETURN']
        for label,array in [('correlation',corr),('covariance',cov),('pair_counts',n)]:
            pd.DataFrame(array,index=labels,columns=labels).to_csv(folder/f'{scale}_with_return_{label}.csv')


def stock_associations(frame, names):
    results = []
    for stock, g in frame.groupby('id',sort=True):
        x = g[names]
        y = g.observed_ret_ld1.to_numpy(float)
        n,pearson,cov = paired_columns(x.to_numpy(float),y,min_pairs=24)
        valid = np.isfinite(x.to_numpy(float))&np.isfinite(y)[:,None]
        xr = x.where(valid).rank(method='average',pct=True)
        yr = pd.DataFrame(np.broadcast_to(y[:,None],x.shape),index=x.index,columns=x.columns).where(valid).rank(method='average',pct=True)
        _,spearman,_ = paired_columns(xr.to_numpy(),yr.to_numpy(),min_pairs=24)
        results.append(pd.DataFrame(dict(id=stock,characteristic=names,n_observed_pairs=n,
            temporal_pearson=pearson,temporal_spearman=spearman,raw_return_covariance=cov)))
    return pd.concat(results,ignore_index=True)


def charts(selected, scores, out):
    import matplotlib
    matplotlib.use('Agg')
    import matplotlib.pyplot as plt
    plt.rcParams.update({'font.family':'DejaVu Sans','font.size':10})
    folder = out/'charts';folder.mkdir(exist_ok=True)
    # Actual counts and scores from this run, rather than an illustrative funnel.
    eligible = int(scores.eligible.sum())
    fig = plt.figure(figsize=(16, 12))
    grid = fig.add_gridspec(3, 2, height_ratios=[1.1, 1, 2.5])
    ax = fig.add_subplot(grid[0, :]); ax.axis('off')
    steps = [(f'{len(scores)} verified JKP candidates', 'Published metadata + WRDS\nUS high-cap; 2005–2014'),
             (f'{eligible} eligible candidates', 'Coverage ≥60%\n≥60 valid IC months'),
             (f'{int(scores.selected_200.sum())} shortlisted', 'Group coverage + score\nRedundancy penalty 0.10'),
             ('50 selected', '13 groups; 2–6 per group\nRedundancy penalty 0.20')]
    for i, (title, detail) in enumerate(steps):
        x = .125 + i*.25
        ax.text(x, .60, title+'\n\n'+detail, ha='center', va='center',
                fontsize=9, bbox=dict(boxstyle='round,pad=.8', facecolor='#edf4f7', edgecolor='#247b9b'),
                transform=ax.transAxes, wrap=True)
        if i<3: ax.annotate('', xy=(x+.16,.60), xytext=(x+.10,.60),
                            xycoords='axes fraction', arrowprops=dict(arrowstyle='->',color='#247b9b'))
    ax.set_title('How the 50 characteristics were chosen', loc='left', fontsize=21, weight='bold', pad=25)
    ax = fig.add_subplot(grid[1, 0])
    ax.barh(['Return association (absolute mean monthly rank IC)', 'Observed data coverage',
             'Yearly association-sign stability', 'Rank persistence'], [55,20,15,10], color='#247b9b')
    ax.invert_yaxis(); ax.set_xlim(0,65); ax.set_xlabel('Fixed score weight (%)')
    for i,v in enumerate([55,20,15,10]): ax.text(v+1,i,str(v)+'%',va='center')
    ax = fig.add_subplot(grid[1, 1]); ax.axis('off')
    ax.text(0,1,'Selection rules',weight='bold',fontsize=14,va='top')
    ax.text(0,.80,'Score uses percentile ranks of the four measures.\n'
            'Greedy priority = score + 0.02 for published label\n'
            '− penalty × largest absolute correlation with chosen signals.\n\n'
            'A: 13 published anchors (one per group)\n'
            'T: 13 additions to reach two members per group\n'
            'S: 24 further choices by score and redundancy',va='top',fontsize=11,linespacing=1.6)
    # Show all 50, their measured score, group and actual inclusion reason.
    palette={'A':'#247b9b','T':'#8c79a2','S':'#5c956d'}
    for column in range(2):
        ax=fig.add_subplot(grid[2,column])
        rows=selected.iloc[column*25:(column+1)*25]
        ax.barh(range(len(rows)),rows.final_score,color=[palette[c] for c in rows.reason_code])
        ax.set_yticks(range(len(rows)),[f'{r.characteristic} [{r.reason_code}] · {r.theme}' for r in rows.itertuples()],fontsize=8)
        ax.invert_yaxis();ax.set_xlim(0,1);ax.set_xlabel('Association score before redundancy adjustment')
        for i,v in enumerate(rows.final_score):ax.text(v+.01,i,f'{v:.3f}',va='center',fontsize=7)
    fig.text(.02,.012,'No ML fit. A/T/S explains inclusion, not model weights. Group quotas and correlation penalties mean the final 50 are not simply the 50 highest individual correlations.',fontsize=10)
    fig.tight_layout(rect=(0,.04,1,.98),h_pad=2.5,w_pad=3)
    for ext in ('png','svg','pdf'):
        name='selection_decision_path_jkp_only.pdf' if ext=='pdf' else f'selection_decision_path.{ext}'
        fig.savefig(folder/name,dpi=180)
    plt.close(fig)
    ordered = selected.sort_values(['theme','final_score'],ascending=[True,False])
    names = ordered.characteristic.tolist()
    for kind, title in [('correlation','Correlation'),('covariance','Covariance')]:
        a = pd.read_csv(out/f'matrices/selected50/rank_{kind}.csv',index_col=0).loc[names,names]
        fig,ax = plt.subplots(figsize=(14,12))
        limit = 1 if kind=='correlation' else float(np.nanmax(abs(a.to_numpy())))
        plot = ax.imshow(a,cmap='RdBu_r',vmin=-limit,vmax=limit)
        ax.set_xticks(range(50), names,rotation=90,fontsize=7)
        ax.set_yticks(range(50), names,fontsize=7)
        ax.set_title(f'US high-cap: selected 50 — ranked-feature {kind}\nSelection sample only; pairwise observed values, no ML model',loc='left',pad=20,weight='bold')
        fig.colorbar(plot,ax=ax,shrink=.75,label=f'{title} of monthly within-US-high-cap percentile ranks')
        fig.tight_layout();fig.savefig(folder/f'selected50_rank_{kind}.png',dpi=180);fig.savefig(folder/f'selected50_rank_{kind}.svg');plt.close(fig)
    fig,axes = plt.subplots(1,2,figsize=(15,8),gridspec_kw={'width_ratios':[1.1,1]})
    groups = selected.groupby('theme').agg(total=('characteristic','size'),published=('published_jkp_153','sum')).sort_index()
    y = np.arange(len(groups))
    axes[0].barh(y,groups.published,color='#247b9b',label='Published labels')
    if (groups.total-groups.published).sum():
        axes[0].barh(y,groups.total-groups.published,left=groups.published,color='#df9d36',label='Provisional US 2005–2014 correlation labels')
    axes[0].set_yticks(y,groups.index);axes[0].invert_yaxis();axes[0].set_xlim(0,7)
    for i,n in enumerate(groups.total):axes[0].text(n+.08,i,str(n),va='center',weight='bold')
    axes[0].set_title('13 groups; 2–6 inputs per group',loc='left',weight='bold');axes[0].set_xlabel('Selected characteristics');axes[0].legend(loc='lower left',bbox_to_anchor=(0,-.2),fontsize=8)
    reasons = selected.reason_code.value_counts().reindex(['A','T','S'],fill_value=0)
    axes[1].barh(['A: published group anchor','T: minimum group quota','S: association score after redundancy'],reasons.values,color=['#247b9b','#8c79a2','#5c956d'])
    axes[1].invert_yaxis()
    for i,n in enumerate(reasons):axes[1].text(n+.3,i,str(n),va='center',weight='bold')
    axes[1].set_xlim(0,max(reasons)+5);axes[1].set_xlabel('Selected characteristics');axes[1].set_title('Why each characteristic was included',loc='left',weight='bold')
    fig.suptitle('Why these 50? US high-cap selection only',fontsize=19,weight='bold',x=.02,ha='left')
    fig.text(.02,.015,'Fixed score: 55% oriented monthly rank IC + 20% coverage + 15% yearly direction stability + 10% persistence.\nAll selected names, groups and directions are verified against published JKP metadata. A/T/S are inclusion reasons, not ML coefficients.',fontsize=10)
    fig.tight_layout(rect=(0,.1,1,.94));fig.savefig(folder/'selection_groups_and_reasons.png',dpi=180);fig.savefig(folder/'selection_groups_and_reasons.svg');plt.close(fig)
    fig,ax = plt.subplots(figsize=(14,15))
    colors = plt.get_cmap('tab20')(np.linspace(0,1,13));palette = dict(zip(sorted(ordered.theme.unique()),colors))
    ax.barh(range(50),ordered.score_period_mean_rank_ic.to_numpy()*100,color=[palette[t] for t in ordered.theme])
    labels = [f'{r.characteristic}  [{r.reason_code}]' for r in ordered.itertuples()]
    ax.set_yticks(range(50),labels,fontsize=9);ax.invert_yaxis();ax.axvline(0,color='#243644',lw=.8)
    ax.set_xlabel('Mean monthly Spearman correlation × 100 (IC +2 means correlation +0.02)')
    ax.set_title('US high-cap return association for each selected characteristic\nSigned association with next-month observed excess return; colours identify groups',loc='left',weight='bold',pad=20)
    ax.grid(axis='x',alpha=.15);fig.tight_layout();fig.savefig(folder/'selected50_return_association.png',dpi=180);fig.savefig(folder/'selected50_return_association.svg');plt.close(fig)


def report(selected, scores, monthly, frame, manifest, out):
    title = 'US high-cap-only characteristic selection'
    columns = ['characteristic','description','theme','published_jkp_153','reason_code','training_mean_rank_ic','score_period_mean_rank_ic','score_coverage','directional_year_stability','mean_persistence','final_score']
    payload = json.dumps(selected[columns].replace({np.nan:None}).to_dict('records'),allow_nan=False).replace('<','\\u003c')
    page = '''<!doctype html><html lang="en"><meta charset="utf-8"><title>US high-cap characteristic selection</title>
<style>body{max-width:1250px;margin:35px auto;padding:0 22px;font:15px/1.55 system-ui;color:#19384b}h1{font-size:32px}h2{margin-top:32px}.note{padding:16px;background:#eef4f7;border-radius:9px}.scroll{overflow:auto}table{border-collapse:collapse;font-size:13px;width:100%}th,td{padding:8px;border-bottom:1px solid #dbe4eb;text-align:right}th:first-child,td:first-child{text-align:left}th{background:#eef4f7}img{max-width:100%}select,input{font:inherit;padding:7px;margin:6px}a{color:#126083}summary{cursor:pointer;font-weight:bold}</style>
<h1>US high-cap-only: 50 characteristics</h1><p>Selection uses US high-cap membership only. US mid-cap and European stocks do not contribute to scores, groups, matrices or selection.</p>
<div class="note">SAMPLE<br>Only names appearing in both published JKP Cluster Labels and Factor Details are eligible. All selected inputs use published group labels and directions. These 50 follow the declared association/diversity rule; no regression, forecast or ML portfolio was fitted.</div>
<h2>How were these 50 selected?</h2><img src="charts/selection_decision_path.png" alt="Verified JKP candidates, eligibility, final 50 scores and inclusion reasons"><p><a href="charts/selection_decision_path_jkp_only.pdf">Download JKP-only selection chart as PDF</a></p>
<h2>Why these 50 and their groups?</h2><img src="charts/selection_groups_and_reasons.png" alt="Group coverage and selection reasons">
<p>A = published group anchor; T = minimum theme representation; S = score after redundancy penalty. Every group assignment is taken from published JKP metadata.</p>
<label>Group<select id="theme"><option value="">All groups</option></select></label><label>Reason<select id="reason"><option value="">All reasons</option><option>A</option><option>T</option><option>S</option></select></label><label>Find<input id="query" placeholder="Characteristic or description"></label><p id="count"></p><div class="scroll"><table id="features"></table></div>
<h2>Correlation and covariance matrices</h2><p>The chart below switches between ranked-feature correlation and covariance. Raw-value and return-augmented matrices are downloadable too. Raw covariance magnitudes depend on units and should not be interpreted as comparable predictive importance.</p><select id="matrix"><option value="correlation">Rank correlation</option><option value="covariance">Rank covariance</option></select><p><a id="matrixcsv" href="matrices/selected50/rank_correlation.csv">Download shown 50×50 matrix</a></p><img id="matriximage" src="charts/selected50_rank_correlation.png" alt="Selected feature matrix">
<details><summary>Each characteristic’s association with next-month return</summary><img src="charts/selected50_return_association.png" alt="Return correlations"></details>
<h2>Data and audit files</h2>LINKS
<h2>Methods, assumptions and exclusions</h2><p>METHODS</p>
<script>const DATA=PAYLOAD;const $=x=>document.getElementById(x);const esc=x=>String(x??'').replace(/[&<>"']/g,c=>({'&':'&amp;','<':'&lt;','>':'&gt;','"':'&quot;',"'":'&#39;'}[c]));const num=x=>Number(x).toFixed(4);$('theme').innerHTML+=[...new Set(DATA.map(x=>x.theme))].sort().map(x=>'<option>'+esc(x)+'</option>').join('');function draw(){let rows=DATA.filter(x=>(!$('theme').value||x.theme===$('theme').value)&&(!$('reason').value||x.reason_code===$('reason').value)&&(`${x.characteristic} ${x.description}`.toLowerCase().includes($('query').value.toLowerCase())));$('count').textContent=rows.length+' of 50 characteristics';$('features').innerHTML='<tr><th>Characteristic</th><th>Description</th><th>Group</th><th>Group label</th><th>Reason</th><th>Mean monthly rank IC</th><th>Coverage</th><th>Score</th></tr>'+rows.map(x=>`<tr><td>${esc(x.characteristic)}</td><td>${esc(x.description)}</td><td>${esc(x.theme)}</td><td>${x.published_jkp_153?'Published':'Provisional proxy'}</td><td>${x.reason_code}</td><td>${num(x.score_period_mean_rank_ic)}</td><td>${(100*x.score_coverage).toFixed(1)}%</td><td>${num(x.final_score)}</td></tr>`).join('');}['theme','reason','query'].forEach(id=>$(id).oninput=draw);$('matrix').onchange=()=>{const kind=$('matrix').value;$('matriximage').src='charts/selected50_rank_'+kind+'.png';$('matrixcsv').href='matrices/selected50/rank_'+kind+'.csv';};draw();</script></html>'''
    methods = ('Monthly cross-sectional Pearson/Spearman correlations compare contemporaneous characteristics with next-month observed excess returns. '
        'Selection score = 55% oriented rank-IC percentile + 20% coverage percentile + 15% yearly sign-stability percentile + 10% persistence percentile. '
        'Candidate membership requires a WRDS source column and exact matches in published JKP Cluster Labels and Factor Details. Published groups and directions are used. Empirical association orientation is measured on 2005–2014 US high-cap data. Eligibility requires at least 60% observed feature coverage and 60 valid monthly ICs. '
        'The greedy rule adds 0.02 for published membership and subtracts 0.20 times maximum absolute training rank-correlation with already selected inputs. Each group has 2–6 members and a published anchor. '
        'Matrices use pairwise-complete observations, at least 120 pairs, and sample covariance (n−1); pair counts are supplied. Raw NaNs and unavailable returns stay missing. '
        'Rank matrices pool within-month percentile ranks; their return column is a pooled diagnostic and differs from the equal-month average IC used for selection. '
        'Per-stock correlations are time-series associations, requiring 24 observed pairs, and are reported separately. No cumulative return target is used. '
        'Pairwise covariance matrices may not be positive semidefinite and are descriptive diagnostics, not fitted portfolio risk matrices. '
        'The dataset retains its supplied source vintage and original membership assumptions; these checks do not establish point-in-time or survivorship-free data. '
        'No outcomes from 2015 onward, other-market return outcomes, regression R², fitted forecasts or portfolio utility enter the new selection. Raw covariance is a diagnostic; return rank correlation and correlation redundancy drive selection.')
    links = ['recommended_50_jkp_only.csv','recommended_50.json','all_candidate_scores.csv','excluded_nonpublished_candidates.csv','candidate_200.csv','group_summary.csv',
             'feature_return_summary.csv','per_stock_feature_return_associations.parquet','per_stock_selected50_pearson.csv',
             'per_stock_selected50_spearman.csv','per_stock_selected50_covariance.csv','selection_manifest.json',
             'verification.json','METHODS_ASSUMPTIONS_AND_EXCLUSIONS.md']
    links += [str(p.relative_to(out)).replace('\\','/') for p in sorted((out/'matrices/selected50').glob('*.csv'))]
    page = page.replace('SAMPLE',html.escape(manifest['sample_description'])).replace('PAYLOAD',payload).replace('METHODS',html.escape(methods))
    page = page.replace('LINKS','<ul>'+''.join(f'<li><a href="{html.escape(p)}">{html.escape(p)}</a></li>' for p in links)+'</ul>')
    (out/'index.html').write_text(page,encoding='utf-8')
    (out/'METHODS_ASSUMPTIONS_AND_EXCLUSIONS.md').write_text('# US high-cap association selection\n\n'+manifest['sample_description']+'\n\n'+methods+'\n',encoding='utf-8')


def select_us_high_cap(mode='training', out=None):
    if mode!='training':raise ValueError('Current selection is restricted to the 2005–2014 measurement window; no later validation outcomes')
    out = Path(out) if out else OUTPUT/'us_high_cap_selection'
    out.mkdir(parents=True,exist_ok=True)
    frame,names,registry,sources = load_sample(mode)
    registry,reference_sources=verified_jkp_registry(registry)
    excluded=sorted(set(names)-set(registry.characteristic))
    exclusion_path=out/'excluded_nonpublished_candidates.csv'
    if not excluded and exclusion_path.exists():
        excluded=sorted(set(pd.read_csv(exclusion_path).characteristic)-set(registry.characteristic))
    names=registry.characteristic.tolist()
    sources+=reference_sources
    frame=frame.drop(columns=excluded,errors='ignore')
    pd.DataFrame({'characteristic':excluded,'decision':'Excluded: not in both published JKP references'}).to_csv(out/'excluded_nonpublished_candidates.csv',index=False)
    if mode == 'training':
        snapshot = DATA/'us_high_cap_selection' if not (DATA/'us_high_cap_sample.parquet').exists() else DATA
        snapshot.mkdir(parents=True,exist_ok=True)
        frame.to_parquet(snapshot/'us_high_cap_sample.parquet',index=False)
        registry.to_csv(snapshot/'candidate_registry.csv',index=False)
    print(f'US high-cap only: {len(frame):,} rows; {len(names)} candidates; mode={mode}',flush=True)
    monthly = panel_metrics(frame,names,'us_large')
    monthly.to_parquet(out/'monthly_feature_return_metrics.parquet',index=False)
    scores = association_scores(monthly,mode)
    training = frame.loc[frame.return_month.dt.year.le(2014)]
    ranks = training.groupby('eom')[names].rank(pct=True)
    correlation,_,_ = pairwise_matrices(ranks.to_numpy(float),min_pairs=120)
    corr = pd.DataFrame(correlation,index=names,columns=names)
    registry = assign_themes(registry,corr)
    scores = scores.merge(registry,on='characteristic',validate='one_to_one')
    cap = diverse_pick(scores,corr,min(200,int(scores.eligible.sum())),min_theme=2,score_col='final_score',redundancy_penalty=.10,published_anchor=True)
    subset = scores.loc[scores.characteristic.isin(cap.characteristic)]
    chosen = diverse_pick(subset,corr,50,min_theme=2,max_theme=6,score_col='final_score',redundancy_penalty=.20,published_anchor=True)
    chosen['reason_code'] = chosen.selection_reason.map(REASONS)
    chosen.to_csv(out/'recommended_50_jkp_only.csv',index=False)
    try:chosen.to_csv(out/'recommended_50.csv',index=False)
    except PermissionError:pass  # A user-open legacy CSV is never used as an active input.
    (out/'recommended_50.json').write_text(json.dumps(chosen.characteristic.tolist(),indent=2)+'\n',encoding='utf-8')
    scores['selected_200'] = scores.characteristic.isin(cap.characteristic)
    scores['selected_50'] = scores.characteristic.isin(chosen.characteristic)
    scores['decision'] = np.select([~scores.eligible,scores.selected_50,~scores.selected_200],['Fails coverage/valid-month gate','Selected 50','Outside US-only 200 cap'],default='Inside 200; not chosen under group/score/redundancy rule')
    scores.sort_values('final_score',ascending=False).to_csv(out/'all_candidate_scores.csv',index=False)
    cap.to_csv(out/'candidate_200.csv',index=False)
    group = chosen.groupby('theme').agg(features=('characteristic','size'),published_representatives=('published_jkp_153','sum'))
    group.to_csv(out/'group_summary.csv')
    summarize(monthly).to_csv(out/'feature_return_summary.csv',index=False)
    export_matrices(training,names,out/'matrices/all_candidates_training')
    export_matrices(training,chosen.characteristic.tolist(),out/'matrices/selected50')
    per_stock = stock_associations(frame,chosen.characteristic.tolist())
    per_stock.to_parquet(out/'per_stock_feature_return_associations.parquet',index=False)
    for metric,label in [('temporal_pearson','pearson'),('temporal_spearman','spearman'),('raw_return_covariance','covariance')]:
        per_stock.pivot(index='id',columns='characteristic',values=metric).to_csv(out/f'per_stock_selected50_{label}.csv')
    manifests = dict(scope='us_large only',mode=mode,candidates=len(names),eligible=int(scores.eligible.sum()),cap=len(cap),selected=50,
        sample_description=f'{len(frame):,} US high-cap held-stock months; {frame.id.nunique()} distinct stocks; measurement return months 2005–2014. No ML training.',
        selection_return_start=str(frame.return_month.min().date()),selection_return_end=str(frame.return_month.max().date()),
        training_only=mode=='training',other_markets_used=False,test_outcomes_used=False,models_fitted=False,regression_r2_used=False,
        selection_weights=dict(oriented_rank_ic=.55,coverage=.20,yearly_direction_stability=.15,rank_persistence=.10),
        reason_counts=chosen.reason_code.value_counts().to_dict(),published_group_count=int(chosen.published_jkp_153.sum()),
        proxy_group_count=int((~chosen.published_jkp_153).sum()),
        candidate_policy='Exact WRDS source names intersect published JKP Cluster Labels and Factor Details',
        jkp_only=True,published_groups_only=True,excluded_unverified_count=len(excluded),
        source_files=[dict(path=str(p.resolve()),sha256=sha(p)) for p in sources],
        caveat='Best under the declared association/diversity rule; no global optimality or joint-ML performance claim.')
    (out/'selection_manifest.json').write_text(json.dumps(manifests,indent=2)+'\n',encoding='utf-8')
    checks = dict(exactly_50_unique=len(chosen)==chosen.characteristic.nunique()==50,
        only_us_high_cap=bool(frame.member_us_large.all()),no_test_outcomes=bool(frame.return_month.max()<pd.Timestamp('2020-01-01')),
        all_eligible=bool(chosen.eligible.all()),thirteen_groups_with_2_to_6=bool(len(group)==13 and group.features.between(2,6).all()),
        published_anchor_each_group=bool(group.published_representatives.ge(1).all()),
        no_models_fitted=True,reproduced_order=diverse_pick(subset,corr,50,min_theme=2,max_theme=6,score_col='final_score',redundancy_penalty=.20,published_anchor=True).characteristic.tolist()==chosen.characteristic.tolist())
    checks['all_selected_verified_jkp']=bool(chosen.jkp_membership_verified.all() and chosen.published_jkp_153.all())
    for path in (out/'matrices').rglob('*.csv'):
        a=pd.read_csv(path,index_col=0).to_numpy(float)
        checks['symmetric_'+str(path.relative_to(out))]=bool(np.allclose(a,a.T,equal_nan=True,rtol=1e-10,atol=1e-12))
    if not all(checks.values()):raise AssertionError({k:v for k,v in checks.items() if not v})
    (out/'verification.json').write_text(json.dumps(dict(status='PASS',checks=checks),indent=2)+'\n',encoding='utf-8')
    charts(chosen,scores,out)
    report(chosen,scores,monthly,frame,manifests,out)
    print(json.dumps({k:v for k,v in manifests.items() if k!='source_files'},indent=2),flush=True)
    return out


def main(argv=None):
    parser=argparse.ArgumentParser(description=__doc__)
    parser.add_argument('--selection-mode',choices=['training','validation'],default='training')
    args=parser.parse_args(argv)
    select_us_high_cap(args.selection_mode)
    return 0


### `scripts/verify.py`

Verify local inputs, exact feature substitution and orchestration without fits. The checks hash the immutable source snapshot, inspect every prepared panel's keys/timing, and independently recompute ranks on three full cross-sections. The report distinguishes a verified build from completed model experiments.

**Role:** Pipeline implementation or configuration. This cell writes the file without running its main function.

**Input:** the embedded source below. **Output:** `scripts/verify.py`. Expand the code cell to read or edit it.


In [ ]:
%%writefile scripts/verify.py
"""Verify local inputs, exact feature substitution and orchestration without fits.

The checks hash the immutable source snapshot, inspect every prepared panel's
keys/timing, and independently recompute ranks on three full cross-sections.
The report distinguishes a verified build from completed model experiments.
"""
from __future__ import annotations

import argparse
import ast
import hashlib
import json
from pathlib import Path

import numpy as np
import pandas as pd
import pyarrow.parquet as pq

from workflow import ROOT, UNIVERSES, config_path, originals_unchanged
from jkmp.config import load_config
from jkmp.features import load_theme_directions, theme_scores
from prepare_inputs import bootstrap, digest
from run_all import plan
from script_groups import normalise_group, script_path


def verify():
    checks = []

    def check(name, passed, detail=""):
        checks.append(dict(check=name, status="PASS" if bool(passed) else "FAIL", detail=detail))

    manifest = bootstrap()  # Cached path verifies local hashes, without siblings.
    names = json.loads((ROOT / "data/characteristics/recommended_50.json").read_text())
    check("exactly_50_unique_inputs", len(names) == len(set(names)) == 50)
    forbidden = {"id", "eom", "ret_exc", "ret_exc_lead1m", "ret_ld1", "tr_ld1", "lambda", "g"}
    check("no_targets_or_identifiers_added_as_predictors", not set(names) & forbidden)
    check("original_keats_source_unchanged", originals_unchanged())
    if manifest.get('selection_scope')=='us_large':
        selection=json.loads((ROOT/'outputs/us_high_cap_selection/selection_manifest.json').read_text())
        check('association_selection_is_us_high_cap_2005_2014_only',selection['scope']=='us_large only' and
              selection['selection_return_start']=='2005-01-31' and selection['selection_return_end']=='2014-12-31' and not selection['other_markets_used'])
        check('no_ml_regression_or_test_outcomes_in_selection',not selection['models_fitted'] and
              not selection['regression_r2_used'] and not selection['test_outcomes_used'])
        check('active_names_match_us_association_report',json.loads((ROOT/'outputs/us_high_cap_selection/recommended_50.json').read_text())==names)
        clusters=pd.read_parquet(ROOT/'data/reference/cluster_labels.parquet')
        details=pd.read_parquet(ROOT/'data/reference/factor_details.parquet')
        check('all_50_in_both_published_jkp_references',set(names).issubset(set(clusters.characteristic)&set(details.abr_jkp.dropna())))
        check('jkp_only_policy_enforced',selection.get('jkp_only') and selection['published_group_count']==50 and selection['proxy_group_count']==0)
        baseline=json.loads((ROOT/'data/characteristics/implementation_baseline.json').read_text())
        core=list((ROOT/'scripts/jkmp/models').glob('*.py'))+list((ROOT/'scripts/jkmp/portfolio').glob('*.py'))
        old_core=[p for p in core if p.relative_to(ROOT).as_posix() in baseline['source_hashes']]
        # The return-model validation population is an authorised adaptation.
        # All other original coefficient/portfolio mathematics stays pinned.
        preserved_core=[p for p in old_core if p.relative_to(ROOT).as_posix()!='scripts/jkmp/models/return_model.py']
        check('other_ml_and_weight_learning_source_preserved',all(digest(p)==baseline['source_hashes'][p.relative_to(ROOT).as_posix()] for p in preserved_core))
        check('only_authorised_static_star_library_added',set(p.name for p in core if p not in old_core)=={'static_ml_star.py'})
        from jkmp.config import _read_yaml
        same=True
        for config in (ROOT/'configs').rglob('*.yaml'):
            if config.stem.startswith('iteration_'):continue
            previous=baseline['non_feature_configs'][config.relative_to(ROOT).as_posix()].copy()
            current=_read_yaml(config,seen=())
            previous.pop('features',None);current.pop('features',None)
            expected_scope='all' if config.name=='kati_base.yaml' else 'universe'
            check(f'{config.relative_to(ROOT)}:authorised_rff_validation_scope',
                  current['return_model'].get('validation_scope')==expected_scope)
            current['return_model'].pop('validation_scope',None)
            expected_static=dict(k_grid=[1.0,1/3,0.2,0.1,0.05],u_grid=[0.0,0.1,0.25,0.5,1.0],g_grid=[0.0,1.0,2.0,3.0])
            if config.parent.name=='us_large' and current.get('static_ml')==expected_static:
                # Authorised base-only grid expansion; iteration loading pins
                # its unused Static ML settings to the original shared defaults.
                current['static_ml']=previous['static_ml']
            # The base model mandate and plot labels changed; economic/model settings did not.
            for section,key in [('reporting','main_types'),('regimes','plot_types')]:
                previous.get(section,{}).pop(key,None);current.get(section,{}).pop(key,None)
            same=same and previous==json.loads(json.dumps(current,default=str))
        check('non_feature_economic_model_settings_preserved',same)
        check('old_characteristic_archives_removed',not any((ROOT/'data/archives').glob('*')) and not any((ROOT/'outputs/archive').glob('*')))
    check("snapshot_all_three_markets_and_training_panels", len(manifest["regional_audits"]) == 2 and
          all(x["exact_broad_keys_preserved"] and x["absent_source_rows"] == 0 for x in manifest["regional_audits"]))
    spec = pd.read_csv(ROOT / "data/characteristics/theme_directions.csv")
    selected_metadata=pd.read_csv(ROOT/'data/characteristics/selected_feature_metadata.csv')
    check("frozen_group_metadata_matches_selected_50", len(spec) == 50 and
          int(spec.published_jkp_153.sum()) == int(selected_metadata.published_jkp_153.sum()) and spec.model_theme.nunique() == 13)
    check("theme_sign_policy", spec.loc[spec.published_jkp_153, "model_theme_direction"].equals(
          spec.loc[spec.published_jkp_153, "direction"]) and
          np.array_equal(spec.loc[~spec.published_jkp_153, "model_theme_direction"], spec.loc[~spec.published_jkp_153, "training_direction"]))

    configs = [(u, "base") for u in UNIVERSES] + [("us_large", f"iteration_{i}") for i in range(1,6)]
    for market, group in configs:
        cfg = load_config(config_path(market, group))
        check(f"{market}/{group}:rff_tuning_scope", cfg.return_model.validation_scope == 'universe')
        check(f"{market}/{group}:ridge_penalties", len(cfg.ridge_grid.values)==101 and cfg.ridge_grid.values[0]==0.0)
        check(f"{market}/{group}:frozen_50_only", len(cfg.features.names) == 50 and set(cfg.features.names) == set(names))
        check(f"{market}/{group}:local_paths", all(p.resolve().is_relative_to(ROOT) for p in
              (cfg.paths.raw, cfg.paths.processed, cfg.paths.outputs)))
        directions = load_theme_directions(cfg.paths.raw, cfg.features.themes)
        check(f"{market}/{group}:all_covariance_themes_supported", len(directions) == 50 and directions.theme.nunique() == 13)

    dates = [pd.Timestamp(d) for d in ("2004-12-31", "2019-12-31", "2025-11-30")]
    for market in UNIVERSES:
        cfg = load_config(config_path(market))
        path = cfg.paths.processed / "panel.parquet"
        check(f"{market}:prepared_panel_exists", path.exists())
        if not path.exists():
            continue
        frame = pd.read_parquet(path, columns=["id", "eom", "in_universe", "ret_ld1", "ret_ld1_original", "ret_ld1_imputed_zero"])
        check(f"{market}:unique_formation_keys", not frame.duplicated(["id", "eom"]).any())
        check(f"{market}:declared_horizon", frame.eom.max() == pd.Timestamp("2025-11-30") and
              frame.eom.dt.is_month_end.all() and (frame.eom + pd.offsets.MonthEnd()).le(pd.Timestamp(cfg.dates.test_end)).all())
        assumed = frame.ret_ld1_imputed_zero
        check(f"{market}:explicit_zero_return_policy", frame.loc[assumed, "ret_ld1_original"].isna().all() and
              frame.loc[assumed, "ret_ld1"].eq(0).all() and frame.loc[assumed, "in_universe"].all(), f"{int(assumed.sum())} labelled assumptions")
        source = ROOT / f"data/source/screened_{'EU' if market=='eu_large' else 'US'}.parquet"
        source_keys = pd.read_parquet(source, columns=["id", "eom"])
        a = pd.MultiIndex.from_frame(frame[["id", "eom"]])
        b = pd.MultiIndex.from_frame(source_keys)
        check(f"{market}:broad_training_population_preserved", len(a) == len(b) and a.difference(b).empty and b.difference(a).empty)
        sample = pd.read_parquet(source, columns=["id", "eom", *names], filters=[("eom", "in", dates)])
        expected = sample[names].groupby(sample.eom).rank(method="average", pct=True).fillna(.5)
        expected.index = pd.MultiIndex.from_frame(sample[["id", "eom"]])
        actual = pd.read_parquet(path, columns=["id", "eom", *names], filters=[("eom", "in", dates)]).set_index(["id", "eom"])
        actual = actual.reindex(expected.index)[names]
        check(f"{market}:independent_three_month_rank_recalculation", np.allclose(expected, actual, atol=1e-12, rtol=0),
              f"{len(actual)} rows; all 50 ranks, missing raw observations become 0.5")
        ds = load_theme_directions(cfg.paths.raw, cfg.features.themes)
        themes = theme_scores(actual, ds)
        saved = pd.read_parquet(path, columns=["id", "eom", *themes.columns], filters=[("eom", "in", dates)]).set_index(["id", "eom"]).reindex(expected.index)
        check(f"{market}:independent_theme_recalculation", np.allclose(themes, saved[themes.columns], rtol=0, atol=1e-12))
        actual_liquidity=pd.read_parquet(path,columns=['id','eom','dolvol'],filters=[('eom','in',dates)]).set_index(['id','eom']).reindex(expected.index).dolvol
        raw_liquidity=pd.read_parquet(source,columns=['id','eom','dolvol_126d'],filters=[('eom','in',dates)]).set_index(['id','eom']).reindex(expected.index).dolvol_126d
        check(f"{market}:cost_liquidity_stays_raw_even_when_selected_as_signal",np.allclose(actual_liquidity,raw_liquidity,equal_nan=True))

    jobs = plan(UNIVERSES, "full", "all")
    available = set()
    for job in jobs:
        check(f"plan:{job.name}:dependencies_precede_job", set(job.dependencies).issubset(available))
        available.add(job.name)
        if job.stage in {"03", "04", "05", "06", "07", "10", "11", "12", "13", "14", "monitor","experiment"}:
            check(f"plan:{job.name}:script_available", script_path(job.stage, normalise_group(job.variant)).is_file())
    check("iterations_only_us_high_cap", all(job.universe == "us_large" for job in jobs if job.variant != "base"))
    check('five_sequential_experiments',len([j for j in jobs if j.stage=='experiment'])==5)
    check("static_star_only_us_high_cap_base",[j.universe for j in jobs if j.stage=='05']==['us_large'])
    cfg=load_config(config_path('us_large'))
    check('expanded_static_star_grid_has_100_candidates',len(cfg.static_ml.k_grid)*len(cfg.static_ml.u_grid)*len(cfg.static_ml.g_grid)==100)
    check("frontier_toggle_independent", any(j.stage == "figure1" for j in jobs) and
          not any(j.stage == "figure1" for j in plan(UNIVERSES, "full", "all", skip_figure1=True)))

    # Check the actual lag-key contract before covariance/ML fits are started.
    # The earliest covariance window is unavailable without fitting; checking
    # from 2014 onward covers all headline and validation portfolio decisions.
    for market, group in [(u, "base") for u in UNIVERSES] + [("us_large", "iteration_2")]:
        cfg = load_config(config_path(market, group))
        path = cfg.paths.processed / "panel.parquet"
        if not path.exists():
            continue
        keys = pd.read_parquet(path, columns=["id", "eom", "in_universe"])
        held = keys.loc[keys.in_universe & keys.eom.ge("2013-12-31"), ["id", "eom"]]
        needed = pd.concat([held.assign(eom=held.eom-pd.offsets.MonthEnd(i)) for i in range(cfg.universe.lookback_months+2)]).drop_duplicates()
        available = pd.MultiIndex.from_frame(keys[["id", "eom"]])
        region = "EU" if market == "eu_large" else "US"
        lag = pd.read_parquet(ROOT / f"data/inputs/lag_history_{region}.parquet", columns=["id", "eom"])
        available = available.union(pd.MultiIndex.from_frame(lag))
        absent = pd.MultiIndex.from_frame(needed).difference(available)
        check(f"{market}/{group}:actual_12_month_signal_history_keys", len(absent) == 0, f"{len(needed)} required distinct keys; {len(absent)} unavailable after exclusions")

    for path in list((ROOT / "scripts").rglob("*.py")) + list((ROOT / "tests").rglob("*.py")):
        ast.parse(path.read_text(encoding="utf-8"), filename=str(path))
    check("all_python_sources_parse", True)
    notebook = ROOT / "Kati_tryout.ipynb"
    if notebook.exists():
        book = json.loads(notebook.read_text(encoding="utf-8"))
        cells = book["cells"]
        check("every_code_cell_has_prior_markdown", all(i > 0 and cells[i-1]["cell_type"] == "markdown" for i, cell in enumerate(cells) if cell["cell_type"] == "code"))
        embedded = book["metadata"].get("kati_source_manifest", {})
        # The builder stores every source path/hash as provenance metadata.
        if isinstance(embedded, dict) and embedded:
            check("notebook_embedded_source_matches_files", all(hashlib.sha256(((ROOT / p).read_text(encoding="utf-8").rstrip()+"\n").encode("utf-8")).hexdigest() == h for p, h in embedded.items()))
            actual_paths = {p.relative_to(ROOT).as_posix() for p in (ROOT / "scripts").rglob("*.py")}
            actual_paths.update(p.relative_to(ROOT).as_posix() for p in (ROOT / "tests").rglob("*.py"))
            check("notebook_contains_all_python_source", actual_paths.issubset(embedded))
            for cell in cells:
                source = cell["source"]
                if cell["cell_type"] == "code" and source.startswith("%%writefile "):
                    header, body = source.split("\n", 1)
                    relative = header.removeprefix("%%writefile ").strip()
                    check(f"notebook_cell:{relative}", (ROOT / relative).read_text(encoding="utf-8").rstrip() == body.rstrip())
    out = ROOT / "outputs/verification"
    out.mkdir(parents=True, exist_ok=True)
    report = dict(status="FAIL" if any(c["status"] == "FAIL" for c in checks) else "PASS", checks=len(checks),
                  passed=sum(c["status"] == "PASS" for c in checks), models_refitted=False, model_backtest_complete=False,
                  data_stock_months=sum(x["stock_months"] for x in manifest["regional_audits"]),
                  details=checks)
    (out / "build_verification.json").write_text(json.dumps(report, indent=2)+"\n", encoding="utf-8")
    pd.DataFrame(checks).to_csv(out / "build_checks.csv", index=False)
    print(json.dumps({k: v for k, v in report.items() if k != "details"}, indent=2))
    return 0 if report["status"] == "PASS" else 2


def main(argv=None):
    argparse.ArgumentParser(description=__doc__).parse_args(argv)
    return verify()


if __name__ == "__main__":
    raise SystemExit(main())


### `scripts/verify_iterations.py`

Independent saved-result checks for the five-step sequence; no investment models fitted.

**Role:** Pipeline implementation or configuration. This cell writes the file without running its main function.

**Input:** the embedded source below. **Output:** `scripts/verify_iterations.py`. Expand the code cell to read or edit it.


In [ ]:
%%writefile scripts/verify_iterations.py
"""Independent saved-result checks for the five-step sequence; no investment models fitted."""
from __future__ import annotations
import json
from pathlib import Path
import numpy as np
import pandas as pd
from workflow import ROOT,config_path
from jkmp.config import load_config


def main():
    protocol=json.loads((ROOT/'configs/us_high_cap_iterations.json').read_text())
    cfg=load_config(config_path('us_large'))
    panel=pd.read_parquet(cfg.paths.processed/'panel.parquet',columns=['id','eom','ret_ld1','lambda'])
    wealth=pd.read_parquet(cfg.paths.processed/'wealth.parquet',columns=['eom','wealth'])
    base=ROOT/'outputs/base/us_large/portfolio_ml/rff'
    previous=pd.read_parquet(base/'pf.parquet').sort_values('eom_ret')
    accepted=[1];checks=[];sensitivity=[]
    def check(name,ok):
        checks.append(dict(check=name,status='PASS' if bool(ok) else 'FAIL'))
    for n in range(1,6):
        folder=ROOT/f'outputs/iteration_{n}/us_large'
        decision=json.loads((folder/'decision.json').read_text())
        pf=pd.read_parquet(folder/'pf.parquet').sort_values('eom_ret')
        w=pd.read_parquet(folder/'weights.parquet')
        state=decision['candidate_state']
        check(f'{n}:complete_identical_test_calendar',pd.DatetimeIndex(pf.eom_ret).equals(pd.date_range('2020-01-31','2025-12-31',freq='ME')))
        check(f'{n}:inherits_only_accepted',decision['inherited_iterations']==accepted)
        d=w.merge(panel,on=['id','eom'],how='left',validate='many_to_one').merge(wealth,on='eom',validate='many_to_one')
        check(f'{n}:all_held_targets_observed_or_original_explicit_zero',d.loc[~d.exit,'ret_ld1'].notna().all())
        # Iteration 3 can introduce members previously unheld: recover only its
        # explicitly saved zero-return assumptions, never invent a target here.
        if d.loc[~d.exit,'ret_ld1'].isna().any():raise ValueError('Accounting requires an explicit new-member missing-return audit')
        cost=.5*d.wealth*d['lambda']*state['cost_multiple']*(d.w-d.w_start)**2
        gross=d.w*d.ret_ld1.where(~d.exit,0.)
        recomputed=pd.DataFrame(dict(eom=d.eom,r=gross,tc=cost,turnover=(d.w-d.w_start).abs())).groupby('eom').sum()
        recomputed.index+=pd.offsets.MonthEnd(1)
        observed=pf.set_index('eom_ret')
        check(f'{n}:independent_stock_return_cost_turnover_sums',np.allclose(recomputed.loc[observed.index],observed[recomputed.columns],rtol=1e-10,atol=1e-12))
        # Literal additive path, separate from the shared reporting helper.
        total=peak=0.;worst=0.
        for value in (pf.r-pf.tc):
            total+=value;peak=max(peak,total);worst=min(worst,total-peak)
        reported=pd.read_csv(folder/'performance_summary.csv').set_index('strategy').loc['Candidate']
        check(f'{n}:additive_cumulative_and_drawdown',reported.path_convention=='additive'
              and np.isclose(reported.cumulative_net_excess,total,rtol=1e-12,atol=1e-12)
              and np.isclose(reported.max_drawdown_excess,worst,rtol=1e-12,atol=1e-12))
        if n==1:
            check('1:bitwise_base_returns',np.array_equal(pf[['r','tc']].to_numpy(),previous[['r','tc']].to_numpy()))
            check('1:p_is_not_applicable',decision['p_better'] is None)
        else:
            a=pf.r.to_numpy();ac=pf.tc.to_numpy();b=previous.r.to_numpy();bc=previous.tc.to_numpy()
            for block in (3,6,12):
                rng=np.random.default_rng(protocol['bootstrap_seed'])
                starts=rng.integers(0,len(a),size=(protocol['bootstrap_reps'],int(np.ceil(len(a)/block))))
                index=((starts[:,:,None]+np.arange(block))%len(a)).reshape(protocol['bootstrap_reps'],-1)[:,:len(a)]
                delta=np.sqrt(12)*((a[index]-ac[index]).mean(axis=1)/a[index].std(axis=1,ddof=1)
                                   -(b[index]-bc[index]).mean(axis=1)/b[index].std(axis=1,ddof=1))
                p=float(np.mean((delta>1e-12)+.5*np.isclose(delta,0,rtol=0,atol=1e-12)))
                sensitivity.append(dict(iteration=n,block_months=block,P_better=p,passes_80=p>=.8))
                if block==6:
                    if decision['p_better'] is None:check(f'{n}:identical_candidate',np.allclose(a-ac,b-bc,rtol=0,atol=1e-12))
                    else:check(f'{n}:literal_paired_bootstrap_probability',abs(p-decision['p_better'])<1e-12)
                    check(f'{n}:acceptance_rule',decision['accepted']==(decision['p_better'] is not None and p>=.8))
        if decision['accepted'] and n not in accepted:accepted.append(n)
        check(f'{n}:included_after_matches_decision',accepted==decision['included_after'])
        if decision['accepted']:previous=pf
        if state['benchmark_relative']:
            calibration=pd.read_csv(folder/'calibration.csv')
            check(f'{n}:calibration_uses_only_pretest_returns',pd.to_datetime(calibration.last_return).le(pd.Timestamp(cfg.dates.validation_end)).all())
        chosen=pd.read_csv(folder/'chosen.csv',parse_dates=['eom_ret']) if n>1 else None
        if chosen is not None:check(f'{n}:settings_chosen_before_forecast_year',(chosen.eom_ret.dt.year+1).eq(chosen.year).all())
    summary=json.loads((ROOT/'outputs/iteration_summary/final/summary.json').read_text())
    check('final:accepted_ancestry_matches_all_five_decisions',summary['included_iterations']==accepted)
    final=pd.read_parquet(ROOT/'outputs/iteration_summary/final/pf.parquet').sort_values('eom_ret')
    check('final:equals_last_accepted_not_last_tried',np.array_equal(final[['r','tc']],previous[['r','tc']]))
    result=pd.DataFrame(checks)
    result.to_csv(ROOT/'outputs/iteration_summary/independent_verification.csv',index=False)
    pd.DataFrame(sensitivity).to_csv(ROOT/'outputs/iteration_summary/bootstrap_block_sensitivity.csv',index=False)
    print(result.to_string(index=False))
    print('Accepted:',accepted,'Checks:',len(result),'Passed:',int(result.status.eq('PASS').sum()))
    return 0 if result.status.eq('PASS').all() else 2


if __name__=='__main__':raise SystemExit(main())


### `scripts/workflow.py`

Offline adapters for the independent Kati monthly panels. The original portfolio mathematics lives in this folder's private jkmp snapshot. These adapters change data loading, membership, timing and output paths only.

**Role:** Pipeline implementation or configuration. This cell writes the file without running its main function.

**Input:** the embedded source below. **Output:** `scripts/workflow.py`. Expand the code cell to read or edit it.


In [ ]:
%%writefile scripts/workflow.py
"""Offline adapters for the independent Kati monthly panels.

The original portfolio mathematics lives in this folder's private jkmp snapshot.
These adapters change data loading, membership, timing and output paths only.
"""
from __future__ import annotations

import hashlib
import json
from pathlib import Path
import shutil
import sys

import numpy as np
import pandas as pd
import pyarrow.parquet as pq

ROOT = Path(__file__).resolve().parents[1]
sys.path.insert(0, str(ROOT / "scripts"))
from jkmp import costs, features
from jkmp.config import load_config
from jkmp.data.prepare import add_timing_columns, ff12_industry, market_total_returns, risk_free_monthly
from jkmp.data.wrds_pull import RAW_FILES

MONTH = pd.offsets.MonthEnd(1)
UNIVERSES = ("us_large", "us_mid", "eu_large")
VARIANTS = ("base", *(f"iteration_{i}" for i in range(1,6)))


def settings():
    return json.loads((ROOT / "settings.json").read_text())


def source_dir():
    return (ROOT / settings()["source"]).resolve()


def config_path(universe, variant="base"):
    from script_groups import normalise_group
    if universe not in UNIVERSES:raise ValueError("Unknown market")
    group=normalise_group(variant)
    return ROOT/"configs"/universe/f"{group}.yaml"


def region(universe):
    return "EU" if universe == "eu_large" else "US"


def assert_new_tree(path):
    path = Path(path).resolve()
    if not path.is_relative_to(ROOT):
        raise ValueError(f"Refusing to write outside the new workflow: {path}")


def write_json(path, value):
    assert_new_tree(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    path.write_text(json.dumps(value, indent=2, default=str) + "\n", encoding="utf-8")


def recover_current_returns(rows):
    """R^excess_t is the prior row's t+1 return only for consecutive calendar months.

    Never use the current row's forward return to drift inherited weights.
    A screened-out month is a real gap, not a zero return or forward fill.
    """
    out = rows.sort_values(["id", "eom"]).reset_index(drop=True).copy()
    previous_date = out.groupby("id", sort=False)["eom"].shift()
    previous_return = out.groupby("id", sort=False)["ret_exc_lead1m"].shift()
    out["ret_exc"] = previous_return.where(previous_date + MONTH == out["eom"])
    return out


def load_monthly_supplement(universe):
    path = ROOT / f"data/inputs/monthly_returns_{region(universe)}.parquet"
    if not path.exists():
        return None
    provenance = json.loads(path.with_suffix(".json").read_text())
    if provenance.get("currency") != "USD" or provenance.get("return_type") != "excess":
        raise ValueError("Monthly supplements must declare USD excess returns")
    frame = numeric_ids(pd.read_parquet(path))
    needed = {"ret_exc", "ret_exc_lead1m"}
    if not needed.issubset(frame.columns):
        raise ValueError("Monthly supplement lacks required return columns")
    return frame


def fill_forward_gaps(rows, supplement):
    """Use exact source observations only; never invent a post-delisting return."""
    if supplement is None:
        return rows
    direct = supplement.set_index(["id", "eom"])["ret_exc_lead1m"]
    adjacent = supplement[["id", "eom", "ret_exc"]].copy()
    adjacent["eom"] = adjacent.eom - MONTH
    adjacent = adjacent.set_index(["id", "eom"])["ret_exc"]
    keys = pd.MultiIndex.from_frame(rows[["id", "eom"]])
    fill = pd.Series(direct.reindex(keys).to_numpy(dtype=float, na_value=np.nan), index=rows.index)
    fill = fill.fillna(pd.Series(adjacent.reindex(keys).to_numpy(dtype=float, na_value=np.nan), index=rows.index))
    out = rows.copy()
    out["ret_exc_lead1m"] = out.ret_exc_lead1m.fillna(fill)
    return out


def zero_missing_held_returns(panel, rf):
    """User-selected assumption: missing held-stock EXCESS returns are zero.

    Preserve the observed series and flag assumptions. Total returns then equal
    the next month's risk-free return; missing current returns are not changed.
    """
    out = panel.copy()
    out["ret_ld1_original"] = out["ret_ld1"]
    missing = out["in_universe"] & out["ret_ld1"].isna()
    out["ret_ld1_imputed_zero"] = missing
    next_rf = (out.loc[missing, "eom"] + MONTH).map(rf)
    if next_rf.isna().any():
        raise ValueError("Cannot apply zero excess-return assumption without next-month RF")
    out.loc[missing, "ret_ld1"] = 0.0
    out.loc[missing, "tr_ld1"] = next_rf.to_numpy()
    return out


def numeric_ids(frame):
    out = frame.copy()
    ids = pd.to_numeric(out["id"], errors="raise")
    if ids.isna().any() or not np.equal(ids, np.floor(ids)).all():
        raise ValueError("Expected non-missing, integral JKP security identifiers")
    out["id"] = ids.astype("int64")
    out["eom"] = pd.to_datetime(out["eom"])
    if not out["eom"].dt.is_month_end.all() or out.duplicated(["id", "eom"]).any():
        raise ValueError("Keats rows must be unique by security and calendar month-end")
    return out


def broaden_membership(screened, universe):
    """Apply Keats' annual buffered rule at 250 names, preserving the mid-cap centre.

    Large caps: ranks 1..250, entry <=220, retention <=300.
    Mid caps: ranks 376..625 (centre 500.5), entry 406..595, retention 326..675.
    Retained positions may exceed target, as in the saved Keats 80-stock universes.
    Missing a month removes membership until the next December.
    """
    middle = universe == "us_mid"
    lo, hi = (376, 625) if middle else (1, 250)
    entry_lo, entry_hi = (406, 595) if middle else (1, 220)
    keep_lo, keep_hi = (326, 675) if middle else (1, 300)
    dec = screened[screened.eom.dt.month.eq(12)].copy()
    # Keats uses method='first' on the original screened row order.
    dec["mcap_rank"] = dec.groupby("eom")["me"].rank(ascending=False, method="first")
    previous, memberships = set(), []
    for eom, group in dec.groupby("eom", sort=True):
        ranks = group.set_index("id")["mcap_rank"]
        retained = {i for i in previous if i in ranks.index and keep_lo <= ranks[i] <= keep_hi}
        candidates = ranks[ranks.between(entry_lo, entry_hi)].drop(list(retained), errors="ignore")
        if middle:
            candidates = candidates.iloc[np.argsort(abs(candidates.to_numpy() - (lo + hi) / 2), kind="stable")]
            candidates = candidates.iloc[:max(0, 250 - len(retained))]
        statuses = {i: "retained" for i in retained} | {i: "new" for i in candidates.index}
        fill = ranks.drop(list(statuses), errors="ignore")
        fill = fill.iloc[np.argsort(abs(fill.to_numpy() - (lo + hi) / 2), kind="stable")]
        statuses.update({i: "fill" for i in fill.index[:max(0, 250 - len(statuses))]})
        selected = group[group.id.isin(statuses)][["id", "mcap_rank"]].copy()
        selected["form_year"] = eom.year
        selected["status"] = selected.id.map(statuses)
        memberships.append(selected)
        previous = set(statuses)
    membership = pd.concat(memberships, ignore_index=True)
    rows = screened[["id", "eom"]].copy()
    rows["form_year"] = np.where(rows.eom.dt.month.eq(12), rows.eom.dt.year, rows.eom.dt.year - 1)
    rows = rows.merge(membership, on=["id", "form_year"], validate="many_to_one")
    rows = rows.sort_values(["id", "form_year", "eom"])
    prior = rows.groupby(["id", "form_year"])["eom"].shift()
    broken = prior.notna() & (prior + MONTH != rows.eom)
    rows = rows[~broken.groupby([rows.id, rows.form_year]).cummax()]
    return rows


def missing_inputs(universe):
    cfg = load_config(config_path(universe))
    daily = ROOT / settings()["daily_inputs"][region(universe)]
    meta = daily.with_suffix(".json")
    missing = []
    for path, purpose in ((daily, "Daily stock-level USD TOTAL returns: id,date,ret; id is the JKP security id"),
                          (meta, 'Daily provenance: currency=USD, return_type=total, identifier=jkp_id, source'),
                          (ROOT / settings()["reference_data"] / RAW_FILES["ff_daily"], "Daily USD risk-free rate")):
        if not path.exists():
            missing.append({"path": str(path), "purpose": purpose})
    for series in cfg.regimes.fred_series:
        candidate = cfg.paths.raw / f"fred_{series}.parquet"
        reference = ROOT / settings()["reference_data"] / candidate.name
        shared = ROOT / f"data/inputs/fred_{series}.parquet"
        if not candidate.exists() and not reference.exists() and not shared.exists():
            missing.append({"path": str(candidate), "purpose": f"Optional cached FRED {series} for stages 10 and 13"})
    return missing


def audit():
    report = {"downloads_allowed": False, "source": str(source_dir()), "files": {}, "universes": {}}
    for path in sorted(source_dir().glob("*.parquet")):
        parquet = pq.ParquetFile(path)
        dates = pd.read_parquet(path, columns=["eom"])["eom"]
        report["files"][path.name] = {"rows": parquet.metadata.num_rows, "columns": parquet.schema.names,
            "first_formation_month": str(dates.min().date()), "last_formation_month": str(dates.max().date()),
            "frequency": "monthly", "has_daily_stock_returns": False}
    for universe in UNIVERSES:
        cfg = load_config(config_path(universe))
        report["universes"][universe] = {"features": len(cfg.features.names),
            "missing_inputs": missing_inputs(universe), "test": [str(cfg.dates.test_start), str(cfg.dates.test_end)],
            "separate_holdout_available": False}
    write_json(ROOT / "outputs/input_audit.json", report)
    print(json.dumps({u: report["universes"][u] for u in UNIVERSES}, indent=2))
    return report


def prepare(universe, variant="base"):
    cfg = load_config(config_path(universe, variant))
    source = source_dir()
    fingerprint = {p.name: {"size": p.stat().st_size, "mtime_ns": p.stat().st_mtime_ns}
                   for p in (source / f"screened_{region(universe)}.parquet", source / f"panel_{universe}.parquet")}
    supplement_path = ROOT / f"data/inputs/monthly_returns_{region(universe)}.parquet"
    if supplement_path.exists():
        fingerprint[supplement_path.name] = {"size": supplement_path.stat().st_size,
                                              "mtime_ns": supplement_path.stat().st_mtime_ns}
    if (cfg.paths.processed / "preparation.json").exists():
        previous = json.loads((cfg.paths.processed / "preparation.json").read_text())
        current_config = json.loads(json.dumps(cfg.as_dict(), default=str))
        fields = ("dates", "features", "trading_costs")
        if previous.get("snapshot") != fingerprint or any(
            previous["config"][key] != current_config[key] for key in fields
        ) or any(previous["config"]["investor"][key] != current_config["investor"][key]
                 for key in ("wealth", "wealth_end_date")):
            raise ValueError("Prepared data is stale. Existing files retained; choose a new output folder before rebuilding.")
        if previous.get("missing_forward_return_policy", "stop") != settings().get("missing_forward_return_policy", "stop"):
            raise ValueError("Prepared return policy differs; retain existing files and prepare into a new output folder")
        print(f"Already prepared: {cfg.paths.processed} (existing files retained)")
        return
    assert_new_tree(cfg.paths.processed)
    assert_new_tree(cfg.paths.raw)
    chars = list(cfg.features.names)
    # Load the full screened region so training and cross-sectional ranks do not use only held names.
    use = ["id", "eom", "sic", "size_grp", "me", "ret_exc_lead1m", "hist_months", "dolvol_126d", *chars]
    use = list(dict.fromkeys(use))
    raw = numeric_ids(pd.read_parquet(source / f"screened_{region(universe)}.parquet", columns=use))
    supplement = load_monthly_supplement(universe)
    raw = fill_forward_gaps(raw, supplement)
    if variant in ("universe_250", "halflife"):
        members = broaden_membership(raw, universe)
    else:
        members = numeric_ids(pd.read_parquet(source / f"panel_{universe}.parquet",
            columns=["id", "eom", "form_year", "mcap_rank", "status", *[c + "_rank" for c in chars]]))

    cfg.paths.raw.mkdir(parents=True, exist_ok=True)
    reference = (ROOT / settings()["reference_data"]).resolve()
    # Tiny reference files are copied once; the large source panels stay in their original location.
    for name in (RAW_FILES["ff_daily"], RAW_FILES["factor_details"], RAW_FILES["cluster_labels"]):
        dest = cfg.paths.raw / name
        if not dest.exists() and (reference / name).exists():
            shutil.copy2(reference / name, dest)
    for series in cfg.regimes.fred_series:
        path = reference / f"fred_{series}.parquet"
        if not path.exists():
            path = ROOT / f"data/inputs/fred_{series}.parquet"
        if path.exists() and not (cfg.paths.raw / path.name).exists():
            shutil.copy2(path, cfg.paths.raw / path.name)
    ff = pd.read_parquet(reference / RAW_FILES["ff_monthly"])[["date", "rf"]]
    ff["date"] = pd.to_datetime(ff.date)
    if ff.date.duplicated().any():
        raise ValueError("Duplicate monthly RF dates")

    # A regional value-weighted market wealth path, built with formation-month market caps.
    # This avoids using the US market as the European wealth-growth benchmark.
    market_rows = raw[raw.ret_exc_lead1m.notna() & raw.me.gt(0)].copy()
    market_rows["weighted"] = market_rows.me * market_rows.ret_exc_lead1m
    monthly = market_rows.groupby("eom")[["weighted", "me"]].sum()
    market = (monthly.weighted / monthly.me).rename("mkt_rf").reset_index()
    market["date"] = market.eom + MONTH
    market = market[["date", "mkt_rf"]].merge(ff, on="date", how="left", validate="one_to_one")
    market = market[market.date.le(pd.Timestamp(cfg.dates.data_end))]
    if market.rf.isna().any():
        raise ValueError("Monthly RF missing for a regional market-return month")
    market.to_parquet(cfg.paths.raw / RAW_FILES["ff_monthly"], index=False)
    wealth = costs.wealth_path(cfg.investor.wealth, cfg.investor.wealth_end_date,
        market_total_returns(market, market.date.min(), market.date.max()))
    # RF at the very first formation month is needed in addition to the forward-return RF dates.
    timed = recover_current_returns(raw)
    if supplement is not None:
        known = supplement.set_index(["id", "eom"])["ret_exc"]
        keys = pd.MultiIndex.from_frame(timed[["id", "eom"]])
        timed["ret_exc"] = timed.ret_exc.fillna(pd.Series(known.reindex(keys).to_numpy(dtype=float, na_value=np.nan), index=timed.index))
    timed = add_timing_columns(timed, risk_free_monthly(ff), wealth)
    timed["dolvol"] = timed["dolvol_126d"].astype(float)
    if timed.dolvol.isna().any() or not timed.dolvol.gt(0).all():
        raise ValueError("Expected positive Keats USD daily dollar volume")
    timed["lambda"] = costs.price_impact_lambda(timed.dolvol, cfg.trading_costs.price_impact)
    ranked = raw[chars].groupby(raw.eom).rank(pct=True).fillna(0.5)
    # recover_current_returns sorted the rows; align by explicit keys, never by row position.
    ranked = pd.concat([raw[["id", "eom"]], ranked], axis=1)
    if variant not in ("universe_250", "halflife"):
        check = members.merge(ranked, on=["id", "eom"], validate="one_to_one")
        if len(check) != len(members):
            raise ValueError("Saved member rows not found in the screened region")
        for char in chars:
            if not np.allclose(check[char].to_numpy(dtype=float), check[char + "_rank"].to_numpy(dtype=float), atol=1e-12):
                raise ValueError(f"Rebuilt Kati member ranks disagree with regional ranks for {char}")
    timed = timed.drop(columns=chars).merge(ranked, on=["id", "eom"], validate="one_to_one")
    directions = features.load_theme_directions(cfg.paths.raw, cfg.features.themes)
    theme = features.theme_scores(timed[chars], directions)
    timed[theme.columns] = theme
    # Keats permits ff49/gics when SIC is missing. Do not drop saved members or invent a sector.
    timed["ff12"] = ff12_industry(timed.sic)
    timed.loc[timed.sic.isna(), "ff12"] = "Other"
    member_keys = members[["id", "eom", "form_year", "mcap_rank", "status"]].copy()
    timed = timed.merge(member_keys, on=["id", "eom"], how="left", validate="one_to_one")
    timed["in_universe"] = timed.form_year.notna()
    from jkmp.data.lag_history import exclude_unavailable
    timed, history_exclusions = exclude_unavailable(timed, universe, cfg.universe.lookback_months)
    timed["recon_eom"] = pd.to_datetime(timed.form_year.astype("Int64").astype("string") + "-12-31")
    if settings().get("missing_forward_return_policy") == "zero_excess":
        timed = zero_missing_held_returns(timed, risk_free_monthly(ff))
    timed["valid_data"] = timed.hist_months.ge(12) & timed.ret_ld1.notna() & timed.tr_ld0.notna()
    timed["g"] = costs.drift_factor(timed.tr_ld0, timed.mu_ld0)
    # The final formation month earns a 2026 return and cannot enter the declared 2020-25 test.
    timed = timed[(timed.eom + MONTH).le(pd.Timestamp(cfg.dates.test_end))].copy()
    for col in [*chars, *theme.columns, "me", "dolvol", "lambda", "ret_ld1", "ret_exc", "tr_ld0", "tr_ld1", "g"]:
        timed[col] = pd.to_numeric(timed[col], errors="raise").astype(float)
    if not np.isfinite(timed[chars].to_numpy()).all():
        raise ValueError("Nonfinite ranked ML input")
    # Top-500 covariance estimation pool belongs to the same region as the traded universe.
    top = raw.assign(rank=raw.groupby("eom")["me"].rank(ascending=False, method="first"))
    top = top[top["rank"].le(cfg.covariance.factor_universe_top_n)][["id", "eom"]]
    top.to_parquet(cfg.paths.raw / f"top{cfg.covariance.factor_universe_top_n}_by_month.parquet", index=False)
    cfg.paths.processed.mkdir(parents=True, exist_ok=True)
    timed = timed.sort_values(["eom", "id"], ignore_index=True)
    timed.to_parquet(cfg.paths.processed / "panel.parquet", index=False)
    history_exclusions.to_csv(cfg.paths.processed / "history_exclusions.csv", index=False)
    if "ret_ld1_imputed_zero" in timed:
        timed.loc[timed.ret_ld1_imputed_zero, ["id", "eom", "ret_ld1_original", "ret_ld1", "tr_ld1"]].to_csv(
            cfg.paths.processed / "zero_return_imputations.csv", index=False)
    wealth.to_parquet(cfg.paths.processed / "wealth.parquet", index=False)
    member_keys.to_parquet(cfg.paths.processed / "universe_membership.parquet", index=False)
    counts = timed[timed.in_universe].groupby("eom").size().rename("n_members")
    counts.to_csv(cfg.paths.processed / "universe_stats.csv")
    screen = pd.DataFrame([{"note": "Keats screens preserved; no additional selection on forward returns",
        "screened_rows": len(raw), "prepared_rows": len(timed), "invalid_current_return": int(timed.tr_ld0.isna().sum())}])
    screen.to_csv(cfg.paths.processed / "screen_log.csv", index=False)
    report = {"universe": universe, "variant": variant, "features": len(chars), "rows": len(timed),
        "members": int(timed.in_universe.sum()), "valid_training_rows": int(timed.valid_data.sum()),
        "missing_current_return_rows": int(timed.tr_ld0.isna().sum()),
        "missing_SIC_member_rows": int((timed.in_universe & timed.sic.isna()).sum()),
        "missing_forward_return_policy": settings().get("missing_forward_return_policy", "stop"),
        "zero_imputed_forward_returns": int(timed.get("ret_ld1_imputed_zero", pd.Series(False, index=timed.index)).sum()),
        "history_excluded_stock_months": len(history_exclusions),
        "ranks": "Regional average percentile ranks, imputed to 0.5; all 50 independently rebuilt member ranks verified",
        "characteristic_source": "US high-cap-only 2005–2014 association-selected 50; local raw snapshot",
        "theme_direction_policy": "Published signs or US high-cap empirical signs for provisional groups; raw ML ranks unflipped",
        "wealth_market": "formation-cap-weighted screened regional USD excess returns + USD RF",
        "wealth_anchor": str(cfg.investor.wealth_end_date), "dollar_volume_units": "USD, not millions",
        "snapshot": fingerprint, "config": cfg.as_dict(), "missing_inputs": missing_inputs(universe)}
    write_json(cfg.paths.processed / "preparation.json", report)
    from jkmp.imputation_audit import report_if_enabled
    report_if_enabled(cfg)
    print(f"Prepared {universe}/{variant}: {len(timed):,} rows, {int(timed.in_universe.sum()):,} member rows; {len(chars)} features")


def covariance(universe, variant="base"):
    from jkmp import covariance as cov
    cfg = load_config(config_path(universe, variant))
    out = cfg.paths.processed / "covariance"
    if (out / "cov_factor.parquet").exists():
        print(f"Existing covariance retained: {out}")
        return
    daily_path = ROOT / settings()["daily_inputs"][region(universe)]
    if not daily_path.exists() or not daily_path.with_suffix(".json").exists():
        raise FileNotFoundError(f"Missing {daily_path} and/or its provenance JSON. No downloads were attempted. "
            "Provide stock-level USD total returns (id,date,ret); see README.md and outputs/input_audit.json.")
    provenance = json.loads(daily_path.with_suffix(".json").read_text())
    expected = {"currency": "USD", "return_type": "total", "identifier": "jkp_id"}
    if any(provenance.get(k) != v for k, v in expected.items()) or not provenance.get("source"):
        raise ValueError("Daily return provenance must declare USD total returns and JKP security ids")
    daily = pd.read_parquet(daily_path, columns=["id", "date", "ret"])
    daily["date"] = pd.to_datetime(daily.date)
    daily["id"] = pd.to_numeric(daily.id, errors="raise")
    if daily.id.isna().any() or not np.equal(daily.id, np.floor(daily.id)).all():
        raise ValueError("Daily ids must be integral JKP security ids")
    daily["id"] = daily.id.astype("int64")
    if daily.duplicated(["id", "date"]).any() or not np.isfinite(daily.ret).all() or (daily.ret < -1).any():
        raise ValueError("Invalid/duplicate daily return observations")
    panel = pd.read_parquet(cfg.paths.processed / "panel.parquet")
    # Covariance depends on historical daily returns/loadings, not current/forward monthly targets.
    # Return completeness is checked by the portfolio accounting/model stages that actually use it.
    top = pd.read_parquet(cfg.paths.raw / f"top{cfg.covariance.factor_universe_top_n}_by_month.parquet")
    needed = set(top.id) | set(panel.loc[panel.in_universe, "id"])
    absent = needed - set(daily.id)
    if absent:
        raise ValueError(f"Daily file has no observations for {len(absent)} required stocks (e.g. {sorted(absent)[:5]})")
    # 'permno' is only the original covariance function's input-column name. EU values remain JKP ids.
    daily = daily[daily.id.isin(needed)].rename(columns={"id": "permno"})
    result = cov.build_covariance(panel, top, daily, pd.read_parquet(cfg.paths.raw / RAW_FILES["ff_daily"]), cfg)
    expected_months = pd.date_range("2013-12-31", "2025-11-30", freq="ME")
    missing_months = expected_months.difference(pd.DatetimeIndex(result.parts))
    if len(missing_months):
        raise ValueError(f"Daily data does not support {len(missing_months)} required risk months; "
                         f"first missing {missing_months[0].date()}. No partial covariance was saved.")
    cov.save_covariance(result.parts, out)
    result.factor_returns.to_parquet(out / "factor_returns.parquet")
    result.diagnostics.to_csv(out / "cov_diagnostics.csv", index=False)
    write_json(out / "provenance.json", provenance)


def validate_prepared():
    """Publish the observed data gaps instead of claiming a complete backtest."""
    summary = []
    for universe in UNIVERSES:
        cfg = load_config(config_path(universe))
        path = cfg.paths.processed / "panel.parquet"
        if not path.exists():
            continue
        cols = ["id", "eom", "in_universe", "tr_ld0", "ret_ld1", "sic"]
        if "ret_ld1_imputed_zero" in pq.ParquetFile(path).schema.names:
            cols.append("ret_ld1_imputed_zero")
        panel = pd.read_parquet(path, columns=cols)
        scored = panel[panel.in_universe & panel.eom.ge(pd.Timestamp("2013-12-31"))].copy()
        gaps = scored[scored.tr_ld0.isna() | scored.ret_ld1.isna()].copy()
        gaps["missing_current_month_return"] = gaps.tr_ld0.isna()
        gaps["missing_next_month_return"] = gaps.ret_ld1.isna()
        gap_path = ROOT / "outputs" / universe / "monthly_return_gaps.csv"
        gap_path.parent.mkdir(parents=True, exist_ok=True)
        gaps[["id", "eom", "missing_current_month_return", "missing_next_month_return"]].to_csv(gap_path, index=False)
        summary.append({"universe": universe, "prepared_rows": len(panel),
            "member_rows": int(panel.in_universe.sum()),
            "held_missing_current_return_since_2013_12": int(scored.tr_ld0.isna().sum()),
            "held_missing_forward_return_since_2013_12": int(scored.ret_ld1.isna().sum()),
            "zero_imputed_forward_returns_all_history": int(panel.get("ret_ld1_imputed_zero", pd.Series(False, index=panel.index)).sum()),
            "gap_details": str(gap_path.relative_to(ROOT)), "full_backtest_complete": False})
    write_json(ROOT / "outputs/prepared_validation.json", summary)
    print(json.dumps(summary, indent=2))
    return summary


def originals_unchanged():
    """Check accessible clone sources; the standalone pipeline also works without them."""
    manifest_path = ROOT / "data/provenance/source_code_sha256.json"
    if not manifest_path.exists():
        return True
    manifest = json.loads(manifest_path.read_text(encoding="utf-8"))["files"]
    changed = [name for name, digest in manifest.items()
               if (ROOT.parent / name).is_file()
               and hashlib.sha256((ROOT.parent / name).read_bytes()).hexdigest() != digest]
    if changed:
        raise AssertionError(f"Source Keats files changed since the clone snapshot: {changed}")
    return True


def ensure_prediction_coverage(cfg):
    """Fill absent held-stock forecasts from already fitted models, never fabricate returns."""
    import datetime as dt
    from uuid import uuid4
    from jkmp.models.return_model import load_models
    p=cfg.paths.processed
    prediction_path=p/"predictions.parquet"
    if not prediction_path.exists():
        raise FileNotFoundError(f"Run the return-model stage first: {prediction_path}")
    cache=p/"return_model/prediction_coverage.json"
    def signature():
        return {name:{"size":(p/name).stat().st_size,"mtime_ns":(p/name).stat().st_mtime_ns}
                for name in ("panel.parquet","predictions.parquet")}
    before=signature()
    if cache.exists() and json.loads(cache.read_text()).get("inputs")==before:
        return
    columns=["id","eom","in_universe",*cfg.features.names]
    held=pd.read_parquet(p/"panel.parquet",columns=columns,filters=[("in_universe","=",True)])
    return_dates=held.eom+MONTH
    held=held[return_dates.between(pd.Timestamp(cfg.return_model.pred_years[0],1,31),pd.Timestamp(cfg.dates.test_end))]
    predictions=pd.read_parquet(prediction_path)
    present=pd.MultiIndex.from_frame(predictions[["id","eom"]])
    missing=held[~pd.MultiIndex.from_frame(held[["id","eom"]]).isin(present)].copy()
    if len(missing):
        models,z=load_models(p/"return_model/models.npz")
        missing["year"]=(missing.eom+MONTH).dt.year
        additions=[]
        for year,group in missing.groupby("year"):
            if int(year) not in models:
                raise ValueError(f"No fitted return model for held-stock forecast year {year}")
            predicted=models[int(year)].predict(group[list(cfg.features.names)].to_numpy(dtype=float),z)
            additions.append(group[["id","eom","year"]].assign(pred_ld1=predicted))
        extra=pd.concat(additions,ignore_index=True)
        archive=p/"return_model/previous_predictions"/(dt.datetime.now().strftime("%Y%m%d_%H%M%S")+"_"+uuid4().hex[:8])
        archive.mkdir(parents=True)
        shutil.copy2(prediction_path,archive/prediction_path.name)
        pd.concat([predictions,extra],ignore_index=True).sort_values(["eom","id"]).to_parquet(prediction_path,index=False)
        extra.to_csv(p/"return_model/forecast_coverage_additions.csv",index=False)
        print("Additional held-stock forecasts from existing fitted models (no new training labels):",flush=True)
        print(extra.to_string(index=False),flush=True)
    write_json(cache,{"inputs":signature(),"missing_held_forecasts":0,"added_forecasts":len(missing)})


## Source — Shared data and estimation stages


### `scripts/shared/01_prepare.py`

Keats adapter. Use run.py to select universe and variant.

**Role:** Pipeline implementation or configuration. This cell writes the file without running its main function.

**Input:** the embedded source below. **Output:** `scripts/shared/01_prepare.py`. Expand the code cell to read or edit it.


In [ ]:
%%writefile scripts/shared/01_prepare.py
"""Keats adapter. Use run.py to select universe and variant."""
from pathlib import Path as _ScriptPath
import sys as _script_sys
_script_sys.path.insert(0,str(_ScriptPath(__file__).resolve().parents[1]))
from _bootstrap import ROOT
from workflow import prepare

if __name__ == "__main__":
    prepare("us_large")


### `scripts/shared/02_covariance.py`

Keats adapter. Use run.py to select universe and variant.

**Role:** Pipeline implementation or configuration. This cell writes the file without running its main function.

**Input:** the embedded source below. **Output:** `scripts/shared/02_covariance.py`. Expand the code cell to read or edit it.


In [ ]:
%%writefile scripts/shared/02_covariance.py
"""Keats adapter. Use run.py to select universe and variant."""
from pathlib import Path as _ScriptPath
import sys as _script_sys
_script_sys.path.insert(0,str(_ScriptPath(__file__).resolve().parents[1]))
from _bootstrap import ROOT
from workflow import covariance

if __name__ == "__main__":
    covariance("us_large")


### `scripts/shared/03_return_model.py`

Stage 3 runner: one-month return predictions pred_ld1 (random-feature ridge, yearly expanding refit). Writes data/processed/predictions.parquet and return_model/{hp_search,chosen,diagnostics}.* python scripts/03_return_model.py

**Role:** Pipeline implementation or configuration. This cell writes the file without running its main function.

**Input:** the embedded source below. **Output:** `scripts/shared/03_return_model.py`. Expand the code cell to read or edit it.


In [ ]:
%%writefile scripts/shared/03_return_model.py
"""Stage 3 runner: one-month return predictions pred_ld1 (random-feature ridge, yearly expanding refit).

Writes data/processed/predictions.parquet and return_model/{hp_search,chosen,diagnostics}.*

    python scripts/03_return_model.py
"""

from __future__ import annotations

from pathlib import Path as _ScriptPath
import sys as _script_sys
_script_sys.path.insert(0,str(_ScriptPath(__file__).resolve().parents[1]))
from _bootstrap import configure
configure(__file__)

import argparse
import logging

import pandas as pd

from jkmp.config import load_config
from jkmp.evaluation.holdout import last_open_month
from jkmp.models.return_model import fit_return_model, prediction_diagnostics, save_models, tuning_marker, tuning_provenance


def main() -> None:
    parser = argparse.ArgumentParser(description=__doc__, formatter_class=argparse.RawDescriptionHelpFormatter)
    parser.add_argument("--config", default=None, help="YAML config (default: configs/base.yaml)")
    parser.add_argument("--override", action="append", default=[], help="dotted override")
    args = parser.parse_args()

    logging.basicConfig(level=logging.INFO, format="%(asctime)s %(levelname)s %(name)s: %(message)s")
    cfg = load_config(args.config, args.override)
    processed = cfg.paths.processed
    panel = pd.read_parquet(processed / "panel.parquet")
    logging.info("RFF return-model market=%s; validation_scope=%s; fitting=screened regional panel",
                 cfg.paths.raw.name, cfg.return_model.validation_scope)

    result = fit_return_model(panel, cfg)
    preds = result.predictions
    open_preds = preds[preds["eom"] + pd.offsets.MonthEnd(1) <= last_open_month(cfg.dates)]  # no holdout diagnostics
    diag = prediction_diagnostics(open_preds, panel, cfg.dates)
    out = processed / "return_model"
    out.mkdir(parents=True, exist_ok=True)
    result.predictions.to_parquet(processed / "predictions.parquet", index=False)
    result.hp_search.to_parquet(out / "hp_search.parquet", index=False)
    result.chosen.to_csv(out / "chosen.csv", index=False)
    save_models(result, out / "models.npz")
    diag.to_csv(out / "diagnostics.csv", index=False)
    import json
    tuning_marker(cfg).write_text(json.dumps(tuning_provenance(cfg), indent=2, default=str), encoding="utf-8")

    pd.set_option("display.width", 200)
    print("\nChosen hyperparameters by return year:")
    print(result.chosen[["market", "validation_scope", "year", "g", "p", "log_lam", "val_mse", "n_train", "n_val", "n_val_regional", "n_test"]].to_string(index=False))
    print("\nOut-of-sample diagnostics (R² vs zero forecast; mean monthly rank IC):")
    print(diag.pivot(index="year", columns="scope", values=["r2_oos", "rank_ic"]).round(4).to_string())


if __name__ == "__main__":
    main()


### `scripts/shared/04_benchmarks.py`

Market comparator for the test and broader forecast-history windows.

**Role:** Pipeline implementation or configuration. This cell writes the file without running its main function.

**Input:** the embedded source below. **Output:** `scripts/shared/04_benchmarks.py`. Expand the code cell to read or edit it.


In [ ]:
%%writefile scripts/shared/04_benchmarks.py
"""Market comparator for the test and broader forecast-history windows."""
from pathlib import Path as _ScriptPath
import sys as _script_sys
_script_sys.path.insert(0,str(_ScriptPath(__file__).resolve().parents[1]))
from _bootstrap import configure
configure(__file__)
import argparse
import pandas as pd
from jkmp.config import load_config
from jkmp.covariance import load_covariance
from jkmp.evaluation.holdout import evaluation_window,last_open_month
from jkmp.evaluation.metrics import table2
from jkmp.portfolio.benchmarks import market_targets
from jkmp.portfolio.accounting import add_start_weights,portfolio_returns


def main():
    parser=argparse.ArgumentParser(description=__doc__);parser.add_argument("--config");parser.add_argument("--override",action="append",default=[])
    args=parser.parse_args();cfg=load_config(args.config,args.override)
    if cfg.paths.outputs.parent.name not in ("iteration_1","iteration_2"):raise ValueError("Use base market for the base comparison reference")
    p=cfg.paths.processed
    panel=pd.read_parquet(p/"panel.parquet");wealth=pd.read_parquet(p/"wealth.parquet");cov=load_covariance(p/"covariance")
    formation=pd.DatetimeIndex(sorted(cov));returns=formation+pd.offsets.MonthEnd(1)
    first,last=cfg.return_model.pred_years;start,end=evaluation_window(cfg.dates)
    windows={"test":formation[(returns>=start)&(returns<=end)],
             "oos":formation[(returns>=pd.Timestamp(first,1,31))&(returns<=min(pd.Timestamp(last,12,31),last_open_month(cfg.dates)))]}
    out=cfg.paths.outputs/"benchmarks";out.mkdir(parents=True,exist_ok=True)
    for label,dates in windows.items():
        universe=panel.loc[panel.in_universe & panel.eom.isin(dates)]
        weights=add_start_weights(market_targets(universe),panel,wealth,dates,cfg.portfolio.charge_exit_tc).assign(type="Market")
        pf=portfolio_returns(weights,panel,wealth).assign(type="Market")
        weights.to_parquet(out/f"weights_{label}.parquet",index=False);pf.to_parquet(out/f"pf_{label}.parquet",index=False)
        table=table2(pf,cfg.investor.gamma,cfg.dates);table.to_csv(out/f"table2_{label}.csv");print(label,table.to_string())

if __name__=="__main__":main()


### `scripts/shared/06_pfml_inputs.py`

Build the default Portfolio-ML Fourier-signal moments from past stock history.

**Role:** Pipeline implementation or configuration. This cell writes the file without running its main function.

**Input:** the embedded source below. **Output:** `scripts/shared/06_pfml_inputs.py`. Expand the code cell to read or edit it.


In [ ]:
%%writefile scripts/shared/06_pfml_inputs.py
"""Build the default Portfolio-ML Fourier-signal moments from past stock history."""

from __future__ import annotations

from pathlib import Path as _ScriptPath
import sys as _script_sys
_script_sys.path.insert(0,str(_ScriptPath(__file__).resolve().parents[1]))
from _bootstrap import configure
configure(__file__)

import argparse
import logging
import time

import pandas as pd

from jkmp.config import load_config
from jkmp.covariance import load_covariance
from jkmp.data.prepare import risk_free_monthly
from jkmp.evaluation.holdout import evaluation_window
from jkmp.portfolio.portfolio_ml import build_moments, save_moments, signal_set

MONTH = pd.offsets.MonthEnd(1)


def main() -> None:
    parser = argparse.ArgumentParser(description=__doc__, formatter_class=argparse.RawDescriptionHelpFormatter)
    parser.add_argument("--config", default=None, help="YAML config (default: configs/base.yaml)")
    parser.add_argument("--override", action="append", default=[], help="dotted override")
    args = parser.parse_args()

    logging.basicConfig(level=logging.INFO, format="%(asctime)s %(levelname)s %(name)s: %(message)s")
    cfg = load_config(args.config, args.override)
    processed = cfg.paths.processed
    panel = pd.read_parquet(processed / "panel.parquet")
    wealth = pd.read_parquet(processed / "wealth.parquet")
    cov = load_covariance(processed / "covariance")
    rf = risk_free_monthly(pd.read_parquet(cfg.paths.raw / "ff_factors_monthly.parquet"))

    lb = cfg.universe.lookback_months
    first = min(cov) + MONTH * (lb + 1)
    dates = pd.DatetimeIndex([d for d in sorted(cov) if first <= d <= evaluation_window(cfg.dates)[1] - MONTH])
    pm = cfg.portfolio_ml
    common = dict(gamma=cfg.investor.gamma, mu=cfg.investor.mu, lb=lb, iterations=cfg.investor.m_iterations,
                  scale_by_vol=pm.scale_by_vol)
    _, tags, _ = signal_set(True,pm.rff_variance_grid)
    runs=[(tag,dict(rff_variance=g,p_max=max(pm.p_grid),seed=cfg.seed))
          for tag,g in zip(tags,pm.rff_variance_grid)]
    logging.info("PF-ML inputs for %d formation months, %s .. %s", len(dates), dates[0].date(), dates[-1].date())
    for tag, kw in runs:
        start = time.time()
        res = build_moments(panel, wealth, cov, rf, dates, cfg.features.names, **common, **kw)
        save_moments(res, processed / "pfml" / tag)
        logging.info("%s: %d months, %d signals, %.1f min", tag, len(res.months), len(res.columns),
                     (time.time() - start) / 60)


if __name__ == "__main__":
    main()


### `scripts/shared/15_holdout.py`

No separate unseen holdout follows the saved Keats test period.

**Role:** Pipeline implementation or configuration. This cell writes the file without running its main function.

**Input:** the embedded source below. **Output:** `scripts/shared/15_holdout.py`. Expand the code cell to read or edit it.


In [ ]:
%%writefile scripts/shared/15_holdout.py
"""No separate unseen holdout follows the saved Keats test period."""
from pathlib import Path as _ScriptPath
import sys as _script_sys
_script_sys.path.insert(0,str(_ScriptPath(__file__).resolve().parents[1]))
from _bootstrap import ROOT

def main():
    print("No separate Keats holdout: 2021-2025 is already in the declared 2020-2025 test period.")

if __name__ == "__main__":
    main()


### `scripts/shared/active_engine.py`

Benchmark-relative Portfolio ML shared by the three US high-cap iterations. Use scripts/run.py 12 --group iteration_1|iteration_2|iteration_3. Steps: calibrate (pretest tracking-error targets), run (one budget), capacity (fund-size sensitivity), report (saved results). Configurations and outputs resolve inside Kati_tryout. No historical Keats model results are read.

**Role:** Pipeline implementation or configuration. This cell writes the file without running its main function.

**Input:** the embedded source below. **Output:** `scripts/shared/active_engine.py`. Expand the code cell to read or edit it.


In [ ]:
%%writefile scripts/shared/active_engine.py
"""Benchmark-relative Portfolio ML shared by the three US high-cap iterations.

Use scripts/run.py 12 --group iteration_1|iteration_2|iteration_3.
Steps: calibrate (pretest tracking-error targets), run (one budget), capacity
(fund-size sensitivity), report (saved results). Configurations and outputs
resolve inside Kati_tryout. No historical Keats model results are read.
"""

from __future__ import annotations

from pathlib import Path as _ScriptPath
import sys as _script_sys
_script_sys.path.insert(0,str(_ScriptPath(__file__).resolve().parents[1]))
from _bootstrap import configure
configure(__file__)

import argparse
import dataclasses
import logging
import time

import pandas as pd

from jkmp.config import load_config
from jkmp.covariance import load_covariance
from jkmp.data.prepare import risk_free_monthly
from jkmp.evaluation.metrics import (active_returns, active_stats, active_utility_flow, active_weights,
                                     ex_ante_exposures, prob_greater)
from jkmp.evaluation.active import plot_active_exposures
from jkmp.portfolio.active import LABEL, calibrate, calibration_point, run_active
from jkmp.portfolio.grids import grid_inputs, scaled_wealth
from script_groups import baseline_reference_dir

MONTH = pd.offsets.MonthEnd(1)


def main() -> None:
    parser = argparse.ArgumentParser(description=__doc__, formatter_class=argparse.RawDescriptionHelpFormatter)
    parser.add_argument("step", choices=["calibrate", "run", "capacity", "report"])
    parser.add_argument("--config", default=None, help="YAML config (configs/experiments/active_pfml.yaml)")
    parser.add_argument("--override", action="append", default=[], help="dotted override")
    args = parser.parse_args()

    logging.basicConfig(level=logging.INFO, format="%(asctime)s %(levelname)s %(name)s: %(message)s")
    cfg = load_config(args.config, args.override)
    if cfg.paths.outputs.parent.name not in ("iteration_1","iteration_2","iteration_3"):raise ValueError("The active engine requires an iteration output group")
    processed, outputs, inv = cfg.paths.processed, cfg.paths.outputs, cfg.investor
    out = outputs / "active"
    out.mkdir(parents=True, exist_ok=True)
    pd.set_option("display.width", 220)

    preds = pd.read_parquet(processed / "predictions.parquet")[["id", "eom", "pred_ld1"]]
    panel = pd.read_parquet(processed / "panel.parquet").merge(preds, on=["id", "eom"], how="left")
    wealth = pd.read_parquet(processed / "wealth.parquet")
    cov = load_covariance(processed / "covariance")
    rf = risk_free_monthly(pd.read_parquet(cfg.paths.raw / "ff_factors_monthly.parquet"))
    inp = grid_inputs(cfg, panel, wealth, cov, rf)
    del panel

    if args.step == "calibrate":
        rows = []
        for g in inv.gamma_te_grid:
            start = time.time()
            rows.append(calibration_point(inp, cfg, g))
            logging.info("gamma_te %g: ex-ante TE %.4f (%.1f min)", g, rows[-1]["te_ex_ante"], (time.time() - start) / 60)
        points = pd.DataFrame(rows)
        points.to_csv(out / "calibration.csv", index=False)
        chosen = calibrate(points, inv.te_targets)
        chosen.to_csv(out / "calibrated.csv", index=False)
        print(points.to_string(index=False, float_format=lambda v: f"{v:.4f}"))
        print("\nCalibrated gamma_te per ex-ante TE budget (validation years only):")
        print(chosen.to_string(index=False, float_format=lambda v: f"{v:.4g}"))
        return

    if args.step == "run":
        start = time.time()
        res = run_active(inp, cfg, inv.gamma_te, inp.eval_dates, inp.hp_years)
        run_dir = out / f"gte{inv.gamma_te:g}"
        run_dir.mkdir(parents=True, exist_ok=True)
        for name in ("aims", "weights", "pf"):
            res[name].to_parquet(run_dir / f"{name}.parquet", index=False)
        res["chosen"].to_csv(run_dir / "chosen.csv", index=False)
        logging.info("gamma_te %g: test window %s .. %s in %.1f min -> %s", inv.gamma_te,
                     (inp.eval_dates[0] + MONTH).date(), (inp.eval_dates[-1] + MONTH).date(),
                     (time.time() - start) / 60, run_dir)
        return

    if args.step == "capacity":
        for level in inv.wealth_grid:
            start = time.time()
            point = dataclasses.replace(inp, wealth=scaled_wealth(inp.wealth, level, inv.wealth))
            res = run_active(point, cfg, inv.gamma_te, inp.eval_dates, inp.hp_years)
            run_dir = out / "capacity" / f"gte{inv.gamma_te:g}_w{level / 1e9:g}B"
            run_dir.mkdir(parents=True, exist_ok=True)
            for name in ("weights", "pf"):
                res[name].to_parquet(run_dir / f"{name}.parquet", index=False)
            logging.info("wealth $%gB: %.1f min -> %s", level / 1e9, (time.time() - start) / 60, run_dir)
        return

    # report
    budgets = pd.read_csv(out / "calibrated.csv")
    bench_all = pd.read_parquet(outputs / "benchmarks" / "pf_test.parquet")
    bench = bench_all[bench_all["type"] == "Market"]  # the value-weighted universe
    baseline_output = baseline_reference_dir(cfg)
    base_pf = pd.read_parquet(baseline_output / "portfolio_ml" / "rff" / "pf.parquet")
    base_w = pd.read_parquet(baseline_output / "portfolio_ml" / "rff" / "weights.parquet")
    rows, exposures = [], []
    base_ex = ex_ante_exposures(active_weights(base_w, inp.panel), cov).assign(type="Portfolio-ML (baseline)")
    exposures.append(base_ex)
    for b in budgets.itertuples(index=False):
        run_dir = out / f"gte{b.gamma_te:g}"
        pf, w = pd.read_parquet(run_dir / "pf.parquet"), pd.read_parquet(run_dir / "weights.parquet")
        s = active_stats(pf, bench, w, inp.panel, cov, b.gamma_te, cfg.dates)
        sb = active_stats(base_pf, bench, base_w, inp.panel, cov, b.gamma_te, cfg.dates)
        u = active_utility_flow(active_returns(pf, bench, cfg.dates), b.gamma_te)
        ub = active_utility_flow(active_returns(base_pf, bench, cfg.dates), b.gamma_te)
        p_beat = prob_greater(u.reset_index(drop=True), ub.reset_index(drop=True))
        label = f"{LABEL}, TE {b.te_target:.0%}"
        rows.append({**s.to_dict(), "type": label, "te_target": b.te_target, "gamma_te": b.gamma_te,
                     "p_beats_baseline": p_beat})
        rows.append({**sb.to_dict(), "type": f"Portfolio-ML (baseline) at γ_TE of TE {b.te_target:.0%}",
                     "te_target": b.te_target, "gamma_te": b.gamma_te, "p_beats_baseline": float("nan")})
        exposures.append(ex_ante_exposures(active_weights(w, inp.panel), cov).assign(type=label))
    summary = pd.DataFrame(rows).set_index("type")
    summary.to_csv(out / "summary.csv")
    exp = pd.concat(exposures, ignore_index=True)
    exp.to_parquet(out / "exposures.parquet", index=False)
    written = plot_active_exposures(exp, outputs / "figures" / "active_theme_exposure")
    logging.info("wrote %s", ", ".join(p.name for p in written))

    m = bench
    print(f"Benchmark (VW universe) own trading: turnover {m['turnover'].mean():.4f} a month, "
          f"TC {m['tc'].mean() * 12:.5f} a year")
    cols = ["gamma_te", "active", "active_net", "te_ex_ante", "te_realised", "ir_gross", "ir_net", "p_ir_net_pos",
            "beta", "neg_weight_share", "active_utility", "p_beats_baseline", "tc"]
    print(summary[cols].to_string(float_format=lambda v: f"{v:.4f}"))
    act = summary[summary.index.str.startswith(LABEL)]
    keep = act[act["p_beats_baseline"] >= 0.80]
    killed = not (act["p_ir_net_pos"] >= 0.80).any()
    print(f"\nPre-registered rule: keep budgets {list(keep['te_target'])} (P(beats baseline) >= 80%); "
          f"kill = {killed} (no budget with P(net IR > 0) >= 80%)")


if __name__ == "__main__":
    main()


## Source — Base: US high-cap Static ML* and regional Portfolio ML


### `scripts/base/05_static_ml.py`

Run annually tuned Static-ML* for the US high-cap base only.

**Role:** Pipeline implementation or configuration. This cell writes the file without running its main function.

**Input:** the embedded source below. **Output:** `scripts/base/05_static_ml.py`. Expand the code cell to read or edit it.


In [ ]:
%%writefile scripts/base/05_static_ml.py
"""Run annually tuned Static-ML* for the US high-cap base only."""
from pathlib import Path as _ScriptPath
import sys as _script_sys
_script_sys.path.insert(0,str(_ScriptPath(__file__).resolve().parents[1]))
from _bootstrap import configure
configure(__file__)
import argparse
import pandas as pd
from jkmp.config import load_config
from jkmp.covariance import load_covariance
from jkmp.evaluation.holdout import evaluation_window
from jkmp.evaluation.metrics import table2
from jkmp.portfolio.static_ml_star import hyper_grid, run_static_ml_star, tuning_grid_marker


def main():
    parser=argparse.ArgumentParser(description=__doc__);parser.add_argument("--config");parser.add_argument("--override",action="append",default=[])
    args=parser.parse_args();cfg=load_config(args.config,args.override)
    if cfg.paths.outputs.parent.name != "base":raise ValueError("This script writes base results; choose the matching iteration script")
    if cfg.paths.outputs.name != "us_large":raise ValueError("Base Static-ML* applies only to US high-cap")
    p=cfg.paths.processed
    predictions=pd.read_parquet(p/"predictions.parquet")[["id","eom","pred_ld1"]]
    panel=pd.read_parquet(p/"panel.parquet").merge(predictions,on=["id","eom"],how="left",validate="one_to_one")
    wealth=pd.read_parquet(p/"wealth.parquet");cov=load_covariance(p/"covariance")
    formation=pd.DatetimeIndex(sorted(cov));returns=formation+pd.offsets.MonthEnd(1);start,end=evaluation_window(cfg.dates)
    history=formation[(returns>=pd.Timestamp(cfg.return_model.pred_years[0],1,31))&(returns<=end)]
    evaluation=formation[(returns>=start)&(returns<=end)]
    grid=hyper_grid(cfg.static_ml.k_grid,cfg.static_ml.u_grid,cfg.static_ml.g_grid)
    result=run_static_ml_star(panel,wealth,cov,history,evaluation,cfg.investor.gamma,grid,cfg.portfolio.charge_exit_tc)
    out=cfg.paths.outputs/"static_ml_star";out.mkdir(parents=True,exist_ok=True)
    for name,frame in result.items():
        if name=="chosen":frame.to_csv(out/"chosen.csv",index=False)
        else:frame.to_parquet(out/f"{name}.parquet",index=False)
    print("Static-ML*: k/u/g selected each December on cumulative preceding utility; applied to the next return year.")
    print(result["chosen"].to_string(index=False))
    print(table2(result["pf"],cfg.investor.gamma,cfg.dates).to_string())
    import json
    tuning_grid_marker(cfg).write_text(json.dumps(dict(method="Static-ML*",candidates=len(grid),
        k_grid=cfg.static_ml.k_grid,u_grid=cfg.static_ml.u_grid,g_grid=cfg.static_ml.g_grid)),encoding="utf-8")

if __name__=="__main__":main()


### `scripts/base/07_portfolio_ml.py`

Fit and evaluate the sole default Portfolio-ML strategy with expanding annual updates.

**Role:** Pipeline implementation or configuration. This cell writes the file without running its main function.

**Input:** the embedded source below. **Output:** `scripts/base/07_portfolio_ml.py`. Expand the code cell to read or edit it.


In [ ]:
%%writefile scripts/base/07_portfolio_ml.py
"""Fit and evaluate the sole default Portfolio-ML strategy with expanding annual updates."""

from __future__ import annotations

from pathlib import Path as _ScriptPath
import sys as _script_sys
_script_sys.path.insert(0,str(_ScriptPath(__file__).resolve().parents[1]))
from _bootstrap import configure
configure(__file__)

import argparse
import logging
import json

import numpy as np
import pandas as pd

from jkmp.config import load_config
from jkmp.covariance import load_covariance
from jkmp.data.prepare import risk_free_monthly
from jkmp.evaluation.holdout import evaluation_window
from jkmp.evaluation.metrics import table2
from jkmp.portfolio.portfolio_ml import load_moments, return_year, run_portfolio_ml, search, signal_set

MONTH = pd.offsets.MonthEnd(1)


def main() -> None:
    parser = argparse.ArgumentParser(description=__doc__, formatter_class=argparse.RawDescriptionHelpFormatter)
    parser.add_argument("--config", default=None, help="YAML config (default: configs/base.yaml)")
    parser.add_argument("--override", action="append", default=[], help="dotted override")
    args = parser.parse_args()

    logging.basicConfig(level=logging.INFO, format="%(asctime)s %(levelname)s %(name)s: %(message)s")
    cfg = load_config(args.config, args.override)
    if cfg.paths.outputs.parent.name != "base":raise ValueError("This script writes base results; choose the matching iteration script")
    processed, outputs, pm = cfg.paths.processed, cfg.paths.outputs, cfg.portfolio_ml
    sig, tags, label = signal_set(True,pm.rff_variance_grid)
    start, end = evaluation_window(cfg.dates)
    hp_years = list(range(cfg.return_model.pred_years[0], end.year + 1))

    results, aim_dates = {}, None
    for tag in tags:
        moments = load_moments(processed / "pfml" / tag, with_risk=False)
        aim_dates = [d for d in sorted(moments.months) if hp_years[0] < return_year(d) <= hp_years[-1]]
        results[tag] = search(moments, tag, pm.p_grid, cfg.ridge_grid.values, hp_years, aim_dates)
        del moments

    panel = pd.read_parquet(processed / "panel.parquet")
    wealth = pd.read_parquet(processed / "wealth.parquet")
    cov = load_covariance(processed / "covariance")
    rf = risk_free_monthly(pd.read_parquet(cfg.paths.raw / "ff_factors_monthly.parquet"))
    ret = pd.DatetimeIndex(aim_dates) + MONTH
    eval_dates = pd.DatetimeIndex(aim_dates)[(ret >= start) & (ret <= end)]
    res = run_portfolio_ml(results, panel, wealth, cov, rf, aim_dates, eval_dates, gamma=cfg.investor.gamma,
                           mu=cfg.investor.mu, iterations=cfg.investor.m_iterations,
                           charge_exit_tc=cfg.portfolio.charge_exit_tc)

    out = outputs / "portfolio_ml" / sig
    out.mkdir(parents=True, exist_ok=True)
    for name in ("validation", "aims", "aim_coefs", "weights", "pf"):
        res[name].to_parquet(out / f"{name}.parquet", index=False)
    res["chosen"].to_csv(out / "chosen.csv", index=False)

    from jkmp.strategies import base_result_paths
    pfs = base_result_paths(outputs)
    t2 = table2(pd.concat([pd.read_parquet(f) for f in pfs if f.exists()], ignore_index=True), cfg.investor.gamma,
            cfg.dates)
    t2.to_csv(outputs / "table2_test.csv")
    from jkmp.strategies import portfolio_tuning_marker
    portfolio_tuning_marker(cfg).write_text(json.dumps(dict(
        method="Portfolio-ML", market=outputs.name, penalties=cfg.ridge_grid.values)),encoding="utf-8")

    pd.set_option("display.width", 200)
    chosen = res["chosen"]
    print(f"\n{label} choices by return year:")
    print(chosen.assign(log_l=np.log(chosen["l"].where(chosen["l"] > 0)))  # NaN for λ = 0
          [["year", "tag", "p", "log_l", "cum_obj"]].to_string(index=False))
    s = res["aims"].groupby("eom")["w_aim"].agg(lev=lambda w: w.abs().sum(), net="sum")
    print(f"\nAims {s.index[0].date()} .. {s.index[-1].date()}: median Σ|w| {s['lev'].median():.2f}, "
          f"median Σw {s['net'].median():.2f}")
    print(f"\nTable 2 — test: returns {eval_dates[0] + MONTH:%Y-%m} to {eval_dates[-1] + MONTH:%Y-%m}, "
          f"wealth ${cfg.investor.wealth / 1e9:.0f}B by {cfg.investor.wealth_end_date:%Y}, gamma {cfg.investor.gamma:g}")
    print(t2.to_string(float_format=lambda v: f"{v:.3f}"))


if __name__ == "__main__":
    main()


### `scripts/base/08_figure1.py`

Retained-model Figure 1; see Kati_tryout/scripts/figure1.py for the implementation.

**Role:** Pipeline implementation or configuration. This cell writes the file without running its main function.

**Input:** the embedded source below. **Output:** `scripts/base/08_figure1.py`. Expand the code cell to read or edit it.


In [ ]:
%%writefile scripts/base/08_figure1.py
"""Retained-model Figure 1; see Kati_tryout/scripts/figure1.py for the implementation."""
from pathlib import Path
import sys
ROOT=Path(__file__).resolve().parents[2]
sys.path.insert(0,str(ROOT/"scripts"))
from figure1 import main
if __name__=="__main__":raise SystemExit(main())


### `scripts/base/10_regimes.py`

Regimes over the test window: FRED USREC / VIXCLS / FEDFUNDS labels, cumulative utility with shaded episodes, Table 2 statistics by regime and method, and every method's trades re-costed with Λ ∝ VIX. Uses the test-window performance of every method present (outputs/benchmarks/pf_test.parquet, outputs/static_ml/pf.parquet, outputs/portfolio_ml/rff/pf.parquet). FRED files are cached in data/raw/fred_{series}.parquet (--force re-downloads). Writes outputs/regimes/{labels,regime_table,vix_recost,cumulative_utility}.csv and outputs/figures/cumulati

**Role:** Pipeline implementation or configuration. This cell writes the file without running its main function.

**Input:** the embedded source below. **Output:** `scripts/base/10_regimes.py`. Expand the code cell to read or edit it.


In [ ]:
%%writefile scripts/base/10_regimes.py
"""Regimes over the test window: FRED USREC / VIXCLS / FEDFUNDS labels, cumulative utility with shaded
episodes, Table 2 statistics by regime and method, and every method's trades re-costed with Λ ∝ VIX.

Uses the test-window performance of every method present (outputs/benchmarks/pf_test.parquet,
outputs/static_ml/pf.parquet, outputs/portfolio_ml/rff/pf.parquet). FRED files are cached in
data/raw/fred_{series}.parquet (--force re-downloads).
Writes outputs/regimes/{labels,regime_table,vix_recost,cumulative_utility}.csv and
outputs/figures/cumulative_utility.{png,pdf}.

    python scripts/10_regimes.py [--force]
"""

from __future__ import annotations

from pathlib import Path as _ScriptPath
import sys as _script_sys
_script_sys.path.insert(0,str(_ScriptPath(__file__).resolve().parents[1]))
from _bootstrap import configure
configure(__file__)

import argparse
import logging

import pandas as pd

from jkmp.config import load_config
from jkmp.evaluation import regimes as rg


def main() -> None:
    parser = argparse.ArgumentParser(description=__doc__, formatter_class=argparse.RawDescriptionHelpFormatter)
    parser.add_argument("--config", default=None, help="YAML config (default: configs/base.yaml)")
    parser.add_argument("--override", action="append", default=[], help="dotted override")
    parser.add_argument("--force", action="store_true", help="re-download the FRED series")
    args = parser.parse_args()

    logging.basicConfig(level=logging.INFO, format="%(asctime)s %(levelname)s %(name)s: %(message)s")
    cfg = load_config(args.config, args.override)
    if cfg.paths.outputs.parent.name != "base":raise ValueError("This script writes base results; choose the matching iteration script")
    outputs, rc, gamma = cfg.paths.outputs, cfg.regimes, cfg.investor.gamma

    from jkmp.strategies import base_result_paths
    files = base_result_paths(outputs)
    pfs = pd.concat([pd.read_parquet(f) for f in files if f.exists()], ignore_index=True)
    from jkmp.strategies import require_strategies
    require_strategies(pfs,complete=True,universe=outputs.name)
    pfs = pfs[["type", "eom", "eom_ret", "r", "tc", "turnover", "inv", "shorting"]]
    ret_months = pd.DatetimeIndex(sorted(pfs["eom_ret"].unique()))
    form_months = pd.DatetimeIndex(sorted(pfs["eom"].unique()))

    fred = rg.load_fred(cfg.data_sources.fred_csv_url, rc.fred_series, cfg.paths.raw, force=args.force)
    monthly = rg.monthly_series(fred)
    labels = rg.label_regimes(monthly, ret_months, rc.fed_change_months, rc.fed_threshold)
    table = rg.regime_table(pfs, labels, gamma, cfg.dates)
    scale = rg.vix_cost_scale(monthly, form_months, rc.vix_norm)
    recost = rg.recost_vix(pfs, scale, gamma, cfg.dates)
    cum = rg.cumulative_utility(pfs, gamma, cfg.dates)

    out = outputs / "regimes"
    out.mkdir(parents=True, exist_ok=True)
    labels.to_csv(out / "labels.csv", index=False)
    table.to_csv(out / "regime_table.csv", index=False)
    recost.to_csv(out / "vix_recost.csv", index=False)
    cum.to_csv(out / "cumulative_utility.csv", index=False)
    import json
    (out/"model_scope.json").write_text(json.dumps(sorted(pfs.type.unique())),encoding="utf-8")
    written = rg.plot_cumulative_utility(cum, rc.episodes, rc.plot_types, gamma,
                                         outputs / "figures" / "cumulative_utility", cfg.dates)
    logging.info("wrote %s", ", ".join(p.name for p in written))

    pd.set_option("display.width", 220)
    order = list(dict.fromkeys(pfs["type"]))
    print("Months per regime (return months):")
    for dim in rg.REGIME_COLUMNS:
        print(f"  {dim}: {labels[dim].value_counts().to_dict()}")
    for col in ("Utility", "TC", "Turnover"):
        piv = table.pivot_table(index=["dimension", "regime"], columns="type", values=col)[order]
        print(f"\n{col} by regime (annualised; Turnover monthly):")
        print(piv.to_string(float_format=lambda v: f"{v:.4f}"))
    print(f"\nVIX re-costing (Λ_t = Λ · VIX_t / mean VIX, mean over the {rc.vix_norm}):")
    print(recost.to_string(index=False, float_format=lambda v: f"{v:.4f}"))
    from jkmp.strategies import base_sources_marker
    base_sources_marker(outputs,"regimes").write_text(json.dumps(dict(types=sorted(pfs.type.unique()))),encoding="utf-8")


if __name__ == "__main__":
    main()


### `scripts/base/11_tables.py`

Base tables: US high-cap Static-ML*, plus Portfolio-ML in all three markets.

**Role:** Pipeline implementation or configuration. This cell writes the file without running its main function.

**Input:** the embedded source below. **Output:** `scripts/base/11_tables.py`. Expand the code cell to read or edit it.


In [ ]:
%%writefile scripts/base/11_tables.py
"""Base tables: US high-cap Static-ML*, plus Portfolio-ML in all three markets."""
from pathlib import Path as _ScriptPath
import sys as _script_sys
_script_sys.path.insert(0,str(_ScriptPath(__file__).resolve().parents[1]))
from _bootstrap import configure
configure(__file__)
import argparse
import pandas as pd
from jkmp.config import load_config
from jkmp.evaluation.metrics import table2,prob_outperformance,return_correlations,subperiod_table
from jkmp.strategies import base_strategies,base_result_paths,base_sources_marker,require_strategies


def main():
    parser=argparse.ArgumentParser(description=__doc__);parser.add_argument("--config");parser.add_argument("--override",action="append",default=[])
    args=parser.parse_args();cfg=load_config(args.config,args.override)
    if cfg.paths.outputs.parent.name != "base":raise ValueError("This script writes base results; choose the matching iteration script")
    out=cfg.paths.outputs
    paths=base_result_paths(out);types=base_strategies(out.name)
    pfs=require_strategies(pd.concat([pd.read_parquet(p) for p in paths],ignore_index=True),complete=True,universe=out.name)
    folder=out/"tables";folder.mkdir(parents=True,exist_ok=True)
    t2=table2(pfs,cfg.investor.gamma,cfg.dates).reindex(types);t2.to_csv(out/"table2_test.csv");t2.to_csv(folder/"table2.csv")
    prob_outperformance(pfs,cfg.investor.gamma,cfg.dates,types).to_csv(folder/"table3.csv")
    return_correlations(pfs,cfg.dates,types).to_csv(folder/"table4.csv")
    periods={p.label:(p.start,p.end) for p in cfg.reporting.subperiods}
    subperiod_table(pfs,cfg.investor.gamma,cfg.dates,periods,["R_TC","SR_net","Utility","TC"]).to_csv(folder/"subperiods.csv")
    print(t2.to_string())
    import json
    (folder/"model_scope.json").write_text(json.dumps(list(types)),encoding="utf-8")
    base_sources_marker(out,"tables").write_text(json.dumps(dict(types=list(types))),encoding="utf-8")

if __name__=="__main__":main()


### `scripts/base/compare_results.py`

Refresh base-only tables and charts from saved Static ML / Portfolio ML outputs.

**Role:** Pipeline implementation or configuration. This cell writes the file without running its main function.

**Input:** the embedded source below. **Output:** `scripts/base/compare_results.py`. Expand the code cell to read or edit it.


In [ ]:
%%writefile scripts/base/compare_results.py
"""Refresh base-only tables and charts from saved Static ML / Portfolio ML outputs."""
from pathlib import Path
import sys
ROOT=Path(__file__).resolve().parents[2]
sys.path.insert(0,str(ROOT/"scripts"))
from analysis.ml_comparison.cli import main
if __name__=="__main__":raise SystemExit(main())


## Source — Iteration 1: annual refitting already in Base


### `scripts/iteration_1/run.py`

Annual refitting: retained as already part of Base. No extra fitted experiment.

**Role:** Pipeline implementation or configuration. This cell writes the file without running its main function.

**Input:** the embedded source below. **Output:** `scripts/iteration_1/run.py`. Expand the code cell to read or edit it.


In [ ]:
%%writefile scripts/iteration_1/run.py
"""Annual refitting: retained as already part of Base. No extra fitted experiment."""
from pathlib import Path
import sys
sys.path.insert(0,str(Path(__file__).resolve().parents[1]))
from iteration_sequence import main
if __name__=='__main__':raise SystemExit(main(['--upto','1',*sys.argv[1:]]))


## Source — Iteration 2: gamma 5


### `scripts/iteration_2/run.py`

Gamma 5 on the last accepted US high-cap setup; paired test-P comparison.

**Role:** Pipeline implementation or configuration. This cell writes the file without running its main function.

**Input:** the embedded source below. **Output:** `scripts/iteration_2/run.py`. Expand the code cell to read or edit it.


In [ ]:
%%writefile scripts/iteration_2/run.py
"""Gamma 5 on the last accepted US high-cap setup; paired test-P comparison."""
from pathlib import Path
import sys
sys.path.insert(0,str(Path(__file__).resolve().parents[1]))
from iteration_sequence import main
if __name__=='__main__':raise SystemExit(main(['--upto','2',*sys.argv[1:]]))


## Source — Iteration 3: exit 120 and doubled costs


### `scripts/iteration_3/run.py`

Exit 120 and doubled impact costs on the last accepted US high-cap setup.

**Role:** Pipeline implementation or configuration. This cell writes the file without running its main function.

**Input:** the embedded source below. **Output:** `scripts/iteration_3/run.py`. Expand the code cell to read or edit it.


In [ ]:
%%writefile scripts/iteration_3/run.py
"""Exit 120 and doubled impact costs on the last accepted US high-cap setup."""
from pathlib import Path
import sys
sys.path.insert(0,str(Path(__file__).resolve().parents[1]))
from iteration_sequence import main
if __name__=='__main__':raise SystemExit(main(['--upto','3',*sys.argv[1:]]))


## Source — Iteration 4: Market plus 1% TE tilt


### `scripts/iteration_4/run.py`

Market plus pretest-calibrated 1% TE tilt, inheriting only accepted changes.

**Role:** Pipeline implementation or configuration. This cell writes the file without running its main function.

**Input:** the embedded source below. **Output:** `scripts/iteration_4/run.py`. Expand the code cell to read or edit it.


In [ ]:
%%writefile scripts/iteration_4/run.py
"""Market plus pretest-calibrated 1% TE tilt, inheriting only accepted changes."""
from pathlib import Path
import sys
sys.path.insert(0,str(Path(__file__).resolve().parents[1]))
from iteration_sequence import main
if __name__=='__main__':raise SystemExit(main(['--upto','4',*sys.argv[1:]]))


## Source — Iteration 5: recent-history weighting


### `scripts/iteration_5/run.py`

Recent-history weighting on the last accepted setup, retaining rejected results.

**Role:** Pipeline implementation or configuration. This cell writes the file without running its main function.

**Input:** the embedded source below. **Output:** `scripts/iteration_5/run.py`. Expand the code cell to read or edit it.


In [ ]:
%%writefile scripts/iteration_5/run.py
"""Recent-history weighting on the last accepted setup, retaining rejected results."""
from pathlib import Path
import sys
sys.path.insert(0,str(Path(__file__).resolve().parents[1]))
from iteration_sequence import main
if __name__=='__main__':raise SystemExit(main(['--upto','5',*sys.argv[1:]]))


## Source — Legacy iteration source (reference only)


### `scripts/legacy_iterations/iteration_1/run.py`

iteration_1: original active Portfolio-ML engine; use calibrate/run/capacity/report as applicable.

**Role:** Pipeline implementation or configuration. This cell writes the file without running its main function.

**Input:** the embedded source below. **Output:** `scripts/legacy_iterations/iteration_1/run.py`. Expand the code cell to read or edit it.


In [ ]:
%%writefile scripts/legacy_iterations/iteration_1/run.py
"""iteration_1: original active Portfolio-ML engine; use calibrate/run/capacity/report as applicable."""
from pathlib import Path
import runpy,sys
ROOT=Path(__file__).resolve().parents[2]
if "--config" not in sys.argv:sys.argv += ["--config",str(ROOT/"configs/us_large/iteration_1.yaml")]
runpy.run_path(str(ROOT/"scripts/shared/active_engine.py"),run_name="__main__")


### `scripts/legacy_iterations/iteration_2/compare_breadth.py`

Trial 3 (docs/research_log.md): 250-stock vs 80-stock active Portfolio-ML — the pre-registered net-IR comparison, rank IC by size bucket, capacity curves and the cost sensitivity (re-costed weights, no re-training). python scripts/13_breadth.py --te 0.02 Reads the reference (configs/experiments/active_pfml.yaml: outputs/active/) and the variant (configs/experiments/universe_250.yaml: outputs/u250/active/), each with its calibrated γ_TE for the budget --te, its `run` and its `capacity` outputs (scripts/12_active_pfml.py). Writes outputs/u250/bre

**Role:** Pipeline implementation or configuration. This cell writes the file without running its main function.

**Input:** the embedded source below. **Output:** `scripts/legacy_iterations/iteration_2/compare_breadth.py`. Expand the code cell to read or edit it.


In [ ]:
%%writefile scripts/legacy_iterations/iteration_2/compare_breadth.py
"""Trial 3 (docs/research_log.md): 250-stock vs 80-stock active Portfolio-ML — the pre-registered net-IR comparison,
rank IC by size bucket, capacity curves and the cost sensitivity (re-costed weights, no re-training).

    python scripts/13_breadth.py --te 0.02

Reads the reference (configs/experiments/active_pfml.yaml: outputs/active/) and the variant
(configs/experiments/universe_250.yaml: outputs/u250/active/), each with its calibrated γ_TE for the budget
--te, its `run` and its `capacity` outputs (scripts/12_active_pfml.py). Writes outputs/u250/breadth/*.csv and
outputs/u250/figures/capacity_ir.{png,pdf}.
"""

from __future__ import annotations

from pathlib import Path as _ScriptPath
import sys as _script_sys
_script_sys.path.insert(0,str(_ScriptPath(__file__).resolve().parents[1]))
from _bootstrap import configure
configure(__file__)

import argparse
import logging

import matplotlib

matplotlib.use("Agg")
import matplotlib.pyplot as plt  # noqa: E402
import pandas as pd  # noqa: E402

from jkmp.config import PROJECT_ROOT, load_config  # noqa: E402
from jkmp.covariance import load_covariance  # noqa: E402
from jkmp.evaluation import regimes as rg  # noqa: E402
from jkmp.evaluation.cost_sensitivity import capacity_aum, recost, vol_cost_scale  # noqa: E402
from jkmp.evaluation.metrics import active_returns, active_stats, prob_greater  # noqa: E402
from jkmp.models.return_model import rank_ic_by_size  # noqa: E402

CONFIGS = {"80-stock": "configs/us_large/iteration_1.yaml", "250-stock": "configs/us_large/iteration_2.yaml"}


def load_side(name: str, te: float) -> dict:
    cfg = load_config(PROJECT_ROOT / CONFIGS[name])
    out = cfg.paths.outputs / "active"
    cal = pd.read_csv(out / "calibrated.csv")
    gamma_te = float(cal.loc[(cal["te_target"] - te).abs() < 1e-12, "gamma_te"].iloc[0])
    bench = pd.read_parquet(cfg.paths.outputs / "benchmarks" / "pf_test.parquet").query("type == 'Market'")
    return dict(cfg=cfg, gamma_te=gamma_te, run=out / f"gte{gamma_te:g}", capacity=out / "capacity", bench=bench,
                panel=pd.read_parquet(cfg.paths.processed / "panel.parquet"),
                wealth=pd.read_parquet(cfg.paths.processed / "wealth.parquet"),
                cov=load_covariance(cfg.paths.processed / "covariance"))


def main() -> None:
    parser = argparse.ArgumentParser(description=__doc__, formatter_class=argparse.RawDescriptionHelpFormatter)
    parser.add_argument("--te", type=float, default=0.02, help="TE budget of the compared runs (a calibrated target)")
    args = parser.parse_args()
    logging.basicConfig(level=logging.INFO, format="%(asctime)s %(levelname)s %(name)s: %(message)s")
    sides = {name: load_side(name, args.te) for name in CONFIGS}
    cfg = sides["250-stock"]["cfg"]
    dates, rep, inv = cfg.dates, cfg.reporting, cfg.investor
    out = cfg.paths.outputs / "breadth"
    out.mkdir(parents=True, exist_ok=True)
    pd.set_option("display.width", 220)

    # 1. Pre-registered decision: net IR improvement of 250-stock over 80-stock
    stats, scaled = {}, {}
    for name, s in sides.items():
        pf, w = pd.read_parquet(s["run"] / "pf.parquet"), pd.read_parquet(s["run"] / "weights.parquet")
        stats[name] = active_stats(pf, s["bench"], w, s["panel"], s["cov"], s["gamma_te"], dates)
        act = active_returns(pf, s["bench"], dates)
        scaled[name] = (act["active_net"] / act["active"].std(ddof=1)).reset_index(drop=True)
    p_improve = prob_greater(scaled["250-stock"], scaled["80-stock"])
    summary = pd.DataFrame(stats).T.assign(gamma_te=[sides[n]["gamma_te"] for n in stats])
    summary.to_csv(out / "summary.csv")
    print(f"Active Portfolio-ML at the {args.te:.0%} TE budget, 2020–2025:")
    print(summary[["gamma_te", "active_net", "te_ex_ante", "te_realised", "ir_gross", "ir_net", "p_ir_net_pos",
                   "beta", "neg_weight_share", "tc"]].to_string(float_format=lambda v: f"{v:.4f}"))
    print(f"\nP(net IR 250-stock > 80-stock) = {p_improve:.4f} -> {'KEEP' if p_improve >= 0.80 else 'KILL'} "
          f"(pre-registered bar 0.80)")

    # 2. Reference-model rank IC by size bucket. Held-specific zero targets can
    # change fitted forecasts between the 80- and 250-stock variants.
    base = sides["80-stock"]
    preds = pd.read_parquet(base["cfg"].paths.processed / "predictions.parquet")
    preds = preds[preds["eom"] + pd.offsets.MonthEnd(1) <= pd.Timestamp(dates.test_end)]
    ic = rank_ic_by_size(preds, base["panel"], rep.ic_size_buckets, dates)
    ic.to_csv(out / "rank_ic_by_size.csv", index=False)
    print("\nReference 80-stock model: mean monthly rank IC by market-cap rank bucket (return years 2015–2025):")
    print(ic.pivot(index="decade", columns="bucket", values="rank_ic").to_string(float_format=lambda v: f"{v:.4f}"))

    # 3. Capacity and cost sensitivity: net IR vs wealth, base costs, VIX-scaled (2020–2025 (VIX subset)), stock-vol-scaled
    vix = rg.monthly_series(rg.load_fred(cfg.data_sources.fred_csv_url, cfg.regimes.fred_series, cfg.paths.raw,
                                         force=False))
    rows = []
    vix_start = pd.Timestamp(rep.cost_vix_start)
    for name, s in sides.items():
        vol_scale = vol_cost_scale(s["cov"], rep.cost_vol_window)
        for level in inv.wealth_grid:
            d = s["capacity"] / f"gte{s['gamma_te']:g}_w{level / 1e9:g}B"
            w = pd.read_parquet(d / "weights.parquet")
            wealth = s["wealth"].assign(wealth=s["wealth"]["wealth"] * level / inv.wealth)
            # "paper index": the benchmark earns its return at no cost (the capacity headline). "traded VW": the
            # benchmark pays its own rebalancing cost at this wealth (∝ wealth), the definition pre-registered at $10B;
            # at large wealth it rewards a fund for skipping the rebalancing it cannot afford.
            benches = {"paper index": s["bench"].assign(tc=0.0),
                       "traded VW": s["bench"].assign(tc=s["bench"]["tc"] * level / inv.wealth)}
            form = pd.DatetimeIndex(sorted(w["eom"].unique()))
            late = form[form + pd.offsets.MonthEnd(1) >= vix_start]
            vix_scale = rg.vix_cost_scale(vix, late, "window")
            runs = {"base": recost(w, s["panel"], wealth, pd.Series(1.0, index=form)),
                    "stock vol": recost(w, s["panel"], wealth, vol_scale)}
            for (model, pf), (bname, bench) in ((r, b) for r in runs.items() for b in benches.items()):
                st = active_stats(pf, bench, w, s["panel"], s["cov"], s["gamma_te"], dates)
                rows.append({"universe": name, "benchmark": bname, "wealth": level, "costs": model,
                             "period": "2020–2025", "ir_net": st["ir_net"], "tc": st["tc"], "te": st["te_realised"]})
            w_late = w[w["eom"].isin(late)]
            for (model, scale), (bname, bench) in ((m, b) for m in (("base", pd.Series(1.0, index=late)), ("VIX", vix_scale))
                                                    for b in benches.items()):
                pf = recost(w_late, s["panel"], wealth, scale)
                st = active_stats(pf, bench[bench["eom"].isin(late)], w_late, s["panel"], s["cov"], s["gamma_te"], dates)
                rows.append({"universe": name, "benchmark": bname, "wealth": level, "costs": model,
                             "period": "2020–2025 (VIX subset)", "ir_net": st["ir_net"], "tc": st["tc"], "te": st["te_realised"]})
    cap = pd.DataFrame(rows)
    cap.to_csv(out / "capacity_curve.csv", index=False)
    aum = (cap.groupby(["benchmark", "universe", "period", "costs"])
           .apply(lambda g: capacity_aum(g["wealth"], g["ir_net"], rep.capacity_ir), include_groups=False)
           .rename("capacity_aum").reset_index())
    aum.to_csv(out / "capacity_aum.csv", index=False)
    print("\nNet IR by wealth ($B), cost model and period:")
    print(cap.assign(wealth=cap["wealth"] / 1e9).pivot_table(index=["benchmark", "universe", "period", "costs"], columns="wealth",
                                                             values="ir_net").to_string(float_format=lambda v: f"{v:.3f}"))
    print(f"\nCapacity: AUM where net IR falls below {rep.capacity_ir} ($B; inf = not reached by $250B):")
    print(aum.assign(capacity_aum=aum["capacity_aum"] / 1e9).to_string(index=False, float_format=lambda v: f"{v:.1f}"))

    fig, axes = plt.subplots(1, 2, figsize=(10, 3.8), sharey=True)
    for ax, period in zip(axes, ["2020–2025", "2020–2025 (VIX subset)"]):
        for (name, model), g in cap[(cap["period"] == period) & (cap["benchmark"] == "paper index")].groupby(["universe", "costs"]):
            ax.plot(g["wealth"] / 1e9, g["ir_net"], marker="o", ms=3, lw=1.2,
                    ls="-" if model == "base" else "--", color="C0" if name == "80-stock" else "C3", label=f"{name}, {model}")
        ax.axhline(rep.capacity_ir, color="grey", lw=0.6, ls=":")
        ax.set(xscale="log", title=f"{period}, vs the paper index", xlabel="Wealth at end-2019 ($B, log scale)")
        ax.legend(fontsize=7)
    axes[0].set_ylabel(f"Net information ratio (TE budget {args.te:.0%})")
    fig.tight_layout()
    figs = cfg.paths.outputs / "figures"
    figs.mkdir(parents=True, exist_ok=True)
    for ext in ("png", "pdf"):
        fig.savefig(figs / f"capacity_ir.{ext}", dpi=200, bbox_inches="tight")
    plt.close(fig)


if __name__ == "__main__":
    main()


### `scripts/legacy_iterations/iteration_2/monitor.py`

Monitor a freshly fitted Kati test run.

**Role:** Pipeline implementation or configuration. This cell writes the file without running its main function.

**Input:** the embedded source below. **Output:** `scripts/legacy_iterations/iteration_2/monitor.py`. Expand the code cell to read or edit it.


In [ ]:
%%writefile scripts/legacy_iterations/iteration_2/monitor.py
"""Monitor a freshly fitted Kati test run."""
from __future__ import annotations

import argparse
from pathlib import Path
import sys

ROOT = Path(__file__).resolve().parents[2]
sys.path.insert(0, str(ROOT / "scripts"))
import pandas as pd
from jkmp.config import load_config
from jkmp.covariance import load_covariance
from jkmp.evaluation.metrics import active_returns, active_weights, ex_ante_exposures
from jkmp.evaluation.monitor import half_life_steps, monitor_table, plot_dashboard
from jkmp.models.return_model import monthly_rank_ic
from jkmp.imputation_audit import enable_reporting, reporting_enabled
if not reporting_enabled():
    enable_reporting(Path(__file__).name)

SELECTED_CONFIG = ROOT / "configs/us_large/iteration_2.yaml"


def main():
    parser = argparse.ArgumentParser(description=__doc__)
    parser.add_argument("--from", dest="start", default="2020-01-31")
    args = parser.parse_args()
    cfg = load_config(SELECTED_CONFIG)
    cal = pd.read_csv(cfg.paths.outputs / "active/calibrated.csv")
    gamma = float(cal.loc[(cal.te_target - 0.02).abs().lt(1e-12), "gamma_te"].iloc[0])
    run = cfg.paths.outputs / "active" / f"gte{gamma:g}"
    panel = pd.read_parquet(cfg.paths.processed / "panel.parquet")
    cov = load_covariance(cfg.paths.processed / "covariance")
    pf = pd.read_parquet(run / "pf.parquet")
    weights = pd.read_parquet(run / "weights.parquet")
    benchmark = pd.read_parquet(cfg.paths.outputs / "benchmarks/pf_test.parquet").query("type == 'Market'").assign(tc=0.0)
    active = active_returns(pf, benchmark, cfg.dates)
    exposure = ex_ante_exposures(active_weights(weights, panel), cov)
    ic = monthly_rank_ic(pd.read_parquet(cfg.paths.processed / "predictions.parquet"), panel, cfg.dates)
    chosen = pd.read_csv(run / "chosen.csv")
    hl = half_life_steps(chosen, cfg.portfolio_ml.half_life_years) if "h" in chosen else None
    # Real execution costs are unobserved: this check must stay missing, not substitute modelled costs.
    realised = pd.Series(float("nan"), index=pd.DatetimeIndex(pf.eom_ret))
    table = monitor_table(active, exposure, ic, realised, hl, cfg.monitor, cfg.dates)
    out = cfg.paths.outputs / "monitor"
    out.mkdir(parents=True, exist_ok=True)
    table.to_csv(out / "monitor.csv", index=False)
    shown = table[table.eom_ret.ge(pd.Timestamp(args.start))]
    plot_dashboard(shown, cfg.monitor, "Kati 2020-2025 test monitor; no separate holdout", out / "dashboard")


if __name__ == "__main__":
    main()


### `scripts/legacy_iterations/iteration_2/run.py`

iteration_2: original active Portfolio-ML engine; use calibrate/run/capacity/report as applicable.

**Role:** Pipeline implementation or configuration. This cell writes the file without running its main function.

**Input:** the embedded source below. **Output:** `scripts/legacy_iterations/iteration_2/run.py`. Expand the code cell to read or edit it.


In [ ]:
%%writefile scripts/legacy_iterations/iteration_2/run.py
"""iteration_2: original active Portfolio-ML engine; use calibrate/run/capacity/report as applicable."""
from pathlib import Path
import runpy,sys
ROOT=Path(__file__).resolve().parents[2]
if "--config" not in sys.argv:sys.argv += ["--config",str(ROOT/"configs/us_large/iteration_2.yaml")]
runpy.run_path(str(ROOT/"scripts/shared/active_engine.py"),run_name="__main__")


### `scripts/legacy_iterations/iteration_3/compare_halflife.py`

Trial 4 (docs/research_log.md): time-decayed β (configs/experiments/halflife.yaml) vs trial 3's 250-stock active Portfolio-ML — the pre-registered net-IR comparison against the paper index, the half-life chosen each year, active theme exposures over time, and net IR by decade. python scripts/14_halflife.py Writes outputs/u250_hl/halflife/{summary,chosen_h,by_decade}.csv and outputs/u250_hl/figures/halflife_exposures.{png,pdf}.

**Role:** Pipeline implementation or configuration. This cell writes the file without running its main function.

**Input:** the embedded source below. **Output:** `scripts/legacy_iterations/iteration_3/compare_halflife.py`. Expand the code cell to read or edit it.


In [ ]:
%%writefile scripts/legacy_iterations/iteration_3/compare_halflife.py
"""Trial 4 (docs/research_log.md): time-decayed β (configs/experiments/halflife.yaml) vs trial 3's 250-stock active
Portfolio-ML — the pre-registered net-IR comparison against the paper index, the half-life chosen each year,
active theme exposures over time, and net IR by decade.

    python scripts/14_halflife.py

Writes outputs/u250_hl/halflife/{summary,chosen_h,by_decade}.csv and outputs/u250_hl/figures/halflife_exposures.{png,pdf}.
"""

from __future__ import annotations

from pathlib import Path as _ScriptPath
import sys as _script_sys
_script_sys.path.insert(0,str(_ScriptPath(__file__).resolve().parents[1]))
from _bootstrap import configure
configure(__file__)

import argparse
import logging

import numpy as np
import pandas as pd

from jkmp.config import PROJECT_ROOT, load_config
from jkmp.covariance import load_covariance
from jkmp.evaluation.active import plot_active_exposures
from jkmp.evaluation.metrics import active_returns, active_stats, active_weights, ex_ante_exposures, prob_greater

REF, VAR = "configs/us_large/iteration_2.yaml", "configs/us_large/iteration_3.yaml"


def main() -> None:
    parser = argparse.ArgumentParser(description=__doc__, formatter_class=argparse.RawDescriptionHelpFormatter)
    parser.parse_args()
    logging.basicConfig(level=logging.INFO, format="%(asctime)s %(levelname)s %(name)s: %(message)s")
    ref, var = load_config(PROJECT_ROOT / REF), load_config(PROJECT_ROOT / VAR)
    dates = var.dates
    cal = pd.read_csv(ref.paths.outputs / "active/calibrated.csv")
    g = float(cal.loc[(cal.te_target - 0.02).abs().lt(1e-12), "gamma_te"].iloc[0])
    panel = pd.read_parquet(var.paths.processed / "panel.parquet")
    cov = load_covariance(var.paths.processed / "covariance")
    bench = (pd.read_parquet(ref.paths.outputs / "benchmarks" / "pf_test.parquet").query("type == 'Market'")
             .assign(tc=0.0))  # paper index (trial 3 decision)
    runs = {"trial 3 (h = ∞)": ref.paths.outputs / "active" / f"gte{g:g}",
            "trial 4 (h tuned)": var.paths.outputs / "active" / f"gte{g:g}"}
    out = var.paths.outputs / "halflife"
    out.mkdir(parents=True, exist_ok=True)
    pd.set_option("display.width", 220)

    stats, scaled, decades, exposures = {}, {}, {}, []
    for name, d in runs.items():
        pf, w = pd.read_parquet(d / "pf.parquet"), pd.read_parquet(d / "weights.parquet")
        stats[name] = active_stats(pf, bench, w, panel, cov, g, dates)
        act = active_returns(pf, bench, dates)
        scaled[name] = (act["active_net"] / act["active"].std(ddof=1)).reset_index(drop=True)
        dec = act["eom_ret"].dt.year // 10 * 10
        decades[name] = act.groupby(dec).apply(
            lambda x: x["active_net"].mean() / x["active"].std(ddof=1) * np.sqrt(12), include_groups=False)
        exposures.append(ex_ante_exposures(active_weights(w, panel), cov).assign(type=name))
    p_improve = prob_greater(scaled["trial 4 (h tuned)"], scaled["trial 3 (h = ∞)"])
    summary = pd.DataFrame(stats).T
    summary.to_csv(out / "summary.csv")
    by_decade = pd.DataFrame(decades).rename_axis("decade")
    by_decade.to_csv(out / "by_decade.csv")

    chosen = pd.read_csv(runs["trial 4 (h tuned)"] / "chosen.csv")
    test = chosen[chosen["year"].between(dates.test_start.year, dates.test_end.year)]
    share_inf = float(np.isinf(test["h"]).mean())
    test[["year", "tag", "p", "l", "h"]].to_csv(out / "chosen_h.csv", index=False)
    exp = pd.concat(exposures, ignore_index=True)
    written = plot_active_exposures(exp, var.paths.outputs / "figures" / "halflife_exposures")
    logging.info("wrote %s", ", ".join(p.name for p in written))

    print("Active Portfolio-ML, 250-stock, 2% budget, vs the paper index, 2020–2025:")
    print(summary[["active_net", "te_ex_ante", "te_realised", "ir_net", "p_ir_net_pos", "beta", "tc"]]
          .to_string(float_format=lambda v: f"{v:.4f}"))
    print("\nNet IR by decade:")
    print(by_decade.to_string(float_format=lambda v: f"{v:.3f}"))
    print("\nChosen half-life by return year (inf = no decay):")
    print(test.groupby("h")["year"].apply(lambda s: f"{len(s)} years: {s.min()}–{s.max()}").to_string())
    print(" ".join(f"{y}:{'∞' if np.isinf(h) else int(h)}" for y, h in zip(test["year"], test["h"])))
    exp["decade"] = (exp["eom"] + pd.offsets.MonthEnd(1)).dt.year // 10 * 10
    print("\nMean active low-risk / quality exposure by decade:")
    print(exp.pivot_table(index="decade", columns="type", values=["theme_low_risk", "theme_quality"])
          .to_string(float_format=lambda v: f"{v:.3f}"))
    killed_inf = share_inf > 0.5
    keep = p_improve >= 0.80 and not killed_inf
    print(f"\nP(net IR trial 4 > trial 3) = {p_improve:.4f}; h = ∞ chosen in {share_inf:.0%} of test years "
          f"-> {'KEEP' if keep else 'KILL'} (pre-registered: P >= 0.80 and h = ∞ in <= 50% of years)")


if __name__ == "__main__":
    main()


### `scripts/legacy_iterations/iteration_3/run.py`

iteration_3: original active Portfolio-ML engine; use calibrate/run/capacity/report as applicable.

**Role:** Pipeline implementation or configuration. This cell writes the file without running its main function.

**Input:** the embedded source below. **Output:** `scripts/legacy_iterations/iteration_3/run.py`. Expand the code cell to read or edit it.


In [ ]:
%%writefile scripts/legacy_iterations/iteration_3/run.py
"""iteration_3: original active Portfolio-ML engine; use calibrate/run/capacity/report as applicable."""
from pathlib import Path
import runpy,sys
ROOT=Path(__file__).resolve().parents[2]
if "--config" not in sys.argv:sys.argv += ["--config",str(ROOT/"configs/us_large/iteration_3.yaml")]
runpy.run_path(str(ROOT/"scripts/shared/active_engine.py"),run_name="__main__")


## Source — Reporting and statistical comparisons


### `scripts/analysis/__init__.py`

Supporting implementation for the Kati workflow.

**Role:** Reporting source. Reads available local results when called; this cell writes the module.

**Input:** the embedded source below. **Output:** `scripts/analysis/__init__.py`. Expand the code cell to read or edit it.


In [ ]:
%%writefile scripts/analysis/__init__.py
# Package marker for the isolated Kati pipeline.


### `scripts/analysis/ml_comparison/__init__.py`

Read-only evaluation of saved Kati portfolios; never fits a model.

**Role:** Reporting source. Reads available local results when called; this cell writes the module.

**Input:** the embedded source below. **Output:** `scripts/analysis/ml_comparison/__init__.py`. Expand the code cell to read or edit it.


In [ ]:
%%writefile scripts/analysis/ml_comparison/__init__.py
"""Read-only evaluation of saved Kati portfolios; never fits a model."""


### `scripts/analysis/ml_comparison/cli.py`

Compare US high-cap Static ML* and the three regional base Portfolio ML models.

**Role:** Reporting source. Reads available local results when called; this cell writes the module.

**Input:** the embedded source below. **Output:** `scripts/analysis/ml_comparison/cli.py`. Expand the code cell to read or edit it.


In [ ]:
%%writefile scripts/analysis/ml_comparison/cli.py
"""Compare US high-cap Static ML* and the three regional base Portfolio ML models."""
from __future__ import annotations
import argparse
import html
import json
from pathlib import Path
import sys

import numpy as np
import pandas as pd

KATI=Path(__file__).resolve().parents[3]
ROOT=KATI
COMPARISON=KATI/'outputs/base'
sys.path.insert(0,str(KATI/'scripts'))
from jkmp.config import load_config
from jkmp.models.return_model import cutoffs_for_year
from jkmp.strategies import STRATEGIES
from .data import discover,align,sha256
from .metrics import table,annual,paths_and_rolling,metric_dictionary
from .inference import compare_pairs,bootstrap_rank_stability


TIMING='''# Training, testing and out-of-sample timing

| Item | Current behavior |
|---|---|
| Initial training return years | 2005-2014 |
| Declared validation return years | 2015-2019 |
| Headline test return years | 2020-2025 |
| Broader return-forecast OOS history | 2015-2025; this overlaps validation and test |
| Separate unseen holdout | None available after the saved test |
| Market (iteration reference only) | No ML fitting; market-cap weights updated monthly |
| Static ML* (US high-cap only) | Annual expanding return forecasts; k/u/g selected each December from cumulative preceding utility and applied to the next return year; monthly weights |
| Portfolio ML | Coefficients refitted annually using past return years; variance/model-size/ridge candidates selected from cumulative past utility each December; weights updated monthly |

The models are **not frozen after 2014 or 2019**. For example, the 2020 return forecaster is fitted through December 2019 returns, while the 2021 fit includes 2020 returns. This is an expanding walk-forward backtest: a completed test year may train a later year's model. No future return from the forecast year enters that year's coefficient fit. Hyperparameter candidates are internal choices of a retained strategy, not separate reported strategies.

The nominal split does not mean every estimator has usable inputs from January 2005. Portfolio ML needs sufficient daily covariance and lag history; the actual start of each newly fitted moment history is reported in annual_training_schedule.csv. Static ML's return forecaster uses available valid regional training observations beginning in 2005. Later fits expand each estimator's available past history through the prior return year. These dates must be read from Kati's new caches, not inherited from Keats' old-feature results.

The current characteristic recommendation uses only US high-cap observations from 2005–2014, ranked by association statistics with group representation and redundancy controls. No regression or ML model is fitted to select these inputs. No 2015–2025 return outcome enters this selection. The existing ML implementation remains available for explicit future fitting, with headline evaluation in 2020–2025 and annual expanding refits.

Return-forecast tuning for year Y uses an earlier training segment and the trailing five available years as internal validation, then refits the selected candidate on all data through year Y-1. For the first 2015 forecast, internal training is 2005-2009, internal validation is 2010-2014, and the final refit uses 2005-2014. The schedule moves forward annually.

Static ML* validates fixed k/u/g candidates from January 2015, chooses each next return year using completed preceding Decembers, and starts its test portfolio with value weights in January 2020. Later choices may use completed earlier test years. US mid-cap and EU high-cap have Portfolio ML and Market only. The existing iterations retain their historical fixed Static ML reference.

Portfolio ML's candidate utility history begins in 2015; the first December candidate choice supports aims from 2016. The saved headline portfolio is evaluated in 2020-2025. There is no separate third dataset named OOS beyond the test: the existing OOS label denotes historical forward predictions, not an additional unseen holdout. The scripts are finite backtests, not a continuously running live service.

The saved candidate-choice table can include a prospective 2026 choice based on completed 2025 observations. No 2026 fitted return forecast, portfolio return or performance is generated by this workflow.
'''


def verify(inventory,monthly,stats,out):
    findings=[]
    def check(name,scope,passed,detail=''):
        findings.append(dict(check=name,scope=scope,status='SKIPPED' if passed is None else ('PASS' if passed else 'FAIL'),detail=detail))
    baseline_path=KATI/'outputs/model_cleanup/original_retained_test_returns.parquet'
    baseline=pd.read_parquet(baseline_path) if baseline_path.exists() else None
    for row in inventory.itertuples():
        universe,strategy=row.universe,row.strategy
        pf=monthly.loc[(monthly.universe==universe)&(monthly.strategy==strategy)].sort_values('eom').reset_index(drop=True)
        fields=['r','tc','turnover','inv','shorting']
        if baseline is not None and strategy != "Static-ML*":
            original=baseline.loc[(baseline.universe==universe)&(baseline.type==strategy)].sort_values('eom').reset_index(drop=True)
            same=len(pf)==len(original) and pf.eom.equals(original.eom) and np.allclose(pf[fields],original[fields],rtol=1e-10,atol=1e-12)
            check('retained_strategy_matches_original',f'{universe}/{strategy}',same,'Original 72-month performance preserved within numerical tolerance')
        else:
            check('retained_strategy_matches_original',f'{universe}/{strategy}',None,'No pre-cleanup reference file in this checkout; independent accounting still runs')
        weights=pd.read_parquet(row.weights_path)
        check('weights_have_only_the_retained_method',f'{universe}/{strategy}',set(weights.type)=={strategy})
        panel=pd.read_parquet(Path(row.processed_path)/'panel.parquet',columns=['id','eom','ret_ld1','lambda'])
        wealth=pd.read_parquet(Path(row.processed_path)/'wealth.parquet').set_index('eom').wealth
        joined=weights.merge(panel,on=['id','eom'],how='left',validate='one_to_one')
        held=~joined['exit']
        complete=joined['lambda'].notna().all() and joined.loc[held,'ret_ld1'].notna().all()
        check('accounting_source_coverage',f'{universe}/{strategy}',complete)
        joined['r_i']=joined.w*joined.ret_ld1.where(held,0.)
        joined['tc_i']=.5*joined.eom.map(wealth)*joined['lambda']*(joined.w-joined.w_start)**2
        joined['turnover_i']=(joined.w-joined.w_start).abs()
        joined['inv_i']=joined.w.abs();joined['short_i']=-joined.w.clip(upper=0)
        recalculated=joined.groupby('eom').agg(r=('r_i','sum'),tc=('tc_i','sum'),turnover=('turnover_i','sum'),inv=('inv_i','sum'),shorting=('short_i','sum')).reindex(pf.eom)
        check('independent_weight_return_cost_accounting',f'{universe}/{strategy}',np.allclose(recalculated[fields],pf[fields],rtol=1e-10,atol=1e-12))
        r=pf.r.to_numpy();tc=pf.tc.to_numpy();n=len(r);mean=sum(r)/n;variance=sum((x-mean)**2 for x in r)/(n-1)
        cost=sum(tc)/n
        expected=dict(R=12*mean,TC=12*cost,R_TC=12*(mean-cost),Vol=np.sqrt(12*variance),Utility=12*(mean-cost-5*variance))
        observed=stats.loc[(stats.dataset==f'{universe}/base')&(stats.strategy==strategy)].iloc[0]
        check('independent_report_formulas',f'{universe}/{strategy}',all(np.isclose(observed[k],v,rtol=1e-11,atol=1e-11) for k,v in expected.items()))
    hash_path=KATI/'outputs/model_cleanup/retained_input_hashes_before.csv'
    if hash_path.exists():
        for row in pd.read_csv(hash_path).itertuples():
            check('retained_input_hash_unchanged',row.path,sha256(KATI/row.path)==row.sha256)
    else:
        check('retained_input_hash_unchanged','pre-cleanup baseline',None,'No pre-cleanup hash manifest in this checkout')
    for universe in inventory.universe.unique():
        cfg=load_config(KATI/f'configs/{universe}/base.yaml')
        expected=json.loads((KATI/'data/characteristics/recommended_50.json').read_text())
        check('recommended_50_inputs_active',universe,len(cfg.features.names)==50 and set(cfg.features.names)==set(expected))
    result=pd.DataFrame(findings);result.to_csv(out/'independent_verification.csv',index=False)
    if result.status.eq('FAIL').any():raise AssertionError('Independent base verification failed; inspect independent_verification.csv')
    return result


def timing_schedule(universes):
    records=[]
    for universe in universes:
        processed=load_config(KATI/f'configs/{universe}/base.yaml').paths.processed
        folders=sorted((processed/'pfml').glob('rff_g*'))
        first_moments=[min(p.glob('????-??-??.npz')).stem for p in folders]
        if not first_moments or len(set(first_moments))!=1:raise ValueError('Default moment candidates have different history starts')
        first_return=pd.Timestamp(first_moments[0])+pd.offsets.MonthEnd(1)
        for year in range(2015,2026):
            train,full=cutoffs_for_year(year,5)
            records.append(dict(universe=universe,forecast_return_year=year,internal_training_last_return_month=train+pd.offsets.MonthEnd(1),
                internal_validation_last_return_month=full+pd.offsets.MonthEnd(1),final_refit_last_return_month=full+pd.offsets.MonthEnd(1),
                return_forecaster_first_allowed_training_return_month='2005-01-31',portfolio_ml_first_available_training_return_month=first_return,
                portfolio_ml_coefficient_last_allowed_return_year=year-1,test_year_updates_use_previously_realised_test_data=year>=2021))
    return pd.DataFrame(records)


def charts(monthly,stats,out):
    import matplotlib
    matplotlib.use('Agg')
    import matplotlib.pyplot as plt
    folder=out/'charts';folder.mkdir(exist_ok=True)
    for dataset,g in monthly.groupby('dataset'):
        fig,axes=plt.subplots(1,2,figsize=(12,4.5))
        for strategy,d in g.groupby('strategy'):
            d=d.sort_values('eom_ret')
            axes[0].plot(d.eom_ret,100*d.r.cumsum(),label=strategy)
            axes[1].plot(d.eom_ret,100*d.tc.cumsum(),label=strategy)
        axes[0].set(title='Cumulative gross excess returns (arithmetic)',ylabel='Percentage points')
        axes[1].set(title='Cumulative modelled trading costs',ylabel='Percentage points')
        if g.tc.max()>1:axes[1].set_yscale('symlog',linthresh=1);axes[1].set_title('Cumulative modelled trading costs (symlog)')
        for ax in axes:ax.grid(alpha=.2);ax.legend(fontsize=8)
        fig.suptitle(dataset+' | test return years 2020-2025');fig.tight_layout()
        for ext in ('png','pdf'):fig.savefig(folder/(dataset.replace('/','__')+'.'+ext),dpi=180)
        plt.close(fig)


def main(argv=None):
    parser=argparse.ArgumentParser(description=__doc__)
    parser.add_argument('--selected-universe',choices=['all','us_large','us_mid','eu_large'],default='all')
    parser.add_argument('--bootstrap-reps',type=int,default=2000)
    args=parser.parse_args(argv)
    universes=['us_large','us_mid','eu_large'] if args.selected_universe=='all' else [args.selected_universe]
    out=COMPARISON/'results';out.mkdir(parents=True,exist_ok=True)
    inventory,raw=discover(ROOT,universes);monthly,across,alignment=align(raw)
    inventory.to_csv(out/'source_inventory.csv',index=False);monthly.to_csv(out/'monthly_returns.csv',index=False)
    alignment.to_csv(out/'date_alignment_exclusions.csv',index=False)
    stats=table(monthly);stats.to_csv(out/'performance_summary.csv',index=False)
    annual(monthly).to_csv(out/'annual_performance.csv',index=False)
    paths_and_rolling(monthly).to_csv(out/'rolling_and_cumulative.csv',index=False)
    metric_dictionary().to_csv(out/'metric_dictionary.csv',index=False)
    pairs=[]
    for block in [3,6,12]:
        p=compare_pairs(monthly,bootstrap_reps=args.bootstrap_reps,block_length=block,seed=20261101)
        p['bootstrap_block_months']=block;pairs.append(p)
    pd.concat(pairs,ignore_index=True).to_csv(out/'paired_inference.csv',index=False)
    bootstrap_rank_stability(monthly,bootstrap_reps=args.bootstrap_reps,seed=20261101).to_csv(out/'bootstrap_rank_stability.csv',index=False)
    corr=[]
    for dataset,g in monthly.groupby('dataset'):
        matrix=g.pivot(index='eom_ret',columns='strategy',values='r').corr()
        matrix.to_csv(out/(dataset.replace('/','__')+'_return_correlations.csv'))
    scenarios=[]
    for multiple in [0.,.5,1.,2.,3.]:
        d=monthly.copy();d['tc']=d.tc*multiple
        scenarios.append(table(d).assign(cost_multiplier=multiple,weights_reoptimised=False))
    pd.concat(scenarios,ignore_index=True).to_csv(out/'cost_sensitivity.csv',index=False)
    timing_schedule(universes).to_csv(out/'annual_training_schedule.csv',index=False)
    (COMPARISON/'TRAIN_TEST_AND_OOS.md').write_text(TIMING,encoding='utf-8')
    checks=verify(inventory,monthly,stats,out)
    charts(monthly,stats,out)
    assumptions='''# Assumptions, exclusions and changes

The base report contains Static ML* only for US high-cap and default Fourier-signal Portfolio ML for US high-cap, US mid-cap and European high-cap, freshly fitted with the frozen suggested 50 from Characteristics_tryout. The original memberships, return and liquidity infrastructure, portfolio mathematics and annual expanding timing are retained. Covariance and model caches are rebuilt for the new inputs.

Base covers all three markets. Iterations 1–5 apply only to US high-cap Portfolio ML. The 50 inputs are selected exclusively from US high-cap 2005–2014 association data, within a whitelist verified against both published JKP reference files and WRDS source availability. All 50 have published group labels and directions. Raw ML ranks are unflipped. Earlier-feature fits have been removed; current-feature performance requires new fits.

This reporting command reads only completed Kati outputs; fitting is performed by the upstream model stages. It independently reconstructs returns and costs from saved weights. Input hashes, membership preservation, feature ranks and theme exposures are checked separately by scripts/verify.py. No equality with old-feature Keats performance is claimed.

The user-selected zero-excess-return assumption remains for 15 US-large, 21 US-mid and 20 European saved held-stock observations over full history. They are flagged assumptions, not observed zeros. No original 80-stock membership is removed by the separate unavailable-lag-history rule. Missing characteristics retain the original neutral rank 0.5 convention. Existing daily-data currency, source-vintage and terminal-return limitations remain; the cleanup does not establish point-in-time or survivorship-free data.

All headline methods use the same 72 return months, January 2020 through December 2025, separately within each market. No returns or stocks are pooled across regions. OOS overlaps test; no additional unseen holdout is claimed. Static ML* uses 2015-onward candidate validation and value-weighted test-start initialization; Market and Portfolio ML retain their original test-start initialization.

Paper-convention annual mean return, cost and utility multiply monthly figures by 12. Volatility multiplies gross-return SD by sqrt(12); net Sharpe uses gross-return SD. Utility uses gamma=10 and sample gross variance. Monthly turnover is two-sided with no half factor. Cumulative returns use cumsum(r-tc), matching the paper replication charts. Additive drawdown equals that path minus its running peak including initial zero, in percentage points after multiplication by 100. Drawdown is an explicitly labelled reporting extension, not total-wealth loss. CAGR and geometric Calmar are omitted. No return is silently removed.

HAC tests use Bartlett/Newey-West covariances with normal-reference inference. Holm correction is within each market's pair family. Paired circular block bootstrap uses 2,000 draws, seed 20261101 and 3/6/12-month blocks. These intervals do not correct all prior research/model selection. The paper-style utility probability is an approximation, not a HAC-robust posterior or frequentist p-value.

Cost sensitivities scale modelled costs at fixed weights and do not re-optimise. Unconstrained weights and quadratic impact can imply infeasible turnover or losses; arithmetic results in those cases are diagnostics, not executable investor wealth or proven profits. Net performance and economic utility must be read together with cost, leverage and feasibility flags. Regimes remain descriptive and use cached current-vintage US/global context even for Europe.
'''
    (COMPARISON/'EXCLUSIONS_ASSUMPTIONS_AND_CHANGES.md').write_text(assumptions,encoding='utf-8')
    show=stats[['dataset','strategy','n','R','TC','R_TC','Vol','SR_net','Utility','Lev','Turnover']].copy()
    for c in ['R','TC','R_TC','Vol','Utility']:show[c]=100*show[c]
    show=show.rename(columns={'R':'Gross return %','TC':'Cost %','R_TC':'Net return %','Vol':'Volatility %','Utility':'Utility %'})
    description='''<h1>Kati base: US high-cap Static ML* and regional Portfolio ML</h1><p>Test return months: January 2020–December 2025. Each region is evaluated separately. Returns, costs, volatility and utility below are annualised percentages; leverage and turnover are ratios.</p><p><strong>Timing:</strong> annual expanding refits and monthly weight updates. Models are not frozen after the initial training period. OOS is the broader forward-prediction history, not a separate unseen holdout.</p><h2>Current results</h2>'''
    economic='''<h2>Economic interpretation</h2><p>Read net return and utility after costs alongside gross return, leverage and turnover. High gross forecasting performance can be outweighed by quadratic trading costs. Large estimated costs indicate infeasibility under the existing unconstrained implementation; removing other strategies does not fix that constraint problem. These are new-feature walk-forward backtests, subject to the documented feature-selection and data-vintage limits.</p><p>US high-cap Static ML* converts annual return forecasts into monthly cost-aware holdings with k/u/g parameters selected each December using preceding utility. Portfolio ML estimates signal coefficients against a risk-and-trading-cost objective and moves monthly holdings toward its aim portfolio. Differences measure these existing portfolio rules; they do not isolate a causal benefit of ML.</p>'''
    links='''<p><a href="../index.html">Base and all five iteration groups</a></p><p>Market folders: <a href="us_large/index.html">US high cap</a> | <a href="us_mid/index.html">US mid cap</a> | <a href="eu_large/index.html">European high cap</a></p><h2>Files</h2><ul><li><a href="TRAIN_TEST_AND_OOS.md">Training, test and OOS timing</a></li><li><a href="EXCLUSIONS_ASSUMPTIONS_AND_CHANGES.md">Assumptions, exclusions and changes</a></li><li><a href="results/performance_summary.csv">Full performance table and feasibility flags</a></li><li><a href="results/paired_inference.csv">Paired HAC tests and block-bootstrap intervals</a></li><li><a href="results/independent_verification.csv">Independent accounting and preservation checks</a></li></ul>'''
    style='<style>body{max-width:1300px;margin:36px auto;padding:0 20px;font:15px/1.5 system-ui;color:#21323e}table{border-collapse:collapse;font-size:13px}td,th{border:1px solid #ced8df;padding:8px}th{background:#e7eff4}.table{overflow-x:auto}img{width:100%;margin:20px 0}</style>'
    images=''.join(f'<img src="results/charts/{u}__base.png" alt="{u} gross returns and trading costs">' for u in universes)
    (COMPARISON/'index.html').write_text('<!doctype html><meta charset="utf-8"><title>Kati base results</title>'+style+description+'<div class="table">'+show.to_html(index=False,float_format=lambda x:f'{x:,.4g}')+'</div>'+economic+links+images,encoding='utf-8')
    (COMPARISON/'START_HERE.md').write_text('''# Base results: US high-cap Static ML* and regional Portfolio ML

Open [index.html](index.html) for the readable result tables, economic interpretation and charts.

- [Performance and feasibility flags](results/performance_summary.csv)
- [Annual results](results/annual_performance.csv)
- [Training, test and OOS explanation](TRAIN_TEST_AND_OOS.md)
- [Assumptions, exclusions and changes](EXCLUSIONS_ASSUMPTIONS_AND_CHANGES.md)
- [Independent verification](results/independent_verification.csv)
- [Paired inference](results/paired_inference.csv)

Base contains US high-cap Static ML* and default Portfolio ML for all three markets. US mid-cap and EU high-cap have no Static ML strategy. New iteration results are separated into ../iteration_1, ../iteration_2 and ../iteration_3; unfinished runs are labelled explicitly.

Run `python Kati_tryout/scripts/base/compare_results.py` to refresh the saved-output comparison. This comparison does not fit any models. Use `python Kati_tryout/scripts/run_all.py --group base --scope full` for the dependency-ordered workflow.
''',encoding='utf-8')
    summary=dict(status='complete',strategies=list(STRATEGIES),universes=universes,series=len(inventory),monthly_rows=len(monthly),
                 test_months_per_series=72,independent_checks=len(checks),all_checks_pass=True,separate_holdout_available=False,
                 annually_refitted=True,recommended_50_active=True,old_keats_results_imported=False)
    (out/'completion.json').write_text(json.dumps(summary,indent=2),encoding='utf-8')
    print(json.dumps(summary,indent=2));return 0


if __name__=='__main__':raise SystemExit(main())


### `scripts/analysis/ml_comparison/data.py`

Only the three retained base strategies, with strict monthly alignment.

**Role:** Reporting source. Reads available local results when called; this cell writes the module.

**Input:** the embedded source below. **Output:** `scripts/analysis/ml_comparison/data.py`. Expand the code cell to read or edit it.


In [ ]:
%%writefile scripts/analysis/ml_comparison/data.py
"""Only the three retained base strategies, with strict monthly alignment."""
from pathlib import Path
import hashlib
import numpy as np
import pandas as pd
from jkmp.config import load_config
from jkmp.strategies import base_result_paths,require_strategies

def sha256(path: Path) -> str:
    h = hashlib.sha256()
    with path.open('rb') as stream:
        for block in iter(lambda: stream.read(1024 * 1024), b''):
            h.update(block)
    return h.hexdigest()


def align(monthly: pd.DataFrame):
    """Common months per dataset, and then across datasets; never fill absent returns."""
    if monthly.empty:
        raise ValueError('No performance observations to align.')
    monthly=monthly.copy()
    monthly['eom_ret']=pd.to_datetime(monthly.eom_ret)
    if monthly[['dataset','strategy','eom_ret']].isna().any().any():
        raise ValueError('Missing dataset, strategy or return month.')
    if monthly.duplicated(['dataset','strategy','eom_ret']).any():
        raise ValueError('Duplicate dataset/strategy/return-month observations.')
    if not monthly.eom_ret.dt.is_month_end.all():
        raise ValueError('Saved monthly returns must be dated at month-end.')
    aligned, logs = [], []
    for dataset, group in monthly.groupby('dataset', sort=False):
        date_sets = [set(g.eom_ret) for _, g in group.groupby('strategy')]
        common = set.intersection(*date_sets)
        if len(common) < 12:
            raise ValueError(f'Fewer than 12 common observations: {dataset}')
        ordered=pd.DatetimeIndex(sorted(common)).to_period('M').asi8
        if not np.all(np.diff(ordered)==1):
            raise ValueError(f'Calendar gaps in common monthly sample: {dataset}; no missing return is fabricated.')
        for strategy, g in group.groupby('strategy'):
            drop = g.loc[~g.eom_ret.isin(common)]
            logs.append(dict(dataset=dataset, strategy=strategy, original_months=len(g),
                common_months=len(common), excluded_months=len(drop),
                excluded_dates=';'.join(drop.eom_ret.dt.strftime('%Y-%m-%d'))))
        aligned.append(group.loc[group.eom_ret.isin(common)])
    within = pd.concat(aligned, ignore_index=True).sort_values(['dataset','strategy','eom_ret'])
    common_all = set.intersection(*(set(g.eom_ret) for _, g in within.groupby('dataset')))
    across = within.loc[within.eom_ret.isin(common_all)].copy()
    return within, across, pd.DataFrame(logs)



def discover(root,universes=None):
    frames=[];inventory=[]
    for universe in universes or ("us_large","us_mid","eu_large"):
        cfg=load_config(root/"configs"/universe/"base.yaml")
        output=cfg.paths.outputs
        paths=[("Static-ML*" if p.parent.name=="static_ml_star" else "Portfolio-ML",p,p.parent/"weights.parquet") for p in base_result_paths(output)]
        for strategy,pf_path,weight_path in paths:
            pf=pd.read_parquet(pf_path)
            if set(pf.type)!={strategy}:raise ValueError(f"Unexpected strategy in {pf_path}")
            pf.eom=pd.to_datetime(pf.eom);pf.eom_ret=pd.to_datetime(pf.eom_ret)
            expected=pd.date_range(cfg.dates.test_start,cfg.dates.test_end,freq="ME")
            if not pd.DatetimeIndex(sorted(pf.eom_ret)).equals(expected):raise ValueError("Incomplete test calendar")
            if not (pf.eom+pd.offsets.MonthEnd(1)).eq(pf.eom_ret).all():raise ValueError("Formation/return mismatch")
            require_strategies(pf,universe=universe)
            frames.append(pf.assign(dataset=f"{universe}/base",strategy=strategy,universe=universe))
            inventory.append(dict(universe=universe,strategy=strategy,pf_path=str(pf_path),weights_path=str(weight_path),
                                  processed_path=str(cfg.paths.processed),sha256=sha256(pf_path)))
        require_strategies(pd.concat([f for f in frames if f.universe.iloc[0]==universe]),complete=True,universe=universe)
    return pd.DataFrame(inventory),pd.concat(frames,ignore_index=True)


### `scripts/analysis/ml_comparison/inference.py`

Paired, saved-output inference; this module never fits an investment model. Paper formulas are from ``reference/jkmp_r/6 - Base analysis.R``, lines 25-40 and 102-115. Its Table 3 normal probability is labelled a *paper posterior approximation*, not a frequentist p-value or an autocorrelation-robust posterior. For Table 2 utility U=12[mean(r-tc)-gamma*var(r,ddof=1)/2], the finite-sample HAC flow uses n/(n-1)*(r-mean(r))**2. Paper Table 3 instead uses the uncorrected centered square. Both differences are retained so this small mismatch is visible

**Role:** Reporting source. Reads available local results when called; this cell writes the module.

**Input:** the embedded source below. **Output:** `scripts/analysis/ml_comparison/inference.py`. Expand the code cell to read or edit it.


In [ ]:
%%writefile scripts/analysis/ml_comparison/inference.py
"""Paired, saved-output inference; this module never fits an investment model.

Paper formulas are from ``reference/jkmp_r/6 - Base analysis.R``, lines 25-40
and 102-115. Its Table 3 normal probability is labelled a *paper posterior
approximation*, not a frequentist p-value or an autocorrelation-robust posterior.

For Table 2 utility U=12[mean(r-tc)-gamma*var(r,ddof=1)/2], the finite-sample
HAC flow uses n/(n-1)*(r-mean(r))**2. Paper Table 3 instead uses the uncorrected
centered square. Both differences are retained so this small mismatch is visible.

HAC uses Bartlett/Newey-West covariances and a normal approximation. Net Sharpe
means sqrt(12)*mean(r-tc)/sd(r,ddof=1), using gross-return volatility as in R.
Its paired HAC test uses the delta-method influence function. Confidence
intervals use paired circular moving-block bootstrap samples and recompute
sample means, variances, and Sharpe ratios in every draw. These are percentile
intervals, not studentized intervals. They are research-sample uncertainty,
not protection against prior model selection, survivorship, or missing data.
"""

from __future__ import annotations

from itertools import combinations

import numpy as np
import pandas as pd
from scipy.stats import norm, rankdata


def holm_adjust(pvalues: np.ndarray) -> np.ndarray:
    """Holm family-wise adjustment, preserving NaNs and input ordering."""
    pvalues = np.asarray(pvalues, dtype=float)
    result = np.full(pvalues.shape, np.nan)
    valid = np.flatnonzero(np.isfinite(pvalues))
    if not len(valid):
        return result
    ordered = valid[np.argsort(pvalues[valid], kind="stable")]
    adjusted = np.maximum.accumulate((len(ordered) - np.arange(len(ordered))) * pvalues[ordered])
    result[ordered] = np.minimum(adjusted, 1.0)
    return result


def newey_west_mean(values: np.ndarray, lags: int | None = None) -> dict[str, float]:
    """Mean and HAC SE, using centered lag covariance sums divided by n.

    Default lag is floor(4*(n/100)**(2/9)), capped at n-1. No prewhitening or
    additional degrees-of-freedom multiplier is used. Constant differences
    produce SE zero: p=1 at zero, p=0 for a deterministic nonzero difference.
    """
    x = np.asarray(values, dtype=float)
    if x.ndim != 1 or len(x) < 2 or not np.isfinite(x).all():
        raise ValueError("HAC requires at least two finite observations")
    n = len(x)
    lag = min(n - 1, max(0, int(np.floor(4 * (n / 100) ** (2 / 9))) if lags is None else int(lags)))
    mean = float(x.mean())
    residual = x - mean
    variance = float(residual @ residual / n)
    for k in range(1, lag + 1):
        variance += 2 * (1 - k / (lag + 1)) * float(residual[k:] @ residual[:-k] / n)
    se = float(np.sqrt(max(variance, 0.0) / n))
    if se <= np.finfo(float).eps * max(1.0, abs(mean)):
        z = 0.0 if abs(mean) <= np.finfo(float).eps else np.sign(mean) * np.inf
        se = 0.0
    else:
        z = mean / se
    return {"mean": mean, "se": se, "z": float(z), "p": float(2 * norm.sf(abs(z))), "lags": lag}


def _normal_probability(mean: float, se: float) -> float:
    if not np.isfinite(mean) or not np.isfinite(se):
        return float("nan")
    if se <= np.finfo(float).eps * max(1.0, abs(mean)):
        return 0.5 if abs(mean) <= np.finfo(float).eps else float(mean > 0)
    return float(norm.cdf(mean / se))


def utility_flows(r: np.ndarray, tc: np.ndarray, gamma: float, sample_corrected: bool = False) -> np.ndarray:
    """Monthly centered-gross-variance flow; corrected flow averages to Table 2 U/12."""
    r, tc = np.asarray(r, dtype=float), np.asarray(tc, dtype=float)
    if len(r) < 2 or r.shape != tc.shape or not np.isfinite(r).all() or not np.isfinite(tc).all():
        raise ValueError("utility requires matching finite arrays with at least two months")
    factor = len(r) / (len(r) - 1) if sample_corrected else 1.0
    return r - tc - 0.5 * gamma * factor * (r - r.mean()) ** 2


def net_sharpe(r: np.ndarray, tc: np.ndarray) -> float:
    """Paper annualized SR: mean net divided by sample SD of gross monthly return."""
    r, tc = np.asarray(r, dtype=float), np.asarray(tc, dtype=float)
    sd = float(np.std(r, ddof=1))
    threshold = 64 * np.finfo(float).eps * max(1.0, float(np.max(np.abs(r))))
    return float(np.sqrt(12) * np.mean(r - tc) / sd) if sd > threshold else float("nan")


def sharpe_influence(r: np.ndarray, tc: np.ndarray) -> np.ndarray:
    """Plug-in influence values of annualized net/gross-SD Sharpe (mean zero).

    The n/(n-1) correction matches sample variance in the plug-in denominator.
    Valid for positive gross-return variance; returns NaN for constant gross r.
    """
    r, tc = np.asarray(r, dtype=float), np.asarray(tc, dtype=float)
    n = len(r)
    variance = float(r.var(ddof=1))
    threshold = 64 * np.finfo(float).eps * max(1.0, float(np.max(np.abs(r))))
    if n < 2 or variance <= threshold**2:
        return np.full(r.shape, np.nan)
    sd, mean_net = np.sqrt(variance), (r - tc).mean()
    squares = (r - r.mean()) ** 2
    variance_if = n / (n - 1) * (squares - squares.mean())
    return np.sqrt(12) * ((r - tc - mean_net) / sd - mean_net * variance_if / (2 * sd**3))


def circular_block_indices(n: int, replicates: int, block_length: int, rng: np.random.Generator) -> np.ndarray:
    """Paired circular blocks with random starts, concatenated and truncated to n."""
    if n < 2 or replicates < 1 or not 1 <= block_length <= n:
        raise ValueError("require n>=2, replicates>=1 and 1<=block_length<=n")
    blocks = int(np.ceil(n / block_length))
    starts = rng.integers(0, n, size=(replicates, blocks))
    return ((starts[..., None] + np.arange(block_length)) % n).reshape(replicates, -1)[:, :n]


def _bootstrap_differences(ra: np.ndarray, ta: np.ndarray, rb: np.ndarray, tb: np.ndarray,
                           gamma: float, reps: int, block: int, rng: np.random.Generator) -> tuple[np.ndarray, np.ndarray]:
    indices = circular_block_indices(len(ra), reps, block, rng)
    a, ac, b, bc = ra[indices], ta[indices], rb[indices], tb[indices]
    utility = 12 * ((a - ac).mean(axis=1) - (b - bc).mean(axis=1)
                    - 0.5 * gamma * (a.var(axis=1, ddof=1) - b.var(axis=1, ddof=1)))
    sd_a, sd_b = a.std(axis=1, ddof=1), b.std(axis=1, ddof=1)
    with np.errstate(divide="ignore", invalid="ignore"):
        sharpe = np.sqrt(12) * ((a - ac).mean(axis=1) / sd_a - (b - bc).mean(axis=1) / sd_b)
    threshold_a = 64 * np.finfo(float).eps * np.maximum(1.0, np.max(np.abs(a), axis=1))
    threshold_b = 64 * np.finfo(float).eps * np.maximum(1.0, np.max(np.abs(b), axis=1))
    sharpe[(sd_a <= threshold_a) | (sd_b <= threshold_b)] = np.nan
    return utility, sharpe


def _interval(values: np.ndarray) -> tuple[float, float]:
    valid = values[np.isfinite(values)]
    if not len(valid):
        return float("nan"), float("nan")
    low, high = np.quantile(valid, [0.025, 0.975])
    return float(low), float(high)


def compare_pairs(pf: pd.DataFrame, gamma: float = 10.0, bootstrap_reps: int = 2000,
                  block_length: int = 6, seed: int = 20261101) -> pd.DataFrame:
    """Compare unordered strategy pairs separately within each dataset.

    Each row is A minus B, with P(A>B), paired two-sided frequentist HAC tests,
    and paired block-bootstrap percentile CIs. Finite joint monthly observations
    are used; excluded counts are explicit. Nonconsecutive overlapping months
    raise ValueError because row-adjacency blocks would misrepresent time.
    Duplicate dataset/strategy/month keys also raise. Holm adjustment families
    comprise all unordered pairs in a dataset, separately for utility and SR.
    """
    required = {"dataset", "strategy", "eom_ret", "r", "tc"}
    if not required <= set(pf):
        raise ValueError(f"missing comparison columns: {sorted(required - set(pf))}")
    if bootstrap_reps < 1 or block_length < 1 or not np.isfinite(gamma) or gamma < 0:
        raise ValueError("require positive bootstrap settings and finite nonnegative gamma")
    data = pf.copy()
    data["eom_ret"] = pd.to_datetime(data["eom_ret"])
    if data[["dataset", "strategy", "eom_ret"]].isna().any().any():
        raise ValueError("missing dataset, strategy, or return-month identifier")
    if not data.eom_ret.dt.is_month_end.all():
        raise ValueError("return observations must be dated at month-end")
    if data.duplicated(["dataset", "strategy", "eom_ret"]).any():
        raise ValueError("duplicate dataset/strategy/return-month observations")
    rng = np.random.default_rng(seed)
    rows = []
    for dataset, sample in data.groupby("dataset", sort=True):
        strategies = sorted(sample.strategy.unique())
        for a, b in combinations(strategies, 2):
            left = sample.loc[sample.strategy.eq(a), ["eom_ret", "r", "tc"]].set_index("eom_ret")
            right = sample.loc[sample.strategy.eq(b), ["eom_ret", "r", "tc"]].set_index("eom_ret")
            joined = left.join(right, how="inner", lsuffix="_a", rsuffix="_b").sort_index()
            valid = np.isfinite(joined.to_numpy()).all(axis=1)
            paired = joined.loc[valid]
            n = len(paired)
            row = {"dataset": dataset, "strategy_a": a, "strategy_b": b, "n": n,
                   "n_a_available": len(left), "n_b_available": len(right),
                   "n_a_excluded": len(left) - n, "n_b_excluded": len(right) - n,
                   "n_joint_nonfinite_excluded": int((~valid).sum()), "gamma_evaluation": gamma,
                   "bootstrap_reps": bootstrap_reps, "bootstrap_block_months": min(block_length, n),
                   "seed": seed, "ci_level": 0.95,
                   "start": paired.index.min() if n else pd.NaT,
                   "end": paired.index.max() if n else pd.NaT,
                   "posterior_method": "paper iid normal approximation; not HAC; not a p-value",
                   "bootstrap_method": "paired circular moving blocks; percentile intervals; re-estimated means/variances",
                   "multiple_testing_family": "all unordered strategy pairs within dataset, separately utility and Sharpe"}
            if n < 3:
                row.update(status="SKIPPED", reason="fewer than three finite paired months")
                rows.append(row)
                continue
            periods = paired.index.to_period("M").asi8
            if not np.all(np.diff(periods) == 1):
                raise ValueError(f"nonconsecutive paired return months for {dataset}: {a} vs {b}")
            ra, ta, rb, tb = (paired[c].to_numpy(dtype=float) for c in ["r_a", "tc_a", "r_b", "tc_b"])
            paper_diff = utility_flows(ra, ta, gamma) - utility_flows(rb, tb, gamma)
            sample_diff = utility_flows(ra, ta, gamma, True) - utility_flows(rb, tb, gamma, True)
            hac = newey_west_mean(sample_diff)
            posterior_se = float(paper_diff.std(ddof=1) / np.sqrt(n))
            bootstrap_u, bootstrap_sr = _bootstrap_differences(ra, ta, rb, tb, gamma, bootstrap_reps,
                                                              min(block_length, n), rng)
            ulo, uhi = _interval(bootstrap_u)
            slo, shi = _interval(bootstrap_sr)
            sr_diff = net_sharpe(ra, ta) - net_sharpe(rb, tb)
            influence = sharpe_influence(ra, ta) - sharpe_influence(rb, tb)
            row.update(status="PASS", reason="", posterior_probability_a_gt_b=_normal_probability(float(paper_diff.mean()), posterior_se),
                       paper_flow_mean_diff_annual=12 * paper_diff.mean(),
                       utility_diff_annual=12 * sample_diff.mean(), utility_hac_se_annual=12 * hac["se"],
                       utility_hac_p=hac["p"], utility_hac_z=hac["z"], hac_lags=int(hac["lags"]),
                       utility_hac_ci_low=12 * (hac["mean"] - norm.ppf(.975) * hac["se"]),
                       utility_hac_ci_high=12 * (hac["mean"] + norm.ppf(.975) * hac["se"]),
                       utility_boot_ci_low=ulo, utility_boot_ci_high=uhi,
                       sharpe_diff=sr_diff, sharpe_boot_ci_low=slo, sharpe_boot_ci_high=shi,
                       utility_boot_valid_reps=int(np.isfinite(bootstrap_u).sum()),
                       sharpe_boot_valid_reps=int(np.isfinite(bootstrap_sr).sum()))
            if np.isfinite(influence).all() and np.isfinite(sr_diff):
                ihac = newey_west_mean(influence)
                se = ihac["se"]
                z = (0.0 if abs(sr_diff) <= np.finfo(float).eps else np.sign(sr_diff) * np.inf) if se == 0 else sr_diff / se
                row.update(sharpe_hac_se=se, sharpe_hac_z=z, sharpe_hac_p=float(2 * norm.sf(abs(z))),
                           sharpe_hac_ci_low=sr_diff - norm.ppf(.975) * se,
                           sharpe_hac_ci_high=sr_diff + norm.ppf(.975) * se,
                           sharpe_status="PASS")
            else:
                row.update(sharpe_hac_se=np.nan, sharpe_hac_z=np.nan, sharpe_hac_p=np.nan,
                           sharpe_hac_ci_low=np.nan, sharpe_hac_ci_high=np.nan,
                           sharpe_status="SKIPPED: zero gross volatility")
            rows.append(row)
    result = pd.DataFrame(rows)
    if result.empty:
        return result
    for metric in ["utility", "sharpe"]:
        raw = f"{metric}_hac_p"
        if raw not in result:
            result[raw] = np.nan
        result[f"{metric}_hac_p_holm"] = result.groupby("dataset", sort=False)[raw].transform(
            lambda x: holm_adjust(x.to_numpy()))
    return result


def bootstrap_rank_stability(pf: pd.DataFrame, gamma: float = 10.0,
                             bootstrap_reps: int = 2000, block_length: int = 6,
                             seed: int = 20261101) -> pd.DataFrame:
    """Joint-block uncertainty of rankings conditional on saved portfolios.

    Every strategy in a dataset receives the SAME sampled months in each draw.
    Rank 1 is best. The 95% rank interval uses empirical inverse-CDF quantiles
    (integer ranks). Ties receive their minimum rank; the probability of best
    splits tied first place equally. These are bootstrap stability frequencies,
    not Bayesian posterior probabilities, corrected selection inference, or
    probabilities a strategy will win in a future sample. No model is re-fitted.
    """
    required={"dataset","strategy","eom_ret","r","tc"}
    if not required <= set(pf):
        raise ValueError(f"missing ranking columns: {sorted(required-set(pf))}")
    if bootstrap_reps<1 or block_length<1 or not np.isfinite(gamma) or gamma<0:
        raise ValueError("require positive bootstrap settings and finite nonnegative gamma")
    data=pf.copy()
    data['eom_ret']=pd.to_datetime(data.eom_ret)
    if data[['dataset','strategy','eom_ret']].isna().any().any():
        raise ValueError('missing dataset, strategy or return-month identifier')
    if data.duplicated(['dataset','strategy','eom_ret']).any():
        raise ValueError('duplicate dataset/strategy/return-month observations')
    rng=np.random.default_rng(seed)
    rows=[]
    for dataset,g in data.groupby('dataset',sort=True):
        gross=g.pivot(index='eom_ret',columns='strategy',values='r').sort_index()
        costs=g.pivot(index='eom_ret',columns='strategy',values='tc').reindex_like(gross)
        if len(gross)<3 or not np.isfinite(gross.to_numpy()).all() or not np.isfinite(costs.to_numpy()).all():
            raise ValueError(f'ranking requires at least 3 identical finite saved months for {dataset}')
        if not gross.index.is_month_end.all() or not np.all(np.diff(gross.index.to_period('M').asi8)==1):
            raise ValueError(f'ranking requires consecutive month-end observations for {dataset}')
        r,tc=gross.to_numpy(),costs.to_numpy()
        indices=circular_block_indices(len(gross),bootstrap_reps,min(block_length,len(gross)),rng)
        boot_r,boot_tc=r[indices],tc[indices]
        boot_net_mean=(boot_r-boot_tc).mean(axis=1)
        boot_variance=boot_r.var(axis=1,ddof=1)
        utility=12*(boot_net_mean-.5*gamma*boot_variance)
        with np.errstate(divide='ignore',invalid='ignore'):
            sharpe=np.sqrt(12)*boot_net_mean/np.sqrt(boot_variance)
        threshold=64*np.finfo(float).eps*np.maximum(1.,np.max(np.abs(boot_r),axis=1))
        sharpe[np.sqrt(boot_variance)<=threshold]=np.nan
        obs_net=(r-tc).mean(axis=0)
        obs_var=r.var(axis=0,ddof=1)
        obs_u=12*(obs_net-.5*gamma*obs_var)
        with np.errstate(divide='ignore',invalid='ignore'):
            obs_sr=np.sqrt(12)*obs_net/np.sqrt(obs_var)
        obs_threshold=64*np.finfo(float).eps*np.maximum(1.,np.max(np.abs(r),axis=0))
        obs_sr[np.sqrt(obs_var)<=obs_threshold]=np.nan
        for metric,scores,observed in [('utility',utility,obs_u),('paper_net_sharpe',sharpe,obs_sr)]:
            ordering=rankdata(-scores,method='min',axis=1,nan_policy='omit')
            observed_rank=rankdata(-observed,method='min',nan_policy='omit')
            tie_count=(ordering==1).sum(axis=1)
            for i,strategy in enumerate(gross.columns):
                valid=np.isfinite(ordering[:,i])
                sample_ranks=ordering[valid,i]
                sample_scores=scores[valid,i]
                if len(sample_ranks):
                    rank_ci=np.quantile(sample_ranks,[.025,.975],method='inverted_cdf')
                    score_ci=np.quantile(sample_scores,[.025,.975])
                    p_best=float(np.mean((sample_ranks==1)/tie_count[valid]))
                    p_top3=float(np.mean(sample_ranks<=min(3,len(gross.columns))))
                else:
                    rank_ci=score_ci=[np.nan,np.nan]
                    p_best=p_top3=np.nan
                rows.append(dict(dataset=dataset,strategy=strategy,ranking_metric=metric,
                    n=len(gross),sample_start=gross.index.min(),sample_end=gross.index.max(),
                    n_strategies=len(gross.columns),observed_metric=observed[i],observed_rank=observed_rank[i],
                    rank_ci_low=rank_ci[0],rank_ci_high=rank_ci[1],
                    score_boot_ci_low=score_ci[0],score_boot_ci_high=score_ci[1],
                    bootstrap_probability_best_ties_split=p_best,bootstrap_probability_top3=p_top3,
                    bootstrap_valid_reps=int(valid.sum()),bootstrap_reps=bootstrap_reps,
                    block_length=min(block_length,len(gross)),seed=seed,gamma_evaluation=gamma,
                    interpretation='Joint circular-block bootstrap rank stability, conditional on saved portfolios; not a posterior or future-win probability.'))
    return pd.DataFrame(rows)


### `scripts/analysis/ml_comparison/metrics.py`

Paper-convention metrics plus explicitly labelled evaluation extensions.

**Role:** Reporting source. Reads available local results when called; this cell writes the module.

**Input:** the embedded source below. **Output:** `scripts/analysis/ml_comparison/metrics.py`. Expand the code cell to read or edit it.


In [ ]:
%%writefile scripts/analysis/ml_comparison/metrics.py
"""Paper-convention metrics plus explicitly labelled evaluation extensions."""
from __future__ import annotations

import numpy as np
import pandas as pd


def additive_path(net):
    """Paper cumulative return and signed drop from its peak, including initial 0.

    Drawdown extends the paper's additive chart convention; multiply by 100 for
    percentage points. It is not a percentage loss of investor wealth.
    """
    values = np.asarray(net, dtype=float)
    if values.ndim != 1 or not len(values) or not np.isfinite(values).all():
        raise ValueError('Additive paths require a nonempty finite return vector.')
    cumulative = np.cumsum(values)
    peak = np.maximum.accumulate(np.r_[0., cumulative])[1:]
    return cumulative, cumulative - peak


def statistics(group: pd.DataFrame, gamma=10.0) -> dict:
    """Use project Table 2 formulas; independently checked in the validation suite."""
    from jkmp.evaluation.metrics import summary_stats
    g = group.sort_values('eom_ret')
    if len(g) < 2:
        raise ValueError('Performance statistics require at least two months.')
    if g.eom_ret.duplicated().any():
        raise ValueError('Duplicate return months in performance statistics.')
    if not np.isfinite(g[['r','tc','turnover','inv','shorting']].to_numpy(float)).all():
        raise ValueError('Nonfinite saved performance; no missing returns are filled.')
    out = summary_stats(g, gamma).to_dict()
    net = (g.r - g.tc).to_numpy(float)
    cumulative, dd = additive_path(net)
    downside = np.sqrt(np.mean(np.minimum(net, 0.0) ** 2))
    sd_net = np.std(net, ddof=1)
    out.update(sample_start=str(g.eom_ret.min().date()), sample_end=str(g.eom_ret.max().date()),
        evaluation_gamma=gamma, max_drawdown_excess=float(dd.min()),
        path_convention='additive', drawdown_unit='decimal return difference (x100 = percentage points)',
        sortino_zero_excess=float(np.mean(net) / downside * np.sqrt(12)) if downside > 0 else np.nan,
        cumulative_net_excess=float(cumulative[-1]),
        cumulative_net_excess_arithmetic=float(net.sum()),
        net_return_volatility=sd_net * np.sqrt(12),
        net_sharpe_net_sd=float(np.mean(net) / sd_net * np.sqrt(12)) if sd_net > 0 else np.nan)
    return out


def table(monthly, gamma=10.0):
    return pd.DataFrame([dict(dataset=dataset,strategy=strategy,**statistics(g,gamma))
        for (dataset,strategy),g in monthly.groupby(['dataset','strategy'],sort=False)])


def annual(monthly,gamma=10.0):
    rows=[]
    for (dataset,strategy,year),g in monthly.groupby(['dataset','strategy',monthly.eom_ret.dt.year]):
        rows.append(dict(dataset=dataset,strategy=strategy,year=year,**statistics(g,gamma)))
    return pd.DataFrame(rows)


def paths_and_rolling(monthly,gamma=10.0,window=24):
    rows=[]
    for (dataset,strategy),g in monthly.groupby(['dataset','strategy'],sort=False):
        g=g.sort_values('eom_ret')
        net=g.r-g.tc
        cumulative, drawdown = additive_path(net)
        mean=g.r.rolling(window,min_periods=window).mean()
        sd=g.r.rolling(window,min_periods=window).std(ddof=1)
        costs=g.tc.rolling(window,min_periods=window).mean()
        p=pd.DataFrame(dict(dataset=dataset,strategy=strategy,eom_ret=g.eom_ret,
            net_excess=net,drawdown_excess=drawdown,path_convention='additive',
            cumulative_net_excess=cumulative,cumulative_net_excess_arithmetic=cumulative,
            cumulative_utility_paper=(net-.5*gamma*(g.r-g.r.mean())**2).cumsum(),
            rolling_net_sharpe_paper=(mean-costs)/sd*np.sqrt(12),
            rolling_utility=(mean-costs-.5*gamma*sd**2)*12,
            rolling_window=window))
        rows.append(p)
    return pd.concat(rows,ignore_index=True)


def ranks(t):
    out=t.copy()
    for column,ascending in [('Utility',False),('SR_net',False),('R_TC',False),('Vol',True),('TC',True),('Turnover',True)]:
        out[f'rank_{column}']=out.groupby('dataset')[column].rank(ascending=ascending,method='min')
    return out.sort_values(['dataset','rank_Utility'])


def metric_dictionary():
    descriptions=[
        ('R','12*mean(r)','decimal/year','Annual arithmetic gross excess return.'),
        ('Vol','sqrt(12)*sd(r, ddof=1)','decimal/year^0.5','Gross-return volatility; paper risk convention.'),
        ('SR_gross','sqrt(12)*mean(r)/sd(r, ddof=1)','ratio','Gross excess Sharpe.'),
        ('TC','12*mean(tc)','fraction/year','Modelled quadratic trading costs, not observed execution costs.'),
        ('R_TC','12*mean(r-tc)','decimal/year','Annual arithmetic net excess return.'),
        ('SR_net','sqrt(12)*mean(r-tc)/sd(r, ddof=1)','ratio','Paper net Sharpe uses GROSS volatility.'),
        ('Utility','12*(mean(r-tc)-gamma*var(r, ddof=1)/2)','decimal/year','Sample mean-variance utility; common evaluation gamma=10.'),
        ('Turnover','mean(sum_i(abs(w_i-w_start_i)))','fraction/month','Two-sided absolute traded weight; NOT divided by two.'),
        ('Lev','mean(sum_i(abs(w_i)))','gross weight','Average monthly gross leverage.'),
        ('Shorting','mean(sum_i(max(-w_i,0)))','gross weight','Average short exposure.'),
        ('max_drawdown_excess','min(C_t-max(0,C_1,...,C_t)); C=cumsum(r-tc)','decimal return difference','Additive net excess-return drawdown; x100 gives percentage points, not percentage loss of wealth. Reporting extension of the paper additive chart.'),
        ('sortino_zero_excess','sqrt(12)*mean(r-tc)/sqrt(mean(min(r-tc,0)^2))','ratio','Zero excess-return hurdle, includes all months in downside denominator.'),
        ('path_convention','additive','label','Paper reporting convention; no compounded growth or CAGR in active performance reports.'),
        ('cumulative_net_excess','sum(r-tc)','decimal return sum','Additive cumulative net excess return; x100 gives percentage points, not investor wealth.'),
        ('cumulative_net_excess_arithmetic','sum(r-tc)','fraction','Paper cumulative net-excess path; arithmetic sum, not investor wealth.'),
        ('cumulative_utility_paper','cumsum(r-tc-gamma*(r-mean(r))^2/2)','fraction','Paper cumulative utility flow using full evaluated sample gross mean; descriptive ex-post path.'),
        ('net_sharpe_net_sd','sqrt(12)*mean(r-tc)/sd(r-tc, ddof=1)','ratio','Conventional net-return SD denominator; provided separately from paper metric.'),
        ('posterior_probability_a_gt_b','Phi(mean(u_A-u_B)/(sd(u_A-u_B)/sqrt(T)))','probability','Paper normal posterior-style probability; not a frequentist p-value; no HAC adjustment.'),
        ('utility_flow','r-tc-gamma*(r-mean(r))^2/2','fraction/month','Exact R flow; its mean uses population variance and differs from Table 2 finite-sample utility.'),
        ('utility_hac_p','2*Phi(-abs(mean(d)/SE_NW(d)))','p-value','Paired sample-variance-adjusted utility-flow mean difference; Bartlett Newey-West.'),
        ('sharpe_hac_p','Normal test of paired paper-SR difference using delta-method influences','p-value','Asymptotic HAC inference; bootstrap supplies non-normal uncertainty.'),
        ('bootstrap_CI','Paired circular moving-block resamples, re-estimate each statistic','same as statistic','Percentile 95% confidence intervals; serial dependence preserved within blocks.'),
        ('Holm_p','Step-down Holm on all unordered pairs per dataset and test family','adjusted p-value','Controls family-wise error within predeclared family; does not remove prior research selection.'),
        ('weight_cosine','dot(wA,wB)/(norm(wA)*norm(wB))','[-1,1]','Average monthly similarity on union of saved IDs, absent weights zero.'),
        ('cost_multiplier','r - m*tc; w and gross r unchanged','scenario','Counterfactual re-costing, not re-optimised portfolio.'),
        ('fixed_weight_AUM','tc_new=tc_saved*AUM_anchor_new/AUM_anchor_saved','scenario','Quadratic impact at unchanged weights/relative wealth path; no participation constraint.'),
        ('forecast_volatility','sqrt(12*w^T*Sigma_t*w)','decimal/year^0.5','Saved contemporaneous covariance forecast; compare with realised gross risk descriptively.')]
    return pd.DataFrame(descriptions,columns=['metric','formula','unit','interpretation_and_method'])


## Source — Calculation libraries


### `scripts/jkmp/__init__.py`

Python replication of Jensen, Kelly, Malamud & Pedersen (2026), "Machine Learning and the Implementable Efficient Frontier", RFS.

**Role:** Calculation library. Other stages import its functions; this cell writes the module.

**Input:** the embedded source below. **Output:** `scripts/jkmp/__init__.py`. Expand the code cell to read or edit it.


In [ ]:
%%writefile scripts/jkmp/__init__.py
"""Python replication of Jensen, Kelly, Malamud & Pedersen (2026),
"Machine Learning and the Implementable Efficient Frontier", RFS."""

from jkmp.config import Config, ConfigError, load_config

__all__ = ["Config", "ConfigError", "load_config"]


### `scripts/jkmp/config.py`

Load ``configs/base.yaml`` (plus run-time overrides) into a frozen, typed ``Config``. Usage:: from jkmp.config import load_config cfg = load_config() # configs/base.yaml cfg = load_config(overrides=["investor.gamma=5"]) # dotted-key strings cfg = load_config(overrides={"investor.wealth": 1.0e9}) # or a dotted-key mapping The YAML file is the single source of truth: every dataclass field must appear in it, unknown keys are rejected, and overrides may only replace keys that already exist. An override replaces the whole value at its key (no deep m

**Role:** Calculation library. Other stages import its functions; this cell writes the module.

**Input:** the embedded source below. **Output:** `scripts/jkmp/config.py`. Expand the code cell to read or edit it.


In [ ]:
%%writefile scripts/jkmp/config.py
"""Load ``configs/base.yaml`` (plus run-time overrides) into a frozen, typed ``Config``.

Usage::

    from jkmp.config import load_config

    cfg = load_config()                                         # configs/base.yaml
    cfg = load_config(overrides=["investor.gamma=5"])           # dotted-key strings
    cfg = load_config(overrides={"investor.wealth": 1.0e9})     # or a dotted-key mapping

The YAML file is the single source of truth: every dataclass field must appear in it,
unknown keys are rejected, and overrides may only replace keys that already exist.
An override replaces the whole value at its key (no deep merge).
"""

from __future__ import annotations

import copy
import dataclasses
import datetime as dt
import math
from collections.abc import Mapping, Sequence
from dataclasses import dataclass
from pathlib import Path
from types import MappingProxyType, NoneType, UnionType
from typing import Any, Union, get_args, get_origin, get_type_hints

import yaml

PROJECT_ROOT = Path(__file__).resolve().parents[2]
DEFAULT_CONFIG_PATH = PROJECT_ROOT / "configs" / "us_large" / "base.yaml"


class ConfigError(ValueError):
    """Raised for missing, unknown, mistyped or inconsistent configuration values."""


def _require(condition: bool, message: str) -> None:
    if not condition:
        raise ConfigError(message)


def _require_even(values: tuple[int, ...], name: str) -> None:
    # Random Fourier features come in sin/cos pairs, so p must be even.
    _require(all(p > 0 and p % 2 == 0 for p in values), f"{name} must contain positive even integers")


# --------------------------------------------------------------------------- sections


@dataclass(frozen=True)
class PathsConfig:
    raw: Path
    processed: Path
    outputs: Path
    reference_r: Path


@dataclass(frozen=True)
class DataSourcesConfig:
    jkp_table: str
    crsp_daily_table: str
    factor_details_url: str
    cluster_labels_url: str
    ff_factors_url: str
    ff_factors_daily_url: str
    fred_csv_url: str


@dataclass(frozen=True)
class DatesConfig:
    warmup_start: dt.date
    validation_start: dt.date
    validation_end: dt.date
    test_start: dt.date
    test_end: dt.date
    holdout_start: dt.date | None
    holdout_end: dt.date | None
    data_end: dt.date
    holdout_locked: bool
    evaluate: str

    def __post_init__(self) -> None:
        order = ["warmup_start", "validation_start", "validation_end", "test_start", "test_end"]
        for earlier, later in zip(order, order[1:]):
            _require(
                getattr(self, earlier) < getattr(self, later),
                f"dates.{earlier} must be before dates.{later}",
            )
        _require(self.test_end <= self.data_end, "test_end must not exceed data_end")
        if self.holdout_start is None or self.holdout_end is None:
            _require(self.holdout_start is None and self.holdout_end is None,
                     "both holdout dates must be null when no separate holdout exists")
            _require(self.holdout_locked and self.evaluate == "test",
                     "No separate Keats holdout is available; evaluate the declared test period only")
        else:
            _require(self.test_end < self.holdout_start <= self.holdout_end <= self.data_end,
                     "holdout must follow test_end and not exceed data_end")
        _require(self.evaluate in ("test", "holdout"), "dates.evaluate must be 'test' or 'holdout'")
        _require(self.evaluate == "test" or not self.holdout_locked, "dates.evaluate: holdout needs holdout_locked: false")


@dataclass(frozen=True)
class UniverseConfig:
    exchanges: tuple[int, ...]
    share_codes: tuple[int, ...]
    jkp_flags: tuple[str, ...]
    target_n: int
    entry_rank: int
    exit_rank: int
    reconstitution_month: int
    min_feature_pct: float
    lookback_months: int

    def __post_init__(self) -> None:
        _require(
            0 < self.entry_rank <= self.target_n <= self.exit_rank,
            "universe ranks must satisfy 0 < entry_rank <= target_n <= exit_rank",
        )
        _require(1 <= self.reconstitution_month <= 12, "universe.reconstitution_month must be 1..12")
        _require(0.0 < self.min_feature_pct <= 1.0, "universe.min_feature_pct must be in (0, 1]")
        _require(self.lookback_months >= 0, "universe.lookback_months must be >= 0")


@dataclass(frozen=True)
class FeaturesConfig:
    percentile_rank: bool
    impute_value: float
    themes: Mapping[str, tuple[str, ...]]

    def __post_init__(self) -> None:
        _require(len(self.themes) > 0, "features.themes must not be empty")
        _require(all(len(f) > 0 for f in self.themes.values()), "every theme needs at least one feature")
        _require(len(self.names) == len(set(self.names)), "a feature appears in more than one theme")

    @property
    def names(self) -> tuple[str, ...]:
        """All features, in theme order."""
        return tuple(f for feats in self.themes.values() for f in feats)

    @property
    def theme_of(self) -> Mapping[str, str]:
        """Feature name -> theme name."""
        return MappingProxyType({f: theme for theme, feats in self.themes.items() for f in feats})


@dataclass(frozen=True)
class CovarianceConfig:
    factor_universe_top_n: int
    sectors: Mapping[str, tuple[str, ...]]
    window_days: int
    min_window_days: int
    half_life_corr_days: int
    half_life_var_days: int
    half_life_idio_days: int
    idio_min_obs: int
    idio_min_window: int
    idio_initial_obs: int
    days_per_month: int

    def __post_init__(self) -> None:
        _require(0 < self.min_window_days <= self.window_days, "covariance.min_window_days must be in (0, window_days]")
        _require(0 < self.idio_min_obs <= self.idio_min_window, "covariance.idio_min_obs must be in (0, idio_min_window]")
        industries = [i for group in self.sectors.values() for i in group]
        _require(len(industries) == len(set(industries)), "covariance.sectors: an industry is in two sectors")


@dataclass(frozen=True)
class TradingCostConfig:
    price_impact: float
    dolvol_column: str
    dolvol_units: str

    def __post_init__(self) -> None:
        _require(self.price_impact > 0, "trading_costs.price_impact must be > 0")
        _require(self.dolvol_units in {"dollars", "millions"}, "trading_costs.dolvol_units must be 'dollars' or 'millions'")


@dataclass(frozen=True)
class InvestorConfig:
    gamma: float
    wealth: float
    wealth_end_date: dt.date
    mu: float
    m_iterations: int
    gamma_te: float
    te_targets: tuple[float, ...]
    gamma_te_grid: tuple[float, ...]
    gamma_grid: tuple[float, ...]
    wealth_grid: tuple[float, ...]

    def __post_init__(self) -> None:
        _require(self.gamma > 0 and all(g > 0 for g in self.gamma_grid), "risk aversion must be > 0")
        _require(self.wealth > 0 and all(w > 0 for w in self.wealth_grid), "wealth must be > 0")
        _require(self.m_iterations > 0, "investor.m_iterations must be > 0")
        _require(self.gamma_te > 0 and all(g > 0 for g in self.gamma_te_grid), "gamma_te must be > 0")
        _require(all(0 < t < 1 for t in self.te_targets), "investor.te_targets are annual fractions in (0, 1)")


@dataclass(frozen=True)
class PortfolioConfig:
    charge_exit_tc: bool


@dataclass(frozen=True)
class RidgeGridConfig:
    include_zero: bool
    log_min: float
    log_max: float
    n: int

    def __post_init__(self) -> None:
        _require(self.n >= 2 and self.log_min < self.log_max, "ridge_grid needs n >= 2 and log_min < log_max")

    @property
    def values(self) -> tuple[float, ...]:
        """Penalties: optional zero, then exp of n evenly spaced log penalties."""
        step = (self.log_max - self.log_min) / (self.n - 1)
        grid = tuple(math.exp(self.log_min + i * step) for i in range(self.n))
        return ((0.0,) + grid) if self.include_zero else grid


@dataclass(frozen=True)
class ReturnModelConfig:
    p_grid: tuple[int, ...]
    rff_variance_grid: tuple[float, ...]
    horizons: tuple[int, ...]
    val_years: int
    pred_years: tuple[int, ...]
    validation_scope: str

    def __post_init__(self) -> None:
        _require_even(self.p_grid, "return_model.p_grid")
        _require(all(h >= 1 for h in self.horizons), "return_model.horizons must be >= 1")
        _require(self.val_years >= 1, "return_model.val_years must be >= 1")
        _require(self.validation_scope in ("all", "universe"),
                 "return_model.validation_scope must be all or universe")
        _require(len(self.pred_years) == 2 and self.pred_years[0] <= self.pred_years[1],
                 "return_model.pred_years must be [first, last]")


@dataclass(frozen=True)
class PortfolioMLConfig:
    use_rff: bool
    p_grid: tuple[int, ...]
    rff_variance_grid: tuple[float, ...]
    scale_by_vol: bool
    add_original_features: bool
    add_constant: bool
    benchmark_relative: bool
    half_life_years: tuple[float | None, ...]

    def __post_init__(self) -> None:
        _require_even(self.p_grid, "portfolio_ml.p_grid")
        _require(len(self.half_life_years) >= 1 and all(h is None or h > 0 for h in self.half_life_years),
                 "portfolio_ml.half_life_years: positive years or null (= no decay)")


@dataclass(frozen=True)
class StaticMLConfig:
    k_grid: tuple[float, ...]
    u_grid: tuple[float, ...]
    g_grid: tuple[float, ...]


@dataclass(frozen=True)
class FactorMLConfig:
    n_portfolios: int


@dataclass(frozen=True)
class FeatureImportanceConfig:
    markowitz_gamma: float
    n_seeds: int

    def __post_init__(self) -> None:
        _require(self.markowitz_gamma > 0, "feature_importance.markowitz_gamma must be > 0")
        _require(self.n_seeds >= 1, "feature_importance.n_seeds must be >= 1")

    def seeds(self, seed: int) -> tuple[int, ...]:
        """Permutation seeds: the run's seed and the next n_seeds - 1."""
        return tuple(range(seed, seed + self.n_seeds))


@dataclass(frozen=True)
class EpisodeConfig:
    label: str
    start: dt.date
    end: dt.date

    def __post_init__(self) -> None:
        _require(self.start <= self.end, f"regimes episode {self.label!r}: start after end")


@dataclass(frozen=True)
class RegimesConfig:
    fred_series: tuple[str, ...]
    fed_change_months: int
    fed_threshold: float
    vix_norm: str
    plot_types: tuple[str, ...]
    episodes: tuple[EpisodeConfig, ...]

    def __post_init__(self) -> None:
        _require(self.fed_change_months >= 1, "regimes.fed_change_months must be >= 1")
        _require(self.fed_threshold >= 0, "regimes.fed_threshold must be >= 0")
        _require(self.vix_norm in ("window", "full"), "regimes.vix_norm must be 'window' or 'full'")


@dataclass(frozen=True)
class ReportingConfig:
    main_types: tuple[str, ...]
    subperiods: tuple[EpisodeConfig, ...]   # label, start, end (return months)
    ic_size_buckets: tuple[tuple[int, ...], ...]
    capacity_ir: float
    cost_vix_start: dt.date
    cost_vol_window: int

    def __post_init__(self) -> None:
        _require(len(self.main_types) >= 1, "reporting.main_types needs at least one method")
        _require(all(len(b) == 2 and 1 <= b[0] <= b[1] for b in self.ic_size_buckets),
                 "reporting.ic_size_buckets must be [first, last] rank pairs")
        _require(self.cost_vol_window >= 1, "reporting.cost_vol_window must be >= 1")


@dataclass(frozen=True)
class MonitorConfig:
    ir_window: int
    drawdown_te_multiple: float
    te_budget: float
    beta_limits: tuple[float, ...]
    theme_limit: float
    ic_window: int
    ic_negative_months: int
    tc_months: int
    bias_window: int

    def __post_init__(self) -> None:
        _require(len(self.beta_limits) == 2 and self.beta_limits[0] < self.beta_limits[1], "monitor.beta_limits: [low, high]")
        _require(min(self.ir_window, self.ic_window, self.ic_negative_months, self.tc_months, self.bias_window) >= 1,
                 "monitor windows must be >= 1")


@dataclass(frozen=True)
class GridsConfig:
    n_jobs: int
    frontier_vol_max: float
    frontier_vol_step: float

    def __post_init__(self) -> None:
        _require(self.n_jobs != 0, "grids.n_jobs must be non-zero (joblib: -1 = all cores)")
        _require(0 < self.frontier_vol_step < self.frontier_vol_max, "grids needs 0 < frontier_vol_step < frontier_vol_max")


@dataclass(frozen=True)
class Config:
    seed: int
    paths: PathsConfig
    data_sources: DataSourcesConfig
    dates: DatesConfig
    universe: UniverseConfig
    features: FeaturesConfig
    covariance: CovarianceConfig
    trading_costs: TradingCostConfig
    investor: InvestorConfig
    portfolio: PortfolioConfig
    ridge_grid: RidgeGridConfig
    return_model: ReturnModelConfig
    portfolio_ml: PortfolioMLConfig
    static_ml: StaticMLConfig
    factor_ml: FactorMLConfig
    monitor: MonitorConfig
    grids: GridsConfig
    feature_importance: FeatureImportanceConfig
    reporting: ReportingConfig
    regimes: RegimesConfig

    def as_dict(self) -> dict[str, Any]:
        """Plain-Python copy (dicts, lists, str paths) for saving alongside run outputs."""
        return _to_plain(self)


# --------------------------------------------------------------------------- loading


def load_config(
    path: str | Path | None = None,
    overrides: Mapping[str, Any] | Sequence[str] | None = None,
) -> Config:
    """Read a YAML config, apply dotted-key overrides, and build a validated ``Config``.

    A file may start with ``base: <path>`` (relative to that file): the base is read first and this
    file's keys are merged on top, so an experiment lists only what it changes.
    """
    path = Path(path) if path is not None else DEFAULT_CONFIG_PATH
    raw = _read_yaml(path, seen=())
    raw = _apply_overrides(raw, overrides)
    cfg = _build(Config, raw, "")
    for target in (cfg.paths.raw, cfg.paths.processed, cfg.paths.outputs):
        _require(target.resolve().is_relative_to(PROJECT_ROOT.resolve()),
                 f"Kati writes must stay inside its own folder: {target}")
    if cfg.paths.outputs.parent.name == "base":
        from jkmp.strategies import base_strategies
        _require(cfg.reporting.main_types == base_strategies(cfg.paths.outputs.name), "Base reports require US high-cap Static-ML* and regional Portfolio-ML only")
        _require(cfg.portfolio_ml.use_rff and not cfg.portfolio_ml.benchmark_relative
                 and cfg.portfolio_ml.add_constant and cfg.portfolio_ml.half_life_years == (None,),
                 "Base requires the original default Portfolio-ML; choose an iteration group for experimental options")
    from jkmp.imputation_audit import report_if_enabled
    report_if_enabled(cfg)
    return cfg


def _read_yaml(path: Path, seen: tuple[Path, ...]) -> dict:
    path = path.resolve()
    _require(path not in seen, f"circular 'base' chain at {path}")
    with path.open() as fh:
        raw = yaml.safe_load(fh)
    _require(isinstance(raw, dict), f"{path} must contain a YAML mapping")
    base = raw.pop("base", None)
    if base is None:
        return raw
    return _merge(_read_yaml(path.parent / base, (*seen, path)), raw, "")


def _merge(base: dict, top: dict, where: str) -> dict:
    """Deep merge; every key in ``top`` must exist in ``base`` (typos fail loudly)."""
    out = copy.deepcopy(base)
    for key, value in top.items():
        name = f"{where}.{key}" if where else key
        _require(key in out, f"unknown key {name!r} in experiment config")
        out[key] = _merge(out[key], value, name) if isinstance(value, dict) and isinstance(out[key], dict) else value
    return out


def _apply_overrides(raw: dict, overrides: Mapping[str, Any] | Sequence[str] | None) -> dict:
    if not overrides:
        return raw
    if isinstance(overrides, Mapping):
        items = list(overrides.items())
    elif isinstance(overrides, str):
        raise ConfigError("overrides must be a list of 'key=value' strings, not a single string")
    else:
        items = [_parse_assignment(s) for s in overrides]
    raw = copy.deepcopy(raw)
    for dotted, value in items:
        *parents, leaf = dotted.split(".")
        node = raw
        for key in parents:
            _require(isinstance(node.get(key), dict), f"unknown override key {dotted!r}")
            node = node[key]
        _require(leaf in node, f"unknown override key {dotted!r}")
        node[leaf] = value
    return raw


def _parse_assignment(text: str) -> tuple[str, Any]:
    key, sep, value = text.partition("=")
    _require(bool(sep) and bool(key.strip()), f"override {text!r} must look like 'section.key=value'")
    return key.strip(), yaml.safe_load(value)  # parse the value exactly as YAML would


def _build(cls: type, data: Any, where: str) -> Any:
    _require(isinstance(data, Mapping), f"{where or 'config'} must be a mapping")
    hints = get_type_hints(cls)
    names = {f.name for f in dataclasses.fields(cls)}
    unknown, missing = set(data) - names, names - set(data)
    _require(not unknown, f"unknown key(s) in {where or 'config'}: {sorted(unknown)}")
    _require(not missing, f"missing key(s) in {where or 'config'}: {sorted(missing)}")
    return cls(**{n: _coerce(hints[n], data[n], f"{where}.{n}" if where else n) for n in names})


def _coerce(tp: Any, value: Any, where: str) -> Any:
    if dataclasses.is_dataclass(tp):
        return _build(tp, value, where)
    origin = get_origin(tp)
    if origin in (Union, UnionType) and NoneType in get_args(tp):  # X | None: YAML null is allowed
        (inner,) = [a for a in get_args(tp) if a is not NoneType]
        return None if value is None else _coerce(inner, value, where)
    if origin is tuple:
        _require(isinstance(value, Sequence) and not isinstance(value, str), f"{where} must be a list")
        (item_tp, _ellipsis) = get_args(tp)
        return tuple(_coerce(item_tp, v, f"{where}[{i}]") for i, v in enumerate(value))
    if origin is Mapping:
        _require(isinstance(value, Mapping), f"{where} must be a mapping")
        key_tp, val_tp = get_args(tp)
        return MappingProxyType(
            {_coerce(key_tp, k, f"{where} key"): _coerce(val_tp, v, f"{where}.{k}") for k, v in value.items()}
        )
    if tp is bool:
        _require(isinstance(value, bool), f"{where} must be true/false, got {value!r}")
        return value
    if tp is int:
        _require(isinstance(value, int) and not isinstance(value, bool), f"{where} must be an integer, got {value!r}")
        return value
    if tp is float:
        # Catches the PyYAML pitfall where "1e9" is read as a string.
        _require(
            isinstance(value, (int, float)) and not isinstance(value, bool),
            f"{where} must be a number, got {value!r} (write 1.0e+9, not 1e9)",
        )
        return float(value)
    if tp is str:
        _require(isinstance(value, str), f"{where} must be a string, got {value!r}")
        return value
    if tp is dt.date:
        if isinstance(value, str):
            value = dt.date.fromisoformat(value)
        _require(isinstance(value, dt.date) and not isinstance(value, dt.datetime), f"{where} must be a YYYY-MM-DD date")
        return value
    if tp is Path:
        _require(isinstance(value, (str, Path)), f"{where} must be a path")
        p = Path(value)
        return p if p.is_absolute() else PROJECT_ROOT / p
    raise ConfigError(f"{where}: unsupported field type {tp!r}")


def _to_plain(obj: Any) -> Any:
    if dataclasses.is_dataclass(obj) and not isinstance(obj, type):
        return {f.name: _to_plain(getattr(obj, f.name)) for f in dataclasses.fields(obj)}
    if isinstance(obj, Mapping):
        return {k: _to_plain(v) for k, v in obj.items()}
    if isinstance(obj, tuple):
        return [_to_plain(v) for v in obj]
    if isinstance(obj, Path):
        return str(obj)
    return obj


### `scripts/jkmp/costs.py`

Trading-cost inputs and the investor's wealth path. * ``price_impact_lambda`` — diagonal of Kyle's lambda (Eq. 35): Λ_i = 2·π / dolvol_i = 0.2 / dolvol_i. * ``drift_factor`` — g_t = (1 + tr_ld0) / (1 + mu_ld0), how a weight drifts over month t (Eq. 6). * ``wealth_path`` — the fund grows with the market (Sec. 5.1.3), so its size is a stable share of the market; wealth is labelled by its value at the end date (``investor.wealth_end_date``). Port of ``wealth_func`` in reference/jkmp_r/1 - Prepare Data.R.

**Role:** Calculation library. Other stages import its functions; this cell writes the module.

**Input:** the embedded source below. **Output:** `scripts/jkmp/costs.py`. Expand the code cell to read or edit it.


In [ ]:
%%writefile scripts/jkmp/costs.py
"""Trading-cost inputs and the investor's wealth path.

* ``price_impact_lambda`` — diagonal of Kyle's lambda (Eq. 35): Λ_i = 2·π / dolvol_i = 0.2 / dolvol_i.
* ``drift_factor`` — g_t = (1 + tr_ld0) / (1 + mu_ld0), how a weight drifts over month t (Eq. 6).
* ``wealth_path`` — the fund grows with the market (Sec. 5.1.3), so its size is a stable share of
  the market; wealth is labelled by its value at the end date (``investor.wealth_end_date``). Port of
  ``wealth_func`` in reference/jkmp_r/1 - Prepare Data.R.
"""

from __future__ import annotations

import datetime as dt

import numpy as np
import pandas as pd


def price_impact_lambda(dolvol: pd.Series, price_impact: float) -> pd.Series:
    """Λ_i = 2·π / V_i, with V_i the expected daily dollar volume (R: ``lambda := 2/dolvol*settings$pi``).

    With π = 0.1, trading 1% of daily volume moves the price by 0.1% (Frazzini et al., 2018).
    """
    return 2.0 * price_impact / dolvol


def drift_factor(tr_ld0: pd.Series, mu_ld0: pd.Series) -> pd.Series:
    """Weight growth over month t, g_t = (1 + r_f + r_t) / (1 + g^w_t)  (Eq. 6).

    A weight π_{t-1} grows to g_t·π_{t-1} before trading at t; the fund grows with the market, so
    g^w_t = mu_ld0. As in R (``pfml_input_fun``), a missing g is set to 1, i.e. the stock is assumed
    to have earned the fund's return.
    """
    return ((1.0 + tr_ld0) / (1.0 + mu_ld0)).fillna(1.0)


def wealth_path(wealth_end: float, end: dt.date, market: pd.DataFrame) -> pd.DataFrame:
    """Investor wealth at each month-end, equal to ``wealth_end`` at the anchor ``end``: rolled back
    before it and forward after it, over every month in ``market``.

    ``market`` has one row per month: ``eom_ret`` (month-end the return is realised) and ``tret``
    (market *total* return over that month). Returns columns ``eom``, ``wealth`` and ``mu_ld1``,
    the market total return over the month after ``eom`` (NaN at the last month).

    w_t = w_{t-1} (1 + R_{m,t})  (Sec. 5.1.3): forward after the anchor, w_{t-1} = w_t / (1 + R_{m,t})
    before it. The R code only rolls back (its sample ends at the anchor), with w_{t-1} = w_t (1 - R_{m,t}),
    a first-order approximation that compounds to a noticeably lower early-sample wealth; we use the exact
    recursion (docs/deviations.md D9, D14).
    """
    end = pd.Timestamp(end)
    mkt = market[["eom_ret", "tret"]].sort_values("eom_ret")
    if not (mkt["eom_ret"] == end).any():
        raise ValueError(f"market returns must run through the wealth end date {end.date()}")
    if mkt["tret"].isna().any():
        raise ValueError("market returns contain missing values")
    eom = pd.DatetimeIndex([mkt["eom_ret"].iloc[0] - pd.offsets.MonthEnd(1), *mkt["eom_ret"]])
    tret = mkt["tret"].to_numpy()
    # Growth index at each month-end; wealth is the index rescaled to wealth_end at the anchor.
    level = np.concatenate([[1.0], np.cumprod(1.0 + tret)])
    return pd.DataFrame({
        "eom": eom,
        "wealth": wealth_end * level / level[eom.get_loc(end)],
        "mu_ld1": np.append(tret, np.nan),
    })


### `scripts/jkmp/covariance.py`

Barra-style covariance matrix (Eqs. 36–37). Port of reference/jkmp_r/3 - Estimate Covariance Matrix.R. Σ_t = X_t F_t X_tᵀ + diag(ivar_t) (monthly units: daily estimates × 21) * Loadings X_t: 6 sector dummies (merged Fama-French 12) + 6 theme scores, standardised each month over the factor-estimation universe (the top 500 by market cap). * Factor returns: each trading day, a no-intercept OLS of the top-500 stocks' excess returns on the previous month-end's loadings (R: ``lm(ret_exc ~ -1 + industries + clusters)``). * F_t: exponentially weighted 

**Role:** Calculation library. Other stages import its functions; this cell writes the module.

**Input:** the embedded source below. **Output:** `scripts/jkmp/covariance.py`. Expand the code cell to read or edit it.


In [ ]:
%%writefile scripts/jkmp/covariance.py
"""Barra-style covariance matrix (Eqs. 36–37). Port of reference/jkmp_r/3 - Estimate Covariance Matrix.R.

    Σ_t = X_t F_t X_tᵀ + diag(ivar_t)          (monthly units: daily estimates × 21)

* Loadings X_t: 6 sector dummies (merged Fama-French 12) + 6 theme scores, standardised each month
  over the factor-estimation universe (the top 500 by market cap).
* Factor returns: each trading day, a no-intercept OLS of the top-500 stocks' excess returns on the
  previous month-end's loadings (R: ``lm(ret_exc ~ -1 + industries + clusters)``).
* F_t: exponentially weighted factor covariance over the last 10 years of daily factor returns, with
  correlations from a 378-day half-life and variances from a 126-day half-life, combined as sd·cor·sd.
* ivar_t: EWMA of each stock's squared residuals (126-day half-life; ``ewma_vol`` ports ewma.cpp),
  kept only with 200 observations in the last 252 trading days; otherwise the median of the stock's
  size group that month.

Residuals are also computed for stocks outside the regression sample (return minus loadings × factor
returns), so a universe stock that falls out of the top 500 keeps an idiosyncratic-risk history.
"""

from __future__ import annotations

import logging
from collections.abc import Mapping, Sequence
from dataclasses import dataclass
from pathlib import Path

import numpy as np
import pandas as pd
from scipy.signal import lfilter

from jkmp.config import Config, DatesConfig
from jkmp.data.prepare import FF12_RANGES
from jkmp.evaluation.holdout import check_return_months
from jkmp.features import THEME_PREFIX, standardize_by_month

log = logging.getLogger(__name__)

SECTOR_PREFIX = "sec_"
FF12_NAMES = [name for name, _ in FF12_RANGES] + ["Other"]
COV_FILES = {"loadings": "cov_loadings.parquet", "factor": "cov_factor.parquet"}


# --------------------------------------------------------------------------- container


@dataclass(frozen=True, eq=False)
class CovParts:
    """Factor-model pieces of Σ at one month-end, for the universe stocks ``ids`` (sorted)."""

    eom: pd.Timestamp
    ids: np.ndarray            # (n,) stock ids
    factors: tuple[str, ...]   # (k,) factor names: sec_* then theme_*
    X: np.ndarray              # (n, k) loadings
    F: np.ndarray              # (k, k) monthly factor covariance
    ivar: np.ndarray           # (n,) monthly idiosyncratic variance

    def _rows(self, ids: Sequence[int] | None) -> np.ndarray | slice:
        if ids is None:
            return slice(None)
        pos = pd.Index(self.ids).get_indexer(np.asarray(ids))
        if (pos < 0).any():
            missing = np.asarray(ids)[pos < 0]
            raise KeyError(f"{len(missing)} ids have no covariance at {self.eom.date()}, e.g. {missing[:5]}")
        return pos

    def full(self, ids: Sequence[int] | None = None) -> np.ndarray:
        """Σ = X F Xᵀ + diag(ivar) for ``ids`` in the given order (default: all stocks)."""
        rows = self._rows(ids)
        X = self.X[rows]
        with np.errstate(all="ignore"):  # spurious Accelerate matmul flags; checked below
            sigma = X @ self.F @ X.T + np.diag(self.ivar[rows])
        return _finite(sigma, "Σ")

    def vol(self, ids: Sequence[int] | None = None) -> np.ndarray:
        """Monthly total volatility sqrt(diag Σ) without building Σ."""
        rows = self._rows(ids)
        X = self.X[rows]
        return _finite(np.sqrt(np.einsum("ij,jk,ik->i", X, self.F, X) + self.ivar[rows]), "vol")


def _finite(a: np.ndarray, what: str) -> np.ndarray:
    if not np.isfinite(a).all():
        raise FloatingPointError(f"non-finite {what}")
    return a


# --------------------------------------------------------------------------- estimators


def ewma_vol(x: np.ndarray, lam: float, start: int) -> np.ndarray:
    """EWMA volatility, a port of ``ewma_c`` (reference/jkmp_r/ewma.cpp).

    var[start] = sum(x[:start]²) / (start - 1 - #missing)       (mean assumed 0)
    var[j]     = lam·var[j-1] + (1 - lam)·x[j-1]²   for j > start  (uses data up to j-1 only)
                 var[j-1]                           if x[j-1] is missing
    Returns sqrt(var); NaN before ``start`` (and everywhere if len(x) <= start).
    """
    x = np.asarray(x, dtype=float)
    n = len(x)
    out = np.full(n, np.nan)
    if n <= start:
        return out
    head = x[:start]
    denom = start - 1 - int(np.isnan(head).sum())
    if denom <= 0:
        return out
    var = np.full(n, np.nan)
    var[start] = np.nansum(head**2) / denom
    sq = x**2
    if not np.isnan(sq[start:n - 1]).any():  # fast path: first-order recursive filter
        var[start + 1:] = lfilter([1.0 - lam], [1.0, -lam], sq[start:n - 1], zi=[lam * var[start]])[0]
    else:
        for j in range(start + 1, n):
            var[j] = var[j - 1] if np.isnan(sq[j - 1]) else lam * var[j - 1] + (1.0 - lam) * sq[j - 1]
    return np.sqrt(var)


def weighted_cov(x: np.ndarray, w: np.ndarray) -> np.ndarray:
    """R ``cov.wt(x, wt, center = TRUE, method = "unbiased")``: weights normalised to sum to one."""
    w = w / w.sum()
    # Apple Accelerate BLAS raises spurious divide/overflow flags in matmul with tiny weights (the
    # result matches einsum to 1e-16), so silence them and check the result explicitly instead.
    with np.errstate(all="ignore"):
        xc = x - w @ x
        cov = (xc * w[:, None]).T @ xc / (1.0 - np.sum(w**2))
    if not np.isfinite(cov).all():
        raise FloatingPointError("non-finite weighted covariance")
    return cov


def ewma_factor_cov(r: np.ndarray, half_life_corr: float, half_life_var: float) -> np.ndarray:
    """sd·cor·sd from two EWMA weightings of daily factor returns ``r`` (T × k, oldest first).

    R: weights (0.5^(1/hl))^(obs:1), tail(·, T) — the newest day has exponent 1.
    """
    age = np.arange(len(r), 0, -1, dtype=float)
    cov_c = weighted_cov(r, 0.5 ** (age / half_life_corr))
    sd_c = np.sqrt(np.diag(cov_c))
    cor = cov_c / np.outer(sd_c, sd_c)
    sd = np.sqrt(np.diag(weighted_cov(r, 0.5 ** (age / half_life_var))))
    return cor * np.outer(sd, sd)


# --------------------------------------------------------------------------- loadings


def sector_dummies(ff12: pd.Series, sectors: Mapping[str, Sequence[str]]) -> pd.DataFrame:
    """0/1 columns ``sec_<sector>``; every Fama-French 12 industry must map to exactly one sector."""
    lookup = {ind: sector for sector, inds in sectors.items() for ind in inds}
    unmapped = sorted(set(FF12_NAMES) - set(lookup))
    if unmapped:
        raise ValueError(f"covariance.sectors does not cover FF12 industries: {unmapped}")
    unknown = sorted(set(ff12.unique()) - set(lookup))
    if unknown:
        raise ValueError(f"unknown FF12 industries in data: {unknown}")
    sector = ff12.map(lookup)
    return pd.DataFrame({SECTOR_PREFIX + s: (sector == s).astype(float) for s in sectors}, index=ff12.index)


def monthly_loadings(panel: pd.DataFrame, top: pd.DataFrame, ids: Sequence[int],
                     sectors: Mapping[str, Sequence[str]]) -> pd.DataFrame:
    """Loadings at each month-end for ``ids``: eom, id, size_grp, in_top, in_universe, sec_*, theme_*.

    Theme scores are standardised by month with the mean and sd of the top-N stocks (R standardises
    over its estimation universe) and applied to every stock.
    """
    themes = [c for c in panel.columns if c.startswith(THEME_PREFIX)]
    rows = panel.loc[panel["id"].isin(ids), ["eom", "id", "size_grp", "ff12", "in_universe", *themes]]
    rows = rows.merge(top[["eom", "id"]].assign(in_top=True), on=["eom", "id"], how="left")
    rows["in_top"] = rows["in_top"].notna()
    rows[themes] = standardize_by_month(rows[themes], rows["eom"], within=rows["in_top"])
    dummies = sector_dummies(rows["ff12"], sectors)
    return pd.concat([rows.drop(columns="ff12"), dummies], axis=1)


# --------------------------------------------------------------------------- daily regressions


def daily_factor_regressions(loadings: pd.DataFrame, daily: pd.DataFrame, rf_daily: pd.Series,
                             factors: Sequence[str]) -> tuple[pd.DataFrame, pd.DataFrame]:
    """Daily cross-sectional OLS (no intercept) on the top-N stocks.

    Returns factor returns (index ``date``, one column per factor, plus ``n_obs`` and ``r2``) and
    residuals for every stock with loadings (``id, date, res, in_regression``). Returns in month t+1
    use loadings from month-end t.
    """
    d = daily.dropna(subset=["ret"])
    rf = d["date"].map(rf_daily)
    if rf.isna().any():
        log.warning("Dropping %d daily returns on dates without a Ken French RF", int(rf.isna().sum()))
    d = d.assign(ret_exc=d["ret"] - rf, eom=(d["date"].dt.to_period("M") - 1).dt.end_time.dt.normalize())
    d = d.dropna(subset=["ret_exc"]).rename(columns={"permno": "id"})
    d = d.merge(loadings[["eom", "id", "in_top", *factors]], on=["id", "eom"], how="inner")
    d = d.dropna(subset=list(factors)).sort_values(["date", "id"], ignore_index=True)

    X = d[list(factors)].to_numpy()
    y = d["ret_exc"].to_numpy()
    top = d["in_top"].to_numpy()
    dates, starts = np.unique(d["date"].to_numpy(), return_index=True)
    ends = np.append(starts[1:], len(d))
    k = len(factors)
    f = np.full((len(dates), k), np.nan)
    n_obs = np.zeros(len(dates), dtype=int)
    r2 = np.full(len(dates), np.nan)
    res = np.full(len(d), np.nan)
    with np.errstate(all="ignore"):  # spurious Accelerate matmul flags; checked below
        for i, (a, b) in enumerate(zip(starts, ends)):
            Xi, yi, ti = X[a:b], y[a:b], top[a:b]
            n_obs[i] = ti.sum()
            if n_obs[i] <= k:
                continue
            f[i] = np.linalg.lstsq(Xi[ti], yi[ti], rcond=None)[0]
            res[a:b] = yi - Xi @ f[i]
            rt, yt = res[a:b][ti], yi[ti]
            r2[i] = 1.0 - (rt @ rt) / ((yt - yt.mean()) @ (yt - yt.mean()))
    estimated = n_obs > k
    if not (np.isfinite(f[estimated]).all() and np.isfinite(res[np.repeat(estimated, ends - starts)]).all()):
        raise FloatingPointError("non-finite factor returns or residuals")
    fct = pd.DataFrame(f, index=pd.DatetimeIndex(dates, name="date"), columns=list(factors))
    fct = fct.assign(n_obs=n_obs, r2=r2).dropna(subset=list(factors))
    residuals = pd.DataFrame({"id": d["id"], "date": d["date"], "res": res, "in_regression": top}).dropna(subset=["res"])
    return fct, residuals


# --------------------------------------------------------------------------- idiosyncratic risk


def idiosyncratic_vol(residuals: pd.DataFrame, trading_days: pd.DatetimeIndex, half_life: float,
                      initial_obs: int, min_obs: int, min_window: int) -> pd.DataFrame:
    """Month-end EWMA residual volatility per stock (daily units): columns id, eom, res_vol.

    R: ``ewma_c(res, 0.5^(1/hl), start)`` by stock; keep a day only if the stock's observation
    ``min_obs`` rows back is within the last ``min_window`` trading days and the vol is defined;
    take the last remaining day of each month.
    """
    r = residuals.sort_values(["id", "date"], ignore_index=True)
    lam = 0.5 ** (1.0 / half_life)
    ids, starts = np.unique(r["id"].to_numpy(), return_index=True)
    ends = np.append(starts[1:], len(r))
    res = r["res"].to_numpy()
    vol = np.concatenate([ewma_vol(res[a:b], lam, initial_obs) for a, b in zip(starts, ends)])
    r["res_vol"] = vol

    days = pd.Series(trading_days.sort_values())
    window_start = pd.Series(days.shift(min_window).to_numpy(), index=days.to_numpy())
    date_back = r.groupby("id")["date"].shift(min_obs)
    keep = (date_back >= r["date"].map(window_start)) & r["res_vol"].notna()
    r = r[keep].assign(eom=lambda x: x["date"] + pd.offsets.MonthEnd(0))
    return r.groupby(["id", "eom"], as_index=False).tail(1)[["id", "eom", "res_vol"]].reset_index(drop=True)


# --------------------------------------------------------------------------- assembly


@dataclass
class CovarianceResult:
    parts: dict[pd.Timestamp, CovParts]
    factor_returns: pd.DataFrame   # daily, with n_obs and r2
    diagnostics: pd.DataFrame      # per month-end


def build_covariance(panel: pd.DataFrame, top: pd.DataFrame, daily: pd.DataFrame,
                     ff_daily: pd.DataFrame, cfg: Config) -> CovarianceResult:
    """CovParts for the universe stocks at every universe month-end with enough factor history."""
    c = cfg.covariance
    themes = [col for col in panel.columns if col.startswith(THEME_PREFIX)]
    factors = [SECTOR_PREFIX + s for s in c.sectors] + themes

    loadings = monthly_loadings(panel, top, daily["permno"].unique(), c.sectors)
    rf_daily = ff_daily.set_index("date")["rf"]
    fct, residuals = daily_factor_regressions(loadings, daily, rf_daily, factors)
    log.info("Daily regressions: %d days (%s to %s), median %d stocks, mean R² %.3f",
             len(fct), fct.index.min().date(), fct.index.max().date(), int(fct["n_obs"].median()), fct["r2"].mean())

    ivol = idiosyncratic_vol(residuals, fct.index, c.half_life_idio_days, c.idio_initial_obs,
                             c.idio_min_obs, c.idio_min_window)
    ivol_by_eom = dict(tuple(ivol.groupby("eom")))
    load_by_eom = dict(tuple(loadings.groupby("eom")))
    fr = fct[factors].to_numpy()
    fdates = fct.index.to_numpy()

    universe = panel.loc[panel["in_universe"], ["eom", "id"]]
    universe_size = universe.groupby("eom").size()
    parts, diag, skipped = {}, [], []
    for eom in pd.DatetimeIndex(universe_size.index).sort_values():
        end = int(np.searchsorted(fdates, np.datetime64(eom), side="right"))
        begin = max(0, end - c.window_days)
        if end - begin < c.min_window_days:
            skipped.append(eom)
            continue
        F = ewma_factor_cov(fr[begin:end], c.half_life_corr_days, c.half_life_var_days) * c.days_per_month

        L = load_by_eom[eom].set_index("id")
        pool = L[L["in_top"] | L["in_universe"]].join(
            ivol_by_eom.get(eom, pd.DataFrame(columns=["id", "res_vol"])).set_index("id")["res_vol"], how="left")
        # R: missing res_vol -> median of the stock's size group that month, then the overall median.
        fill = pool.groupby("size_grp")["res_vol"].transform("median").fillna(pool["res_vol"].median())
        res_vol = pool["res_vol"].fillna(fill)

        ids = np.sort(L.index[L["in_universe"]].to_numpy())
        if len(ids) != universe_size[eom]:
            raise ValueError(f"{universe_size[eom] - len(ids)} universe stocks at {eom.date()} have no loadings "
                             "(no CRSP daily data?)")
        parts[eom] = CovParts(
            eom=eom, ids=ids, factors=tuple(factors), X=L.loc[ids, factors].to_numpy(), F=F,
            ivar=res_vol.loc[ids].to_numpy() ** 2 * c.days_per_month,
        )
        diag.append({"eom": eom, "n_stocks": len(ids), "n_ivar_imputed": int(pool.loc[ids, "res_vol"].isna().sum()),
                     "n_factor_days": end - begin, "median_vol_annual": float(np.median(parts[eom].vol()) * np.sqrt(12))})
    if skipped:
        log.info("Skipped %d month-ends with fewer than %d factor-return days (%s to %s)",
                 len(skipped), c.min_window_days, skipped[0].date(), skipped[-1].date())
    if not parts:
        raise ValueError("no month-end has enough factor history")
    return CovarianceResult(parts, fct, pd.DataFrame(diag))


# --------------------------------------------------------------------------- storage


def save_covariance(parts: Mapping[pd.Timestamp, CovParts], out_dir: Path) -> None:
    """Two long tables: loadings + ivar per (eom, id), and F per (eom, factor)."""
    out_dir.mkdir(parents=True, exist_ok=True)
    load, fac = [], []
    for eom, p in parts.items():
        load.append(pd.DataFrame(p.X, columns=p.factors).assign(eom=eom, id=p.ids, ivar=p.ivar))
        fac.append(pd.DataFrame(p.F, columns=p.factors).assign(eom=eom, factor=p.factors))
    pd.concat(load, ignore_index=True).to_parquet(out_dir / COV_FILES["loadings"], index=False)
    pd.concat(fac, ignore_index=True).to_parquet(out_dir / COV_FILES["factor"], index=False)


def load_covariance(in_dir: Path) -> dict[pd.Timestamp, CovParts]:
    load = pd.read_parquet(in_dir / COV_FILES["loadings"])
    fac = pd.read_parquet(in_dir / COV_FILES["factor"])
    factors = tuple(c for c in fac.columns if c not in ("eom", "factor"))
    parts = {}
    for eom, F in fac.groupby("eom", sort=True):
        L = load[load["eom"] == eom].sort_values("id")
        F = F.set_index("factor").loc[list(factors), list(factors)]
        parts[eom] = CovParts(eom=eom, ids=L["id"].to_numpy(), factors=factors,
                              X=L[list(factors)].to_numpy(), F=F.to_numpy(), ivar=L["ivar"].to_numpy())
    return parts


# --------------------------------------------------------------------------- calibration


def vol_bias(parts: Mapping[pd.Timestamp, CovParts], panel: pd.DataFrame, months: Sequence[pd.Timestamp],
             dates: DatesConfig) -> pd.DataFrame:
    """Standardised next-month excess returns z = r / σ̂, with σ̂ from Σ at formation month ``eom``, for each
    universe stock (``stock``) and the equal- and value-weighted universe portfolios (``ew``, ``vw``; weights
    over the stocks with a return). A calibrated Σ has E[z²] = 1 (Barra bias statistic). Columns eom,
    eom_ret, scope, z."""
    months = pd.DatetimeIndex(sorted(months))
    check_return_months(months + pd.offsets.MonthEnd(1), dates, "vol_bias")
    rows = panel.loc[panel["in_universe"] & panel["eom"].isin(months) & panel["ret_ld1"].notna(),
                     ["eom", "id", "me", "ret_ld1"]]
    out = []
    for eom, g in rows.groupby("eom", sort=True):
        p = parts[eom]
        ids, r = g["id"].to_numpy(), g["ret_ld1"].to_numpy()
        out.append(pd.DataFrame({"eom": eom, "scope": "stock", "z": r / p.vol(ids)}))
        sigma = p.full(ids)
        for scope, w in (("ew", np.full(len(ids), 1.0 / len(ids))), ("vw", g["me"].to_numpy() / g["me"].sum())):
            with np.errstate(all="ignore"):  # spurious Accelerate matmul flags; checked below
                var = float(_finite(w @ sigma @ w, "portfolio variance"))
            out.append(pd.DataFrame({"eom": [eom], "scope": [scope], "z": [w @ r / np.sqrt(var)]}))
    z = pd.concat(out, ignore_index=True)
    return z.assign(eom_ret=z["eom"] + pd.offsets.MonthEnd(1))[["eom", "eom_ret", "scope", "z"]]


def bias_by_decade(z: pd.DataFrame) -> pd.DataFrame:
    """Realised/predicted volatility ratio sqrt(mean z²) per (decade of the return month, scope)."""
    d = z.assign(decade=z["eom_ret"].dt.year // 10 * 10)
    return (d.groupby(["decade", "scope"])["z"].agg(n="size", ratio=lambda s: float(np.sqrt(np.mean(s ** 2))))
            .reset_index())


### `scripts/jkmp/data/__init__.py`

Raw-data access and preparation.

**Role:** Calculation library. Other stages import its functions; this cell writes the module.

**Input:** the embedded source below. **Output:** `scripts/jkmp/data/__init__.py`. Expand the code cell to read or edit it.


In [ ]:
%%writefile scripts/jkmp/data/__init__.py
"""Raw-data access and preparation."""


### `scripts/jkmp/data/lag_history.py`

Offline lag-history completion and explicit per-stock/month exclusions.

**Role:** Calculation library. Other stages import its functions; this cell writes the module.

**Input:** the embedded source below. **Output:** `scripts/jkmp/data/lag_history.py`. Expand the code cell to read or edit it.


In [ ]:
%%writefile scripts/jkmp/data/lag_history.py
"""Offline lag-history completion and explicit per-stock/month exclusions."""
from pathlib import Path
import json

import pandas as pd
import pyarrow.parquet as pq

ROOT = Path(__file__).resolve().parents[3]


def complete_rows(rows, need, features):
    missing = pd.MultiIndex.from_frame(need[["id","eom"]]).difference(
        pd.MultiIndex.from_frame(rows[["id","eom"]]))
    if not len(missing):
        return rows
    additions = []
    columns = ["id","eom","g","ret_ld1","lambda",*features]
    for region in ("US","EU"):
        path = ROOT / f"data/inputs/lag_history_{region}.parquet"
        if path.exists():
            if not set(columns).issubset(pq.ParquetFile(path).schema.names):
                continue  # A different feature specification must not read an incompatible supplement.
            frame = pd.read_parquet(path, columns=columns)
            keys = pd.MultiIndex.from_frame(frame[["id","eom"]])
            additions.append(frame[keys.isin(missing)])
    if additions:
        rows = pd.concat([rows,*additions], ignore_index=True)
    still_missing = pd.MultiIndex.from_frame(need[["id","eom"]]).difference(pd.MultiIndex.from_frame(rows[["id","eom"]]))
    if len(still_missing):
        raise ValueError(f"{len(still_missing)} stock/months have no row in actual lagged signal history (unavailable); "
                         "apply the explicit membership-exclusion policy rather than invent history")
    return rows


def exclude_unavailable(panel, universe, lookback):
    """Exclude only affected formation months, allowing a later month once history exists."""
    region = "EU" if universe.startswith("eu") else "US"
    path = ROOT / f"data/inputs/unavailable_lag_history_{region}.csv"
    result = panel.copy()
    result["excluded_missing_lag_history"] = False
    events = []
    if path.exists():
        unavailable = pd.read_csv(path, parse_dates=["eom"])
        for row in unavailable.itertuples():
            mask = result.in_universe & result.id.eq(row.id) & result.eom.between(
                row.eom, row.eom+pd.offsets.MonthEnd(lookback+1))
            for affected in result.loc[mask,["id","eom"]].itertuples():
                events.append({"stock_id":int(affected.id), "formation_month_end":affected.eom,
                    "return_month_end":affected.eom+pd.offsets.MonthEnd(1),
                    "missing_history_month_end":row.eom,
                    "reason":"Excluded for this stock/month: required lagged history not reported by WRDS"})
            result.loc[mask,"excluded_missing_lag_history"] = True
            result.loc[mask,"in_universe"] = False
    fields=["stock_id","formation_month_end","return_month_end","missing_history_month_end","reason"]
    return result,pd.DataFrame(events,columns=fields)


### `scripts/jkmp/data/prepare.py`

Stage 1: the screened stock-month panel. Port of reference/jkmp_r/1 - Prepare Data.R. Timing (one row per stock ``id`` and formation month-end ``eom`` = t): * ``ret_ld1`` — excess return over month t+1 (what a portfolio formed at t earns) * ``tr_ld1`` — total return over month t+1 (drifts weights from t to t+1) * ``tr_ld0`` — total return over month t (the growth term g_t) * ``mu_ld1`` / ``mu_ld0`` — market total return over month t+1 / month t (wealth growth) Also on each row: ``lambda`` (Λ_i), ``g`` (weight drift over month t), ranked charact

**Role:** Calculation library. Other stages import its functions; this cell writes the module.

**Input:** the embedded source below. **Output:** `scripts/jkmp/data/prepare.py`. Expand the code cell to read or edit it.


In [ ]:
%%writefile scripts/jkmp/data/prepare.py
"""Stage 1: the screened stock-month panel. Port of reference/jkmp_r/1 - Prepare Data.R.

Timing (one row per stock ``id`` and formation month-end ``eom`` = t):

* ``ret_ld1`` — excess return over month t+1 (what a portfolio formed at t earns)
* ``tr_ld1``  — total return over month t+1 (drifts weights from t to t+1)
* ``tr_ld0``  — total return over month t (the growth term g_t)
* ``mu_ld1`` / ``mu_ld0`` — market total return over month t+1 / month t (wealth growth)

Also on each row: ``lambda`` (Λ_i), ``g`` (weight drift over month t), ranked characteristics,
``theme_*`` scores, ``valid_data`` (passes every screen and has 12 months of history) and
``in_universe`` (member of the top-80 buffered universe, ``jkmp.data.universe``).
"""

from __future__ import annotations

import logging
from dataclasses import dataclass

import numpy as np
import pandas as pd

from jkmp import costs, features
from jkmp.config import Config
from jkmp.data import universe
from jkmp.data.wrds_pull import RAW_FILES

log = logging.getLogger(__name__)

MONTH = pd.offsets.MonthEnd(1)

# Fama-French 12 industries from SIC codes (R: the case_when in 1 - Prepare Data.R; first match wins).
FF12_RANGES: list[tuple[str, list[tuple[int, int]]]] = [
    ("NoDur", [(100, 999), (2000, 2399), (2700, 2749), (2770, 2799), (3100, 3199), (3940, 3989)]),
    ("Durbl", [(2500, 2519), (3630, 3659), (3710, 3711), (3714, 3714), (3716, 3716), (3750, 3751),
               (3792, 3792), (3900, 3939), (3990, 3999)]),
    ("Manuf", [(2520, 2589), (2600, 2699), (2750, 2769), (3000, 3099), (3200, 3569), (3580, 3629),
               (3700, 3709), (3712, 3713), (3715, 3715), (3717, 3749), (3752, 3791), (3793, 3799),
               (3830, 3839), (3860, 3899)]),
    ("Enrgy", [(1200, 1399), (2900, 2999)]),
    ("Chems", [(2800, 2829), (2840, 2899)]),
    ("BusEq", [(3570, 3579), (3660, 3692), (3694, 3699), (3810, 3829), (7370, 7379)]),
    ("Telcm", [(4800, 4899)]),
    ("Utils", [(4900, 4949)]),
    ("Shops", [(5000, 5999), (7200, 7299), (7600, 7699)]),
    ("Hlth", [(2830, 2839), (3693, 3693), (3840, 3859), (8000, 8099)]),
    ("Money", [(6000, 6999)]),
]


# --------------------------------------------------------------------------- market inputs


def risk_free_monthly(ff_monthly: pd.DataFrame) -> pd.Series:
    """Ken French RF by month-end: the T-bill return over that month (decimal)."""
    return ff_monthly.set_index("date")["rf"].rename_axis("eom")


def market_total_returns(ff_monthly: pd.DataFrame, start: pd.Timestamp, end: pd.Timestamp) -> pd.DataFrame:
    """Market total return (Mkt-RF + RF) per return month in ``start .. end``: columns eom_ret, tret.

    R uses JKP's value-weighted US market (``market_returns.csv``); Ken French's market is the same
    concept — value-weighted CRSP common stocks (docs/deviations.md).
    """
    ff = ff_monthly[ff_monthly["date"].between(pd.Timestamp(start), pd.Timestamp(end))]
    return pd.DataFrame({"eom_ret": ff["date"], "tret": ff["mkt_rf"] + ff["rf"]}).reset_index(drop=True)


# --------------------------------------------------------------------------- panel construction


def add_timing_columns(chars: pd.DataFrame, rf: pd.Series, wealth: pd.DataFrame) -> pd.DataFrame:
    """Add ret_ld1, tr_ld1, tr_ld0, mu_ld1, mu_ld0 to JKP rows (needs ``ret_exc``, ``ret_exc_lead1m``).

    JKP's ``ret_exc`` at month t is the CRSP total return over t minus RF of month t, so total
    returns add back the RF of the *return* month: t+1 for ``tr_ld1``, t for ``tr_ld0``.
    (R adds RF of month t to ``tr_ld1``; see docs/deviations.md.)
    """
    needed = pd.Index(chars["eom"]).union(pd.Index(chars["eom"] + MONTH))
    missing = needed.difference(rf.index)
    in_sample = missing[missing <= rf.index.max()]
    if len(in_sample):
        raise ValueError(f"risk-free rate missing for {len(in_sample)} months, e.g. {in_sample[0].date()}")
    out = chars.copy()
    mu = wealth.set_index("eom")["mu_ld1"]
    out["ret_ld1"] = out["ret_exc_lead1m"]
    out["tr_ld1"] = out["ret_ld1"] + (out["eom"] + MONTH).map(rf)
    out["tr_ld0"] = out["ret_exc"] + out["eom"].map(rf)
    out["mu_ld1"] = out["eom"].map(mu)
    out["mu_ld0"] = (out["eom"] - MONTH).map(mu)
    return out


def apply_screens(panel: pd.DataFrame, cfg: Config) -> tuple[pd.DataFrame, pd.DataFrame]:
    """Drop rows failing each screen in turn (R order); return the panel and a per-screen log."""
    feats = list(cfg.features.names)
    u, d = cfg.universe, cfg.dates
    min_feat = int(np.floor(len(feats) * u.min_feature_pct))  # R: floor(length(features)*feat_pct)
    steps = [
        ("common stock (exchange, share code)",
         lambda p: p["crsp_exchcd"].isin(u.exchanges) & p["crsp_shrcd"].isin(u.share_codes)),
        ("sample dates", lambda p: p["eom"].between(pd.Timestamp(d.warmup_start), pd.Timestamp(d.data_end))),
        ("non-missing me", lambda p: p["me"].notna()),
        ("returns at t and t+1", lambda p: p["tr_ld0"].notna() & p["tr_ld1"].notna()),
        ("dolvol > 0", lambda p: p["dolvol"].gt(0)),  # NaN fails too
        ("SIC code", lambda p: p["sic"].notna()),
        (f"at least {min_feat} of {len(feats)} features", lambda p: p[feats].notna().sum(axis=1) >= min_feat),
    ]
    n_start, me_start = len(panel), panel["me"].sum()
    rows = []
    for name, keep in steps:
        mask = keep(panel)
        excluded = int((~mask).sum())
        rows.append({"screen": name, "rows_before": len(panel), "excluded": excluded,
                     "pct_excluded": 100.0 * excluded / max(len(panel), 1)})
        log.info("   %-40s excludes %6.2f%% of the observations", name, rows[-1]["pct_excluded"])
        panel = panel[mask]
    log.info("   Screened panel keeps %.2f%% of the observations and %.2f%% of the market cap",
             100.0 * len(panel) / n_start, 100.0 * panel["me"].sum() / me_start)
    return panel.reset_index(drop=True), pd.DataFrame(rows)


def flag_history(panel: pd.DataFrame, n_lags: int) -> pd.Series:
    """True where the row ``n_lags`` back for the same stock is exactly ``n_lags`` months earlier.

    R: ``eom_lag := shift(eom, n = lb)``, ``month_diff == lb``, with lb = lb_hor + 1 = 12, applied
    after the screens — so the stock must pass every screen in each of the last 13 months.
    """
    ordered = panel.sort_values(["id", "eom"])
    lag = ordered.groupby("id")["eom"].shift(n_lags)
    month_diff = (ordered["eom"].dt.year - lag.dt.year) * 12 + (ordered["eom"].dt.month - lag.dt.month)
    return month_diff.eq(n_lags).reindex(panel.index)


def ff12_industry(sic: pd.Series) -> pd.Series:
    out = pd.Series("Other", index=sic.index, dtype="object")
    assigned = pd.Series(False, index=sic.index)
    for name, ranges in FF12_RANGES:
        hit = pd.Series(False, index=sic.index)
        for lo, hi in ranges:
            hit |= sic.between(lo, hi)
        hit &= ~assigned
        out[hit] = name
        assigned |= hit
    return out


@dataclass
class PreparedData:
    panel: pd.DataFrame        # screened stock-months with timing, costs, features, themes, universe flags
    wealth: pd.DataFrame       # eom, wealth, mu_ld1
    screen_log: pd.DataFrame   # rows excluded by each screen
    membership: pd.DataFrame   # universe decisions per reconstitution
    universe_stats: pd.DataFrame


def build_panel(cfg: Config) -> PreparedData:
    """Read data/raw and build every Stage 1 output."""
    raw = cfg.paths.raw
    feats = list(cfg.features.names)
    chars = pd.read_parquet(raw / RAW_FILES["jkp"])
    ff = pd.read_parquet(raw / RAW_FILES["ff_monthly"])

    wealth = costs.wealth_path(
        cfg.investor.wealth, cfg.investor.wealth_end_date,
        market_total_returns(ff, start=pd.Timestamp(cfg.dates.warmup_start), end=pd.Timestamp(cfg.dates.data_end)),
    )
    dolvol_scale = 1e6 if cfg.trading_costs.dolvol_units == "millions" else 1.0
    chars["dolvol"] = chars[cfg.trading_costs.dolvol_column] * dolvol_scale

    panel = add_timing_columns(chars, risk_free_monthly(ff), wealth)
    panel, screen_log = apply_screens(panel, cfg)

    n_lags = cfg.universe.lookback_months + 1  # R: lb <- pf_set$lb_hor + 1
    panel["valid_data"] = flag_history(panel, n_lags)
    log.info("   %d-month history flag marks %.2f%% of rows invalid", n_lags, 100.0 * (~panel["valid_data"]).mean())

    panel[feats] = features.rank_features(panel, feats, cfg.features.percentile_rank, cfg.features.impute_value)
    directions = features.load_theme_directions(raw, cfg.features.themes)
    themes = features.theme_scores(panel[feats], directions)
    panel[themes.columns] = themes
    panel["ff12"] = ff12_industry(panel["sic"])
    panel["lambda"] = costs.price_impact_lambda(panel["dolvol"], cfg.trading_costs.price_impact)
    panel["g"] = costs.drift_factor(panel["tr_ld0"], panel["mu_ld0"])

    u = cfg.universe
    panel, membership, stats = universe.build_universe(
        panel, u.target_n, u.entry_rank, u.exit_rank, u.reconstitution_month)

    columns = ["id", "eom", "sic", "ff12", "size_grp", "crsp_exchcd", "crsp_shrcd", "me", "dolvol", "lambda",
               "ret_exc", "ret_ld1", "tr_ld0", "tr_ld1", "mu_ld0", "mu_ld1", "g",
               "valid_data", "recon_eom", "in_universe", *feats, *themes.columns]
    panel = panel[columns].sort_values(["eom", "id"], ignore_index=True)
    return PreparedData(panel, wealth, screen_log, membership, stats)


### `scripts/jkmp/data/universe.py`

Investable universe: the largest stocks with a rank buffer, reconstituted once a year. At each reconstitution month-end (December of year y), among stocks with ``valid_data``: 1. rank by market cap ``me`` (1 = largest; ties broken by ``id``); 2. keep current members still ranked <= ``exit_rank`` (100); 3. add every non-member ranked <= ``entry_rank`` (70); 4. fill with the next-largest non-members until there are ``target_n`` (80). Steps 2–3 can give more than ``target_n`` members (no trimming), so the size is reported each year. Members hold f

**Role:** Calculation library. Other stages import its functions; this cell writes the module.

**Input:** the embedded source below. **Output:** `scripts/jkmp/data/universe.py`. Expand the code cell to read or edit it.


In [ ]:
%%writefile scripts/jkmp/data/universe.py
"""Investable universe: the largest stocks with a rank buffer, reconstituted once a year.

At each reconstitution month-end (December of year y), among stocks with ``valid_data``:

1. rank by market cap ``me`` (1 = largest; ties broken by ``id``);
2. keep current members still ranked <= ``exit_rank`` (100);
3. add every non-member ranked <= ``entry_rank`` (70);
4. fill with the next-largest non-members until there are ``target_n`` (80).

Steps 2–3 can give more than ``target_n`` members (no trimming), so the size is reported each year.

Members hold for the 12 formation months after reconstitution (Jan–Dec of y+1). A member leaves
early only when it has no valid row — it delists or fails a screen. Its last month in the universe is
its last valid row, whose ``ret_ld1`` is its final return (JKP returns include delisting returns), and
it does not re-enter before the next reconstitution, so it cannot flicker in and out.

Replaces R's size screen + 12/12 addition/deletion rule (docs/deviations.md D3).
"""

from __future__ import annotations

import logging

import numpy as np
import pandas as pd

log = logging.getLogger(__name__)

HOLD_MONTHS = 12


def reconstitute(panel: pd.DataFrame, target_n: int, entry_rank: int, exit_rank: int,
                 month: int) -> pd.DataFrame:
    """Membership decisions: one row per (``recon_eom``, ``id``) with ``me_rank`` and ``status``.

    ``status`` is ``kept`` (incumbent within the exit rank), ``added`` (newcomer within the entry
    rank) or ``filled`` (next-largest newcomer, to reach ``target_n``).
    """
    eligible = panel.loc[panel["valid_data"] & (panel["eom"].dt.month == month), ["eom", "id", "me"]]
    members: set[int] = set()
    decisions = []
    for recon_eom, cs in eligible.groupby("eom", sort=True):
        cs = cs.sort_values(["me", "id"], ascending=[False, True]).reset_index(drop=True)
        cs["me_rank"] = np.arange(1, len(cs) + 1)
        incumbent = cs["id"].isin(members)
        keep = incumbent & (cs["me_rank"] <= exit_rank)
        add = ~incumbent & (cs["me_rank"] <= entry_rank)
        fill = pd.Series(False, index=cs.index)
        short = target_n - int((keep | add).sum())
        if short > 0:
            fill[cs.index[~incumbent & ~add][:short]] = True  # cs is in rank order
        chosen = cs[keep | add | fill].assign(
            status=np.select([keep, add, fill], ["kept", "added", "filled"], default="")[keep | add | fill],
            recon_eom=recon_eom,
        )
        decisions.append(chosen[["recon_eom", "id", "me_rank", "status"]])
        members = set(chosen["id"])
    if not decisions:
        raise ValueError(f"no valid stock-months in reconstitution month {month}")
    return pd.concat(decisions, ignore_index=True)


def _add_months(eom: pd.Series, k: pd.Series | int) -> pd.Series:
    return (eom.dt.to_period("M") + k).dt.end_time.dt.normalize()


def flag_universe(panel: pd.DataFrame, membership: pd.DataFrame) -> tuple[pd.DataFrame, pd.DataFrame]:
    """Mark ``in_universe`` on the panel and record how each membership ended.

    Returns the panel with ``recon_eom`` (the reconstitution behind the membership; NaT outside it) and
    ``in_universe``, and the membership table with ``last_eom`` and ``exit``: ``held`` (all months to
    the next reconstitution or the end of the data), ``left_sample`` (no later valid rows at all —
    delisted, merged or dropped by JKP) or ``screened_out`` (later rows exist but failed a screen).
    """
    last_month = panel["eom"].max()
    grid = membership[["recon_eom", "id"]].merge(pd.DataFrame({"k": range(1, HOLD_MONTHS + 1)}), how="cross")
    grid["eom"] = _add_months(grid["recon_eom"], grid["k"])
    grid = grid[grid["eom"] <= last_month]

    valid = panel.loc[panel["valid_data"], ["id", "eom"]].assign(ok=1)
    grid = grid.merge(valid, on=["id", "eom"], how="left").fillna({"ok": 0})
    grid = grid.sort_values(["recon_eom", "id", "k"])
    grid["active"] = grid.groupby(["recon_eom", "id"])["ok"].cummin().astype(bool)  # out once a month fails

    active = grid[grid["active"]]
    out = panel.merge(active[["id", "eom", "recon_eom"]].assign(in_universe=True), on=["id", "eom"], how="left")
    out["in_universe"] = out["in_universe"].notna()  # True where matched to an active membership

    ends = active.groupby(["recon_eom", "id"]).agg(last_eom=("eom", "max"), n_months=("k", "size")).reset_index()
    possible = grid.groupby(["recon_eom", "id"]).size().rename("n_possible").reset_index()
    m = membership.merge(possible, on=["recon_eom", "id"], how="left").merge(ends, on=["recon_eom", "id"], how="left")
    m["n_months"] = m["n_months"].fillna(0).astype(int)
    last_valid = valid.groupby("id")["eom"].max()
    held = m["n_months"] >= m["n_possible"].fillna(0)
    gone = m["last_eom"].fillna(m["recon_eom"]) >= m["id"].map(last_valid)
    m["exit"] = np.select([held, gone], ["held", "left_sample"], default="screened_out")
    return out, m.drop(columns="n_possible")


def universe_stats(membership: pd.DataFrame) -> pd.DataFrame:
    """Per membership year: size, kept / added / filled, exits at reconstitution, turnover, early exits.

    ``turnover`` = share of names that are new at reconstitution (added + filled) / size.
    """
    rows, prev_size = [], 0
    for recon_eom, g in membership.groupby("recon_eom", sort=True):
        counts = g["status"].value_counts()
        kept, entrants = int(counts.get("kept", 0)), int(counts.get("added", 0) + counts.get("filled", 0))
        exits = g["exit"].value_counts()
        rows.append({
            "year": recon_eom.year + 1,
            "size": len(g),
            "kept": kept,
            "added": int(counts.get("added", 0)),
            "filled": int(counts.get("filled", 0)),
            "dropped_at_recon": prev_size - kept,
            "turnover": entrants / len(g),
            "early_exit_left_sample": int(exits.get("left_sample", 0)),
            "early_exit_screened_out": int(exits.get("screened_out", 0)),
            "avg_monthly_size": g["n_months"].sum() / max(g["n_months"].max(), 1),
        })
        prev_size = len(g)
    return pd.DataFrame(rows)


def build_universe(panel: pd.DataFrame, target_n: int, entry_rank: int, exit_rank: int,
                   month: int) -> tuple[pd.DataFrame, pd.DataFrame, pd.DataFrame]:
    """Reconstitute, flag the panel, and summarise: returns (panel, membership, stats)."""
    membership = reconstitute(panel, target_n, entry_rank, exit_rank, month)
    panel, membership = flag_universe(panel, membership)
    stats = universe_stats(membership)
    log.info("Universe: %d reconstitutions, size %d-%d, mean turnover %.1f%%", len(stats),
             stats["size"].min(), stats["size"].max(), 100 * stats["turnover"].iloc[1:].mean())
    return panel, membership, stats


### `scripts/jkmp/data/wrds_pull.py`

Stage 0: locate and pull the raw data (WRDS, JKP reference files, Ken French). Credentials never live in code or config: ``connect()`` reads them from environment variables, which may be set in the shell or in the git-ignored project ``.env`` (template: ``.env.example``). ``explore()`` finds candidate WRDS tables and checks them against what the pipeline needs; ``pull_all()`` writes every raw input to ``data/raw/`` (skipping files that already exist).

**Role:** Calculation library. Other stages import its functions; this cell writes the module.

**Input:** the embedded source below. **Output:** `scripts/jkmp/data/wrds_pull.py`. Expand the code cell to read or edit it.


In [ ]:
%%writefile scripts/jkmp/data/wrds_pull.py
"""Stage 0: locate and pull the raw data (WRDS, JKP reference files, Ken French).

Credentials never live in code or config: ``connect()`` reads them from environment variables, which may be
set in the shell or in the git-ignored project ``.env`` (template: ``.env.example``).

``explore()`` finds candidate WRDS tables and checks them against what the pipeline needs;
``pull_all()`` writes every raw input to ``data/raw/`` (skipping files that already exist).
"""

from __future__ import annotations

import io
import logging
import os
import re
import urllib.request
import zipfile
from collections.abc import Callable, Sequence
from dataclasses import dataclass, field
from pathlib import Path

from typing import TYPE_CHECKING

import pandas as pd

from jkmp.config import PROJECT_ROOT, Config

if TYPE_CHECKING:  # wrds (and its sqlalchemy) are imported only where a connection is used
    import wrds

log = logging.getLogger(__name__)

# Identifier and bookkeeping columns the JKP extract needs besides the characteristics in the config.
# size_grp: size-group median for missing idiosyncratic variance (R, 3 - Estimate Covariance Matrix.R).
# ret_exc: return over the month ending at eom, for the growth term g_t (tr_ld0 in R).
JKP_BASE_COLUMNS = (
    "id", "eom", "excntry", "sic", "size_grp", "me", "crsp_exchcd", "crsp_shrcd",
    "dolvol_126d", "ret_exc", "ret_exc_lead1m",
)

# crsp.dsf_v2 column names -> ours.
CRSP_V2_COLUMNS = {"permno": "permno", "dlycaldt": "date", "dlyret": "ret"}
CRSP_PERMNO_BATCH = 250

# Output files in data/raw/ (the top-N file name uses covariance.factor_universe_top_n).
RAW_FILES = {
    "jkp": "jkp_usa.parquet",
    "daily": "crsp_daily.parquet",
    "ff_monthly": "ff_factors_monthly.parquet",
    "ff_daily": "ff_factors_daily.parquet",
    "factor_details": "factor_details.parquet",
    "cluster_labels": "cluster_labels.parquet",
    "counts": "row_counts_by_month.csv",
}

_IDENTIFIER = re.compile(r"^[a-z_][a-z0-9_]*$")

# Table-name patterns (SQL ILIKE) for each dataset we need to locate.
CANDIDATE_PATTERNS = {
    "jkp": ["%global_factor%", "%jkp%"],
    "crsp_daily": ["dsf", "dsf_v2", "dsf62", "stkdlysecuritydata", "dsi", "wrds_dsfv2%"],
}

# Probe stock for cheap date-coverage checks: Apple, PERMNO 14593 (also its JKP id).
PROBE_PERMNO = 14593
DATE_COLUMN_CANDIDATES = ("eom", "date", "dlycaldt", "caldt")
QUERY_TIMEOUT_SECONDS = 180


# --------------------------------------------------------------------------- connection


def connect(env_file: Path = PROJECT_ROOT / ".env") -> wrds.Connection:
    """Open a WRDS connection: ``wrds.Connection(wrds_username=..., api_key=...)``.

    Reads WRDS_USERNAME and WRDS_API_KEY from the environment, after loading ``env_file`` if it exists;
    variables already set in the shell win over the file.
    wrds 3.5.0 accepts but ignores ``api_key``; the password is then found in ~/.pgpass.
    Set WRDS_PASSWORD only to override ~/.pgpass — never put the API key there.
    """
    raise RuntimeError('Downloads disabled: Keats workflow uses local inputs only.')
    from dotenv import load_dotenv
    import wrds

    load_dotenv(env_file, override=False)
    kwargs = {}
    if username := os.environ.get("WRDS_USERNAME"):
        kwargs["wrds_username"] = username
    if api_key := os.environ.get("WRDS_API_KEY"):
        kwargs["api_key"] = api_key
    if password := os.environ.get("WRDS_PASSWORD"):
        kwargs["wrds_password"] = password
    return wrds.Connection(**kwargs)


def _set_timeout(db: wrds.Connection, seconds: int) -> None:
    import sqlalchemy as sa

    db.connection.execute(sa.text(f"set statement_timeout = {int(seconds * 1000)}"))


# --------------------------------------------------------------------------- exploration


@dataclass
class TableReport:
    dataset: str
    schema: str
    table: str
    n_columns: int
    date_column: str | None
    missing_columns: list[str] = field(default_factory=list)
    coverage: str = ""


@dataclass
class ExploreReport:
    libraries: list[str]
    candidates: pd.DataFrame
    tables: list[TableReport]
    feature_check: pd.DataFrame


def find_candidate_tables(db: wrds.Connection) -> pd.DataFrame:
    """Tables/views the user can see whose names match ``CANDIDATE_PATTERNS``."""
    frames = []
    for dataset, patterns in CANDIDATE_PATTERNS.items():
        df = db.raw_sql(
            """
            select table_schema as schema, table_name as table, table_type
            from information_schema.tables
            where table_name ilike any(%(patterns)s)
            order by table_schema, table_name
            """,
            params={"patterns": patterns},
        )
        if dataset == "crsp_daily":
            df = df[df["schema"].str.lower().str.startswith("crsp")]
        frames.append(df.assign(dataset=dataset))
    return pd.concat(frames, ignore_index=True)


def table_columns(db: wrds.Connection, schema: str, table: str) -> list[str]:
    df = db.raw_sql(
        """
        select column_name from information_schema.columns
        where table_schema = %(schema)s and table_name = %(table)s
        order by ordinal_position
        """,
        params={"schema": schema, "table": table},
    )
    return df["column_name"].str.lower().tolist()


def probe_coverage(db: wrds.Connection, schema: str, table: str, columns: list[str]) -> tuple[str | None, str]:
    """Date range for the probe stock (index-friendly), as a printable string."""
    date_col = next((c for c in DATE_COLUMN_CANDIDATES if c in columns), None)
    id_col = next((c for c in ("permno", "id") if c in columns), None)
    if date_col is None:
        return None, "no recognised date column"
    if id_col is None:
        return date_col, "no permno/id column to probe"
    where = f"{id_col} = %(probe)s"
    if "excntry" in columns:
        where += " and excntry = 'USA'"
    try:
        _set_timeout(db, QUERY_TIMEOUT_SECONDS)
        df = db.raw_sql(
            f'select min({date_col}) as first, max({date_col}) as last, count(*) as n '
            f'from "{schema}"."{table}" where {where}',
            params={"probe": PROBE_PERMNO},
        )
    except Exception as err:  # timeouts and permission errors are findings, not failures
        return date_col, f"probe failed: {type(err).__name__}: {str(err).splitlines()[0][:120]}"
    finally:
        _set_timeout(db, 0)
    row = df.iloc[0]
    return date_col, f"{row['first']} to {row['last']} ({row['n']} rows for PERMNO {PROBE_PERMNO})"


def download_reference_file(url: str, dest_dir: Path) -> Path:
    """Download a small public file (JKP reference files) unless it already exists."""
    dest = dest_dir / urllib.request.url2pathname(url.rsplit("/", 1)[-1])
    if not dest.exists():
        dest_dir.mkdir(parents=True, exist_ok=True)
        log.info("Downloading %s", url)
        dest.write_bytes(_http_get(url))
    return dest


def check_features(cfg: Config, factor_details: Path, cluster_labels: Path) -> pd.DataFrame:
    """Compare config features against JKP's Factor Details (names) and Cluster Labels (themes)."""
    details = pd.read_excel(factor_details)
    known = set(details["abr_jkp"].dropna().astype(str))
    direction = details.dropna(subset=["abr_jkp"]).set_index("abr_jkp")["direction"]
    labels = pd.read_csv(cluster_labels)
    # R (1 - Prepare Data.R): lower-case, spaces/hyphens -> underscores; rvol_252d added to low_risk.
    clusters = labels["cluster"].str.lower().str.replace(r"[\s-]", "_", regex=True)
    theme_of = dict(zip(labels["characteristic"], clusters)) | {"rvol_252d": "low_risk"}
    rows = []
    for feature in cfg.features.names:
        jkp_theme = theme_of.get(feature)
        rows.append({
            "feature": feature,
            "config_theme": cfg.features.theme_of[feature],
            "in_factor_details": feature in known,
            "jkp_theme": jkp_theme,
            "theme_matches": jkp_theme == cfg.features.theme_of[feature],
            "direction": direction.get(feature),
        })
    return pd.DataFrame(rows)


def explore(db: wrds.Connection, cfg: Config) -> ExploreReport:
    """Find JKP and CRSP daily candidates, check required columns and coverage, check features."""
    libraries = sorted(db.list_libraries())
    candidates = find_candidate_tables(db)
    required = {"jkp": list(JKP_BASE_COLUMNS) + list(cfg.features.names), "crsp_daily": ["permno"]}
    tables = []
    for row in candidates.itertuples(index=False):
        columns = table_columns(db, row.schema, row.table)
        date_col, coverage = probe_coverage(db, row.schema, row.table, columns)
        tables.append(TableReport(
            dataset=row.dataset, schema=row.schema, table=row.table, n_columns=len(columns),
            date_column=date_col,
            missing_columns=[c for c in required[row.dataset] if c not in columns],
            coverage=coverage,
        ))
    details = download_reference_file(cfg.data_sources.factor_details_url, cfg.paths.raw)
    clusters = download_reference_file(cfg.data_sources.cluster_labels_url, cfg.paths.raw)
    return ExploreReport(libraries, candidates, tables, check_features(cfg, details, clusters))


def format_report(report: ExploreReport) -> str:
    interesting = [lib for lib in report.libraries if any(k in lib.lower() for k in ("crsp", "contrib", "jkp", "global"))]
    lines = [
        f"WRDS libraries visible: {len(report.libraries)}",
        f"  relevant: {', '.join(interesting) or '(none)'}",
        "",
        "Candidate tables:",
    ]
    for t in report.tables:
        missing = ", ".join(t.missing_columns) if t.missing_columns else "none"
        lines += [
            f"  [{t.dataset}] {t.schema}.{t.table}  ({t.n_columns} columns, date column: {t.date_column})",
            f"      coverage: {t.coverage}",
            f"      missing required columns: {missing}",
        ]
    if not report.tables:
        lines.append("  (none found)")
    lines += ["", "Feature check against Factor Details.xlsx / Cluster Labels.csv:",
              report.feature_check.to_string(index=False)]
    return "\n".join(lines)


# --------------------------------------------------------------------------- SQL helpers


def _identifier(name: str) -> str:
    """Quote a column/schema/table name after checking it is a plain identifier (no injection)."""
    if not _IDENTIFIER.match(name):
        raise ValueError(f"not a plain SQL identifier: {name!r}")
    return f'"{name}"'


def _qualified(table: str) -> str:
    schema, sep, name = table.partition(".")
    if not sep:
        raise ValueError(f"table must be 'schema.table', got {table!r}")
    return f"{_identifier(schema)}.{_identifier(name)}"


# --------------------------------------------------------------------------- pulls


def pull_jkp(db: wrds.Connection, cfg: Config) -> pd.DataFrame:
    """JKP characteristics for US common stocks on the configured exchanges, warm-up to data end."""
    columns = [c for c in JKP_BASE_COLUMNS if c != "excntry"] + list(cfg.features.names)
    flags = "".join(f" and {_identifier(f)} = 1" for f in cfg.universe.jkp_flags)
    sql = f"""
        select {", ".join(_identifier(c) for c in columns)}
        from {_qualified(cfg.data_sources.jkp_table)}
        where excntry = 'USA'
          and eom between %(start)s and %(end)s
          and id <= 99999
          and crsp_exchcd = any(%(exchanges)s)
          and crsp_shrcd = any(%(share_codes)s){flags}
    """
    params = {
        "start": cfg.dates.warmup_start,
        "end": cfg.dates.data_end,
        "exchanges": list(cfg.universe.exchanges),
        "share_codes": list(cfg.universe.share_codes),
    }
    log.info("Pulling JKP from %s (%s to %s) ...", cfg.data_sources.jkp_table, params["start"], params["end"])
    df = db.raw_sql(sql, params=params, date_cols=["eom"])
    df = _tidy_numeric(df, int_cols=("id",))
    dupes = df.duplicated(["id", "eom"]).sum()
    if dupes:
        log.warning("JKP extract has %d duplicate (id, eom) rows", dupes)
    return df.sort_values(["eom", "id"], ignore_index=True)


def top_n_by_month(jkp: pd.DataFrame, n: int) -> pd.DataFrame:
    """The n largest stocks by market cap each month: columns eom, id, me, me_rank (1 = largest)."""
    ranked = jkp.loc[jkp["me"].notna(), ["eom", "id", "me"]].sort_values(["eom", "id"])
    ranked["me_rank"] = ranked.groupby("eom")["me"].rank(method="first", ascending=False).astype("int64")
    return ranked[ranked["me_rank"] <= n].sort_values(["eom", "me_rank"], ignore_index=True)


def pull_crsp_daily(db: wrds.Connection, cfg: Config, permnos: Sequence[int]) -> pd.DataFrame:
    """Daily total returns from the CRSP v2 file for the given PERMNOs, over the full sample."""
    select = ", ".join(f"{_identifier(src)} as {_identifier(dst)}" for src, dst in CRSP_V2_COLUMNS.items())
    sql = f"""
        select {select}
        from {_qualified(cfg.data_sources.crsp_daily_table)}
        where permno = any(%(permnos)s) and dlycaldt between %(start)s and %(end)s
    """
    start, end = cfg.dates.warmup_start.replace(day=1), cfg.dates.data_end
    permnos = sorted(int(p) for p in permnos)
    frames = []
    for i in range(0, len(permnos), CRSP_PERMNO_BATCH):
        batch = permnos[i:i + CRSP_PERMNO_BATCH]
        frames.append(db.raw_sql(sql, params={"permnos": batch, "start": start, "end": end}, date_cols=["date"]))
        log.info("CRSP daily: %d/%d PERMNOs, %d rows so far",
                 i + len(batch), len(permnos), sum(len(f) for f in frames))
    df = _tidy_numeric(pd.concat(frames, ignore_index=True), int_cols=("permno",))
    return df.sort_values(["permno", "date"], ignore_index=True)


def _http_get(url: str) -> bytes:
    raise RuntimeError("Downloads disabled: Keats workflow uses local inputs only.")
    request = urllib.request.Request(url, headers={"User-Agent": "Mozilla/5.0 (jkmp replication)"})
    with urllib.request.urlopen(request, timeout=120) as resp:
        return resp.read()


def read_french_zip(content: bytes) -> pd.DataFrame:
    """Parse a Ken French factor CSV zip (monthly or daily section) into decimals.

    Returns columns ``date`` (month-end for monthly files) plus e.g. mkt_rf, smb, hml, rf.
    Rows with 4-digit keys (the annual section of monthly files) are dropped.
    """
    with zipfile.ZipFile(io.BytesIO(content)) as zf:
        text = zf.read(zf.namelist()[0]).decode("latin-1")
    header, rows = None, []
    for line in text.splitlines():
        parts = [p.strip() for p in line.split(",")]
        if header is None and len(parts) > 1 and parts[0] == "" and parts[1]:
            header = [p.lower().replace("-", "_") for p in parts[1:]]
        elif header is not None and parts[0].isdigit() and len(parts[0]) in (6, 8):
            rows.append(parts[: len(header) + 1])
    if header is None or not rows:
        raise ValueError("unrecognised Ken French file layout")
    df = pd.DataFrame(rows, columns=["key", *header])
    key_len = df["key"].str.len()
    if key_len.nunique() != 1:
        raise ValueError("Ken French file mixes monthly and daily rows")
    if key_len.iloc[0] == 6:
        date = pd.to_datetime(df["key"], format="%Y%m") + pd.offsets.MonthEnd(0)
    else:
        date = pd.to_datetime(df["key"], format="%Y%m%d")
    values = df[header].astype("float64") / 100.0  # French files are in percent
    return pd.concat([date.rename("date"), values], axis=1)


def _tidy_numeric(df: pd.DataFrame, int_cols: Sequence[str] = ()) -> pd.DataFrame:
    """Nullable WRDS dtypes -> plain int64 identifiers and float64 values (NaN for missing)."""
    df = df.copy()
    for col in df.columns:
        if col in int_cols:
            df[col] = df[col].astype("int64")
        elif pd.api.types.is_numeric_dtype(df[col]) and not pd.api.types.is_bool_dtype(df[col]):
            df[col] = df[col].astype("float64")
    return df


def _stringify_objects(df: pd.DataFrame) -> pd.DataFrame:
    """Mixed-type spreadsheet columns -> strings, so they can be written to parquet."""
    df = df.copy()
    for col in df.columns[df.dtypes == object]:
        df[col] = df[col].astype("string")
    return df


# --------------------------------------------------------------------------- bookkeeping


def row_counts_by_month(jkp: pd.DataFrame, top: pd.DataFrame, daily: pd.DataFrame) -> pd.DataFrame:
    """Per month: JKP rows, non-missing me / next-month return, top-N size, CRSP daily rows and stocks."""
    jkp_counts = jkp.groupby("eom").agg(
        jkp_rows=("id", "size"),
        me_nonnull=("me", "count"),
        ret_lead1m_nonnull=("ret_exc_lead1m", "count"),
    )
    top_counts = top.groupby("eom").size().rename("top_n")
    month = (daily["date"] + pd.offsets.MonthEnd(0)).rename("eom")
    daily_counts = daily.groupby(month).agg(daily_rows=("ret", "size"), daily_permnos=("permno", "nunique"))
    counts = jkp_counts.join(top_counts, how="outer").join(daily_counts, how="outer")
    return counts.fillna(0).astype("int64").rename_axis("eom").reset_index()


def log_row_counts(counts: pd.DataFrame) -> None:
    for row in counts.itertuples(index=False):
        log.info(
            "%s  jkp=%5d  me=%5d  ret_lead1m=%5d  top_n=%3d  daily_rows=%6d  daily_permnos=%4d",
            row.eom.strftime("%Y-%m"), row.jkp_rows, row.me_nonnull, row.ret_lead1m_nonnull,
            row.top_n, row.daily_rows, row.daily_permnos,
        )
    for col in ("jkp_rows", "top_n", "daily_permnos"):
        s = counts.loc[counts[col] > 0, col]
        log.info("%-14s per month: min %d, median %d, max %d", col, s.min(), int(s.median()), s.max())


def _cached(path: Path, build: Callable[[], pd.DataFrame], force: bool) -> pd.DataFrame:
    if path.exists() and not force:
        log.info("Using existing %s (pass --force to re-pull)", path.name)
        return pd.read_parquet(path)
    df = build()
    df.to_parquet(path, index=False)
    log.info("Wrote %s: %d rows x %d columns", path.name, len(df), df.shape[1])
    return df


# --------------------------------------------------------------------------- orchestration


def pull_all(db: wrds.Connection, cfg: Config, force: bool = False) -> dict[str, Path]:
    """Pull every raw input into data/raw/. Existing files are reused unless ``force``."""
    raw = cfg.paths.raw
    raw.mkdir(parents=True, exist_ok=True)
    n = cfg.covariance.factor_universe_top_n
    paths = {key: raw / name for key, name in RAW_FILES.items()}
    paths["top"] = raw / f"top{n}_by_month.parquet"
    paths["csv"] = raw / f"jkp_usa_top{n}.csv"

    jkp = _cached(paths["jkp"], lambda: pull_jkp(db, cfg), force)
    top = _cached(paths["top"], lambda: top_n_by_month(jkp, n), force)

    # Deliverable: characteristics for the top-N stocks each month (the full extract stays in parquet).
    jkp.merge(top[["eom", "id", "me_rank"]], on=["eom", "id"]).to_csv(paths["csv"], index=False)
    log.info("Wrote %s", paths["csv"].name)

    daily = _cached(paths["daily"], lambda: pull_crsp_daily(db, cfg, top["id"].unique()), force)
    _cached(paths["ff_monthly"], lambda: read_french_zip(_http_get(cfg.data_sources.ff_factors_url)), force)
    _cached(paths["ff_daily"], lambda: read_french_zip(_http_get(cfg.data_sources.ff_factors_daily_url)), force)

    details = download_reference_file(cfg.data_sources.factor_details_url, raw)
    clusters = download_reference_file(cfg.data_sources.cluster_labels_url, raw)
    _cached(paths["factor_details"], lambda: _stringify_objects(pd.read_excel(details)), force)
    _cached(paths["cluster_labels"], lambda: pd.read_csv(clusters), force)

    counts = row_counts_by_month(jkp, top, daily)
    counts.to_csv(paths["counts"], index=False)
    log_row_counts(counts)
    return paths


### `scripts/jkmp/evaluation/__init__.py`

Performance evaluation: Table 2 statistics and related tests.

**Role:** Calculation library. Other stages import its functions; this cell writes the module.

**Input:** the embedded source below. **Output:** `scripts/jkmp/evaluation/__init__.py`. Expand the code cell to read or edit it.


In [ ]:
%%writefile scripts/jkmp/evaluation/__init__.py
"""Performance evaluation: Table 2 statistics and related tests."""


### `scripts/jkmp/evaluation/active.py`

Active theme exposures over time for benchmark-relative Portfolio-ML (extension, D34).

**Role:** Calculation library. Other stages import its functions; this cell writes the module.

**Input:** the embedded source below. **Output:** `scripts/jkmp/evaluation/active.py`. Expand the code cell to read or edit it.


In [ ]:
%%writefile scripts/jkmp/evaluation/active.py
"""Active theme exposures over time for benchmark-relative Portfolio-ML (extension, D34)."""

from __future__ import annotations

from pathlib import Path

import matplotlib

matplotlib.use("Agg")
import matplotlib.pyplot as plt  # noqa: E402
import pandas as pd  # noqa: E402

from jkmp.features import THEME_PREFIX  # noqa: E402


def plot_active_exposures(exposures: pd.DataFrame, path: Path, window: int = 12) -> list[Path]:
    """One panel per theme: the active exposure Xᵀa (theme loadings are standardised scores) of each method,
    as a ``window``-month rolling mean. ``exposures``: eom, type, theme_* columns (``ex_ante_exposures``)."""
    themes = [c for c in exposures.columns if c.startswith(THEME_PREFIX)]
    fig, axes = plt.subplots(2, (len(themes) + 1) // 2, figsize=(12, 5.6), sharex=True)
    for ax, theme in zip(axes.flat, themes):
        for t, g in exposures.groupby("type", sort=False):
            s = g.set_index("eom")[theme].sort_index().rolling(window).mean()
            ax.plot(s.index, s, lw=1.0, label=t)
        ax.axhline(0, color="grey", lw=0.5)
        ax.set_title(theme.removeprefix(THEME_PREFIX).replace("_", " "), fontsize=9)
        ax.tick_params(labelsize=8)
    for ax in list(axes.flat)[len(themes):]:
        ax.set_visible(False)
    axes.flat[0].legend(fontsize=6)
    fig.supylabel(f"Active theme exposure Xᵀa ({window}-month rolling mean)", fontsize=8)
    fig.tight_layout()
    path.parent.mkdir(parents=True, exist_ok=True)
    out = [path.with_suffix(".png"), path.with_suffix(".pdf")]
    fig.savefig(out[0], dpi=200, bbox_inches="tight")
    fig.savefig(out[1], bbox_inches="tight")
    plt.close(fig)
    return out


### `scripts/jkmp/evaluation/cost_sensitivity.py`

Trading-cost sensitivity by re-costing fixed weights (extension, research log trial 3). No re-training: the weights of a run are kept and only Λ changes, so tc = (w/2)·Σ λ_i s_i (Δw_i)² with a scale s per stock-month. * VIX: s_t = VIX_t / mean VIX (``regimes.vix_cost_scale``), the same for every stock in month t. * Stock volatility: s_i,t = σ_i,t / σ̄_i,t, σ_i,t = √Σ_ii at t and σ̄_i,t the mean of σ_i over the stock's last ``window`` month-ends up to and including t (past data only). Stocks without Σ at t (exits) keep s = 1. ``capacity_aum``: t

**Role:** Calculation library. Other stages import its functions; this cell writes the module.

**Input:** the embedded source below. **Output:** `scripts/jkmp/evaluation/cost_sensitivity.py`. Expand the code cell to read or edit it.


In [ ]:
%%writefile scripts/jkmp/evaluation/cost_sensitivity.py
"""Trading-cost sensitivity by re-costing fixed weights (extension, research log trial 3). No re-training: the
weights of a run are kept and only Λ changes, so tc = (w/2)·Σ λ_i s_i (Δw_i)² with a scale s per stock-month.

* VIX: s_t = VIX_t / mean VIX (``regimes.vix_cost_scale``), the same for every stock in month t.
* Stock volatility: s_i,t = σ_i,t / σ̄_i,t, σ_i,t = √Σ_ii at t and σ̄_i,t the mean of σ_i over the stock's last
  ``window`` month-ends up to and including t (past data only). Stocks without Σ at t (exits) keep s = 1.

``capacity_aum``: the wealth at which the net information ratio first falls below a threshold, by linear
interpolation in log wealth over the wealth grid.
"""

from __future__ import annotations

from collections.abc import Mapping, Sequence

import numpy as np
import pandas as pd

from jkmp.covariance import CovParts
from jkmp.portfolio.accounting import portfolio_returns


def vol_cost_scale(cov_parts: Mapping[pd.Timestamp, CovParts], window: int) -> pd.Series:
    """s_i,t = σ_i,t / σ̄_i,t indexed by (id, eom); σ̄ over the last ``window`` month-ends ≤ t with a Σ for the stock."""
    vol = pd.concat([pd.DataFrame({"id": c.ids, "eom": eom, "vol": c.vol()}) for eom, c in sorted(cov_parts.items())])
    vol = vol.sort_values(["id", "eom"])
    vbar = vol.groupby("id")["vol"].transform(lambda s: s.rolling(window, min_periods=1).mean())
    return pd.Series((vol["vol"] / vbar).to_numpy(), index=pd.MultiIndex.from_frame(vol[["id", "eom"]]),
                     name="cost_scale")


def recost(weights: pd.DataFrame, panel: pd.DataFrame, wealth: pd.DataFrame, scale: pd.Series) -> pd.DataFrame:
    """``portfolio_returns`` with Λ multiplied by ``scale``: indexed by eom (one value a month) or by (id, eom).
    Stock-months without a scale keep Λ."""
    p = panel[["id", "eom", "ret_ld1", "lambda"]]
    if isinstance(scale.index, pd.MultiIndex):
        s = pd.Series(scale.reindex(pd.MultiIndex.from_frame(p[["id", "eom"]])).to_numpy(), index=p.index)
    else:
        s = p["eom"].map(scale)
    return portfolio_returns(weights, p.assign(**{"lambda": p["lambda"] * s.fillna(1.0)}), wealth)


def capacity_aum(wealth: Sequence[float], ir: Sequence[float], threshold: float) -> float:
    """First wealth where IR crosses below ``threshold`` (log-linear interpolation between grid points).
    NaN-free edge cases: −inf if already below at the smallest wealth, +inf if never below on the grid."""
    w, r = np.asarray(wealth, dtype=float), np.asarray(ir, dtype=float)
    order = np.argsort(w)
    w, r = w[order], r[order]
    if r[0] < threshold:
        return float("-inf")
    for k in range(1, len(w)):
        if r[k] < threshold:
            f = (r[k - 1] - threshold) / (r[k - 1] - r[k])
            return float(np.exp(np.log(w[k - 1]) + f * (np.log(w[k]) - np.log(w[k - 1]))))
    return float("inf")


### `scripts/jkmp/evaluation/holdout.py`

Holdout lock: no performance number may use a return month after ``dates.test_end`` while ``dates.holdout_locked`` is true (holdout ``dates.holdout_start`` .. ``dates.holdout_end``). Every evaluation path calls ``check_return_months`` on the rows it summarises: Table 2 (``metrics.table2``, so also ``grids.summarise``), the grid windows (``grids.grid_inputs``), the scaled frontier, feature importance, the regime statistics and the cumulative-utility plot, and the return-model diagnostics. The frontier and Fig. 5 plots only draw frames those func

**Role:** Calculation library. Other stages import its functions; this cell writes the module.

**Input:** the embedded source below. **Output:** `scripts/jkmp/evaluation/holdout.py`. Expand the code cell to read or edit it.


In [ ]:
%%writefile scripts/jkmp/evaluation/holdout.py
"""Holdout lock: no performance number may use a return month after ``dates.test_end`` while
``dates.holdout_locked`` is true (holdout ``dates.holdout_start`` .. ``dates.holdout_end``).

Every evaluation path calls ``check_return_months`` on the rows it summarises: Table 2
(``metrics.table2``, so also ``grids.summarise``), the grid windows (``grids.grid_inputs``), the scaled
frontier, feature importance, the regime statistics and the cumulative-utility plot, and the return-model
diagnostics. The frontier and Fig. 5 plots only draw frames those functions produced. Model fitting is not
checked: it may use data up to the end of each walk-forward year.
"""

from __future__ import annotations

from collections.abc import Sequence

import pandas as pd

from jkmp.config import DatesConfig

MONTH = pd.offsets.MonthEnd(1)


class HoldoutLockedError(RuntimeError):
    """A performance computation reached a return month inside the locked holdout."""


def last_open_month(dates: DatesConfig) -> pd.Timestamp:
    """The last return month any performance number may use: test_end while locked, else holdout_end."""
    return pd.Timestamp(dates.test_end if dates.holdout_locked else dates.holdout_end)


def check_return_months(rows: pd.DataFrame | Sequence[pd.Timestamp], dates: DatesConfig, what: str) -> None:
    """Raise ``HoldoutLockedError`` if any return month is after ``last_open_month(dates)``.

    ``rows`` is a frame with ``eom_ret`` (the return month) or ``eom`` (the formation month, return month
    eom + 1), or a sequence of return months. ``what`` names the computation in the error message.
    """
    if isinstance(rows, pd.DataFrame):
        if "eom_ret" in rows:
            months = pd.DatetimeIndex(rows["eom_ret"])
        elif "eom" in rows:
            months = pd.DatetimeIndex(rows["eom"]) + MONTH
        else:
            raise ValueError(f"{what}: rows need an eom_ret or eom column to check the holdout lock")
    else:
        months = pd.DatetimeIndex(rows)
    last = last_open_month(dates)
    if len(months) and months.max() > last:
        raise HoldoutLockedError(
            f"{what}: return months up to {months.max().date()} but the holdout is locked after {last.date()} "
            f"(dates.holdout_locked = true)")


def evaluation_window(dates: DatesConfig) -> tuple[pd.Timestamp, pd.Timestamp]:
    """First and last return month evaluated by scripts 04–07 and 12: the test window, or — only with
    ``dates.evaluate: holdout`` and the lock off — the holdout (the walk-forward continues through it: models,
    β and hyperparameters keep updating each year on everything up to the year before)."""
    if dates.evaluate == "test":
        return pd.Timestamp(dates.test_start), pd.Timestamp(dates.test_end)
    if dates.holdout_locked:
        raise HoldoutLockedError("dates.evaluate: holdout while dates.holdout_locked is true")
    return pd.Timestamp(dates.holdout_start), pd.Timestamp(dates.holdout_end)


### `scripts/jkmp/evaluation/metrics.py`

Table 2 statistics, Table 3 (probability of outperformance) and Table 4 (correlations). Ports of ``pf_summary``, ``utility_t``, ``prob_outperformance`` and the return correlations in reference/jkmp_r/6 - Base analysis.R. Active (benchmark-relative) statistics against the value-weighted universe are our extension (D34). Input: monthly performance rows (``jkmp.portfolio.accounting.portfolio_returns``) with columns ``type, r, tc, turnover, inv, shorting``. As in R, volatility, the net Sharpe ratio and utility all use the variance of the **gross** 

**Role:** Calculation library. Other stages import its functions; this cell writes the module.

**Input:** the embedded source below. **Output:** `scripts/jkmp/evaluation/metrics.py`. Expand the code cell to read or edit it.


In [ ]:
%%writefile scripts/jkmp/evaluation/metrics.py
"""Table 2 statistics, Table 3 (probability of outperformance) and Table 4 (correlations). Ports of ``pf_summary``,
``utility_t``, ``prob_outperformance`` and the return correlations in reference/jkmp_r/6 - Base analysis.R.
Active (benchmark-relative) statistics against the value-weighted universe are our extension (D34).

Input: monthly performance rows (``jkmp.portfolio.accounting.portfolio_returns``) with columns
``type, r, tc, turnover, inv, shorting``. As in R, volatility, the net Sharpe ratio and utility all use
the variance of the **gross** return r (not r − tc); sample moments use n − 1 like R's ``sd``/``var``.
"""

from __future__ import annotations

from collections.abc import Mapping, Sequence

import numpy as np
import pandas as pd
from scipy.stats import norm

from jkmp.config import DatesConfig
from jkmp.covariance import CovParts
from jkmp.evaluation.holdout import check_return_months

TABLE2_COLUMNS = ["n", "R", "Vol", "SR_gross", "TC", "R_TC", "SR_net", "Utility", "Turnover", "Lev", "Shorting"]


def utility_flow(pf: pd.DataFrame, gamma: float) -> pd.Series:
    """Monthly realised utility r − tc − ½γ(r − r̄)², r̄ the mean gross return of each ``type``."""
    rbar = pf.groupby("type")["r"].transform("mean")
    return pf["r"] - pf["tc"] - 0.5 * gamma * (pf["r"] - rbar) ** 2


def summary_stats(pf: pd.DataFrame, gamma: float) -> pd.Series:
    """One method's Table 2 row (annualised except Turnover, Lev and Shorting)."""
    r, tc = pf["r"], pf["tc"]
    sd = r.std(ddof=1)
    return pd.Series({
        "n": len(pf),
        "R": r.mean() * 12,
        "Vol": sd * np.sqrt(12),
        "SR_gross": r.mean() / sd * np.sqrt(12),
        "TC": tc.mean() * 12,
        "R_TC": (r - tc).mean() * 12,
        "SR_net": (r - tc).mean() / sd * np.sqrt(12),
        "Utility": (r.mean() - 0.5 * r.var(ddof=1) * gamma - tc.mean()) * 12,
        "Turnover": pf["turnover"].mean(),
        "Lev": pf["inv"].mean(),
        "Shorting": pf["shorting"].mean(),
    })


def table2(pfs: pd.DataFrame, gamma: float, dates: DatesConfig) -> pd.DataFrame:
    """Table 2 rows, one per ``type`` (in order of first appearance); refuses locked holdout months."""
    check_return_months(pfs, dates, "table2")
    rows = {t: summary_stats(g, gamma) for t, g in pfs.groupby("type", sort=False)}
    out = pd.DataFrame(rows).T[TABLE2_COLUMNS]
    out["n"] = out["n"].astype(int)
    return out.rename_axis("type")


def _wide(pfs: pd.DataFrame, value: pd.Series, types: Sequence[str]) -> pd.DataFrame:
    """Return month × type; every type must cover the same months."""
    wide = pfs.assign(value=value).pivot(index="eom_ret", columns="type", values="value")[list(types)]
    if wide.isna().any().any():
        raise ValueError("methods cover different return months")
    return wide


def prob_outperformance(pfs: pd.DataFrame, gamma: float, dates: DatesConfig, types: Sequence[str]) -> pd.DataFrame:
    """Table 3: posterior probability that the row method's mean utility flow beats the column method's.
    R ``prob_outperformance``: d_t = u_row,t − u_col,t, P = 1 − Φ(0; mean d, sd(d)/√T), with u_t = ``utility_flow``."""
    check_return_months(pfs, dates, "prob_outperformance")
    u = _wide(pfs, utility_flow(pfs, gamma), types)
    out = pd.DataFrame(np.nan, index=list(types), columns=list(types))
    for a in types:
        for b in types:
            if a != b:
                d = u[a] - u[b]
                out.loc[a, b] = 1.0 - norm.cdf(0.0, loc=d.mean(), scale=d.std(ddof=1) / np.sqrt(len(d)))
    return out.rename_axis("type")


def return_correlations(pfs: pd.DataFrame, dates: DatesConfig, types: Sequence[str]) -> pd.DataFrame:
    """Table 4: Pearson correlation of the monthly gross returns r (R: ``cor(method = "pearson")``)."""
    check_return_months(pfs, dates, "return_correlations")
    return _wide(pfs, pfs["r"], types).corr(method="pearson").rename_axis("type")


def subperiod_table(pfs: pd.DataFrame, gamma: float, dates: DatesConfig,
                    periods: Mapping[str, tuple[str, str]], columns: Sequence[str]) -> pd.DataFrame:
    """Table 2 ``columns`` within each (label: (first, last) return month) period; type × (column, period)."""
    parts = {label: table2(pfs[pfs["eom_ret"].between(pd.Timestamp(a), pd.Timestamp(b))], gamma, dates)[list(columns)]
             for label, (a, b) in periods.items()}
    out = pd.concat(parts, axis=1).swaplevel(axis=1)
    return out[[(c, label) for c in columns for label in periods]]


# --- Benchmark-relative statistics (extension, D34) ---------------------------------------------------

def active_returns(pf: pd.DataFrame, bench: pd.DataFrame, dates: DatesConfig) -> pd.DataFrame:
    """Monthly active return vs the benchmark: ``active`` = r − r_b (gross), ``active_net`` = (r − tc) − (r_b − tc_b),
    plus ``r`` and ``r_b``. ``pf`` and ``bench`` are single-method performance rows over the same months."""
    check_return_months(pf, dates, "active_returns")
    d = pf[["eom", "eom_ret", "r", "tc"]].merge(bench[["eom_ret", "r", "tc"]], on="eom_ret", suffixes=("", "_b"),
                                                validate="one_to_one")
    if len(d) != len(pf) or len(d) != len(bench):
        raise ValueError("strategy and benchmark cover different return months")
    return d.assign(active=d["r"] - d["r_b"], active_net=(d["r"] - d["tc"]) - (d["r_b"] - d["tc_b"]))


def active_utility_flow(active: pd.DataFrame, gamma_te: float) -> pd.Series:
    """Monthly active utility: active net return − (γ_TE/2)(active − mean active)² (gross active variance, as R4)."""
    a = active["active"]
    return active["active_net"] - 0.5 * gamma_te * (a - a.mean()) ** 2


def prob_greater(x: pd.Series, y: pd.Series | float = 0.0) -> float:
    """Posterior P(mean of x − y > 0) under R's ``prob_outperformance`` rule: 1 − Φ(0; mean d, sd(d)/√T)."""
    d = np.asarray(x - y, dtype=float)
    return float(1.0 - norm.cdf(0.0, loc=d.mean(), scale=d.std(ddof=1) / np.sqrt(len(d))))


def active_weights(weights: pd.DataFrame, panel: pd.DataFrame) -> pd.DataFrame:
    """a = w − π^VW per universe stock and formation month (exit rows dropped): ``id, eom, w, vw, a``."""
    w = weights.loc[~weights["exit"], ["id", "eom", "w"]].merge(panel[["id", "eom", "me"]], on=["id", "eom"], how="left")
    if w["me"].isna().any():
        raise ValueError("held stocks without market cap")
    vw = w["me"] / w.groupby("eom")["me"].transform("sum")
    return w.assign(vw=vw, a=w["w"] - vw)[["id", "eom", "w", "vw", "a"]]


def ex_ante_exposures(active: pd.DataFrame, cov_parts: Mapping[pd.Timestamp, CovParts]) -> pd.DataFrame:
    """Per formation month: ex-ante active volatility √(aᵀΣa) (monthly; ``te``) and the factor exposures Xᵀa
    (one column per covariance factor: sectors and themes)."""
    rows = []
    for eom, g in active.groupby("eom", sort=True):
        c = cov_parts[eom]
        a = pd.Series(g["a"].to_numpy(), index=g["id"].to_numpy()).reindex(c.ids, fill_value=0.0)
        if not set(g["id"]) <= set(c.ids):
            raise KeyError(f"active positions without covariance at {eom.date()}")
        a = a.to_numpy()
        with np.errstate(all="ignore"):  # spurious Accelerate matmul flags; checked below
            b = c.X.T @ a
            var = float(b @ c.F @ b + (c.ivar * a**2).sum())
        if not (np.isfinite(var) and np.isfinite(b).all()):
            raise FloatingPointError(f"non-finite active risk at {eom.date()}")
        rows.append({"eom": eom, "te": np.sqrt(var), **dict(zip(c.factors, b))})
    return pd.DataFrame(rows)


def active_stats(pf: pd.DataFrame, bench: pd.DataFrame, weights: pd.DataFrame, panel: pd.DataFrame,
                 cov_parts: Mapping[pd.Timestamp, CovParts], gamma_te: float, dates: DatesConfig) -> pd.Series:
    """Annualised active statistics of one method against the benchmark (the VW universe, ``bench``)."""
    act = active_returns(pf, bench, dates)
    aw = active_weights(weights[weights["eom"].isin(pf["eom"])], panel)
    ex = ex_ante_exposures(aw, cov_parts)
    a, an, sd = act["active"], act["active_net"], act["active"].std(ddof=1)
    rb = act["r_b"]
    return pd.Series({
        "n": len(act),
        "active": a.mean() * 12, "active_net": an.mean() * 12,
        "te_realised": sd * np.sqrt(12), "te_ex_ante": ex["te"].mean() * np.sqrt(12),
        "ir_gross": a.mean() / sd * np.sqrt(12), "ir_net": an.mean() / sd * np.sqrt(12),
        "p_ir_net_pos": prob_greater(an),
        "beta": float(np.cov(act["r"], rb, ddof=1)[0, 1] / rb.var(ddof=1)),
        "neg_weight_share": float((aw["w"] < 0).groupby(aw["eom"]).mean().mean()),
        "active_utility": active_utility_flow(act, gamma_te).mean() * 12,
        "tc": act["tc"].mean() * 12, "tc_bench": act["tc_b"].mean() * 12,
    })


def deflated_sharpe(returns: pd.Series, trial_sharpes: Sequence[float], n_trials: int) -> dict:
    """Deflated Sharpe ratio (Bailey and López de Prado, 2014) of per-period ``returns`` after ``n_trials``
    trials whose per-period Sharpe ratios are ``trial_sharpes`` (their variance sets the expected maximum):
    SR₀ = √V[SR]·((1 − γₑ)Φ⁻¹(1 − 1/N) + γₑΦ⁻¹(1 − 1/(N e))),
    DSR = Φ((SR − SR₀)√(T − 1) / √(1 − γ₃SR + (γ₄ − 1)/4·SR²)), γ₃ skewness, γ₄ (non-excess) kurtosis."""
    r = np.asarray(returns, dtype=float)
    t, sr = len(r), r.mean() / r.std(ddof=1)
    z = (r - r.mean()) / r.std(ddof=0)
    skew, kurt = float((z**3).mean()), float((z**4).mean())
    euler = 0.5772156649015329
    v = float(np.var(np.asarray(trial_sharpes, dtype=float), ddof=1)) if len(trial_sharpes) > 1 else 0.0
    sr0 = (np.sqrt(v) * ((1 - euler) * norm.ppf(1 - 1 / n_trials) + euler * norm.ppf(1 - 1 / (n_trials * np.e)))
           if n_trials > 1 else 0.0)
    dsr = norm.cdf((sr - sr0) * np.sqrt(t - 1) / np.sqrt(1 - skew * sr + (kurt - 1) / 4 * sr**2))
    return {"T": t, "sr": float(sr), "sr_annual": float(sr * np.sqrt(12)), "sr0": float(sr0),
            "sr0_annual": float(sr0 * np.sqrt(12)), "skew": skew, "kurtosis": kurt, "n_trials": n_trials,
            "var_trial_sr": v, "dsr": float(dsr)}


### `scripts/jkmp/evaluation/monitor.py`

Monthly monitoring of the live active strategy (extension; scripts/11_monitor.py). Every check uses only data up to the month it flags. One row per return month with the statistic and its flag: * ``ir``: rolling ``ir_window``-month net information ratio < 0. * ``drawdown``: drawdown of the cumulative net active return > ``drawdown_te_multiple`` × the TE budget. * ``beta``: rolling ``ir_window``-month beta to the benchmark outside ``beta_limits``. * ``theme``: largest |active theme exposure Xᵀa| at formation > ``theme_limit``. * ``ic``: rolling 

**Role:** Calculation library. Other stages import its functions; this cell writes the module.

**Input:** the embedded source below. **Output:** `scripts/jkmp/evaluation/monitor.py`. Expand the code cell to read or edit it.


In [ ]:
%%writefile scripts/jkmp/evaluation/monitor.py
"""Monthly monitoring of the live active strategy (extension; scripts/11_monitor.py). Every check uses only data
up to the month it flags. One row per return month with the statistic and its flag:

* ``ir``: rolling ``ir_window``-month net information ratio < 0.
* ``drawdown``: drawdown of the cumulative net active return > ``drawdown_te_multiple`` × the TE budget.
* ``beta``: rolling ``ir_window``-month beta to the benchmark outside ``beta_limits``.
* ``theme``: largest |active theme exposure Xᵀa| at formation > ``theme_limit``.
* ``ic``: rolling ``ic_window``-month mean rank IC < 0 for ``ic_negative_months`` consecutive months.
* ``tc``: realised trading cost (stock-vol-scaled Λ) above the model's prediction for ``tc_months`` months in a row.
* ``half_life``: the December pick of the half-life moves more than one grid step from the previous year's.
* ``bias``: bias statistic sd(active_t / ex-ante TE_t) over ``bias_window`` months outside 1 ± √(2/T).
"""

from __future__ import annotations

from collections.abc import Sequence
from pathlib import Path

import matplotlib

matplotlib.use("Agg")
import matplotlib.pyplot as plt  # noqa: E402
import numpy as np  # noqa: E402
import pandas as pd  # noqa: E402

from jkmp.config import DatesConfig, MonitorConfig  # noqa: E402
from jkmp.evaluation.holdout import check_return_months  # noqa: E402
from jkmp.features import THEME_PREFIX  # noqa: E402

MONTH = pd.offsets.MonthEnd(1)
CHECKS = ["ir", "drawdown", "beta", "theme", "ic", "tc", "half_life", "bias"]


def _run_length(flag: pd.Series) -> pd.Series:
    """Consecutive True count ending at each row."""
    grp = (~flag).cumsum()
    return flag.astype(int).groupby(grp).cumsum()


def half_life_steps(chosen: pd.DataFrame, grid: Sequence[float | None]) -> pd.Series:
    """Per return year: grid steps the chosen half-life moved from the previous year (None/inf = last step)."""
    order = [np.inf if h is None else float(h) for h in grid]
    pos = chosen.set_index("year")["h"].map(lambda h: order.index(np.inf if pd.isna(h) else float(h))).sort_index()
    return pos.diff().abs()


def monitor_table(active: pd.DataFrame, exposures: pd.DataFrame, rank_ic: pd.Series, tc_realised: pd.Series,
                  hl_steps: pd.Series | None, cfg: MonitorConfig, dates: DatesConfig) -> pd.DataFrame:
    """``active``: eom, eom_ret, r, r_b, tc, active, active_net (``metrics.active_returns``); ``exposures``: eom,
    te, theme_* (``metrics.ex_ante_exposures``); ``rank_ic``: monthly IC by return month; ``tc_realised``: re-costed
    tc by return month; ``hl_steps``: ``half_life_steps`` by return year (None if h is not tuned)."""
    check_return_months(active, dates, "monitor")
    d = active.sort_values("eom_ret").set_index("eom_ret")
    ex = exposures.assign(eom_ret=exposures["eom"] + MONTH).set_index("eom_ret").reindex(d.index)
    themes = [c for c in exposures.columns if c.startswith(THEME_PREFIX)]
    w, sq12 = cfg.ir_window, np.sqrt(12)
    out = pd.DataFrame(index=d.index)
    out["ir"] = d["active_net"].rolling(w).mean() / d["active"].rolling(w).std() * sq12
    cum = d["active_net"].cumsum()
    out["drawdown"] = cum.cummax().clip(lower=0) - cum
    out["beta"] = d["r"].rolling(w).cov(d["r_b"]) / d["r_b"].rolling(w).var()
    out["theme"] = ex[themes].abs().max(axis=1)
    out["ic"] = rank_ic.reindex(d.index).rolling(cfg.ic_window).mean()
    out["tc"] = tc_realised.reindex(d.index) - d["tc"]
    out["half_life"] = (d.index.year.map(hl_steps) if hl_steps is not None else np.nan)
    z = d["active"] / ex["te"]
    out["bias"] = z.rolling(cfg.bias_window).std()
    band = np.sqrt(2 / cfg.bias_window)
    flags = pd.DataFrame({
        "ir": out["ir"] < 0,
        "drawdown": out["drawdown"] > cfg.drawdown_te_multiple * cfg.te_budget,
        "beta": (out["beta"] < cfg.beta_limits[0]) | (out["beta"] > cfg.beta_limits[1]),
        "theme": out["theme"] > cfg.theme_limit,
        "ic": _run_length(out["ic"] < 0) >= cfg.ic_negative_months,
        "tc": _run_length(out["tc"] > 0) >= cfg.tc_months,
        "half_life": pd.Series(out["half_life"], index=out.index) > 1,
        "bias": (out["bias"] < 1 - band) | (out["bias"] > 1 + band),
    }, index=out.index)
    return out.join(flags.add_prefix("flag_")).reset_index()


def plot_dashboard(table: pd.DataFrame, cfg: MonitorConfig, title: str, path: Path) -> list[Path]:
    """One page: the eight statistics with their limits and flagged months shaded; plus an HTML version with the
    flag summary table. Returns the PNG and HTML paths."""
    band = np.sqrt(2 / cfg.bias_window)
    limits = {"ir": [0.0], "drawdown": [cfg.drawdown_te_multiple * cfg.te_budget], "beta": list(cfg.beta_limits),
              "theme": [cfg.theme_limit], "ic": [0.0], "tc": [0.0], "half_life": [1.0], "bias": [1 - band, 1 + band]}
    names = {"ir": f"Rolling {cfg.ir_window}m net IR", "drawdown": "Drawdown, cum. net active",
             "beta": f"Rolling {cfg.ir_window}m beta", "theme": "Max |theme exposure|",
             "ic": f"Rolling {cfg.ic_window}m rank IC", "tc": "Realised − predicted TC",
             "half_life": "Half-life jump (grid steps)", "bias": f"TE bias stat ({cfg.bias_window}m)"}
    fig, axes = plt.subplots(4, 2, figsize=(11, 10), sharex=True)
    x = pd.to_datetime(table["eom_ret"])
    for ax, c in zip(axes.flat, CHECKS):
        ax.plot(x, table[c], lw=1.0, color="C0")
        for lim in limits[c]:
            ax.axhline(lim, color="C3", lw=0.7, ls="--")
        flagged = table[f"flag_{c}"].fillna(False).to_numpy()
        ax.fill_between(x, 0, 1, where=flagged, transform=ax.get_xaxis_transform(), color="C3", alpha=0.15, lw=0)
        ax.set_title(f"{names[c]} — flagged {int(flagged.sum())} of {len(flagged)} months", fontsize=9)
        ax.tick_params(labelsize=7)
    fig.suptitle(title, fontsize=11)
    fig.tight_layout()
    path.parent.mkdir(parents=True, exist_ok=True)
    png, html = path.with_suffix(".png"), path.with_suffix(".html")
    fig.savefig(png, dpi=160, bbox_inches="tight")
    plt.close(fig)
    summary = pd.DataFrame({"check": [names[c] for c in CHECKS],
                            "flagged months": [int(table[f"flag_{c}"].fillna(False).sum()) for c in CHECKS],
                            "flagged in last month": [bool(table[f"flag_{c}"].fillna(False).iloc[-1]) for c in CHECKS]})
    html.write_text(f"<!doctype html><html><head><meta charset='utf-8'><title>Strategy monitor</title>"
                    f"<style>body{{font-family:sans-serif;max-width:1100px;margin:auto;padding:16px}}"
                    f"table{{border-collapse:collapse}}td,th{{border:1px solid #ccc;padding:4px 8px}}"
                    f"img{{max-width:100%}}</style></head><body><h1>{title}</h1>{summary.to_html(index=False)}"
                    f"<p><img src='{png.name}' alt='monitor charts'></p></body></html>")
    return [png, html]


### `scripts/jkmp/evaluation/regimes.py`

Market regimes over the test window (extension; not in the paper or the R code). FRED series (public CSV, cached in data/raw/fred_{series}.parquet): USREC NBER recession indicator, monthly (1 = recession) VIXCLS CBOE VIX daily close → monthly mean FEDFUNDS effective federal funds rate, monthly average (%) Labels per month-end: ``recession``, ``vix_regime`` (low / mid / high tercile of monthly VIX over the evaluation window) and ``fed_regime`` (hiking / cutting / on hold from the change in FEDFUNDS over the last ``fed_change_months`` months, wit

**Role:** Calculation library. Other stages import its functions; this cell writes the module.

**Input:** the embedded source below. **Output:** `scripts/jkmp/evaluation/regimes.py`. Expand the code cell to read or edit it.


In [ ]:
%%writefile scripts/jkmp/evaluation/regimes.py
"""Market regimes over the test window (extension; not in the paper or the R code).

FRED series (public CSV, cached in data/raw/fred_{series}.parquet):
  USREC    NBER recession indicator, monthly (1 = recession)
  VIXCLS   CBOE VIX daily close → monthly mean
  FEDFUNDS effective federal funds rate, monthly average (%)
Labels per month-end: ``recession``, ``vix_regime`` (low / mid / high tercile of monthly VIX over the
evaluation window) and ``fed_regime`` (hiking / cutting / on hold from the change in FEDFUNDS over the
last ``fed_change_months`` months, with a ±``fed_threshold`` pp band).

Performance rows are labelled by the month the return is realised (``eom_ret``). The VIX re-costing
scales each month's trading cost by VIX_t / mean VIX at the formation month t, when the trade happens:
the weights are unchanged and tc = (w/2) ΔπᵀΛΔπ is linear in Λ, so Λ_t → s_t Λ gives tc_t → s_t tc_t.
"""

from __future__ import annotations

import io
import logging
import urllib.request
from collections.abc import Sequence
from pathlib import Path

import matplotlib

matplotlib.use("Agg")
import matplotlib.pyplot as plt  # noqa: E402
import numpy as np  # noqa: E402
import pandas as pd  # noqa: E402

from jkmp.config import DatesConfig  # noqa: E402
from jkmp.evaluation.holdout import check_return_months  # noqa: E402
from jkmp.evaluation.metrics import summary_stats, utility_flow  # noqa: E402

log = logging.getLogger(__name__)

MONTH = pd.offsets.MonthEnd(1)
REGIME_COLUMNS = ("vix_regime", "fed_regime", "recession")


# --- FRED -------------------------------------------------------------------------------------------

def fetch_fred(url_template: str, series: str, timeout: int = 60) -> pd.DataFrame:
    """One FRED series as ``date, value`` (missing observations dropped)."""
    raise FileNotFoundError(f"Missing local FRED cache for {series}; downloads are disabled.")
    url = url_template.format(series=series)
    log.info("Downloading %s", url)
    with urllib.request.urlopen(url, timeout=timeout) as resp:
        raw = pd.read_csv(io.BytesIO(resp.read()))
    raw.columns = ["date", "value"]
    out = pd.DataFrame({"date": pd.to_datetime(raw["date"]), "value": pd.to_numeric(raw["value"], errors="coerce")})
    return out.dropna().reset_index(drop=True)


def load_fred(url_template: str, series: Sequence[str], raw_dir: Path, force: bool = False) -> dict[str, pd.DataFrame]:
    """Cached FRED downloads: ``raw_dir/fred_{series}.parquet``."""
    out = {}
    for s in series:
        path = raw_dir / f"fred_{s}.parquet"
        if force or not path.exists():
            fetch_fred(url_template, s).to_parquet(path, index=False)
        out[s] = pd.read_parquet(path)
    return out


def monthly_series(fred: dict[str, pd.DataFrame]) -> pd.DataFrame:
    """``eom, usrec, vix, fedfunds`` by month-end (VIX: mean of daily closes in the month)."""
    def by_month(df: pd.DataFrame, name: str) -> pd.Series:
        return df.groupby(df["date"].dt.to_period("M"))["value"].mean().rename(name)
    parts = [by_month(fred["USREC"], "usrec"), by_month(fred["VIXCLS"], "vix"), by_month(fred["FEDFUNDS"], "fedfunds")]
    m = pd.concat(parts, axis=1)
    m.index = m.index.to_timestamp(how="end").normalize()
    return m.rename_axis("eom").reset_index()


# --- labels ------------------------------------------------------------------------------------------

def label_regimes(monthly: pd.DataFrame, months: Sequence[pd.Timestamp], fed_change_months: int,
                  fed_threshold: float) -> pd.DataFrame:
    """Regime labels for ``months``; VIX terciles are cut on those months only."""
    m = monthly.sort_values("eom").set_index("eom")
    d_ff = m["fedfunds"] - m["fedfunds"].shift(fed_change_months)
    fed = np.select([d_ff > fed_threshold, d_ff < -fed_threshold], ["hiking", "cutting"], default="on hold")
    m = m.assign(fed_change=d_ff, fed_regime=pd.Series(fed, index=m.index).where(d_ff.notna()))
    sub = m.reindex(pd.DatetimeIndex(months))
    if sub[["vix", "fedfunds", "usrec"]].isna().any().any():
        missing = sub.index[sub[["vix", "fedfunds", "usrec"]].isna().any(axis=1)]
        raise ValueError(f"FRED data missing for {len(missing)} months, e.g. {missing[0].date()}")
    sub["vix_regime"] = pd.qcut(sub["vix"], 3, labels=["low VIX", "mid VIX", "high VIX"]).astype(str)
    sub["recession"] = np.where(sub["usrec"] > 0.5, "recession", "expansion")
    return sub.rename_axis("eom").reset_index()


# --- analysis ----------------------------------------------------------------------------------------

def cumulative_utility(pfs: pd.DataFrame, gamma: float, dates: DatesConfig) -> pd.DataFrame:
    """Monthly utility flow r − tc − ½γ(r − r̄)² (Table 2's utility, month by month) and its running sum."""
    check_return_months(pfs, dates, "cumulative_utility")
    d = pfs.sort_values(["type", "eom_ret"]).reset_index(drop=True)
    d["u"] = utility_flow(d, gamma)
    d["cum_u"] = d.groupby("type", sort=False)["u"].cumsum()
    return d[["type", "eom_ret", "u", "cum_u"]]


def regime_table(pfs: pd.DataFrame, labels: pd.DataFrame, gamma: float, dates: DatesConfig) -> pd.DataFrame:
    """Table 2 statistics within each regime (rows labelled by the return month)."""
    check_return_months(pfs, dates, "regime_table")
    d = pfs.merge(labels[["eom", *REGIME_COLUMNS]].rename(columns={"eom": "eom_ret"}), on="eom_ret", how="left")
    if d[list(REGIME_COLUMNS)].isna().any().any():
        raise ValueError("some return months have no regime label")
    rows = []
    for dim in REGIME_COLUMNS:
        for (regime, t), g in d.groupby([dim, "type"], sort=False):
            rows.append(summary_stats(g, gamma).rename(None).to_frame().T.assign(dimension=dim, regime=regime, type=t))
    out = pd.concat(rows, ignore_index=True)
    first = ["dimension", "regime", "type"]
    return out[first + [c for c in out.columns if c not in first]]


def vix_cost_scale(monthly: pd.DataFrame, months: Sequence[pd.Timestamp], norm: str) -> pd.Series:
    """s_t = VIX_t / mean VIX, with the mean over ``months`` ('window') or all VIX history ('full')."""
    v = monthly.set_index("eom")["vix"]
    base = v.reindex(pd.DatetimeIndex(months)).mean() if norm == "window" else v.mean()
    return (v / base).rename("vix_scale")


def recost_vix(pfs: pd.DataFrame, scale: pd.Series, gamma: float, dates: DatesConfig) -> pd.DataFrame:
    """Each method's trades re-costed with Λ_t = s_t·Λ (s at the formation month): Table 2 rows for the
    original and re-costed costs side by side."""
    check_return_months(pfs, dates, "recost_vix")
    s = pfs["eom"].map(scale)
    if s.isna().any():
        raise ValueError("VIX scale missing for some formation months")
    re = pfs.assign(tc=pfs["tc"] * s)
    rows = []
    for t, g in pfs.groupby("type", sort=False):
        a, b = summary_stats(g, gamma), summary_stats(re[re["type"] == t], gamma)
        rows.append({"type": t, "TC": a["TC"], "TC_vix": b["TC"], "R_TC": a["R_TC"], "R_TC_vix": b["R_TC"],
                     "Utility": a["Utility"], "Utility_vix": b["Utility"], "dUtility": b["Utility"] - a["Utility"],
                     "TC_ratio": b["TC"] / a["TC"] if a["TC"] > 0 else np.nan,
                     "corr_turnover_vix": _corr(g["turnover"], g["eom"].map(scale))})
    return pd.DataFrame(rows)


def _corr(a: pd.Series, b: pd.Series) -> float:
    return float(np.corrcoef(a, b)[0, 1]) if a.std() > 0 and b.std() > 0 else float("nan")


# --- figure ------------------------------------------------------------------------------------------

def plot_cumulative_utility(cum: pd.DataFrame, episodes, types: Sequence[str], gamma: float,
                            path: Path, dates: DatesConfig) -> list[Path]:
    check_return_months(cum, dates, "plot_cumulative_utility")
    fig, ax = plt.subplots(figsize=(8, 4.6))
    for ep in episodes:
        ax.axvspan(pd.Timestamp(ep.start), pd.Timestamp(ep.end) + pd.Timedelta(days=1), color="grey", alpha=0.18, lw=0)
        ax.text(pd.Timestamp(ep.start), 1.0, f" {ep.label}", transform=ax.get_xaxis_transform(), fontsize=6.5,
                va="top", ha="left", rotation=90, color="dimgrey")
    for t in types:
        g = cum[cum["type"] == t]
        if len(g):
            start = pd.DataFrame({"eom_ret": [g["eom_ret"].min() - MONTH], "cum_u": [0.0]})
            g = pd.concat([start, g[["eom_ret", "cum_u"]]])
            ax.plot(g["eom_ret"], g["cum_u"], lw=1.4 if "ML" in t else 1.0, label=t)
    ax.axhline(0, color="grey", lw=0.5)
    ax.set(ylabel=f"Cumulative realised utility (γ = {gamma:g}, monthly flows)", title=f"Cumulative utility, {cum['eom_ret'].min():%Y}–{cum['eom_ret'].max():%Y}")
    ax.legend(fontsize=7, ncol=2, loc="lower left")
    path.parent.mkdir(parents=True, exist_ok=True)
    out = [path.with_suffix(".png"), path.with_suffix(".pdf")]
    fig.savefig(out[0], dpi=200, bbox_inches="tight")
    fig.savefig(out[1], bbox_inches="tight")
    plt.close(fig)
    return out


### `scripts/jkmp/features.py`

Characteristic standardisation and theme scores. * ``rank_features`` — cross-sectional rank in [0, 1] per month, exact zeros -> 0, missing -> 0.5 (reference/jkmp_r/1 - Prepare Data.R, feature standardisation and imputation). * ``theme_scores`` — per theme, the average rank of its characteristics after flipping those with direction -1 to ``1 - rank`` (reference/jkmp_r/3 - Estimate Covariance Matrix.R, ``cluster_ranks``). These are the theme loadings of the Barra-style covariance model; R standardises them by month within the estimation universe 

**Role:** Calculation library. Other stages import its functions; this cell writes the module.

**Input:** the embedded source below. **Output:** `scripts/jkmp/features.py`. Expand the code cell to read or edit it.


In [ ]:
%%writefile scripts/jkmp/features.py
"""Characteristic standardisation and theme scores.

* ``rank_features`` — cross-sectional rank in [0, 1] per month, exact zeros -> 0, missing -> 0.5
  (reference/jkmp_r/1 - Prepare Data.R, feature standardisation and imputation).
* ``theme_scores`` — per theme, the average rank of its characteristics after flipping those with
  direction -1 to ``1 - rank`` (reference/jkmp_r/3 - Estimate Covariance Matrix.R, ``cluster_ranks``).
  These are the theme loadings of the Barra-style covariance model; R standardises them by month
  within the estimation universe just before the factor regressions (``standardize_by_month``).
"""

from __future__ import annotations

from collections.abc import Mapping, Sequence
from pathlib import Path

import pandas as pd

from jkmp.data.wrds_pull import RAW_FILES

THEME_PREFIX = "theme_"


def rank_features(panel: pd.DataFrame, features: Sequence[str], percentile_rank: bool = True,
                  impute_value: float = 0.5) -> pd.DataFrame:
    """Rank each characteristic within each month to [0, 1]; return the standardised columns.

    ``rank(method="max", pct=True)`` equals R's ``ecdf(x)(x)`` (ties take the highest rank).
    R then sets characteristics whose raw value is exactly 0 to rank 0 ("ecdf always returns >0")
    and imputes missing values with 0.5. Without ranking, R imputes the monthly median instead.
    """
    x = panel[list(features)]
    if percentile_rank:
        ranks = x.groupby(panel["eom"]).rank(method="max", pct=True)
        return ranks.mask(x.eq(0), 0.0).fillna(impute_value)
    return x.fillna(x.groupby(panel["eom"]).transform("median"))


def normalise_theme_label(label: str) -> str:
    """JKP cluster label -> config theme name (R: lower-case, spaces and hyphens -> underscores)."""
    return label.strip().lower().replace(" ", "_").replace("-", "_")


def load_theme_directions(raw_dir: Path, themes: Mapping[str, Sequence[str]]) -> pd.DataFrame:
    """Load the frozen research groups and audited covariance direction policy.

    Published features retain published signs; provisional group members use
    their 2005–2014 training sign. These signs apply only to covariance theme
    exposures; return-model and Portfolio ML raw ranks remain unflipped.
    """
    # Kati's frozen metadata explicitly distinguishes provisional group members.
    # Keep the published reference files intact: do not mislabel proxies as JKP.
    frozen = Path(__file__).resolve().parents[2] / "data/characteristics/theme_directions.csv"
    if frozen.exists():
        spec = pd.read_csv(frozen)
        expected = {f: t for t, feats in themes.items() for f in feats}
        if spec.characteristic.duplicated().any() or set(spec.characteristic) != set(expected):
            raise ValueError("Frozen Kati theme metadata differs from configured 50 predictors")
        if any(expected[row.characteristic] != row.model_theme for row in spec.itertuples()):
            raise ValueError("Configured theme differs from the frozen research assignment")
        if not spec.model_theme_direction.isin([-1, 1]).all():
            raise ValueError("Frozen theme directions must be +1/-1")
        indexed = spec.set_index("characteristic")
        return pd.DataFrame([dict(feature=f, theme=t, direction=float(indexed.loc[f, "model_theme_direction"]),
                                 direction_source=indexed.loc[f, "model_direction_source"],
                                 published_group=bool(indexed.loc[f, "published_jkp_153"]))
                             for f, t in expected.items()])
    raise FileNotFoundError("Kati requires data/characteristics/theme_directions.csv; run scripts/prepare_inputs.py first")


def theme_scores(ranks: pd.DataFrame, directions: pd.DataFrame) -> pd.DataFrame:
    """Average signed rank per theme: columns ``theme_<name>`` (R: ``rowMeans`` of flipped ranks)."""
    out = {}
    for theme, group in directions.groupby("theme", sort=False):
        signed = [ranks[f] if d == 1 else 1.0 - ranks[f] for f, d in zip(group["feature"], group["direction"])]
        out[THEME_PREFIX + theme] = pd.concat(signed, axis=1).mean(axis=1)
    return pd.DataFrame(out, index=ranks.index)


def standardize_by_month(x: pd.DataFrame, eom: pd.Series, within: pd.Series | None = None) -> pd.DataFrame:
    """Subtract the monthly cross-sectional mean and divide by the standard deviation (R: ``sd``, n-1).

    With ``within`` (a boolean mask), the mean and sd come from those rows only but are applied to
    every row — e.g. moments from the covariance estimation universe, applied to all stocks.
    """
    base = x if within is None else x[within]
    grouped = base.groupby(eom if within is None else eom[within])
    mean, sd = grouped.mean(), grouped.std()
    return (x - mean.reindex(eom).to_numpy()) / sd.reindex(eom).to_numpy()


### `scripts/jkmp/imputation_audit.py`

Visible, stage-specific disclosure of zero-imputed next-month returns. Reporting is opt-in at script entry points so importing/loading configurations in libraries does not normally print tables or create output files.

**Role:** Calculation library. Other stages import its functions; this cell writes the module.

**Input:** the embedded source below. **Output:** `scripts/jkmp/imputation_audit.py`. Expand the code cell to read or edit it.


In [ ]:
%%writefile scripts/jkmp/imputation_audit.py
"""Visible, stage-specific disclosure of zero-imputed next-month returns.

Reporting is opt-in at script entry points so importing/loading configurations
in libraries does not normally print tables or create output files.
"""
from __future__ import annotations

from contextlib import contextmanager
from pathlib import Path
import re

import pandas as pd

ROOT = Path(__file__).resolve().parents[2]
_stage = None
_reported = set()


def reporting_enabled():
    return _stage is not None


def enable_reporting(stage):
    global _stage, _reported
    _stage = str(stage)
    _reported = set()


@contextmanager
def reporting_context(stage):
    global _stage, _reported
    previous = (_stage, _reported)
    enable_reporting(stage)
    try:
        yield
    finally:
        _stage, _reported = previous


def build_table(cfg):
    """Read the actual prepared-data audit; do not infer assumptions from NaNs."""
    source = cfg.paths.processed / "zero_return_imputations.csv"
    if not source.exists():
        return None
    data = pd.read_csv(source, parse_dates=["eom"])
    required = {"id", "eom", "ret_ld1_original", "ret_ld1", "tr_ld1"}
    if not required.issubset(data):
        raise ValueError(f"Incomplete zero-return audit: {source}")
    data["eom"] = pd.to_datetime(data.eom)
    if not data.ret_ld1.eq(0).all() or data.ret_ld1_original.notna().any():
        raise ValueError(f"Audit must identify genuinely missing source returns assumed zero: {source}")
    table = data.rename(columns={"id": "stock_id", "eom": "formation_month_end",
        "ret_ld1_original": "source_excess_return", "ret_ld1": "assumed_excess_return",
        "tr_ld1": "resulting_total_return"}).copy()
    table["stock_id"] = pd.to_numeric(table.stock_id, errors="raise").astype("int64")
    table["return_month_end"] = table.formation_month_end + pd.offsets.MonthEnd(1)
    universe = cfg.paths.raw.name
    table["universe"] = universe
    table["company_label"] = "Name unavailable locally"
    # Membership labels can be current company names; the JKP security id is the stable identifier.
    membership = ROOT / "data/source" / f"membership_{universe}.csv"
    if membership.exists() and len(table):
        names = pd.read_csv(membership, usecols=["id", "form_year", "conm"])
        names = names.dropna(subset=["id", "form_year", "conm"])
        names["stock_id"] = pd.to_numeric(names.id, errors="raise").astype("int64")
        labels = names.groupby(["stock_id", "form_year"]).conm.agg(
            lambda values: " / ".join(sorted(set(values.astype(str)))))
        formation_year = table.formation_month_end.dt.year - table.formation_month_end.dt.month.ne(12).astype(int)
        keys = pd.MultiIndex.from_arrays([table.stock_id, formation_year])
        table["company_label"] = labels.reindex(keys).to_numpy()
        table["company_label"] = table.company_label.fillna("Name unavailable locally")
    columns = ["universe", "stock_id", "company_label", "formation_month_end", "return_month_end",
               "source_excess_return", "assumed_excess_return", "resulting_total_return"]
    return table[columns].sort_values(["return_month_end", "stock_id"], ignore_index=True)


def report_if_enabled(cfg):
    if not reporting_enabled():
        return
    source = cfg.paths.processed / "zero_return_imputations.csv"
    key = (_stage, str(cfg.paths.processed.resolve()), str(cfg.paths.outputs.resolve()))
    if key in _reported:
        return
    table = build_table(cfg)
    if table is None:
        print(f"\nZERO-RETURN AUDIT | {_stage} | {cfg.paths.raw.name}: prepared audit not yet available.", flush=True)
        return  # Preparation can create it later in this same run.
    destination = cfg.paths.outputs.resolve() / "imputation_audit"
    if not destination.is_relative_to(ROOT.resolve()):
        raise ValueError("Refusing to write the stage imputation audit outside the isolated Kati workflow")
    destination.mkdir(parents=True, exist_ok=True)
    filename = re.sub(r"[^A-Za-z0-9_.-]+", "_", _stage).strip("_") or "stage"
    path = destination / f"{filename}.csv"
    saved = table.copy()
    for column in ("formation_month_end", "return_month_end"):
        saved[column] = saved[column].dt.strftime("%Y-%m-%d")
    saved.to_csv(path, index=False, na_rep="MISSING")
    print(f"\nZERO-RETURN ASSUMPTIONS | {_stage} | {cfg.paths.raw.name}", flush=True)
    print(f"Prepared input: {cfg.paths.processed}", flush=True)
    print(f"{len(table)} input-dataset observations have MISSING next-month EXCESS returns assumed to be 0%.", flush=True)
    if len(table):
        shown = saved.copy()
        shown["source_excess_return"] = "MISSING"
        for column in ("assumed_excess_return", "resulting_total_return"):
            shown[column] = shown[column].map(lambda value: f"{value:.6%}")
        print(shown.to_string(index=False, max_rows=None, max_cols=None), flush=True)
        print("These are assumptions, not observed zero returns. Total return equals the next-month risk-free rate.", flush=True)
        print("The table covers the full prepared history, including warmup; not every row necessarily affects this stage.", flush=True)
        print("Company labels come from local membership files; use stock_id as the stable security identifier.", flush=True)
    stage_name = Path(_stage.split()[0]).name
    if stage_name == "02" or stage_name.startswith("02_"):
        print("Covariance uses historical daily returns; these next-month values are not risk-estimation inputs.", flush=True)
    print(f"Saved stage audit: {path}\n", flush=True)
    exclusions = cfg.paths.processed / "history_exclusions.csv"
    if exclusions.exists():
        excluded = pd.read_csv(exclusions)
        if len(excluded):
            print(f"HISTORY EXCLUSIONS | {_stage}: {len(excluded)} stock/month exclusions", flush=True)
            print(excluded.to_string(index=False, max_rows=None, max_cols=None), flush=True)
            print("Membership resumes only when the required historical rows are available; no synthetic lag history was inserted.", flush=True)
        excluded.to_csv(destination / f"{filename}_history_exclusions.csv", index=False)
    _reported.add(key)


### `scripts/jkmp/models/__init__.py`

Return-prediction models: random Fourier features + ridge.

**Role:** Calculation library. Other stages import its functions; this cell writes the module.

**Input:** the embedded source below. **Output:** `scripts/jkmp/models/__init__.py`. Expand the code cell to read or edit it.


In [ ]:
%%writefile scripts/jkmp/models/__init__.py
"""Return-prediction models: random Fourier features + ridge."""


### `scripts/jkmp/models/random_features.py`

Random Fourier features (Eqs. 38–39). Port of ``rff`` in reference/jkmp_r/0 - Return prediction functions.R. W ~ N(0, g·I) (k × p/2; g is a VARIANCE, as in R's ``mvrnorm(Sigma = g*diag(k))``) RF(s) = p^{-1/2} [cos(sW), sin(sW)] R calls ``set.seed(seed)`` before each g, so every g shares the same standard-normal draws scaled by √g; a model with p features uses the first p/2 cos and the first p/2 sin columns of the p_max draw. NumPy's generator differs from R's, so the draws themselves are not R's (docs/deviations.md).

**Role:** Calculation library. Other stages import its functions; this cell writes the module.

**Input:** the embedded source below. **Output:** `scripts/jkmp/models/random_features.py`. Expand the code cell to read or edit it.


In [ ]:
%%writefile scripts/jkmp/models/random_features.py
"""Random Fourier features (Eqs. 38–39). Port of ``rff`` in reference/jkmp_r/0 - Return prediction functions.R.

    W ~ N(0, g·I)  (k × p/2; g is a VARIANCE, as in R's ``mvrnorm(Sigma = g*diag(k))``)
    RF(s) = p^{-1/2} [cos(sW), sin(sW)]

R calls ``set.seed(seed)`` before each g, so every g shares the same standard-normal draws scaled by
√g; a model with p features uses the first p/2 cos and the first p/2 sin columns of the p_max draw.
NumPy's generator differs from R's, so the draws themselves are not R's (docs/deviations.md).
"""

from __future__ import annotations

import numpy as np


def base_draws(n_inputs: int, p_max: int, seed: int) -> np.ndarray:
    """Standard-normal matrix Z (n_inputs × p_max/2), shared by every g."""
    if p_max % 2:
        raise ValueError("p_max must be even")
    return np.random.default_rng(seed).standard_normal((n_inputs, p_max // 2))


def weights_for(z: np.ndarray, variance: float) -> np.ndarray:
    """W = √g · Z, i.e. W ~ N(0, g·I)."""
    return np.sqrt(variance) * z


def rff_raw(x: np.ndarray, w: np.ndarray) -> np.ndarray:
    """Unscaled features [cos(xW), sin(xW)] for every column of W (n × 2·W.shape[1])."""
    with np.errstate(all="ignore"):  # spurious Accelerate matmul flags; checked below
        xw = x @ w
    if not np.isfinite(xw).all():
        raise FloatingPointError("non-finite random-feature projections")
    return np.hstack([np.cos(xw), np.sin(xw)])


def column_subset(p: int, p_max: int) -> np.ndarray:
    """Columns of ``rff_raw`` (p_max features) that make up the first p features: cos 1..p/2, sin 1..p/2."""
    if p % 2 or p > p_max:
        raise ValueError(f"p must be even and <= {p_max}")
    half, half_max = p // 2, p_max // 2
    return np.concatenate([np.arange(half), half_max + np.arange(half)])


def rff(x: np.ndarray, w: np.ndarray, p: int) -> np.ndarray:
    """RF(x) with p features: p^{-1/2} [cos(xW[:, :p/2]), sin(xW[:, :p/2])] (R scales by ``p^(-0.5)``)."""
    return rff_raw(x, w[:, : p // 2]) / np.sqrt(p)


### `scripts/jkmp/models/return_model.py`

One-month return predictions ``pred_ld1``: random-feature ridge refit every year (expanding window). Port of ``rff_hp_search`` (0 - Return prediction functions.R) with ``data_split`` (yearly updates), with the validation window shortened to ``val_years`` (R: 10). For each return year y: * train : formation months with returns realised up to Dec y−1−val_years (eom ≤ Nov y−1−val_years) * val : the next ``val_years`` years of realised returns (Dec y−1−val_years .. Nov y−1) * pick (g, p, λ) by validation MSE; refit on train + val (eom ≤ Nov y−1) * 

**Role:** Calculation library. Other stages import its functions; this cell writes the module.

**Input:** the embedded source below. **Output:** `scripts/jkmp/models/return_model.py`. Expand the code cell to read or edit it.


In [ ]:
%%writefile scripts/jkmp/models/return_model.py
"""One-month return predictions ``pred_ld1``: random-feature ridge refit every year (expanding window).

Port of ``rff_hp_search`` (0 - Return prediction functions.R) with ``data_split`` (yearly updates),
with the validation window shortened to ``val_years`` (R: 10). For each return year y:

* train : formation months with returns realised up to Dec y−1−val_years  (eom ≤ Nov y−1−val_years)
* val   : the next ``val_years`` years of realised returns                (Dec y−1−val_years .. Nov y−1)
* pick (g, p, λ) by validation MSE; refit on train + val                  (eom ≤ Nov y−1)
* test  : predict for formation months Dec y−1 .. Nov y (returns realised in year y)

Training rows are all screened stocks with a 12-month history (``valid_data``), not just the portfolio
universe — R also fits its return models on all US stocks. With ``validation_scope=universe``,
only this market's historical members score the candidate (g, p, lambda) settings. Each market
uses its own panel, memberships, selected settings and saved coefficients. The target is
``ret_ld1``; inputs are the ranked characteristics in [0, 1].
"""

from __future__ import annotations

import logging
import json
import hashlib
from collections.abc import Sequence
from dataclasses import dataclass, field
from pathlib import Path

import numpy as np
import pandas as pd

from jkmp.config import Config, DatesConfig
from jkmp.evaluation.holdout import check_return_months, last_open_month
from jkmp.models.random_features import base_draws, column_subset, rff, rff_raw, weights_for
from jkmp.models.ridge import Moments, mse_from_moments, ridge_path

log = logging.getLogger(__name__)


def validation_rows(rows: pd.DataFrame, scope: str) -> pd.DataFrame:
    """Choose scoring stocks using membership at each historical formation month.

    Coefficient fitting still uses the screened regional panel, as in the paper.
    No present-day membership list or future stock return enters this filter.
    """
    if scope == "all":
        return rows
    if scope != "universe":
        raise ValueError(f"Unknown return-model validation scope: {scope}")
    if "in_universe" not in rows:
        raise ValueError("Market-specific RFF validation requires historical in_universe membership")
    selected = rows.loc[rows["in_universe"].eq(True)]
    if selected.empty:
        raise ValueError("No eligible historical market members for RFF validation")
    return selected


def tuning_provenance(cfg: Config) -> dict:
    """Identify the market, search settings, source and prepared panel without fitting."""
    panel = cfg.paths.processed / "panel.parquet"
    info = panel.stat() if panel.exists() else None
    source = Path(__file__).resolve()
    return dict(version=2, market=cfg.paths.raw.name, processed=str(cfg.paths.processed),
                return_model=cfg.as_dict()["return_model"], ridge_grid=cfg.as_dict()["ridge_grid"],
                features=list(cfg.features.names), seed=cfg.seed,
                panel=dict(size=info.st_size, mtime_ns=info.st_mtime_ns) if info else None,
                sources={name: hashlib.sha256((source.parent / name).read_bytes()).hexdigest()
                         for name in (source.name, "random_features.py", "ridge.py")})


def tuning_marker(cfg: Config) -> Path:
    """Old broad-panel fits cannot satisfy a market-specific tuning checkpoint."""
    token = hashlib.sha256(json.dumps(tuning_provenance(cfg), sort_keys=True).encode()).hexdigest()[:16]
    return cfg.paths.processed / "return_model" / f"tuning_{token}.json"


@dataclass
class YearModel:
    """The fitted model that predicts return year ``year``: pred = RF_p(x; √g·Z) @ coef + b0."""
    year: int
    g: float
    p: int
    lam: float
    coef: np.ndarray   # (p,)
    b0: float

    def predict(self, x: np.ndarray, z: np.ndarray) -> np.ndarray:
        with np.errstate(all="ignore"):  # spurious Accelerate matmul flags; checked below
            pred = rff(x, weights_for(z, self.g), self.p) @ self.coef + self.b0
        if not np.isfinite(pred).all():
            raise FloatingPointError("non-finite predictions")
        return pred


@dataclass
class ReturnModelResult:
    predictions: pd.DataFrame   # id, eom, year, pred_ld1
    hp_search: pd.DataFrame     # year, g, p, lam, val_mse
    chosen: pd.DataFrame        # one row per year: g, p, lam, val_mse, n_train, n_val, n_full
    models: dict[int, YearModel] = field(default_factory=dict)
    z: np.ndarray | None = None  # standard-normal draws shared by every g (n_features × p_max/2)


def cutoffs_for_year(year: int, val_years: int) -> tuple[pd.Timestamp, pd.Timestamp]:
    """(last training formation month, last train+val formation month) for return year ``year``."""
    return pd.Timestamp(year - 1 - val_years, 11, 30), pd.Timestamp(year - 1, 11, 30)


def formation_months_for_year(year: int) -> tuple[pd.Timestamp, pd.Timestamp]:
    """Formation months whose next-month return falls in ``year``: Dec y−1 .. Nov y."""
    return pd.Timestamp(year - 1, 12, 31), pd.Timestamp(year, 11, 30)


def prefix_moments(rows: pd.DataFrame, features: list[str], w: np.ndarray,
                   snapshots: list[pd.Timestamp]) -> dict[pd.Timestamp, Moments]:
    """Running sufficient statistics of [cos(sW), sin(sW)] and ret_ld1, recorded at each snapshot month."""
    wanted = set(snapshots)
    running = Moments.zeros(2 * w.shape[1])
    out = {}
    for eom, g in rows.groupby("eom", sort=True):
        if eom > max(snapshots):
            break
        running = running + Moments.of(rff_raw(g[features].to_numpy(), w), g["ret_ld1"].to_numpy())
        if eom in wanted:
            out[eom] = running
    missing = wanted - set(out)
    if missing:
        raise ValueError(f"no data in snapshot months {sorted(missing)[:3]}")
    return out


def prediction_years(cfg: Config) -> list[int]:
    """Return years to fit and predict: ``return_model.pred_years``, cut at the last open year while the holdout is
    locked, so no validation MSE is ever computed on holdout return months."""
    first, last = cfg.return_model.pred_years
    return list(range(first, min(last, last_open_month(cfg.dates).year) + 1))


def fit_return_model(panel: pd.DataFrame, cfg: Config) -> ReturnModelResult:
    rm = cfg.return_model
    feats = list(cfg.features.names)
    lambdas = np.asarray(cfg.ridge_grid.values)
    p_grid, g_grid = list(rm.p_grid), list(rm.rff_variance_grid)
    p_max = max(p_grid)
    years = prediction_years(cfg)

    columns = ["id", "eom", *feats, "ret_ld1"]
    if "in_universe" in panel:
        columns.append("in_universe")
    rows = panel.loc[panel["valid_data"] & ((panel["eom"] + pd.offsets.MonthEnd(1)) >= pd.Timestamp("2005-01-31")), columns]
    scoring_rows = validation_rows(rows, rm.validation_scope)
    # A held stock can be forecast from current characteristics even when its
    # current realised return makes that row ineligible for fitting.
    prediction_rows = panel.loc[panel["valid_data"] | panel["in_universe"], ["id", "eom", *feats, "ret_ld1"]] if "in_universe" in panel else rows
    cut = {y: cutoffs_for_year(y, rm.val_years) for y in years}
    snapshots = sorted({c for pair in cut.values() for c in pair})
    z = base_draws(len(feats), p_max, cfg.seed)  # same draws for every g (R: set.seed before each g)
    moments = {g: prefix_moments(rows, feats, weights_for(z, g), snapshots) for g in g_grid}
    scoring_moments = moments if rm.validation_scope == "all" else {
        g: prefix_moments(scoring_rows, feats, weights_for(z, g), snapshots) for g in g_grid}
    log.info("Moments: %d rows, %d snapshot months, p_max %d, %d variances", len(rows), len(snapshots), p_max, len(g_grid))

    searches, chosen, preds, models = [], [], [], {}
    for y in years:
        tr_cut, full_cut = cut[y]
        records = []
        for g in g_grid:
            m_train, m_full = moments[g][tr_cut], moments[g][full_cut]
            m_val = scoring_moments[g][full_cut] - scoring_moments[g][tr_cut]
            if m_val.n <= 0:
                raise ValueError(f"No {cfg.paths.raw.name} validation rows for return year {y}")
            for p in p_grid:
                cols, scale = column_subset(p, p_max), p ** -0.5
                coef, b0 = ridge_path(m_train.select(cols, scale), lambdas)
                mse = mse_from_moments(m_val.select(cols, scale), coef, b0)
                records.append(pd.DataFrame({"year": y, "g": g, "p": p, "lam": lambdas, "val_mse": mse,
                                             "market": cfg.paths.raw.name, "validation_scope": rm.validation_scope,
                                             "n_val": m_val.n}))
        search = pd.concat(records, ignore_index=True)
        best = search.loc[search["val_mse"].idxmin()]
        g, p, lam = float(best["g"]), int(best["p"]), float(best["lam"])

        cols, scale = column_subset(p, p_max), p ** -0.5
        m_full = moments[g][full_cut]
        coef, b0 = ridge_path(m_full.select(cols, scale), np.array([lam]))
        lo, hi = formation_months_for_year(y)
        test = prediction_rows[prediction_rows["eom"].between(lo, hi)]
        models[y] = YearModel(y, g, p, lam, coef[:, 0].copy(), float(b0[0]))
        pred = models[y].predict(test[feats].to_numpy(), z)
        preds.append(test[["id", "eom"]].assign(year=y, pred_ld1=pred))
        n_train = moments[g][tr_cut].n
        n_val = scoring_moments[g][full_cut].n - scoring_moments[g][tr_cut].n
        chosen.append({"year": y, "g": g, "p": p, "lam": lam, "log_lam": np.log(lam) if lam > 0 else -np.inf,
                       "val_mse": float(best["val_mse"]), "n_train": n_train, "n_val": n_val,
                       "n_val_regional": m_full.n - n_train, "n_full": m_full.n, "n_test": len(test),
                       "market": cfg.paths.raw.name, "validation_scope": rm.validation_scope})
        searches.append(search)
        log.info("Year %d: g=%.3f p=%d log(λ)=%s val MSE %.6f; %d train / %d val / %d test rows",
                 y, g, p, f"{np.log(lam):.1f}" if lam > 0 else "-inf (λ=0)", best["val_mse"],
                 n_train, n_val, len(test))
    return ReturnModelResult(pd.concat(preds, ignore_index=True), pd.concat(searches, ignore_index=True),
                             pd.DataFrame(chosen), models, z)


def save_models(result: ReturnModelResult, path: Path) -> None:
    """Fitted yearly models + the shared draws Z, so predictions can be recomputed on new inputs."""
    years = sorted(result.models)
    m = [result.models[y] for y in years]
    np.savez(path, z=result.z, years=np.array(years),
             meta=json.dumps([{"year": x.year, "g": x.g, "p": x.p, "lam": x.lam, "b0": x.b0} for x in m]),
             **{f"coef_{x.year}": x.coef for x in m})


def load_models(path: Path) -> tuple[dict[int, YearModel], np.ndarray]:
    with np.load(path) as f:
        meta = json.loads(str(f["meta"]))
        models = {d["year"]: YearModel(d["year"], d["g"], d["p"], d["lam"], f[f"coef_{d['year']}"], d["b0"])
                  for d in meta}
        return models, f["z"]


def rank_ic_by_size(predictions: pd.DataFrame, panel: pd.DataFrame, buckets: Sequence[Sequence[int]],
                    dates: DatesConfig) -> pd.DataFrame:
    """Mean monthly Spearman rank IC of pred_ld1 vs ret_ld1 within market-cap rank buckets (rank among the
    screened ``valid_data`` stocks each month, 1 = largest), per decade of the return month and overall."""
    check_return_months(predictions, dates, "rank_ic_by_size")
    d = predictions[["id", "eom", "pred_ld1"]].merge(panel.loc[panel["valid_data"], ["id", "eom", "me", "ret_ld1"]],
                                                     on=["id", "eom"])
    d["size_rank"] = d.groupby("eom")["me"].rank(ascending=False, method="first")
    out = []
    for lo, hi in buckets:
        b = d[d["size_rank"].between(lo, hi)]
        ic = b.groupby("eom")[["pred_ld1", "ret_ld1"]].apply(lambda x: x["pred_ld1"].rank().corr(x["ret_ld1"].rank()))
        decade = (ic.index + pd.offsets.MonthEnd(1)).year // 10 * 10
        label = f"{lo}-{hi}"
        out.append(ic.groupby(decade).agg(["mean", "size"]).assign(bucket=label).rename_axis("decade").reset_index())
        out.append(pd.DataFrame({"decade": ["all"], "mean": [ic.mean()], "size": [len(ic)], "bucket": [label]}))
    return pd.concat(out, ignore_index=True).rename(columns={"mean": "rank_ic", "size": "months"})


def monthly_rank_ic(predictions: pd.DataFrame, panel: pd.DataFrame, dates: DatesConfig) -> pd.Series:
    """Spearman rank IC of pred_ld1 vs ret_ld1 across the portfolio universe, one value per return month."""
    check_return_months(predictions, dates, "monthly_rank_ic")
    d = predictions[["id", "eom", "pred_ld1"]].merge(panel.loc[panel["in_universe"], ["id", "eom", "ret_ld1"]],
                                                     on=["id", "eom"])
    ic = d.groupby("eom")[["pred_ld1", "ret_ld1"]].apply(lambda x: x["pred_ld1"].rank().corr(x["ret_ld1"].rank()))
    return pd.Series(ic.to_numpy(), index=ic.index + pd.offsets.MonthEnd(1), name="rank_ic").rename_axis("eom_ret")


def prediction_diagnostics(predictions: pd.DataFrame, panel: pd.DataFrame, dates: DatesConfig) -> pd.DataFrame:
    """Per return year: out-of-sample R² against a zero forecast (Gu, Kelly and Xiu, 2020) and the mean
    monthly Spearman rank IC, for all predicted stocks and for the portfolio universe."""
    check_return_months(predictions, dates, "prediction_diagnostics")
    d = predictions.merge(panel[["id", "eom", "ret_ld1", "in_universe"]], on=["id", "eom"], how="left")
    out = []
    for (year, scope), g in pd.concat([d.assign(scope="all"), d[d["in_universe"]].assign(scope="universe")]).groupby(["year", "scope"]):
        ic = g.groupby("eom")[["pred_ld1", "ret_ld1"]].apply(lambda x: x["pred_ld1"].rank().corr(x["ret_ld1"].rank()))
        out.append({"year": year, "scope": scope, "n": len(g),
                    "r2_oos": 1 - ((g["ret_ld1"] - g["pred_ld1"]) ** 2).sum() / (g["ret_ld1"] ** 2).sum(),
                    "rank_ic": ic.mean(), "pred_sd": g["pred_ld1"].std()})
    return pd.DataFrame(out)


### `scripts/jkmp/models/ridge.py`

Closed-form ridge with an unpenalised intercept, for a whole λ grid from one eigendecomposition. Replaces R's ``glmnet(alpha = 0, standardize = FALSE)``, which minimises (1/2n) Σ (y − β₀ − xβ)² + (λ/2) ‖β‖² ⇒ β(λ) = (C + λI)⁻¹ c, β₀ = ȳ − x̄β, with C = X̃ᵀX̃/n and c = X̃ᵀỹ/n on centred data. With C = V diag(d) Vᵀ every λ costs one matrix product: β(λ) = V diag(1/(d + λ)) Vᵀ c. All quantities come from additive sufficient statistics (``Moments``), so a training window is a sum of monthly moments and a validation window is a difference of two run

**Role:** Calculation library. Other stages import its functions; this cell writes the module.

**Input:** the embedded source below. **Output:** `scripts/jkmp/models/ridge.py`. Expand the code cell to read or edit it.


In [ ]:
%%writefile scripts/jkmp/models/ridge.py
"""Closed-form ridge with an unpenalised intercept, for a whole λ grid from one eigendecomposition.

Replaces R's ``glmnet(alpha = 0, standardize = FALSE)``, which minimises

    (1/2n) Σ (y − β₀ − xβ)² + (λ/2) ‖β‖²   ⇒   β(λ) = (C + λI)⁻¹ c,  β₀ = ȳ − x̄β,

with C = X̃ᵀX̃/n and c = X̃ᵀỹ/n on centred data. With C = V diag(d) Vᵀ every λ costs one matrix
product: β(λ) = V diag(1/(d + λ)) Vᵀ c.

All quantities come from additive sufficient statistics (``Moments``), so a training window is a sum
of monthly moments and a validation window is a difference of two running sums; validation MSE for
every λ is computed from the validation moments without materialising the validation design matrix.
"""

from __future__ import annotations

from dataclasses import dataclass

import numpy as np
from scipy.linalg import eigh


@dataclass
class Moments:
    """Sums over rows of a design X (n × P) and target y."""

    n: int
    sx: np.ndarray    # Σ x        (P,)
    sy: float         # Σ y
    sxx: np.ndarray   # Σ x xᵀ     (P, P)
    sxy: np.ndarray   # Σ x y      (P,)
    syy: float        # Σ y²

    @classmethod
    def zeros(cls, n_cols: int) -> "Moments":
        return cls(0, np.zeros(n_cols), 0.0, np.zeros((n_cols, n_cols)), np.zeros(n_cols), 0.0)

    @classmethod
    def of(cls, x: np.ndarray, y: np.ndarray) -> "Moments":
        with np.errstate(all="ignore"):  # spurious Accelerate matmul flags; checked below
            m = cls(len(y), x.sum(axis=0), float(y.sum()), x.T @ x, x.T @ y, float(y @ y))
        if not (np.isfinite(m.sxx).all() and np.isfinite(m.sxy).all()):
            raise FloatingPointError("non-finite moments")
        return m

    def __add__(self, o: "Moments") -> "Moments":
        return Moments(self.n + o.n, self.sx + o.sx, self.sy + o.sy, self.sxx + o.sxx, self.sxy + o.sxy, self.syy + o.syy)

    def __sub__(self, o: "Moments") -> "Moments":
        return Moments(self.n - o.n, self.sx - o.sx, self.sy - o.sy, self.sxx - o.sxx, self.sxy - o.sxy, self.syy - o.syy)

    def select(self, cols: np.ndarray, scale: float) -> "Moments":
        """Moments of the design ``scale · X[:, cols]``."""
        return Moments(self.n, self.sx[cols] * scale, self.sy, self.sxx[np.ix_(cols, cols)] * scale**2,
                       self.sxy[cols] * scale, self.syy)


def ridge_path(train: Moments, lambdas: np.ndarray) -> tuple[np.ndarray, np.ndarray]:
    """Coefficients (P × L) and intercepts (L,) for every λ, from one eigendecomposition.

    λ = 0 gives OLS; directions with a numerically zero eigenvalue (and λ = 0) get zero weight, i.e.
    the minimum-norm least-squares solution.
    """
    n = train.n
    xbar, ybar = train.sx / n, train.sy / n
    C = train.sxx / n - np.outer(xbar, xbar)
    c = train.sxy / n - xbar * ybar
    d, V = eigh(0.5 * (C + C.T))
    d = np.clip(d, 0.0, None)
    lam = np.asarray(lambdas, dtype=float)
    denom = d[:, None] + lam[None, :]
    tiny = denom <= 1e-12 * max(d.max(), 1e-300)
    inv = np.where(tiny, 0.0, 1.0 / np.where(tiny, 1.0, denom))
    with np.errstate(all="ignore"):
        coef = V @ (inv * (V.T @ c)[:, None])
        intercept = ybar - xbar @ coef
    if not (np.isfinite(coef).all() and np.isfinite(intercept).all()):
        raise FloatingPointError("non-finite ridge coefficients")
    return coef, intercept


def mse_from_moments(m: Moments, coef: np.ndarray, intercept: np.ndarray) -> np.ndarray:
    """Mean squared error of y − β₀ − xβ over the rows summarised by ``m``, for each column of ``coef``."""
    with np.errstate(all="ignore"):
        quad = np.einsum("pl,pl->l", coef, m.sxx @ coef)
        sse = (m.syy - 2 * intercept * m.sy - 2 * (m.sxy @ coef) + m.n * intercept**2
               + 2 * intercept * (m.sx @ coef) + quad)
    return sse / m.n


### `scripts/jkmp/portfolio/__init__.py`

Portfolio construction: accounting, trading speed, and portfolio methods.

**Role:** Calculation library. Other stages import its functions; this cell writes the module.

**Input:** the embedded source below. **Output:** `scripts/jkmp/portfolio/__init__.py`. Expand the code cell to read or edit it.


In [ ]:
%%writefile scripts/jkmp/portfolio/__init__.py
"""Portfolio construction: accounting, trading speed, and portfolio methods."""


### `scripts/jkmp/portfolio/accounting.py`

Weight bookkeeping and trading costs. Port of ``w_fun`` (0 - Portfolio choice functions.R) and ``pf_ts_fun`` (0 - General functions.R). Every portfolio method produces target weights ``w`` per formation month t. Trading happens from the *inherited* weights ``w_start``: in the first month the value-weighted universe portfolio (R: ``initial_weights_new(w_type = "vw")``); afterwards last month's weights drifted by ``(1 + tr_ld1) / (1 + mu_ld1)`` (Eqs. 5–6). Stocks without a previous weight start at 0. Exits: R drops a stock that leaves the univers

**Role:** Calculation library. Other stages import its functions; this cell writes the module.

**Input:** the embedded source below. **Output:** `scripts/jkmp/portfolio/accounting.py`. Expand the code cell to read or edit it.


In [ ]:
%%writefile scripts/jkmp/portfolio/accounting.py
"""Weight bookkeeping and trading costs. Port of ``w_fun`` (0 - Portfolio choice functions.R) and
``pf_ts_fun`` (0 - General functions.R).

Every portfolio method produces target weights ``w`` per formation month t. Trading happens from the
*inherited* weights ``w_start``: in the first month the value-weighted universe portfolio (R:
``initial_weights_new(w_type = "vw")``); afterwards last month's weights drifted by
``(1 + tr_ld1) / (1 + mu_ld1)`` (Eqs. 5–6). Stocks without a previous weight start at 0.

Exits: R drops a stock that leaves the universe, so selling it is free. With ``charge_exit_tc`` we add
a row ``w = 0`` for every stock held at t−1 that is not in the universe at t but still has a panel
row at t (it still trades, so its Λ_t is known). A stock with no row at t delisted or left the data:
its position was settled by the delisting return, so there is no sale and no cost.
"""

from __future__ import annotations

import logging
from collections.abc import Sequence

import numpy as np
import pandas as pd

log = logging.getLogger(__name__)

MONTH = pd.offsets.MonthEnd(1)


def drift_weights(weights: pd.DataFrame, panel: pd.DataFrame, wealth: pd.DataFrame) -> pd.DataFrame:
    """Weights at t (``id, eom, w``) grown to the start of t+1: ``w·(1 + tr_ld1)/(1 + mu_ld1)``.

    Returns ``id, eom, w_start`` with ``eom`` = t+1.
    """
    d = (weights[["id", "eom", "w"]]
         .merge(panel[["id", "eom", "tr_ld1"]], on=["id", "eom"], how="left")
         .merge(wealth[["eom", "mu_ld1"]], on="eom", how="left"))
    if d[["tr_ld1", "mu_ld1"]].isna().any().any():
        bad = d[d[["tr_ld1", "mu_ld1"]].isna().any(axis=1)]
        raise ValueError(f"{len(bad)} held positions lack tr_ld1/mu_ld1, e.g. {bad.iloc[0][['id', 'eom']].tolist()}")
    return pd.DataFrame({"id": d["id"], "eom": d["eom"] + MONTH,
                         "w_start": d["w"] * (1.0 + d["tr_ld1"]) / (1.0 + d["mu_ld1"])})


def start_weights_for(eom: pd.Timestamp, universe_ids: np.ndarray, drifted: pd.DataFrame | None,
                      panel_t: pd.DataFrame, charge_exit_tc: bool) -> pd.DataFrame:
    """Inherited weights at one month for the universe (+ exit rows): ``id, w_start, exit``.

    ``drifted`` holds last month's drifted weights (``id, w_start``); ``None`` in the first month, when
    the start is the value-weighted universe. ``panel_t`` is the panel at ``eom`` (``id, me``).
    """
    uni = pd.DataFrame({"id": np.asarray(universe_ids)})
    if drifted is None:
        me = uni.merge(panel_t[["id", "me"]], on="id", how="left")["me"]
        return uni.assign(w_start=(me / me.sum()).to_numpy(), exit=False)
    out = uni.merge(drifted[["id", "w_start"]], on="id", how="left").fillna({"w_start": 0.0}).assign(exit=False)
    leaving = drifted[~drifted["id"].isin(uni["id"]) & (drifted["w_start"] != 0.0)]
    if charge_exit_tc and len(leaving):
        trades = leaving["id"].isin(panel_t["id"])
        out = pd.concat([out, leaving.loc[trades, ["id", "w_start"]].assign(exit=True)], ignore_index=True)
    return out


def add_start_weights(targets: pd.DataFrame, panel: pd.DataFrame, wealth: pd.DataFrame,
                      dates: Sequence[pd.Timestamp], charge_exit_tc: bool) -> pd.DataFrame:
    """R ``w_fun`` for methods whose targets do not depend on inherited weights.

    ``targets``: ``id, eom, w`` for the universe at each date (consecutive month-ends). Returns
    ``id, eom, w, w_start, exit``; exit rows (``w = 0``) are added only with ``charge_exit_tc``.
    """
    dates = pd.DatetimeIndex(sorted(dates))
    if len(dates) > 1 and not (dates[1:] == dates[:-1] + MONTH).all():
        raise ValueError("dates must be consecutive month-ends")
    tg = targets.loc[targets["eom"].isin(dates), ["id", "eom", "w"]]
    drifted = drift_weights(tg[tg["eom"] < dates[-1]], panel, wealth)
    drifted_by = dict(tuple(drifted.groupby("eom")))
    panel_by = dict(tuple(panel.loc[panel["eom"].isin(dates), ["id", "eom", "me"]].groupby("eom")))

    rows, n_charged, n_free = [], 0, 0
    for i, (eom, g) in enumerate(tg.groupby("eom", sort=True)):
        prev = None if i == 0 else drifted_by.get(eom, pd.DataFrame(columns=["id", "w_start"]))
        start = start_weights_for(eom, g["id"].to_numpy(), prev, panel_by[eom], charge_exit_tc)
        if prev is not None:
            leaving = prev[~prev["id"].isin(g["id"]) & (prev["w_start"] != 0.0)]
            charged = int(start["exit"].sum())
            n_charged += charged
            n_free += len(leaving) - charged
        rows.append(start.merge(g[["id", "w"]], on="id", how="left").fillna({"w": 0.0}).assign(eom=eom))
    out = pd.concat(rows, ignore_index=True)[["id", "eom", "w", "w_start", "exit"]]
    log.info("Exits: %d charged as sales, %d without a sale (delisted/left data or charge_exit_tc off)",
             n_charged, n_free)
    return out


def portfolio_returns(weights: pd.DataFrame, panel: pd.DataFrame, wealth: pd.DataFrame) -> pd.DataFrame:
    """R ``pf_ts_fun``: per formation month t, leverage, shorting, turnover, gross return over t+1 and
    trading cost ``wealth_t/2 · Σ λ_i (w_i − w_start_i)²`` (fraction of wealth). Row dated ``eom_ret = t+1``.
    """
    c = (weights.merge(panel[["id", "eom", "ret_ld1", "lambda"]], on=["id", "eom"], how="left")
         .merge(wealth[["eom", "wealth"]], on="eom", how="left"))
    held = ~c["exit"]
    if c.loc[held, ["ret_ld1", "lambda"]].isna().any().any() or c["lambda"].isna().any() or c["wealth"].isna().any():
        raise ValueError("missing ret_ld1, lambda or wealth for some weights")
    c["ret_ld1"] = c["ret_ld1"].where(held, 0.0)  # exit rows: w = 0, nothing earned
    dw = c["w"] - c["w_start"]
    c = c.assign(abs_w=c["w"].abs(), short=c["w"].clip(upper=0).abs(), abs_dw=dw.abs(),
                 r_i=c["w"] * c["ret_ld1"], tc_i=0.5 * c["wealth"] * c["lambda"] * dw**2)
    pf = c.groupby("eom").agg(inv=("abs_w", "sum"), shorting=("short", "sum"), turnover=("abs_dw", "sum"),
                              r=("r_i", "sum"), tc=("tc_i", "sum")).reset_index()
    pf.insert(1, "eom_ret", pf["eom"] + MONTH)
    return pf


### `scripts/jkmp/portfolio/active.py`

Benchmark-relative (active) Portfolio-ML — our extension (docs/deviations.md D34, research log trial 2). π_t = π^VW_t + a_t with a_t the Portfolio-ML aim/recursion on active weights. The objective is the Portfolio-ML objective with the active risk aversion γ_TE in place of γ, so the inputs (r̃ = Ωᵀr, the active return of each signal; Σ̃ = γ_TE·ΩᵀΣΩ + wΔΩᵀΛΔΩ) and m come from the unchanged functions. Runs use the yearly sums of ``grids`` (same β and December selection as scripts 06–07, without monthly files). γ_TE is calibrated to ex-ante tracki

**Role:** Calculation library. Other stages import its functions; this cell writes the module.

**Input:** the embedded source below. **Output:** `scripts/jkmp/portfolio/active.py`. Expand the code cell to read or edit it.


In [ ]:
%%writefile scripts/jkmp/portfolio/active.py
"""Benchmark-relative (active) Portfolio-ML — our extension (docs/deviations.md D34, research log trial 2).

π_t = π^VW_t + a_t with a_t the Portfolio-ML aim/recursion on active weights. The objective is the
Portfolio-ML objective with the active risk aversion γ_TE in place of γ, so the inputs (r̃ = Ωᵀr, the active
return of each signal; Σ̃ = γ_TE·ΩᵀΣΩ + wΔΩᵀΛΔΩ) and m come from the unchanged functions. Runs use the
yearly sums of ``grids`` (same β and December selection as scripts 06–07, without monthly files).

γ_TE is calibrated to ex-ante tracking-error budgets **before** the test window: each grid value is run with hp
years up to the end of validation only, and the mean ex-ante TE of the implemented active weights over those
validation return months is interpolated (log TE against log γ_TE) at each budget.
"""

from __future__ import annotations

import logging
from collections.abc import Sequence

import numpy as np
import pandas as pd

from jkmp.config import Config
from jkmp.evaluation.metrics import active_weights, ex_ante_exposures
from jkmp.portfolio.grids import GridInputs, _signal_kwargs
from jkmp.portfolio.portfolio_ml import build_yearly_moments, run_portfolio_ml, search_yearly, signal_set

log = logging.getLogger(__name__)

LABEL = "Portfolio-ML (active)"


def run_active(inp: GridInputs, cfg: Config, gamma_te: float, eval_dates: Sequence[pd.Timestamp],
               hp_years: Sequence[int]) -> dict[str, pd.DataFrame]:
    """Active Portfolio-ML at ``gamma_te``: search (g, p, λ) over ``hp_years``, build aims for ``eval_dates`` and
    run the active recursion over them from VW initial weights (``run_portfolio_ml`` outputs), as scripts/07 runs
    the baseline over the test window."""
    pm = cfg.portfolio_ml
    if not pm.benchmark_relative:
        raise ValueError("run_active needs portfolio_ml.benchmark_relative: true (configs/experiments/active_pfml.yaml)")
    eval_dates = pd.DatetimeIndex(sorted(eval_dates))
    _, tags, _ = signal_set(pm.use_rff, pm.rff_variance_grid)
    results = {}
    for tag, g in zip(tags, pm.rff_variance_grid if pm.use_rff else [None]):
        ym = build_yearly_moments(inp.panel, inp.wealth, inp.cov, inp.rf, inp.pfml_dates, cfg.features.names,
                                  signal_dates=eval_dates, gamma=gamma_te, mu=cfg.investor.mu,
                                  lb=cfg.universe.lookback_months, iterations=cfg.investor.m_iterations,
                                  scale_by_vol=pm.scale_by_vol, add_constant=pm.add_constant, **_signal_kwargs(cfg, g))
        results[tag] = search_yearly(ym, tag, pm.p_grid, cfg.ridge_grid.values, hp_years, pm.half_life_years)
    return run_portfolio_ml(results, inp.panel, inp.wealth, inp.cov, inp.rf, eval_dates, eval_dates, gamma=gamma_te,
                            mu=cfg.investor.mu, iterations=cfg.investor.m_iterations,
                            charge_exit_tc=cfg.portfolio.charge_exit_tc, label=LABEL, benchmark_relative=True)


def calibration_point(inp: GridInputs, cfg: Config, gamma_te: float) -> dict:
    """Mean ex-ante TE (annual) of the active weights over the validation return months, hp years ≤ validation end.
    Touches no return after ``dates.validation_end`` (the yearly sums of later years are never used)."""
    d = cfg.dates
    formation = pd.DatetimeIndex(sorted(inp.cov))
    ret = formation + pd.offsets.MonthEnd(1)
    hp_years = [y for y in inp.hp_years if y <= d.validation_end.year]
    aim_dates = formation[(ret.year > hp_years[0]) & (ret <= pd.Timestamp(d.validation_end))]
    res = run_active(inp, cfg, gamma_te, aim_dates, hp_years)
    ex = ex_ante_exposures(active_weights(res["weights"], inp.panel), inp.cov)
    return {"gamma_te": gamma_te, "months": len(aim_dates), "first_ret": (aim_dates[0] + pd.offsets.MonthEnd(1)).date(),
            "te_ex_ante": float(ex["te"].mean() * np.sqrt(12))}


def calibrate(points: pd.DataFrame, targets: Sequence[float]) -> pd.DataFrame:
    """γ_TE for each TE budget by linear interpolation of log γ_TE on log TE (``points``: gamma_te, te_ex_ante).
    Raises if a budget lies outside the TE range spanned by the grid (extend ``investor.gamma_te_grid``)."""
    p = points.sort_values("te_ex_ante")
    x, y = np.log(p["te_ex_ante"].to_numpy()), np.log(p["gamma_te"].to_numpy())
    if not (np.diff(y) < 0).all():
        log.warning("ex-ante TE is not monotone in gamma_te over the grid:\n%s", p.to_string(index=False))
    out = []
    for t in targets:
        if not x[0] <= np.log(t) <= x[-1]:
            raise ValueError(f"TE budget {t:.3f} outside the grid's range "
                             f"{p['te_ex_ante'].min():.4f} .. {p['te_ex_ante'].max():.4f}")
        # 4 significant figures: the value passed to the run (--override investor.gamma_te) and its folder name
        out.append({"te_target": t, "gamma_te": float(f"{np.exp(np.interp(np.log(t), x, y)):.4g}")})
    return pd.DataFrame(out)


### `scripts/jkmp/portfolio/benchmarks.py`

Benchmarks without a return model: Market, 1/N and Minimum variance. Ports ``mkt_implement``, ``ew_implement`` and ``mv_implement`` (0 - Portfolio choice functions.R): the target weights are traded to in full each month; costs come from ``jkmp.portfolio.accounting``.

**Role:** Calculation library. Other stages import its functions; this cell writes the module.

**Input:** the embedded source below. **Output:** `scripts/jkmp/portfolio/benchmarks.py`. Expand the code cell to read or edit it.


In [ ]:
%%writefile scripts/jkmp/portfolio/benchmarks.py
"""Benchmarks without a return model: Market, 1/N and Minimum variance.

Ports ``mkt_implement``, ``ew_implement`` and ``mv_implement`` (0 - Portfolio choice functions.R): the
target weights are traded to in full each month; costs come from ``jkmp.portfolio.accounting``.
"""

from __future__ import annotations

from collections.abc import Mapping, Sequence

import numpy as np
import pandas as pd
from scipy.linalg import cho_factor, cho_solve

from jkmp.covariance import CovParts
from jkmp.portfolio.accounting import add_start_weights, portfolio_returns


def market_targets(universe: pd.DataFrame) -> pd.DataFrame:
    """Value weights me / Σ me over the universe each month."""
    w = universe["me"] / universe.groupby("eom")["me"].transform("sum")
    return universe[["id", "eom"]].assign(w=w.to_numpy())


def equal_weight_targets(universe: pd.DataFrame) -> pd.DataFrame:
    """1/N over the universe each month (DeMiguel et al., 2009)."""
    return universe[["id", "eom"]].assign(w=(1.0 / universe.groupby("eom")["id"].transform("size")).to_numpy())


def min_variance_targets(universe: pd.DataFrame, cov_parts: Mapping[pd.Timestamp, CovParts]) -> pd.DataFrame:
    """Ex-ante minimum-variance weights Σ⁻¹1 / (1ᵀΣ⁻¹1), via a Cholesky solve (no explicit inverse)."""
    out = []
    for eom, g in universe.groupby("eom", sort=True):
        ids = g["id"].to_numpy()
        x = cho_solve(cho_factor(cov_parts[eom].full(ids)), np.ones(len(ids)))
        out.append(pd.DataFrame({"id": ids, "eom": eom, "w": x / x.sum()}))
    return pd.concat(out, ignore_index=True)


def run_benchmarks(panel: pd.DataFrame, wealth: pd.DataFrame, cov_parts: Mapping[pd.Timestamp, CovParts],
                   dates: Sequence[pd.Timestamp], charge_exit_tc: bool) -> tuple[pd.DataFrame, pd.DataFrame]:
    """Weights and monthly performance (``type`` column) for the three benchmarks over ``dates``."""
    dates = pd.DatetimeIndex(sorted(dates))
    missing = [d for d in dates if d not in cov_parts]
    if missing:
        raise ValueError(f"no covariance for {len(missing)} dates, e.g. {missing[0].date()}")
    universe = panel.loc[panel["in_universe"] & panel["eom"].isin(dates)]
    targets = {
        "Market": market_targets(universe),
        "1/N": equal_weight_targets(universe),
        "Minimum variance": min_variance_targets(universe, cov_parts),
    }
    weights, pfs = [], []
    for name, tg in targets.items():
        w = add_start_weights(tg, panel, wealth, dates, charge_exit_tc)
        weights.append(w.assign(type=name))
        pfs.append(portfolio_returns(w, panel, wealth).assign(type=name))
    return pd.concat(weights, ignore_index=True), pd.concat(pfs, ignore_index=True)


### `scripts/jkmp/portfolio/grids.py`

Shared date windows, wealth scaling and signal settings for Kati frontiers and iterations. Only helper functions used by the retained Static ML and Portfolio ML are included. Each frontier point is fitted by scripts/figure1.py.

**Role:** Calculation library. Other stages import its functions; this cell writes the module.

**Input:** the embedded source below. **Output:** `scripts/jkmp/portfolio/grids.py`. Expand the code cell to read or edit it.


In [ ]:
%%writefile scripts/jkmp/portfolio/grids.py
"""Shared date windows, wealth scaling and signal settings for Kati frontiers and iterations.

Only helper functions used by the retained Static ML and Portfolio ML are included.
Each frontier point is fitted by scripts/figure1.py.
"""

from __future__ import annotations

import logging
from collections.abc import Mapping, Sequence
from dataclasses import dataclass
from pathlib import Path

import pandas as pd

from jkmp.config import Config, DatesConfig
from jkmp.covariance import CovParts
from jkmp.evaluation.holdout import check_return_months, evaluation_window
from jkmp.evaluation.metrics import table2

log = logging.getLogger(__name__)

MONTH = pd.offsets.MonthEnd(1)


@dataclass
class GridInputs:
    """Everything a grid point needs, small enough to ship to joblib workers."""
    panel: pd.DataFrame            # stocks ever in the universe, with pred_ld1
    wealth: pd.DataFrame           # base wealth path (cfg.investor.wealth at the end of the sample)
    cov: Mapping[pd.Timestamp, CovParts]
    rf: pd.Series                  # French monthly RF by month-end
    pfml_dates: pd.DatetimeIndex   # Portfolio-ML formation months (first Σ + lb + 1 .. test end − 1)
    val_dates: pd.DatetimeIndex    # Forecast-history formation months (returns from the first prediction year)
    eval_dates: pd.DatetimeIndex   # test window formation months
    hp_years: list[int]


def grid_inputs(cfg: Config, panel: pd.DataFrame, wealth: pd.DataFrame, cov: Mapping[pd.Timestamp, CovParts],
                rf: pd.Series) -> GridInputs:
    """Date windows as in scripts 05 and 07; the panel cut to stocks that are ever in the universe
    (the only ones that are held, enter m and Σ̃, or are sold on exit)."""
    formation = pd.DatetimeIndex(sorted(cov))
    ret = formation + MONTH
    first_year = cfg.return_model.pred_years[0]
    test_start, test_end = evaluation_window(cfg.dates)
    lb = cfg.universe.lookback_months
    first = formation[0] + MONTH * (lb + 1)
    keep = panel["id"].isin(panel.loc[panel["in_universe"], "id"].unique())
    eval_dates = formation[(ret >= test_start) & (ret <= test_end)]
    val_dates = formation[(ret >= pd.Timestamp(first_year, 1, 31)) & (ret <= test_end)]
    check_return_months(eval_dates.union(val_dates) + MONTH, cfg.dates, "grid windows")
    cols = ["id", "eom", "me", "in_universe", "lambda", "g", "tr_ld1", "ret_ld1", "pred_ld1", *cfg.features.names]
    return GridInputs(
        panel=panel.loc[keep & (panel["eom"] >= formation[0]), cols].reset_index(drop=True),
        wealth=wealth, cov=cov, rf=rf,
        pfml_dates=formation[(formation >= first) & (ret <= test_end)],
        val_dates=val_dates, eval_dates=eval_dates,
        hp_years=list(range(first_year, test_end.year + 1)))


def point_key(gamma: float, wealth: float) -> str:
    return f"gamma{gamma:g}_wealth{wealth / 1e9:g}B"


def scaled_wealth(base: pd.DataFrame, level: float, base_level: float) -> pd.DataFrame:
    """Wealth path ending at ``level``: w_{t−1} = w_t/(1 + R_m) is linear in the end value."""
    return base.assign(wealth=base["wealth"] * (level / base_level))


def _signal_kwargs(cfg: Config, g: float | None) -> dict:
    """build_moments arguments for one signal set (RFF with variance g, or raw characteristics if g is None)."""
    pm = cfg.portfolio_ml
    if g is None:
        return dict(use_rff=False, rff_variance=None, p_max=None, seed=None, add_orig=False)
    return dict(use_rff=True, rff_variance=g, p_max=max(pm.p_grid), seed=cfg.seed, add_orig=pm.add_original_features)


### `scripts/jkmp/portfolio/portfolio_ml.py`

Portfolio-ML (Eqs. 35–37 and Proposition 3). Ports of ``pfml_input_fun``, ``pfml_search_coef``, ``pfml_hp_reals_fun``, ``pfml_aims_fun``, ``pfml_w`` and ``pfml_implement`` (0 - Portfolio choice functions.R). For each formation month t and the universe stocks at t (n of them), with P signals + a constant: signals S_s (n × P+1), s = t, t−1, …, t−lb−1: features of the *current* ids at month s (random Fourier features of the ranked characteristics, or the characteristics themselves), demeaned across the n stocks, a constant added, every column scal

**Role:** Calculation library. Other stages import its functions; this cell writes the module.

**Input:** the embedded source below. **Output:** `scripts/jkmp/portfolio/portfolio_ml.py`. Expand the code cell to read or edit it.


In [ ]:
%%writefile scripts/jkmp/portfolio/portfolio_ml.py
"""Portfolio-ML (Eqs. 35–37 and Proposition 3). Ports of ``pfml_input_fun``, ``pfml_search_coef``,
``pfml_hp_reals_fun``, ``pfml_aims_fun``, ``pfml_w`` and ``pfml_implement`` (0 - Portfolio choice functions.R).

For each formation month t and the universe stocks at t (n of them), with P signals + a constant:

  signals  S_s (n × P+1), s = t, t−1, …, t−lb−1: features of the *current* ids at month s
           (random Fourier features of the ranked characteristics, or the characteristics themselves),
           demeaned across the n stocks, a constant added, every column scaled to unit sum of squares,
           then rows multiplied by 1/vol (vol = sqrt(diag Σ_s); median of the month if missing).
  weights  G_s = m·diag(g_s), with m the trading speed at t and g_s = (1+tr_ld0)/(1+mu_ld0) at s;
           A_0 = I, A_{i+1} = A_i G_{t−i}   and   A¹_0 = I, A¹_{i+1} = A¹_i G_{t−i−1}.
  Ω_t    = (Σ_i A_i)⁻¹ Σ_i A_i S_{t−i},             i = 0..lb
  Ω_{t−1} = (Σ_i A¹_i)⁻¹ Σ_i A¹_i S_{t−i−1}
  ΔΩ     = Ω_t − diag(g_t) Ω_{t−1}
  r̃ = Ωᵀ r_{t+1},   risk = γ ΩᵀΣΩ (factor form of Σ),   tc = w ΔΩᵀ Λ ΔΩ,   Σ̃ = risk + tc.

Columns are ordered as R's ``pfml_feat_fun(p_max)``: constant, rff1_cos … rff{p/2}_cos,
rff1_sin … rff{p/2}_sin (then the original features if ``add_orig``), so the p-feature model is
``feature_index(p, p_max)``. ``use_rff=False`` uses the ranked characteristics as signals (linear
Portfolio-ML extension). One call handles one RFF variance g, as R runs ``pfml_input_fun`` per g.
"""

from __future__ import annotations

import json
import logging
import datetime as dt
from uuid import uuid4
from collections.abc import Callable, Mapping, Sequence
from dataclasses import dataclass, field
from pathlib import Path

import numpy as np
import pandas as pd

from jkmp.covariance import CovParts
from jkmp.models.random_features import base_draws, rff_raw, weights_for
from jkmp.portfolio.accounting import portfolio_returns
from jkmp.portfolio.recursion import MonthInputs, month_inputs, recursive_weights
from jkmp.portfolio.trading_speed import trading_speed

log = logging.getLogger(__name__)

MONTH = pd.offsets.MonthEnd(1)


@dataclass
class MonthMoments:
    eom: pd.Timestamp
    ids: np.ndarray        # (n,) universe ids at eom, sorted
    r_tilde: np.ndarray    # (P+1,)        Ωᵀ r
    denom: np.ndarray      # (P+1, P+1)    Σ̃ = risk + tc
    risk: np.ndarray | None  # (P+1, P+1)  γ ΩᵀΣΩ (None if loaded without it)
    signal: np.ndarray     # (n, P+1)      S_t, the signals that map coefficients to the aim portfolio

    @property
    def tc(self) -> np.ndarray:
        return self.denom - self.risk


@dataclass
class PfmlMoments:
    columns: list[str]
    meta: dict
    months: dict[pd.Timestamp, MonthMoments] = field(default_factory=dict)
    rff_w: np.ndarray | None = None   # (k, p_max/2) random weights W, None for raw characteristics


def signal_set(use_rff: bool, rff_variance_grid: Sequence[float]) -> tuple[str, list[str], str]:
    """(output folder, signal-set tags, method label) for ``portfolio_ml.use_rff``."""
    if use_rff:
        return "rff", [f"rff_g{g:.4f}" for g in rff_variance_grid], "Portfolio-ML"
    return "raw", ["raw"], "Portfolio-ML (linear)"


def signal_names(features: Sequence[str], p_max: int | None, add_orig: bool = False,
                 add_constant: bool = True) -> list[str]:
    """R ``pfml_feat_fun(p_max, orig_feat)``; ``p_max=None`` means the raw characteristics."""
    const = ["constant"] if add_constant else []
    if p_max is None:
        return [*const, *features]
    half = p_max // 2
    names = [*const, *[f"rff{j}_cos" for j in range(1, half + 1)], *[f"rff{j}_sin" for j in range(1, half + 1)]]
    return names + list(features) if add_orig else names


def feature_index(p: int, p_max: int, n_orig: int = 0, constant: bool = True) -> np.ndarray:
    """Columns of the p-feature model in the p_max layout: [constant], cos 1..p/2, sin 1..p/2 (+ originals)."""
    if p % 2 or p > p_max:
        raise ValueError(f"p must be even and <= {p_max}")
    half, half_max, c = p // 2, p_max // 2, int(constant)
    idx = [0] * c + [*range(c, c + half), *range(c + half_max, c + half_max + half)]
    return np.array(idx + list(range(c + p_max, c + p_max + n_orig)))


def scale_signals(raw: np.ndarray, vol: np.ndarray | None, add_constant: bool = True) -> np.ndarray:
    """Demean each column, prepend a constant, scale every column to unit sum of squares, divide rows by vol.

    R (``balanced = F``): ``x - mean(x)``, ``constant := 1``, ``x*sqrt(1/sum(x^2))``, ``diag(1/vol_scale) %*% s``.
    A column with no cross-sectional variation would be 0/0 in R; it is left at zero here. Without
    ``add_constant`` (``portfolio_ml.add_constant: false``) there is no constant column: after the 1/vol scaling it
    is an inverse-volatility long book, which a benchmark-relative active book should not hold (D34).
    """
    x = raw - raw.mean(axis=0)
    s = np.hstack([np.ones((len(raw), 1)), x]) if add_constant else x
    ss = (s**2).sum(axis=0)
    s = np.divide(s, np.sqrt(ss), out=np.zeros_like(s), where=ss > 0)
    return s if vol is None else s / vol[:, None]


def weighted_signals(signals: Sequence[np.ndarray], g: Sequence[np.ndarray], m: np.ndarray, lb: int
                     ) -> tuple[np.ndarray, np.ndarray]:
    """Ω_t and Ω_{t−1} from ``signals[i]`` = S_{t−i} and ``g[i]`` = g_{t−i}, i = 0..lb+1 (R's gtm loop)."""
    if len(signals) != lb + 2 or len(g) != lb + 2:
        raise ValueError(f"need lb + 2 = {lb + 2} months of signals and growth factors")
    n = m.shape[0]
    with np.errstate(all="ignore"):  # spurious Accelerate matmul flags; checked below
        gtm = [m * gi[None, :] for gi in g]                      # m %*% diag(gt)
        agg, agg_l1 = [np.eye(n)], [np.eye(n)]
        for i in range(lb):
            agg.append(agg[i] @ gtm[i])
            agg_l1.append(agg_l1[i] @ gtm[i + 1])
        omega = sum(a @ signals[i] for i, a in enumerate(agg))
        omega_l1 = sum(a @ signals[i + 1] for i, a in enumerate(agg_l1))
        omega = np.linalg.solve(sum(agg), omega)
        omega_l1 = np.linalg.solve(sum(agg_l1), omega_l1)
    if not (np.isfinite(omega).all() and np.isfinite(omega_l1).all()):
        raise FloatingPointError("non-finite weighted signals")
    return omega, omega_l1


def month_moments(omega: np.ndarray, omega_l1: np.ndarray, g_t: np.ndarray, r: np.ndarray, cov: CovParts,
                  ids: np.ndarray, lam: np.ndarray, wealth: float, gamma: float) -> tuple[np.ndarray, ...]:
    """r̃ = Ωᵀr, risk = γΩᵀΣΩ via Σ = XFXᵀ + diag(ivar), tc = wΔΩᵀΛΔΩ; returns (r̃, risk, tc)."""
    rows = np.searchsorted(cov.ids, ids)
    if (rows >= len(cov.ids)).any() or (cov.ids[np.minimum(rows, len(cov.ids) - 1)] != ids).any():
        raise KeyError(f"universe stocks without covariance loadings at {cov.eom.date()}")
    X, ivar = cov.X[rows], cov.ivar[rows]
    d_omega = omega - g_t[:, None] * omega_l1
    with np.errstate(all="ignore"):  # spurious Accelerate matmul flags; checked below
        r_tilde = omega.T @ r
        b = X.T @ omega
        risk = gamma * (b.T @ cov.F @ b + omega.T @ (ivar[:, None] * omega))
        tc = wealth * (d_omega.T @ (lam[:, None] * d_omega))
    out = r_tilde, 0.5 * (risk + risk.T), 0.5 * (tc + tc.T)  # symmetric up to round-off
    if not all(np.isfinite(a).all() for a in out):
        raise FloatingPointError("non-finite Portfolio-ML moments")
    return out


def build_moments(panel: pd.DataFrame, wealth: pd.DataFrame, cov_parts: Mapping[pd.Timestamp, CovParts],
                  rf: pd.Series, dates: Sequence[pd.Timestamp], features: Sequence[str], *, gamma: float,
                  mu: float, lb: int, iterations: int, scale_by_vol: bool = True, use_rff: bool = True,
                  rff_variance: float | None = None, p_max: int | None = None, seed: int | None = None,
                  add_orig: bool = False, add_constant: bool = True,
                  sink: Callable[[MonthMoments], None] | None = None) -> PfmlMoments:
    """R ``pfml_input_fun`` (``balanced = F``) for formation months ``dates``.

    With ``sink``, each month is passed to ``sink`` instead of being kept (``months`` stays empty), so
    callers can reduce months on the fly (``build_yearly_moments``).

    ``rf`` is Ken French's monthly RF indexed by month-end; m at t uses RF of the return month t+1
    (docs/deviations.md D20). Needs ``cov_parts`` for every month t−lb−1 .. t and panel rows for
    every universe stock at t in each of those months (guaranteed by ``valid_data``).
    """
    features = list(features)
    if use_rff:
        if rff_variance is None or p_max is None or seed is None:
            raise ValueError("use_rff needs rff_variance, p_max and seed")
        w_rff = weights_for(base_draws(len(features), p_max, seed), rff_variance)  # R: set.seed(seed); rff(p_max, g)
    else:
        w_rff, p_max, add_orig = None, None, False
    columns = signal_names(features, p_max, add_orig, add_constant)
    dates = pd.DatetimeIndex(sorted(dates))
    w_by = wealth.set_index("eom")["wealth"]
    lag_months = [pd.DatetimeIndex(dates - MONTH * i) for i in range(lb + 2)]

    # Raw signal inputs, once per (id, eom) needed: ids of the universe at t, months t−lb−1 .. t
    universe = panel.loc[panel["in_universe"] & panel["eom"].isin(dates), ["id", "eom"]]
    need = pd.concat([universe.assign(eom=universe["eom"] - MONTH * i) for i in range(lb + 2)]).drop_duplicates()
    rows = panel[["id", "eom", "g", "ret_ld1", "lambda", *features]].merge(need, on=["id", "eom"])
    from jkmp.data.lag_history import complete_rows
    rows = complete_rows(rows, need, features)
    x = rows[features].to_numpy(dtype=float)
    raw = rff_raw(x, w_rff) if use_rff else x
    if use_rff and add_orig:
        raw = np.hstack([raw, x])
    rows = rows[["id", "eom", "g", "ret_ld1", "lambda"]].reset_index(drop=True)
    by_eom = {eom: idx.to_numpy() for eom, idx in rows.groupby("eom").groups.items()}  # eom -> row positions
    row_id, row_g, row_ret, row_lam = (rows[c].to_numpy() for c in ("id", "g", "ret_ld1", "lambda"))
    vol_by = {}  # eom -> (vol Series over that month's Σ ids, median)

    def vol_for(eom: pd.Timestamp, ids: np.ndarray) -> np.ndarray | None:
        if not scale_by_vol:
            return None
        if eom not in vol_by:
            if eom not in cov_parts:
                raise KeyError(f"no covariance at {eom.date()} (needed for the vol scale)")
            c = cov_parts[eom]
            v = pd.Series(c.vol(), index=c.ids)
            vol_by[eom] = (v, float(v.median()))
        v, med = vol_by[eom]
        return v.reindex(ids).fillna(med).to_numpy()  # R: impute missing with the month's median

    out = PfmlMoments(columns=columns, rff_w=w_rff, meta={
        "use_rff": use_rff, "rff_variance": rff_variance, "p_max": p_max, "seed": seed, "add_orig": add_orig,
        "features": features, "gamma": gamma, "mu": mu, "lb": lb, "iterations": iterations,
        "scale_by_vol": scale_by_vol, "add_constant": add_constant})
    for j, t in enumerate(dates):
        ids = np.sort(universe.loc[universe["eom"] == t, "id"].to_numpy())
        signals, growth, now = [], [], None
        for i in range(lb + 2):
            s_month = lag_months[i][j]
            idx = by_eom.get(s_month, np.array([], dtype=int))
            pos = pd.Index(row_id[idx]).get_indexer(ids)
            if (pos < 0).any():
                raise ValueError(f"{(pos < 0).sum()} universe stocks at {t.date()} have no row at {s_month.date()}")
            r_idx = idx[pos]
            signals.append(scale_signals(raw[r_idx], vol_for(s_month, ids), add_constant))
            growth.append(row_g[r_idx])
            if i == 0:
                now = r_idx
        cov, w_t = cov_parts[t], float(w_by[t])
        lam = row_lam[now]
        m = trading_speed(cov.full(ids), lam, w_t, gamma, float(rf[t + MONTH]), mu, iterations)
        omega, omega_l1 = weighted_signals(signals, growth, m, lb)
        r_tilde, risk, tc = month_moments(omega, omega_l1, growth[0], row_ret[now], cov, ids, lam, w_t, gamma)
        mm = MonthMoments(t, ids, r_tilde, risk + tc, risk, signals[0])
        if sink is None:
            out.months[t] = mm
        else:
            sink(mm)
        if t.month == 12:
            log.info("PF-ML inputs %s: n=%d, P+1=%d", t.date(), len(ids), len(columns))
    return out


def save_moments(moments: PfmlMoments, directory: Path) -> None:
    """``directory/{meta.json, rff_w.npy, YYYY-MM-DD.npz}``; each npz holds ids, r_tilde, denom, risk, signal."""
    directory.mkdir(parents=True, exist_ok=True)
    old_files = list(directory.glob("*.npz"))
    if old_files:
        archive = directory / "previous_runs" / (dt.datetime.now().strftime("%Y%m%d_%H%M%S") + "_" + uuid4().hex[:8])
        # Preserve existing caches; never delete or replace them when rebuilding.
        if not archive.resolve().is_relative_to(directory.resolve()):
            raise ValueError("Cache archive must stay inside its moment directory")
        archive.mkdir(parents=True)
        previous_metadata = [directory / name for name in ("meta.json", "rff_w.npy") if (directory / name).exists()]
        for old in [*old_files, *previous_metadata]:
            old.rename(archive / old.name)
    meta = {**moments.meta, "columns": moments.columns, "dates": [f"{d:%Y-%m-%d}" for d in sorted(moments.months)]}
    (directory / "meta.json").write_text(json.dumps(meta, indent=1))
    if moments.rff_w is not None:
        np.save(directory / "rff_w.npy", moments.rff_w)
    for eom, mm in moments.months.items():
        np.savez(directory / f"{eom:%Y-%m-%d}.npz", ids=mm.ids, r_tilde=mm.r_tilde, denom=mm.denom,
                 risk=mm.risk, signal=mm.signal)


def load_moments(directory: Path, with_risk: bool = True) -> PfmlMoments:
    """Inverse of ``save_moments``; ``with_risk=False`` skips the risk matrices (not needed for the search)."""
    meta = json.loads((directory / "meta.json").read_text())
    columns, dates = meta.pop("columns"), meta.pop("dates")
    w_path = directory / "rff_w.npy"
    out = PfmlMoments(columns=columns, meta=meta, rff_w=np.load(w_path) if w_path.exists() else None)
    for d in dates:
        eom = pd.Timestamp(d)
        with np.load(directory / f"{d}.npz") as z:
            risk = z["risk"] if with_risk else None
            out.months[eom] = MonthMoments(eom, z["ids"], z["r_tilde"], z["denom"], risk, z["signal"])
    return out


# --- Coefficients, validation and aim portfolios (pfml_search_coef, pfml_hp_reals_fun, pfml_aims_fun) ---

def return_year(eom: pd.Timestamp) -> int:
    """Year in which the return of formation month ``eom`` is realised (our year convention)."""
    return (eom + MONTH).year


def model_columns(moments, p: int) -> np.ndarray:
    """Columns of the p-feature model; raw characteristics always use every column."""
    meta = moments.meta
    if not meta["use_rff"]:
        return np.arange(len(moments.columns))
    return feature_index(p, meta["p_max"], len(meta["features"]) if meta["add_orig"] else 0,
                         meta.get("add_constant", True))


def p_values(moments, p_grid: Sequence[int]) -> list[int]:
    """The p grid for RFF signals; for raw characteristics the single model with all of them."""
    return [int(p) for p in p_grid] if moments.meta["use_rff"] else [len(moments.meta["features"])]


def ridge_coefs(denom_mean: np.ndarray, r_mean: np.ndarray, lambdas: Sequence[float]) -> np.ndarray:
    """β(λ) = (Σ̄ + λI)⁻¹ r̄ for every λ (columns) from one eigendecomposition: V diag(1/(d + λ)) Vᵀ r̄.

    R ``solve(denom_sub + l*diag(p+1)) %*% r_tilde_sub``: the constant is penalised like every signal.
    """
    lam = np.asarray(lambdas, dtype=float)
    d, V = np.linalg.eigh(0.5 * (denom_mean + denom_mean.T))
    with np.errstate(all="ignore"):  # spurious Accelerate matmul flags; checked below
        coef = V @ ((V.T @ r_mean)[:, None] / (d[:, None] + lam[None, :]))
    if not np.isfinite(coef).all():
        raise FloatingPointError("non-finite Portfolio-ML coefficients (singular Σ̄ at λ = 0?)")
    return coef


@dataclass
class YearSums:
    """Sums over the months of one return year: count, Σ r̃_t and Σ Σ̃_t."""
    n: int
    r_sum: np.ndarray
    d_sum: np.ndarray


def yearly_sums(months: Mapping[pd.Timestamp, MonthMoments]) -> dict[int, YearSums]:
    out: dict[int, YearSums] = {}
    for d in sorted(months):
        mm, y = months[d], return_year(d)
        if y not in out:
            out[y] = YearSums(0, np.zeros_like(mm.r_tilde), np.zeros_like(mm.denom))
        out[y].n += 1
        out[y].r_sum += mm.r_tilde
        out[y].d_sum += mm.denom
    return out


def coefs_from_sums(sums: Mapping[int, YearSums], owner, p_grid: Sequence[int], lambdas: Sequence[float],
                    hp_years: Sequence[int], half_life: float | None = None) -> dict[int, dict[int, np.ndarray]]:
    """β for each hp year y from the running means of r̃ and Σ̃ over return years ≤ y−1.
    ``owner`` supplies ``meta`` and ``columns`` (a PfmlMoments or YearlyMoments).

    ``half_life`` h (years, D36): year s gets weight δ^{y−s}, δ = 0.5^{1/h}, so r̄_y = Σ δ^{y−s} r_sum_s / Σ δ^{y−s} n_s.
    The accumulators hold Σ δ^{−(s−s₀)} x_s (the common factor δ^{y−s₀} cancels in the ratio); ``None`` gives weight
    1.0, i.e. exactly the equally weighted running means of R ``pfml_search_coef``."""
    n_cols = len(owner.columns)
    r_sum, d_sum, n = np.zeros(n_cols), np.zeros((n_cols, n_cols)), 0.0
    years, k, out = sorted(sums), 0, {}
    for y in sorted(hp_years):
        while k < len(years) and years[k] <= y - 1:
            ys = sums[years[k]]
            wt = 1.0 if half_life is None else 0.5 ** (-(years[k] - years[0]) / half_life)
            if wt == 1.0:
                r_sum, d_sum, n = r_sum + ys.r_sum, d_sum + ys.d_sum, n + ys.n
            else:
                r_sum, d_sum, n = r_sum + wt * ys.r_sum, d_sum + wt * ys.d_sum, n + wt * ys.n
            k += 1
        if n == 0:
            raise ValueError(f"no Portfolio-ML training months before return year {y}")
        r_bar, d_bar = r_sum / n, d_sum / n
        out[y] = {}
        for p in p_values(owner, p_grid):
            c = model_columns(owner, p)
            out[y][p] = ridge_coefs(d_bar[np.ix_(c, c)], r_bar[c], lambdas)
        log.debug("PF-ML coefficients for %d: %d training months", y, n)
    return out


def search_coef(moments: PfmlMoments, p_grid: Sequence[int], lambdas: Sequence[float],
                hp_years: Sequence[int]) -> dict[int, dict[int, np.ndarray]]:
    """R ``pfml_search_coef``: ``{year: {p: β (P_p × L)}}``, β for hp year y from the means of r̃ and Σ̃
    over every month whose return is realised by December y−1 (running sums, updated year by year)."""
    return coefs_from_sums(yearly_sums(moments.months), moments, p_grid, lambdas, hp_years)


def hp_reals(moments: PfmlMoments, coefs: Mapping[int, Mapping[int, np.ndarray]], lambdas: Sequence[float]
             ) -> pd.DataFrame:
    """R ``pfml_hp_reals_fun``: realised utility r̃ᵀβ − ½βᵀΣ̃β of each (p, λ) in each month of hp year y
    (using the β of year y), its cumulative mean since the first hp year, and its rank by return month."""
    lam = np.asarray(lambdas, dtype=float)
    frames = []
    for y, by_p in coefs.items():
        months = [d for d in sorted(moments.months) if return_year(d) == y]
        for p, B in by_p.items():
            c = model_columns(moments, p)
            for d in months:
                mm = moments.months[d]
                with np.errstate(all="ignore"):  # spurious Accelerate matmul flags; checked below
                    obj = mm.r_tilde[c] @ B - 0.5 * np.einsum("kl,kl->l", B, mm.denom[np.ix_(c, c)] @ B)
                if not np.isfinite(obj).all():
                    raise FloatingPointError(f"non-finite realised utility at {d.date()}")
                frames.append(pd.DataFrame({"hp_end": y, "eom": d, "eom_ret": d + MONTH, "p": p,
                                            "l_no": np.arange(len(lam)), "l": lam, "obj": obj}))
    v = pd.concat(frames, ignore_index=True).sort_values(["p", "l_no", "eom_ret"], ignore_index=True)
    v["cum_obj"] = v.groupby(["p", "l_no"])["obj"].transform(lambda s: s.expanding().mean())
    v["rank"] = v.groupby("eom_ret")["cum_obj"].rank(ascending=False, method="average")  # R frank(-cum_obj)
    return v


@dataclass
class SearchResult:
    """One signal set (one RFF variance g, or the raw characteristics) after the coefficient search."""
    tag: str
    g: float
    columns: list[str]
    meta: dict
    coefs: dict[int, dict[int, np.ndarray]]
    validation: pd.DataFrame
    signals: dict[pd.Timestamp, tuple[np.ndarray, np.ndarray]]  # eom -> (ids, S_t) for the aim months
    coefs_by_h: dict[float, dict[int, dict[int, np.ndarray]]] | None = None  # half-life (inf = none) -> coefs (D36)


def search(moments: PfmlMoments, tag: str, p_grid: Sequence[int], lambdas: Sequence[float],
           hp_years: Sequence[int], aim_dates: Sequence[pd.Timestamp]) -> SearchResult:
    """Coefficients and validation for one signal set; keeps only the signals needed for the aims."""
    coefs = search_coef(moments, p_grid, lambdas, hp_years)
    g = moments.meta["rff_variance"] if moments.meta["use_rff"] else float("nan")
    validation = hp_reals(moments, coefs, lambdas).assign(tag=tag, g=g)
    signals = {d: (moments.months[d].ids, moments.months[d].signal) for d in aim_dates}
    return SearchResult(tag, g, moments.columns, moments.meta, coefs, validation, signals)


def december_utility(sums: Mapping[int, YearSums], owner, coefs: Mapping[int, Mapping[int, np.ndarray]],
                     lambdas: Sequence[float]) -> pd.DataFrame:
    """``hp_reals`` at the December return months only, from yearly sums. β is fixed within an hp year,
    so the year's total utility is Σr̃ᵀβ − ½βᵀ(ΣΣ̃)β and the December ``cum_obj`` (all R's selection
    uses) is exact: the running total over hp years divided by the running month count."""
    lam = np.asarray(lambdas, dtype=float)
    frames = []
    for y in sorted(coefs):
        ys = sums[y]
        for p, B in coefs[y].items():
            c = model_columns(owner, p)
            with np.errstate(all="ignore"):  # spurious Accelerate matmul flags; checked below
                total = ys.r_sum[c] @ B - 0.5 * np.einsum("kl,kl->l", B, ys.d_sum[np.ix_(c, c)] @ B)
            if not np.isfinite(total).all():
                raise FloatingPointError(f"non-finite realised utility in {y}")
            frames.append(pd.DataFrame({"hp_end": y, "eom_ret": pd.Timestamp(y, 12, 31), "p": p, "n": ys.n,
                                        "l_no": np.arange(len(lam)), "l": lam, "total": total}))
    v = pd.concat(frames, ignore_index=True).sort_values(["p", "l_no", "eom_ret"], ignore_index=True)
    grp = v.groupby(["p", "l_no"])
    v["obj"] = v["total"] / v["n"]
    v["cum_obj"] = grp["total"].cumsum() / grp["n"].cumsum()
    v["rank"] = v.groupby("eom_ret")["cum_obj"].rank(ascending=False, method="average")
    return v.drop(columns="total")


@dataclass
class YearlyMoments:
    """Compact Portfolio-ML inputs: yearly sums of r̃ and Σ̃ plus the signals of selected months."""
    columns: list[str]
    meta: dict
    years: dict[int, YearSums]
    signals: dict[pd.Timestamp, tuple[np.ndarray, np.ndarray]]
    rff_w: np.ndarray | None = None


def build_yearly_moments(panel: pd.DataFrame, wealth: pd.DataFrame, cov_parts: Mapping[pd.Timestamp, CovParts],
                         rf: pd.Series, dates: Sequence[pd.Timestamp], features: Sequence[str],
                         signal_dates: Sequence[pd.Timestamp], **kwargs) -> YearlyMoments:
    """``build_moments`` reduced on the fly to ``yearly_sums`` (one month in memory at a time)."""
    keep = set(pd.DatetimeIndex(signal_dates))
    years: dict[int, YearSums] = {}
    signals: dict[pd.Timestamp, tuple[np.ndarray, np.ndarray]] = {}

    def sink(mm: MonthMoments) -> None:
        y = return_year(mm.eom)
        if y not in years:
            years[y] = YearSums(0, np.zeros_like(mm.r_tilde), np.zeros_like(mm.denom))
        years[y].n += 1
        years[y].r_sum += mm.r_tilde
        years[y].d_sum += mm.denom
        if mm.eom in keep:
            signals[mm.eom] = (mm.ids, mm.signal)

    shell = build_moments(panel, wealth, cov_parts, rf, dates, features, sink=sink, **kwargs)
    missing = keep - set(signals)
    if missing:
        raise ValueError(f"{len(missing)} signal months are outside the formation months, e.g. {min(missing).date()}")
    return YearlyMoments(shell.columns, shell.meta, years, signals, shell.rff_w)


def save_yearly_moments(ym: YearlyMoments, path: Path) -> None:
    years = sorted(ym.years)
    dates = sorted(ym.signals)
    np.savez(path, meta=json.dumps({**ym.meta, "columns": ym.columns}), years=np.array(years),
             n=np.array([ym.years[y].n for y in years]), r_sum=np.stack([ym.years[y].r_sum for y in years]),
             d_sum=np.stack([ym.years[y].d_sum for y in years]),
             signal_dates=np.array([f"{d:%Y-%m-%d}" for d in dates]),
             counts=np.array([len(ym.signals[d][0]) for d in dates]),
             ids=np.concatenate([ym.signals[d][0] for d in dates]),
             signal=np.concatenate([ym.signals[d][1] for d in dates]),
             rff_w=ym.rff_w if ym.rff_w is not None else np.zeros((0, 0)))


def load_yearly_moments(path: Path) -> YearlyMoments:
    with np.load(path) as z:
        meta = json.loads(str(z["meta"]))
        columns = meta.pop("columns")
        years = {int(y): YearSums(int(n), r, d) for y, n, r, d in zip(z["years"], z["n"], z["r_sum"], z["d_sum"])}
        bounds = np.concatenate([[0], np.cumsum(z["counts"])])
        ids, signal = z["ids"], z["signal"]  # each z[...] access re-reads the file
        signals = {pd.Timestamp(str(d)): (ids[a:b], signal[a:b])
                   for d, a, b in zip(z["signal_dates"], bounds[:-1], bounds[1:])}
        rff_w = z["rff_w"] if z["rff_w"].size else None
    return YearlyMoments(columns, meta, years, signals, rff_w)


def search_yearly(ym: YearlyMoments, tag: str, p_grid: Sequence[int], lambdas: Sequence[float],
                  hp_years: Sequence[int], half_lives: Sequence[float | None] = (None,)) -> SearchResult:
    """``search`` from yearly sums: same β and the same December selection, without monthly matrices.
    With several ``half_lives`` (D36) every h gets its own β; the validation rows carry ``h`` (inf = no decay) and
    are scored by the unchanged equally weighted cumulative utility, so h is picked each December like (g, p, λ)."""
    g = ym.meta["rff_variance"] if ym.meta["use_rff"] else float("nan")
    by_h, frames = {}, []
    for h in half_lives:
        key = np.inf if h is None else float(h)
        by_h[key] = coefs_from_sums(ym.years, ym, p_grid, lambdas, hp_years, half_life=h)
        frames.append(december_utility(ym.years, ym, by_h[key], lambdas).assign(h=key))
    validation = pd.concat(frames, ignore_index=True).assign(tag=tag, g=g)
    main = by_h.get(np.inf, next(iter(by_h.values())))
    return SearchResult(tag, g, ym.columns, ym.meta, main, validation, ym.signals, coefs_by_h=by_h)


def select_hypers(validation: pd.DataFrame) -> pd.DataFrame:
    """R ``pfml_implement``: across every (g, p, λ), the highest cumulative realised utility at each
    December return month; that (g, p, λ) is used in the next return year (column ``year``)."""
    v = validation.reset_index(drop=True)
    if "h" not in v:
        v = v.assign(h=np.inf)  # no half-life search: equal weights
    dec = v[v["eom_ret"].dt.month == 12]
    best = dec.loc[dec.groupby("eom_ret")["cum_obj"].idxmax()]
    best = best.assign(year=best["eom_ret"].dt.year + 1)
    return best[["year", "eom_ret", "tag", "g", "p", "l_no", "l", "h", "cum_obj"]].reset_index(drop=True)


def aim_portfolios(results: Mapping[str, SearchResult], chosen: pd.DataFrame, dates: Sequence[pd.Timestamp]
                   ) -> tuple[pd.DataFrame, pd.DataFrame]:
    """R ``pfml_aims_fun``: aim weights S_t β for formation months ``dates``, with the (g, p, λ) chosen in
    December y−1 and the β of hp year y (trained on returns through y−1), y = return year of t.
    Returns ``aims`` (id, eom, w_aim) and ``coefs`` (eom, year, tag, p, l, column, beta)."""
    by_year = chosen.set_index("year")
    aims, coefs = [], []
    for d in sorted(dates):
        y = return_year(d)
        if y not in by_year.index:
            raise ValueError(f"no Portfolio-ML hyperparameters chosen for return year {y}")
        h = by_year.loc[y]
        res = results[h["tag"]]
        p, l_no = int(h["p"]), int(h["l_no"])
        c = model_columns(res, p)
        res_coefs = res.coefs if res.coefs_by_h is None else res.coefs_by_h[float(h.get("h", np.inf))]
        beta = res_coefs[y][p][:, l_no]
        ids, signal = res.signals[d]
        with np.errstate(all="ignore"):  # spurious Accelerate matmul flags; checked below
            w_aim = signal[:, c] @ beta
        if not np.isfinite(w_aim).all():
            raise FloatingPointError(f"non-finite aim portfolio at {d.date()}")
        aims.append(pd.DataFrame({"id": ids, "eom": d, "w_aim": w_aim}))
        coefs.append(pd.DataFrame({"eom": d, "year": y, "tag": h["tag"], "p": p, "l": h["l"],
                                   "column": [res.columns[i] for i in c], "beta": beta}))
    return pd.concat(aims, ignore_index=True), pd.concat(coefs, ignore_index=True)


# --- Weights and implementation (pfml_w, pfml_implement) -----------------------------------------

def pfml_weights(inputs: Mapping[pd.Timestamp, MonthInputs], aims: pd.DataFrame, panel: pd.DataFrame,
                 wealth: pd.DataFrame, rf: pd.Series, *, gamma: float, mu: float, iterations: int,
                 charge_exit_tc: bool, benchmark_relative: bool = False) -> pd.DataFrame:
    """R ``pfml_w``: π_t = m·π_start + (I − m)·aim_t, with m (Lemma 1) at t for the universe and π_start
    the VW universe in the first month, then the drifted π_{t−1} (new stocks at 0; ``accounting``).

    ``benchmark_relative`` (D34): the aim is an *active* book A_t and the recursion runs on active weights,
    a_t = m·a_inh + (I − m)·A_t with a_inh = π_start − π^VW_t, then π_t = π^VW_t + a_t (π^VW_t: value weights of
    the universe at t, ``benchmarks.market_targets``). Defining a_inh against today's value weights makes the
    traded Δπ = π_t − π_start equal Δa exactly, so the cost charged on the total portfolio is the cost of the
    active trades. The approximation is in the objective only: its ΔΩ = Ω_t − diag(g)Ω_{t−1} lets the active
    book drift with g and ignores the benchmark's own rebalancing (π^VW_t vs the drifted π^VW_{t−1}: universe
    changes and issuance), which is small — the Market row's turnover and cost measure it.

    m uses RF of the return month t+1 (docs/deviations.md D20). Returns ``id, eom, w, w_start, exit``.
    """
    aim_by = {eom: g.set_index("id")["w_aim"] for eom, g in aims.groupby("eom")}

    def target(eom: pd.Timestamp, x: MonthInputs, held: np.ndarray) -> np.ndarray:
        if eom not in aim_by:
            raise ValueError(f"no aim portfolio at {eom.date()}")
        aim = aim_by[eom].reindex(x.ids)
        if aim.isna().any():
            raise ValueError(f"{aim.isna().sum()} universe stocks lack an aim weight at {eom.date()}")
        aim = aim.to_numpy()
        m = trading_speed(x.sigma, x.lam, x.wealth, gamma, float(rf[eom + MONTH]), mu, iterations)
        vw = np.zeros_like(held)
        if benchmark_relative:
            me = x.panel_t.set_index("id")["me"].reindex(x.ids).to_numpy()
            vw = me / me.sum()
        with np.errstate(all="ignore"):  # spurious Accelerate matmul flags; checked in recursive_weights
            return vw + m @ (held - vw) + aim - m @ aim

    return recursive_weights(inputs, panel, wealth, target, charge_exit_tc)


def run_portfolio_ml(results: Mapping[str, SearchResult], panel: pd.DataFrame, wealth: pd.DataFrame,
                     cov_parts: Mapping[pd.Timestamp, CovParts], rf: pd.Series, aim_dates: Sequence[pd.Timestamp],
                     eval_dates: Sequence[pd.Timestamp], *, gamma: float, mu: float, iterations: int,
                     charge_exit_tc: bool, label: str = "Portfolio-ML", benchmark_relative: bool = False
                     ) -> dict[str, pd.DataFrame]:
    """R ``pfml_implement`` after the per-g search: pick (g, p, λ) each December across all g, build the
    aim portfolios for ``aim_dates`` and run the weights over ``eval_dates`` from VW initial weights."""
    validation = pd.concat([r.validation for r in results.values()], ignore_index=True)
    chosen = select_hypers(validation)
    years = {return_year(d) for d in aim_dates}
    aims, coefs = aim_portfolios(results, chosen[chosen["year"].isin(years)], aim_dates)
    eval_dates = pd.DatetimeIndex(sorted(eval_dates))
    inputs = month_inputs(panel, wealth, cov_parts, eval_dates, mu_col=None)
    weights = pfml_weights(inputs, aims, panel, wealth, rf, gamma=gamma, mu=mu, iterations=iterations,
                           charge_exit_tc=charge_exit_tc, benchmark_relative=benchmark_relative)
    pf = portfolio_returns(weights, panel, wealth).assign(type=label)
    log.info("%s: chose\n%s", label, chosen.to_string(index=False))
    return {"validation": validation, "chosen": chosen, "aims": aims, "aim_coefs": coefs,
            "weights": weights.assign(type=label), "pf": pf}


### `scripts/jkmp/portfolio/recursion.py`

Month-by-month weights for methods whose target depends on the inherited weights (Static-ML, Portfolio-ML). R ``static_val_fun`` and ``pfml_w`` share this loop: start from value weights in the first month, then from last month's weights drifted by (1 + tr_ld1)/(1 + mu_ld1), new stocks at 0 (``accounting``); the method maps (month, inherited weights of the universe) to new weights.

**Role:** Calculation library. Other stages import its functions; this cell writes the module.

**Input:** the embedded source below. **Output:** `scripts/jkmp/portfolio/recursion.py`. Expand the code cell to read or edit it.


In [ ]:
%%writefile scripts/jkmp/portfolio/recursion.py
"""Month-by-month weights for methods whose target depends on the inherited weights (Static-ML, Portfolio-ML).

R ``static_val_fun`` and ``pfml_w`` share this loop: start from value weights in the first month, then
from last month's weights drifted by (1 + tr_ld1)/(1 + mu_ld1), new stocks at 0 (``accounting``); the
method maps (month, inherited weights of the universe) to new weights.
"""

from __future__ import annotations

from collections.abc import Callable, Mapping, Sequence
from dataclasses import dataclass

import numpy as np
import pandas as pd

from jkmp.covariance import CovParts
from jkmp.portfolio.accounting import drift_weights, start_weights_for


@dataclass
class MonthInputs:
    ids: np.ndarray        # universe ids, sorted
    sigma: np.ndarray      # monthly Σ for ids
    lam: np.ndarray        # Λ diagonal for ids
    mu: np.ndarray | None  # pred_ld1 for ids (None if not requested)
    wealth: float
    panel_t: pd.DataFrame  # id, me at t (for the VW start and exit detection)


def month_inputs(panel: pd.DataFrame, wealth: pd.DataFrame, cov_parts: Mapping[pd.Timestamp, CovParts],
                 dates: Sequence[pd.Timestamp], mu_col: str | None = "pred_ld1") -> dict[pd.Timestamp, MonthInputs]:
    """Per-month arrays for the universe, built once and shared by every run over the same months."""
    w_by = wealth.set_index("eom")["wealth"]
    p = panel[panel["eom"].isin(dates)]
    out = {}
    for eom, g in p.groupby("eom", sort=True):
        u = g[g["in_universe"]].sort_values("id")
        if mu_col is not None and u[mu_col].isna().any():
            raise ValueError(f"missing {mu_col} in the universe at {eom.date()}")
        ids = u["id"].to_numpy()
        out[eom] = MonthInputs(ids, cov_parts[eom].full(ids), u["lambda"].to_numpy(),
                               None if mu_col is None else u[mu_col].to_numpy(), float(w_by[eom]), g[["id", "me"]])
    return out


Target = Callable[[pd.Timestamp, MonthInputs, np.ndarray], np.ndarray]


def recursive_weights(inputs: Mapping[pd.Timestamp, MonthInputs], panel: pd.DataFrame, wealth: pd.DataFrame,
                      target: Target, charge_exit_tc: bool) -> pd.DataFrame:
    """Run ``target(eom, inputs, w_start of the universe) -> w`` over the months in ``inputs`` (consecutive).

    Returns ``id, eom, w, w_start, exit`` (exit rows only with ``charge_exit_tc``), ready for
    ``accounting.portfolio_returns``.
    """
    rows, previous_ids, previous_start = [], None, None
    # Index once: repeated counterfactual/grid runs must not rejoin the full panel
    # for every individual month. Initial weights, drift and exit rules match accounting.py.
    panel_returns = {date: group.set_index("id")["tr_ld1"] for date, group in
                     panel.loc[panel["eom"].isin(inputs), ["id","eom","tr_ld1"]].groupby("eom", sort=False)}
    market = wealth.set_index("eom")["mu_ld1"]
    for eom in sorted(inputs):
        x = inputs[eom]
        exit_ids, exit_start = np.array([],dtype=x.ids.dtype), np.array([],dtype=float)
        if previous_ids is None:
            capital=x.panel_t.set_index("id")["me"].reindex(x.ids).to_numpy()
            held=capital/capital.sum()
        else:
            positions=pd.Index(previous_ids).get_indexer(x.ids)
            held=np.zeros(len(x.ids))
            found=positions>=0
            held[found]=previous_start[positions[found]]
            if charge_exit_tc:
                leaving=(~pd.Index(previous_ids).isin(x.ids)) & (previous_start!=0.0)
                leaving &= pd.Index(previous_ids).isin(x.panel_t["id"])
                exit_ids,exit_start=previous_ids[leaving],previous_start[leaving]
        w = target(eom, x, held)
        if w.shape != held.shape or not np.isfinite(w).all():
            raise FloatingPointError(f"bad target weights at {eom.date()}")
        month=pd.DataFrame({"id":np.concatenate([x.ids,exit_ids]),"eom":eom,
            "w":np.concatenate([w,np.zeros(len(exit_ids))]),
            "w_start":np.concatenate([held,exit_start]),
            "exit":np.concatenate([np.zeros(len(x.ids),dtype=bool),np.ones(len(exit_ids),dtype=bool)])})
        rows.append(month)
        nonzero=w!=0.0
        previous_ids=x.ids[nonzero]
        if nonzero.any():
            returns=panel_returns[eom].reindex(previous_ids).to_numpy()
            growth=market.get(eom,np.nan)
            if pd.isna(returns).any() or pd.isna(growth):
                raise ValueError(f"Held positions lack tr_ld1/mu_ld1 at {eom.date()}")
            previous_start=w[nonzero]*(1.0+returns)/(1.0+growth)
        else:
            previous_start=np.array([],dtype=float)
    return pd.concat(rows, ignore_index=True)[["id", "eom", "w", "w_start", "exit"]]


### `scripts/jkmp/portfolio/static_ml.py`

Original Static-ML, with fixed cost/shrinkage/risk multipliers (1,1,0). The return forecaster is annually refitted. Portfolio weights are updated monthly. The portfolio recursion begins in 2015 and continues into the 2020-2025 test, preserving the original non-starred strategy's inherited weights.

**Role:** Calculation library. Other stages import its functions; this cell writes the module.

**Input:** the embedded source below. **Output:** `scripts/jkmp/portfolio/static_ml.py`. Expand the code cell to read or edit it.


In [ ]:
%%writefile scripts/jkmp/portfolio/static_ml.py
"""Original Static-ML, with fixed cost/shrinkage/risk multipliers (1,1,0).

The return forecaster is annually refitted. Portfolio weights are updated monthly.
The portfolio recursion begins in 2015 and continues into the 2020-2025 test,
preserving the original non-starred strategy's inherited weights.
"""
from collections.abc import Mapping,Sequence
import numpy as np
import pandas as pd
from scipy.linalg import cho_factor,cho_solve
from jkmp.covariance import CovParts
from jkmp.portfolio.accounting import portfolio_returns
from jkmp.portfolio.recursion import month_inputs,recursive_weights


def run_static_ml(panel: pd.DataFrame,wealth: pd.DataFrame,cov_parts: Mapping[pd.Timestamp,CovParts],
                  val_dates: Sequence[pd.Timestamp],eval_dates: Sequence[pd.Timestamp],gamma: float,
                  charge_exit_tc: bool) -> dict[str,pd.DataFrame]:
    history=pd.DatetimeIndex(sorted(val_dates));evaluation=pd.DatetimeIndex(sorted(eval_dates))
    if not set(evaluation).issubset(history):raise ValueError("Static evaluation must be within its continuous history")
    inputs=month_inputs(panel,wealth,cov_parts,history)
    ids=np.unique(np.concatenate([x.ids for x in inputs.values()]))
    panel=panel.loc[panel.id.isin(ids)&panel.eom.isin(history)].reset_index(drop=True)
    def target(date,x,held):
        cost=x.wealth*x.lam
        return cho_solve(cho_factor(gamma*x.sigma+np.diag(cost)),x.mu+cost*held)
    weights=recursive_weights(inputs,panel,wealth,target,charge_exit_tc).assign(type="Static-ML")
    oos=portfolio_returns(weights,panel,wealth).assign(type="Static-ML")
    return dict(pf=oos.loc[oos.eom.isin(evaluation)].reset_index(drop=True),
                weights=weights.loc[weights.eom.isin(evaluation)].reset_index(drop=True),
                pf_oos=oos,weights_oos=weights)


### `scripts/jkmp/portfolio/static_ml_star.py`

US high-cap base Static-ML* (Eq. 34 with the Eq. 41 tuning layer). Port of ``static_val_fun``, ``static_implement`` (0 - Portfolio choice functions.R) and ``sigma_gam_adj`` (0 - General functions.R). Each month, given inherited weights w_start (VW in the first month, then drifted): w_t = (γΣ* + W·kΛ)⁻¹ (u·μ_t + W·kΛ·w_start), γΣ* = γΣ + g·diag(γΣ) (``cov_add``) u shrinks the predicted returns ``pred_ld1``, g inflates the variances, k = 1/φ scales trading costs. Static-ML is (k, u, g) = (1, 1, 0); Static-ML* picks (k, u, g) from the grid each De

**Role:** Calculation library. Other stages import its functions; this cell writes the module.

**Input:** the embedded source below. **Output:** `scripts/jkmp/portfolio/static_ml_star.py`. Expand the code cell to read or edit it.


In [ ]:
%%writefile scripts/jkmp/portfolio/static_ml_star.py
"""US high-cap base Static-ML* (Eq. 34 with the Eq. 41 tuning layer). Port of ``static_val_fun``,
``static_implement`` (0 - Portfolio choice functions.R) and ``sigma_gam_adj`` (0 - General functions.R).

Each month, given inherited weights w_start (VW in the first month, then drifted):

    w_t = (γΣ* + W·kΛ)⁻¹ (u·μ_t + W·kΛ·w_start),     γΣ* = γΣ + g·diag(γΣ)   (``cov_add``)

u shrinks the predicted returns ``pred_ld1``, g inflates the variances, k = 1/φ scales trading costs.
Static-ML is (k, u, g) = (1, 1, 0); Static-ML* picks (k, u, g) from the grid each December by the
highest cumulative realised utility since the validation start and uses it for the next return year.
"""

from __future__ import annotations

import itertools
import hashlib
import json
import logging
from collections.abc import Callable, Mapping, Sequence

import numpy as np
import pandas as pd
from scipy.linalg import cho_factor, cho_solve

from jkmp.covariance import CovParts
from jkmp.portfolio.accounting import portfolio_returns
from jkmp.portfolio.recursion import MonthInputs, month_inputs, recursive_weights

log = logging.getLogger(__name__)

MONTH = pd.offsets.MonthEnd(1)
Hyper = tuple[float, float, float]  # (k, u, g)


def tuning_grid_marker(cfg):
    """A changed grid or return forecast must not reuse a previous fit."""
    payload=dict(k=cfg.static_ml.k_grid,u=cfg.static_ml.u_grid,g=cfg.static_ml.g_grid)
    predictions=cfg.paths.processed/"predictions.parquet"
    info=predictions.stat() if predictions.exists() else None
    payload["predictions"]=dict(size=info.st_size,mtime_ns=info.st_mtime_ns) if info else None
    token=hashlib.sha256(json.dumps(payload,sort_keys=True).encode()).hexdigest()[:16]
    return cfg.paths.outputs/"static_ml_star"/f"tuning_grid_{token}.json"


def sigma_gam_adj(sigma_gam: np.ndarray, g: float, cov_type: str = "cov_add") -> np.ndarray:
    """R ``sigma_gam_adj``: ``cov_add`` adds g × the diagonal; ``cov_mult`` scales by g."""
    if cov_type == "cov_add":
        return sigma_gam + np.diag(np.diag(sigma_gam) * g)
    if cov_type == "cov_mult":
        return sigma_gam * g
    raise ValueError(f"unsupported cov_type {cov_type!r}")


def static_weights(inputs: Mapping[pd.Timestamp, MonthInputs], panel: pd.DataFrame, wealth: pd.DataFrame,
                   gamma: float, hyper_for: Callable[[pd.Timestamp], Hyper], charge_exit_tc: bool) -> pd.DataFrame:
    """Recursive Static-ML weights over the months in ``inputs``: ``id, eom, w, w_start, exit``."""
    def target(eom: pd.Timestamp, x: MonthInputs, held: np.ndarray) -> np.ndarray:
        k, u, g = hyper_for(eom)
        lam_w = x.wealth * k * x.lam
        with np.errstate(all="ignore"):  # spurious Accelerate matmul flags; checked in recursive_weights
            a = sigma_gam_adj(gamma * x.sigma, g) + np.diag(lam_w)
            return cho_solve(cho_factor(a), u * x.mu + lam_w * held)

    return recursive_weights(inputs, panel, wealth, target, charge_exit_tc)


def hyper_grid(k_grid: Sequence[float], u_grid: Sequence[float], g_grid: Sequence[float]) -> list[Hyper]:
    return [(float(k), float(u), float(g)) for k, u, g in itertools.product(k_grid, u_grid, g_grid)]


def validate(inputs, panel, wealth, gamma, grid: Sequence[Hyper], charge_exit_tc) -> pd.DataFrame:
    """Monthly performance for every fixed (k, u, g) over the validation months (one run each, VW start)."""
    # All fixed settings share the same monthly membership and observed returns.
    # Batch their inherited weights; this avoids thousands of identical pandas joins.
    # Each (k,g) Cholesky factor is shared by its u settings; formulas are unchanged.
    dates = sorted(inputs)
    count = len(grid)
    if not count:
        raise ValueError("Static-ML validation needs a nonempty hyperparameter grid")
    groups = {}
    for index, (k, u, g) in enumerate(grid):
        groups.setdefault((k, g), []).append(index)
    panels = {date: group.set_index("id") for date, group in panel.loc[
        panel["eom"].isin(dates), ["id", "eom", "me", "lambda", "ret_ld1", "tr_ld1"]].groupby("eom", sort=False)}
    market_growth = wealth.set_index("eom")["mu_ld1"]
    previous_ids, inherited, out = None, None, []
    for date in dates:
        x = inputs[date]
        monthly = panels[date]
        current = monthly.loc[x.ids]
        if previous_ids is None:
            capital = current["me"].to_numpy()
            held = np.broadcast_to((capital/capital.sum())[:,None], (len(x.ids), count)).copy()
            exit_turnover = np.zeros(count)
            exit_cost = np.zeros(count)
        else:
            positions = pd.Index(previous_ids).get_indexer(x.ids)
            held = np.zeros((len(x.ids), count))
            found = positions >= 0
            held[found] = inherited[positions[found]]
            exits = ~pd.Index(previous_ids).isin(x.ids)
            exits &= pd.Index(previous_ids).isin(monthly.index)
            exits &= np.any(inherited != 0.0, axis=1)
            if charge_exit_tc and exits.any():
                sold = inherited[exits]
                sale_lambda = monthly.loc[previous_ids[exits], "lambda"].to_numpy()
                if np.isnan(sale_lambda).any():
                    raise ValueError("Missing lambda on exiting positions")
                exit_turnover = np.abs(sold).sum(axis=0)
                exit_cost = (sale_lambda[:,None] * sold**2).sum(axis=0)
            else:
                exit_turnover = np.zeros(count)
                exit_cost = np.zeros(count)
        weights = np.empty_like(held)
        for (k,g), indices in groups.items():
            lam_w = x.wealth * k * x.lam
            matrix = sigma_gam_adj(gamma * x.sigma, g) + np.diag(lam_w)
            u_values = np.array([grid[i][1] for i in indices])
            rhs = x.mu[:,None] * u_values + lam_w[:,None] * held[:,indices]
            weights[:,indices] = cho_solve(cho_factor(matrix), rhs)
        target_returns = current["ret_ld1"].to_numpy()
        total_returns = current["tr_ld1"].to_numpy()
        growth = float(market_growth.loc[date])
        if (not np.isfinite(weights).all() or np.isnan(target_returns).any()
                or np.isnan(total_returns).any() or not np.isfinite(growth)):
            raise ValueError(f"Missing/nonfinite Static-ML validation inputs at {date.date()}")
        trades = weights-held
        month = pd.DataFrame({"eom": date, "eom_ret": date+MONTH,
            "inv": np.abs(weights).sum(axis=0), "shorting": -weights.clip(max=0).sum(axis=0),
            "turnover": np.abs(trades).sum(axis=0)+exit_turnover,
            "r": (weights*target_returns[:,None]).sum(axis=0),
            "tc": .5*x.wealth*((x.lam[:,None]*trades**2).sum(axis=0)+exit_cost),
            "hp_no": np.arange(count), "k": [hp[0] for hp in grid],
            "u": [hp[1] for hp in grid], "g": [hp[2] for hp in grid]})
        out.append(month)
        inherited = weights * ((1+total_returns)/(1+growth))[:,None]
        previous_ids = x.ids
        if date.month == 12:
            log.info("Static-ML validation %s: %d hyperparameter settings", date.date(), count)
    return pd.concat(out, ignore_index=True).sort_values(["hp_no", "eom"], ignore_index=True)


def select_hypers(validation: pd.DataFrame, gamma: float) -> pd.DataFrame:
    """Best (k, u, g) at each December by cumulative realised utility (R ``static_implement``).

    cum_var = cummean(r²) − cummean(r)²;  cum_obj = cummean(r − tc − ½γ·cum_var); rank by eom_ret.
    The December-Y choice is used for return year Y+1 (column ``year``).
    """
    v = validation.sort_values(["hp_no", "eom_ret"]).reset_index(drop=True)  # idxmax needs unique labels
    grp = v.groupby("hp_no")
    mean_r = grp["r"].transform(lambda s: s.expanding().mean())
    mean_r2 = (v["r"] ** 2).groupby(v["hp_no"]).transform(lambda s: s.expanding().mean())
    v["cum_var"] = mean_r2 - mean_r**2
    flow = v["r"] - v["tc"] - 0.5 * v["cum_var"] * gamma
    v["cum_obj"] = flow.groupby(v["hp_no"]).transform(lambda s: s.expanding().mean())
    dec = v[v["eom_ret"].dt.month == 12]
    best = dec.loc[dec.groupby("eom_ret")["cum_obj"].idxmax()]
    return best.assign(year=best["eom_ret"].dt.year + 1)[["year", "eom_ret", "hp_no", "k", "u", "g", "cum_obj"]].reset_index(drop=True)


def run_static_ml_star(panel: pd.DataFrame, wealth: pd.DataFrame, cov_parts: Mapping[pd.Timestamp, CovParts],
                  val_dates: Sequence[pd.Timestamp], eval_dates: Sequence[pd.Timestamp], gamma: float,
                  grid: Sequence[Hyper], charge_exit_tc: bool) -> dict[str, pd.DataFrame]:
    """Select each return year using completed prior Decembers, then trade monthly.

    The validation history expands through preceding realised years. Headline
    test weights start from value weights, as in the repository paper port.
    No fixed Static-ML strategy is returned or reported.
    """
    if not grid or len(set(grid)) != len(grid):
        raise ValueError("Static-ML* needs a nonempty unique parameter grid")
    if any(k <= 0 or u < 0 or g < 0 for k, u, g in grid):
        raise ValueError("Require k>0, u>=0, g>=0")
    inputs = month_inputs(panel, wealth, cov_parts, sorted(set(val_dates) | set(eval_dates)))
    # Every traded or exiting name must occur in one of these monthly universes.
    # Keep all rows for those names/dates (including non-members used for exits),
    # rather than repeatedly joining the entire screened regional training panel.
    held_ids = np.unique(np.concatenate([item.ids for item in inputs.values()]))
    panel = panel.loc[panel["id"].isin(held_ids) & panel["eom"].isin(inputs)].reset_index(drop=True)
    val_inputs = {d: inputs[d] for d in sorted(val_dates)}
    validation = validate(val_inputs, panel, wealth, gamma, grid, charge_exit_tc)
    chosen = select_hypers(validation, gamma)
    by_year = {int(r.year): (r.k, r.u, r.g) for r in chosen.itertuples()}

    def hyper_for(eom: pd.Timestamp) -> Hyper:
        year = (eom + MONTH).year  # return year of formation month eom
        if year not in by_year:
            raise ValueError(f"no Static-ML* hyperparameters chosen for return year {year}")
        return by_year[year]

    eval_dates = pd.DatetimeIndex(sorted(eval_dates))
    w_star = static_weights({d: inputs[d] for d in eval_dates}, panel, wealth, gamma, hyper_for, charge_exit_tc)
    pf_star = portfolio_returns(w_star, panel, wealth).assign(type="Static-ML*")
    log.info("Static-ML*: chose\n%s", chosen.to_string(index=False))
    return {"validation": validation, "chosen": chosen, "weights": w_star.assign(type="Static-ML*"),
            "pf": pf_star}


### `scripts/jkmp/portfolio/trading_speed.py`

Trading-speed matrix m (Lemma 1). Port of ``m_func`` (reference/jkmp_r/0 - Portfolio choice functions.R). With Λ diagonal, Lemma 1's fixed point reads m̃ = (x + y − m̃ ∘ G)⁻¹, m = Λ^{-1/2} m̃ Λ^{1/2}, G = 11ᵀ + Σ / (1 + r_f + μ)² second moment of the weight growth g (ḡ = 1) x = w⁻¹ Λ^{-1/2} γΣ Λ^{-1/2} risk relative to trading cost y = I + diag(diag G) and ∘ is the elementwise product. The optimal portfolio trades a fraction I − m towards the aim each month (Eq. 20): m → 0 (trade fully) as wealth → 0 and m → I (do not trade) as wealth → ∞.

**Role:** Calculation library. Other stages import its functions; this cell writes the module.

**Input:** the embedded source below. **Output:** `scripts/jkmp/portfolio/trading_speed.py`. Expand the code cell to read or edit it.


In [ ]:
%%writefile scripts/jkmp/portfolio/trading_speed.py
"""Trading-speed matrix m (Lemma 1). Port of ``m_func`` (reference/jkmp_r/0 - Portfolio choice functions.R).

With Λ diagonal, Lemma 1's fixed point reads

    m̃ = (x + y − m̃ ∘ G)⁻¹,      m = Λ^{-1/2} m̃ Λ^{1/2},

    G = 11ᵀ + Σ / (1 + r_f + μ)²         second moment of the weight growth g (ḡ = 1)
    x = w⁻¹ Λ^{-1/2} γΣ Λ^{-1/2}         risk relative to trading cost
    y = I + diag(diag G)

and ∘ is the elementwise product. The optimal portfolio trades a fraction I − m towards the aim each
month (Eq. 20): m → 0 (trade fully) as wealth → 0 and m → I (do not trade) as wealth → ∞.
"""

from __future__ import annotations

import numpy as np
from scipy.linalg import solve, sqrtm


def _parts(sigma: np.ndarray, lam: np.ndarray, wealth: float, gamma: float, rf: float, mu: float):
    lam = np.asarray(lam, dtype=float)
    if (lam <= 0).any():
        raise ValueError("lambda must be positive")
    n = len(lam)
    G = np.ones((n, n)) + sigma / (1.0 + rf + mu) ** 2
    s = 1.0 / np.sqrt(lam)  # diagonal of Λ^{-1/2}
    x = (gamma / wealth) * (s[:, None] * sigma * s[None, :])
    y = np.diag(1.0 + np.diag(G))
    return lam, G, x, y


def trading_speed(sigma: np.ndarray, lam: np.ndarray, wealth: float, gamma: float, rf: float, mu: float,
                  iterations: int = 10, return_tilde: bool = False) -> np.ndarray:
    """m (n × n) for monthly covariance ``sigma``, diagonal price impact ``lam`` (vector) and ``wealth``.

    ``rf`` and ``mu`` are the monthly risk-free rate and expected portfolio growth (``investor.mu``);
    callers pass the RF of the return month t+1 (docs/deviations.md D13; R passes RF of month t).

    Start value (R): m̃₀ = ½(σ̂ − √(σ̂² − 4I)) with σ̂ = x + 2I, the solution when G = 11ᵀ. It is
    computed in the algebraically identical, cancellation-free form 2·(σ̂ + √(σ̂² − 4I))⁻¹, with the
    square root from ``scipy.linalg.sqrtm``. Then ``iterations`` fixed-point steps, each solving
    (x + y − m̃∘G) m̃ = I with a symmetric solver (no explicit inverse).
    """
    if not np.isfinite(wealth) or wealth < 0:
        raise ValueError("wealth must be finite and nonnegative")
    if wealth == 0:
        # Figure 1's exact frictionless limit: fully rebalance to the aim.
        # This avoids dividing by zero in gamma/wealth; positive-wealth math
        # below is unchanged.
        return np.zeros_like(sigma, dtype=float)
    lam, G, x, y = _parts(sigma, lam, wealth, gamma, rf, mu)
    n = len(lam)
    eye = np.eye(n)
    with np.errstate(all="ignore"):  # spurious Accelerate matmul flags; checked below
        sig_hat = x + 2.0 * eye
        root = np.real(sqrtm(sig_hat @ sig_hat - 4.0 * eye))
        m_tilde = solve(sig_hat + root, 2.0 * eye, assume_a="sym")
        for _ in range(iterations):
            m_tilde = solve(x + y - m_tilde * G, eye, assume_a="sym")
            m_tilde = 0.5 * (m_tilde + m_tilde.T)  # keep exact symmetry against round-off
    if not np.isfinite(m_tilde).all():
        raise FloatingPointError("non-finite trading-speed matrix")
    if return_tilde:
        return m_tilde
    return (1.0 / np.sqrt(lam))[:, None] * m_tilde * np.sqrt(lam)[None, :]


def lemma1_residual(m_tilde: np.ndarray, sigma: np.ndarray, lam: np.ndarray, wealth: float, gamma: float,
                    rf: float, mu: float) -> float:
    """max |(x + y − m̃∘G) m̃ − I|: zero at the Lemma 1 fixed point."""
    _, G, x, y = _parts(sigma, lam, wealth, gamma, rf, mu)
    with np.errstate(all="ignore"):
        r = (x + y - m_tilde * G) @ m_tilde - np.eye(len(m_tilde))
    return float(np.abs(r).max())


### `scripts/jkmp/strategies.py`

The public strategy set for the base results; iterations have their own mandates.

**Role:** Calculation library. Other stages import its functions; this cell writes the module.

**Input:** the embedded source below. **Output:** `scripts/jkmp/strategies.py`. Expand the code cell to read or edit it.


In [ ]:
%%writefile scripts/jkmp/strategies.py
"""The public strategy set for the base results; iterations have their own mandates."""
STRATEGIES = ("Static-ML*", "Portfolio-ML")


def base_strategies(universe):
    return STRATEGIES if universe == "us_large" else ("Portfolio-ML",)


def base_result_paths(outputs):
    """Active base files only; never discover historical fixed Static-ML files."""
    return ([outputs/"static_ml_star/pf.parquet"] if outputs.name=="us_large" else []) + [outputs/"portfolio_ml/rff/pf.parquet"]


def base_sources_marker(outputs,folder):
    """Refresh derived tables/charts whenever their saved model returns change."""
    import hashlib
    signature=hashlib.sha256()
    for path in base_result_paths(outputs):
        signature.update(str(path.relative_to(outputs)).encode())
        if not path.exists():return outputs/folder/"sources_pending.json"
        with path.open("rb") as stream:
            signature.update(hashlib.file_digest(stream,"sha256").digest())
    return outputs/folder/f"sources_{signature.hexdigest()[:16]}.json"


def portfolio_tuning_marker(cfg):
    """Recompute base portfolio coefficients when penalties, inputs or settings change."""
    import hashlib
    import json
    from pathlib import Path
    plain=cfg.as_dict()
    payload={key:plain[key] for key in ("seed","features","dates","investor","portfolio","portfolio_ml","ridge_grid")}
    payload["prediction_years"]=cfg.return_model.pred_years
    payload["market"]=cfg.paths.outputs.name
    inputs=[cfg.paths.processed/name for name in ("panel.parquet","wealth.parquet","covariance/cov_factor.parquet","covariance/cov_loadings.parquet")]
    for g in cfg.portfolio_ml.rff_variance_grid:
        folder=cfg.paths.processed/"pfml"/f"rff_g{g:.4f}"
        inputs += sorted(folder.glob("*.npz")) + [folder/"meta.json",folder/"rff_w.npy"]
    payload["inputs"]={str(p):dict(size=p.stat().st_size,mtime_ns=p.stat().st_mtime_ns) if p.exists() else None for p in inputs}
    scripts=Path(__file__).resolve().parents[1]
    sources=[scripts/"base/07_portfolio_ml.py",scripts/"jkmp/portfolio/portfolio_ml.py",scripts/"jkmp/models/ridge.py"]
    payload["sources"]={str(p):hashlib.sha256(p.read_bytes()).hexdigest() for p in sources}
    token=hashlib.sha256(json.dumps(payload,sort_keys=True,default=str).encode()).hexdigest()[:16]
    return cfg.paths.outputs/"portfolio_ml/rff"/f"tuning_{token}.json"


def require_strategies(frame, complete=False, universe="us_large"):
    allowed=base_strategies(universe)
    names=set(frame["type"])
    if names-set(allowed) or (complete and names!=set(allowed)):
        raise ValueError(f"Expected only {allowed}; found {sorted(names)}")
    if frame.duplicated(["type","eom_ret"]).any():
        raise ValueError("Duplicate strategy/return-month rows")
    return frame


## Source — Verification tests


### `tests/conftest.py`

Use only Kati's private engine and workspace-local test scratch.

**Role:** Verification source. Tests run only when explicitly invoked; this cell writes the test file.

**Input:** the embedded source below. **Output:** `tests/conftest.py`. Expand the code cell to read or edit it.


In [ ]:
%%writefile tests/conftest.py
"""Use only Kati's private engine and workspace-local test scratch."""
from pathlib import Path
import sys

ROOT = Path(__file__).resolve().parents[1]
sys.path.insert(0, str(ROOT / "scripts"))


def pytest_configure(config):
    if config.option.basetemp is not None:
        return
    import os
    from uuid import uuid4
    scratch = (ROOT / "outputs/verification/test_tmp" / uuid4().hex).resolve()
    if not scratch.is_relative_to(ROOT) or scratch.exists():
        raise RuntimeError("Unexpected test scratch path")
    scratch.parent.mkdir(parents=True, exist_ok=True)
    config.option.basetemp = ("\\\\?\\" + str(scratch)) if os.name == "nt" else str(scratch)


### `tests/test_accounting.py`

A short/entry/exit example checked against explicit economic accounting.

**Role:** Verification source. Tests run only when explicitly invoked; this cell writes the test file.

**Input:** the embedded source below. **Output:** `tests/test_accounting.py`. Expand the code cell to read or edit it.


In [ ]:
%%writefile tests/test_accounting.py
"""A short/entry/exit example checked against explicit economic accounting."""
import numpy as np
import pandas as pd
import pytest

from jkmp.portfolio.accounting import add_start_weights, portfolio_returns


def test_inherited_short_entry_and_exit_costs_by_hand():
    jan, feb = pd.to_datetime(["2020-01-31", "2020-02-29"])
    panel = pd.DataFrame({
        "id": [1, 2, 1, 2, 3], "eom": [jan, jan, feb, feb, feb],
        "me": [60., 40., 60., 40., 50.],
        "tr_ld1": [.10, -.05, .04, np.nan, -.01],
        "ret_ld1": [.09, -.06, .03, np.nan, -.02],
        "lambda": [1e-10, 2e-10, 1e-10, 2e-10, 3e-10],
    })
    wealth = pd.DataFrame({"eom": [jan, feb], "wealth": [1e9, 1e9], "mu_ld1": [.01, .02]})
    targets = pd.DataFrame({"id": [1, 2, 1, 3], "eom": [jan, jan, feb, feb], "w": [.6, -.2, .2, .5]})
    weights = add_start_weights(targets, panel, wealth, [jan, feb], charge_exit_tc=True)
    feb_weights = weights[weights.eom.eq(feb)].set_index("id")
    assert feb_weights.loc[1, "w_start"] == pytest.approx(.6 * 1.10 / 1.01)
    assert feb_weights.loc[2, "w_start"] == pytest.approx(-.2 * .95 / 1.01)
    assert feb_weights.loc[3, "w_start"] == 0
    assert feb_weights.loc[2, "exit"] and feb_weights.loc[2, "w"] == 0

    pf = portfolio_returns(weights, panel, wealth).set_index("eom")
    assert pf.loc[jan, "r"] == pytest.approx(.6 * .09 + (-.2) * (-.06))
    assert pf.loc[jan, "tc"] == pytest.approx(.5 * 1e9 * 2e-10 * (-.2 - .4) ** 2)
    delta1, delta2, delta3 = .2 - .6 * 1.10 / 1.01, .2 * .95 / 1.01, .5
    assert pf.loc[feb, "r"] == pytest.approx(.2 * .03 + .5 * (-.02))
    assert pf.loc[feb, "tc"] == pytest.approx(.5 * 1e9 * (1e-10 * delta1**2 + 2e-10 * delta2**2 + 3e-10 * delta3**2))
    assert pf.loc[feb, "turnover"] == pytest.approx(abs(delta1) + abs(delta2) + abs(delta3))
    assert pf.loc[feb, "inv"] == pytest.approx(.7)
    assert pf.loc[feb, "eom_ret"] == pd.Timestamp("2020-03-31")

    # A zero exit position can have no forward return; a held position cannot.
    missing = panel.copy()
    missing.loc[missing.eom.eq(feb) & missing.id.eq(3), "ret_ld1"] = np.nan
    with pytest.raises(ValueError, match="missing ret_ld1"):
        portfolio_returns(weights, missing, wealth)


### `tests/test_active_math.py`

Benchmark-relative statistics (jkmp.evaluation.metrics active_*) and γ_TE calibration (jkmp.portfolio.active).

**Role:** Verification source. Tests run only when explicitly invoked; this cell writes the test file.

**Input:** the embedded source below. **Output:** `tests/test_active_math.py`. Expand the code cell to read or edit it.


In [ ]:
%%writefile tests/test_active_math.py
"""Benchmark-relative statistics (jkmp.evaluation.metrics active_*) and γ_TE calibration (jkmp.portfolio.active)."""

import numpy as np
import pandas as pd
import pytest

from jkmp.config import load_config
from jkmp.covariance import CovParts
from jkmp.evaluation.metrics import (active_returns, active_stats, active_utility_flow, active_weights,
                                     ex_ante_exposures, prob_greater)
from jkmp.portfolio.active import calibrate

pytestmark = pytest.mark.filterwarnings("ignore:.*encountered in matmul:RuntimeWarning")
DATES = load_config().dates
RNG = np.random.default_rng(11)
N, T = 4, 120
FORM = pd.date_range("1990-01-31", periods=T, freq="ME")


def _setup():
    rb = RNG.normal(0.007, 0.04, T)
    r = 0.9 * rb + RNG.normal(0.002, 0.01, T)
    pf = pd.DataFrame({"eom": FORM, "eom_ret": FORM + pd.offsets.MonthEnd(1), "r": r, "tc": 0.0005})
    bench = pf.assign(r=rb, tc=0.0001)
    me = np.array([4.0, 3.0, 2.0, 1.0])
    a = np.array([0.02, -0.01, 0.0, -0.01])
    panel = pd.DataFrame([{"id": i, "eom": e, "me": me[i]} for e in FORM for i in range(N)])
    weights = pd.DataFrame([{"id": i, "eom": e, "w": me[i] / me.sum() + a[i] - (0.3 if i == 3 else 0.0),
                             "w_start": 0.0, "exit": False} for e in FORM for i in range(N)])
    X = RNG.normal(size=(N, 2))
    cov = {e: CovParts(e, np.arange(N), ("sec_a", "theme_x"), X, np.diag([0.002, 0.001]), np.full(N, 0.003))
           for e in FORM}
    return pf, bench, weights, panel, cov, X


def test_active_returns_and_utility():
    pf, bench, *_ = _setup()
    act = active_returns(pf, bench, DATES)
    np.testing.assert_allclose(act["active"], pf["r"] - bench["r"])
    np.testing.assert_allclose(act["active_net"], (pf["r"] - 0.0005) - (bench["r"] - 0.0001))
    u = active_utility_flow(act, 50.0)
    np.testing.assert_allclose(u, act["active_net"] - 25.0 * (act["active"] - act["active"].mean()) ** 2)
    assert prob_greater(pd.Series([0.01, 0.02, 0.015])) > 0.99
    assert prob_greater(pd.Series([1.0, 2.0, 3.0]), pd.Series([6.0, 7.5, 7.8])) < 1e-3
    with pytest.raises(ValueError, match="different return months"):
        active_returns(pf, bench.iloc[1:], DATES)


def test_active_stats_by_hand():
    pf, bench, weights, panel, cov, X = _setup()
    s = active_stats(pf, bench, weights, panel, cov, 50.0, DATES)
    act = active_returns(pf, bench, DATES)
    sd = act["active"].std(ddof=1)
    assert s["ir_net"] == pytest.approx(act["active_net"].mean() / sd * np.sqrt(12))
    assert s["te_realised"] == pytest.approx(sd * np.sqrt(12))
    assert s["beta"] == pytest.approx(np.polyfit(bench["r"], pf["r"], 1)[0], rel=1e-10)
    a = weights.loc[weights["eom"] == FORM[0], "w"].to_numpy() - np.array([0.4, 0.3, 0.2, 0.1])
    sigma = X @ np.diag([0.002, 0.001]) @ X.T + np.diag(np.full(N, 0.003))
    assert s["te_ex_ante"] == pytest.approx(np.sqrt(a @ sigma @ a) * np.sqrt(12), rel=1e-10)
    assert s["neg_weight_share"] == pytest.approx(0.25)                     # stock 3: 0.1 − 0.01 − 0.3 < 0
    ex = ex_ante_exposures(active_weights(weights, panel), cov)
    np.testing.assert_allclose(ex.loc[0, ["sec_a", "theme_x"]].to_numpy(dtype=float), X.T @ a, rtol=1e-12)


def test_calibrate_interpolates_log_te_on_log_gamma():
    points = pd.DataFrame({"gamma_te": [10.0, 100.0, 1000.0], "te_ex_ante": [0.08, 0.02, 0.005]})
    out = calibrate(points, [0.02, 0.01]).set_index("te_target")["gamma_te"]
    assert out[0.02] == pytest.approx(100.0)
    assert out[0.01] == pytest.approx(np.exp(np.interp(np.log(0.01), np.log([0.005, 0.02]), np.log([1000.0, 100.0]))),
                                      rel=1e-3)                          # rounded to 4 significant figures
    with pytest.raises(ValueError, match="outside"):
        calibrate(points, [0.2])


### `tests/test_additive_reporting.py`

Economic reporting checks distinguish additive paths from investor wealth.

**Role:** Verification source. Tests run only when explicitly invoked; this cell writes the test file.

**Input:** the embedded source below. **Output:** `tests/test_additive_reporting.py`. Expand the code cell to read or edit it.


In [ ]:
%%writefile tests/test_additive_reporting.py
"""Economic reporting checks distinguish additive paths from investor wealth."""
import numpy as np
import pandas as pd
import pytest

from analysis.ml_comparison.metrics import additive_path, paths_and_rolling, statistics


def monthly(net):
    return pd.DataFrame(dict(dataset='example', strategy='Portfolio ML',
        eom_ret=pd.date_range('2020-01-31', periods=len(net), freq='ME'),
        r=np.array(net)+.001, tc=.001, turnover=.1, inv=1., shorting=0.))


def test_additive_losses_include_initial_zero_and_recovery():
    # Two 10% losses accumulate to -20 pp, not the compounded -19%.
    path, dd = additive_path([-.1, -.1, .3, -.15])
    np.testing.assert_allclose(path, [-.1, -.2, .1, -.05], atol=1e-15)
    np.testing.assert_allclose(dd, [-.1, -.2, 0., -.15], atol=1e-15)
    stats = statistics(monthly([-.1, -.1, .3, -.15]))
    assert stats['max_drawdown_excess'] == pytest.approx(-.2)
    assert stats['cumulative_net_excess'] == pytest.approx(-.05)
    assert stats['path_convention'] == 'additive'
    assert 'cagr_excess' not in stats and 'calmar_excess' not in stats


def test_statistical_and_monthly_reports_agree_without_discarding_large_losses():
    frame = monthly([.2, -1.2, .4, -.1]).sample(frac=1, random_state=8)
    stats = statistics(frame)
    paths = paths_and_rolling(frame)
    np.testing.assert_allclose(paths.cumulative_net_excess, [.2, -1., -.6, -.7])
    np.testing.assert_allclose(paths.drawdown_excess, [0., -1.2, -.8, -.9])
    assert stats['max_drawdown_excess'] == pytest.approx(paths.drawdown_excess.min())
    assert stats['cumulative_net_excess'] == pytest.approx(paths.cumulative_net_excess.iloc[-1])
    assert stats['R_TC'] == pytest.approx(12*frame.eval('r-tc').mean())
    assert len(paths) == len(frame) and paths.drawdown_excess.notna().all()


def test_additive_active_path_is_sum_of_paired_monthly_differences():
    candidate = np.array([.02, -.03, .04, .01])
    market = np.array([.01, -.02, .03, .005])
    a, _ = additive_path(candidate)
    b, _ = additive_path(market)
    np.testing.assert_allclose(a-b, np.cumsum(candidate-market), atol=1e-15)
    with pytest.raises(ValueError, match='finite'):
        additive_path([0., np.nan])


### `tests/test_base_static_star.py`

Base scope, lagged annual tuning, and independently reconstructed Static ML* trades.

**Role:** Verification source. Tests run only when explicitly invoked; this cell writes the test file.

**Input:** the embedded source below. **Output:** `tests/test_base_static_star.py`. Expand the code cell to read or edit it.


In [ ]:
%%writefile tests/test_base_static_star.py
"""Base scope, lagged annual tuning, and independently reconstructed Static ML* trades."""
import json
from types import SimpleNamespace

import numpy as np
import pandas as pd
import pytest

from jkmp.covariance import CovParts
from jkmp.portfolio.accounting import portfolio_returns
from jkmp.portfolio.recursion import month_inputs
from jkmp.portfolio.static_ml_star import hyper_grid, run_static_ml_star, select_hypers, static_weights, validate
from jkmp.strategies import base_strategies
from run_all import plan
from workflow import UNIVERSES


def example():
    dates=pd.date_range("2018-12-31",periods=25,freq="ME")
    rng=np.random.default_rng(91)
    rows=[];cov={}
    for i,date in enumerate(dates):
        ids=np.arange(1,4)
        cov[date]=CovParts(date,ids,("f",),np.ones((3,1)),np.array([[.0003]]),np.array([.002,.003,.004]))
        for stock in ids:
            # Membership change forces a real exit and a new position.
            rows.append(dict(id=stock,eom=date,me=100.+stock,**{"lambda":1e-10*stock},
                pred_ld1=.004*stock,ret_ld1=rng.normal(.005,.03),tr_ld1=rng.normal(.006,.03),
                in_universe=stock in ((1,2) if i<13 else (1,3))))
    panel=pd.DataFrame(rows)
    wealth=pd.DataFrame(dict(eom=dates,wealth=1e9,mu_ld1=.006))
    return dates,panel,wealth,cov


@pytest.mark.parametrize("charge_exit",[False,True])
def test_batched_validation_matches_independent_recursive_accounting(charge_exit):
    dates,panel,wealth,cov=example()
    inputs=month_inputs(panel,wealth,cov,dates)
    grid=hyper_grid([1.,.2],[0.,.5,1.],[0.,2.])
    observed=validate(inputs,panel,wealth,10.,grid,charge_exit)
    fields=["r","tc","turnover","inv","shorting"]
    for i,hp in enumerate(grid):
        weights=static_weights(inputs,panel,wealth,10.,lambda date:hp,charge_exit)
        expected=portfolio_returns(weights,panel,wealth).sort_values("eom")
        actual=observed.loc[observed.hp_no.eq(i)].sort_values("eom")
        np.testing.assert_allclose(actual[fields],expected[fields],rtol=1e-11,atol=1e-13)


def test_future_outcomes_do_not_change_prior_year_choice_or_weights():
    dates,panel,wealth,cov=example();grid=hyper_grid([1.,.2],[.25,1.],[0.,2.])
    evaluation=dates[(dates+pd.offsets.MonthEnd(1)).year==2020]
    first=run_static_ml_star(panel,wealth,cov,dates,evaluation,10.,grid,True)
    changed=panel.copy()
    future=(changed.eom+pd.offsets.MonthEnd(1)).dt.year>=2020
    changed.loc[future,"ret_ld1"]=-2.
    second=run_static_ml_star(changed,wealth,cov,dates,evaluation,10.,grid,True)
    pd.testing.assert_frame_equal(first["weights"],second["weights"])
    pd.testing.assert_frame_equal(first["chosen"].query("year<=2020"),second["chosen"].query("year<=2020"))
    assert set(first["pf"].type)=={"Static-ML*"}
    assert first["chosen"].eom_ret.dt.year.add(1).eq(first["chosen"].year).all()


def test_december_choice_matches_literal_paper_cumulative_objective():
    _,panel,wealth,cov=example();dates=sorted(cov);grid=hyper_grid([1.,.2],[.25,1.],[0.,2.])
    validation=validate(month_inputs(panel,wealth,cov,dates),panel,wealth,10.,grid,True)
    chosen=select_hypers(validation,10.)
    for row in chosen.itertuples():
        scores=[]
        for i in range(len(grid)):
            d=validation.loc[validation.hp_no.eq(i)&validation.eom_ret.le(row.eom_ret)].sort_values("eom_ret")
            variance=[np.mean(d.r.iloc[:n]**2)-np.mean(d.r.iloc[:n])**2 for n in range(1,len(d)+1)]
            scores.append(np.mean(d.r-d.tc-5*np.array(variance)))
        assert row.hp_no==int(np.argmax(scores))
        assert row.cum_obj==pytest.approx(max(scores))


def test_base_scope_and_iterations_are_separate():
    jobs=plan(UNIVERSES,"full","all")
    assert [j.universe for j in jobs if j.stage=="05"]==["us_large"]
    assert base_strategies("us_large")==("Static-ML*","Portfolio-ML")
    assert base_strategies("us_mid")==base_strategies("eu_large")==("Portfolio-ML",)
    assert len([j for j in jobs if j.stage=="experiment"])==5
    from comparison_analysis import base_sources,iteration_baseline_sources
    sources=base_sources()
    assert len([s for s in sources if s["expected_type"]=="Static-ML*"])==1
    assert not any(s["expected_type"]=="Static-ML" for s in sources)
    assert any(s["expected_type"]=="Static-ML*" for s in iteration_baseline_sources())
    from jkmp.config import load_config
    from workflow import config_path
    for i in range(1,6):
        cfg=load_config(config_path("us_large",f"iteration_{i}"))
        assert cfg.reporting.main_types==("Static-ML*","Portfolio-ML")
        assert cfg.static_ml.k_grid==(1.,1/3,.2,.1,.05)
        assert cfg.static_ml.g_grid==(0.,1.,2.,3.)


def test_portfolio_cache_reuse_rejects_changed_data_or_model_settings():
    from figure1 import portfolio_compatible
    import copy
    old=dict(config=dict(reporting={},static_ml={},regimes={},portfolio_ml={"p_grid":[8]}),sources={"panel.parquet":"aaa","/static_ml.py":"old","/portfolio_ml.py":"bbb"})
    new=copy.deepcopy(old);new["config"]["reporting"]={"main_types":["Portfolio-ML"]}
    new["sources"].pop("/static_ml.py");new["sources"]["/static_ml_star.py"]="new"
    assert portfolio_compatible(old,new)
    for part in ("data","model"):
        changed=copy.deepcopy(new)
        if part=="data":changed["sources"]["panel.parquet"]="changed"
        else:changed["config"]["portfolio_ml"]["p_grid"]=[16]
        assert not portfolio_compatible(old,changed)


def test_combined_figure_contains_only_all_retained_frontiers(tmp_path,monkeypatch):
    import figure1
    monkeypatch.setattr(figure1,"OUT",tmp_path)
    monkeypatch.setattr(figure1,"provenance",lambda universe:{"signature":"verified"})
    monkeypatch.setattr(figure1,"load_config",lambda path:SimpleNamespace(investor=SimpleNamespace(gamma=10.,wealth=1e10)))
    monkeypatch.setattr(figure1,"market_dir",lambda universe:tmp_path/universe/"market")
    for universe in UNIVERSES:
        folder=tmp_path/universe;folder.mkdir()
        (folder/"market").mkdir()
        pf=pd.DataFrame(dict(r=[.01,.02,-.01],tc=[.001]*3,turnover=[.1]*3,inv=[1.]*3,shorting=[0.]*3))
        pf.to_parquet(folder/"market/pf.parquet",index=False)
        rows=[dict(type=method,gamma=g,wealth=w,Vol=.2/g**.5,R_TC=.1/g**.5)
              for method,_ in figure1.frontier_methods(universe) for w in (0.,1e10) for g in (1.,10.,100.)]
        pd.DataFrame(rows).to_csv(folder/"frontier_summary.csv",index=False)
        (folder/"completion.json").write_text(json.dumps(dict(signature="verified")))
    assert figure1.plot_combined()
    result=pd.read_csv(tmp_path/"combined_frontier_summary.csv")
    assert len(result)==24
    assert set(result.loc[result.type.eq("Static-ML*"),"universe"])=={"us_large"}
    assert (tmp_path/"charts/figure1_combined.pdf").stat().st_size>1000


### `tests/test_dashboard.py`

Keep the results viewer honest about missing fits and source identities.

**Role:** Verification source. Tests run only when explicitly invoked; this cell writes the test file.

**Input:** the embedded source below. **Output:** `tests/test_dashboard.py`. Expand the code cell to read or edit it.


In [ ]:
%%writefile tests/test_dashboard.py
"""Keep the results viewer honest about missing fits and source identities."""
import json

import numpy as np
import pandas as pd
import pytest

from dashboard import build_dashboard, collect_series


def test_new_build_displays_unavailable_results_without_substituting_keats(tmp_path):
    original = tmp_path / "keats_pipeline/outputs/base/us_large/static_ml"
    original.mkdir(parents=True)
    (original / "pf.parquet").write_bytes(b"not a Kati result")
    kati = tmp_path / "Kati_tryout"
    kati.mkdir()
    assert collect_series(kati) == []
    build_dashboard(kati)
    status = json.loads((kati / "outputs/dashboard_status.json").read_text())
    assert status["completed_series"] == 0
    assert status["models_fitted_by_dashboard"] is False
    assert "No completed Kati model results yet" in (kati / "outputs/dashboard.html").read_text()


def test_viewer_uses_net_returns_and_costs_from_completed_local_file(tmp_path):
    folder = tmp_path / "outputs/base/us_large/static_ml_star"
    folder.mkdir(parents=True)
    frame = pd.DataFrame(dict(eom_ret=pd.date_range("2020-01-31", periods=3, freq="ME"),
        r=[.02, -.01, .03], tc=[.001, .002, .003], turnover=[.1, .2, .3], type="Static-ML*"))
    frame.to_parquet(folder / "pf.parquet", index=False)
    rows = collect_series(tmp_path)
    assert len(rows) == 1
    assert np.isclose(rows[0]["summary"]["annual_net"], 12*(frame.r-frame.tc).mean())
    assert np.isclose(rows[0]["summary"]["annual_cost"], 12*frame.tc.mean())
    assert np.isclose(rows[0]["points"][0]["net"], .019)
    frame["type"] = "Factor-ML"
    frame.to_parquet(folder / "pf.parquet", index=False)
    with pytest.raises(ValueError, match="strategy identity"):
        collect_series(tmp_path)


### `tests/test_isolation.py`

Check the declared experiment and protect the original pipeline's files.

**Role:** Verification source. Tests run only when explicitly invoked; this cell writes the test file.

**Input:** the embedded source below. **Output:** `tests/test_isolation.py`. Expand the code cell to read or edit it.


In [ ]:
%%writefile tests/test_isolation.py
"""Check the declared experiment and protect the original pipeline's files."""
import json
from pathlib import Path

import pytest

from workflow import ROOT, UNIVERSES, config_path, originals_unchanged
from jkmp.config import ConfigError, load_config
from run_all import plan
from script_groups import normalise_group, script_path


def test_private_engine_and_all_configured_inputs_are_local():
    import jkmp.config
    import analysis.ml_comparison.cli as comparison

    expected = json.loads((ROOT / "data/characteristics/recommended_50.json").read_text())
    assert len(expected) == len(set(expected)) == 50
    assert Path(jkmp.config.__file__).resolve().is_relative_to(ROOT / "scripts")
    assert comparison.KATI == ROOT
    choices = [(u, "base") for u in UNIVERSES]
    choices += [("us_large", f'iteration_{i}') for i in range(1,6)]
    for market, group in choices:
        cfg = load_config(config_path(market, group))
        assert set(cfg.features.names) == set(expected)
        assert len(cfg.features.names) == 50
        for target in (cfg.paths.raw, cfg.paths.processed, cfg.paths.outputs):
            assert target.resolve().is_relative_to(ROOT)
        assert cfg.dates.holdout_start is None and cfg.dates.holdout_end is None
    assert originals_unchanged()


@pytest.mark.parametrize("field", ["raw", "processed", "outputs"])
def test_configuration_refuses_to_write_into_original_keats(field):
    target = ROOT.parent / "keats_pipeline" / "outputs" / "forbidden_test"
    with pytest.raises(ConfigError, match="inside its own folder"):
        load_config(config_path("us_large"), {f"paths.{field}": str(target)})
    assert not target.exists()


def test_plan_keeps_all_base_markets_and_only_us_high_cap_iterations():
    jobs = plan(UNIVERSES, "full", "all")
    names = {job.name for job in jobs}
    assert len(names) == len(jobs)
    assert all(set(job.dependencies).issubset(names) for job in jobs)
    for market in UNIVERSES:
        for stage in ("portfolio", "tables", "market_reference"):
            assert f"{market}.base.{stage}" in names
        assert (f"{market}.base.static" in names)==(market=="us_large")
    for group in (f'iteration_{i}' for i in range(1,6)):
        assert any(f"us_large.{group}." in name for name in names)
    assert not any(job.universe in ("us_mid", "eu_large") and job.variant != "base" for job in jobs)
    numerical = {"03", "04", "05", "06", "07", "10", "11", "12", "13", "14", "monitor","experiment"}
    for job in jobs:
        if job.stage in numerical:
            path = script_path(job.stage, normalise_group(job.variant))
            assert path.is_file() and path.resolve().is_relative_to(ROOT / "scripts")


def test_final_plan_has_no_hardcoded_external_source_dependency():
    settings = json.loads((ROOT / "settings.json").read_text())
    for key in ("source", "reference_data"):
        assert (ROOT / settings[key]).resolve().is_relative_to(ROOT / "data")
    assert not settings["downloads_allowed"]


### `tests/test_iteration_sequence.py`

Acceptance, paired statistics and accepted-only ancestry, without real-data model fits.

**Role:** Verification source. Tests run only when explicitly invoked; this cell writes the test file.

**Input:** the embedded source below. **Output:** `tests/test_iteration_sequence.py`. Expand the code cell to read or edit it.


In [ ]:
%%writefile tests/test_iteration_sequence.py
"""Acceptance, paired statistics and accepted-only ancestry, without real-data model fits."""
import json
from types import SimpleNamespace

import numpy as np
import pandas as pd
import pytest

import iteration_sequence as seq


def pf(offset=0.):
    rng=np.random.default_rng(100)
    return pd.DataFrame(dict(eom_ret=pd.date_range('2020-01-31',periods=72,freq='ME'),
        r=rng.normal(.005,.03,72)+offset,tc=.001,turnover=.1,inv=1.,shorting=.1))


def test_bootstrap_is_paired_reproducible_and_reverses():
    a=pf(.003);b=pf()
    first=seq.paired_probability(a,b,reps=500)
    second=seq.paired_probability(a,b,reps=500)
    reverse=seq.paired_probability(b,a,reps=500)
    assert first==second
    assert first['probability']>.95
    assert first['probability']+reverse['probability']==pytest.approx(1)
    assert seq.paired_probability(a,a)['probability'] is None
    with pytest.raises(ValueError,match='calendars'):
        seq.paired_probability(a,b.iloc[1:])


def test_buffer_does_not_use_future_returns():
    rows=pd.DataFrame([dict(id=i,eom=pd.Timestamp(y,12,31),me=100-i,ret_ld1=0.)
                       for y in (2018,2019,2020) for i in range(1,11)])
    a=seq.buffered_membership(rows,5,target_n=3,entry_rank=2)
    b=seq.buffered_membership(rows.assign(ret_ld1=999.),5,target_n=3,entry_rank=2)
    pd.testing.assert_frame_equal(a,b)


def test_rejected_experiments_never_enter_next_candidate(tmp_path,monkeypatch):
    monkeypatch.setattr(seq,'ROOT',tmp_path)
    monkeypatch.setattr(seq,'OUT',tmp_path/'outputs/iteration_summary')
    base=tmp_path/'outputs/base/us_large/portfolio_ml/rff';base.mkdir(parents=True)
    monkeypatch.setattr(seq,'BASE',base)
    for relative in ('portfolio_ml/rff','static_ml_star'):
        folder=tmp_path/'outputs/base/us_large'/relative;folder.mkdir(parents=True,exist_ok=True)
        pf().to_parquet(folder/'pf.parquet')
    for market in ('us_mid','eu_large'):
        folder=tmp_path/f'outputs/base/{market}/portfolio_ml/rff';folder.mkdir(parents=True)
        pf().to_parquet(folder/'pf.parquet')
    market=tmp_path/'outputs/base/overall analysis/market_references/us_large';market.mkdir(parents=True)
    pf().to_parquet(market/'pf.parquet')
    (base/'weights.parquet').write_bytes(b'unchanged')
    (base/'chosen.csv').write_text('year,lambda\n2020,1\n')
    protocol=json.loads(seq.PROTOCOL.read_text())
    file=tmp_path/'protocol.json';file.write_text(json.dumps(protocol))
    monkeypatch.setattr(seq,'PROTOCOL',file)
    calls=[]
    class Fake:
        signature='test'
        cfg=SimpleNamespace(investor=SimpleNamespace(gamma=10.),universe=SimpleNamespace(exit_rank=100),
                            dates=SimpleNamespace(test_start='2020-01-31',test_end='2025-12-31'))
        wealth=None
        def run_candidate(self,state,folder):calls.append(dict(state));return pf(len(calls)*.001)
        def panel_for(self,state,folder):return None
    monkeypatch.setattr(seq,'ExperimentEngine',Fake)
    probabilities=iter([.9,.3,.95,.1])
    monkeypatch.setattr(seq,'paired_probability',lambda *a,**k:dict(probability=next(probabilities),identical=False))
    monkeypatch.setattr(seq,'verify_accounting',lambda *a:None)
    monkeypatch.setattr(seq,'write_iteration_report',lambda *a:None)
    monkeypatch.setattr(seq,'report_saved',lambda *a,**k:0)
    assert seq.main([])==0
    assert calls[1]['gamma']==5. and calls[1]['exit_rank']==120
    assert calls[2]['gamma']==5. and calls[2]['exit_rank']==100 and calls[2]['cost_multiple']==1.
    assert calls[3]['benchmark_relative'] and calls[3]['half_lives']==[3,5,10,None]
    d=json.loads((tmp_path/'outputs/iteration_4/us_large/decision.json').read_text())
    assert d['inherited_iterations']==[1,2] and d['included_after']==[1,2,4]
    d=json.loads((tmp_path/'outputs/iteration_5/us_large/decision.json').read_text())
    assert not d['accepted'] and d['included_after']==[1,2,4]


def test_frontier_moment_reuse_never_reuses_changed_risk_or_observations():
    from figure1 import moments_compatible
    import copy
    old=dict(config=dict(ridge_grid={'n':100},return_model={},investor={'gamma':10}),
             sources={'/panel.parquet':'a','/predictions.parquet':'b','/portfolio_ml.py':'c'})
    current=copy.deepcopy(old);current['config']['ridge_grid']={'n':200};current['sources']['/predictions.parquet']='new'
    assert moments_compatible(old,current)
    current['config']['investor']['gamma']=5
    assert not moments_compatible(old,current)
    current=copy.deepcopy(old);current['sources']['/panel.parquet']='new'
    assert not moments_compatible(old,current)


def test_filtered_reader_preserves_all_declared_members_and_feature_values():
    engine=seq.ExperimentEngine()
    cfg=engine.cfg
    keys=pd.MultiIndex.from_frame(engine.panel[['id','eom']])
    held=pd.read_parquet(cfg.paths.processed/'panel.parquet',columns=['id','eom'],filters=[('in_universe','=',True)])
    assert pd.MultiIndex.from_frame(held).difference(keys).empty
    for members in engine._memberships.values():
        assert pd.MultiIndex.from_frame(members[['id','eom']]).difference(keys).empty
    date=pd.Timestamp('2019-12-31')
    columns=['id','eom',*cfg.features.names]
    original=pd.read_parquet(cfg.paths.processed/'panel.parquet',columns=columns,filters=[('eom','=',date)])
    selected=engine.panel[engine.panel.eom.eq(date)][columns]
    original=original[original.id.isin(selected.id)].sort_values('id').reset_index(drop=True)
    pd.testing.assert_frame_equal(selected.sort_values('id').reset_index(drop=True),original)


### `tests/test_portfolio_math.py`

Portfolio-ML (jkmp.portfolio.portfolio_ml): inputs (signal scaling, the lag weighting against a literal transcription of R's gtm loop, moments against the full Σ), the coefficient search against lstsq, the December selection and aims, and the weights π_t = m·π_start + (I − m)·aim with their wealth limits.

**Role:** Verification source. Tests run only when explicitly invoked; this cell writes the test file.

**Input:** the embedded source below. **Output:** `tests/test_portfolio_math.py`. Expand the code cell to read or edit it.


In [ ]:
%%writefile tests/test_portfolio_math.py
"""Portfolio-ML (jkmp.portfolio.portfolio_ml): inputs (signal scaling, the lag weighting against a literal
transcription of R's gtm loop, moments against the full Σ), the coefficient search against lstsq, the
December selection and aims, and the weights π_t = m·π_start + (I − m)·aim with their wealth limits."""

import numpy as np
import pandas as pd
import pytest

from jkmp.covariance import CovParts
from jkmp.models.random_features import column_subset
from jkmp.portfolio.portfolio_ml import (MonthMoments, PfmlMoments, aim_portfolios, build_moments, build_yearly_moments,
                                         december_utility, feature_index, load_yearly_moments, save_yearly_moments,
                                         search_yearly, yearly_sums,
                                         hp_reals, load_moments, month_moments, pfml_weights, ridge_coefs,
                                         save_moments,
                                         scale_signals, search, search_coef, select_hypers, signal_names,
                                         weighted_signals)
from jkmp.portfolio.accounting import portfolio_returns
from jkmp.portfolio.recursion import month_inputs
from jkmp.portfolio.trading_speed import trading_speed

pytestmark = pytest.mark.filterwarnings("ignore:.*encountered in matmul:RuntimeWarning")

RNG = np.random.default_rng(11)


def _r_gtm_loop(signals, g, m, lb):
    """Literal transcription of the weighting block of R pfml_input_fun (lists named by lag)."""
    n = m.shape[0]
    gtm = [m @ np.diag(gi) for gi in g]
    gtm_agg, gtm_agg_l1 = [None] * (len(gtm) - 1), [None] * (len(gtm) - 1)
    gtm_agg[0] = np.eye(n)
    gtm_agg_l1[0] = np.eye(n)
    for i in range(1, lb + 1):                                   # R: for (i in 1:lb)
        gtm_agg[i] = gtm_agg[i - 1] @ gtm[i - 1]
        gtm_agg_l1[i] = gtm_agg_l1[i - 1] @ gtm[i]
    omega = const = omega_l1 = const_l1 = 0
    for i in range(lb + 1):                                      # R: for (i in 0:lb)
        omega = omega + gtm_agg[i] @ signals[i]
        const = const + gtm_agg[i]
        omega_l1 = omega_l1 + gtm_agg_l1[i] @ signals[i + 1]
        const_l1 = const_l1 + gtm_agg_l1[i]
    return np.linalg.solve(const, omega), np.linalg.solve(const_l1, omega_l1)


def _random_cov(eom, ids, k=3):
    return CovParts(eom, np.asarray(ids), tuple(f"f{j}" for j in range(k)), RNG.normal(size=(len(ids), k)) * 0.05,
                    np.diag(RNG.uniform(0.001, 0.003, k)), RNG.uniform(0.002, 0.006, len(ids)))


def test_scale_signals_demeans_and_normalises():
    raw = RNG.normal(size=(6, 4))
    s = scale_signals(raw, None)
    np.testing.assert_allclose(s[:, 0], 1 / np.sqrt(6))                  # constant → 1/√n
    np.testing.assert_allclose(s[:, 1:].sum(axis=0), 0, atol=1e-14)      # demeaned
    np.testing.assert_allclose((s**2).sum(axis=0), 1)                    # unit sum of squares
    vol = RNG.uniform(0.05, 0.1, 6)
    np.testing.assert_allclose(scale_signals(raw, vol), s / vol[:, None])  # rows × 1/vol
    flat = scale_signals(np.ones((6, 1)), None)
    assert np.all(flat[:, 1] == 0)                                       # 0/0 column left at zero


@pytest.mark.parametrize("lb", [0, 1, 3, 11])
def test_weighted_signals_matches_r_loop(lb):
    n, P = 5, 4
    signals = [RNG.normal(size=(n, P)) for _ in range(lb + 2)]
    g = [RNG.uniform(0.9, 1.1, n) for _ in range(lb + 2)]
    m = RNG.uniform(0, 0.2, (n, n)) + np.diag(RNG.uniform(0.3, 0.6, n))
    got = weighted_signals(signals, g, m, lb)
    want = _r_gtm_loop(signals, g, m, lb)
    for a, b in zip(got, want):
        np.testing.assert_allclose(a, b, rtol=1e-12, atol=1e-14)


def test_weighted_signals_limits():
    n, P, lb = 4, 3, 2
    signals = [RNG.normal(size=(n, P)) for _ in range(lb + 2)]
    g = [np.ones(n)] * (lb + 2)
    omega, omega_l1 = weighted_signals(signals, g, np.zeros((n, n)), lb)    # m = 0: trade fully to today's aim
    np.testing.assert_allclose(omega, signals[0])
    np.testing.assert_allclose(omega_l1, signals[1])
    omega, omega_l1 = weighted_signals(signals, g, np.eye(n), lb)           # m = I, g = 1: equal-weighted lags
    np.testing.assert_allclose(omega, np.mean(signals[: lb + 1], axis=0))
    np.testing.assert_allclose(omega_l1, np.mean(signals[1:], axis=0))


def test_month_moments_match_full_sigma():
    n, P = 6, 5
    ids = np.arange(10, 10 + n)
    cov = _random_cov(pd.Timestamp("2020-01-31"), np.arange(8, 18))      # Σ covers more ids than the universe
    omega, omega_l1 = RNG.normal(size=(n, P)), RNG.normal(size=(n, P))
    g_t, r, lam = RNG.uniform(0.9, 1.1, n), RNG.normal(0, 0.05, n), RNG.uniform(1e-11, 1e-10, n)
    r_tilde, risk, tc = month_moments(omega, omega_l1, g_t, r, cov, ids, lam, 1e10, 10.0)
    sigma = cov.full(ids)
    d = omega - np.diag(g_t) @ omega_l1
    np.testing.assert_allclose(r_tilde, omega.T @ r)
    np.testing.assert_allclose(risk, 10.0 * omega.T @ sigma @ omega, rtol=1e-12)
    np.testing.assert_allclose(tc, 1e10 * d.T @ np.diag(lam) @ d, rtol=1e-12)
    with pytest.raises(KeyError):
        month_moments(omega, omega_l1, g_t, r, cov, ids + 100, lam, 1e10, 10.0)


def test_feature_index_matches_r_feature_names():
    names = signal_names([], 8)
    idx = feature_index(4, 8)
    assert [names[i] for i in idx] == ["constant", "rff1_cos", "rff2_cos", "rff1_sin", "rff2_sin"]
    np.testing.assert_array_equal(idx[1:] - 1, column_subset(4, 8))      # same layout as the return model


def _panel(n_ids=7, n_months=8, features=("a", "b", "c")):
    dates = pd.date_range("2019-01-31", periods=n_months, freq="ME")
    rows = []
    for eom in dates:
        for i in range(n_ids):
            rows.append({"id": i + 1, "eom": eom, "in_universe": i < 5, "g": RNG.uniform(0.95, 1.05),
                         "ret_ld1": RNG.normal(0, 0.05), "lambda": RNG.uniform(1e-11, 1e-10),
                         **{f: RNG.uniform() for f in features}})
    panel = pd.DataFrame(rows)
    wealth = pd.DataFrame({"eom": dates, "wealth": 1e10, "mu_ld1": 0.01})
    cov = {eom: _random_cov(eom, np.arange(1, 6)) for eom in dates}
    rf = pd.Series(0.001, index=pd.date_range("2018-12-31", periods=n_months + 2, freq="ME"))
    return panel, wealth, cov, rf, dates, list(features)


@pytest.mark.parametrize("use_rff", [True, False])
def test_build_moments_end_to_end(use_rff, tmp_path):
    panel, wealth, cov, rf, dates, feats = _panel()
    lb = 2
    t = dates[-1:]
    cov[dates[-2]] = _random_cov(dates[-2], np.arange(1, 5))           # id 5 has no Σ at lag 1 → month median vol
    kw = dict(gamma=10.0, mu=0.007, lb=lb, iterations=10, use_rff=use_rff, rff_variance=0.1, p_max=8, seed=1)
    res = build_moments(panel, wealth, cov, rf, t, feats, **kw)
    P1 = 9 if use_rff else 4
    assert res.columns == signal_names(feats, 8 if use_rff else None)
    assert sorted(res.months) == list(t)

    # Recompute one month by hand from the panel
    d = t[-1]
    mm = res.months[d]
    ids = np.arange(1, 6)
    np.testing.assert_array_equal(mm.ids, ids)
    sigs, gs = [], []
    for i in range(lb + 2):
        s = panel[(panel["eom"] == d - pd.offsets.MonthEnd(i)) & panel["id"].isin(ids)].sort_values("id")
        x = s[feats].to_numpy()
        raw = np.hstack([np.cos(x @ res.rff_w), np.sin(x @ res.rff_w)]) if use_rff else x
        c = cov[s["eom"].iloc[0]]
        vol = pd.Series(c.vol(), index=c.ids)
        sigs.append(scale_signals(raw, vol.reindex(ids).fillna(vol.median()).to_numpy()))
        gs.append(s["g"].to_numpy())
    now = panel[(panel["eom"] == d) & panel["id"].isin(ids)].sort_values("id")
    m = trading_speed(cov[d].full(ids), now["lambda"].to_numpy(), 1e10, 10.0, 0.001, 0.007, 10)
    omega, omega_l1 = _r_gtm_loop(sigs, gs, m, lb)
    dlt = omega - np.diag(gs[0]) @ omega_l1
    assert mm.signal.shape == (5, P1)
    np.testing.assert_allclose(mm.signal, sigs[0], rtol=1e-12)
    np.testing.assert_allclose(mm.r_tilde, omega.T @ now["ret_ld1"].to_numpy(), rtol=1e-10)
    np.testing.assert_allclose(mm.risk, 10.0 * omega.T @ cov[d].full(ids) @ omega, rtol=1e-10)
    np.testing.assert_allclose(mm.tc, 1e10 * dlt.T @ np.diag(now["lambda"].to_numpy()) @ dlt, rtol=1e-8, atol=1e-14)

    save_moments(res, tmp_path)
    back = load_moments(tmp_path)
    assert back.columns == res.columns and back.meta["use_rff"] == use_rff
    np.testing.assert_array_equal(back.months[d].denom, mm.denom)
    assert (back.rff_w is None) == (not use_rff)


def test_build_moments_requires_lag_rows():
    panel, wealth, cov, rf, dates, feats = _panel()
    panel = panel[~((panel["id"] == 3) & (panel["eom"] == dates[3]))]
    with pytest.raises(ValueError, match="no row"):
        build_moments(panel, wealth, cov, rf, dates[4:5], feats, gamma=10.0, mu=0.007, lb=2, iterations=10,
                      use_rff=False)


# --- coefficient search, validation, aims ---------------------------------------------------------

P_MAX = 6
LAMBDAS = [0.0, 0.05, 2.0]


def _toy_moments(seed=3, start="2011-12-31", end="2016-11-30"):
    """r̃_t = A_tᵀy_t and Σ̃_t = A_tᵀA_t, so the PF-ML ridge is a stacked least-squares problem."""
    rng = np.random.default_rng(seed)
    cols = signal_names([], P_MAX)
    months, data = {}, {}
    for d in pd.date_range(start, end, freq="ME"):
        a, y = rng.normal(size=(4, len(cols))), rng.normal(size=4)
        data[d] = (a, y)
        months[d] = MonthMoments(d, np.arange(1, 5), a.T @ y, a.T @ a, None, rng.normal(size=(4, len(cols))))
    meta = {"use_rff": True, "p_max": P_MAX, "add_orig": False, "features": ["x1", "x2"], "rff_variance": 0.1}
    return PfmlMoments(cols, meta, months), data


def _lstsq_beta(data, year, cols, lam):
    """Brute force: minimise Σ_t ‖A_t β − y_t‖²/N + λ‖β‖² over the months with returns realised by Dec year−1."""
    use = [d for d in data if (d + pd.offsets.MonthEnd(1)).year <= year - 1]
    n = len(use)
    A = np.vstack([data[d][0][:, cols] for d in use]) / np.sqrt(n)
    y = np.concatenate([data[d][1] for d in use]) / np.sqrt(n)
    A = np.vstack([A, np.sqrt(lam) * np.eye(len(cols))])
    y = np.concatenate([y, np.zeros(len(cols))])
    return np.linalg.lstsq(A, y, rcond=None)[0]


@pytest.mark.parametrize("p", [2, 4, 6])
def test_closed_form_beta_matches_lstsq(p):
    moments, data = _toy_moments()
    coefs = search_coef(moments, [2, 4, 6], LAMBDAS, [2014, 2015, 2016])
    cols = feature_index(p, P_MAX)
    for year in (2014, 2015, 2016):
        for j, lam in enumerate(LAMBDAS):
            np.testing.assert_allclose(coefs[year][p][:, j], _lstsq_beta(data, year, cols, lam), rtol=1e-9, atol=1e-12)


def test_ridge_coefs_equals_solve():
    rng = np.random.default_rng(0)
    a = rng.normal(size=(30, 7))
    d, r = a.T @ a / 30, rng.normal(size=7)
    for j, lam in enumerate(LAMBDAS):
        np.testing.assert_allclose(ridge_coefs(d, r, LAMBDAS)[:, j], np.linalg.solve(d + lam * np.eye(7), r), rtol=1e-10)


def test_coefficients_do_not_peek_at_the_hp_year():
    moments, _ = _toy_moments()
    before = search_coef(moments, [4], LAMBDAS, [2015])
    for d, mm in moments.months.items():
        if (d + pd.offsets.MonthEnd(1)).year >= 2015:                 # scramble everything realised in 2015+
            mm.r_tilde, mm.denom = mm.r_tilde * -7.0, mm.denom * 3.0
    after = search_coef(moments, [4], LAMBDAS, [2015])
    np.testing.assert_array_equal(before[2015][4], after[2015][4])


def test_hp_reals_objective_and_cumulative_mean():
    moments, _ = _toy_moments()
    coefs = search_coef(moments, [2, 6], LAMBDAS, [2015, 2016])
    v = hp_reals(moments, coefs, LAMBDAS)
    assert set(v["hp_end"]) == {2015, 2016} and len(v) == 2 * 12 * 2 * len(LAMBDAS)
    d = pd.Timestamp("2016-03-31")                                     # return month Apr 2016 → β of 2016
    c, b = feature_index(2, P_MAX), coefs[2016][2][:, 1]
    mm = moments.months[d]
    want = mm.r_tilde[c] @ b - 0.5 * b @ mm.denom[np.ix_(c, c)] @ b
    row = v[(v["eom"] == d) & (v["p"] == 2) & (v["l_no"] == 1)].iloc[0]
    assert row["obj"] == pytest.approx(want, rel=1e-12)
    path = v[(v["p"] == 2) & (v["l_no"] == 1)].sort_values("eom_ret")
    assert row["cum_obj"] == pytest.approx(path.loc[path["eom_ret"] <= row["eom_ret"], "obj"].mean(), rel=1e-12)
    assert (v.groupby("eom_ret")["rank"].min() == 1).all()


def test_select_hypers_across_signal_sets_for_next_year():
    months = pd.date_range("2015-01-31", "2016-12-31", freq="ME")
    rows = []
    for tag, p, r15, r16 in [("a", 2, 0.02, 0.0), ("b", 4, 0.0, 0.05)]:
        obj = np.where(months.year == 2015, r15, r16)
        cum = pd.Series(obj).expanding().mean().to_numpy()
        rows.append(pd.DataFrame({"eom_ret": months, "tag": tag, "g": 0.1, "p": p, "l_no": 0, "l": 0.0,
                                  "obj": obj, "cum_obj": cum}))
    best = select_hypers(pd.concat(rows)).set_index("year")
    assert best.loc[2016, "tag"] == "a" and best.loc[2017, "tag"] == "b"


def test_aim_portfolios_use_chosen_model_and_year_coefficients():
    moments, _ = _toy_moments()
    aim_dates = [pd.Timestamp("2015-12-31"), pd.Timestamp("2016-05-31")]     # return year 2016
    res = search(moments, "rff_g0.1000", [2, 4], LAMBDAS, [2015, 2016], aim_dates)
    chosen = pd.DataFrame({"year": [2016], "tag": ["rff_g0.1000"], "p": [4], "l_no": [2], "l": [LAMBDAS[2]]})
    aims, coefs = aim_portfolios({"rff_g0.1000": res}, chosen, aim_dates)
    c = feature_index(4, P_MAX)
    for d in aim_dates:
        want = moments.months[d].signal[:, c] @ res.coefs[2016][4][:, 2]
        np.testing.assert_allclose(aims.loc[aims["eom"] == d, "w_aim"].to_numpy(), want)
    assert list(coefs.loc[coefs["eom"] == aim_dates[0], "column"]) == ["constant", "rff1_cos", "rff2_cos",
                                                                        "rff1_sin", "rff2_sin"]
    with pytest.raises(ValueError, match="no Portfolio-ML hyperparameters"):
        aim_portfolios({"rff_g0.1000": res}, chosen, [pd.Timestamp("2016-12-31")])


# --- weights: π_t = m·π_start + (I − m)·aim ------------------------------------------------------

W_DATES = pd.date_range("2020-01-31", periods=5, freq="ME")
W_IDS = np.array([1, 2, 3])
W_GAMMA, W_MU, W_RF = 10.0, 0.007, 0.001


def _weights_setup(wealth_level):
    rng = np.random.default_rng(5)
    rows, aims = [], []
    base_aim = np.array([0.5, -0.3, 0.8])
    for t, eom in enumerate(W_DATES):
        for j, i in enumerate(W_IDS):
            rows.append({"id": i, "eom": eom, "me": 100.0 * i, "in_universe": True, "lambda": 2e-10 * i,
                         "tr_ld1": 0.01 * (i - 1), "ret_ld1": 0.005 * i})
        aims.append(pd.DataFrame({"id": W_IDS, "eom": eom, "w_aim": base_aim * (-1) ** t}))  # flips every month
    panel = pd.DataFrame(rows)
    wealth = pd.DataFrame({"eom": W_DATES, "wealth": wealth_level, "mu_ld1": 0.01})
    cov = {eom: CovParts(eom, W_IDS, ("f",), rng.normal(size=(3, 1)) * 0.05, np.array([[0.002]]),
                         np.array([0.004, 0.003, 0.005])) for eom in W_DATES}
    rf = pd.Series(W_RF, index=pd.date_range("2020-01-31", periods=7, freq="ME"))
    return panel, wealth, cov, rf, pd.concat(aims, ignore_index=True)


def _run_weights(wealth_level):
    panel, wealth, cov, rf, aims = _weights_setup(wealth_level)
    inputs = month_inputs(panel, wealth, cov, W_DATES, mu_col=None)
    w = pfml_weights(inputs, aims, panel, wealth, rf, gamma=W_GAMMA, mu=W_MU, iterations=10, charge_exit_tc=True)
    return w, portfolio_returns(w, panel, wealth), aims, cov, panel


def test_tiny_wealth_trades_fully_to_the_aim():
    w, pf, aims, _, _ = _run_weights(1e-6)                               # m ≈ 0
    merged = w.merge(aims, on=["id", "eom"])
    np.testing.assert_allclose(merged["w"], merged["w_aim"], atol=1e-10)
    assert (pf["turnover"] > 1.0).all()                                  # the aim flips sign every month


def test_huge_wealth_stops_trading():
    w, pf, _, _, _ = _run_weights(1e20)                                  # m → I
    np.testing.assert_allclose(w.loc[w["eom"] == W_DATES[0], "w"], [1 / 6, 2 / 6, 3 / 6], atol=1e-4)  # stays VW
    assert (pf["turnover"] < 1e-4).all()


def test_turnover_falls_with_wealth():
    turnover = [_run_weights(level)[1]["turnover"].mean() for level in (1e-6, 1e8, 1e10, 1e12, 1e20)]
    assert all(a > b for a, b in zip(turnover, turnover[1:]))


def test_one_recursion_step_by_hand():
    w, _, aims, cov, panel = _run_weights(1e10)
    lam = 2e-10 * W_IDS
    m = trading_speed(cov[W_DATES[0]].full(W_IDS), lam, 1e10, W_GAMMA, W_RF, W_MU, 10)
    vw = np.array([1, 2, 3]) / 6
    aim0 = aims.loc[aims["eom"] == W_DATES[0], "w_aim"].to_numpy()
    first = w[w["eom"] == W_DATES[0]].sort_values("id")
    np.testing.assert_allclose(first["w_start"], vw)
    np.testing.assert_allclose(first["w"], m @ vw + (np.eye(3) - m) @ aim0, rtol=1e-12)
    second = w[w["eom"] == W_DATES[1]].sort_values("id")
    np.testing.assert_allclose(second["w_start"], first["w"].to_numpy() * (1 + 0.01 * (W_IDS - 1)) / 1.01, rtol=1e-12)


def test_missing_aim_raises():
    panel, wealth, cov, rf, aims = _weights_setup(1e10)
    inputs = month_inputs(panel, wealth, cov, W_DATES, mu_col=None)
    with pytest.raises(ValueError, match="aim"):
        pfml_weights(inputs, aims[aims["id"] != 2], panel, wealth, rf, gamma=W_GAMMA, mu=W_MU, iterations=10,
                     charge_exit_tc=True)


# --- yearly sums (grid runs): same β and December selection without monthly matrices ---------------

def test_december_utility_equals_monthly_cumulative_mean():
    moments, _ = _toy_moments(start="2011-12-31", end="2017-11-30")
    coefs = search_coef(moments, [2, 6], LAMBDAS, [2015, 2016, 2017])
    monthly = hp_reals(moments, coefs, LAMBDAS)
    dec = december_utility(yearly_sums(moments.months), moments, coefs, LAMBDAS)
    m = monthly[monthly["eom_ret"].dt.month == 12].merge(dec, on=["eom_ret", "p", "l_no"], suffixes=("_m", "_y"))
    assert len(m) == len(dec) == 3 * 2 * len(LAMBDAS)
    np.testing.assert_allclose(m["cum_obj_m"], m["cum_obj_y"], rtol=1e-10)
    np.testing.assert_array_equal(m["rank_m"], m["rank_y"])


def test_yearly_moments_match_monthly_build_and_round_trip(tmp_path):
    panel, wealth, cov, rf, dates, feats = _panel(n_months=14)
    kw = dict(gamma=10.0, mu=0.007, lb=2, iterations=10, use_rff=True, rff_variance=0.1, p_max=8, seed=1)
    t = dates[3:]
    full = build_moments(panel, wealth, cov, rf, t, feats, **kw)
    ym = build_yearly_moments(panel, wealth, cov, rf, t, feats, signal_dates=t[-2:], **kw)
    want = yearly_sums(full.months)
    assert sorted(ym.years) == sorted(want)
    for y in want:
        assert ym.years[y].n == want[y].n
        np.testing.assert_allclose(ym.years[y].d_sum, want[y].d_sum, rtol=1e-12)
        np.testing.assert_allclose(ym.years[y].r_sum, want[y].r_sum, rtol=1e-12)
    np.testing.assert_array_equal(ym.signals[t[-1]][1], full.months[t[-1]].signal)
    save_yearly_moments(ym, tmp_path / "m.npz")
    back = load_yearly_moments(tmp_path / "m.npz")
    assert back.columns == ym.columns and back.meta == ym.meta and sorted(back.signals) == sorted(ym.signals)
    np.testing.assert_array_equal(back.years[2019].d_sum, ym.years[2019].d_sum)
    np.testing.assert_array_equal(back.signals[t[-1]][0], ym.signals[t[-1]][0])
    np.testing.assert_array_equal(back.rff_w, ym.rff_w)


def test_yearly_search_picks_the_same_hyperparameters():
    moments, _ = _toy_moments(start="2011-12-31", end="2017-11-30")
    aim_dates = [pd.Timestamp("2016-06-30")]
    monthly = search(moments, "t", [2, 4, 6], LAMBDAS, [2015, 2016, 2017], aim_dates)
    sigs = {d: (moments.months[d].ids, moments.months[d].signal) for d in aim_dates}
    from jkmp.portfolio.portfolio_ml import YearlyMoments
    ym = YearlyMoments(moments.columns, moments.meta, yearly_sums(moments.months), sigs)
    yearly = search_yearly(ym, "t", [2, 4, 6], LAMBDAS, [2015, 2016, 2017])
    a, b = select_hypers(monthly.validation), select_hypers(yearly.validation)
    pd.testing.assert_frame_equal(a[["year", "p", "l_no"]], b[["year", "p", "l_no"]])
    np.testing.assert_allclose(a["cum_obj"], b["cum_obj"], rtol=1e-10)


# --- benchmark-relative (active) Portfolio-ML, D34 -------------------------------------------------

def test_scale_signals_and_feature_index_without_constant():
    raw = np.random.default_rng(1).normal(size=(6, 4))
    with_c, without = scale_signals(raw, None), scale_signals(raw, None, add_constant=False)
    assert with_c.shape == (6, 5) and without.shape == (6, 4)
    np.testing.assert_allclose(without, with_c[:, 1:])
    np.testing.assert_allclose(without.sum(axis=0), 0.0, atol=1e-12)            # every column demeaned
    assert feature_index(4, 8, constant=False).tolist() == [0, 1, 4, 5]          # cos 1–2, sin 1–2
    assert feature_index(4, 8).tolist() == [0, 1, 2, 5, 6]


def test_active_recursion_tracks_vw_plus_active_book():
    panel, wealth, cov, rf, aims = _weights_setup(1e10)
    inputs = month_inputs(panel, wealth, cov, W_DATES, mu_col=None)
    kw = dict(gamma=W_GAMMA, mu=W_MU, iterations=10, charge_exit_tc=True, benchmark_relative=True)
    vw = np.array([1, 2, 3]) / 6
    zero = aims.assign(w_aim=0.0)
    w0 = pfml_weights(inputs, zero, panel, wealth, rf, **kw)    # no active view: VW, then trade drift back at speed m
    for k, (eom, g) in enumerate(w0.groupby("eom")):
        g = g.sort_values("id")
        mk = trading_speed(cov[eom].full(W_IDS), 2e-10 * W_IDS, 1e10, W_GAMMA, W_RF, W_MU, 10)
        expect = vw if k == 0 else vw + mk @ (g["w_start"].to_numpy() - vw)
        np.testing.assert_allclose(g["w"], expect, atol=1e-12)
    w = pfml_weights(inputs, aims, panel, wealth, rf, **kw)
    m = trading_speed(cov[W_DATES[0]].full(W_IDS), 2e-10 * W_IDS, 1e10, W_GAMMA, W_RF, W_MU, 10)
    aim0 = aims.loc[aims["eom"] == W_DATES[0], "w_aim"].to_numpy()
    first = w[w["eom"] == W_DATES[0]].sort_values("id")
    np.testing.assert_allclose(first["w"], vw + (np.eye(3) - m) @ aim0, rtol=1e-12)   # a_inh = 0 at the VW start
    second = w[w["eom"] == W_DATES[1]].sort_values("id")
    m1 = trading_speed(cov[W_DATES[1]].full(W_IDS), 2e-10 * W_IDS, 1e10, W_GAMMA, W_RF, W_MU, 10)
    aim1 = aims.loc[aims["eom"] == W_DATES[1], "w_aim"].to_numpy()
    a_inh = second["w_start"].to_numpy() - vw
    np.testing.assert_allclose(second["w"], vw + m1 @ a_inh + (np.eye(3) - m1) @ aim1, rtol=1e-12)
    # Δπ = Δa exactly: the trade is the change of the active book against today's value weights
    np.testing.assert_allclose(second["w"] - second["w_start"], (second["w"].to_numpy() - vw) - a_inh, atol=1e-15)


# --- half-life discounting of β, D36 -----------------------------------------------------------------

def _toy_yearly():
    from jkmp.portfolio.portfolio_ml import YearlyMoments
    moments, data = _toy_moments(start="2009-12-31", end="2017-11-30")
    d = pd.Timestamp("2016-06-30")
    return YearlyMoments(moments.columns, moments.meta, yearly_sums(moments.months),
                         {d: (moments.months[d].ids, moments.months[d].signal)}), data


def test_infinite_half_life_reproduces_the_undiscounted_betas_exactly():
    ym, _ = _toy_yearly()
    years = [2013, 2014, 2015, 2016, 2017]
    base = search_yearly(ym, "t", [2, 4, 6], LAMBDAS, years)
    tuned = search_yearly(ym, "t", [2, 4, 6], LAMBDAS, years, half_lives=(3, 5, 10, None))
    for y in years:
        for p in (2, 4, 6):
            np.testing.assert_array_equal(tuned.coefs_by_h[np.inf][y][p], base.coefs[y][p])   # bit for bit
    inf_rows = tuned.validation[tuned.validation["h"] == np.inf].drop(columns="h").reset_index(drop=True)
    pd.testing.assert_frame_equal(inf_rows, base.validation.reset_index(drop=True)[inf_rows.columns])
    assert set(tuned.validation["h"]) == {3.0, 5.0, 10.0, np.inf}
    assert "h" in select_hypers(tuned.validation)


def test_discounted_beta_by_hand():
    ym, _ = _toy_yearly()
    h, y = 3.0, 2016
    delta = 0.5 ** (1 / h)
    use = [s for s in sorted(ym.years) if s <= y - 1]
    wts = np.array([delta ** (y - s) for s in use])
    r_bar = sum(w * ym.years[s].r_sum for w, s in zip(wts, use)) / sum(w * ym.years[s].n for w, s in zip(wts, use))
    d_bar = sum(w * ym.years[s].d_sum for w, s in zip(wts, use)) / sum(w * ym.years[s].n for w, s in zip(wts, use))
    from jkmp.portfolio.portfolio_ml import coefs_from_sums
    got = coefs_from_sums(ym.years, ym, [4], LAMBDAS, [y], half_life=h)[y][4]
    c = feature_index(4, P_MAX)
    np.testing.assert_allclose(got, ridge_coefs(d_bar[np.ix_(c, c)], r_bar[c], LAMBDAS), rtol=1e-10)


def test_aims_use_the_chosen_half_life():
    ym, _ = _toy_yearly()
    res = search_yearly(ym, "t", [2, 4, 6], LAMBDAS, [2016], half_lives=(3, None))
    d = pd.Timestamp("2016-06-30")
    chosen = pd.DataFrame({"year": [2016], "tag": ["t"], "p": [4], "l_no": [1], "l": [LAMBDAS[1]], "h": [3.0]})
    aims, _ = aim_portfolios({"t": res}, chosen, [d])
    c = feature_index(4, P_MAX)
    np.testing.assert_allclose(aims["w_aim"], ym.signals[d][1][:, c] @ res.coefs_by_h[3.0][2016][4][:, 1])
    assert not np.allclose(res.coefs_by_h[3.0][2016][4], res.coefs_by_h[np.inf][2016][4])


def test_half_life_config_accepts_null():
    from jkmp.config import load_config
    from workflow import config_path
    assert load_config(config_path("us_large","iteration_3"),overrides=["portfolio_ml.half_life_years=[3, 5, 10, null]"]).portfolio_ml.half_life_years == (3.0, 5.0, 10.0, None)


### `tests/test_rff_forecast.py`

RFF identities and a synthetic walk-forward fit, independent of cached results.

**Role:** Verification source. Tests run only when explicitly invoked; this cell writes the test file.

**Input:** the embedded source below. **Output:** `tests/test_rff_forecast.py`. Expand the code cell to read or edit it.


In [ ]:
%%writefile tests/test_rff_forecast.py
"""RFF identities and a synthetic walk-forward fit, independent of cached results."""
import numpy as np
import pandas as pd
import pytest

from workflow import config_path
from jkmp.config import load_config
from jkmp.models.random_features import base_draws, weights_for, rff, rff_raw, column_subset
from jkmp.models.return_model import fit_return_model, validation_rows, tuning_marker


def test_validation_rows_follow_historical_membership():
    rows = pd.DataFrame(dict(id=[1, 2, 1, 2],
        eom=pd.to_datetime(["2013-12-31", "2013-12-31", "2014-12-31", "2014-12-31"]),
        in_universe=[True, False, False, True], ret_ld1=[.01, 99., 99., .02]))
    selected = validation_rows(rows, "universe")
    assert list(zip(selected.id, selected.eom.dt.year)) == [(1, 2013), (2, 2014)]
    assert validation_rows(rows, "all") is rows
    with pytest.raises(ValueError, match="historical in_universe"):
        validation_rows(rows.drop(columns="in_universe"), "universe")
    with pytest.raises(ValueError, match="No eligible"):
        validation_rows(rows.assign(in_universe=False), "universe")


def test_market_configs_and_checkpoints_are_independent():
    from run_all import Job, output_paths
    paths=[]
    for market in ("us_large", "us_mid", "eu_large"):
        cfg=load_config(config_path(market))
        assert cfg.return_model.validation_scope=="universe"
        marker=tuning_marker(cfg)
        assert marker.parent==cfg.paths.processed/"return_model"
        assert marker in output_paths(Job(f"{market}.base.returns", "03", market))
        legacy=load_config(config_path(market), {"return_model.validation_scope":"all"})
        assert marker!=tuning_marker(legacy)
        paths.append(marker)
    assert len(set(paths))==3
    for i in range(1,6):
        assert load_config(config_path("us_large",f"iteration_{i}")).return_model.validation_scope=="universe"


def test_base_and_iteration_ridge_grid_checkpoints():
    from jkmp.strategies import portfolio_tuning_marker
    from run_all import Job, output_paths
    for market in ("us_large","us_mid","eu_large"):
        cfg=load_config(config_path(market))
        assert len(cfg.ridge_grid.values)==101 and cfg.ridge_grid.values[0]==0.0
        different=load_config(config_path(market),{"ridge_grid.n":3})
        assert len(different.ridge_grid.values)==4
        assert tuning_marker(cfg)!=tuning_marker(different)
        assert portfolio_tuning_marker(cfg)!=portfolio_tuning_marker(different)
        assert portfolio_tuning_marker(cfg) in output_paths(Job(market,"07",market))
    for i in (1,2,3):
        cfg=load_config(config_path("us_large",f"iteration_{i}"))
        assert len(cfg.ridge_grid.values)==101


def test_restore_does_not_reuse_stale_outputs_with_an_old_marker(tmp_path):
    import os
    from run_all import cached_outputs_ready
    output=tmp_path/"table.csv"
    old_marker=tmp_path/"sources_old.json"
    new_marker=tmp_path/"sources_new.json"
    output.write_text("old result")
    old_marker.write_text("old settings")
    os.utime(output,ns=(100,100));os.utime(old_marker,ns=(200,200))
    assert cached_outputs_ready([output,old_marker])
    output.write_text("new result")
    os.utime(output,ns=(300,300))
    new_marker.write_text("new settings")
    os.utime(new_marker,ns=(400,400))
    assert not cached_outputs_ready([output,old_marker])
    assert cached_outputs_ready([output,new_marker])


def test_fourier_pairs_scaling_and_nested_model_sizes():
    x = np.array([[0., .2, 1.], [.5, .8, .3]])
    z = base_draws(3, 8, seed=12)
    np.testing.assert_array_equal(z, base_draws(3, 8, seed=12))
    w = weights_for(z, .25)
    np.testing.assert_allclose(w, .5 * z)
    projection = x @ w
    expected = np.column_stack([np.cos(projection[:, :2]), np.sin(projection[:, :2])]) / 2
    np.testing.assert_allclose(rff(x, w, 4), expected)
    np.testing.assert_allclose(rff_raw(x, w)[:, column_subset(4, 8)] / 2, expected)
    np.testing.assert_allclose((expected ** 2).sum(axis=1), .5)
    with pytest.raises(FloatingPointError, match="non-finite"):
        rff_raw(np.array([[np.nan, 0., 1.]]), w)


def test_forecaster_learns_coefficients_without_seeing_forecast_year_returns():
    cfg = load_config(config_path("us_large"), {
        "return_model.pred_years": [2015, 2015],
        "return_model.p_grid": [2, 4],
        "return_model.rff_variance_grid": [.05],
        "ridge_grid.n": 3,
    })
    features = list(cfg.features.names)
    dates = pd.date_range("2004-12-31", "2015-11-30", freq="ME")
    rng = np.random.default_rng(58)
    x = rng.uniform(0, 1, (len(dates) * 6, len(features)))
    panel = pd.DataFrame(x, columns=features)
    panel["id"] = np.tile(np.arange(1, 7), len(dates))
    panel["eom"] = np.repeat(dates, 6)
    panel["valid_data"] = True
    panel["in_universe"] = panel.id.le(3)
    panel["ret_ld1"] = .08 * x[:, 0] - .03 * x[:, 1] + rng.normal(0, .002, len(x))

    first = fit_return_model(panel, cfg)
    future_changed = panel.copy()
    future_changed.loc[future_changed.eom.ge("2014-12-31"), "ret_ld1"] = 99.
    second = fit_return_model(future_changed, cfg)
    pd.testing.assert_frame_equal(first.predictions, second.predictions)
    pd.testing.assert_frame_equal(first.chosen, second.chosen)
    assert first.models[2015].coef.shape == (first.models[2015].p,)
    assert np.linalg.norm(first.models[2015].coef) > 0
    assert first.predictions.pred_ld1.std() > 0
    assert first.chosen.n_test.iloc[0] == 72
    assert first.chosen.validation_scope.iloc[0] == "universe"
    assert first.chosen.n_val.iloc[0] == 5 * 12 * 3
    assert first.chosen.n_val_regional.iloc[0] == 5 * 12 * 6

    # A changed past target changes the fit: these are learned coefficients,
    # rather than fixed characteristic weights or only a random projection.
    past_changed = panel.copy()
    past_changed.loc[past_changed.eom.lt("2014-12-31"), "ret_ld1"] *= -1
    third = fit_return_model(past_changed, cfg)
    np.testing.assert_allclose(third.predictions.pred_ld1, -first.predictions.pred_ld1, rtol=1e-10, atol=1e-12)


### `tests/test_themes.py`

Provisional assignments are explicit inputs, not silently published classifications.

**Role:** Verification source. Tests run only when explicitly invoked; this cell writes the test file.

**Input:** the embedded source below. **Output:** `tests/test_themes.py`. Expand the code cell to read or edit it.


In [ ]:
%%writefile tests/test_themes.py
"""Provisional assignments are explicit inputs, not silently published classifications."""
from copy import deepcopy

import numpy as np
import pandas as pd
import pytest

from workflow import ROOT, config_path
from jkmp.config import load_config
from jkmp.features import load_theme_directions, theme_scores


def test_frozen_group_override_is_complete_and_refuses_relabelled_config():
    cfg = load_config(config_path("us_large"))
    directions = load_theme_directions(cfg.paths.raw, cfg.features.themes)
    assert len(directions) == 50
    assert directions.feature.is_unique
    assert directions.direction.isin([-1, 1]).all()
    metadata=pd.read_csv(ROOT/'data/characteristics/selected_feature_metadata.csv')
    assert directions.published_group.sum() == metadata.published_jkp_153.sum()
    assert (~directions.published_group).sum() == (~metadata.published_jkp_153).sum()
    assert directions.direction_source.notna().all()
    changed = {name: list(values) for name, values in cfg.features.themes.items()}
    group = next(iter(changed))
    changed["incorrect_theme"] = [changed[group].pop()]
    with pytest.raises(ValueError, match="theme differs"):
        load_theme_directions(cfg.paths.raw, changed)


def test_signed_theme_exposures_do_not_mutate_raw_ml_ranks():
    ranks = pd.DataFrame({"a": [.2, .9], "b": [.8, .1]})
    original = ranks.copy(deep=True)
    directions = pd.DataFrame({"feature": ["a", "b"], "theme": ["quality", "quality"], "direction": [1, -1]})
    result = theme_scores(ranks, directions)
    np.testing.assert_allclose(result.theme_quality, [.2, .9])
    pd.testing.assert_frame_equal(ranks, original)


## Source — Experiment configurations


### `configs/eu_large/base.yaml`

Declarative settings for the local Kati workflow; paths resolve inside Kati_tryout.

**Role:** Pipeline implementation or configuration. This cell writes the file without running its main function.

**Input:** the embedded source below. **Output:** `configs/eu_large/base.yaml`. Expand the code cell to read or edit it.


In [ ]:
%%writefile configs/eu_large/base.yaml
base: ../kati_base.yaml
paths:
  raw: data/raw/eu_large
  processed: data/processed/eu_large/with_zero_returns
  outputs: outputs/base/eu_large
  reference_r: data/reference
dates:
  warmup_start: '1994-12-31'
  validation_start: '2015-01-31'
  validation_end: '2019-12-31'
  test_start: '2020-01-31'
  test_end: '2025-12-31'
  holdout_start: null
  holdout_end: null
  data_end: '2025-12-31'
  holdout_locked: true
  evaluate: test
features:
  themes:
    accruals:
    - seas_16_20na
    - cowc_gr1a
    debt_issuance:
    - ni_ar1
    - fnl_gr1a
    investment:
    - mispricing_mgmt
    - inv_gr1a
    - aliq_at
    - capx_gr3
    - sale_gr1
    low_leverage:
    - netdebt_me
    - age
    - ni_ivol
    - bidaskhl_21d
    low_risk:
    - rmax5_21d
    - betadown_252d
    - seas_6_10na
    - betabab_1260d
    - rvol_21d
    - rmax1_21d
    momentum:
    - ret_6_1
    - ret_12_1
    - resff3_6_1
    profit_growth:
    - sale_emp_gr1
    - ret_12_7
    - seas_1_1an
    - dsale_dsga
    - dsale_drec
    profitability:
    - ebit_sale
    - ebit_bev
    - turnover_var_126d
    quality:
    - at_turnover
    - mispricing_perf
    - sale_bev
    - qmj_safety
    - niq_at
    - qmj_prof
    seasonality:
    - corr_1260d
    - seas_2_5an
    - seas_11_15an
    - coskew_21d
    short_term_reversal:
    - rmax5_rvol_21d
    - rskew_21d
    size:
    - market_equity
    - dolvol_126d
    value:
    - ebitda_mev
    - eqnpo_12m
    - div12m_me
    - fcf_me
    - debt_me
    - at_me
return_model:
  validation_scope: universe
  pred_years:
  - 2015
  - 2025
  val_years: 5
investor:
  wealth_end_date: '2019-12-31'
reporting:
  main_types: [Portfolio-ML]
  subperiods:
  - label: 2020-2022
    start: '2020-01-31'
    end: '2022-12-31'
  - label: 2023-2025
    start: '2023-01-31'
    end: '2025-12-31'
  cost_vix_start: '2020-01-31'
grids:
  n_jobs: 1
regimes:
  plot_types: [Portfolio-ML]


### `configs/kati_base.yaml`

Declarative settings for the local Kati workflow; paths resolve inside Kati_tryout.

**Role:** Pipeline implementation or configuration. This cell writes the file without running its main function.

**Input:** the embedded source below. **Output:** `configs/kati_base.yaml`. Expand the code cell to read or edit it.


In [ ]:
%%writefile configs/kati_base.yaml
seed: 1
paths:
  raw: data/raw
  processed: data/processed
  outputs: outputs
  reference_r: data/reference
data_sources:
  jkp_table: contrib.global_factor
  crsp_daily_table: crsp.dsf_v2
  factor_details_url: https://raw.githubusercontent.com/bkelly-lab/ReplicationCrisis/master/GlobalFactors/Factor%20Details.xlsx
  cluster_labels_url: https://raw.githubusercontent.com/bkelly-lab/ReplicationCrisis/master/GlobalFactors/Cluster%20Labels.csv
  ff_factors_url: https://mba.tuck.dartmouth.edu/pages/faculty/ken.french/ftp/F-F_Research_Data_Factors_CSV.zip
  ff_factors_daily_url: https://mba.tuck.dartmouth.edu/pages/faculty/ken.french/ftp/F-F_Research_Data_Factors_daily_CSV.zip
  fred_csv_url: https://fred.stlouisfed.org/graph/fredgraph.csv?id={series}
dates:
  warmup_start: '1994-12-31'
  validation_start: '2015-01-31'
  validation_end: '2019-12-31'
  test_start: '2020-01-31'
  test_end: '2025-12-31'
  holdout_start: null
  holdout_end: null
  data_end: 2025-12-31
  holdout_locked: true
  evaluate: test
universe:
  exchanges:
  - 1
  - 3
  share_codes:
  - 10
  - 11
  - 12
  jkp_flags:
  - common
  - primary_sec
  - obs_main
  target_n: 80
  entry_rank: 70
  exit_rank: 100
  reconstitution_month: 12
  min_feature_pct: 0.5
  lookback_months: 11
features:
  percentile_rank: true
  impute_value: 0.5
  themes:
    accruals:
    - seas_16_20na
    - cowc_gr1a
    debt_issuance:
    - ni_ar1
    - fnl_gr1a
    investment:
    - mispricing_mgmt
    - inv_gr1a
    - aliq_at
    - capx_gr3
    - sale_gr1
    low_leverage:
    - netdebt_me
    - age
    - ni_ivol
    - bidaskhl_21d
    low_risk:
    - rmax5_21d
    - betadown_252d
    - seas_6_10na
    - betabab_1260d
    - rvol_21d
    - rmax1_21d
    momentum:
    - ret_6_1
    - ret_12_1
    - resff3_6_1
    profit_growth:
    - sale_emp_gr1
    - ret_12_7
    - seas_1_1an
    - dsale_dsga
    - dsale_drec
    profitability:
    - ebit_sale
    - ebit_bev
    - turnover_var_126d
    quality:
    - at_turnover
    - mispricing_perf
    - sale_bev
    - qmj_safety
    - niq_at
    - qmj_prof
    seasonality:
    - corr_1260d
    - seas_2_5an
    - seas_11_15an
    - coskew_21d
    short_term_reversal:
    - rmax5_rvol_21d
    - rskew_21d
    size:
    - market_equity
    - dolvol_126d
    value:
    - ebitda_mev
    - eqnpo_12m
    - div12m_me
    - fcf_me
    - debt_me
    - at_me
covariance:
  factor_universe_top_n: 500
  sectors:
    consumer:
    - NoDur
    - Durbl
    - Shops
    industrials:
    - Manuf
    - Chems
    - Other
    tech_telecom:
    - BusEq
    - Telcm
    energy_utilities:
    - Enrgy
    - Utils
    health:
    - Hlth
    finance:
    - Money
  window_days: 2520
  min_window_days: 2520
  half_life_corr_days: 378
  half_life_var_days: 126
  half_life_idio_days: 126
  idio_min_obs: 200
  idio_min_window: 252
  idio_initial_obs: 63
  days_per_month: 21
trading_costs:
  price_impact: 0.1
  dolvol_column: dolvol_126d
  dolvol_units: dollars
investor:
  gamma: 10.0
  wealth: 10000000000.0
  wealth_end_date: '2019-12-31'
  mu: 0.007
  m_iterations: 10
  gamma_te: 100.0
  te_targets:
  - 0.01
  - 0.02
  - 0.04
  gamma_te_grid:
  - 1.0
  - 2.5
  - 5.0
  - 10.0
  - 25.0
  - 50.0
  - 100.0
  - 250.0
  gamma_grid:
  - 1.0
  - 5.0
  - 10.0
  - 20.0
  - 100.0
  wealth_grid:
  - 1000000000.0
  - 10000000000.0
  - 50000000000.0
  - 100000000000.0
  - 250000000000.0
portfolio:
  charge_exit_tc: true
ridge_grid:
  include_zero: true
  log_min: -10.0
  log_max: 10.0
  n: 100
return_model:
  validation_scope: all
  p_grid:
  - 2
  - 4
  - 8
  - 16
  - 32
  - 64
  - 128
  - 256
  - 512
  rff_variance_grid:
  - 0.049787068367863944
  - 0.1353352832366127
  horizons:
  - 1
  val_years: 5
  pred_years:
  - 2015
  - 2025
portfolio_ml:
  use_rff: true
  p_grid:
  - 8
  - 16
  - 32
  - 64
  - 128
  - 256
  - 512
  rff_variance_grid:
  - 0.049787068367863944
  - 0.1353352832366127
  scale_by_vol: true
  add_original_features: false
  add_constant: true
  benchmark_relative: false
  half_life_years:
  - null
static_ml:
  k_grid:
  - 1.0
  - 0.3333333333333333
  - 0.2
  u_grid:
  - 0.0
  - 0.1
  - 0.25
  - 0.5
  - 1.0
  g_grid:
  - 0.0
  - 1.0
  - 2.0
factor_ml:
  n_portfolios: 5
monitor:
  ir_window: 36
  drawdown_te_multiple: 2.0
  te_budget: 0.02
  beta_limits:
  - 0.8
  - 1.2
  theme_limit: 1.0
  ic_window: 12
  ic_negative_months: 6
  tc_months: 3
  bias_window: 36
grids:
  n_jobs: 5
  frontier_vol_max: 0.5
  frontier_vol_step: 0.01
feature_importance:
  markowitz_gamma: 100.0
  n_seeds: 20
reporting:
  main_types:
  - Static-ML*
  - Portfolio-ML
  subperiods:
  - label: "1981\u20132000"
    start: 1981-01-31
    end: 2000-12-31
  - label: "2001\u20132020"
    start: 2001-01-31
    end: 2020-12-31
  ic_size_buckets:
  - - 1
    - 80
  - - 81
    - 250
  capacity_ir: 0.3
  cost_vix_start: 1990-01-31
  cost_vol_window: 60
regimes:
  fred_series:
  - USREC
  - VIXCLS
  - FEDFUNDS
  fed_change_months: 3
  fed_threshold: 0.1
  vix_norm: window
  plot_types:
  - Static-ML*
  - Portfolio-ML
  episodes:
  - label: COVID crash
    start: 2020-02-01
    end: 2020-04-30
  - label: 2022 hikes / bear market
    start: 2022-01-01
    end: 2022-12-31
  - label: SVB / regional banks
    start: 2023-03-01
    end: 2023-03-31
  - label: Tariff shock
    start: 2025-04-01
    end: 2025-04-30


### `configs/us_large/base.yaml`

Declarative settings for the local Kati workflow; paths resolve inside Kati_tryout.

**Role:** Pipeline implementation or configuration. This cell writes the file without running its main function.

**Input:** the embedded source below. **Output:** `configs/us_large/base.yaml`. Expand the code cell to read or edit it.


In [ ]:
%%writefile configs/us_large/base.yaml
base: ../kati_base.yaml
static_ml:
  # Expanded base-only candidate range; the Static ML* formula is unchanged.
  k_grid: [1.0, 0.3333333333333333, 0.2, 0.1, 0.05]
  u_grid: [0.0, 0.1, 0.25, 0.5, 1.0]
  g_grid: [0.0, 1.0, 2.0, 3.0]
paths:
  raw: data/raw/us_large
  processed: data/processed/us_large/with_zero_returns
  outputs: outputs/base/us_large
  reference_r: data/reference
dates:
  warmup_start: '1994-12-31'
  validation_start: '2015-01-31'
  validation_end: '2019-12-31'
  test_start: '2020-01-31'
  test_end: '2025-12-31'
  holdout_start: null
  holdout_end: null
  data_end: '2025-12-31'
  holdout_locked: true
  evaluate: test
features:
  themes:
    accruals:
    - seas_16_20na
    - cowc_gr1a
    debt_issuance:
    - ni_ar1
    - fnl_gr1a
    investment:
    - mispricing_mgmt
    - inv_gr1a
    - aliq_at
    - capx_gr3
    - sale_gr1
    low_leverage:
    - netdebt_me
    - age
    - ni_ivol
    - bidaskhl_21d
    low_risk:
    - rmax5_21d
    - betadown_252d
    - seas_6_10na
    - betabab_1260d
    - rvol_21d
    - rmax1_21d
    momentum:
    - ret_6_1
    - ret_12_1
    - resff3_6_1
    profit_growth:
    - sale_emp_gr1
    - ret_12_7
    - seas_1_1an
    - dsale_dsga
    - dsale_drec
    profitability:
    - ebit_sale
    - ebit_bev
    - turnover_var_126d
    quality:
    - at_turnover
    - mispricing_perf
    - sale_bev
    - qmj_safety
    - niq_at
    - qmj_prof
    seasonality:
    - corr_1260d
    - seas_2_5an
    - seas_11_15an
    - coskew_21d
    short_term_reversal:
    - rmax5_rvol_21d
    - rskew_21d
    size:
    - market_equity
    - dolvol_126d
    value:
    - ebitda_mev
    - eqnpo_12m
    - div12m_me
    - fcf_me
    - debt_me
    - at_me
return_model:
  validation_scope: universe
  pred_years:
  - 2015
  - 2025
  val_years: 5
investor:
  wealth_end_date: '2019-12-31'
reporting:
  subperiods:
  - label: 2020-2022
    start: '2020-01-31'
    end: '2022-12-31'
  - label: 2023-2025
    start: '2023-01-31'
    end: '2025-12-31'
  cost_vix_start: '2020-01-31'
grids:
  n_jobs: 1


### `configs/us_large/iteration_1.yaml`

Declarative settings for the local Kati workflow; paths resolve inside Kati_tryout.

**Role:** Pipeline implementation or configuration. This cell writes the file without running its main function.

**Input:** the embedded source below. **Output:** `configs/us_large/iteration_1.yaml`. Expand the code cell to read or edit it.


In [ ]:
%%writefile configs/us_large/iteration_1.yaml
base: base.yaml
portfolio_ml:
  add_constant: true
  benchmark_relative: false
paths:
  outputs: outputs/iteration_1/us_large
  reference_r: data/reference
features:
  themes:
    accruals:
    - seas_16_20na
    - cowc_gr1a
    debt_issuance:
    - ni_ar1
    - fnl_gr1a
    investment:
    - mispricing_mgmt
    - inv_gr1a
    - aliq_at
    - capx_gr3
    - sale_gr1
    low_leverage:
    - netdebt_me
    - age
    - ni_ivol
    - bidaskhl_21d
    low_risk:
    - rmax5_21d
    - betadown_252d
    - seas_6_10na
    - betabab_1260d
    - rvol_21d
    - rmax1_21d
    momentum:
    - ret_6_1
    - ret_12_1
    - resff3_6_1
    profit_growth:
    - sale_emp_gr1
    - ret_12_7
    - seas_1_1an
    - dsale_dsga
    - dsale_drec
    profitability:
    - ebit_sale
    - ebit_bev
    - turnover_var_126d
    quality:
    - at_turnover
    - mispricing_perf
    - sale_bev
    - qmj_safety
    - niq_at
    - qmj_prof
    seasonality:
    - corr_1260d
    - seas_2_5an
    - seas_11_15an
    - coskew_21d
    short_term_reversal:
    - rmax5_rvol_21d
    - rskew_21d
    size:
    - market_equity
    - dolvol_126d
    value:
    - ebitda_mev
    - eqnpo_12m
    - div12m_me
    - fcf_me
    - debt_me
    - at_me


### `configs/us_large/iteration_2.yaml`

Declarative settings for the local Kati workflow; paths resolve inside Kati_tryout.

**Role:** Pipeline implementation or configuration. This cell writes the file without running its main function.

**Input:** the embedded source below. **Output:** `configs/us_large/iteration_2.yaml`. Expand the code cell to read or edit it.


In [ ]:
%%writefile configs/us_large/iteration_2.yaml
base: base.yaml
paths:
  processed: data/processed/us_large/with_zero_returns
  outputs: outputs/iteration_2/us_large
  reference_r: data/reference
investor:
  gamma: 5.0
features:
  themes:
    accruals:
    - seas_16_20na
    - cowc_gr1a
    debt_issuance:
    - ni_ar1
    - fnl_gr1a
    investment:
    - mispricing_mgmt
    - inv_gr1a
    - aliq_at
    - capx_gr3
    - sale_gr1
    low_leverage:
    - netdebt_me
    - age
    - ni_ivol
    - bidaskhl_21d
    low_risk:
    - rmax5_21d
    - betadown_252d
    - seas_6_10na
    - betabab_1260d
    - rvol_21d
    - rmax1_21d
    momentum:
    - ret_6_1
    - ret_12_1
    - resff3_6_1
    profit_growth:
    - sale_emp_gr1
    - ret_12_7
    - seas_1_1an
    - dsale_dsga
    - dsale_drec
    profitability:
    - ebit_sale
    - ebit_bev
    - turnover_var_126d
    quality:
    - at_turnover
    - mispricing_perf
    - sale_bev
    - qmj_safety
    - niq_at
    - qmj_prof
    seasonality:
    - corr_1260d
    - seas_2_5an
    - seas_11_15an
    - coskew_21d
    short_term_reversal:
    - rmax5_rvol_21d
    - rskew_21d
    size:
    - market_equity
    - dolvol_126d
    value:
    - ebitda_mev
    - eqnpo_12m
    - div12m_me
    - fcf_me
    - debt_me
    - at_me


### `configs/us_large/iteration_3.yaml`

Declarative settings for the local Kati workflow; paths resolve inside Kati_tryout.

**Role:** Pipeline implementation or configuration. This cell writes the file without running its main function.

**Input:** the embedded source below. **Output:** `configs/us_large/iteration_3.yaml`. Expand the code cell to read or edit it.


In [ ]:
%%writefile configs/us_large/iteration_3.yaml
base: base.yaml
paths:
  outputs: outputs/iteration_3/us_large
  reference_r: data/reference
universe:
  exit_rank: 120
trading_costs:
  price_impact: 0.2
features:
  themes:
    accruals:
    - seas_16_20na
    - cowc_gr1a
    debt_issuance:
    - ni_ar1
    - fnl_gr1a
    investment:
    - mispricing_mgmt
    - inv_gr1a
    - aliq_at
    - capx_gr3
    - sale_gr1
    low_leverage:
    - netdebt_me
    - age
    - ni_ivol
    - bidaskhl_21d
    low_risk:
    - rmax5_21d
    - betadown_252d
    - seas_6_10na
    - betabab_1260d
    - rvol_21d
    - rmax1_21d
    momentum:
    - ret_6_1
    - ret_12_1
    - resff3_6_1
    profit_growth:
    - sale_emp_gr1
    - ret_12_7
    - seas_1_1an
    - dsale_dsga
    - dsale_drec
    profitability:
    - ebit_sale
    - ebit_bev
    - turnover_var_126d
    quality:
    - at_turnover
    - mispricing_perf
    - sale_bev
    - qmj_safety
    - niq_at
    - qmj_prof
    seasonality:
    - corr_1260d
    - seas_2_5an
    - seas_11_15an
    - coskew_21d
    short_term_reversal:
    - rmax5_rvol_21d
    - rskew_21d
    size:
    - market_equity
    - dolvol_126d
    value:
    - ebitda_mev
    - eqnpo_12m
    - div12m_me
    - fcf_me
    - debt_me
    - at_me


### `configs/us_large/iteration_4.yaml`

Declarative settings for the local Kati workflow; paths resolve inside Kati_tryout.

**Role:** Pipeline implementation or configuration. This cell writes the file without running its main function.

**Input:** the embedded source below. **Output:** `configs/us_large/iteration_4.yaml`. Expand the code cell to read or edit it.


In [ ]:
%%writefile configs/us_large/iteration_4.yaml
base: base.yaml
paths:
  outputs: outputs/iteration_4/us_large
portfolio_ml:
  benchmark_relative: true
  add_constant: false
investor:
  te_targets: [0.01]


### `configs/us_large/iteration_5.yaml`

Declarative settings for the local Kati workflow; paths resolve inside Kati_tryout.

**Role:** Pipeline implementation or configuration. This cell writes the file without running its main function.

**Input:** the embedded source below. **Output:** `configs/us_large/iteration_5.yaml`. Expand the code cell to read or edit it.


In [ ]:
%%writefile configs/us_large/iteration_5.yaml
base: base.yaml
paths:
  outputs: outputs/iteration_5/us_large
portfolio_ml:
  half_life_years: [3, 5, 10, null]


### `configs/us_mid/base.yaml`

Declarative settings for the local Kati workflow; paths resolve inside Kati_tryout.

**Role:** Pipeline implementation or configuration. This cell writes the file without running its main function.

**Input:** the embedded source below. **Output:** `configs/us_mid/base.yaml`. Expand the code cell to read or edit it.


In [ ]:
%%writefile configs/us_mid/base.yaml
base: ../kati_base.yaml
paths:
  raw: data/raw/us_mid
  processed: data/processed/us_mid/with_zero_returns
  outputs: outputs/base/us_mid
  reference_r: data/reference
dates:
  warmup_start: '1994-12-31'
  validation_start: '2015-01-31'
  validation_end: '2019-12-31'
  test_start: '2020-01-31'
  test_end: '2025-12-31'
  holdout_start: null
  holdout_end: null
  data_end: '2025-12-31'
  holdout_locked: true
  evaluate: test
features:
  themes:
    accruals:
    - seas_16_20na
    - cowc_gr1a
    debt_issuance:
    - ni_ar1
    - fnl_gr1a
    investment:
    - mispricing_mgmt
    - inv_gr1a
    - aliq_at
    - capx_gr3
    - sale_gr1
    low_leverage:
    - netdebt_me
    - age
    - ni_ivol
    - bidaskhl_21d
    low_risk:
    - rmax5_21d
    - betadown_252d
    - seas_6_10na
    - betabab_1260d
    - rvol_21d
    - rmax1_21d
    momentum:
    - ret_6_1
    - ret_12_1
    - resff3_6_1
    profit_growth:
    - sale_emp_gr1
    - ret_12_7
    - seas_1_1an
    - dsale_dsga
    - dsale_drec
    profitability:
    - ebit_sale
    - ebit_bev
    - turnover_var_126d
    quality:
    - at_turnover
    - mispricing_perf
    - sale_bev
    - qmj_safety
    - niq_at
    - qmj_prof
    seasonality:
    - corr_1260d
    - seas_2_5an
    - seas_11_15an
    - coskew_21d
    short_term_reversal:
    - rmax5_rvol_21d
    - rskew_21d
    size:
    - market_equity
    - dolvol_126d
    value:
    - ebitda_mev
    - eqnpo_12m
    - div12m_me
    - fcf_me
    - debt_me
    - at_me
return_model:
  validation_scope: universe
  pred_years:
  - 2015
  - 2025
  val_years: 5
investor:
  wealth_end_date: '2019-12-31'
reporting:
  main_types: [Portfolio-ML]
  ic_size_buckets:
  - - 461
    - 540
  - - 376
    - 460
  - - 541
    - 625
  subperiods:
  - label: 2020-2022
    start: '2020-01-31'
    end: '2022-12-31'
  - label: 2023-2025
    start: '2023-01-31'
    end: '2025-12-31'
  cost_vix_start: '2020-01-31'
grids:
  n_jobs: 1
regimes:
  plot_types: [Portfolio-ML]


### `configs/us_high_cap_iterations.json`

Declarative settings for the local Kati workflow; paths resolve inside Kati_tryout.

**Role:** Pipeline implementation or configuration. This cell writes the file without running its main function.

**Input:** the embedded source below. **Output:** `configs/us_high_cap_iterations.json`. Expand the code cell to read or edit it.


In [ ]:
%%writefile configs/us_high_cap_iterations.json
{
  "schema_version": 1,
  "universe": "us_large",
  "evaluation_period": "existing Kati test period; no separate unseen holdout",
  "acceptance_threshold": 0.8,
  "statistic": "paired circular-block bootstrap frequency of a positive paper-net-Sharpe difference",
  "bootstrap_reps": 10000,
  "bootstrap_block_months": 6,
  "bootstrap_seed": 20261102,
  "drawdown_is_acceptance_gate": false,
  "iterations": [
    {"number": 1, "name": "Refit yearly", "kind": "already_in_base"},
    {"number": 2, "name": "Gamma 5", "override": {"gamma": 5.0}},
    {"number": 3, "name": "Exit 120, costs x2", "override": {"exit_rank": 120, "cost_multiple": 2.0}},
    {"number": 4, "name": "Market + 1% TE-target tilt", "override": {"benchmark_relative": true, "te_target": 0.01}},
    {"number": 5, "name": "Recent-history weighting", "override": {"half_lives": [3, 5, 10, null]}}
  ]
}


### `settings.json`

Declarative settings for the local Kati workflow; paths resolve inside Kati_tryout.

**Role:** Pipeline implementation or configuration. This cell writes the file without running its main function.

**Input:** the embedded source below. **Output:** `settings.json`. Expand the code cell to read or edit it.


In [ ]:
%%writefile settings.json
{
  "source": "data/source",
  "reference_data": "data/reference",
  "universes": [
    "us_large",
    "us_mid",
    "eu_large"
  ],
  "features": "Published JKP-only US high-cap association-selected 50",
  "downloads_allowed": false,
  "missing_forward_return_policy": "zero_excess",
  "missing_lag_history_policy": "exclude_affected_stock_months",
  "train_start": "2005-01-31",
  "validation_start": "2015-01-31",
  "validation_end": "2019-12-31",
  "test_start": "2020-01-31",
  "test_end": "2025-12-31",
  "wealth_anchor": "2019-12-31",
  "currency": "USD",
  "daily_inputs": {
    "US": "data/inputs/daily_US.parquet",
    "EU": "data/inputs/daily_EU.parquet"
  },
  "selection_scope": "us_large",
  "selection_return_start": "2005-01-31",
  "selection_return_end": "2014-12-31",
  "selection_models_fitted": false,
  "old_feature_archives_removed": true,
  "jkp_only": true
}


## US high-cap correlation and covariance selection — no ML training

**Purpose:** reproduce the current 50 from 153 verified published JKP candidates using only US high-cap observations in 2005–2014. **Input:** the self-contained US high-cap snapshot and published reference metadata. **Output:** rankings, A/T/S reasons, 50×50 feature matrices, return-augmented matrices, per-stock associations and interactive charts. This optional cell performs no ML fit and does not install a changed list over fitted models.


In [ ]:
RECOMPUTE_ASSOCIATIONS = False
if RECOMPUTE_ASSOCIATIONS:
    subprocess.run([sys.executable, "-X", "utf8", str(KATI_ROOT / "scripts/select_characteristics.py")],
                   cwd=KATI_ROOT, check=True)
else:
    print("Existing association report: outputs/us_high_cap_selection/index.html")


## Inspect the workflow before fitting

**Purpose:** list the real job dependencies. **Input:** organised scripts/configurations. **Output:** printed execution plan. `--list-jobs` does not launch model fits. The first run needs fresh forecasts, covariance and portfolio coefficients for the new features.


In [ ]:
command = [sys.executable, "-X", "utf8", str(KATI_ROOT / "scripts/run_all.py"),
           "--group", "all", "--scope", "full", "--universe", "all", "--list-jobs"]
subprocess.run(command, cwd=KATI_ROOT, check=True)


## Verify the local build

**Purpose:** check the frozen inputs, local paths, source/notebook structure and implementation safeguards. **Input:** this local build. **Output:** the verification report and its printed status. This check does not establish portfolio performance; model results require the fresh fitting stages below.


In [ ]:
subprocess.run([sys.executable, "-X", "utf8", str(KATI_ROOT / "scripts/verify.py")],
               cwd=KATI_ROOT, check=True)


## Choose whether to run models

**Purpose:** make the long computation explicit. **Input:** the controls below. **Output:** an execution command, followed by model results only if `RUN_MODELS=True`.

Leave both flags false to browse and verify the build. Set `RUN_MODELS=True` for the full current experiment scope. `RUN_FIGURE1` controls the additional frontier fits: false adds `--skip-figure1`, while true includes the frontier stage. The full CLI command is in the user guide. The model run may take many hours and reuses only compatible completed Kati outputs on a later invocation.


In [ ]:
RUN_MODELS = False
RUN_FIGURE1 = False
RUN_GROUP = "all"
MARKETS = "all"

runner = KATI_ROOT / "scripts/run_all.py"
model_command = [sys.executable, "-u", "-X", "utf8", str(runner),
                 "--group", RUN_GROUP, "--scope", "full", "--universe", MARKETS]
if not RUN_FIGURE1:
    model_command.append("--skip-figure1")
print("Command:", subprocess.list2cmdline(model_command))
if RUN_MODELS:
    subprocess.run(model_command, cwd=KATI_ROOT, check=True)
else:
    print("Model fits have not been started by this cell.")


## Refresh the results explorer

**Purpose:** create a local dashboard from results that actually exist. **Input:** Kati result files. **Output:** charts, tables and report navigation. Before fitting, the explorer shows selection evidence and marks missing model results as unavailable. It does not substitute Keats performance. Use market, run and measure controls in the explorer where offered.


In [ ]:
subprocess.run([sys.executable, "-X", "utf8", str(KATI_ROOT / "scripts/dashboard.py")],
               cwd=KATI_ROOT, check=True)
from IPython.display import HTML, display
dashboard = next((p for p in [KATI_ROOT / "outputs/dashboard.html", KATI_ROOT / "outputs/index.html"]
                  if p.exists()), None)
if dashboard is not None:
    relative = dashboard.relative_to(KATI_ROOT).as_posix()
    display(HTML(f'<p><a href="{relative}" target="_blank">Open the full results explorer</a></p>'
                 f'<iframe src="{relative}" style="width:100%;height:850px;border:1px solid #d5dee5"></iframe>'))
else:
    print("No dashboard has been generated yet; inspect the dashboard command's output.")


## Explain the frozen 50 and their groups

**Purpose:** display the shortlist explanation, separately from fitted investment performance. **Input:** selection charts copied into Kati outputs. **Output:** presentation overview and detailed evidence chart, when available. The three selection codes mean **A** published group anchor, **T** minimum theme representation, and **S** balanced score with a redundancy penalty. They are reasons for inclusion, not ML coefficients or importance weights.


In [ ]:
from IPython.display import Image, display
for filename in ("selection_groups_and_reasons.png", "selected50_rank_correlation.png", "selected50_rank_covariance.png"):
    image_path = KATI_ROOT / "outputs/us_high_cap_selection/charts" / filename
    if image_path.exists():
        display(Image(filename=str(image_path)))
    else:
        print(filename, "is not present; the frozen names and metadata are under data/characteristics/.")


## Where to inspect completed results

| Question | Local result location |
|---|---|
| Which characteristic data and labels are used? | `data/characteristics/` and `data/SNAPSHOT_MANIFEST.json` |
| Which base model/market performs better? | `outputs/base/overall analysis/` |
| What changed in a specific iteration? | `outputs/iteration_N/us_large/overall analysis/` |
| How do the four Portfolio ML versions compare? | `outputs/base/overall analysis/us_high_cap_progression/` |
| How do all models and iterations compare? | `outputs/base/overall analysis/all_models_and_iterations/` |
| Where are Figure 1 charts? | `outputs/base/overall analysis/figure1/` and iteration extensions |
| What ran, failed, or was blocked? | `outputs/organisation_audit/run_logs/` for the all-group workflow |

CSV/Parquet tables and charts sit beside their corresponding report. Read the availability/status files before citing a result. A blocked downstream job means its prerequisite did not finish; it is not a zero-return observation. Read `docs/ASSUMPTIONS_AND_CHANGES.md` for timing, imputation, selection and interpretation limits.
